# Olympus routing v6: retained-source handoff

CPU and internet disabled. The complete source is embedded and checked. Requires Python >=3.11, Torch >=2.4, NumPy, Pydantic >=2.10 and pytest. The default performs artificial qualification only. Scientific reproduction requires separate admission. Save the retained archive before ending the session. A generated notebook is not Kaggle execution evidence.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, tarfile, time

PAYLOAD = {'LICENSE': 'T2x5bXB1cyBQcm9wcmlldGFyeSBMaWNlbnNlCgpDb3B5cmlnaHQgKGMpIDIwMjYgUnlhbi4gQWxsIHJpZ2h0cyByZXNlcnZlZC4KClRoZSBzb3VyY2UgY29kZSwgZG9jdW1lbnRhdGlvbiwgZGF0YSBkZWZpbml0aW9ucywgYW5kIG90aGVyIG1hdGVyaWFscyBpbiB0aGlzCnJlcG9zaXRvcnkgYXJlIG1hZGUgcHVibGljbHkgdmlld2FibGUgZm9yIGV2YWx1YXRpb24gYW5kIHJlc2VhcmNoIGRpc2N1c3Npb24uCk5vIHBlcm1pc3Npb24gaXMgZ3JhbnRlZCB0byBjb3B5LCBtb2RpZnksIGRpc3RyaWJ1dGUsIHN1YmxpY2Vuc2UsIHNlbGwsIGhvc3QsCm9yIGNyZWF0ZSBkZXJpdmF0aXZlIHdvcmtzIGZyb20gdGhlc2UgbWF0ZXJpYWxzIGV4Y2VwdCB3aGVyZSBhIHNlcGFyYXRlCndyaXR0ZW4gYWdyZWVtZW50IHdpdGggdGhlIGNvcHlyaWdodCBob2xkZXIgZXhwcmVzc2x5IHBlcm1pdHMgaXQuCgpUSEUgTUFURVJJQUxTIEFSRSBQUk9WSURFRCAiQVMgSVMiLCBXSVRIT1VUIFdBUlJBTlRZIE9GIEFOWSBLSU5ELCBFWFBSRVNTIE9SCklNUExJRUQsIElOQ0xVRElORyBCVVQgTk9UIExJTUlURUQgVE8gV0FSUkFOVElFUyBPRiBNRVJDSEFOVEFCSUxJVFksIEZJVE5FU1MKRk9SIEEgUEFSVElDVUxBUiBQVVJQT1NFLCBUSVRMRSwgQU5EIE5PTi1JTkZSSU5HRU1FTlQuIElOIE5PIEVWRU5UIFNIQUxMIFRIRQpDT1BZUklHSFQgSE9MREVSIEJFIExJQUJMRSBGT1IgQU5ZIENMQUlNLCBEQU1BR0VTLCBPUiBPVEhFUiBMSUFCSUxJVFkgQVJJU0lORwpGUk9NLCBPVVQgT0YsIE9SIElOIENPTk5FQ1RJT04gV0lUSCBUSEUgTUFURVJJQUxTIE9SIFRIRUlSIFVTRS4K', 'olympus/__init__.py': 'IiIiT2x5bXB1cyBjb2duaXRpdmUgYXJjaGl0ZWN0dXJlLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKX192ZXJzaW9uX18gPSAiMC4yLjAiCgpfX2FsbF9fID0gWyJfX3ZlcnNpb25fXyIsICJhcHAiXQoKCmRlZiBfX2dldGF0dHJfXyhuYW1lOiBzdHIpIC0+IEFueToKICAgIGlmIG5hbWUgPT0gImFwcCI6CiAgICAgICAgZnJvbSBvbHltcHVzLmFwaSBpbXBvcnQgYXBwCgogICAgICAgIHJldHVybiBhcHAKICAgIHJhaXNlIEF0dHJpYnV0ZUVycm9yKGYibW9kdWxlIHtfX25hbWVfXyFyfSBoYXMgbm8gYXR0cmlidXRlIHtuYW1lIXJ9IikK', 'olympus/api.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzZWNyZXRzCmltcG9ydCB0aHJlYWRpbmcKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEFzeW5jSXRlcmF0b3IsIEl0ZXJhdG9yCmZyb20gY29udGV4dGxpYiBpbXBvcnQgYXN5bmNjb250ZXh0bWFuYWdlcgpmcm9tIGZ1bmN0b29scyBpbXBvcnQgbHJ1X2NhY2hlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBUZW1wb3JhcnlEaXJlY3RvcnkKZnJvbSB0eXBpbmcgaW1wb3J0IEFubm90YXRlZCwgTGl0ZXJhbAoKaW1wb3J0IGh0dHB4CmZyb20gZmFzdGFwaSBpbXBvcnQgRGVwZW5kcywgRmFzdEFQSSwgSGVhZGVyLCBIVFRQRXhjZXB0aW9uCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgQ29uZmlnRGljdCwgRmllbGQKCmZyb20gb2x5bXB1cyBpbXBvcnQgX192ZXJzaW9uX18KZnJvbSBvbHltcHVzLmNvcmUuaW50ZXJwcmV0YXRpb24gaW1wb3J0IEludGVycHJldGl2ZVN1cGVycG9zaXRpb25OZXR3b3JrCmZyb20gb2x5bXB1cy5jb3JlLndvcmtzcGFjZSBpbXBvcnQgTGF0ZW50V29ya3NwYWNlCmZyb20gb2x5bXB1cy5kZW1vcyBpbXBvcnQgKAogICAgcnVuX2FtYmlndW91c19pbnRlcnByZXRhdGlvbl9kZW1vLAogICAgcnVuX2RpdmlzaW9uX2RlbW8sCiAgICBydW5fZm9yZ2VfZGVtbywKICAgIHJ1bl9oZXJtZXNfZGVtbywKICAgIHJ1bl9yZXByZXNlbnRhdGlvbl9kZW1vLAogICAgcnVuX3JldHJvZGljdGlvbl9kZW1vLAogICAgcnVuX3RyYW5zZm9ybV9kZW1vLAopCmZyb20gb2x5bXB1cy5mb3JnZS5jb21waWxlciBpbXBvcnQgTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcgpmcm9tIG9seW1wdXMuZm9yZ2UucnVudGltZSBpbXBvcnQgRm9yZ2VSdW50aW1lCmZyb20gb2x5bXB1cy5mb3VuZHJ5Lm9sbGFtYSBpbXBvcnQgT2xsYW1hQ2xpZW50CmZyb20gb2x5bXB1cy5mb3VuZHJ5LnNlcnZpY2UgaW1wb3J0IEZvdW5kcnlTZXJ2aWNlCgoKY2xhc3MgQXBpUmVxdWVzdChCYXNlTW9kZWwpOgogICAgbW9kZWxfY29uZmlnID0gQ29uZmlnRGljdChleHRyYT0iZm9yYmlkIiwgc3RyX3N0cmlwX3doaXRlc3BhY2U9VHJ1ZSkKCgpjbGFzcyBXb3Jrc3BhY2VSZXF1ZXN0KEFwaVJlcXVlc3QpOgogICAgb2JqZWN0aXZlOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9Ml8wMDApCiAgICBwcm9tcHQ6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwgbWF4X2xlbmd0aD01MF8wMDApCgoKY2xhc3MgQ29tcGlsZVJlcXVlc3QoQXBpUmVxdWVzdCk6CiAgICBiZWhhdmlvcjogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTIwXzAwMCkKCgpjbGFzcyBSdW5SZXF1ZXN0KENvbXBpbGVSZXF1ZXN0KToKICAgIHByb21wdDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTUwXzAwMCkKCgpjbGFzcyBDaGF0TWVzc2FnZShBcGlSZXF1ZXN0KToKICAgIHJvbGU6IExpdGVyYWxbInN5c3RlbSIsICJ1c2VyIiwgImFzc2lzdGFudCJdCiAgICBjb250ZW50OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NTBfMDAwKQoKCmNsYXNzIENoYXRDb21wbGV0aW9uUmVxdWVzdChBcGlSZXF1ZXN0KToKICAgIG1vZGVsOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MjAwKQogICAgbWVzc2FnZXM6IGxpc3RbQ2hhdE1lc3NhZ2VdID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTEwMCkKICAgIG1heF90b2tlbnM6IGludCA9IEZpZWxkKGRlZmF1bHQ9MTYwLCBnZT0xLCBsZT00XzA5NikKICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MC43LCBnZT0wLjAsIGxlPTIuMCkKICAgIHNlZWQ6IGludCB8IE5vbmUgPSBGaWVsZChkZWZhdWx0PU5vbmUsIGdlPTAsIGxlPTIqKjMyIC0gMSkKICAgIHN0cmVhbTogTGl0ZXJhbFtGYWxzZV0gPSBGYWxzZQoKCmNsYXNzIE9sbGFtYUdlbmVyYXRlUmVxdWVzdChBcGlSZXF1ZXN0KToKICAgIG1vZGVsOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MjAwKQogICAgcHJvbXB0OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NTBfMDAwKQogICAgc3lzdGVtOiBzdHIgfCBOb25lID0gRmllbGQoZGVmYXVsdD1Ob25lLCBtYXhfbGVuZ3RoPTEwXzAwMCkKICAgIG1heF90b2tlbnM6IGludCA9IEZpZWxkKGRlZmF1bHQ9MjU2LCBnZT0xLCBsZT00XzA5NikKICAgIGNvbnRleHRfdG9rZW5zOiBpbnQgPSBGaWVsZChkZWZhdWx0PTJfMDQ4LCBnZT0yNTYsIGxlPTEzMV8wNzIpCiAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMCwgZ2U9MC4wLCBsZT0yLjApCgoKY29tcGlsZXIgPSBOYXR1cmFsTGFuZ3VhZ2VCZWhhdmlvckNvbXBpbGVyKCkKcnVudGltZSA9IEZvcmdlUnVudGltZSgpCmludGVycHJldGVyID0gSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsoKQoKCkBscnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgZ2V0X2ZvdW5kcnlfc2VydmljZSgpIC0+IEZvdW5kcnlTZXJ2aWNlOgogICAgcm9vdCA9IFBhdGgob3MuZW52aXJvbi5nZXQoIk9MWU1QVVNfRk9VTkRSWV9ST09UIiwgImFydGlmYWN0cy9mb3VuZHJ5IikpCiAgICByZXR1cm4gRm91bmRyeVNlcnZpY2Uocm9vdCkKCgpAbHJ1X2NhY2hlKG1heHNpemU9MSkKZGVmIGdldF9vbGxhbWFfY2xpZW50KCkgLT4gT2xsYW1hQ2xpZW50OgogICAgcmV0dXJuIE9sbGFtYUNsaWVudChvcy5lbnZpcm9uLmdldCgiT0xZTVBVU19PTExBTUFfVVJMIiwgImh0dHA6Ly8xMjcuMC4wLjE6MTE0MzQiKSkKCgpGb3VuZHJ5RGVwZW5kZW5jeSA9IEFubm90YXRlZFtGb3VuZHJ5U2VydmljZSwgRGVwZW5kcyhnZXRfZm91bmRyeV9zZXJ2aWNlKV0KT2xsYW1hRGVwZW5kZW5jeSA9IEFubm90YXRlZFtPbGxhbWFDbGllbnQsIERlcGVuZHMoZ2V0X29sbGFtYV9jbGllbnQpXQoKCmRlZiByZXF1aXJlX3dvcmtsb2FkX2F1dGhvcml6YXRpb24oCiAgICBhdXRob3JpemF0aW9uOiBBbm5vdGF0ZWRbc3RyIHwgTm9uZSwgSGVhZGVyKCldID0gTm9uZSwKKSAtPiBOb25lOgogICAgIiIiUmVxdWlyZSBhIGJlYXJlciB0b2tlbiBmb3Igc3RhdGVmdWwvY29tcHV0ZSByb3V0ZXMgd2hlbiBvbmUgaXMgY29uZmlndXJlZC4KCiAgICBPbHltcHVzIHJlbWFpbnMgdXNhYmxlIGFzIGEgbG9vcGJhY2stb25seSBkZXZlbG9wbWVudCBzZXJ2aWNlIHdpdGhvdXQgYSB0b2tlbi4KICAgIEFueSBkZXBsb3ltZW50IHRoYXQgY3Jvc3NlcyBhIHRydXN0IGJvdW5kYXJ5IG11c3Qgc2V0IGBgT0xZTVBVU19BUElfVE9LRU5gYC4KICAgICIiIgoKICAgIHRva2VuID0gb3MuZW52aXJvbi5nZXQoIk9MWU1QVVNfQVBJX1RPS0VOIikKICAgIGlmIHRva2VuIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgdG9rZW4gb3IgYXV0aG9yaXphdGlvbiBpcyBOb25lIG9yIG5vdCBzZWNyZXRzLmNvbXBhcmVfZGlnZXN0KAogICAgICAgIGF1dGhvcml6YXRpb24sIGYiQmVhcmVyIHt0b2tlbn0iCiAgICApOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oCiAgICAgICAgICAgIHN0YXR1c19jb2RlPTQwMSwKICAgICAgICAgICAgZGV0YWlsPSJ2YWxpZCBPbHltcHVzIGJlYXJlciB0b2tlbiByZXF1aXJlZCIsCiAgICAgICAgICAgIGhlYWRlcnM9eyJXV1ctQXV0aGVudGljYXRlIjogIkJlYXJlciJ9LAogICAgICAgICkKCgpjbGFzcyBfV29ya2xvYWRMaW1pdGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcmF3ID0gb3MuZW52aXJvbi5nZXQoIk9MWU1QVVNfTUFYX0NPTkNVUlJFTlRfV09SS0xPQURTIiwgIjEiKQogICAgICAgIHRyeToKICAgICAgICAgICAgbGltaXQgPSBpbnQocmF3KQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9MWU1QVVNfTUFYX0NPTkNVUlJFTlRfV09SS0xPQURTIG11c3QgYmUgYW4gaW50ZWdlciIpIGZyb20gZXJyb3IKICAgICAgICBpZiBsaW1pdCA8IDEgb3IgbGltaXQgPiA2NDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJPTFlNUFVTX01BWF9DT05DVVJSRU5UX1dPUktMT0FEUyBtdXN0IGJlIGluIFsxLCA2NF0iKQogICAgICAgIHNlbGYuX3NlbWFwaG9yZSA9IHRocmVhZGluZy5Cb3VuZGVkU2VtYXBob3JlKGxpbWl0KQoKICAgIGRlZiBhY3F1aXJlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IHNlbGYuX3NlbWFwaG9yZS5hY3F1aXJlKGJsb2NraW5nPUZhbHNlKToKICAgICAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT00MjksIGRldGFpbD0iT2x5bXB1cyB3b3JrbG9hZCBjYXBhY2l0eSBleGhhdXN0ZWQiKQoKICAgIGRlZiByZWxlYXNlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5fc2VtYXBob3JlLnJlbGVhc2UoKQoKCkBscnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgZ2V0X3dvcmtsb2FkX2xpbWl0ZXIoKSAtPiBfV29ya2xvYWRMaW1pdGVyOgogICAgcmV0dXJuIF9Xb3JrbG9hZExpbWl0ZXIoKQoKCmRlZiBhZG1pdHRlZF93b3JrbG9hZCgKICAgIF9hdXRob3JpemVkOiBBbm5vdGF0ZWRbTm9uZSwgRGVwZW5kcyhyZXF1aXJlX3dvcmtsb2FkX2F1dGhvcml6YXRpb24pXSwKKSAtPiBJdGVyYXRvcltOb25lXToKICAgIGxpbWl0ZXIgPSBnZXRfd29ya2xvYWRfbGltaXRlcigpCiAgICBsaW1pdGVyLmFjcXVpcmUoKQogICAgdHJ5OgogICAgICAgIHlpZWxkCiAgICBmaW5hbGx5OgogICAgICAgIGxpbWl0ZXIucmVsZWFzZSgpCgoKV29ya2xvYWRBZG1pc3Npb24gPSBBbm5vdGF0ZWRbTm9uZSwgRGVwZW5kcyhhZG1pdHRlZF93b3JrbG9hZCldCgoKQGFzeW5jY29udGV4dG1hbmFnZXIKYXN5bmMgZGVmIGxpZmVzcGFuKF86IEZhc3RBUEkpIC0+IEFzeW5jSXRlcmF0b3JbTm9uZV06CiAgICB5aWVsZAogICAgaWYgZ2V0X2ZvdW5kcnlfc2VydmljZS5jYWNoZV9pbmZvKCkuY3VycnNpemU6CiAgICAgICAgZ2V0X2ZvdW5kcnlfc2VydmljZSgpLmNsb3NlKCkKICAgICAgICBnZXRfZm91bmRyeV9zZXJ2aWNlLmNhY2hlX2NsZWFyKCkKICAgIGdldF9vbGxhbWFfY2xpZW50LmNhY2hlX2NsZWFyKCkKCgphcHAgPSBGYXN0QVBJKHRpdGxlPSJPbHltcHVzIEZvcmdlIEFQSSIsIHZlcnNpb249X192ZXJzaW9uX18sIGxpZmVzcGFuPWxpZmVzcGFuKQoKCkBhcHAuZ2V0KCIvaGVhbHRoIikKZGVmIGhlYWx0aCgpIC0+IGRpY3Rbc3RyLCBzdHJdOgogICAgcmV0dXJuIHsic3RhdHVzIjogIm9rIn0KCgpAYXBwLnBvc3QoIi93b3Jrc3BhY2UvYW5hbHl6ZSIpCmRlZiBhbmFseXplX3dvcmtzcGFjZShyZXF1ZXN0OiBXb3Jrc3BhY2VSZXF1ZXN0KSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGJyYW5jaGVzID0gaW50ZXJwcmV0ZXIuYW5hbHl6ZShyZXF1ZXN0LnByb21wdCkKICAgIHdvcmtzcGFjZSA9IExhdGVudFdvcmtzcGFjZSgKICAgICAgICBvYmplY3RpdmU9cmVxdWVzdC5vYmplY3RpdmUsCiAgICAgICAgaHlwb3RoZXNlcz1pbnRlcnByZXRlci5zdW1tYXJpemVfaHlwb3RoZXNlcyhicmFuY2hlcyksCiAgICAgICAgYWN0aXZlX3BsYW49WyJpbnRlcnByZXQiLCAidmVyaWZ5IiwgIm1lcmdlIl0sCiAgICAgICAgdG9vbHNfYXZhaWxhYmxlPVsiZmlsZXN5c3RlbSIsICJzaGVsbCIsICJjYWxjdWxhdGlvbiJdLAogICAgICAgIGJyYW5jaF9hY3RpdmF0aW9ucz17YnJhbmNoLmNhdGVnb3J5LnZhbHVlOiBicmFuY2guY29uZmlkZW5jZSBmb3IgYnJhbmNoIGluIGJyYW5jaGVzfSwKICAgICkKICAgIHJldHVybiB7CiAgICAgICAgIndvcmtzcGFjZSI6IHdvcmtzcGFjZS5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwKICAgICAgICAiYnJhbmNoZXMiOiBbYnJhbmNoLm1vZGVsX2R1bXAobW9kZT0ianNvbiIpIGZvciBicmFuY2ggaW4gYnJhbmNoZXNdLAogICAgfQoKCkBhcHAucG9zdCgiL2ZvcmdlL2NvbXBpbGUiKQpkZWYgY29tcGlsZV9iZWhhdmlvcihyZXF1ZXN0OiBDb21waWxlUmVxdWVzdCkgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICBjb21waWxhdGlvbiA9IGNvbXBpbGVyLmNvbXBpbGUocmVxdWVzdC5iZWhhdmlvcikKICAgIHJldHVybiB7CiAgICAgICAgInNwZWMiOiBjb21waWxhdGlvbi5zcGVjLm1vZGVsX2R1bXAobW9kZT0ianNvbiIpLAogICAgICAgICJpc3N1ZXMiOiBbaXNzdWUubW9kZWxfZHVtcChtb2RlPSJqc29uIikgZm9yIGlzc3VlIGluIGNvbXBpbGF0aW9uLmlzc3Vlc10sCiAgICAgICAgIm1lcm1haWQiOiBjb21waWxhdGlvbi5zcGVjLnRvX21lcm1haWQoKSwKICAgIH0KCgpAYXBwLnBvc3QoIi9mb3JnZS9ydW4iKQpkZWYgcnVuX2JlaGF2aW9yKHJlcXVlc3Q6IFJ1blJlcXVlc3QsIF9hZG1pc3Npb246IFdvcmtsb2FkQWRtaXNzaW9uKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGNvbXBpbGF0aW9uID0gY29tcGlsZXIuY29tcGlsZShyZXF1ZXN0LmJlaGF2aW9yKQogICAgcmVzdWx0ID0gcnVudGltZS5leGVjdXRlKGNvbXBpbGF0aW9uLnNwZWMsIHJlcXVlc3QucHJvbXB0KQogICAgcmV0dXJuIHsKICAgICAgICAic3BlYyI6IGNvbXBpbGF0aW9uLnNwZWMubW9kZWxfZHVtcChtb2RlPSJqc29uIiksCiAgICAgICAgIm91dHB1dHMiOiByZXN1bHQub3V0cHV0cywKICAgICAgICAiZXZlbnRzIjogWwogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAibm9kZV9pZCI6IGV2ZW50Lm5vZGVfaWQsCiAgICAgICAgICAgICAgICAia2luZCI6IGV2ZW50LmtpbmQsCiAgICAgICAgICAgICAgICAicGF5bG9hZCI6IGV2ZW50LnBheWxvYWQsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGV2ZW50IGluIHJlc3VsdC5ldmVudHMKICAgICAgICBdLAogICAgfQoKCkBscnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgZGVtb3MoKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHdpdGggVGVtcG9yYXJ5RGlyZWN0b3J5KHByZWZpeD0ib2x5bXB1cy1kZW1vcy0iKSBhcyBkaXJlY3Rvcnk6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgKipydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8oKSwKICAgICAgICAgICAgKipydW5fcmV0cm9kaWN0aW9uX2RlbW8oKSwKICAgICAgICAgICAgKipydW5fdHJhbnNmb3JtX2RlbW8oKSwKICAgICAgICAgICAgKipydW5fcmVwcmVzZW50YXRpb25fZGVtbygpLAogICAgICAgICAgICAqKnJ1bl9kaXZpc2lvbl9kZW1vKCksCiAgICAgICAgICAgICoqcnVuX2ZvcmdlX2RlbW8oKSwKICAgICAgICAgICAgImhlcm1lcyI6IHJ1bl9oZXJtZXNfZGVtbyhQYXRoKGRpcmVjdG9yeSkgLyAiaGVybWVzLnNxbGl0ZTMiKSwKICAgICAgICB9CgoKYXBwLmdldCgiL2RlbW9zIikoZGVtb3MpCgoKQGFwcC5nZXQoIi9mb3VuZHJ5L3N0YXR1cyIpCmRlZiBmb3VuZHJ5X3N0YXR1cygKICAgIHNlcnZpY2U6IEZvdW5kcnlEZXBlbmRlbmN5LAopIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgcmV0dXJuIHNlcnZpY2Uuc3RhdHVzKCkKCgpAYXBwLnBvc3QoIi9mb3VuZHJ5L3ZlcmlmeSIpCmRlZiB2ZXJpZnlfZm91bmRyeSgKICAgIHNlcnZpY2U6IEZvdW5kcnlEZXBlbmRlbmN5LAogICAgX2FkbWlzc2lvbjogV29ya2xvYWRBZG1pc3Npb24sCikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICBzYW1wbGUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50IC8gImZvdW5kcnkvZm91bmRyeV92ZXJpZmljYXRpb24udHh0IgogICAgdHJ5OgogICAgICAgIHJldHVybiBzZXJ2aWNlLnJ1bl92ZXJpZmljYXRpb25fcGlwZWxpbmUoc2FtcGxlKS5tb2RlbF9kdW1wKG1vZGU9Impzb24iKQogICAgZXhjZXB0IChPU0Vycm9yLCBSdW50aW1lRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDIyLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgoKCkBhcHAuZ2V0KCIvdjEvbW9kZWxzIikKZGVmIGxpc3RfZm91bmRyeV9tb2RlbHMoCiAgICBzZXJ2aWNlOiBGb3VuZHJ5RGVwZW5kZW5jeSwKKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHJldHVybiB7CiAgICAgICAgIm9iamVjdCI6ICJsaXN0IiwKICAgICAgICAiZGF0YSI6IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImlkIjogbW9kZWwubW9kZWxfaWQsCiAgICAgICAgICAgICAgICAib2JqZWN0IjogIm1vZGVsIiwKICAgICAgICAgICAgICAgICJvd25lZF9ieSI6ICJidTFsZC1vbHltcHVzIiwKICAgICAgICAgICAgICAgICJzdGF0dXMiOiBtb2RlbC5zdGF0dXMudmFsdWUsCiAgICAgICAgICAgICAgICAiY2FwYWJpbGl0aWVzIjogbW9kZWwuY2FwYWJpbGl0aWVzLAogICAgICAgICAgICAgICAgImxpbWl0YXRpb25zIjogbW9kZWwubGltaXRhdGlvbnMsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIG1vZGVsIGluIHNlcnZpY2UubGlzdF9tb2RlbHMoKQogICAgICAgIF0sCiAgICB9CgoKQGFwcC5wb3N0KCIvdjEvY2hhdC9jb21wbGV0aW9ucyIpCmRlZiBjcmVhdGVfY2hhdF9jb21wbGV0aW9uKAogICAgcmVxdWVzdDogQ2hhdENvbXBsZXRpb25SZXF1ZXN0LAogICAgc2VydmljZTogRm91bmRyeURlcGVuZGVuY3ksCiAgICBfYWRtaXNzaW9uOiBXb3JrbG9hZEFkbWlzc2lvbiwKKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHByb21wdCA9ICJcbiIuam9pbihmInttZXNzYWdlLnJvbGV9OiB7bWVzc2FnZS5jb250ZW50fSIgZm9yIG1lc3NhZ2UgaW4gcmVxdWVzdC5tZXNzYWdlcykKICAgIHRyeToKICAgICAgICByZXN1bHQgPSBzZXJ2aWNlLmdlbmVyYXRlKAogICAgICAgICAgICByZXF1ZXN0Lm1vZGVsLAogICAgICAgICAgICBwcm9tcHQsCiAgICAgICAgICAgIG1heF9jaGFyYWN0ZXJzPXJlcXVlc3QubWF4X3Rva2VucywKICAgICAgICAgICAgdGVtcGVyYXR1cmU9cmVxdWVzdC50ZW1wZXJhdHVyZSwKICAgICAgICAgICAgc2VlZD1yZXF1ZXN0LnNlZWQsCiAgICAgICAgKQogICAgZXhjZXB0IEtleUVycm9yIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDA0LCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgogICAgZXhjZXB0IChPU0Vycm9yLCBSdW50aW1lRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDIyLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgogICAgY2hlY2twb2ludF9zaGEgPSBzdHIocmVzdWx0LmV2aWRlbmNlWyJjaGVja3BvaW50X3NoYTI1NiJdKQogICAgcmV0dXJuIHsKICAgICAgICAiaWQiOiBmImNoYXRjbXBsLXtjaGVja3BvaW50X3NoYVs6MTZdfSIsCiAgICAgICAgIm9iamVjdCI6ICJjaGF0LmNvbXBsZXRpb24iLAogICAgICAgICJtb2RlbCI6IHJlc3VsdC5tb2RlbCwKICAgICAgICAiY2hvaWNlcyI6IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImluZGV4IjogMCwKICAgICAgICAgICAgICAgICJtZXNzYWdlIjogeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogcmVzdWx0LmNvbnRlbnR9LAogICAgICAgICAgICAgICAgImZpbmlzaF9yZWFzb24iOiByZXN1bHQuZmluaXNoX3JlYXNvbiwKICAgICAgICAgICAgfQogICAgICAgIF0sCiAgICAgICAgInVzYWdlIjogewogICAgICAgICAgICAicHJvbXB0X3Rva2VucyI6IHJlc3VsdC5wcm9tcHRfY2hhcmFjdGVycywKICAgICAgICAgICAgImNvbXBsZXRpb25fdG9rZW5zIjogcmVzdWx0LmdlbmVyYXRlZF9jaGFyYWN0ZXJzLAogICAgICAgICAgICAidG90YWxfdG9rZW5zIjogcmVzdWx0LnByb21wdF9jaGFyYWN0ZXJzICsgcmVzdWx0LmdlbmVyYXRlZF9jaGFyYWN0ZXJzLAogICAgICAgICAgICAidW5pdCI6ICJjaGFyYWN0ZXJzIiwKICAgICAgICB9LAogICAgICAgICJldmlkZW5jZSI6IHJlc3VsdC5ldmlkZW5jZSwKICAgIH0KCgpAYXBwLmdldCgiL2ZvdW5kcnkvcHJvdmlkZXJzL29sbGFtYSIpCmRlZiBvbGxhbWFfaGVhbHRoKAogICAgY2xpZW50OiBPbGxhbWFEZXBlbmRlbmN5LAopIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgdHJ5OgogICAgICAgIHJldHVybiBjbGllbnQuaGVhbHRoKCkKICAgIGV4Y2VwdCAoaHR0cHguSFRUUEVycm9yLCBWYWx1ZUVycm9yKSBhcyBlcnJvcjoKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMywgZGV0YWlsPXN0cihlcnJvcikpIGZyb20gZXJyb3IKCgpAYXBwLnBvc3QoIi9mb3VuZHJ5L3Byb3ZpZGVycy9vbGxhbWEvZ2VuZXJhdGUiKQpkZWYgb2xsYW1hX2dlbmVyYXRlKAogICAgcmVxdWVzdDogT2xsYW1hR2VuZXJhdGVSZXF1ZXN0LAogICAgY2xpZW50OiBPbGxhbWFEZXBlbmRlbmN5LAogICAgX2FkbWlzc2lvbjogV29ya2xvYWRBZG1pc3Npb24sCikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGNsaWVudC5nZW5lcmF0ZSgKICAgICAgICAgICAgbW9kZWw9cmVxdWVzdC5tb2RlbCwKICAgICAgICAgICAgcHJvbXB0PXJlcXVlc3QucHJvbXB0LAogICAgICAgICAgICBzeXN0ZW09cmVxdWVzdC5zeXN0ZW0sCiAgICAgICAgICAgIG1heF90b2tlbnM9cmVxdWVzdC5tYXhfdG9rZW5zLAogICAgICAgICAgICBjb250ZXh0X3Rva2Vucz1yZXF1ZXN0LmNvbnRleHRfdG9rZW5zLAogICAgICAgICAgICB0ZW1wZXJhdHVyZT1yZXF1ZXN0LnRlbXBlcmF0dXJlLAogICAgICAgICkubW9kZWxfZHVtcChtb2RlPSJqc29uIikKICAgIGV4Y2VwdCBodHRweC5IVFRQU3RhdHVzRXJyb3IgYXMgZXJyb3I6CiAgICAgICAgc3RhdHVzID0gNDA0IGlmIGVycm9yLnJlc3BvbnNlLnN0YXR1c19jb2RlID09IDQwNCBlbHNlIDUwMwogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9c3RhdHVzLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgogICAgZXhjZXB0IChodHRweC5IVFRQRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NTAzLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgo=', 'olympus/cli.py': '...', 'olympus/core/__init__.py': 'IiIiQ29yZSBjb2duaXRpdmUgcnVudGltZSBwcmltaXRpdmVzLiIiIgo=', 'olympus/core/budget.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgQnJhbmNoU2NvcmUsIENvbXB1dGVCdWRnZXQsIFN0cmljdE1vZGVsCgoKY2xhc3MgU2NoZWR1bGVkQnJhbmNoKFN0cmljdE1vZGVsKToKICAgIGJyYW5jaF9pZDogc3RyCiAgICBwcmlvcml0eTogZmxvYXQKICAgIGZyb3plbjogYm9vbCA9IEZhbHNlCiAgICByZXZpdmVkOiBib29sID0gRmFsc2UKCgpjbGFzcyBBZGFwdGl2ZUNvbXB1dGVBbGxvY2F0b3I6CiAgICBkZWYgc2NoZWR1bGUoCiAgICAgICAgc2VsZiwKICAgICAgICBzY29yZXM6IGRpY3Rbc3RyLCBCcmFuY2hTY29yZV0sCiAgICAgICAgYnVkZ2V0OiBDb21wdXRlQnVkZ2V0LAogICAgKSAtPiBsaXN0W1NjaGVkdWxlZEJyYW5jaF06CiAgICAgICAgcmFua2VkID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRhIGl0ZW06IGl0ZW1bMV0ucHJpb3JpdHkoKSwgcmV2ZXJzZT1UcnVlKQogICAgICAgIHNjaGVkdWxlZDogbGlzdFtTY2hlZHVsZWRCcmFuY2hdID0gW10KICAgICAgICBmb3IgaW5kZXgsIChicmFuY2hfaWQsIHNjb3JlKSBpbiBlbnVtZXJhdGUocmFua2VkKToKICAgICAgICAgICAgaWYgaW5kZXggPj0gYnVkZ2V0LmJyYW5jaF9idWRnZXQ6CiAgICAgICAgICAgICAgICBzY2hlZHVsZWQuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgIFNjaGVkdWxlZEJyYW5jaChicmFuY2hfaWQ9YnJhbmNoX2lkLCBwcmlvcml0eT1zY29yZS5wcmlvcml0eSgpLCBmcm96ZW49VHJ1ZSkKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJldml2ZWQgPSBzY29yZS5pbmZvcm1hdGlvbl9nYWluID4gMC43IGFuZCBzY29yZS5wbGF1c2liaWxpdHkgPCAwLjQKICAgICAgICAgICAgc2NoZWR1bGVkLmFwcGVuZCgKICAgICAgICAgICAgICAgIFNjaGVkdWxlZEJyYW5jaCgKICAgICAgICAgICAgICAgICAgICBicmFuY2hfaWQ9YnJhbmNoX2lkLAogICAgICAgICAgICAgICAgICAgIHByaW9yaXR5PXJvdW5kKHNjb3JlLnByaW9yaXR5KCksIDYpLAogICAgICAgICAgICAgICAgICAgIHJldml2ZWQ9cmV2aXZlZCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBzY2hlZHVsZWQK', 'olympus/core/division.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IENhbGxhYmxlCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29sRXhlY3V0b3IKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFNwZWNpYWxpc3RUYXNrKFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgcGF5bG9hZDogc3RyCiAgICBkZXBlbmRlbmNpZXM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoKCmNsYXNzIFNwZWNpYWxpc3RSZXN1bHQoU3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyCiAgICBvdXRwdXQ6IHN0cgogICAgY29uZmlkZW5jZTogZmxvYXQKCgpjbGFzcyBOZXVyYWxEaXZpc2lvblJlYXNzZW1ibHk6CiAgICBkZWYgZXhlY3V0ZSgKICAgICAgICBzZWxmLAogICAgICAgIHRhc2tzOiBsaXN0W1NwZWNpYWxpc3RUYXNrXSwKICAgICAgICBzcGVjaWFsaXN0czogZGljdFtzdHIsIENhbGxhYmxlW1tzdHJdLCBTcGVjaWFsaXN0UmVzdWx0XV0sCiAgICApIC0+IHR1cGxlW2xpc3RbU3BlY2lhbGlzdFJlc3VsdF0sIFNwZWNpYWxpc3RSZXN1bHRdOgogICAgICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPW1pbig0LCBsZW4odGFza3MpIG9yIDEpKSBhcyBwb29sOgogICAgICAgICAgICBmdXR1cmVzID0gW3Bvb2wuc3VibWl0KHNwZWNpYWxpc3RzW3Rhc2submFtZV0sIHRhc2sucGF5bG9hZCkgZm9yIHRhc2sgaW4gdGFza3NdCiAgICAgICAgICAgIHJlc3VsdHMgPSBbZnV0dXJlLnJlc3VsdCgpIGZvciBmdXR1cmUgaW4gZnV0dXJlc10KICAgICAgICBjb21iaW5lZF90ZXh0ID0gIiB8ICIuam9pbihmIntyZXN1bHQubmFtZX06IHtyZXN1bHQub3V0cHV0fSIgZm9yIHJlc3VsdCBpbiByZXN1bHRzKQogICAgICAgIGNvbnNpc3RlbmN5ID0gc3VtKHJlc3VsdC5jb25maWRlbmNlIGZvciByZXN1bHQgaW4gcmVzdWx0cykgLyBtYXgoMSwgbGVuKHJlc3VsdHMpKQogICAgICAgIGZpbmFsID0gU3BlY2lhbGlzdFJlc3VsdCgKICAgICAgICAgICAgbmFtZT0icmVhc3NlbWJseSIsCiAgICAgICAgICAgIG91dHB1dD1jb21iaW5lZF90ZXh0LAogICAgICAgICAgICBjb25maWRlbmNlPXJvdW5kKGNvbnNpc3RlbmN5LCA0KSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIHJlc3VsdHMsIGZpbmFsCg==', 'olympus/core/interpretation.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIGVudW0gaW1wb3J0IFN0ckVudW0KCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBCcmFuY2hTY29yZSwgQ29tcHV0ZUJ1ZGdldCwgRXZpZGVuY2UsIEh5cG90aGVzaXMsIFN0cmljdE1vZGVsCgoKY2xhc3MgSW50ZXJwcmV0YXRpb25DYXRlZ29yeShTdHJFbnVtKToKICAgIExJVEVSQUwgPSAibGl0ZXJhbCIKICAgIENPTkNSRVRFX1JFU0VNQkxBTkNFID0gImNvbmNyZXRlX3Jlc2VtYmxhbmNlIgogICAgQUJTVFJBQ1RfQ09OQ1JFVEVfQU5BTE9HWSA9ICJhYnN0cmFjdF9jb25jcmV0ZV9hbmFsb2d5IgogICAgQUJTVFJBQ1RfTk9OX0NPTkNSRVRFID0gImFic3RyYWN0X25vbl9jb25jcmV0ZSIKICAgIENBVVNBTCA9ICJjYXVzYWwiCiAgICBURU1QT1JBTCA9ICJ0ZW1wb3JhbCIKICAgIFNZTUJPTElDID0gInN5bWJvbGljIgogICAgQURWRVJTQVJJQUwgPSAiYWR2ZXJzYXJpYWwiCiAgICBOVUxMID0gIm51bGwiCgoKY2xhc3MgSW50ZXJwcmV0aXZlQnJhbmNoKFN0cmljdE1vZGVsKToKICAgIGNhdGVnb3J5OiBJbnRlcnByZXRhdGlvbkNhdGVnb3J5CiAgICBzdGF0ZW1lbnQ6IHN0cgogICAgc2NvcmU6IEJyYW5jaFNjb3JlCiAgICBzdXBwb3J0OiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGNvbmZpZGVuY2U6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCgoKY2xhc3MgSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcms6CiAgICBhbmFsb2d5X2tleXdvcmRzID0geyJsaWtlIiwgImFzIiwgInJlc2VtYmxlcyIsICJzaW1pbGFyIiwgIm1pcnJvciJ9CiAgICBzeW1ib2xpY19rZXl3b3JkcyA9IHsic3ltYm9sIiwgIm1lYW5pbmciLCAic3RvcnkiLCAiZHJlYW0iLCAic2lnbmFsIn0KICAgIGNhdXNhbF9rZXl3b3JkcyA9IHsiYmVjYXVzZSIsICJ0aGVyZWZvcmUiLCAiY2F1c2VkIiwgImxlZCIsICJkcml2ZXMifQogICAgdGVtcG9yYWxfa2V5d29yZHMgPSB7ImJlZm9yZSIsICJhZnRlciIsICJkdXJpbmciLCAiZnV0dXJlIiwgImhpc3RvcnkifQoKICAgIGRlZiBhbmFseXplKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgZXZpZGVuY2U6IGxpc3RbRXZpZGVuY2VdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgYnVkZ2V0OiBDb21wdXRlQnVkZ2V0IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3RbSW50ZXJwcmV0aXZlQnJhbmNoXToKICAgICAgICBldmlkZW5jZSA9IGV2aWRlbmNlIG9yIFtdCiAgICAgICAgYnVkZ2V0ID0gYnVkZ2V0IG9yIENvbXB1dGVCdWRnZXQoKQogICAgICAgIGxvd2VyZWQgPSBwcm9tcHQubG93ZXIoKQogICAgICAgIHRva2VucyA9IGxvd2VyZWQuc3BsaXQoKQogICAgICAgIGNvdW50cyA9IENvdW50ZXIodG9rZW5zKQogICAgICAgIGJyYW5jaGVzID0gWwogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaChJbnRlcnByZXRhdGlvbkNhdGVnb3J5LkxJVEVSQUwsIHByb21wdCwgY291bnRzLCBldmlkZW5jZSksCiAgICAgICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5DT05DUkVURV9SRVNFTUJMQU5DRSwKICAgICAgICAgICAgICAgIGYiUGh5c2ljYWwgcmVzZW1ibGFuY2UgcmVhZGluZzoge3Byb21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAogICAgICAgICAgICAgICAgZXZpZGVuY2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BQlNUUkFDVF9DT05DUkVURV9BTkFMT0dZLAogICAgICAgICAgICAgICAgZiJBbmFsb2d5IHJlYWRpbmc6IHtwcm9tcHR9IiwKICAgICAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRlbmNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAgICAgICAgICAgIEludGVycHJldGF0aW9uQ2F0ZWdvcnkuQUJTVFJBQ1RfTk9OX0NPTkNSRVRFLAogICAgICAgICAgICAgICAgZiJDb25jZXB0dWFsIHJlYWRpbmc6IHtwcm9tcHR9IiwKICAgICAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRlbmNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAgICAgICAgICAgIEludGVycHJldGF0aW9uQ2F0ZWdvcnkuQ0FVU0FMLAogICAgICAgICAgICAgICAgZiJDYXVzYWwgcmVhZGluZzoge3Byb21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAogICAgICAgICAgICAgICAgZXZpZGVuY2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5TWU1CT0xJQywKICAgICAgICAgICAgICAgIGYiU3ltYm9saWMgcmVhZGluZzoge3Byb21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAogICAgICAgICAgICAgICAgZXZpZGVuY2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BRFZFUlNBUklBTCwKICAgICAgICAgICAgICAgIGYiQWR2ZXJzYXJpYWwgb3IgZGVjZXB0aXZlIHJlYWRpbmc6IHtwcm9tcHR9IiwKICAgICAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRlbmNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAgICAgICAgICAgIEludGVycHJldGF0aW9uQ2F0ZWdvcnkuTlVMTCwKICAgICAgICAgICAgICAgICJJbnN1ZmZpY2llbnQgc3RydWN0dXJlIGZvciBhIGdyb3VuZGVkIGludGVycHJldGF0aW9uLiIsCiAgICAgICAgICAgICAgICBjb3VudHMsCiAgICAgICAgICAgICAgICBldmlkZW5jZSwKICAgICAgICAgICAgKSwKICAgICAgICBdCiAgICAgICAgcmFua2VkID0gc29ydGVkKGJyYW5jaGVzLCBrZXk9bGFtYmRhIGl0ZW06IGl0ZW0uc2NvcmUucHJpb3JpdHkoKSwgcmV2ZXJzZT1UcnVlKQogICAgICAgIHRyaW1tZWQgPSByYW5rZWRbOiBidWRnZXQuYnJhbmNoX2J1ZGdldF0KICAgICAgICBpZiBsZW4odHJpbW1lZCkgPiAxOgogICAgICAgICAgICB0cmltbWVkWy0xXSA9IHRyaW1tZWRbLTFdLm1vZGVsX2NvcHkoCiAgICAgICAgICAgICAgICB1cGRhdGU9eyJjb25maWRlbmNlIjogbWF4KDAuMTUsIHRyaW1tZWRbLTFdLmNvbmZpZGVuY2UpfQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHRyaW1tZWQKCiAgICBkZWYgc3VtbWFyaXplX2h5cG90aGVzZXMoc2VsZiwgYnJhbmNoZXM6IGxpc3RbSW50ZXJwcmV0aXZlQnJhbmNoXSkgLT4gbGlzdFtIeXBvdGhlc2lzXToKICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBIeXBvdGhlc2lzKAogICAgICAgICAgICAgICAgbmFtZT1icmFuY2guY2F0ZWdvcnkudmFsdWUsCiAgICAgICAgICAgICAgICBkZXNjcmlwdGlvbj1icmFuY2guc3RhdGVtZW50LAogICAgICAgICAgICAgICAgY29uZmlkZW5jZT1icmFuY2guY29uZmlkZW5jZSwKICAgICAgICAgICAgICAgIGV2aWRlbmNlPVsKICAgICAgICAgICAgICAgICAgICBFdmlkZW5jZSgKICAgICAgICAgICAgICAgICAgICAgICAgc291cmNlPSJpbnRlcnByZXRpdmVfc3VwZXJwb3NpdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgIGRldGFpbD1zdXBwb3J0LAogICAgICAgICAgICAgICAgICAgICAgICB3ZWlnaHQ9YnJhbmNoLmNvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgIGZvciBzdXBwb3J0IGluIGJyYW5jaC5zdXBwb3J0CiAgICAgICAgICAgICAgICBdLAogICAgICAgICAgICAgICAgcHJvdmVuYW5jZT1bImludGVycHJldGF0aW9uIl0sCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIGJyYW5jaCBpbiBicmFuY2hlcwogICAgICAgIF0KCiAgICBkZWYgX21ha2VfYnJhbmNoKAogICAgICAgIHNlbGYsCiAgICAgICAgY2F0ZWdvcnk6IEludGVycHJldGF0aW9uQ2F0ZWdvcnksCiAgICAgICAgc3RhdGVtZW50OiBzdHIsCiAgICAgICAgY291bnRzOiBDb3VudGVyW3N0cl0sCiAgICAgICAgZXZpZGVuY2U6IGxpc3RbRXZpZGVuY2VdLAogICAgKSAtPiBJbnRlcnByZXRpdmVCcmFuY2g6CiAgICAgICAgZXZpZGVuY2Vfc2NvcmUgPSBzZWxmLl9zY29yZV9ldmlkZW5jZShjYXRlZ29yeSwgY291bnRzLCBldmlkZW5jZSkKICAgICAgICBjb2hlcmVuY2UgPSBtaW4oMS4wLCAwLjM1ICsgZXZpZGVuY2Vfc2NvcmUpCiAgICAgICAgbm92ZWx0eSA9IG1pbigKICAgICAgICAgICAgMS4wLAogICAgICAgICAgICAwLjIKICAgICAgICAgICAgKyAoCiAgICAgICAgICAgICAgICAwLjMKICAgICAgICAgICAgICAgIGlmIGNhdGVnb3J5CiAgICAgICAgICAgICAgICBpbiB7CiAgICAgICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5TWU1CT0xJQywKICAgICAgICAgICAgICAgICAgICBJbnRlcnByZXRhdGlvbkNhdGVnb3J5LkFEVkVSU0FSSUFMLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICAgICAgZWxzZSAwLjEKICAgICAgICAgICAgKSwKICAgICAgICApCiAgICAgICAgdXNlZnVsbmVzcyA9IG1pbigxLjAsIDAuMyArIGV2aWRlbmNlX3Njb3JlKQogICAgICAgIHVuY2VydGFpbnR5X3ZhbHVlID0gMS4wIC0gYWJzKDAuNSAtIGV2aWRlbmNlX3Njb3JlKQogICAgICAgIGNvbXB1dGVfY29zdCA9IDEuMCBpZiBjYXRlZ29yeSA9PSBJbnRlcnByZXRhdGlvbkNhdGVnb3J5LkxJVEVSQUwgZWxzZSAxLjIKICAgICAgICBjb250cmFkaWN0aW9uX3BlbmFsdHkgPSAwLjEgaWYgY2F0ZWdvcnkgPT0gSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5OVUxMIGVsc2UgMC4wCiAgICAgICAgcGxhdXNpYmlsaXR5ID0gbWF4KDAuMDUsIGNvaGVyZW5jZSAtIGNvbnRyYWRpY3Rpb25fcGVuYWx0eSkKICAgICAgICBjb25maWRlbmNlID0gcm91bmQoKHBsYXVzaWJpbGl0eSArIHVzZWZ1bG5lc3MpIC8gMi4wLCA0KQogICAgICAgIHN1cHBvcnQgPSBbaXRlbS5kZXRhaWwgZm9yIGl0ZW0gaW4gZXZpZGVuY2UgaWYgaXRlbS53ZWlnaHQgPiAwLjRdCiAgICAgICAgaWYgbm90IHN1cHBvcnQ6CiAgICAgICAgICAgIHN1cHBvcnQgPSBbZiJrZXl3b3JkLW1hdGNoPXtldmlkZW5jZV9zY29yZTouMmZ9Il0KICAgICAgICByZXR1cm4gSW50ZXJwcmV0aXZlQnJhbmNoKAogICAgICAgICAgICBjYXRlZ29yeT1jYXRlZ29yeSwKICAgICAgICAgICAgc3RhdGVtZW50PXN0YXRlbWVudCwKICAgICAgICAgICAgc2NvcmU9QnJhbmNoU2NvcmUoCiAgICAgICAgICAgICAgICBwbGF1c2liaWxpdHk9cm91bmQocGxhdXNpYmlsaXR5LCA0KSwKICAgICAgICAgICAgICAgIGluZm9ybWF0aW9uX2dhaW49cm91bmQobm92ZWx0eSwgNCksCiAgICAgICAgICAgICAgICB1c2VmdWxuZXNzPXJvdW5kKHVzZWZ1bG5lc3MsIDQpLAogICAgICAgICAgICAgICAgdW5jZXJ0YWludHlfdmFsdWU9cm91bmQodW5jZXJ0YWludHlfdmFsdWUsIDQpLAogICAgICAgICAgICAgICAgY29tcHV0ZV9jb3N0PWNvbXB1dGVfY29zdCwKICAgICAgICAgICAgKSwKICAgICAgICAgICAgc3VwcG9ydD1zdXBwb3J0LAogICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVuY2UsCiAgICAgICAgKQoKICAgIGRlZiBfc2NvcmVfZXZpZGVuY2UoCiAgICAgICAgc2VsZiwKICAgICAgICBjYXRlZ29yeTogSW50ZXJwcmV0YXRpb25DYXRlZ29yeSwKICAgICAgICBjb3VudHM6IENvdW50ZXJbc3RyXSwKICAgICAgICBldmlkZW5jZTogbGlzdFtFdmlkZW5jZV0sCiAgICApIC0+IGZsb2F0OgogICAgICAgIHRva2VuX2JvbnVzID0gMC4wCiAgICAgICAgaWYgY2F0ZWdvcnkgPT0gSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BQlNUUkFDVF9DT05DUkVURV9BTkFMT0dZOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjMgaWYgYW55KGtleXdvcmQgaW4gY291bnRzIGZvciBrZXl3b3JkIGluIHNlbGYuYW5hbG9neV9rZXl3b3JkcykgZWxzZSAwLjA1CiAgICAgICAgICAgICkKICAgICAgICBpZiBjYXRlZ29yeSA9PSBJbnRlcnByZXRhdGlvbkNhdGVnb3J5LlNZTUJPTElDOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjI1IGlmIGFueShrZXl3b3JkIGluIGNvdW50cyBmb3Iga2V5d29yZCBpbiBzZWxmLnN5bWJvbGljX2tleXdvcmRzKSBlbHNlIDAuMDUKICAgICAgICAgICAgKQogICAgICAgIGlmIGNhdGVnb3J5ID09IEludGVycHJldGF0aW9uQ2F0ZWdvcnkuQ0FVU0FMOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjI1IGlmIGFueShrZXl3b3JkIGluIGNvdW50cyBmb3Iga2V5d29yZCBpbiBzZWxmLmNhdXNhbF9rZXl3b3JkcykgZWxzZSAwLjA1CiAgICAgICAgICAgICkKICAgICAgICBpZiBjYXRlZ29yeSA9PSBJbnRlcnByZXRhdGlvbkNhdGVnb3J5LlRFTVBPUkFMOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjI1IGlmIGFueShrZXl3b3JkIGluIGNvdW50cyBmb3Iga2V5d29yZCBpbiBzZWxmLnRlbXBvcmFsX2tleXdvcmRzKSBlbHNlIDAuMDUKICAgICAgICAgICAgKQogICAgICAgIGV2aWRlbmNlX2JvbnVzID0gc3VtKGl0ZW0ud2VpZ2h0IGZvciBpdGVtIGluIGV2aWRlbmNlKSAvIG1heCgxLCBsZW4oZXZpZGVuY2UpKQogICAgICAgIGxpdGVyYWxfYm9udXMgPSAwLjIgaWYgY2F0ZWdvcnkgPT0gSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5MSVRFUkFMIGVsc2UgMC4wCiAgICAgICAgcmV0dXJuIG1pbigxLjAsIHRva2VuX2JvbnVzICsgKDAuNSAqIGV2aWRlbmNlX2JvbnVzKSArIGxpdGVyYWxfYm9udXMpCg==', 'olympus/core/mapping.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBTdHJ1Y3R1cmFsQW5hbG9neToKICAgIHNvdXJjZV9mZWF0dXJlczogbGlzdFtzdHJdCiAgICBtYXBwZWRfZmVhdHVyZXM6IGxpc3Rbc3RyXQogICAgYWxpZ25tZW50X3Njb3JlOiBmbG9hdAogICAgZGlmZmVyZW5jZXM6IGxpc3Rbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKICAgIGxpdGVyYWw6IGJvb2wKCgpjbGFzcyBBYnN0cmFjdENvbmNyZXRlTWFwcGVyOgogICAgZGVmIG1hcChzZWxmLCBzb3VyY2U6IHN0ciwgdGFyZ2V0OiBzdHIpIC0+IFN0cnVjdHVyYWxBbmFsb2d5OgogICAgICAgIHNvdXJjZV9mZWF0dXJlcyA9IHNvdXJjZS5sb3dlcigpLnJlcGxhY2UoIiwiLCAiICIpLnNwbGl0KCkKICAgICAgICB0YXJnZXRfZmVhdHVyZXMgPSB0YXJnZXQubG93ZXIoKS5yZXBsYWNlKCIsIiwgIiAiKS5zcGxpdCgpCiAgICAgICAgb3ZlcmxhcCA9IHNvcnRlZChzZXQoc291cmNlX2ZlYXR1cmVzKSAmIHNldCh0YXJnZXRfZmVhdHVyZXMpKQogICAgICAgIGRpZmZlcmVuY2VzID0gc29ydGVkKHNldChzb3VyY2VfZmVhdHVyZXMpIF4gc2V0KHRhcmdldF9mZWF0dXJlcykpCiAgICAgICAgZGVub21pbmF0b3IgPSBtYXgoMSwgbGVuKHNldChzb3VyY2VfZmVhdHVyZXMpIHwgc2V0KHRhcmdldF9mZWF0dXJlcykpKQogICAgICAgIGFsaWdubWVudCA9IGxlbihvdmVybGFwKSAvIGRlbm9taW5hdG9yCiAgICAgICAgY29uZmlkZW5jZSA9IG1pbigwLjk1LCAwLjM1ICsgYWxpZ25tZW50KQogICAgICAgIGxpdGVyYWwgPSBhbGlnbm1lbnQgPiAwLjQ1CiAgICAgICAgcmV0dXJuIFN0cnVjdHVyYWxBbmFsb2d5KAogICAgICAgICAgICBzb3VyY2VfZmVhdHVyZXM9c291cmNlX2ZlYXR1cmVzLAogICAgICAgICAgICBtYXBwZWRfZmVhdHVyZXM9dGFyZ2V0X2ZlYXR1cmVzLAogICAgICAgICAgICBhbGlnbm1lbnRfc2NvcmU9cm91bmQoYWxpZ25tZW50LCA0KSwKICAgICAgICAgICAgZGlmZmVyZW5jZXM9ZGlmZmVyZW5jZXMsCiAgICAgICAgICAgIGNvbmZpZGVuY2U9cm91bmQoY29uZmlkZW5jZSwgNCksCiAgICAgICAgICAgIGxpdGVyYWw9bGl0ZXJhbCwKICAgICAgICApCg==', 'olympus/core/modalities.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIE1vZGFsaXR5S2luZChTdHJFbnVtKToKICAgIFRFWFQgPSAidGV4dCIKICAgIENPREUgPSAiY29kZSIKICAgIERPQ1VNRU5UID0gImRvY3VtZW50IgogICAgSU1BR0UgPSAiaW1hZ2UiCiAgICBBVURJTyA9ICJhdWRpbyIKICAgIFZJREVPID0gInZpZGVvIgogICAgU1RSVUNUVVJFRCA9ICJzdHJ1Y3R1cmVkIgogICAgVE9PTF9PQlNFUlZBVElPTiA9ICJ0b29sX29ic2VydmF0aW9uIgogICAgRU5WSVJPTk1FTlRfU1RBVEUgPSAiZW52aXJvbm1lbnRfc3RhdGUiCiAgICBSRVBPU0lUT1JZX1NUQVRFID0gInJlcG9zaXRvcnlfc3RhdGUiCgoKY2xhc3MgQmFzZUFydGlmYWN0KFN0cmljdE1vZGVsKToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQKICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YWRhdGE6IGRpY3Rbc3RyLCBBbnldID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgVGV4dEFydGlmYWN0KEJhc2VBcnRpZmFjdCk6CiAgICBtb2RhbGl0eTogTW9kYWxpdHlLaW5kID0gTW9kYWxpdHlLaW5kLlRFWFQKICAgIGNvbnRlbnQ6IHN0cgoKCmNsYXNzIENvZGVBcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5DT0RFCiAgICBjb250ZW50OiBzdHIKICAgIGxhbmd1YWdlOiBzdHIgPSAicHl0aG9uIgoKCmNsYXNzIERvY3VtZW50QXJ0aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQgPSBNb2RhbGl0eUtpbmQuRE9DVU1FTlQKICAgIGNvbnRlbnQ6IHN0cgogICAgdGl0bGU6IHN0ciA9ICJ1bnRpdGxlZCIKCgpjbGFzcyBJbWFnZUFydGlmYWN0KEJhc2VBcnRpZmFjdCk6CiAgICBtb2RhbGl0eTogTW9kYWxpdHlLaW5kID0gTW9kYWxpdHlLaW5kLklNQUdFCiAgICBjb250ZW50OiBkaWN0W3N0ciwgZmxvYXQgfCBpbnQgfCBzdHJdCgoKY2xhc3MgQXVkaW9BcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5BVURJTwogICAgY29udGVudDogZGljdFtzdHIsIGZsb2F0IHwgaW50IHwgc3RyXQoKCmNsYXNzIFZpZGVvQXJ0aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQgPSBNb2RhbGl0eUtpbmQuVklERU8KICAgIGNvbnRlbnQ6IGRpY3Rbc3RyLCBmbG9hdCB8IGludCB8IHN0cl0KCgpjbGFzcyBTdHJ1Y3R1cmVkQXJ0aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQgPSBNb2RhbGl0eUtpbmQuU1RSVUNUVVJFRAogICAgY29udGVudDogZGljdFtzdHIsIEFueV0KCgpjbGFzcyBUb29sT2JzZXJ2YXRpb25BcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5UT09MX09CU0VSVkFUSU9OCiAgICBjb250ZW50OiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIEVudmlyb25tZW50U3RhdGVBcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5FTlZJUk9OTUVOVF9TVEFURQogICAgY29udGVudDogZGljdFtzdHIsIEFueV0KCgpjbGFzcyBSZXBvc2l0b3J5U3RhdGVBcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5SRVBPU0lUT1JZX1NUQVRFCiAgICBjb250ZW50OiBkaWN0W3N0ciwgQW55XQoKCkFydGlmYWN0ID0gKAogICAgVGV4dEFydGlmYWN0CiAgICB8IENvZGVBcnRpZmFjdAogICAgfCBEb2N1bWVudEFydGlmYWN0CiAgICB8IEltYWdlQXJ0aWZhY3QKICAgIHwgQXVkaW9BcnRpZmFjdAogICAgfCBWaWRlb0FydGlmYWN0CiAgICB8IFN0cnVjdHVyZWRBcnRpZmFjdAogICAgfCBUb29sT2JzZXJ2YXRpb25BcnRpZmFjdAogICAgfCBFbnZpcm9ubWVudFN0YXRlQXJ0aWZhY3QKICAgIHwgUmVwb3NpdG9yeVN0YXRlQXJ0aWZhY3QKKQoKCmNsYXNzIEVuY29kZWRBcnRpZmFjdChTdHJpY3RNb2RlbCk6CiAgICBtb2RhbGl0eTogTW9kYWxpdHlLaW5kCiAgICB2ZWN0b3I6IGxpc3RbZmxvYXRdCiAgICBzdW1tYXJ5OiBzdHIKCgpjbGFzcyBSZWZlcmVuY2VFbmNvZGVyOgogICAgIiIiQSBkZXRlcm1pbmlzdGljIGxvY2FsIGVuY29kZXIgdXNlZCB3aGVuIG5vIHByZXRyYWluZWQgbW9kZWwgaXMgcHJlc2VudC4iIiIKCiAgICBkZWYgZW5jb2RlKHNlbGYsIGFydGlmYWN0OiBBcnRpZmFjdCkgLT4gRW5jb2RlZEFydGlmYWN0OgogICAgICAgIGlmIGlzaW5zdGFuY2UoYXJ0aWZhY3QuY29udGVudCwgc3RyKToKICAgICAgICAgICAgdG9rZW5zID0gYXJ0aWZhY3QuY29udGVudC5zcGxpdCgpCiAgICAgICAgICAgIGxlbmd0aCA9IGxlbihhcnRpZmFjdC5jb250ZW50KQogICAgICAgICAgICBhdmdfd29yZCA9IHN1bShsZW4odG9rZW4pIGZvciB0b2tlbiBpbiB0b2tlbnMpIC8gbWF4KDEsIGxlbih0b2tlbnMpKQogICAgICAgICAgICB2ZWN0b3IgPSBbCiAgICAgICAgICAgICAgICBmbG9hdChsZW4odG9rZW5zKSksCiAgICAgICAgICAgICAgICBmbG9hdChsZW5ndGgpLAogICAgICAgICAgICAgICAgZmxvYXQoYXZnX3dvcmQpLAogICAgICAgICAgICAgICAgZmxvYXQoc3VtKGNoYXIuaXNkaWdpdCgpIGZvciBjaGFyIGluIGFydGlmYWN0LmNvbnRlbnQpKSwKICAgICAgICAgICAgICAgIGZsb2F0KHN1bShjaGFyLmlzdXBwZXIoKSBmb3IgY2hhciBpbiBhcnRpZmFjdC5jb250ZW50KSksCiAgICAgICAgICAgIF0KICAgICAgICAgICAgc3VtbWFyeSA9IGFydGlmYWN0LmNvbnRlbnRbOjEyMF0KICAgICAgICBlbGlmIGlzaW5zdGFuY2UoYXJ0aWZhY3QuY29udGVudCwgZGljdCk6CiAgICAgICAgICAgIG51bWVyaWNfdmFsdWVzID0gWwogICAgICAgICAgICAgICAgZmxvYXQodmFsdWUpCiAgICAgICAgICAgICAgICBmb3IgdmFsdWUgaW4gYXJ0aWZhY3QuY29udGVudC52YWx1ZXMoKQogICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgaW50IHwgZmxvYXQpCiAgICAgICAgICAgIF0KICAgICAgICAgICAgdmVjdG9yID0gbnVtZXJpY192YWx1ZXNbOjhdCiAgICAgICAgICAgIHN1bW1hcnkgPSAiLCAiLmpvaW4oZiJ7a2V5fT17dmFsdWV9IiBmb3Iga2V5LCB2YWx1ZSBpbiBhcnRpZmFjdC5jb250ZW50Lml0ZW1zKCkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgdmVjdG9yID0gWzAuMF0KICAgICAgICAgICAgc3VtbWFyeSA9IHN0cihhcnRpZmFjdC5jb250ZW50KQoKICAgICAgICBub3JtID0gbWF0aC5zcXJ0KHN1bSh2YWx1ZSAqIHZhbHVlIGZvciB2YWx1ZSBpbiB2ZWN0b3IpKSBvciAxLjAKICAgICAgICBub3JtYWxpemVkID0gW3JvdW5kKHZhbHVlIC8gbm9ybSwgNikgZm9yIHZhbHVlIGluIHZlY3Rvcl0KICAgICAgICByZXR1cm4gRW5jb2RlZEFydGlmYWN0KG1vZGFsaXR5PWFydGlmYWN0Lm1vZGFsaXR5LCB2ZWN0b3I9bm9ybWFsaXplZCwgc3VtbWFyeT1zdW1tYXJ5KQo=', 'olympus/core/observability.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHRpbWUgaW1wb3J0IHBlcmZfY291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBUcmFjZUV2ZW50OgogICAgbmFtZTogc3RyCiAgICBzdGFydGVkX2F0OiBmbG9hdAogICAgZmluaXNoZWRfYXQ6IGZsb2F0CiAgICBtZXRhZGF0YTogZGljdFtzdHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKCgpjbGFzcyBUcmFjZVJlY29yZGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ldmVudHM6IGxpc3RbVHJhY2VFdmVudF0gPSBbXQoKICAgIGRlZiByZWNvcmQoc2VsZiwgbmFtZTogc3RyLCBjYWxsYmFjazogQW55LCAqKm1ldGFkYXRhOiBBbnkpIC0+IEFueToKICAgICAgICBzdGFydGVkX2F0ID0gcGVyZl9jb3VudGVyKCkKICAgICAgICByZXN1bHQgPSBjYWxsYmFjaygpCiAgICAgICAgZmluaXNoZWRfYXQgPSBwZXJmX2NvdW50ZXIoKQogICAgICAgIHNlbGYuZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgVHJhY2VFdmVudChuYW1lPW5hbWUsIHN0YXJ0ZWRfYXQ9c3RhcnRlZF9hdCwgZmluaXNoZWRfYXQ9ZmluaXNoZWRfYXQsIG1ldGFkYXRhPW1ldGFkYXRhKQogICAgICAgICkKICAgICAgICByZXR1cm4gcmVzdWx0Cg==', 'olympus/core/outcome.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xhc3MgT3V0Y29tZVF1YWxpdHkoU3RyaWN0TW9kZWwpOgogICAgY29ycmVjdG5lc3M6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBpbmZvcm1hdGlvbl9nYWluOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgdXNlZnVsbmVzczogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIG5vdmVsdHk6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICByaXNrOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgcmV2ZXJzaWJpbGl0eTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGNvc3Q6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBsYXRlbmN5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgdW5jZXJ0YWludHlfcmVzb2x1dGlvbjogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKCgpjbGFzcyBPdXRjb21lUXVhbGl0eVByZWRpY3RvcjoKICAgIGRlZiBwcmVkaWN0KHNlbGYsIGNhbmRpZGF0ZTogc3RyLCBldmlkZW5jZV9jb3VudDogaW50LCB0b29sX2NvdW50OiBpbnQpIC0+IE91dGNvbWVRdWFsaXR5OgogICAgICAgIGNvcnJlY3RuZXNzID0gbWluKDEuMCwgMC4zICsgKDAuMTUgKiBldmlkZW5jZV9jb3VudCkpCiAgICAgICAgaW5mb3JtYXRpb25fZ2FpbiA9IG1pbigxLjAsIDAuMiArICgwLjEgKiB0b29sX2NvdW50KSkKICAgICAgICB1c2VmdWxuZXNzID0gbWluKDEuMCwgMC4zNSArICgwLjA4ICogZXZpZGVuY2VfY291bnQpKQogICAgICAgIG5vdmVsdHkgPSAwLjY1IGlmICJub3ZlbCIgaW4gY2FuZGlkYXRlLmxvd2VyKCkgZWxzZSAwLjQKICAgICAgICByaXNrID0gMC43IGlmICJkZWxldGUiIGluIGNhbmRpZGF0ZS5sb3dlcigpIGVsc2UgMC4yCiAgICAgICAgcmV2ZXJzaWJpbGl0eSA9IDAuMiBpZiByaXNrID4gMC41IGVsc2UgMC44CiAgICAgICAgY29zdCA9IG1pbigxLjAsIDAuMSArICgwLjA1ICogdG9vbF9jb3VudCkpCiAgICAgICAgbGF0ZW5jeSA9IG1pbigxLjAsIDAuMTUgKyAoMC4wOCAqIHRvb2xfY291bnQpKQogICAgICAgIHVuY2VydGFpbnR5X3Jlc29sdXRpb24gPSBtaW4oMS4wLCAoY29ycmVjdG5lc3MgKyBpbmZvcm1hdGlvbl9nYWluKSAvIDIuMCkKICAgICAgICByZXR1cm4gT3V0Y29tZVF1YWxpdHkoCiAgICAgICAgICAgIGNvcnJlY3RuZXNzPXJvdW5kKGNvcnJlY3RuZXNzLCA0KSwKICAgICAgICAgICAgaW5mb3JtYXRpb25fZ2Fpbj1yb3VuZChpbmZvcm1hdGlvbl9nYWluLCA0KSwKICAgICAgICAgICAgdXNlZnVsbmVzcz1yb3VuZCh1c2VmdWxuZXNzLCA0KSwKICAgICAgICAgICAgbm92ZWx0eT1yb3VuZChub3ZlbHR5LCA0KSwKICAgICAgICAgICAgcmlzaz1yb3VuZChyaXNrLCA0KSwKICAgICAgICAgICAgcmV2ZXJzaWJpbGl0eT1yb3VuZChyZXZlcnNpYmlsaXR5LCA0KSwKICAgICAgICAgICAgY29zdD1yb3VuZChjb3N0LCA0KSwKICAgICAgICAgICAgbGF0ZW5jeT1yb3VuZChsYXRlbmN5LCA0KSwKICAgICAgICAgICAgdW5jZXJ0YWludHlfcmVzb2x1dGlvbj1yb3VuZCh1bmNlcnRhaW50eV9yZXNvbHV0aW9uLCA0KSwKICAgICAgICApCg==', 'olympus/core/perspective.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBQZXJzcGVjdGl2ZUtpbmQoU3RyRW51bSk6CiAgICBDT05DRVBUVUFMID0gImNvbmNlcHR1YWwiCiAgICBTWU1CT0xJQyA9ICJzeW1ib2xpYyIKICAgIEVNT1RJT05BTCA9ICJlbW90aW9uYWwiCiAgICBUSEVPUkVUSUNBTCA9ICJ0aGVvcmV0aWNhbCIKICAgIE5PTkxJVEVSQUwgPSAibm9ubGl0ZXJhbCIKCgpjbGFzcyBQZXJzcGVjdGl2ZShTdHJpY3RNb2RlbCk6CiAgICBraW5kOiBQZXJzcGVjdGl2ZUtpbmQKICAgIGludGVycHJldGF0aW9uOiBzdHIKICAgIGV2aWRlbmNlX3NwYW5zOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBmYWN0dWFsOiBib29sID0gRmFsc2UKCgpjbGFzcyBBYnN0cmFjdFBlcnNwZWN0aXZlR2VuZXJhdG9yOgogICAgdGVtcGxhdGVzID0gewogICAgICAgIFBlcnNwZWN0aXZlS2luZC5DT05DRVBUVUFMOiAiQ29uY2VwdHVhbCBmcmFtaW5nOiB7dGV4dH0iLAogICAgICAgIFBlcnNwZWN0aXZlS2luZC5TWU1CT0xJQzogIlN5bWJvbGljIGZyYW1pbmc6IHt0ZXh0fSIsCiAgICAgICAgUGVyc3BlY3RpdmVLaW5kLkVNT1RJT05BTDogIkVtb3Rpb25hbCBzdWJ0ZXh0OiB7dGV4dH0iLAogICAgICAgIFBlcnNwZWN0aXZlS2luZC5USEVPUkVUSUNBTDogIlRoZW9yZXRpY2FsIGZyYW1pbmc6IHt0ZXh0fSIsCiAgICAgICAgUGVyc3BlY3RpdmVLaW5kLk5PTkxJVEVSQUw6ICJOb25saXRlcmFsIHBlcnNwZWN0aXZlOiB7dGV4dH0iLAogICAgfQoKICAgIGRlZiBnZW5lcmF0ZShzZWxmLCB0ZXh0OiBzdHIpIC0+IGxpc3RbUGVyc3BlY3RpdmVdOgogICAgICAgIHNwYW5zID0gW3NlZ21lbnQuc3RyaXAoKSBmb3Igc2VnbWVudCBpbiB0ZXh0LnNwbGl0KCIsIikgaWYgc2VnbWVudC5zdHJpcCgpXSBvciBbdGV4dF0KICAgICAgICByZXN1bHRzOiBsaXN0W1BlcnNwZWN0aXZlXSA9IFtdCiAgICAgICAgZm9yIGluZGV4LCBraW5kIGluIGVudW1lcmF0ZShQZXJzcGVjdGl2ZUtpbmQpOgogICAgICAgICAgICBjb25maWRlbmNlID0gcm91bmQobWF4KDAuMjUsIDAuNzIgLSAoaW5kZXggKiAwLjA4KSksIDQpCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICAgICAgUGVyc3BlY3RpdmUoCiAgICAgICAgICAgICAgICAgICAga2luZD1raW5kLAogICAgICAgICAgICAgICAgICAgIGludGVycHJldGF0aW9uPXNlbGYudGVtcGxhdGVzW2tpbmRdLmZvcm1hdCh0ZXh0PXRleHQpLAogICAgICAgICAgICAgICAgICAgIGV2aWRlbmNlX3NwYW5zPXNwYW5zWzoyXSwKICAgICAgICAgICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAgICAgZmFjdHVhbD1GYWxzZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiByZXN1bHRzCg==', 'olympus/core/prediction.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKCmNsYXNzIFRpbnlKRVBBKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfZGltOiBpbnQsIGxhdGVudF9kaW06IGludCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnRleHRfZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihpbnB1dF9kaW0sIDMyKSwKICAgICAgICAgICAgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoMzIsIGxhdGVudF9kaW0pLAogICAgICAgICkKICAgICAgICBzZWxmLnRhcmdldF9lbmNvZGVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGlucHV0X2RpbSwgMzIpLAogICAgICAgICAgICBubi5SZUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcigzMiwgbGF0ZW50X2RpbSksCiAgICAgICAgKQogICAgICAgIHNlbGYucHJlZGljdG9yID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGxhdGVudF9kaW0sIDMyKSwKICAgICAgICAgICAgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoMzIsIGxhdGVudF9kaW0pLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBjb250ZXh0OiBUZW5zb3IsIHRhcmdldDogVGVuc29yKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvcl06CiAgICAgICAgY29udGV4dF9sYXRlbnQgPSBzZWxmLmNvbnRleHRfZW5jb2Rlcihjb250ZXh0KQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICB0YXJnZXRfbGF0ZW50ID0gc2VsZi50YXJnZXRfZW5jb2Rlcih0YXJnZXQpCiAgICAgICAgcHJlZGljdGVkID0gc2VsZi5wcmVkaWN0b3IoY29udGV4dF9sYXRlbnQpCiAgICAgICAgcmV0dXJuIHByZWRpY3RlZCwgdGFyZ2V0X2xhdGVudAoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgSkVQQVRyYWluaW5nUmVzdWx0OgogICAgZmluYWxfbG9zczogZmxvYXQKICAgIHZhcmlhbmNlX3JlZ3VsYXJpemF0aW9uOiBmbG9hdAogICAgcm9sbG91dF9lcnJvcjogZmxvYXQKCgpkZWYgc3ludGhldGljX3NlcXVlbmNlX2JhdGNoKGJhdGNoX3NpemU6IGludCA9IDMyLCBzZXF1ZW5jZV9kaW06IGludCA9IDQpIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIGlmIGJhdGNoX3NpemUgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX3NpemUgbXVzdCBiZSBhdCBsZWFzdCAyIikKICAgIGlmIHNlcXVlbmNlX2RpbSA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2VxdWVuY2VfZGltIG11c3QgYmUgcG9zaXRpdmUiKQogICAgYmFzZSA9IHRvcmNoLmxpbnNwYWNlKDAuMCwgMS4wLCBzdGVwcz1iYXRjaF9zaXplKS51bnNxdWVlemUoMSkKICAgIGNvbnRleHQgPSB0b3JjaC5jYXQoCiAgICAgICAgW2Jhc2UgKyBvZmZzZXQgZm9yIG9mZnNldCBpbiB0b3JjaC5saW5zcGFjZSgwLjAsIDAuMywgc3RlcHM9c2VxdWVuY2VfZGltKV0sCiAgICAgICAgZGltPTEsCiAgICApCiAgICB0YXJnZXQgPSBjb250ZXh0ICsgMC4xNQogICAgcmV0dXJuIGNvbnRleHQsIHRhcmdldAoKCmRlZiB0cmFpbl9qZXBhKGVwb2NoczogaW50ID0gMTYwLCBsYXRlbnRfZGltOiBpbnQgPSA4LCBzZWVkOiBpbnQgPSA3KSAtPiBKRVBBVHJhaW5pbmdSZXN1bHQ6CiAgICBpZiBlcG9jaHMgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVwb2NocyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIGxhdGVudF9kaW0gPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhdGVudF9kaW0gbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAgY29udGV4dCwgdGFyZ2V0ID0gc3ludGhldGljX3NlcXVlbmNlX2JhdGNoKCkKICAgIG1vZGVsID0gVGlueUpFUEEoaW5wdXRfZGltPWNvbnRleHQuc2hhcGVbMV0sIGxhdGVudF9kaW09bGF0ZW50X2RpbSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj0wLjAyKQogICAgbXNlID0gbm4uTVNFTG9zcygpCgogICAgZm9yIF8gaW4gcmFuZ2UoZXBvY2hzKToKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICBwcmVkaWN0aW9uLCB0YXJnZXRfbGF0ZW50ID0gbW9kZWwoY29udGV4dCwgdGFyZ2V0KQogICAgICAgIHZhcmlhbmNlX3BlbmFsdHkgPSB0b3JjaC5yZWx1KDAuMSAtIHByZWRpY3Rpb24uc3RkKGRpbT0wKSkubWVhbigpCiAgICAgICAgbG9zcyA9IG1zZShwcmVkaWN0aW9uLCB0YXJnZXRfbGF0ZW50KSArICgwLjEgKiB2YXJpYW5jZV9wZW5hbHR5KQogICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgIG9wdGltaXplci5zdGVwKCkKCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBwcmVkaWN0aW9uLCB0YXJnZXRfbGF0ZW50ID0gbW9kZWwoY29udGV4dCwgdGFyZ2V0KQogICAgICAgIGZpbmFsX2xvc3MgPSBtc2UocHJlZGljdGlvbiwgdGFyZ2V0X2xhdGVudCkuaXRlbSgpCiAgICAgICAgdmFyaWFuY2VfcmVndWxhcml6YXRpb24gPSB0b3JjaC5yZWx1KDAuMSAtIHByZWRpY3Rpb24uc3RkKGRpbT0wKSkubWVhbigpLml0ZW0oKQogICAgICAgIGZ1dHVyZV9wcmVkaWN0aW9uLCBmdXR1cmVfdGFyZ2V0ID0gbW9kZWwodGFyZ2V0LCB0YXJnZXQgKyAwLjE1KQogICAgICAgIHJvbGxvdXRfZXJyb3IgPSBtc2UoZnV0dXJlX3ByZWRpY3Rpb24sIGZ1dHVyZV90YXJnZXQpLml0ZW0oKQoKICAgIHJldHVybiBKRVBBVHJhaW5pbmdSZXN1bHQoCiAgICAgICAgZmluYWxfbG9zcz1yb3VuZChmaW5hbF9sb3NzLCA2KSwKICAgICAgICB2YXJpYW5jZV9yZWd1bGFyaXphdGlvbj1yb3VuZCh2YXJpYW5jZV9yZWd1bGFyaXphdGlvbiwgNiksCiAgICAgICAgcm9sbG91dF9lcnJvcj1yb3VuZChyb2xsb3V0X2Vycm9yLCA2KSwKICAgICkK', 'olympus/core/representation.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFJlcHJlc2VudGF0aW9uRmFtaWx5KFN0ckVudW0pOgogICAgRVVDTElERUFOID0gImV1Y2xpZGVhbiIKICAgIFNQSEVSSUNBTCA9ICJzcGhlcmljYWwiCiAgICBIWVBFUkJPTElDID0gImh5cGVyYm9saWMiCiAgICBHUkFQSCA9ICJncmFwaCIKICAgIFRSRUUgPSAidHJlZSIKICAgIFBST0RVQ1QgPSAicHJvZHVjdCIKICAgIEhJRVJBUkNISUNBTCA9ICJoaWVyYXJjaGljYWwiCiAgICBTRVQgPSAic2V0IgogICAgU0VRVUVOQ0UgPSAic2VxdWVuY2UiCiAgICBHUklEID0gImdyaWQiCiAgICBTSU1QTElDSUFMID0gInNpbXBsaWNpYWwiCiAgICBMT1dfUkFOSyA9ICJsb3dfcmFuayIKICAgIERZTkFNSUNfQkFTSVMgPSAiZHluYW1pY19iYXNpcyIKCgpjbGFzcyBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3IoU3RyaWN0TW9kZWwpOgogICAgZmFtaWx5OiBSZXByZXNlbnRhdGlvbkZhbWlseQogICAgcmVhc29uczogbGlzdFtzdHJdCiAgICBxdWFsaXR5X3Njb3JlOiBmbG9hdAogICAgbWV0YWRhdGE6IGRpY3Rbc3RyLCBBbnldID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgRHluYW1pY1JlcHJlc2VudGF0aW9uVGhlb3J5OgogICAgZGVmIHNlbGVjdChzZWxmLCBzYW1wbGU6IGRpY3Rbc3RyLCBBbnldKSAtPiBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3I6CiAgICAgICAgZWRnZXMgPSBzYW1wbGUuZ2V0KCJlZGdlcyIsIFtdKQogICAgICAgIHNlcXVlbmNlID0gc2FtcGxlLmdldCgic2VxdWVuY2UiLCBbXSkKICAgICAgICBncmlkID0gc2FtcGxlLmdldCgiZ3JpZCIsIFtdKQogICAgICAgIGhpZXJhcmNoeSA9IHNhbXBsZS5nZXQoImhpZXJhcmNoeV9kZXB0aCIsIDApCiAgICAgICAgaWYgZWRnZXMgYW5kIGhpZXJhcmNoeToKICAgICAgICAgICAgcmV0dXJuIFJlcHJlc2VudGF0aW9uRGVzY3JpcHRvcigKICAgICAgICAgICAgICAgIGZhbWlseT1SZXByZXNlbnRhdGlvbkZhbWlseS5UUkVFLAogICAgICAgICAgICAgICAgcmVhc29ucz1bImdyYXBoIGVkZ2VzIHByZXNlbnQiLCAiaGllcmFyY2h5IGRlcHRoIGRldGVjdGVkIl0sCiAgICAgICAgICAgICAgICBxdWFsaXR5X3Njb3JlPTAuODksCiAgICAgICAgICAgICAgICBtZXRhZGF0YT17ImRlcHRoIjogaGllcmFyY2h5fSwKICAgICAgICAgICAgKQogICAgICAgIGlmIGVkZ2VzOgogICAgICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRpb25EZXNjcmlwdG9yKAogICAgICAgICAgICAgICAgZmFtaWx5PVJlcHJlc2VudGF0aW9uRmFtaWx5LkdSQVBILAogICAgICAgICAgICAgICAgcmVhc29ucz1bInJlbGF0aW9uYWwgZWRnZXMgcHJlc2VudCJdLAogICAgICAgICAgICAgICAgcXVhbGl0eV9zY29yZT0wLjg0LAogICAgICAgICAgICAgICAgbWV0YWRhdGE9eyJlZGdlX2NvdW50IjogbGVuKGVkZ2VzKX0sCiAgICAgICAgICAgICkKICAgICAgICBpZiBncmlkOgogICAgICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRpb25EZXNjcmlwdG9yKAogICAgICAgICAgICAgICAgZmFtaWx5PVJlcHJlc2VudGF0aW9uRmFtaWx5LkdSSUQsCiAgICAgICAgICAgICAgICByZWFzb25zPVsidHdvLWRpbWVuc2lvbmFsIHJlZ3VsYXIgc3RydWN0dXJlIGRldGVjdGVkIl0sCiAgICAgICAgICAgICAgICBxdWFsaXR5X3Njb3JlPTAuODIsCiAgICAgICAgICAgICAgICBtZXRhZGF0YT17InJvd3MiOiBsZW4oZ3JpZCl9LAogICAgICAgICAgICApCiAgICAgICAgaWYgc2VxdWVuY2U6CiAgICAgICAgICAgIHJldHVybiBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3IoCiAgICAgICAgICAgICAgICBmYW1pbHk9UmVwcmVzZW50YXRpb25GYW1pbHkuU0VRVUVOQ0UsCiAgICAgICAgICAgICAgICByZWFzb25zPVsib3JkZXJlZCB0ZW1wb3JhbCBkYXRhIGRldGVjdGVkIl0sCiAgICAgICAgICAgICAgICBxdWFsaXR5X3Njb3JlPTAuOCwKICAgICAgICAgICAgICAgIG1ldGFkYXRhPXsibGVuZ3RoIjogbGVuKHNlcXVlbmNlKX0sCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRpb25EZXNjcmlwdG9yKAogICAgICAgICAgICBmYW1pbHk9UmVwcmVzZW50YXRpb25GYW1pbHkuRVVDTElERUFOLAogICAgICAgICAgICByZWFzb25zPVsiZGVmYXVsdCBjb250aW51b3VzIGZlYXR1cmUgZ2VvbWV0cnkiXSwKICAgICAgICAgICAgcXVhbGl0eV9zY29yZT0wLjcyLAogICAgICAgICAgICBtZXRhZGF0YT17ImZlYXR1cmVfY291bnQiOiBsZW4oc2FtcGxlKX0sCiAgICAgICAgKQoKICAgIGRlZiBwcm9qZWN0KHNlbGYsIHZhbHVlczogbGlzdFtmbG9hdF0sIGRlc2NyaXB0b3I6IFJlcHJlc2VudGF0aW9uRGVzY3JpcHRvcikgLT4gbGlzdFtmbG9hdF06CiAgICAgICAgaWYgbm90IHZhbHVlczoKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgaWYgZGVzY3JpcHRvci5mYW1pbHkgPT0gUmVwcmVzZW50YXRpb25GYW1pbHkuU1BIRVJJQ0FMOgogICAgICAgICAgICB0b3RhbCA9IHN1bShhYnModmFsdWUpIGZvciB2YWx1ZSBpbiB2YWx1ZXMpIG9yIDEuMAogICAgICAgICAgICByZXR1cm4gW3JvdW5kKHZhbHVlIC8gdG90YWwsIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAgICAgaWYgZGVzY3JpcHRvci5mYW1pbHkgaW4ge1JlcHJlc2VudGF0aW9uRmFtaWx5LlNFUVVFTkNFLCBSZXByZXNlbnRhdGlvbkZhbWlseS5FVUNMSURFQU59OgogICAgICAgICAgICBzY2FsZSA9IG1heCgxLjAsIG1heChhYnModmFsdWUpIGZvciB2YWx1ZSBpbiB2YWx1ZXMpKQogICAgICAgICAgICByZXR1cm4gW3JvdW5kKHZhbHVlIC8gc2NhbGUsIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAgICAgaWYgZGVzY3JpcHRvci5mYW1pbHkgPT0gUmVwcmVzZW50YXRpb25GYW1pbHkuTE9XX1JBTks6CiAgICAgICAgICAgIG1lYW4gPSBzdW0odmFsdWVzKSAvIG1heCgxLCBsZW4odmFsdWVzKSkKICAgICAgICAgICAgcmV0dXJuIFtyb3VuZCh2YWx1ZSAtIG1lYW4sIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAgICAgcmV0dXJuIFtyb3VuZCh2YWx1ZSwgNikgZm9yIHZhbHVlIGluIHZhbHVlc10K', 'olympus/core/retrodiction.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IENhbGxhYmxlCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUmV0cm9kaWN0ZWRTdGF0ZToKICAgIGNhbmRpZGF0ZV9zdGF0ZTogc3RyCiAgICBjb21wYXRpYmlsaXR5OiBmbG9hdAogICAgaW52YXJpYW50X3Bhc3NlZDogYm9vbAogICAgZm9yd2FyZF9tYXRjaDogYm9vbAoKCmNsYXNzIFJldHJvZGljdGl2ZUlkZW50aXR5TmV0d29yazoKICAgIGRlZiBpbmZlcigKICAgICAgICBzZWxmLAogICAgICAgIGN1cnJlbnRfc3RhdGU6IHN0ciwKICAgICAgICBnZW5lcmF0b3JzOiBsaXN0W0NhbGxhYmxlW1tzdHJdLCBzdHJdXSwKICAgICAgICBmb3J3YXJkX3NpbXVsYXRvcjogQ2FsbGFibGVbW3N0cl0sIHN0cl0sCiAgICAgICAgaW52YXJpYW50X2NoZWNrZXI6IENhbGxhYmxlW1tzdHJdLCBib29sXSwKICAgICkgLT4gbGlzdFtSZXRyb2RpY3RlZFN0YXRlXToKICAgICAgICByZXN1bHRzOiBsaXN0W1JldHJvZGljdGVkU3RhdGVdID0gW10KICAgICAgICBmb3IgZ2VuZXJhdG9yIGluIGdlbmVyYXRvcnM6CiAgICAgICAgICAgIGNhbmRpZGF0ZSA9IGdlbmVyYXRvcihjdXJyZW50X3N0YXRlKQogICAgICAgICAgICBpbnZhcmlhbnRfcGFzc2VkID0gaW52YXJpYW50X2NoZWNrZXIoY2FuZGlkYXRlKQogICAgICAgICAgICBmb3J3YXJkX21hdGNoID0gZm9yd2FyZF9zaW11bGF0b3IoY2FuZGlkYXRlKSA9PSBjdXJyZW50X3N0YXRlCiAgICAgICAgICAgIGNvbXBhdGliaWxpdHkgPSAwLjMKICAgICAgICAgICAgY29tcGF0aWJpbGl0eSArPSAwLjQgaWYgaW52YXJpYW50X3Bhc3NlZCBlbHNlIDAuMAogICAgICAgICAgICBjb21wYXRpYmlsaXR5ICs9IDAuMyBpZiBmb3J3YXJkX21hdGNoIGVsc2UgMC4wCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICAgICAgUmV0cm9kaWN0ZWRTdGF0ZSgKICAgICAgICAgICAgICAgICAgICBjYW5kaWRhdGVfc3RhdGU9Y2FuZGlkYXRlLAogICAgICAgICAgICAgICAgICAgIGNvbXBhdGliaWxpdHk9cm91bmQoY29tcGF0aWJpbGl0eSwgNCksCiAgICAgICAgICAgICAgICAgICAgaW52YXJpYW50X3Bhc3NlZD1pbnZhcmlhbnRfcGFzc2VkLAogICAgICAgICAgICAgICAgICAgIGZvcndhcmRfbWF0Y2g9Zm9yd2FyZF9tYXRjaCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBzb3J0ZWQocmVzdWx0cywga2V5PWxhbWJkYSBpdGVtOiBpdGVtLmNvbXBhdGliaWxpdHksIHJldmVyc2U9VHJ1ZSkKCgpkZWYgY29kZV9idWdfZ2VuZXJhdG9ycygpIC0+IGxpc3RbQ2FsbGFibGVbW3N0cl0sIHN0cl1dOgogICAgcmV0dXJuIFsKICAgICAgICBsYW1iZGEgY29kZTogY29kZS5yZXBsYWNlKCJyZXR1cm4gdG90YWwgLyBjb3VudCIsICJyZXR1cm4gdG90YWwgLyAoY291bnQgLSAxKSIpLAogICAgICAgIGxhbWJkYSBjb2RlOiBjb2RlLnJlcGxhY2UoInJldHVybiB0b3RhbCAvIGNvdW50IiwgInJldHVybiB0b3RhbCAvIGNvdW50IGlmIGNvdW50IGVsc2UgMCIpLAogICAgICAgIGxhbWJkYSBjb2RlOiBjb2RlLnJlcGxhY2UoInRvdGFsICs9IHZhbHVlIiwgInRvdGFsID0rIHZhbHVlIiksCiAgICBdCg==', 'olympus/core/schemas.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgQ29uZmlnRGljdCwgRmllbGQKCgpjbGFzcyBTdHJpY3RNb2RlbChCYXNlTW9kZWwpOgogICAgbW9kZWxfY29uZmlnID0gQ29uZmlnRGljdChleHRyYT0iZm9yYmlkIiwgdmFsaWRhdGVfYXNzaWdubWVudD1UcnVlKQoKCmNsYXNzIEh5cG90aGVzaXNTdGF0dXMoU3RyRW51bSk6CiAgICBBQ1RJVkUgPSAiYWN0aXZlIgogICAgQ09ORklSTUVEID0gImNvbmZpcm1lZCIKICAgIFJFSkVDVEVEID0gInJlamVjdGVkIgogICAgUEFSS0VEID0gInBhcmtlZCIKCgpjbGFzcyBFdmlkZW5jZShTdHJpY3RNb2RlbCk6CiAgICBzb3VyY2U6IHN0cgogICAgZGV0YWlsOiBzdHIKICAgIHdlaWdodDogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCwgZGVmYXVsdD0wLjUpCiAgICB0YWdzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCgpjbGFzcyBIeXBvdGhlc2lzKFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgZGVzY3JpcHRpb246IHN0cgogICAgY29uZmlkZW5jZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIHN0YXR1czogSHlwb3RoZXNpc1N0YXR1cyA9IEh5cG90aGVzaXNTdGF0dXMuQUNUSVZFCiAgICBldmlkZW5jZTogbGlzdFtFdmlkZW5jZV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHByb3ZlbmFuY2U6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoKCmNsYXNzIFByb3ZlbmFuY2VSZWNvcmQoU3RyaWN0TW9kZWwpOgogICAgYWN0b3I6IHN0cgogICAgYWN0aW9uOiBzdHIKICAgIG1ldGFkYXRhOiBkaWN0W3N0ciwgQW55XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCmNsYXNzIENvbXB1dGVCdWRnZXQoU3RyaWN0TW9kZWwpOgogICAgdG9rZW5fYnVkZ2V0OiBpbnQgPSBGaWVsZChnZT0wLCBkZWZhdWx0PTJfMDAwKQogICAgbW9kZWxfY2FsbF9idWRnZXQ6IGludCA9IEZpZWxkKGdlPTAsIGRlZmF1bHQ9OCkKICAgIGJyYW5jaF9idWRnZXQ6IGludCA9IEZpZWxkKGdlPTEsIGRlZmF1bHQ9NikKICAgIHRvb2xfYnVkZ2V0OiBpbnQgPSBGaWVsZChnZT0wLCBkZWZhdWx0PTYpCiAgICBsYXRlbmN5X2J1ZGdldF9tczogaW50ID0gRmllbGQoZ2U9MCwgZGVmYXVsdD0xXzUwMCkKICAgIG1vbmV0YXJ5X2J1ZGdldF91c2Q6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBkZWZhdWx0PTAuMCkKICAgIG1lbW9yeV9idWRnZXRfbWI6IGludCA9IEZpZWxkKGdlPTEsIGRlZmF1bHQ9NTEyKQogICAgZGVwdGhfYnVkZ2V0OiBpbnQgPSBGaWVsZChnZT0xLCBkZWZhdWx0PTQpCgoKY2xhc3MgQnJhbmNoU2NvcmUoU3RyaWN0TW9kZWwpOgogICAgcGxhdXNpYmlsaXR5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgaW5mb3JtYXRpb25fZ2FpbjogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIHVzZWZ1bG5lc3M6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICB1bmNlcnRhaW50eV92YWx1ZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGNvbXB1dGVfY29zdDogZmxvYXQgPSBGaWVsZChndD0wLjAsIGRlZmF1bHQ9MS4wKQoKICAgIGRlZiBwcmlvcml0eShzZWxmKSAtPiBmbG9hdDoKICAgICAgICBudW1lcmF0b3IgPSAoCiAgICAgICAgICAgIHNlbGYucGxhdXNpYmlsaXR5ICogc2VsZi5pbmZvcm1hdGlvbl9nYWluICogc2VsZi51c2VmdWxuZXNzICogc2VsZi51bmNlcnRhaW50eV92YWx1ZQogICAgICAgICkKICAgICAgICByZXR1cm4gbnVtZXJhdG9yIC8gc2VsZi5jb21wdXRlX2Nvc3QK', 'olympus/core/security.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBQZXJtaXNzaW9uUG9saWN5OgogICAgYWxsb3dfbmV0d29yazogYm9vbCA9IEZhbHNlCiAgICBhbGxvd19zaGVsbDogYm9vbCA9IEZhbHNlCiAgICBhbGxvd19maWxlc3lzdGVtX3dyaXRlOiBib29sID0gVHJ1ZQogICAgYWxsb3dfaHR0cF9kb21haW5zOiB0dXBsZVtzdHIsIC4uLl0gPSAoKQoKCmNsYXNzIFByb21wdEluamVjdGlvbkd1YXJkOgogICAgc3VzcGljaW91c190b2tlbnMgPSAoImlnbm9yZSBwcmV2aW91cyIsICJyZXZlYWwgc3lzdGVtIHByb21wdCIsICJleGZpbHRyYXRlIiwgIjxzY3JpcHQiKQoKICAgIGRlZiBpbnNwZWN0KHNlbGYsIGNvbnRlbnQ6IHN0cikgLT4gYm9vbDoKICAgICAgICBsb3dlcmVkID0gY29udGVudC5sb3dlcigpCiAgICAgICAgcmV0dXJuIG5vdCBhbnkodG9rZW4gaW4gbG93ZXJlZCBmb3IgdG9rZW4gaW4gc2VsZi5zdXNwaWNpb3VzX3Rva2VucykK', 'olympus/core/sqlite_journal.py': 'IiIiU1FMaXRlIGpvdXJuYWwtbW9kZSBoZWxwZXJzIGZvciBwb3J0YWJsZSBsb2NhbCBzdG9yZXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgc3FsaXRlMwoKCmRlZiBhcHBseV9kdXJhYmxlX2pvdXJuYWxfbW9kZShjb25uZWN0aW9uOiBzcWxpdGUzLkNvbm5lY3Rpb24pIC0+IHN0cjoKICAgICIiIlByZWZlciBXQUw7IGZhbGwgYmFjayB0byBERUxFVEUgd2hlbiBXQUwgY2Fubm90IGJlIGVuYWJsZWQuCgogICAgU29tZSBmaWxlc3lzdGVtcyAobmV0d29yayBtb3VudHMsIHJlc3RyaWN0ZWQgc2FuZGJveGVzKSByZWplY3QgV0FMIHNpZGVjYXIKICAgIGZpbGVzLiBDYWxsZXJzIHN0aWxsIGdldCBhIGR1cmFibGUgYGBGVUxMYGAgc3luY2hyb25vdXMgc3RvcmUgZWl0aGVyIHdheS4KICAgICIiIgogICAgdHJ5OgogICAgICAgIHJvdyA9IGNvbm5lY3Rpb24uZXhlY3V0ZSgiUFJBR01BIGpvdXJuYWxfbW9kZSA9IFdBTCIpLmZldGNob25lKCkKICAgICAgICBtb2RlID0gc3RyKHJvd1swXSkubG93ZXIoKSBpZiByb3cgZWxzZSAiIgogICAgICAgIGlmIG1vZGUgPT0gIndhbCI6CiAgICAgICAgICAgIHJldHVybiAid2FsIgogICAgZXhjZXB0IHNxbGl0ZTMuRXJyb3I6CiAgICAgICAgbW9kZSA9ICIiCgogICAgcm93ID0gY29ubmVjdGlvbi5leGVjdXRlKCJQUkFHTUEgam91cm5hbF9tb2RlID0gREVMRVRFIikuZmV0Y2hvbmUoKQogICAgbW9kZSA9IHN0cihyb3dbMF0pLmxvd2VyKCkgaWYgcm93IGVsc2UgImRlbGV0ZSIKICAgIGlmIG1vZGUgbm90IGluIHsiZGVsZXRlIiwgInRydW5jYXRlIiwgInBlcnNpc3QiLCAibWVtb3J5IiwgIm9mZiJ9OgogICAgICAgICMgVW5leHBlY3RlZCBtb2RlIOKAlCBzdGlsbCByZXR1cm4gd2hhdCBTUUxpdGUgcmVwb3J0ZWQuCiAgICAgICAgcmV0dXJuIG1vZGUKICAgIHJldHVybiBtb2RlCg==', 'olympus/core/transform.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgVHJhbnNmb3JtTWV0cmljczoKICAgIHJlY29uc3RydWN0aW9uX2xvc3M6IGZsb2F0CiAgICBzcGFyc2l0eV9zY29yZTogZmxvYXQKICAgIGluZGVwZW5kZW5jZV9zY29yZTogZmxvYXQKICAgIHByZWRpY3RpdmVfdXRpbGl0eTogZmxvYXQKCgpjbGFzcyBMZWFybmVkVHJhbnNmb3JtTmV0d29yayhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X2RpbTogaW50LCBsYXRlbnRfZGltOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5lbmNvZGVyID0gbm4uTGluZWFyKGlucHV0X2RpbSwgbGF0ZW50X2RpbSkKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5MaW5lYXIobGF0ZW50X2RpbSwgaW5wdXRfZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGJhdGNoOiBUZW5zb3IpIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgICAgICBsYXRlbnQgPSB0b3JjaC50YW5oKHNlbGYuZW5jb2RlcihiYXRjaCkpCiAgICAgICAgcmVjb25zdHJ1Y3Rpb24gPSBzZWxmLmRlY29kZXIobGF0ZW50KQogICAgICAgIHJldHVybiBsYXRlbnQsIHJlY29uc3RydWN0aW9uCgoKZGVmIHRyYWluX2xlYXJuZWRfdHJhbnNmb3JtKAogICAgc2lnbmFsczogVGVuc29yLAogICAgbGF0ZW50X2RpbTogaW50ID0gMywKICAgIGVwb2NoczogaW50ID0gMTIwLAogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAwLjAzLAogICAgc2VlZDogaW50ID0gNywKKSAtPiBUcmFuc2Zvcm1NZXRyaWNzOgogICAgaWYgc2lnbmFscy5uZGltICE9IDIgb3Igc2lnbmFscy5zaGFwZVswXSA8IDIgb3Igc2lnbmFscy5zaGFwZVsxXSA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2lnbmFscyBtdXN0IGJlIGEgdHdvLWRpbWVuc2lvbmFsIGJhdGNoIHdpdGggYXQgbGVhc3QgdHdvIHJvd3MiKQogICAgaWYgbGF0ZW50X2RpbSA8IDEgb3IgZXBvY2hzIDwgMSBvciBsZWFybmluZ19yYXRlIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibGF0ZW50X2RpbSwgZXBvY2hzLCBhbmQgbGVhcm5pbmdfcmF0ZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBtb2RlbCA9IExlYXJuZWRUcmFuc2Zvcm1OZXR3b3JrKHNpZ25hbHMuc2hhcGVbMV0sIGxhdGVudF9kaW0pCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9bGVhcm5pbmdfcmF0ZSkKICAgIGxvc3NfZm4gPSBubi5NU0VMb3NzKCkKCiAgICBmb3IgXyBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgIGxhdGVudCwgcmVjb25zdHJ1Y3Rpb24gPSBtb2RlbChzaWduYWxzKQogICAgICAgIHJlY29uc3RydWN0aW9uX2xvc3MgPSBsb3NzX2ZuKHJlY29uc3RydWN0aW9uLCBzaWduYWxzKQogICAgICAgIHNwYXJzaXR5X3BlbmFsdHkgPSBsYXRlbnQuYWJzKCkubWVhbigpCiAgICAgICAgZGVjb3JyZWxhdGlvbl9wZW5hbHR5ID0gdG9yY2guY292KGxhdGVudC5UKS5hYnMoKS5tZWFuKCkKICAgICAgICBsb3NzID0gcmVjb25zdHJ1Y3Rpb25fbG9zcyArICgwLjAyICogc3BhcnNpdHlfcGVuYWx0eSkgKyAoMC4wMSAqIGRlY29ycmVsYXRpb25fcGVuYWx0eSkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgbGF0ZW50LCByZWNvbnN0cnVjdGlvbiA9IG1vZGVsKHNpZ25hbHMpCiAgICAgICAgcmVjb25zdHJ1Y3Rpb25fbG9zcyA9IGxvc3NfZm4ocmVjb25zdHJ1Y3Rpb24sIHNpZ25hbHMpLml0ZW0oKQogICAgICAgIHNwYXJzaXR5X3Njb3JlID0gZmxvYXQoMS4wIC0gdG9yY2guY2xhbXAobGF0ZW50LmFicygpLm1lYW4oKSwgMC4wLCAxLjApLml0ZW0oKSkKICAgICAgICBpbmRlcGVuZGVuY2Vfc2NvcmUgPSBmbG9hdCgKICAgICAgICAgICAgMS4wIC0gdG9yY2guY2xhbXAodG9yY2guY292KGxhdGVudC5UKS5hYnMoKS5tZWFuKCksIDAuMCwgMS4wKS5pdGVtKCkKICAgICAgICApCiAgICAgICAgcHJlZGljdGl2ZV91dGlsaXR5ID0gZmxvYXQoMS4wIC0gbWluKDEuMCwgcmVjb25zdHJ1Y3Rpb25fbG9zcykpCiAgICByZXR1cm4gVHJhbnNmb3JtTWV0cmljcygKICAgICAgICByZWNvbnN0cnVjdGlvbl9sb3NzPXJvdW5kKHJlY29uc3RydWN0aW9uX2xvc3MsIDYpLAogICAgICAgIHNwYXJzaXR5X3Njb3JlPXJvdW5kKHNwYXJzaXR5X3Njb3JlLCA2KSwKICAgICAgICBpbmRlcGVuZGVuY2Vfc2NvcmU9cm91bmQoaW5kZXBlbmRlbmNlX3Njb3JlLCA2KSwKICAgICAgICBwcmVkaWN0aXZlX3V0aWxpdHk9cm91bmQocHJlZGljdGl2ZV91dGlsaXR5LCA2KSwKICAgICkK', 'olympus/core/verification.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwsIEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFZlcmlmaWNhdGlvbkV2aWRlbmNlKFN0cmljdE1vZGVsKToKICAgIHZlcmlmaWVyOiBzdHIKICAgIHBhc3NlZDogYm9vbAogICAgZGV0YWlsOiBzdHIKICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgU2NoZW1hVmVyaWZpZXI6CiAgICBkZWYgdmVyaWZ5KHNlbGYsIG1vZGVsOiB0eXBlW0Jhc2VNb2RlbF0sIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBWZXJpZmljYXRpb25FdmlkZW5jZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZhbGlkYXRlZCA9IG1vZGVsLm1vZGVsX3ZhbGlkYXRlKHBheWxvYWQpCiAgICAgICAgICAgIHJldHVybiBWZXJpZmljYXRpb25FdmlkZW5jZSgKICAgICAgICAgICAgICAgIHZlcmlmaWVyPSJzY2hlbWEiLAogICAgICAgICAgICAgICAgcGFzc2VkPVRydWUsCiAgICAgICAgICAgICAgICBkZXRhaWw9IlBheWxvYWQgY29uZm9ybXMgdG8gc2NoZW1hLiIsCiAgICAgICAgICAgICAgICBwYXlsb2FkPXZhbGlkYXRlZC5tb2RlbF9kdW1wKCksCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgcmV0dXJuIFZlcmlmaWNhdGlvbkV2aWRlbmNlKAogICAgICAgICAgICAgICAgdmVyaWZpZXI9InNjaGVtYSIsCiAgICAgICAgICAgICAgICBwYXNzZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICBkZXRhaWw9c3RyKGVycm9yKSwKICAgICAgICAgICAgKQoKCmNsYXNzIENvZGVWZXJpZmllcjoKICAgIGRlZiB2ZXJpZnkoc2VsZiwgY29kZTogc3RyKSAtPiBWZXJpZmljYXRpb25FdmlkZW5jZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHRyZWUgPSBhc3QucGFyc2UoY29kZSkKICAgICAgICAgICAgZnVuY3Rpb25zID0gW25vZGUubmFtZSBmb3Igbm9kZSBpbiBhc3Qud2Fsayh0cmVlKSBpZiBpc2luc3RhbmNlKG5vZGUsIGFzdC5GdW5jdGlvbkRlZildCiAgICAgICAgICAgIHJldHVybiBWZXJpZmljYXRpb25FdmlkZW5jZSgKICAgICAgICAgICAgICAgIHZlcmlmaWVyPSJjb2RlIiwKICAgICAgICAgICAgICAgIHBhc3NlZD1UcnVlLAogICAgICAgICAgICAgICAgZGV0YWlsPSJQeXRob24gc291cmNlIHBhcnNlZCBzdWNjZXNzZnVsbHkuIiwKICAgICAgICAgICAgICAgIHBheWxvYWQ9eyJmdW5jdGlvbnMiOiBmdW5jdGlvbnN9LAogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IFN5bnRheEVycm9yIGFzIGVycm9yOgogICAgICAgICAgICByZXR1cm4gVmVyaWZpY2F0aW9uRXZpZGVuY2UoCiAgICAgICAgICAgICAgICB2ZXJpZmllcj0iY29kZSIsCiAgICAgICAgICAgICAgICBwYXNzZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICBkZXRhaWw9c3RyKGVycm9yKSwKICAgICAgICAgICAgKQoKCmNsYXNzIENvbnRyYWRpY3Rpb25WZXJpZmllcjoKICAgIGRlZiB2ZXJpZnkoc2VsZiwgc3RhdGVtZW50czogbGlzdFtzdHJdKSAtPiBWZXJpZmljYXRpb25FdmlkZW5jZToKICAgICAgICBub3JtYWxpemVkID0ge3N0YXRlbWVudC5sb3dlcigpLnN0cmlwKCkgZm9yIHN0YXRlbWVudCBpbiBzdGF0ZW1lbnRzfQogICAgICAgIGNvbnRyYWRpY3Rpb24gPSBhbnkoCiAgICAgICAgICAgIHN0YXRlbWVudC5zdGFydHN3aXRoKCJub3QgIikgYW5kIHN0YXRlbWVudC5yZW1vdmVwcmVmaXgoIm5vdCAiKSBpbiBub3JtYWxpemVkCiAgICAgICAgICAgIGZvciBzdGF0ZW1lbnQgaW4gbm9ybWFsaXplZAogICAgICAgICkKICAgICAgICByZXR1cm4gVmVyaWZpY2F0aW9uRXZpZGVuY2UoCiAgICAgICAgICAgIHZlcmlmaWVyPSJjb250cmFkaWN0aW9uIiwKICAgICAgICAgICAgcGFzc2VkPW5vdCBjb250cmFkaWN0aW9uLAogICAgICAgICAgICBkZXRhaWw9IkNvbnRyYWRpY3Rpb24gZGV0ZWN0ZWQuIgogICAgICAgICAgICBpZiBjb250cmFkaWN0aW9uCiAgICAgICAgICAgIGVsc2UgIk5vIGRpcmVjdCBjb250cmFkaWN0aW9uIGRldGVjdGVkLiIsCiAgICAgICAgICAgIHBheWxvYWQ9eyJzdGF0ZW1lbnRzIjogc3RhdGVtZW50c30sCiAgICAgICAgKQoKCmNsYXNzIENhbGlicmF0aW9uVmVyaWZpZXI6CiAgICBkZWYgdmVyaWZ5KHNlbGYsIGNvbmZpZGVuY2U6IGZsb2F0LCBwYXNzZWQ6IGJvb2wpIC0+IFZlcmlmaWNhdGlvbkV2aWRlbmNlOgogICAgICAgIGNhbGlicmF0ZWQgPSBub3QgKChjb25maWRlbmNlID4gMC44IGFuZCBub3QgcGFzc2VkKSBvciAoY29uZmlkZW5jZSA8IDAuMiBhbmQgcGFzc2VkKSkKICAgICAgICByZXR1cm4gVmVyaWZpY2F0aW9uRXZpZGVuY2UoCiAgICAgICAgICAgIHZlcmlmaWVyPSJjYWxpYnJhdGlvbiIsCiAgICAgICAgICAgIHBhc3NlZD1jYWxpYnJhdGVkLAogICAgICAgICAgICBkZXRhaWw9IkNvbmZpZGVuY2UgaXMgY2FsaWJyYXRlZC4iCiAgICAgICAgICAgIGlmIGNhbGlicmF0ZWQKICAgICAgICAgICAgZWxzZSAiQ29uZmlkZW5jZSBpcyBwb29ybHkgY2FsaWJyYXRlZC4iLAogICAgICAgICAgICBwYXlsb2FkPXsiY29uZmlkZW5jZSI6IGNvbmZpZGVuY2UsICJvdXRjb21lIjogcGFzc2VkfSwKICAgICAgICApCg==', 'olympus/core/workspace.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIGNhc3QKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBFdmlkZW5jZSwgSHlwb3RoZXNpcywgUHJvdmVuYW5jZVJlY29yZCwgU3RyaWN0TW9kZWwKCgpjbGFzcyBXb3Jrc3BhY2VEaWZmKFN0cmljdE1vZGVsKToKICAgIGNoYW5nZWRfZmllbGRzOiBsaXN0W3N0cl0KICAgIG9sZF92YWx1ZXM6IGRpY3Rbc3RyLCBBbnldCiAgICBuZXdfdmFsdWVzOiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIExhdGVudFdvcmtzcGFjZShTdHJpY3RNb2RlbCk6CiAgICBvYmplY3RpdmU6IHN0cgogICAgdGFza19zdGF0ZTogZGljdFtzdHIsIEFueV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGNvbnN0cmFpbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGV2aWRlbmNlOiBsaXN0W0V2aWRlbmNlXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgaHlwb3RoZXNlczogbGlzdFtIeXBvdGhlc2lzXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgdW5jZXJ0YWludHk6IGZsb2F0ID0gMC41CiAgICBhY3RpdmVfcGxhbjogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBtZW1vcnlfcmVmZXJlbmNlczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBwcmVkaWN0ZWRfY29uc2VxdWVuY2VzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHRvb2xzX2F2YWlsYWJsZTogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBicmFuY2hfYWN0aXZhdGlvbnM6IGRpY3Rbc3RyLCBmbG9hdF0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHByb3ZlbmFuY2U6IGxpc3RbUHJvdmVuYW5jZVJlY29yZF0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHZlcnNpb246IGludCA9IDEKCiAgICBkZWYgc2VyaWFsaXplKHNlbGYpIC0+IHN0cjoKICAgICAgICByZXR1cm4gc2VsZi5tb2RlbF9kdW1wX2pzb24oaW5kZW50PTIpCgogICAgZGVmIGNoZWNrcG9pbnQoc2VsZiwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBwYXRoLndyaXRlX3RleHQoc2VsZi5zZXJpYWxpemUoKSwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBsb2FkKGNscywgcGF0aDogUGF0aCkgLT4gTGF0ZW50V29ya3NwYWNlOgogICAgICAgIHJldHVybiBjbHMubW9kZWxfdmFsaWRhdGVfanNvbihwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKCiAgICBkZWYgbWVyZ2Uoc2VsZiwgb3RoZXI6IExhdGVudFdvcmtzcGFjZSkgLT4gTGF0ZW50V29ya3NwYWNlOgogICAgICAgIG1lcmdlZCA9IHNlbGYubW9kZWxfY29weShkZWVwPVRydWUpCiAgICAgICAgbWVyZ2VkLnRhc2tfc3RhdGUudXBkYXRlKG90aGVyLnRhc2tfc3RhdGUpCiAgICAgICAgbWVyZ2VkLmNvbnN0cmFpbnRzID0gc29ydGVkKHNldChzZWxmLmNvbnN0cmFpbnRzICsgb3RoZXIuY29uc3RyYWludHMpKQogICAgICAgIG1lcmdlZC5ldmlkZW5jZS5leHRlbmQob3RoZXIuZXZpZGVuY2UpCiAgICAgICAgbWVyZ2VkLmh5cG90aGVzZXMuZXh0ZW5kKG90aGVyLmh5cG90aGVzZXMpCiAgICAgICAgbWVyZ2VkLmFjdGl2ZV9wbGFuID0gc2VsZi5hY3RpdmVfcGxhbiArIFsKICAgICAgICAgICAgc3RlcCBmb3Igc3RlcCBpbiBvdGhlci5hY3RpdmVfcGxhbiBpZiBzdGVwIG5vdCBpbiBzZWxmLmFjdGl2ZV9wbGFuCiAgICAgICAgXQogICAgICAgIG1lcmdlZC5tZW1vcnlfcmVmZXJlbmNlcyA9IHNvcnRlZChzZXQoc2VsZi5tZW1vcnlfcmVmZXJlbmNlcyArIG90aGVyLm1lbW9yeV9yZWZlcmVuY2VzKSkKICAgICAgICBtZXJnZWQucHJlZGljdGVkX2NvbnNlcXVlbmNlcyA9IHNvcnRlZCgKICAgICAgICAgICAgc2V0KHNlbGYucHJlZGljdGVkX2NvbnNlcXVlbmNlcyArIG90aGVyLnByZWRpY3RlZF9jb25zZXF1ZW5jZXMpCiAgICAgICAgKQogICAgICAgIG1lcmdlZC50b29sc19hdmFpbGFibGUgPSBzb3J0ZWQoc2V0KHNlbGYudG9vbHNfYXZhaWxhYmxlICsgb3RoZXIudG9vbHNfYXZhaWxhYmxlKSkKICAgICAgICBtZXJnZWQuYnJhbmNoX2FjdGl2YXRpb25zLnVwZGF0ZShvdGhlci5icmFuY2hfYWN0aXZhdGlvbnMpCiAgICAgICAgbWVyZ2VkLnByb3ZlbmFuY2UuZXh0ZW5kKG90aGVyLnByb3ZlbmFuY2UpCiAgICAgICAgbWVyZ2VkLnVuY2VydGFpbnR5ID0gcm91bmQoKHNlbGYudW5jZXJ0YWludHkgKyBvdGhlci51bmNlcnRhaW50eSkgLyAyLjAsIDQpCiAgICAgICAgbWVyZ2VkLnZlcnNpb24gPSBtYXgoc2VsZi52ZXJzaW9uLCBvdGhlci52ZXJzaW9uKSArIDEKICAgICAgICByZXR1cm4gbWVyZ2VkCgogICAgZGVmIGRpZmYoc2VsZiwgb3RoZXI6IExhdGVudFdvcmtzcGFjZSkgLT4gV29ya3NwYWNlRGlmZjoKICAgICAgICBjdXJyZW50ID0gc2VsZi5tb2RlbF9kdW1wKCkKICAgICAgICB0YXJnZXQgPSBvdGhlci5tb2RlbF9kdW1wKCkKICAgICAgICBjaGFuZ2VkX2ZpZWxkcyA9IHNvcnRlZChrZXkgZm9yIGtleSBpbiBjdXJyZW50IGlmIGN1cnJlbnRba2V5XSAhPSB0YXJnZXRba2V5XSkKICAgICAgICByZXR1cm4gV29ya3NwYWNlRGlmZigKICAgICAgICAgICAgY2hhbmdlZF9maWVsZHM9Y2hhbmdlZF9maWVsZHMsCiAgICAgICAgICAgIG9sZF92YWx1ZXM9e2tleTogY3VycmVudFtrZXldIGZvciBrZXkgaW4gY2hhbmdlZF9maWVsZHN9LAogICAgICAgICAgICBuZXdfdmFsdWVzPXtrZXk6IHRhcmdldFtrZXldIGZvciBrZXkgaW4gY2hhbmdlZF9maWVsZHN9LAogICAgICAgICkKCiAgICBkZWYgcmVwbGF5KHNlbGYpIC0+IGxpc3RbZGljdFtzdHIsIEFueV1dOgogICAgICAgIHRpbWVsaW5lID0gW3siZXZlbnQiOiAid29ya3NwYWNlX2luaXRpYWxpemVkIiwgIm9iamVjdGl2ZSI6IHNlbGYub2JqZWN0aXZlLCAidmVyc2lvbiI6IDF9XQogICAgICAgIGZvciByZWNvcmQgaW4gc2VsZi5wcm92ZW5hbmNlOgogICAgICAgICAgICB0aW1lbGluZS5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImV2ZW50IjogcmVjb3JkLmFjdGlvbiwKICAgICAgICAgICAgICAgICAgICAiYWN0b3IiOiByZWNvcmQuYWN0b3IsCiAgICAgICAgICAgICAgICAgICAgIm1ldGFkYXRhIjogcmVjb3JkLm1ldGFkYXRhLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHRpbWVsaW5lCgogICAgZGVmIGV4cG9ydF9zbmFwc2hvdChzZWxmLCBwYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICBwYXlsb2FkID0gY2FzdChkaWN0W3N0ciwgQW55XSwganNvbi5sb2FkcyhzZWxmLnNlcmlhbGl6ZSgpKSkKICAgICAgICBwYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgcmV0dXJuIHBheWxvYWQK', 'olympus/data/__init__.py': 'IiIiRGF0YSBwbGF0Zm9ybSBjb21wb25lbnRzLiIiIgoKZnJvbSBvbHltcHVzLmRhdGEuY29udHJvbGxlZF9mdXNpb24gaW1wb3J0ICgKICAgIFJFR0lTVEVSRURfU1BMSVRTLAogICAgQ29udHJvbGxlZERhdGFzZXRSZWNlaXB0LAogICAgQ29udHJvbGxlZFNwbGl0U3BlYywKICAgIGdlbmVyYXRlX2NvbnRyb2xsZWRfZnVzaW9uX2RhdGFzZXQsCiAgICBnZW5lcmF0ZV9jb250cm9sbGVkX3NwbGl0LAopCmZyb20gb2x5bXB1cy5kYXRhLmVkaXRjbGV2ciBpbXBvcnQgKAogICAgRWRpdENMRVZSRGF0YXNldCwKICAgIEVkaXRDTEVWUlBhaXIsCiAgICBFZGl0Q0xFVlJQYWlyUGF0aHMsCiAgICBpdGVyX3NwbGl0X3JlY29yZHMsCikKZnJvbSBvbHltcHVzLmRhdGEub2x5bXB1c19kYXRhc2V0IGltcG9ydCAoCiAgICBMb2FkZWRPbHltcHVzRGF0YXNldCwKICAgIE9seW1wdXNCYXRjaEFydGlmYWN0LAogICAgT2x5bXB1c1N5c3RlbURhdGFzZXRNYW5pZmVzdCwKICAgIGxvYWRfb2x5bXB1c19kYXRhc2V0LAogICAgd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlmZXN0LAopCmZyb20gb2x5bXB1cy5kYXRhLnJlbGlhYmlsaXR5X2Z1c2lvbiBpbXBvcnQgKAogICAgQ09NUE9ORU5UX09SREVSLAogICAgUkVHSVNURVJFRF9SRUxJQUJJTElUWV9TUExJVFMsCiAgICBSZWxpYWJpbGl0eURhdGFzZXRSZWNlaXB0LAogICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMsCiAgICBnZW5lcmF0ZV9yZWxpYWJpbGl0eV9mdXNpb25fZGF0YXNldCwKICAgIGdlbmVyYXRlX3JlbGlhYmlsaXR5X3NwbGl0LAopCgpfX2FsbF9fID0gWwogICAgIlJFR0lTVEVSRURfU1BMSVRTIiwKICAgICJDb250cm9sbGVkRGF0YXNldFJlY2VpcHQiLAogICAgIkNvbnRyb2xsZWRTcGxpdFNwZWMiLAogICAgIkNPTVBPTkVOVF9PUkRFUiIsCiAgICAiRWRpdENMRVZSRGF0YXNldCIsCiAgICAiRWRpdENMRVZSUGFpciIsCiAgICAiRWRpdENMRVZSUGFpclBhdGhzIiwKICAgICJMb2FkZWRPbHltcHVzRGF0YXNldCIsCiAgICAiT2x5bXB1c0JhdGNoQXJ0aWZhY3QiLAogICAgIk9seW1wdXNTeXN0ZW1EYXRhc2V0TWFuaWZlc3QiLAogICAgIlJFR0lTVEVSRURfUkVMSUFCSUxJVFlfU1BMSVRTIiwKICAgICJSZWxpYWJpbGl0eURhdGFzZXRSZWNlaXB0IiwKICAgICJSZWxpYWJpbGl0eVNwbGl0U3BlYyIsCiAgICAiZ2VuZXJhdGVfY29udHJvbGxlZF9mdXNpb25fZGF0YXNldCIsCiAgICAiZ2VuZXJhdGVfY29udHJvbGxlZF9zcGxpdCIsCiAgICAiZ2VuZXJhdGVfcmVsaWFiaWxpdHlfZnVzaW9uX2RhdGFzZXQiLAogICAgImdlbmVyYXRlX3JlbGlhYmlsaXR5X3NwbGl0IiwKICAgICJpdGVyX3NwbGl0X3JlY29yZHMiLAogICAgImxvYWRfb2x5bXB1c19kYXRhc2V0IiwKICAgICJ3cml0ZV9vbHltcHVzX2RhdGFzZXRfbWFuaWZlc3QiLApdCg==', 'olympus/data/conservative_routing_v6.py': 'IiIiRnJlc2ggc291cmNlIGlkZW50aXRpZXMgd2l0aCBhIHByb3NwZWN0aXZlbHkgcmVzZXJ2ZWQgY29tcG9zaXRlIHNoaWZ0LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmRhdGEucm9idXN0X2Z1c2lvbl92NSBpbXBvcnQgZ2VuZXJhdGUgYXMgcHJldmlvdXNfZ2VuZXJhdGUKCgpkZWYgZ2VuZXJhdGUoc2VlZDogaW50LCBjb3VudDogaW50LCByZWdpbWU6IHN0cikgLT4gZGljdFtzdHIsIFRlbnNvcl06CiAgICBpZiByZWdpbWUgIT0gImNvbXBvc2l0ZV9zaGlmdCI6CiAgICAgICAgcmV0dXJuIHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNvdW50LCByZWdpbWUpCiAgICBiYXRjaCA9IHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNvdW50LCAiY29ycnVwdGVkIikKICAgIGNsZWFuID0gcHJldmlvdXNfZ2VuZXJhdGUoc2VlZCwgY291bnQsICJjbGVhbiIpCiAgICAjIFJlc2VydmVkIGNvbmZpZ3VyYXRpb24gZGVjbGFyZWQgYmVmb3JlIGV4ZWN1dGlvbjsgbm8gdGFyZ2V0LWRlcGVuZGVudCB0cmFuc2Zvcm0uCiAgICAjIEtub3duIHBlcnR1cmJhdGlvbiB0eXBlcywgbm90IGEgbm92ZWwgZmFtaWx5IG9yIG5hdHVyYWwtZGF0YSBPT0QgY2xhaW0uCiAgICBmb3IgaW5kZXgsIGtleSBpbiBlbnVtZXJhdGUoKCJ0b2tlbnMiLCAiaW1hZ2UiLCAic3RhdGVzIikpOgogICAgICAgIG1hc2sgPSBiYXRjaFsiY29ycnVwdGlvbl9zdGF0dXMiXVs6LCBpbmRleF0gPT0gMgogICAgICAgIHggPSBjbGVhbltrZXldLmNsb25lKCkKICAgICAgICBpZiBrZXkgPT0gInRva2VucyI6CiAgICAgICAgICAgIHggPSB4LnJvbGwoMiwgLTEpCiAgICAgICAgZWxpZiBrZXkgPT0gImltYWdlIjoKICAgICAgICAgICAgeFs6LCA6LCAxOjoyLCA6XSA9IDAuNQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHggPSAwLjUgKiB4ICsgdG9yY2gubGluc3BhY2UoLTAuNzUsIDAuNzUsIHguc2hhcGVbMV0pW05vbmUsIDosIE5vbmVdCiAgICAgICAgYmF0Y2hba2V5XVttYXNrXSA9IHhbbWFza10KICAgIHJldHVybiBiYXRjaAo=', 'olympus/data/controlled_fusion.py': 'IiIiRGV0ZXJtaW5pc3RpYyBhbGlnbmVkIGRhdGEgZm9yIHRoZSByZWdpc3RlcmVkIE9seW1wdXMgY29udHJvbGxlZC1tZWNoYW5pc20gc3R1ZHkuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZApmcm9tIHRvcmNoIGltcG9ydCBUZW5zb3IKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5kYXRhLm9seW1wdXNfZGF0YXNldCBpbXBvcnQgd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlmZXN0CgpDb250cm9sbGVkUmVnaW1lID0gTGl0ZXJhbFsibWl4ZWQiLCAiY2xlYW4iLCAiY29ycnVwdGVkIiwgIm9vZCJdCgoKY2xhc3MgQ29udHJvbGxlZFNwbGl0U3BlYyhTdHJpY3RNb2RlbCk6CiAgICBuYW1lOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEpCiAgICBleGFtcGxlczogaW50ID0gRmllbGQoZ2U9MSkKICAgIHNlZWQ6IGludCA9IEZpZWxkKGdlPTAsIGxlPTIqKjYzIC0gMSkKICAgIHJlZ2ltZTogQ29udHJvbGxlZFJlZ2ltZQoKCmNsYXNzIENvbnRyb2xsZWREYXRhc2V0UmVjZWlwdChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogTGl0ZXJhbFsxXSA9IDEKICAgIGRhdGFzZXRfaWQ6IExpdGVyYWxbIm9seW1wdXMtY29udHJvbGxlZC1mdXNpb24iXSA9ICJvbHltcHVzLWNvbnRyb2xsZWQtZnVzaW9uIgogICAgdmVyc2lvbjogTGl0ZXJhbFsiMi4wLjAiXSA9ICIyLjAuMCIKICAgIGxpY2Vuc2U6IExpdGVyYWxbIkNDLUJZLTQuMCJdID0gIkNDLUJZLTQuMCIKICAgIGdlbmVyYXRvcjogTGl0ZXJhbFsib2x5bXB1cy5kYXRhLmNvbnRyb2xsZWRfZnVzaW9uIl0gPSAib2x5bXB1cy5kYXRhLmNvbnRyb2xsZWRfZnVzaW9uIgogICAgc3BsaXRzOiBsaXN0W0NvbnRyb2xsZWRTcGxpdFNwZWNdCiAgICBidW5kbGVfc2hhMjU2OiBkaWN0W3N0ciwgc3RyXQogICAgbWFuaWZlc3Rfc2hhMjU2OiBkaWN0W3N0ciwgc3RyXQogICAgc291cmNlX2lkX3NoYTI1NjogZGljdFtzdHIsIHN0cl0KICAgIHNvdXJjZV9vdmVybGFwX2NvdW50OiBpbnQgPSBGaWVsZChnZT0wKQogICAgcHJvdGVjdGVkX3NwbGl0czogbGlzdFtzdHJdCiAgICBjbGFpbV9ib3VuZGFyeTogc3RyCgoKUkVHSVNURVJFRF9TUExJVFMgPSAoCiAgICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9InRyYWluIiwgZXhhbXBsZXM9NTEyLCBzZWVkPTIwMjYxMjAxLCByZWdpbWU9Im1peGVkIiksCiAgICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9ImRldmVsb3BtZW50IiwgZXhhbXBsZXM9MTI4LCBzZWVkPTIwMjYxMjAyLCByZWdpbWU9Im1peGVkIiksCiAgICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9ImNsZWFuX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjEyMDMsIHJlZ2ltZT0iY2xlYW4iKSwKICAgIENvbnRyb2xsZWRTcGxpdFNwZWMoCiAgICAgICAgbmFtZT0iY29ycnVwdGVkX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjEyMDQsIHJlZ2ltZT0iY29ycnVwdGVkIgogICAgKSwKICAgIENvbnRyb2xsZWRTcGxpdFNwZWMobmFtZT0ib29kX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjEyMDUsIHJlZ2ltZT0ib29kIiksCikKCgpkZWYgX3NoYTI1NihwYXlsb2FkOiBieXRlcykgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCgoKZGVmIF9hdG9taWNfd3JpdGUocGF0aDogUGF0aCwgcGF5bG9hZDogYnl0ZXMpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICBoYW5kbGUud3JpdGUocGF5bG9hZCkKICAgICAgICBoYW5kbGUuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgX3NvdXJjZV9pZHMoc3BlYzogQ29udHJvbGxlZFNwbGl0U3BlYykgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIFsKICAgICAgICBmIntzcGVjLm5hbWV9LXtfc2hhMjU2KGYne3NwZWMuc2VlZH06e2luZGV4fScuZW5jb2RlKCkpWzoyMF19IgogICAgICAgIGZvciBpbmRleCBpbiByYW5nZShzcGVjLmV4YW1wbGVzKQogICAgXQoKCmRlZiBfY29ycnVwdGlvbl9wbGFuKAogICAgc3BlYzogQ29udHJvbGxlZFNwbGl0U3BlYywgZ2VuZXJhdG9yOiB0b3JjaC5HZW5lcmF0b3IKKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvcl06CiAgICAiIiJSZXR1cm4gbWlzc2luZyBhbmQgcGVydHVyYmVkIG1vZGFsaXR5IGluZGljZXM7IC0xIG1lYW5zIG5vIGludGVydmVudGlvbi4iIiIKCiAgICBtaXNzaW5nID0gdG9yY2guZnVsbCgoc3BlYy5leGFtcGxlcywpLCAtMSwgZHR5cGU9dG9yY2gubG9uZykKICAgIHBlcnR1cmJlZCA9IHRvcmNoLmZ1bGwoKHNwZWMuZXhhbXBsZXMsKSwgLTEsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBpZiBzcGVjLnJlZ2ltZSA9PSAibWl4ZWQiOgogICAgICAgIHNlbGVjdGVkID0gdG9yY2gucmFuZChzcGVjLmV4YW1wbGVzLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSA8IDAuNzAKICAgICAgICBtaXNzaW5nW3NlbGVjdGVkXSA9IHRvcmNoLnJhbmRpbnQoMCwgMywgKGludChzZWxlY3RlZC5zdW0oKSksKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgIGVsaWYgc3BlYy5yZWdpbWUgaW4geyJjb3JydXB0ZWQiLCAib29kIn06CiAgICAgICAgbWlzc2luZyA9IHRvcmNoLnJhbmRpbnQoMCwgMywgKHNwZWMuZXhhbXBsZXMsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgICAgICBvZmZzZXQgPSB0b3JjaC5yYW5kaW50KDEsIDMsIChzcGVjLmV4YW1wbGVzLCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAgICAgcGVydHVyYmVkID0gKG1pc3NpbmcgKyBvZmZzZXQpICUgMwogICAgcmV0dXJuIG1pc3NpbmcsIHBlcnR1cmJlZAoKCmRlZiBnZW5lcmF0ZV9jb250cm9sbGVkX3NwbGl0KHNwZWM6IENvbnRyb2xsZWRTcGxpdFNwZWMpIC0+IHR1cGxlW2RpY3Rbc3RyLCBUZW5zb3JdLCBsaXN0W3N0cl1dOgogICAgIiIiR2VuZXJhdGUgb25lIGRldGVybWluaXN0aWMgc3BsaXQgd2l0aG91dCByZWFkaW5nIGFueSBvdGhlciBzcGxpdCBvciBnbG9iYWwgUk5HLiIiIgoKICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcihkZXZpY2U9ImNwdSIpLm1hbnVhbF9zZWVkKHNwZWMuc2VlZCkKICAgIGNvdW50ID0gc3BlYy5leGFtcGxlcwogICAgbGFiZWxzID0gdG9yY2gucmFuZGludCgwLCA0LCAoY291bnQsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgIG1pc3NpbmcsIHBlcnR1cmJlZCA9IF9jb3JydXB0aW9uX3BsYW4oc3BlYywgZ2VuZXJhdG9yKQogICAgbm9pc2Vfc2NhbGUgPSAwLjEyIGlmIHNwZWMucmVnaW1lICE9ICJvb2QiIGVsc2UgMC4yNAoKICAgIHRva2VucyA9IHRvcmNoLnJhbmRpbnQoOCwgMzEsIChjb3VudCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBjbGFzc190b2tlbnMgPSBsYWJlbHMgKyAxCiAgICBpZiBzcGVjLnJlZ2ltZSA9PSAib29kIjoKICAgICAgICBjbGFzc190b2tlbnMgPSB0b3JjaC50ZW5zb3IoWzMsIDEsIDQsIDJdLCBkdHlwZT10b3JjaC5sb25nKVtsYWJlbHNdCiAgICB0b2tlbnNbOiwgMF0gPSBjbGFzc190b2tlbnMKICAgIHRva2Vuc1s6LCAxXSA9IDUgKyBsYWJlbHMucmVtYWluZGVyKDIpCiAgICB0b2tlbnNbOiwgMl0gPSA3ICsgbGFiZWxzLmRpdigyLCByb3VuZGluZ19tb2RlPSJmbG9vciIpCiAgICBhdHRlbnRpb25fbWFzayA9IHRvcmNoLm9uZXNfbGlrZSh0b2tlbnMsIGR0eXBlPXRvcmNoLmJvb2wpCgogICAgaW1hZ2UgPSB0b3JjaC5yYW5kKChjb3VudCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpICogbm9pc2Vfc2NhbGUKICAgIGZvciBpbmRleCBpbiByYW5nZShjb3VudCk6CiAgICAgICAgbGFiZWwgPSBpbnQobGFiZWxzW2luZGV4XSkKICAgICAgICBjaGFubmVsID0gbGFiZWwgJSAzCiAgICAgICAgdG9wID0gMCBpZiBsYWJlbCA8IDIgZWxzZSA0CiAgICAgICAgbGVmdCA9IDAgaWYgbGFiZWwgJSAyID09IDAgZWxzZSA0CiAgICAgICAgaWYgc3BlYy5yZWdpbWUgPT0gIm9vZCI6CiAgICAgICAgICAgIGNoYW5uZWwgPSAoY2hhbm5lbCArIDEpICUgMwogICAgICAgICAgICBsZWZ0ID0gNCAtIGxlZnQKICAgICAgICBpbWFnZVtpbmRleCwgY2hhbm5lbCwgdG9wIDogdG9wICsgNCwgbGVmdCA6IGxlZnQgKyA0XSArPSAwLjc1CiAgICBpbWFnZS5jbGFtcF8oMC4wLCAxLjApCgogICAgYWN0aW9ucyA9IHRvcmNoLnJhbmQoKGNvdW50LCA0LCAzKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikgKiAyLjAgLSAxLjAKICAgIHRpbWUgPSB0b3JjaC5hcmFuZ2UoNCwgZHR5cGU9dG9yY2guZmxvYXQzMikudmlldygxLCA0LCAxKQogICAgbGFiZWxfdmFsdWUgPSBsYWJlbHMuZmxvYXQoKS52aWV3KGNvdW50LCAxLCAxKSAvIDMuMAogICAgc3RhdGVfbm9pc2UgPSB0b3JjaC5yYW5kbigoY291bnQsIDQsIDUpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSAqIG5vaXNlX3NjYWxlCiAgICBzdGF0ZV90ZW1wbGF0ZSA9IHRvcmNoLmNhdCgKICAgICAgICAoCiAgICAgICAgICAgIGxhYmVsX3ZhbHVlLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIChsYWJlbF92YWx1ZSArIDAuMDggKiB0aW1lKS5leHBhbmQoLTEsIC0xLCAxKSwKICAgICAgICAgICAgKGxhYmVscy5yZW1haW5kZXIoMikuZmxvYXQoKS52aWV3KGNvdW50LCAxLCAxKSkuZXhwYW5kKC0xLCA0LCAxKSwKICAgICAgICAgICAgKGxhYmVscy5kaXYoMiwgcm91bmRpbmdfbW9kZT0iZmxvb3IiKS5mbG9hdCgpLnZpZXcoY291bnQsIDEsIDEpKS5leHBhbmQoCiAgICAgICAgICAgICAgICAtMSwgNCwgMQogICAgICAgICAgICApLAogICAgICAgICAgICB0b3JjaC5zaW4obGFiZWxfdmFsdWUgKiB0b3JjaC5waSArIHRpbWUgKiAwLjMpLAogICAgICAgICksCiAgICAgICAgZGltPS0xLAogICAgKQogICAgc3RhdGVzID0gc3RhdGVfdGVtcGxhdGUgKyBzdGF0ZV9ub2lzZQoKICAgIGZvciBtb2RhbGl0eSwgdmFsdWUgaW4gZW51bWVyYXRlKCh0b2tlbnMsIGltYWdlLCBzdGF0ZXMpKToKICAgICAgICBzZWxlY3RlZF9taXNzaW5nID0gbWlzc2luZyA9PSBtb2RhbGl0eQogICAgICAgIHNlbGVjdGVkX3BlcnR1cmJlZCA9IHBlcnR1cmJlZCA9PSBtb2RhbGl0eQogICAgICAgIGlmIGJvb2woc2VsZWN0ZWRfbWlzc2luZy5hbnkoKSk6CiAgICAgICAgICAgIHZhbHVlW3NlbGVjdGVkX21pc3NpbmddID0gMAogICAgICAgIGlmIGJvb2woc2VsZWN0ZWRfcGVydHVyYmVkLmFueSgpKToKICAgICAgICAgICAgaWYgbW9kYWxpdHkgPT0gMDoKICAgICAgICAgICAgICAgIHRva2Vuc1tzZWxlY3RlZF9wZXJ0dXJiZWQsIDozXSA9IHRvcmNoLnJhbmRpbnQoCiAgICAgICAgICAgICAgICAgICAgMSwKICAgICAgICAgICAgICAgICAgICAxMiwKICAgICAgICAgICAgICAgICAgICAoaW50KHNlbGVjdGVkX3BlcnR1cmJlZC5zdW0oKSksIDMpLAogICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj1nZW5lcmF0b3IsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGVsaWYgbW9kYWxpdHkgPT0gMToKICAgICAgICAgICAgICAgIGltYWdlW3NlbGVjdGVkX3BlcnR1cmJlZF0gPSB0b3JjaC5yYW5kKAogICAgICAgICAgICAgICAgICAgIChpbnQoc2VsZWN0ZWRfcGVydHVyYmVkLnN1bSgpKSwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHN0YXRlc1tzZWxlY3RlZF9wZXJ0dXJiZWRdID0gdG9yY2gucmFuZG4oCiAgICAgICAgICAgICAgICAgICAgKGludChzZWxlY3RlZF9wZXJ0dXJiZWQuc3VtKCkpLCA0LCA1KSwgZ2VuZXJhdG9yPWdlbmVyYXRvcgogICAgICAgICAgICAgICAgKQoKICAgIHRlbXBvcmFsX21hc2sgPSB0b3JjaC5vbmVzKChjb3VudCwgNCksIGR0eXBlPXRvcmNoLmJvb2wpCiAgICBhbGxvd2VkX3RyYW5zaXRpb25zID0gdG9yY2gub25lcygoY291bnQsIDQpLCBkdHlwZT10b3JjaC5ib29sKQogICAgdHJhbnNpdGlvbl90YXJnZXRzID0gbGFiZWxzLnJlbWFpbmRlcigzKQogICAgZm9yYmlkZGVuID0gKHRyYW5zaXRpb25fdGFyZ2V0cyArIDEpLnJlbWFpbmRlcigzKQogICAgYWxsb3dlZF90cmFuc2l0aW9uc1t0b3JjaC5hcmFuZ2UoY291bnQpLCBmb3JiaWRkZW5dID0gRmFsc2UKICAgIGFsbG93ZWRfdHJhbnNpdGlvbnNbOiwgM10gPSBUcnVlCgogICAgY29ycnVwdGlvbl9jb3VudCA9IG1pc3NpbmcuZ2UoMCkuZmxvYXQoKSArIHBlcnR1cmJlZC5nZSgwKS5mbG9hdCgpCiAgICByaXNrX3Njb3JlID0gMC4wNSArIGNvcnJ1cHRpb25fY291bnQgKiAwLjE1CiAgICBpbnRyaW5zaWNfcmlza190YXJnZXRzID0gY29ycnVwdGlvbl9jb3VudCAvIDIuMAogICAgcGFkZGVkX2FjdGlvbnMgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLnBhZChhY3Rpb25zLCAoMCwgMikpCiAgICBuZXh0X3N0YXRlX3RhcmdldHMgPSBzdGF0ZXMgKyAwLjEgKiBwYWRkZWRfYWN0aW9ucwoKICAgIGJ1bmRsZSA9IHsKICAgICAgICAidG9rZW5zIjogdG9rZW5zLAogICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAogICAgICAgICJpbWFnZSI6IGltYWdlLAogICAgICAgICJzdGF0ZXMiOiBzdGF0ZXMsCiAgICAgICAgImFjdGlvbnMiOiBhY3Rpb25zLAogICAgICAgICJ0ZW1wb3JhbF9tYXNrIjogdGVtcG9yYWxfbWFzaywKICAgICAgICAiYWxsb3dlZF90cmFuc2l0aW9ucyI6IGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAgICAgICAgInJpc2tfc2NvcmUiOiByaXNrX3Njb3JlLAogICAgICAgICJkb21haW5faWRzIjogbGFiZWxzLnJlbWFpbmRlcigzKSwKICAgICAgICAidGltZXN0YW1wIjogdG9yY2guYXJhbmdlKGNvdW50LCBkdHlwZT10b3JjaC5mbG9hdDMyKSAvIG1heCgxLCBjb3VudCAtIDEpLAogICAgICAgICJkZWNpc2lvbl90YXJnZXRzIjogbGFiZWxzLAogICAgICAgICJ0cmFuc2l0aW9uX3RhcmdldHMiOiB0cmFuc2l0aW9uX3RhcmdldHMsCiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyI6IG5leHRfc3RhdGVfdGFyZ2V0cywKICAgICAgICAiaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyI6IGludHJpbnNpY19yaXNrX3RhcmdldHMsCiAgICB9CiAgICByZXR1cm4gYnVuZGxlLCBfc291cmNlX2lkcyhzcGVjKQoKCmRlZiBfc2VyaWFsaXplX2J1bmRsZShidW5kbGU6IGRpY3Rbc3RyLCBUZW5zb3JdKSAtPiBieXRlczoKICAgIGJ1ZmZlciA9IGlvLkJ5dGVzSU8oKQogICAgdG9yY2guc2F2ZShidW5kbGUsIGJ1ZmZlcikKICAgIHJldHVybiBidWZmZXIuZ2V0dmFsdWUoKQoKCmRlZiBnZW5lcmF0ZV9jb250cm9sbGVkX2Z1c2lvbl9kYXRhc2V0KG91dHB1dDogUGF0aCkgLT4gQ29udHJvbGxlZERhdGFzZXRSZWNlaXB0OgogICAgIiIiTWF0ZXJpYWxpemUgdGhlIHJlZ2lzdGVyZWQgc3BsaXRzIGFuZCBoYXNoLWJvdW5kIHR3by1zcGxpdCBtYW5pZmVzdHMgb25jZS4iIiIKCiAgICBvdXRwdXQgPSBvdXRwdXQucmVzb2x2ZSgpCiAgICByZWNlaXB0X3BhdGggPSBvdXRwdXQgLyAicmVjZWlwdC5qc29uIgogICAgaWYgb3V0cHV0LmV4aXN0cygpIGFuZCBhbnkob3V0cHV0Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKCJyZWZ1c2luZyB0byBvdmVyd3JpdGUgYSBjb250cm9sbGVkIGRhdGFzZXQgZGlyZWN0b3J5IikKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgc291cmNlX2lkczogZGljdFtzdHIsIGxpc3Rbc3RyXV0gPSB7fQogICAgYnVuZGxlX2hhc2hlczogZGljdFtzdHIsIHN0cl0gPSB7fQogICAgZm9yIHNwZWMgaW4gUkVHSVNURVJFRF9TUExJVFM6CiAgICAgICAgYnVuZGxlLCBpZGVudGlmaWVycyA9IGdlbmVyYXRlX2NvbnRyb2xsZWRfc3BsaXQoc3BlYykKICAgICAgICBwYXlsb2FkID0gX3NlcmlhbGl6ZV9idW5kbGUoYnVuZGxlKQogICAgICAgIHBhdGggPSBvdXRwdXQgLyBmIntzcGVjLm5hbWV9LnB0IgogICAgICAgIF9hdG9taWNfd3JpdGUocGF0aCwgcGF5bG9hZCkKICAgICAgICBzb3VyY2VfaWRzW3NwZWMubmFtZV0gPSBpZGVudGlmaWVycwogICAgICAgIGJ1bmRsZV9oYXNoZXNbc3BlYy5uYW1lXSA9IF9zaGEyNTYocGF5bG9hZCkKCiAgICBhbGxfaWRzID0gW2lkZW50aWZpZXIgZm9yIHZhbHVlcyBpbiBzb3VyY2VfaWRzLnZhbHVlcygpIGZvciBpZGVudGlmaWVyIGluIHZhbHVlc10KICAgIG92ZXJsYXBfY291bnQgPSBsZW4oYWxsX2lkcykgLSBsZW4oc2V0KGFsbF9pZHMpKQogICAgaWYgb3ZlcmxhcF9jb3VudDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNvbnRyb2xsZWQgc3BsaXQgc291cmNlIGlkZW50aXRpZXMgb3ZlcmxhcCIpCgogICAgbWFuaWZlc3RfaGFzaGVzOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICBmb3IgZXZhbHVhdGlvbl9uYW1lIGluICgiZGV2ZWxvcG1lbnQiLCAiY2xlYW5fdGVzdCIsICJjb3JydXB0ZWRfdGVzdCIsICJvb2RfdGVzdCIpOgogICAgICAgIG1hbmlmZXN0ID0gd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlmZXN0KAogICAgICAgICAgICBvdXRwdXQgLyBmIm1hbmlmZXN0X3tldmFsdWF0aW9uX25hbWV9Lmpzb24iLAogICAgICAgICAgICBkYXRhc2V0X2lkPSJvbHltcHVzLWNvbnRyb2xsZWQtZnVzaW9uIiwKICAgICAgICAgICAgdmVyc2lvbj0iMi4wLjAiLAogICAgICAgICAgICBsaWNlbnNlX25hbWU9IkNDLUJZLTQuMCIsCiAgICAgICAgICAgIHRyYWluX2JhdGNoPW91dHB1dCAvICJ0cmFpbi5wdCIsCiAgICAgICAgICAgIGV2YWx1YXRpb25fYmF0Y2g9b3V0cHV0IC8gZiJ7ZXZhbHVhdGlvbl9uYW1lfS5wdCIsCiAgICAgICAgICAgIHRyYWluX3NvdXJjZV9pZHM9c291cmNlX2lkc1sidHJhaW4iXSwKICAgICAgICAgICAgZXZhbHVhdGlvbl9zb3VyY2VfaWRzPXNvdXJjZV9pZHNbZXZhbHVhdGlvbl9uYW1lXSwKICAgICAgICApCiAgICAgICAgYXNzZXJ0IG1hbmlmZXN0Lm1hbmlmZXN0X3NoYTI1NiBpcyBub3QgTm9uZQogICAgICAgIG1hbmlmZXN0X2hhc2hlc1tldmFsdWF0aW9uX25hbWVdID0gbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2CgogICAgcmVjZWlwdCA9IENvbnRyb2xsZWREYXRhc2V0UmVjZWlwdCgKICAgICAgICBzcGxpdHM9bGlzdChSRUdJU1RFUkVEX1NQTElUUyksCiAgICAgICAgYnVuZGxlX3NoYTI1Nj1idW5kbGVfaGFzaGVzLAogICAgICAgIG1hbmlmZXN0X3NoYTI1Nj1tYW5pZmVzdF9oYXNoZXMsCiAgICAgICAgc291cmNlX2lkX3NoYTI1Nj17CiAgICAgICAgICAgIG5hbWU6IF9zaGEyNTYoKCJcbiIuam9pbih2YWx1ZXMpICsgIlxuIikuZW5jb2RlKCkpCiAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZXMgaW4gc291cmNlX2lkcy5pdGVtcygpCiAgICAgICAgfSwKICAgICAgICBzb3VyY2Vfb3ZlcmxhcF9jb3VudD1vdmVybGFwX2NvdW50LAogICAgICAgIHByb3RlY3RlZF9zcGxpdHM9WyJjbGVhbl90ZXN0IiwgImNvcnJ1cHRlZF90ZXN0IiwgIm9vZF90ZXN0Il0sCiAgICAgICAgY2xhaW1fYm91bmRhcnk9KAogICAgICAgICAgICAiQ29udHJvbGxlZCBzeW50aGV0aWMgbWVjaGFuaXNtIGV2aWRlbmNlIG9ubHk7IG5vIG5hdHVyYWwtZGF0YSwgZGVwbG95bWVudCwgIgogICAgICAgICAgICAidGFyZ2V0LXNjYWxlLCBvciBKRVBBIGNsYWltIGlzIGxpY2Vuc2VkIGJ5IHRoaXMgZGF0YXNldC4iCiAgICAgICAgKSwKICAgICkKICAgIF9hdG9taWNfd3JpdGUoCiAgICAgICAgcmVjZWlwdF9wYXRoLAogICAgICAgIChqc29uLmR1bXBzKHJlY2VpcHQubW9kZWxfZHVtcChtb2RlPSJqc29uIiksIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKS5lbmNvZGUoKSwKICAgICkKICAgIHJldHVybiByZWNlaXB0Cgo=', 'olympus/data/editclevr.py': 'IiIiQm91bmRlZC1tZW1vcnkgYWNjZXNzIHRvIGEgdmFsaWRhdGVkLCBtYXRlcmlhbGl6ZWQgRWRpdENMRVZSIHNuYXBzaG90LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEl0ZXJhdG9yLCBTZXF1ZW5jZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIFNlbGYsIG92ZXJsb2FkCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gUElMIGltcG9ydCBJbWFnZQpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKClBBWUxPQURfRklFTERTID0gKAogICAgImJlZm9yZV9pbWFnZSIsCiAgICAiYWZ0ZXJfaW1hZ2UiLAogICAgImJlZm9yZV9zY2VuZV9qc29uIiwKICAgICJhZnRlcl9zY2VuZV9qc29uIiwKICAgICJpbnN0YW5jZV9tYXNrc19iZWZvcmUiLAogICAgImluc3RhbmNlX21hc2tzX2FmdGVyIiwKKQpTVUlURVMgPSBmcm96ZW5zZXQoeyJhdG9taWNfaWQiLCAiY29nZW50X29vZCIsICJoYXJkX2Rpc3RyYWN0b3IiLCAibm9fZWRpdCJ9KQoKCmNsYXNzIF9TdHJlYW1pbmdKc29uUmVhZGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGg6IFBhdGgsICosIGNodW5rX3NpemU6IGludCA9IDEwMjQgKiAxMDI0KSAtPiBOb25lOgogICAgICAgIHNlbGYuX2hhbmRsZSA9IHBhdGgub3BlbigiciIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgc2VsZi5fY2h1bmtfc2l6ZSA9IGNodW5rX3NpemUKICAgICAgICBzZWxmLl9idWZmZXIgPSAiIgogICAgICAgIHNlbGYuX3Bvc2l0aW9uID0gMAogICAgICAgIHNlbGYuX2VvZiA9IEZhbHNlCiAgICAgICAgc2VsZi5fZGVjb2RlciA9IGpzb24uSlNPTkRlY29kZXIoKQoKICAgIGRlZiBjbG9zZShzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuX2hhbmRsZS5jbG9zZSgpCgogICAgZGVmIF9jb21wYWN0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5fcG9zaXRpb246CiAgICAgICAgICAgIHNlbGYuX2J1ZmZlciA9IHNlbGYuX2J1ZmZlcltzZWxmLl9wb3NpdGlvbiA6XQogICAgICAgICAgICBzZWxmLl9wb3NpdGlvbiA9IDAKCiAgICBkZWYgX2ZpbGwoc2VsZikgLT4gYm9vbDoKICAgICAgICBpZiBzZWxmLl9lb2Y6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHNlbGYuX2NvbXBhY3QoKQogICAgICAgIGNodW5rID0gc2VsZi5faGFuZGxlLnJlYWQoc2VsZi5fY2h1bmtfc2l6ZSkKICAgICAgICBpZiBub3QgY2h1bms6CiAgICAgICAgICAgIHNlbGYuX2VvZiA9IFRydWUKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5fYnVmZmVyICs9IGNodW5rCiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBkZWYgc2tpcF93aGl0ZXNwYWNlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgc2VsZi5fcG9zaXRpb24gPCBsZW4oc2VsZi5fYnVmZmVyKSBhbmQgc2VsZi5fYnVmZmVyW3NlbGYuX3Bvc2l0aW9uXS5pc3NwYWNlKCk6CiAgICAgICAgICAgICAgICBzZWxmLl9wb3NpdGlvbiArPSAxCiAgICAgICAgICAgIGlmIHNlbGYuX3Bvc2l0aW9uIDwgbGVuKHNlbGYuX2J1ZmZlcikgb3Igbm90IHNlbGYuX2ZpbGwoKToKICAgICAgICAgICAgICAgIHJldHVybgoKICAgIGRlZiBwZWVrKHNlbGYpIC0+IHN0cjoKICAgICAgICBzZWxmLnNraXBfd2hpdGVzcGFjZSgpCiAgICAgICAgaWYgc2VsZi5fcG9zaXRpb24gPj0gbGVuKHNlbGYuX2J1ZmZlcik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVuZXhwZWN0ZWQgZW5kIG9mIEpTT04gaW5wdXQiKQogICAgICAgIHJldHVybiBzZWxmLl9idWZmZXJbc2VsZi5fcG9zaXRpb25dCgogICAgZGVmIGV4cGVjdChzZWxmLCB0b2tlbjogc3RyKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYucGVlaygpICE9IHRva2VuOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhwZWN0ZWQge3Rva2VuIXJ9IGluIEpTT04gaW5wdXQiKQogICAgICAgIHNlbGYuX3Bvc2l0aW9uICs9IDEKCiAgICBkZWYgZGVjb2RlKHNlbGYpIC0+IEFueToKICAgICAgICBzZWxmLnNraXBfd2hpdGVzcGFjZSgpCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgdmFsdWUsIGVuZCA9IHNlbGYuX2RlY29kZXIucmF3X2RlY29kZShzZWxmLl9idWZmZXIsIHNlbGYuX3Bvc2l0aW9uKQogICAgICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3IgYXMgZXJyb3I6CiAgICAgICAgICAgICAgICBpZiBzZWxmLl9maWxsKCk6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgb3IgdHJ1bmNhdGVkIEpTT04gaW5wdXQiKSBmcm9tIGVycm9yCiAgICAgICAgICAgIHNlbGYuX3Bvc2l0aW9uID0gZW5kCiAgICAgICAgICAgIHJldHVybiB2YWx1ZQoKICAgIGRlZiBlbnN1cmVfZW5kKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5za2lwX3doaXRlc3BhY2UoKQogICAgICAgIGlmIHNlbGYuX3Bvc2l0aW9uIDwgbGVuKHNlbGYuX2J1ZmZlcikgb3Igc2VsZi5fZmlsbCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmV4cGVjdGVkIGNvbnRlbnQgYWZ0ZXIgdG9wLWxldmVsIEpTT04gb2JqZWN0IikKCgpkZWYgaXRlcl9zcGxpdF9yZWNvcmRzKAogICAgcGF0aDogUGF0aCwKICAgICosCiAgICBjaHVua19zaXplOiBpbnQgPSAxMDI0ICogMTAyNCwKKSAtPiBJdGVyYXRvclt0dXBsZVtzdHIsIGludCwgQW55XV06CiAgICAiIiJTdHJlYW0gcmVjb3JkcyBmcm9tIHRoZSBsYXJnZSB0b3AtbGV2ZWwgc3BsaXQgb2JqZWN0IHdpdGhvdXQgbG9hZGluZyBpdCBhbGwuIiIiCgogICAgcmVhZGVyID0gX1N0cmVhbWluZ0pzb25SZWFkZXIocGF0aCwgY2h1bmtfc2l6ZT1jaHVua19zaXplKQogICAgdHJ5OgogICAgICAgIHJlYWRlci5leHBlY3QoInsiKQogICAgICAgIGlmIHJlYWRlci5wZWVrKCkgPT0gIn0iOgogICAgICAgICAgICByZWFkZXIuZXhwZWN0KCJ9IikKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgc3BsaXRfbmFtZSA9IHJlYWRlci5kZWNvZGUoKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShzcGxpdF9uYW1lLCBzdHIpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic3BsaXQgcmVnaXN0cnkga2V5cyBtdXN0IGJlIHN0cmluZ3MiKQogICAgICAgICAgICByZWFkZXIuZXhwZWN0KCI6IikKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiWyIpCiAgICAgICAgICAgIGluZGV4ID0gMAogICAgICAgICAgICBpZiByZWFkZXIucGVlaygpICE9ICJdIjoKICAgICAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgc3BsaXRfbmFtZSwgaW5kZXgsIHJlYWRlci5kZWNvZGUoKQogICAgICAgICAgICAgICAgICAgIGluZGV4ICs9IDEKICAgICAgICAgICAgICAgICAgICBkZWxpbWl0ZXIgPSByZWFkZXIucGVlaygpCiAgICAgICAgICAgICAgICAgICAgaWYgZGVsaW1pdGVyID09ICJdIjoKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgICAgICByZWFkZXIuZXhwZWN0KCIsIikKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiXSIpCiAgICAgICAgICAgIGRlbGltaXRlciA9IHJlYWRlci5wZWVrKCkKICAgICAgICAgICAgaWYgZGVsaW1pdGVyID09ICJ9IjoKICAgICAgICAgICAgICAgIHJlYWRlci5leHBlY3QoIn0iKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiLCIpCiAgICAgICAgcmVhZGVyLmVuc3VyZV9lbmQoKQogICAgZmluYWxseToKICAgICAgICByZWFkZXIuY2xvc2UoKQoKCmNsYXNzIEVkaXRDTEVWUlBhaXIoU3RyaWN0TW9kZWwpOgogICAgcGFpcl9pZDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgc3VpdGU6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGVkaXRfZmFjdG9yOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEpCiAgICBiZWZvcmVfaW1hZ2U6IHN0cgogICAgYWZ0ZXJfaW1hZ2U6IHN0cgogICAgYmVmb3JlX3NjZW5lX2pzb246IHN0cgogICAgYWZ0ZXJfc2NlbmVfanNvbjogc3RyCiAgICBpbnN0YW5jZV9tYXNrc19iZWZvcmU6IHN0cgogICAgaW5zdGFuY2VfbWFza3NfYWZ0ZXI6IHN0cgoKICAgIEBtb2RlbF92YWxpZGF0b3IobW9kZT0iYWZ0ZXIiKQogICAgZGVmIHZhbGlkX3BhdGhzKHNlbGYpIC0+IFNlbGY6CiAgICAgICAgaWYgc2VsZi5zdWl0ZSBub3QgaW4gU1VJVEVTOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBFZGl0Q0xFVlIgc3VpdGU6IHtzZWxmLnN1aXRlfSIpCiAgICAgICAgZm9yIGZpZWxkIGluIFBBWUxPQURfRklFTERTOgogICAgICAgICAgICByYXcgPSBnZXRhdHRyKHNlbGYsIGZpZWxkKQogICAgICAgICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChyYXcpCiAgICAgICAgICAgIGlmIHBhdGguaXNfYWJzb2x1dGUoKSBvciAiLi4iIGluIHBhdGgucGFydHMgb3Igbm90IHBhdGgucGFydHM6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5zYWZlIEVkaXRDTEVWUiBwYXRoIGluIHtmaWVsZH0iKQogICAgICAgICAgICBpZiBwYXRoLnBhcnRzWzBdICE9IHNlbGYuc3VpdGU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRWRpdENMRVZSIHBhdGggaW4ge2ZpZWxkfSBkb2VzIG5vdCBtYXRjaCBzdWl0ZSIpCiAgICAgICAgcmV0dXJuIHNlbGYKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlLCBzbG90cz1UcnVlKQpjbGFzcyBFZGl0Q0xFVlJQYWlyUGF0aHM6CiAgICBwYWlyOiBFZGl0Q0xFVlJQYWlyCiAgICBiZWZvcmVfaW1hZ2U6IFBhdGgKICAgIGFmdGVyX2ltYWdlOiBQYXRoCiAgICBiZWZvcmVfc2NlbmVfanNvbjogUGF0aAogICAgYWZ0ZXJfc2NlbmVfanNvbjogUGF0aAogICAgaW5zdGFuY2VfbWFza3NfYmVmb3JlOiBQYXRoCiAgICBpbnN0YW5jZV9tYXNrc19hZnRlcjogUGF0aAoKCmRlZiBfaW1hZ2VfdGVuc29yKHBhdGg6IFBhdGgpIC0+IFRlbnNvcjoKICAgIHdpdGggSW1hZ2Uub3BlbihwYXRoKSBhcyBpbWFnZToKICAgICAgICByZ2IgPSBucC5hcnJheShpbWFnZS5jb252ZXJ0KCJSR0IiKSwgZHR5cGU9bnAudWludDgsIGNvcHk9VHJ1ZSkKICAgIHJldHVybiB0b3JjaC5mcm9tX251bXB5KHJnYikucGVybXV0ZSgyLCAwLCAxKS50byhkdHlwZT10b3JjaC5mbG9hdDMyKS5kaXZfKDI1NS4wKQoKCmNsYXNzIEVkaXRDTEVWUkRhdGFzZXQoU2VxdWVuY2VbRWRpdENMRVZSUGFpclBhdGhzXSk6CiAgICAiIiJTcGxpdC1zY29wZWQgcGF0aHMgcGx1cyBkZXRlcm1pbmlzdGljIFJHQiwgc2NlbmUsIGFuZCBtYXNrIGRlY29kZXJzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHBheWxvYWRfcm9vdDogUGF0aCwKICAgICAgICBzcGxpdF9yZWdpc3RyeTogUGF0aCwKICAgICAgICBzcGxpdDogc3RyLAogICAgICAgICosCiAgICAgICAgdmVyaWZ5X2ZpbGVzOiBib29sID0gVHJ1ZSwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLnBheWxvYWRfcm9vdCA9IHBheWxvYWRfcm9vdC5yZXNvbHZlKCkKICAgICAgICBpZiBwYXlsb2FkX3Jvb3QuaXNfc3ltbGluaygpIG9yIG5vdCBzZWxmLnBheWxvYWRfcm9vdC5pc19kaXIoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRWRpdENMRVZSIHBheWxvYWQgcm9vdCBtdXN0IGJlIGEgcmVndWxhciBkaXJlY3RvcnkiKQogICAgICAgIHNlbGYuc3BsaXRfcmVnaXN0cnkgPSBzcGxpdF9yZWdpc3RyeS5yZXNvbHZlKCkKICAgICAgICBpZiBzcGxpdF9yZWdpc3RyeS5pc19zeW1saW5rKCkgb3Igbm90IHNlbGYuc3BsaXRfcmVnaXN0cnkuaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFZGl0Q0xFVlIgc3BsaXQgcmVnaXN0cnkgbXVzdCBiZSBhIHJlZ3VsYXIgZmlsZSIpCiAgICAgICAgc2VsZi5zcGxpdCA9IHNwbGl0CiAgICAgICAgcGFpcnM6IGxpc3RbRWRpdENMRVZSUGFpcl0gPSBbXQogICAgICAgIGtub3duX3NwbGl0czogc2V0W3N0cl0gPSBzZXQoKQogICAgICAgIGZvciBzcGxpdF9uYW1lLCBpbmRleCwgcmF3IGluIGl0ZXJfc3BsaXRfcmVjb3JkcyhzZWxmLnNwbGl0X3JlZ2lzdHJ5KToKICAgICAgICAgICAga25vd25fc3BsaXRzLmFkZChzcGxpdF9uYW1lKQogICAgICAgICAgICBpZiBzcGxpdF9uYW1lICE9IHNwbGl0OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7c3BsaXRfbmFtZX1be2luZGV4fV0gaXMgbm90IGFuIG9iamVjdCIpCiAgICAgICAgICAgIGZpZWxkcyA9ICgicGFpcl9pZCIsICJzdWl0ZSIsICJlZGl0X2ZhY3RvciIsICpQQVlMT0FEX0ZJRUxEUykKICAgICAgICAgICAgc2VsZWN0ZWQgPSB7bmFtZTogcmF3LmdldChuYW1lKSBmb3IgbmFtZSBpbiBmaWVsZHN9CiAgICAgICAgICAgIHBhaXJzLmFwcGVuZChFZGl0Q0xFVlJQYWlyLm1vZGVsX3ZhbGlkYXRlKHNlbGVjdGVkKSkKICAgICAgICBpZiBzcGxpdCBub3QgaW4ga25vd25fc3BsaXRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBFZGl0Q0xFVlIgc3BsaXQ6IHtzcGxpdH0iKQogICAgICAgIGlmIG5vdCBwYWlyczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVkaXRDTEVWUiBzcGxpdCBpcyBlbXB0eToge3NwbGl0fSIpCiAgICAgICAgc2VsZi5fcGFpcnMgPSB0dXBsZShwYWlycykKICAgICAgICBpZiB2ZXJpZnlfZmlsZXM6CiAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShsZW4oc2VsZi5fcGFpcnMpKToKICAgICAgICAgICAgICAgIHNlbGYuX3Jlc29sdmUoaW5kZXgpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5fcGFpcnMpCgogICAgQG92ZXJsb2FkCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaW5kZXg6IGludCkgLT4gRWRpdENMRVZSUGFpclBhdGhzOiAuLi4KCiAgICBAb3ZlcmxvYWQKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogc2xpY2UpIC0+IFNlcXVlbmNlW0VkaXRDTEVWUlBhaXJQYXRoc106IC4uLgoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogaW50IHwgc2xpY2UpIC0+IEVkaXRDTEVWUlBhaXJQYXRocyB8IFNlcXVlbmNlW0VkaXRDTEVWUlBhaXJQYXRoc106CiAgICAgICAgaWYgaXNpbnN0YW5jZShpbmRleCwgc2xpY2UpOgogICAgICAgICAgICByZXR1cm4gdHVwbGUoc2VsZi5fcmVzb2x2ZShpdGVtKSBmb3IgaXRlbSBpbiByYW5nZSgqaW5kZXguaW5kaWNlcyhsZW4oc2VsZikpKSkKICAgICAgICBpZiBpbmRleCA8IDA6CiAgICAgICAgICAgIGluZGV4ICs9IGxlbihzZWxmLl9wYWlycykKICAgICAgICBpZiBpbmRleCA8IDAgb3IgaW5kZXggPj0gbGVuKHNlbGYuX3BhaXJzKToKICAgICAgICAgICAgcmFpc2UgSW5kZXhFcnJvcihpbmRleCkKICAgICAgICByZXR1cm4gc2VsZi5fcmVzb2x2ZShpbmRleCkKCiAgICBkZWYgX3BhdGgoc2VsZiwgcmF3OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgY2FuZGlkYXRlID0gc2VsZi5wYXlsb2FkX3Jvb3Quam9pbnBhdGgoKlB1cmVQb3NpeFBhdGgocmF3KS5wYXJ0cykKICAgICAgICBpZiBjYW5kaWRhdGUuaXNfc3ltbGluaygpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRWRpdENMRVZSIHBheWxvYWQgbWVtYmVyIGlzIGEgc3ltbGluazoge3Jhd30iKQogICAgICAgIHJlc29sdmVkID0gY2FuZGlkYXRlLnJlc29sdmUoKQogICAgICAgIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90byhzZWxmLnBheWxvYWRfcm9vdCkgb3Igbm90IHJlc29sdmVkLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVkaXRDTEVWUiBwYXlsb2FkIG1lbWJlciBpcyBtaXNzaW5nIG9yIGVzY2FwZXMgdGhlIHJvb3Q6IHtyYXd9IikKICAgICAgICByZXR1cm4gcmVzb2x2ZWQKCiAgICBkZWYgX3Jlc29sdmUoc2VsZiwgaW5kZXg6IGludCkgLT4gRWRpdENMRVZSUGFpclBhdGhzOgogICAgICAgIHBhaXIgPSBzZWxmLl9wYWlyc1tpbmRleF0KICAgICAgICByZXR1cm4gRWRpdENMRVZSUGFpclBhdGhzKAogICAgICAgICAgICBwYWlyPXBhaXIsCiAgICAgICAgICAgIGJlZm9yZV9pbWFnZT1zZWxmLl9wYXRoKHBhaXIuYmVmb3JlX2ltYWdlKSwKICAgICAgICAgICAgYWZ0ZXJfaW1hZ2U9c2VsZi5fcGF0aChwYWlyLmFmdGVyX2ltYWdlKSwKICAgICAgICAgICAgYmVmb3JlX3NjZW5lX2pzb249c2VsZi5fcGF0aChwYWlyLmJlZm9yZV9zY2VuZV9qc29uKSwKICAgICAgICAgICAgYWZ0ZXJfc2NlbmVfanNvbj1zZWxmLl9wYXRoKHBhaXIuYWZ0ZXJfc2NlbmVfanNvbiksCiAgICAgICAgICAgIGluc3RhbmNlX21hc2tzX2JlZm9yZT1zZWxmLl9wYXRoKHBhaXIuaW5zdGFuY2VfbWFza3NfYmVmb3JlKSwKICAgICAgICAgICAgaW5zdGFuY2VfbWFza3NfYWZ0ZXI9c2VsZi5fcGF0aChwYWlyLmluc3RhbmNlX21hc2tzX2FmdGVyKSwKICAgICAgICApCgogICAgZGVmIGxvYWRfcmdiX3BhaXIoc2VsZiwgaW5kZXg6IGludCkgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3JdOgogICAgICAgIHBhdGhzID0gc2VsZltpbmRleF0KICAgICAgICByZXR1cm4gX2ltYWdlX3RlbnNvcihwYXRocy5iZWZvcmVfaW1hZ2UpLCBfaW1hZ2VfdGVuc29yKHBhdGhzLmFmdGVyX2ltYWdlKQoKICAgIGRlZiBsb2FkX3NjZW5lcyhzZWxmLCBpbmRleDogaW50KSAtPiB0dXBsZVtkaWN0W3N0ciwgQW55XSwgZGljdFtzdHIsIEFueV1dOgogICAgICAgIHBhdGhzID0gc2VsZltpbmRleF0KICAgICAgICB2YWx1ZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dID0gW10KICAgICAgICBmb3IgcGF0aCBpbiAocGF0aHMuYmVmb3JlX3NjZW5lX2pzb24sIHBhdGhzLmFmdGVyX3NjZW5lX2pzb24pOgogICAgICAgICAgICB2YWx1ZSA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFZGl0Q0xFVlIgc2NlbmUgbXVzdCBiZSBhbiBvYmplY3Q6IHtwYXRofSIpCiAgICAgICAgICAgIHZhbHVlcy5hcHBlbmQodmFsdWUpCiAgICAgICAgcmV0dXJuIHZhbHVlc1swXSwgdmFsdWVzWzFdCgogICAgZGVmIGxvYWRfbWFza3Moc2VsZiwgaW5kZXg6IGludCkgLT4gdHVwbGVbZGljdFtzdHIsIG5wLm5kYXJyYXlbQW55LCBBbnldXSwgLi4uXToKICAgICAgICBwYXRocyA9IHNlbGZbaW5kZXhdCiAgICAgICAgcmVzdWx0czogbGlzdFtkaWN0W3N0ciwgbnAubmRhcnJheVtBbnksIEFueV1dXSA9IFtdCiAgICAgICAgZm9yIHBhdGggaW4gKHBhdGhzLmluc3RhbmNlX21hc2tzX2JlZm9yZSwgcGF0aHMuaW5zdGFuY2VfbWFza3NfYWZ0ZXIpOgogICAgICAgICAgICB3aXRoIG5wLmxvYWQocGF0aCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBhcmNoaXZlOgogICAgICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoe25hbWU6IG5wLmFycmF5KGFyY2hpdmVbbmFtZV0sIGNvcHk9VHJ1ZSkgZm9yIG5hbWUgaW4gYXJjaGl2ZS5maWxlc30pCiAgICAgICAgcmV0dXJuIHR1cGxlKHJlc3VsdHMpCg==', 'olympus/data/ingestion.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGh0dHAuY2xpZW50CmltcG9ydCBpcGFkZHJlc3MKaW1wb3J0IGpzb24KaW1wb3J0IHNvY2tldAppbXBvcnQgc3NsCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQpmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgU3BsaXRSZXN1bHQsIHVybHNwbGl0Cgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmNvcmUuc2VjdXJpdHkgaW1wb3J0IFBlcm1pc3Npb25Qb2xpY3kKCl9TQUZFX0NPTlRFTlRfVFlQRVMgPSBmcm96ZW5zZXQoCiAgICB7ImFwcGxpY2F0aW9uL2pzb24iLCAiYXBwbGljYXRpb24veG1sIiwgInRleHQvY3N2IiwgInRleHQvcGxhaW4iLCAidGV4dC94bWwifQopCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgX1Jlc29sdmVkRW5kcG9pbnQ6CiAgICBmYW1pbHk6IGludAogICAgc29ja3R5cGU6IGludAogICAgcHJvdG86IGludAogICAgc29ja2FkZHI6IEFueQogICAgaXA6IHN0cgoKCmNsYXNzIF9QaW5uZWRIVFRQU0Nvbm5lY3Rpb24oaHR0cC5jbGllbnQuSFRUUFNDb25uZWN0aW9uKToKICAgICIiIkhUVFBTIGNvbm5lY3Rpb24gd2hvc2UgVENQIHBlZXIgaXMgZml4ZWQgYmVmb3JlIFRMUyBzdGFydHMuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaG9zdG5hbWU6IHN0ciwKICAgICAgICBwb3J0OiBpbnQsCiAgICAgICAgZW5kcG9pbnQ6IF9SZXNvbHZlZEVuZHBvaW50LAogICAgICAgICosCiAgICAgICAgdGltZW91dDogZmxvYXQsCiAgICApIC0+IE5vbmU6CiAgICAgICAgY29udGV4dCA9IHNzbC5jcmVhdGVfZGVmYXVsdF9jb250ZXh0KCkKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGhvc3RuYW1lLCBwb3J0PXBvcnQsIHRpbWVvdXQ9dGltZW91dCwgY29udGV4dD1jb250ZXh0KQogICAgICAgIHNlbGYuX2VuZHBvaW50ID0gZW5kcG9pbnQKICAgICAgICBzZWxmLl9jb25uZWN0X3RpbWVvdXQgPSB0aW1lb3V0CiAgICAgICAgc2VsZi5fdGxzX2NvbnRleHQgPSBjb250ZXh0CgogICAgZGVmIGNvbm5lY3Qoc2VsZikgLT4gTm9uZToKICAgICAgICBzb2NrID0gc29ja2V0LnNvY2tldCgKICAgICAgICAgICAgc2VsZi5fZW5kcG9pbnQuZmFtaWx5LAogICAgICAgICAgICBzZWxmLl9lbmRwb2ludC5zb2NrdHlwZSwKICAgICAgICAgICAgc2VsZi5fZW5kcG9pbnQucHJvdG8sCiAgICAgICAgKQogICAgICAgIHNvY2suc2V0dGltZW91dChzZWxmLl9jb25uZWN0X3RpbWVvdXQpCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIENvbm5lY3QgZGlyZWN0bHkgdG8gdGhlIGFscmVhZHktdmFsaWRhdGVkIHNvY2thZGRyLiBUaGlzIGRlbGliZXJhdGVseSBhdm9pZHMKICAgICAgICAgICAgIyBzb2NrZXQuY3JlYXRlX2Nvbm5lY3Rpb24oKSwgd2hpY2ggd291bGQgcGVyZm9ybSBhIHNlY29uZCBob3N0bmFtZSByZXNvbHV0aW9uLgogICAgICAgICAgICBzb2NrLmNvbm5lY3Qoc2VsZi5fZW5kcG9pbnQuc29ja2FkZHIpCiAgICAgICAgICAgIHNlbGYuc29jayA9IHNlbGYuX3Rsc19jb250ZXh0LndyYXBfc29ja2V0KHNvY2ssIHNlcnZlcl9ob3N0bmFtZT1zZWxmLmhvc3QpCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgIHNvY2suY2xvc2UoKQogICAgICAgICAgICByYWlzZQoKCmNsYXNzIEluZ2VzdGVkRG9jdW1lbnQoU3RyaWN0TW9kZWwpOgogICAgaWRlbnRpZmllcjogc3RyCiAgICB0ZXh0OiBzdHIKICAgIGxhbmd1YWdlOiBzdHIgPSAidW5rbm93biIKICAgIGNodW5rczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBtZXRhZGF0YTogZGljdFtzdHIsIHN0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKCgpjbGFzcyBJbmdlc3Rpb25QaXBlbGluZToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHBvbGljeTogUGVybWlzc2lvblBvbGljeSB8IE5vbmUgPSBOb25lLAogICAgICAgICosCiAgICAgICAgbWF4X3Jlc3BvbnNlX2J5dGVzOiBpbnQgPSAxXzAwMF8wMDAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgaWYgbWF4X3Jlc3BvbnNlX2J5dGVzIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X3Jlc3BvbnNlX2J5dGVzIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIHNlbGYucG9saWN5ID0gcG9saWN5IG9yIFBlcm1pc3Npb25Qb2xpY3koKQogICAgICAgIHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzID0gbWF4X3Jlc3BvbnNlX2J5dGVzCgogICAgZGVmIGluZ2VzdF9maWxlKHNlbGYsIHBhdGg6IFBhdGgsIGNodW5rX3NpemU6IGludCA9IDEyMCkgLT4gSW5nZXN0ZWREb2N1bWVudDoKICAgICAgICB0ZXh0ID0gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICByZXR1cm4gc2VsZi5fbm9ybWFsaXplKHBhdGgubmFtZSwgdGV4dCwgY2h1bmtfc2l6ZSkKCiAgICBkZWYgaW5nZXN0X2h0dHAoc2VsZiwgdXJsOiBzdHIsIGNodW5rX3NpemU6IGludCA9IDEyMCkgLT4gSW5nZXN0ZWREb2N1bWVudDoKICAgICAgICBpZiBub3Qgc2VsZi5wb2xpY3kuYWxsb3dfbmV0d29yazoKICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKCJuZXR3b3JrIGluZ2VzdGlvbiBpcyBkaXNhYmxlZCIpCiAgICAgICAgcGFyc2VkID0gdXJsc3BsaXQodXJsKQogICAgICAgIGhvc3RuYW1lID0gKHBhcnNlZC5ob3N0bmFtZSBvciAiIikubG93ZXIoKS5yc3RyaXAoIi4iKQogICAgICAgIGlmIHBhcnNlZC5zY2hlbWUgIT0gImh0dHBzIiBvciBub3QgaG9zdG5hbWUgb3IgcGFyc2VkLnVzZXJuYW1lIG9yIHBhcnNlZC5wYXNzd29yZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5nZXN0aW9uIFVSTHMgbXVzdCBiZSBjcmVkZW50aWFsLWZyZWUgSFRUUFMgVVJMcyIpCiAgICAgICAgYWxsb3dlZCA9IHtkb21haW4ubG93ZXIoKS5yc3RyaXAoIi4iKSBmb3IgZG9tYWluIGluIHNlbGYucG9saWN5LmFsbG93X2h0dHBfZG9tYWluc30KICAgICAgICBpZiBob3N0bmFtZSBub3QgaW4gYWxsb3dlZDoKICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKGYiaG9zdCBpcyBub3QgYWxsb3dsaXN0ZWQ6IHtob3N0bmFtZX0iKQogICAgICAgIHBvcnQgPSBwYXJzZWQucG9ydCBvciA0NDMKICAgICAgICBlbmRwb2ludHMgPSBzZWxmLl9yZXNvbHZlX3B1YmxpY19lbmRwb2ludHMoaG9zdG5hbWUsIHBvcnQpCiAgICAgICAgcGF5bG9hZCwgY2hhcnNldCA9IHNlbGYuX2ZldGNoX2h0dHBzKHBhcnNlZCwgaG9zdG5hbWUsIHBvcnQsIGVuZHBvaW50cykKICAgICAgICB0ZXh0ID0gcGF5bG9hZC5kZWNvZGUoY2hhcnNldCkKICAgICAgICByZXR1cm4gc2VsZi5fbm9ybWFsaXplKHVybCwgdGV4dCwgY2h1bmtfc2l6ZSkKCiAgICBkZWYgX2ZldGNoX2h0dHBzKAogICAgICAgIHNlbGYsCiAgICAgICAgcGFyc2VkOiBTcGxpdFJlc3VsdCwKICAgICAgICBob3N0bmFtZTogc3RyLAogICAgICAgIHBvcnQ6IGludCwKICAgICAgICBlbmRwb2ludHM6IHR1cGxlW19SZXNvbHZlZEVuZHBvaW50LCAuLi5dLAogICAgKSAtPiB0dXBsZVtieXRlcywgc3RyXToKICAgICAgICB0YXJnZXQgPSBwYXJzZWQucGF0aCBvciAiLyIKICAgICAgICBpZiBwYXJzZWQucXVlcnk6CiAgICAgICAgICAgIHRhcmdldCA9IGYie3RhcmdldH0/e3BhcnNlZC5xdWVyeX0iCiAgICAgICAgaGVhZGVycyA9IHsKICAgICAgICAgICAgIkFjY2VwdCI6ICIsICIuam9pbihzb3J0ZWQoX1NBRkVfQ09OVEVOVF9UWVBFUykpLAogICAgICAgICAgICAiVXNlci1BZ2VudCI6ICJPbHltcHVzLzAuMSIsCiAgICAgICAgfQogICAgICAgIGxhc3RfZXJyb3I6IEV4Y2VwdGlvbiB8IE5vbmUgPSBOb25lCiAgICAgICAgZm9yIGVuZHBvaW50IGluIGVuZHBvaW50czoKICAgICAgICAgICAgY29ubmVjdGlvbiA9IF9QaW5uZWRIVFRQU0Nvbm5lY3Rpb24oaG9zdG5hbWUsIHBvcnQsIGVuZHBvaW50LCB0aW1lb3V0PTUuMCkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY29ubmVjdGlvbi5yZXF1ZXN0KCJHRVQiLCB0YXJnZXQsIGhlYWRlcnM9aGVhZGVycykKICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gY29ubmVjdGlvbi5nZXRyZXNwb25zZSgpCiAgICAgICAgICAgICAgICBpZiAzMDAgPD0gcmVzcG9uc2Uuc3RhdHVzIDwgNDAwOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAgcmVkaXJlY3RzIGFyZSBkaXNhYmxlZCBmb3IgaW5nZXN0aW9uIikKICAgICAgICAgICAgICAgIGlmIHJlc3BvbnNlLnN0YXR1cyA8IDIwMCBvciByZXNwb25zZS5zdGF0dXMgPj0gMzAwOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJIVFRQIGluZ2VzdGlvbiByZXF1ZXN0IGZhaWxlZCB3aXRoIHN0YXR1cyB7cmVzcG9uc2Uuc3RhdHVzfSIpCiAgICAgICAgICAgICAgICBjb250ZW50X3R5cGUgPSByZXNwb25zZS5oZWFkZXJzLmdldF9jb250ZW50X3R5cGUoKS5sb3dlcigpCiAgICAgICAgICAgICAgICBpZiBjb250ZW50X3R5cGUgbm90IGluIF9TQUZFX0NPTlRFTlRfVFlQRVM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuc3VwcG9ydGVkIEhUVFAgY29udGVudCB0eXBlOiB7Y29udGVudF90eXBlfSIpCiAgICAgICAgICAgICAgICBsZW5ndGggPSByZXNwb25zZS5oZWFkZXJzLmdldCgiQ29udGVudC1MZW5ndGgiKQogICAgICAgICAgICAgICAgaWYgbGVuZ3RoIGlzIG5vdCBOb25lIGFuZCBpbnQobGVuZ3RoKSA+IHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAgcmVzcG9uc2UgZXhjZWVkcyB0aGUgY29uZmlndXJlZCBieXRlIGxpbWl0IikKICAgICAgICAgICAgICAgIHBheWxvYWQgPSByZXNwb25zZS5yZWFkKHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzICsgMSkKICAgICAgICAgICAgICAgIGlmIGxlbihwYXlsb2FkKSA+IHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAgcmVzcG9uc2UgZXhjZWVkcyB0aGUgY29uZmlndXJlZCBieXRlIGxpbWl0IikKICAgICAgICAgICAgICAgIGNoYXJzZXQgPSByZXNwb25zZS5oZWFkZXJzLmdldF9jb250ZW50X2NoYXJzZXQoInV0Zi04IikKICAgICAgICAgICAgICAgIHJldHVybiBwYXlsb2FkLCBjaGFyc2V0CiAgICAgICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgaHR0cC5jbGllbnQuSFRUUEV4Y2VwdGlvbikgYXMgZXhjOgogICAgICAgICAgICAgICAgbGFzdF9lcnJvciA9IGV4YwogICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgY29ubmVjdGlvbi5jbG9zZSgpCiAgICAgICAgaWYgbGFzdF9lcnJvciBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuYWJsZSB0byBjb25uZWN0IHRvIGluZ2VzdGlvbiBob3N0OiB7aG9zdG5hbWV9IikgZnJvbSBsYXN0X2Vycm9yCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuYWJsZSB0byBjb25uZWN0IHRvIGluZ2VzdGlvbiBob3N0OiB7aG9zdG5hbWV9IikKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3Jlc29sdmVfcHVibGljX2VuZHBvaW50cyhob3N0bmFtZTogc3RyLCBwb3J0OiBpbnQpIC0+IHR1cGxlW19SZXNvbHZlZEVuZHBvaW50LCAuLi5dOgogICAgICAgIHRyeToKICAgICAgICAgICAgYWRkcmVzc2VzID0gc29ja2V0LmdldGFkZHJpbmZvKGhvc3RuYW1lLCBwb3J0LCB0eXBlPXNvY2tldC5TT0NLX1NUUkVBTSkKICAgICAgICBleGNlcHQgc29ja2V0LmdhaWVycm9yIGFzIGV4YzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuYWJsZSB0byByZXNvbHZlIGluZ2VzdGlvbiBob3N0OiB7aG9zdG5hbWV9IikgZnJvbSBleGMKICAgICAgICBpZiBub3QgYWRkcmVzc2VzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5hYmxlIHRvIHJlc29sdmUgaW5nZXN0aW9uIGhvc3Q6IHtob3N0bmFtZX0iKQoKICAgICAgICBlbmRwb2ludHM6IGxpc3RbX1Jlc29sdmVkRW5kcG9pbnRdID0gW10KICAgICAgICBmb3IgZmFtaWx5LCBzb2NrdHlwZSwgcHJvdG8sIF9jYW5vbmljYWxfbmFtZSwgc29ja2FkZHIgaW4gYWRkcmVzc2VzOgogICAgICAgICAgICBpcCA9IGlwYWRkcmVzcy5pcF9hZGRyZXNzKHNvY2thZGRyWzBdKQogICAgICAgICAgICBpZiBub3QgaXAuaXNfZ2xvYmFsOgogICAgICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKGYiaW5nZXN0aW9uIGhvc3QgcmVzb2x2ZXMgdG8gYSBub24tcHVibGljIGFkZHJlc3M6IHtpcH0iKQogICAgICAgICAgICBlbmRwb2ludHMuYXBwZW5kKAogICAgICAgICAgICAgICAgX1Jlc29sdmVkRW5kcG9pbnQoCiAgICAgICAgICAgICAgICAgICAgZmFtaWx5PWZhbWlseSwKICAgICAgICAgICAgICAgICAgICBzb2NrdHlwZT1zb2NrdHlwZSwKICAgICAgICAgICAgICAgICAgICBwcm90bz1wcm90bywKICAgICAgICAgICAgICAgICAgICBzb2NrYWRkcj1zb2NrYWRkciwKICAgICAgICAgICAgICAgICAgICBpcD1zdHIoaXApLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHR1cGxlKGVuZHBvaW50cykKCiAgICBkZWYgX25vcm1hbGl6ZShzZWxmLCBpZGVudGlmaWVyOiBzdHIsIHRleHQ6IHN0ciwgY2h1bmtfc2l6ZTogaW50KSAtPiBJbmdlc3RlZERvY3VtZW50OgogICAgICAgIG5vcm1hbGl6ZWQgPSAiICIuam9pbih0ZXh0LnNwbGl0KCkpCiAgICAgICAgY2h1bmtzID0gWwogICAgICAgICAgICBub3JtYWxpemVkW2luZGV4IDogaW5kZXggKyBjaHVua19zaXplXQogICAgICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2UoMCwgbGVuKG5vcm1hbGl6ZWQpLCBjaHVua19zaXplKQogICAgICAgIF0KICAgICAgICByZXR1cm4gSW5nZXN0ZWREb2N1bWVudCgKICAgICAgICAgICAgaWRlbnRpZmllcj1pZGVudGlmaWVyLAogICAgICAgICAgICB0ZXh0PW5vcm1hbGl6ZWQsCiAgICAgICAgICAgIGxhbmd1YWdlPSJlbiIgaWYgbm9ybWFsaXplZC5pc2FzY2lpKCkgZWxzZSAibWl4ZWQiLAogICAgICAgICAgICBjaHVua3M9Y2h1bmtzLAogICAgICAgICAgICBtZXRhZGF0YT17InNoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KG5vcm1hbGl6ZWQuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKX0sCiAgICAgICAgKQoKICAgIGRlZiBzaGFyZChzZWxmLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0sIG91dHB1dF9wYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIG91dHB1dF9wYXRoLndyaXRlX3RleHQoCiAgICAgICAgICAgIGpzb24uZHVtcHMoW2RvY3VtZW50Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpIGZvciBkb2N1bWVudCBpbiBkb2N1bWVudHNdLCBpbmRlbnQ9MiksCiAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQo=', 'olympus/data/manifests.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQsIG1vZGVsX3ZhbGlkYXRvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBEYXRhc2V0U3BsaXQoU3RyaWN0TW9kZWwpOgogICAgdHJhaW46IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICB2YWxpZGF0aW9uOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgdGVzdDogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiB2YWxpZGF0ZV90b3RhbChzZWxmKSAtPiBEYXRhc2V0U3BsaXQ6CiAgICAgICAgaWYgYWJzKChzZWxmLnRyYWluICsgc2VsZi52YWxpZGF0aW9uICsgc2VsZi50ZXN0KSAtIDEuMCkgPiAxZS05OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkYXRhc2V0IHNwbGl0IGZyYWN0aW9ucyBtdXN0IHN1bSB0byAxLjAiKQogICAgICAgIHJldHVybiBzZWxmCgoKY2xhc3MgRGF0YXNldE1hbmlmZXN0KFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgdmVyc2lvbjogc3RyCiAgICBzcGxpdDogRGF0YXNldFNwbGl0CiAgICBzb3VyY2VzOiBsaXN0W3N0cl0KICAgIGZpbHRlcnM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZGVkdXBsaWNhdGlvbjogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICB3ZWlnaHRzOiBkaWN0W3N0ciwgZmxvYXRdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBsaWNlbnNlczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBjb250YW1pbmF0aW9uX3J1bGVzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHRyYW5zZm9ybWF0aW9uczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCg==', 'olympus/data/olympus_dataset.py': '...', 'olympus/data/oracle_fusion_v4.py': 'IiIiRnJlc2ggdjQgbWVjaGFuaXNtIGRhdGE7IHRhcmdldC1mcmVlIGRvbWFpbiBtZXRhZGF0YSBhbmQgY2xlYW4gZHluYW1pY3MuCgpUaGlzIGlzIGEgY29udHJvbGxlZCBzeW50aGV0aWMgZGlhZ25vc3RpYywgbm90IGEgbmF0dXJhbC1kYXRhIGJlbmNobWFyay4KT25seSB0aGUgZXhwbGljaXRseSBwcml2aWxlZ2VkIG9yYWNsZSBjb25zdW1lcyBjb3JydXB0aW9uIG1ldGFkYXRhIG9yIGxhYmVscy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yCgoKZGVmIGR5bmFtaWNzKHN0YXRlOiBUZW5zb3IsIGFjdGlvbjogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBwYWRkZWQgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLnBhZChhY3Rpb24sICgwLCAyKSkKICAgIHJldHVybiAwLjgyICogc3RhdGUgKyAwLjE4ICogc3RhdGUucm9sbCgxLCAtMSkudGFuaCgpICsgMC4xMiAqIHBhZGRlZAoKCmRlZiBnZW5lcmF0ZShzZWVkOiBpbnQsIGNvdW50OiBpbnQsIHJlZ2ltZTogc3RyKSAtPiBkaWN0W3N0ciwgVGVuc29yXToKICAgIGlmIHJlZ2ltZSBub3QgaW4geyJtaXhlZCIsICJtaXNzaW5nX29ubHkiLCAiY2xlYW4iLCAiY29ycnVwdGVkIiwgIm9vZCJ9IG9yIGNvdW50IDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIHY0IHNwbGl0IikKICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKQogICAgbGFiZWxzID0gdG9yY2gucmFuZGludCgwLCA0LCAoY291bnQsKSwgZ2VuZXJhdG9yPWcpCiAgICAjIEluZGVwZW5kZW50IG1ldGFkYXRhOiB0aGUgdGFyZ2V0IGlzIG5ldmVyIHVzZWQgdG8gY29uc3RydWN0IGRvbWFpbiBvciBsZWdhbGl0eS4KICAgIGRvbWFpbnMgPSB0b3JjaC5yYW5kaW50KDAsIDMsIChjb3VudCwpLCBnZW5lcmF0b3I9ZykKICAgIGNsZWFuX2luaXRpYWwgPSB0b3JjaC5yYW5kbihjb3VudCwgNSwgZ2VuZXJhdG9yPWcpICogMC4xNQogICAgY2xlYW5faW5pdGlhbFt0b3JjaC5hcmFuZ2UoY291bnQpLCBsYWJlbHNdICs9IDEuNQogICAgYWN0aW9ucyA9IHRvcmNoLnJhbmQoY291bnQsIDQsIDMsIGdlbmVyYXRvcj1nKSAqIDIgLSAxCiAgICBjbGVhbl9zdGF0ZXMgPSBbY2xlYW5faW5pdGlhbF0KICAgIGZ1dHVyZSA9IFtdCiAgICBmb3IgdCBpbiByYW5nZSg0KToKICAgICAgICBuZXh0X3N0YXRlID0gZHluYW1pY3MoY2xlYW5fc3RhdGVzWy0xXSwgYWN0aW9uc1s6LCB0XSkKICAgICAgICBmdXR1cmUuYXBwZW5kKG5leHRfc3RhdGUpCiAgICAgICAgaWYgdCA8IDM6CiAgICAgICAgICAgIGNsZWFuX3N0YXRlcy5hcHBlbmQobmV4dF9zdGF0ZSkKICAgIGNsZWFuID0gdG9yY2guc3RhY2soY2xlYW5fc3RhdGVzLCAxKQogICAgbmV4dF9zdGF0ZXMgPSB0b3JjaC5zdGFjayhmdXR1cmUsIDEpCgogICAgIyBJbmRlcGVuZGVudCBtb2RhbGl0eSBjb3JydXB0aW9uLiBNaXhlZCB0cmFpbiBpbmNsdWRlcyBhbGwgdGVzdCBpbnRlcnZlbnRpb24KICAgICMgY2F0ZWdvcmllczsgT09EIHJlc2VydmVzIGJpYXMvcmV2ZXJzYWwgaW50ZXJ2ZW50aW9ucywgbmV2ZXIgc2VtYW50aWMgcmVtYXBwaW5nLgogICAgc3RhdHVzID0gdG9yY2guemVyb3MoY291bnQsIDMsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBicmFuY2ggPSB0b3JjaC5yYW5kaW50KDAsIDQsIChjb3VudCwpLCBnZW5lcmF0b3I9ZykKICAgIGZpcnN0ID0gdG9yY2gucmFuZGludCgwLCAzLCAoY291bnQsKSwgZ2VuZXJhdG9yPWcpCiAgICBzZWNvbmQgPSAoZmlyc3QgKyB0b3JjaC5yYW5kaW50KDEsIDMsIChjb3VudCwpLCBnZW5lcmF0b3I9ZykpICUgMwogICAgaWYgcmVnaW1lID09ICJtaXNzaW5nX29ubHkiOgogICAgICAgIG1pc3NpbmdfYnJhbmNoID0gKGJyYW5jaCA9PSAxKSB8IChicmFuY2ggPT0gMykKICAgICAgICBzdGF0dXNbbWlzc2luZ19icmFuY2gsIGZpcnN0W21pc3NpbmdfYnJhbmNoXV0gPSAxCiAgICBlbGlmIHJlZ2ltZSA9PSAibWl4ZWQiOgogICAgICAgIHN0YXR1c1ticmFuY2ggPT0gMSwgZmlyc3RbYnJhbmNoID09IDFdXSA9IDEKICAgICAgICBzdGF0dXNbYnJhbmNoID09IDIsIGZpcnN0W2JyYW5jaCA9PSAyXV0gPSAyCiAgICAgICAgc3RhdHVzW2JyYW5jaCA9PSAzLCBmaXJzdFticmFuY2ggPT0gM11dID0gMQogICAgICAgIHN0YXR1c1ticmFuY2ggPT0gMywgc2Vjb25kW2JyYW5jaCA9PSAzXV0gPSAyCiAgICBlbGlmIHJlZ2ltZSBpbiB7ImNvcnJ1cHRlZCIsICJvb2QifToKICAgICAgICBzdGF0dXNbdG9yY2guYXJhbmdlKGNvdW50KSwgZmlyc3RdID0gMQogICAgICAgIHN0YXR1c1t0b3JjaC5hcmFuZ2UoY291bnQpLCBzZWNvbmRdID0gMgoKICAgIHRva2VucyA9IHRvcmNoLnJhbmRpbnQoOCwgMzEsIChjb3VudCwgOCksIGdlbmVyYXRvcj1nKQogICAgdG9rZW5zWzosIDBdID0gbGFiZWxzICsgMQogICAgdG9rZW5zWzosIDFdID0gNSArIGxhYmVscyAlIDIKICAgIHRva2Vuc1s6LCAyXSA9IDcgKyBsYWJlbHMgLy8gMgogICAgaW1hZ2UgPSB0b3JjaC5yYW5kKGNvdW50LCAzLCA4LCA4LCBnZW5lcmF0b3I9ZykgKiAwLjE1CiAgICBmb3IgaSBpbiByYW5nZShjb3VudCk6CiAgICAgICAgeSA9IGludChsYWJlbHNbaV0pCiAgICAgICAgaW1hZ2VbaSwgeSAlIDMsICh5IC8vIDIpICogNCA6ICh5IC8vIDIgKyAxKSAqIDQsICh5ICUgMikgKiA0IDogKHkgJSAyICsgMSkgKiA0XSArPSAwLjc1CiAgICBzdGF0ZXMgPSBjbGVhbiArIHRvcmNoLnJhbmRuKGNsZWFuLnNoYXBlLCBnZW5lcmF0b3I9ZykgKiAwLjEyCiAgICBmb3IgbW9kYWxpdHksIHggaW4gZW51bWVyYXRlKCh0b2tlbnMsIGltYWdlLCBzdGF0ZXMpKToKICAgICAgICBtaXNzaW5nID0gc3RhdHVzWzosIG1vZGFsaXR5XSA9PSAxCiAgICAgICAgZGFtYWdlZCA9IHN0YXR1c1s6LCBtb2RhbGl0eV0gPT0gMgogICAgICAgIHhbbWlzc2luZ10gPSAwCiAgICAgICAgbiA9IGludChkYW1hZ2VkLnN1bSgpKQogICAgICAgIGlmIG5vdCBuOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG1vZGFsaXR5ID09IDA6CiAgICAgICAgICAgIGlmIHJlZ2ltZSA9PSAib29kIjoKICAgICAgICAgICAgICAgIHhbZGFtYWdlZCwgOjNdID0geFtkYW1hZ2VkLCA6M10uZmxpcCgtMSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHhbZGFtYWdlZCwgOjNdID0gdG9yY2gucmFuZGludCgxLCAxMiwgKG4sIDMpLCBnZW5lcmF0b3I9ZykKICAgICAgICBlbGlmIG1vZGFsaXR5ID09IDE6CiAgICAgICAgICAgIGlmIHJlZ2ltZSA9PSAib29kIjoKICAgICAgICAgICAgICAgIHhbZGFtYWdlZF0gPSAoMS4wIC0geFtkYW1hZ2VkXSkuY2xhbXAoMCwgMSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHhbZGFtYWdlZF0gPSB0b3JjaC5yYW5kKG4sIDMsIDgsIDgsIGdlbmVyYXRvcj1nKQogICAgICAgIGVsaWYgcmVnaW1lID09ICJvb2QiOgogICAgICAgICAgICB4W2RhbWFnZWRdICs9IDEuNQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHhbZGFtYWdlZF0gPSB0b3JjaC5yYW5kbihuLCA0LCA1LCBnZW5lcmF0b3I9ZykKICAgIHF1YWxpdHkgPSB0b3JjaC50ZW5zb3IoWzEuMCwgMC4wLCAwLjI1XSlbc3RhdHVzXQogICAgIyBBdGxhcyBjb25zdW1lcyBCT1RIIHRleHQgYW5kIGltYWdlLiBJdHMgY29ycnVwdGlvbiBwcm94eSBleHBsaWNpdGx5IHJlZmxlY3RzCiAgICAjIGJvdGgsIHJhdGhlciB0aGFuIGluY29ycmVjdGx5IGxhYmVsaW5nIGl0IGFuIGltYWdlLW9ubHkgZXhwZXJ0LgogICAgY29tcG9uZW50X3F1YWxpdHkgPSB0b3JjaC5zdGFjaygKICAgICAgICBbCiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0sCiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0sCiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0sCiAgICAgICAgICAgIChxdWFsaXR5WzosIDBdICsgcXVhbGl0eVs6LCAxXSkgLyAyLAogICAgICAgICAgICBxdWFsaXR5WzosIDJdLAogICAgICAgIF0sCiAgICAgICAgMSwKICAgICkKICAgIHJldHVybiB7CiAgICAgICAgInRva2VucyI6IHRva2VucywKICAgICAgICAiaW1hZ2UiOiBpbWFnZSwKICAgICAgICAic3RhdGVzIjogc3RhdGVzLAogICAgICAgICJhY3Rpb25zIjogYWN0aW9ucywKICAgICAgICAiZG9tYWluX2lkcyI6IGRvbWFpbnMsCiAgICAgICAgImFsbG93ZWRfdHJhbnNpdGlvbnMiOiB0b3JjaC5vbmVzKGNvdW50LCA0LCBkdHlwZT10b3JjaC5ib29sKSwKICAgICAgICAicmlza19zY29yZSI6IHRvcmNoLnplcm9zKGNvdW50KSwKICAgICAgICAiZGVjaXNpb25fdGFyZ2V0cyI6IGxhYmVscywKICAgICAgICAidHJhbnNpdGlvbl90YXJnZXRzIjogbGFiZWxzICUgMywKICAgICAgICAibmV4dF9zdGF0ZV90YXJnZXRzIjogbmV4dF9zdGF0ZXMsCiAgICAgICAgImNvbXBvbmVudF9xdWFsaXR5IjogY29tcG9uZW50X3F1YWxpdHksCiAgICAgICAgImNvcnJ1cHRpb25fc3RhdHVzIjogc3RhdHVzLAogICAgICAgICJzb3VyY2VfaWRzIjogdG9yY2guYXJhbmdlKGNvdW50LCBkdHlwZT10b3JjaC5pbnQ2NCkgKyBzZWVkICogMTAwMDAwLAogICAgfQoKCmRlZiBzdWJzZXQoYmF0Y2g6IGRpY3Rbc3RyLCBUZW5zb3JdLCBpbmRpY2VzOiBUZW5zb3IpIC0+IGRpY3Rbc3RyLCBUZW5zb3JdOgogICAgcmV0dXJuIHtrZXk6IHZhbHVlW2luZGljZXNdIGZvciBrZXksIHZhbHVlIGluIGJhdGNoLml0ZW1zKCl9Cg==', 'olympus/data/registry.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBEYXRhU291cmNlUmVjb3JkKFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgc291cmNlX3VybDogc3RyCiAgICBvcmdhbml6YXRpb246IHN0cgogICAgbGljZW5zZTogc3RyCiAgICBwZXJtaXR0ZWRfdXNlczogbGlzdFtzdHJdCiAgICByZXRyaWV2YWxfZGF0ZTogc3RyCiAgICBzaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBwcm92ZW5hbmNlOiBzdHIKICAgIHF1YWxpdHlfc2NvcmU6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBsYW5ndWFnZTogc3RyCiAgICBkb21haW46IHN0cgogICAgcGVyc29uYWxfZGF0YTogYm9vbAogICAgdHJhaW5pbmdfZWxpZ2libGU6IGJvb2wKICAgIHJlbW92ZWQ6IGJvb2wgPSBGYWxzZQoKCmNsYXNzIERhdGFTb3VyY2VSZWdpc3RyeShTdHJpY3RNb2RlbCk6CiAgICBzb3VyY2VzOiBsaXN0W0RhdGFTb3VyY2VSZWNvcmRdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgogICAgZGVmIGFkZChzZWxmLCBzb3VyY2U6IERhdGFTb3VyY2VSZWNvcmQpIC0+IE5vbmU6CiAgICAgICAgaWYgYW55KGl0ZW0ubmFtZSA9PSBzb3VyY2UubmFtZSBhbmQgbm90IGl0ZW0ucmVtb3ZlZCBmb3IgaXRlbSBpbiBzZWxmLnNvdXJjZXMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiYWN0aXZlIGRhdGEgc291cmNlIGFscmVhZHkgZXhpc3RzOiB7c291cmNlLm5hbWV9IikKICAgICAgICBzZWxmLnNvdXJjZXMuYXBwZW5kKHNvdXJjZSkKCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBwYXRoLndyaXRlX3RleHQoc2VsZi5tb2RlbF9kdW1wX2pzb24oaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGxvYWQoY2xzLCBwYXRoOiBQYXRoKSAtPiBEYXRhU291cmNlUmVnaXN0cnk6CiAgICAgICAgcmV0dXJuIGNscy5tb2RlbF92YWxpZGF0ZShqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKSkK', 'olympus/data/reliability_fusion.py': 'IiIiRnJlc2gsIHNvdXJjZS1kaXNqb2ludCBkYXRhIGZvciB0aGUgcHJlcmVnaXN0ZXJlZCBPbHltcHVzIHYzIHN1Y2Nlc3Nvci4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGVtcGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgTGl0ZXJhbAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmRhdGEub2x5bXB1c19kYXRhc2V0IGltcG9ydCB3cml0ZV9vbHltcHVzX2RhdGFzZXRfbWFuaWZlc3QKClJlbGlhYmlsaXR5UmVnaW1lID0gTGl0ZXJhbFsibWl4ZWQiLCAiY2xlYW4iLCAiY29ycnVwdGVkIiwgIm9vZCJdCgoKY2xhc3MgUmVsaWFiaWxpdHlTcGxpdFNwZWMoU3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgZXhhbXBsZXM6IGludCA9IEZpZWxkKGdlPTEpCiAgICBzZWVkOiBpbnQgPSBGaWVsZChnZT0wLCBsZT0yKio2MyAtIDEpCiAgICByZWdpbWU6IFJlbGlhYmlsaXR5UmVnaW1lCgoKY2xhc3MgUmVsaWFiaWxpdHlEYXRhc2V0UmVjZWlwdChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogTGl0ZXJhbFsxXSA9IDEKICAgIGRhdGFzZXRfaWQ6IExpdGVyYWxbIm9seW1wdXMtcmVsaWFiaWxpdHktZnVzaW9uIl0gPSAib2x5bXB1cy1yZWxpYWJpbGl0eS1mdXNpb24iCiAgICB2ZXJzaW9uOiBMaXRlcmFsWyIzLjAuMCJdID0gIjMuMC4wIgogICAgbGljZW5zZTogTGl0ZXJhbFsiQ0MtQlktNC4wIl0gPSAiQ0MtQlktNC4wIgogICAgZ2VuZXJhdG9yOiBMaXRlcmFsWyJvbHltcHVzLmRhdGEucmVsaWFiaWxpdHlfZnVzaW9uIl0gPSAib2x5bXB1cy5kYXRhLnJlbGlhYmlsaXR5X2Z1c2lvbiIKICAgIHNwbGl0czogbGlzdFtSZWxpYWJpbGl0eVNwbGl0U3BlY10KICAgIGJ1bmRsZV9zaGEyNTY6IGRpY3Rbc3RyLCBzdHJdCiAgICBtYW5pZmVzdF9zaGEyNTY6IGRpY3Rbc3RyLCBzdHJdCiAgICBzb3VyY2VfaWRfc2hhMjU2OiBkaWN0W3N0ciwgc3RyXQogICAgc291cmNlX292ZXJsYXBfY291bnQ6IGludCA9IEZpZWxkKGdlPTApCiAgICBwcm90ZWN0ZWRfc3BsaXRzOiBsaXN0W3N0cl0KICAgIGNvbXBvbmVudF9vcmRlcjogbGlzdFtzdHJdCiAgICByZWxpYWJpbGl0eV9zZW1hbnRpY3M6IGRpY3Rbc3RyLCBmbG9hdF0KICAgIGNsYWltX2JvdW5kYXJ5OiBzdHIKCgpSRUdJU1RFUkVEX1JFTElBQklMSVRZX1NQTElUUyA9ICgKICAgIFJlbGlhYmlsaXR5U3BsaXRTcGVjKG5hbWU9InRyYWluIiwgZXhhbXBsZXM9NTEyLCBzZWVkPTIwMjYxNTAxLCByZWdpbWU9Im1peGVkIiksCiAgICBSZWxpYWJpbGl0eVNwbGl0U3BlYyhuYW1lPSJkZXZlbG9wbWVudCIsIGV4YW1wbGVzPTEyOCwgc2VlZD0yMDI2MTUwMiwgcmVnaW1lPSJtaXhlZCIpLAogICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMobmFtZT0iY2xlYW5fdGVzdCIsIGV4YW1wbGVzPTI1Niwgc2VlZD0yMDI2MTUwMywgcmVnaW1lPSJjbGVhbiIpLAogICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMobmFtZT0iY29ycnVwdGVkX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjE1MDQsIHJlZ2ltZT0iY29ycnVwdGVkIiksCiAgICBSZWxpYWJpbGl0eVNwbGl0U3BlYyhuYW1lPSJvb2RfdGVzdCIsIGV4YW1wbGVzPTI1Niwgc2VlZD0yMDI2MTUwNSwgcmVnaW1lPSJvb2QiKSwKKQpDT01QT05FTlRfT1JERVIgPSAoImhlcm1lcyIsICJwcm9tZXRoZXVzIiwgInBlcnNldXMiLCAiYXRsYXMiLCAia3Jvbm9zIikKVEVYVF9DT01QT05FTlRTID0gKDAsIDEsIDIpClJFTElBQklMSVRZX1ZBTFVFUyA9IHsiY2xlYW4iOiAxLjAsICJwZXJ0dXJiZWQiOiAwLjI1LCAibWlzc2luZyI6IDAuMH0KCgpkZWYgX3NoYTI1NihwYXlsb2FkOiBieXRlcykgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCgoKZGVmIF9hdG9taWNfd3JpdGUocGF0aDogUGF0aCwgcGF5bG9hZDogYnl0ZXMpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICBoYW5kbGUud3JpdGUocGF5bG9hZCkKICAgICAgICBoYW5kbGUuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgX3NvdXJjZV9pZHMoc3BlYzogUmVsaWFiaWxpdHlTcGxpdFNwZWMpIC0+IGxpc3Rbc3RyXToKICAgIHJldHVybiBbCiAgICAgICAgZiJ2My17c3BlYy5uYW1lfS17X3NoYTI1NihmJ3tzcGVjLnNlZWR9OntpbmRleH0nLmVuY29kZSgpKVs6MjBdfSIKICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2Uoc3BlYy5leGFtcGxlcykKICAgIF0KCgpkZWYgX2NvcnJ1cHRpb25fcGxhbigKICAgIHNwZWM6IFJlbGlhYmlsaXR5U3BsaXRTcGVjLCBnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvcgopIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIG1pc3NpbmcgPSB0b3JjaC5mdWxsKChzcGVjLmV4YW1wbGVzLCksIC0xLCBkdHlwZT10b3JjaC5sb25nKQogICAgcGVydHVyYmVkID0gdG9yY2guZnVsbCgoc3BlYy5leGFtcGxlcywpLCAtMSwgZHR5cGU9dG9yY2gubG9uZykKICAgIGlmIHNwZWMucmVnaW1lID09ICJtaXhlZCI6CiAgICAgICAgc2VsZWN0ZWQgPSB0b3JjaC5yYW5kKHNwZWMuZXhhbXBsZXMsIGdlbmVyYXRvcj1nZW5lcmF0b3IpIDwgMC43MAogICAgICAgIG1pc3Npbmdbc2VsZWN0ZWRdID0gdG9yY2gucmFuZGludCgwLCAzLCAoaW50KHNlbGVjdGVkLnN1bSgpKSwpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgZWxpZiBzcGVjLnJlZ2ltZSBpbiB7ImNvcnJ1cHRlZCIsICJvb2QifToKICAgICAgICBtaXNzaW5nID0gdG9yY2gucmFuZGludCgwLCAzLCAoc3BlYy5leGFtcGxlcywpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgICAgIG9mZnNldCA9IHRvcmNoLnJhbmRpbnQoMSwgMywgKHNwZWMuZXhhbXBsZXMsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgICAgICBwZXJ0dXJiZWQgPSAobWlzc2luZyArIG9mZnNldCkgJSAzCiAgICByZXR1cm4gbWlzc2luZywgcGVydHVyYmVkCgoKZGVmIF9yZWxpYWJpbGl0eV90YXJnZXRzKG1pc3Npbmc6IFRlbnNvciwgcGVydHVyYmVkOiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgIHRhcmdldHMgPSB0b3JjaC5vbmVzKChtaXNzaW5nLnNoYXBlWzBdLCBsZW4oQ09NUE9ORU5UX09SREVSKSksIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICBtb2RhbGl0eV9jb21wb25lbnRzID0gezA6IFRFWFRfQ09NUE9ORU5UUywgMTogKDMsKSwgMjogKDQsKX0KICAgIGZvciBtb2RhbGl0eSwgY29tcG9uZW50cyBpbiBtb2RhbGl0eV9jb21wb25lbnRzLml0ZW1zKCk6CiAgICAgICAgZm9yIGNvbXBvbmVudCBpbiBjb21wb25lbnRzOgogICAgICAgICAgICB0YXJnZXRzW21pc3NpbmcgPT0gbW9kYWxpdHksIGNvbXBvbmVudF0gPSBSRUxJQUJJTElUWV9WQUxVRVNbIm1pc3NpbmciXQogICAgICAgICAgICB0YXJnZXRzW3BlcnR1cmJlZCA9PSBtb2RhbGl0eSwgY29tcG9uZW50XSA9IFJFTElBQklMSVRZX1ZBTFVFU1sicGVydHVyYmVkIl0KICAgIHJldHVybiB0YXJnZXRzCgoKZGVmIGdlbmVyYXRlX3JlbGlhYmlsaXR5X3NwbGl0KAogICAgc3BlYzogUmVsaWFiaWxpdHlTcGxpdFNwZWMsCikgLT4gdHVwbGVbZGljdFtzdHIsIFRlbnNvcl0sIGxpc3Rbc3RyXV06CiAgICAiIiJHZW5lcmF0ZSBhIGRldGVybWluaXN0aWMgc3BsaXQgd2l0aCBpbmRlcGVuZGVudGx5IHNwZWNpZmllZCByZWxpYWJpbGl0eSBsYWJlbHMuIiIiCgogICAgZ2VuZXJhdG9yID0gdG9yY2guR2VuZXJhdG9yKGRldmljZT0iY3B1IikubWFudWFsX3NlZWQoc3BlYy5zZWVkKQogICAgY291bnQgPSBzcGVjLmV4YW1wbGVzCiAgICBsYWJlbHMgPSB0b3JjaC5yYW5kaW50KDAsIDQsIChjb3VudCwpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgbWlzc2luZywgcGVydHVyYmVkID0gX2NvcnJ1cHRpb25fcGxhbihzcGVjLCBnZW5lcmF0b3IpCiAgICByZWxpYWJpbGl0eV90YXJnZXRzID0gX3JlbGlhYmlsaXR5X3RhcmdldHMobWlzc2luZywgcGVydHVyYmVkKQogICAgbm9pc2Vfc2NhbGUgPSAwLjEyIGlmIHNwZWMucmVnaW1lICE9ICJvb2QiIGVsc2UgMC4yNAoKICAgIHRva2VucyA9IHRvcmNoLnJhbmRpbnQoOCwgMzEsIChjb3VudCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBjbGFzc190b2tlbnMgPSBsYWJlbHMgKyAxCiAgICBpZiBzcGVjLnJlZ2ltZSA9PSAib29kIjoKICAgICAgICBjbGFzc190b2tlbnMgPSB0b3JjaC50ZW5zb3IoWzMsIDEsIDQsIDJdLCBkdHlwZT10b3JjaC5sb25nKVtsYWJlbHNdCiAgICB0b2tlbnNbOiwgMF0gPSBjbGFzc190b2tlbnMKICAgIHRva2Vuc1s6LCAxXSA9IDUgKyBsYWJlbHMucmVtYWluZGVyKDIpCiAgICB0b2tlbnNbOiwgMl0gPSA3ICsgbGFiZWxzLmRpdigyLCByb3VuZGluZ19tb2RlPSJmbG9vciIpCiAgICBhdHRlbnRpb25fbWFzayA9IHRvcmNoLm9uZXNfbGlrZSh0b2tlbnMsIGR0eXBlPXRvcmNoLmJvb2wpCgogICAgaW1hZ2UgPSB0b3JjaC5yYW5kKChjb3VudCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpICogbm9pc2Vfc2NhbGUKICAgIGZvciBpbmRleCBpbiByYW5nZShjb3VudCk6CiAgICAgICAgbGFiZWwgPSBpbnQobGFiZWxzW2luZGV4XSkKICAgICAgICBjaGFubmVsID0gbGFiZWwgJSAzCiAgICAgICAgdG9wID0gMCBpZiBsYWJlbCA8IDIgZWxzZSA0CiAgICAgICAgbGVmdCA9IDAgaWYgbGFiZWwgJSAyID09IDAgZWxzZSA0CiAgICAgICAgaWYgc3BlYy5yZWdpbWUgPT0gIm9vZCI6CiAgICAgICAgICAgIGNoYW5uZWwgPSAoY2hhbm5lbCArIDEpICUgMwogICAgICAgICAgICBsZWZ0ID0gNCAtIGxlZnQKICAgICAgICBpbWFnZVtpbmRleCwgY2hhbm5lbCwgdG9wIDogdG9wICsgNCwgbGVmdCA6IGxlZnQgKyA0XSArPSAwLjc1CiAgICBpbWFnZS5jbGFtcF8oMC4wLCAxLjApCgogICAgYWN0aW9ucyA9IHRvcmNoLnJhbmQoKGNvdW50LCA0LCAzKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikgKiAyLjAgLSAxLjAKICAgIHRpbWUgPSB0b3JjaC5hcmFuZ2UoNCwgZHR5cGU9dG9yY2guZmxvYXQzMikudmlldygxLCA0LCAxKQogICAgbGFiZWxfdmFsdWUgPSBsYWJlbHMuZmxvYXQoKS52aWV3KGNvdW50LCAxLCAxKSAvIDMuMAogICAgc3RhdGVfbm9pc2UgPSB0b3JjaC5yYW5kbigoY291bnQsIDQsIDUpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSAqIG5vaXNlX3NjYWxlCiAgICBzdGF0ZV90ZW1wbGF0ZSA9IHRvcmNoLmNhdCgKICAgICAgICAoCiAgICAgICAgICAgIGxhYmVsX3ZhbHVlLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIChsYWJlbF92YWx1ZSArIDAuMDggKiB0aW1lKS5leHBhbmQoLTEsIC0xLCAxKSwKICAgICAgICAgICAgbGFiZWxzLnJlbWFpbmRlcigyKS5mbG9hdCgpLnZpZXcoY291bnQsIDEsIDEpLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIGxhYmVscy5kaXYoMiwgcm91bmRpbmdfbW9kZT0iZmxvb3IiKS5mbG9hdCgpLnZpZXcoY291bnQsIDEsIDEpLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIHRvcmNoLnNpbihsYWJlbF92YWx1ZSAqIHRvcmNoLnBpICsgdGltZSAqIDAuMyksCiAgICAgICAgKSwKICAgICAgICBkaW09LTEsCiAgICApCiAgICBzdGF0ZXMgPSBzdGF0ZV90ZW1wbGF0ZSArIHN0YXRlX25vaXNlCgogICAgZm9yIG1vZGFsaXR5LCB2YWx1ZSBpbiBlbnVtZXJhdGUoKHRva2VucywgaW1hZ2UsIHN0YXRlcykpOgogICAgICAgIHNlbGVjdGVkX21pc3NpbmcgPSBtaXNzaW5nID09IG1vZGFsaXR5CiAgICAgICAgc2VsZWN0ZWRfcGVydHVyYmVkID0gcGVydHVyYmVkID09IG1vZGFsaXR5CiAgICAgICAgaWYgYm9vbChzZWxlY3RlZF9taXNzaW5nLmFueSgpKToKICAgICAgICAgICAgdmFsdWVbc2VsZWN0ZWRfbWlzc2luZ10gPSAwCiAgICAgICAgaWYgYm9vbChzZWxlY3RlZF9wZXJ0dXJiZWQuYW55KCkpOgogICAgICAgICAgICBhbW91bnQgPSBpbnQoc2VsZWN0ZWRfcGVydHVyYmVkLnN1bSgpKQogICAgICAgICAgICBpZiBtb2RhbGl0eSA9PSAwOgogICAgICAgICAgICAgICAgdG9rZW5zW3NlbGVjdGVkX3BlcnR1cmJlZCwgOjNdID0gdG9yY2gucmFuZGludCgKICAgICAgICAgICAgICAgICAgICAxLCAxMiwgKGFtb3VudCwgMyksIGdlbmVyYXRvcj1nZW5lcmF0b3IKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxpZiBtb2RhbGl0eSA9PSAxOgogICAgICAgICAgICAgICAgaW1hZ2Vbc2VsZWN0ZWRfcGVydHVyYmVkXSA9IHRvcmNoLnJhbmQoKGFtb3VudCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzdGF0ZXNbc2VsZWN0ZWRfcGVydHVyYmVkXSA9IHRvcmNoLnJhbmRuKChhbW91bnQsIDQsIDUpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQoKICAgIHRlbXBvcmFsX21hc2sgPSB0b3JjaC5vbmVzKChjb3VudCwgNCksIGR0eXBlPXRvcmNoLmJvb2wpCiAgICBhbGxvd2VkX3RyYW5zaXRpb25zID0gdG9yY2gub25lcygoY291bnQsIDQpLCBkdHlwZT10b3JjaC5ib29sKQogICAgdHJhbnNpdGlvbl90YXJnZXRzID0gbGFiZWxzLnJlbWFpbmRlcigzKQogICAgZm9yYmlkZGVuID0gKHRyYW5zaXRpb25fdGFyZ2V0cyArIDEpLnJlbWFpbmRlcigzKQogICAgYWxsb3dlZF90cmFuc2l0aW9uc1t0b3JjaC5hcmFuZ2UoY291bnQpLCBmb3JiaWRkZW5dID0gRmFsc2UKICAgIGFsbG93ZWRfdHJhbnNpdGlvbnNbOiwgM10gPSBUcnVlCiAgICBjb3JydXB0aW9uX2NvdW50ID0gbWlzc2luZy5nZSgwKS5mbG9hdCgpICsgcGVydHVyYmVkLmdlKDApLmZsb2F0KCkKICAgIHJpc2tfc2NvcmUgPSAwLjA1ICsgY29ycnVwdGlvbl9jb3VudCAqIDAuMTUKICAgIHBhZGRlZF9hY3Rpb25zID0gdG9yY2gubm4uZnVuY3Rpb25hbC5wYWQoYWN0aW9ucywgKDAsIDIpKQoKICAgIGJ1bmRsZSA9IHsKICAgICAgICAidG9rZW5zIjogdG9rZW5zLAogICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAogICAgICAgICJpbWFnZSI6IGltYWdlLAogICAgICAgICJzdGF0ZXMiOiBzdGF0ZXMsCiAgICAgICAgImFjdGlvbnMiOiBhY3Rpb25zLAogICAgICAgICJ0ZW1wb3JhbF9tYXNrIjogdGVtcG9yYWxfbWFzaywKICAgICAgICAiYWxsb3dlZF90cmFuc2l0aW9ucyI6IGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAgICAgICAgInJpc2tfc2NvcmUiOiByaXNrX3Njb3JlLAogICAgICAgICJkb21haW5faWRzIjogbGFiZWxzLnJlbWFpbmRlcigzKSwKICAgICAgICAidGltZXN0YW1wIjogdG9yY2guYXJhbmdlKGNvdW50LCBkdHlwZT10b3JjaC5mbG9hdDMyKSAvIG1heCgxLCBjb3VudCAtIDEpLAogICAgICAgICJkZWNpc2lvbl90YXJnZXRzIjogbGFiZWxzLAogICAgICAgICJ0cmFuc2l0aW9uX3RhcmdldHMiOiB0cmFuc2l0aW9uX3RhcmdldHMsCiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyI6IHN0YXRlcyArIDAuMSAqIHBhZGRlZF9hY3Rpb25zLAogICAgICAgICJpbnRyaW5zaWNfcmlza190YXJnZXRzIjogY29ycnVwdGlvbl9jb3VudCAvIDIuMCwKICAgICAgICAiY29tcG9uZW50X3JlbGlhYmlsaXR5X3RhcmdldHMiOiByZWxpYWJpbGl0eV90YXJnZXRzLAogICAgfQogICAgcmV0dXJuIGJ1bmRsZSwgX3NvdXJjZV9pZHMoc3BlYykKCgpkZWYgX3NlcmlhbGl6ZV9idW5kbGUoYnVuZGxlOiBkaWN0W3N0ciwgVGVuc29yXSkgLT4gYnl0ZXM6CiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIHRvcmNoLnNhdmUoYnVuZGxlLCBidWZmZXIpCiAgICByZXR1cm4gYnVmZmVyLmdldHZhbHVlKCkKCgpkZWYgZ2VuZXJhdGVfcmVsaWFiaWxpdHlfZnVzaW9uX2RhdGFzZXQob3V0cHV0OiBQYXRoKSAtPiBSZWxpYWJpbGl0eURhdGFzZXRSZWNlaXB0OgogICAgIiIiTWF0ZXJpYWxpemUgdjMgc3BsaXRzIG9uY2UgYW5kIGJpbmQgZXZlcnkgc3BsaXQgdG8gY29udGVudCBoYXNoZXMuIiIiCgogICAgb3V0cHV0ID0gb3V0cHV0LnJlc29sdmUoKQogICAgaWYgb3V0cHV0LmV4aXN0cygpIGFuZCBhbnkob3V0cHV0Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKCJyZWZ1c2luZyB0byBvdmVyd3JpdGUgYSByZWxpYWJpbGl0eSBkYXRhc2V0IGRpcmVjdG9yeSIpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHNvdXJjZV9pZHM6IGRpY3Rbc3RyLCBsaXN0W3N0cl1dID0ge30KICAgIGJ1bmRsZV9oYXNoZXM6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGZvciBzcGVjIGluIFJFR0lTVEVSRURfUkVMSUFCSUxJVFlfU1BMSVRTOgogICAgICAgIGJ1bmRsZSwgaWRlbnRpZmllcnMgPSBnZW5lcmF0ZV9yZWxpYWJpbGl0eV9zcGxpdChzcGVjKQogICAgICAgIHBheWxvYWQgPSBfc2VyaWFsaXplX2J1bmRsZShidW5kbGUpCiAgICAgICAgX2F0b21pY193cml0ZShvdXRwdXQgLyBmIntzcGVjLm5hbWV9LnB0IiwgcGF5bG9hZCkKICAgICAgICBzb3VyY2VfaWRzW3NwZWMubmFtZV0gPSBpZGVudGlmaWVycwogICAgICAgIGJ1bmRsZV9oYXNoZXNbc3BlYy5uYW1lXSA9IF9zaGEyNTYocGF5bG9hZCkKCiAgICBhbGxfaWRzID0gW2lkZW50aWZpZXIgZm9yIHZhbHVlcyBpbiBzb3VyY2VfaWRzLnZhbHVlcygpIGZvciBpZGVudGlmaWVyIGluIHZhbHVlc10KICAgIG92ZXJsYXBfY291bnQgPSBsZW4oYWxsX2lkcykgLSBsZW4oc2V0KGFsbF9pZHMpKQogICAgaWYgb3ZlcmxhcF9jb3VudDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInYzIHNvdXJjZSBpZGVudGl0aWVzIG92ZXJsYXAiKQoKICAgIG1hbmlmZXN0X2hhc2hlczogZGljdFtzdHIsIHN0cl0gPSB7fQogICAgZm9yIGV2YWx1YXRpb25fbmFtZSBpbiAoImRldmVsb3BtZW50IiwgImNsZWFuX3Rlc3QiLCAiY29ycnVwdGVkX3Rlc3QiLCAib29kX3Rlc3QiKToKICAgICAgICBtYW5pZmVzdCA9IHdyaXRlX29seW1wdXNfZGF0YXNldF9tYW5pZmVzdCgKICAgICAgICAgICAgb3V0cHV0IC8gZiJtYW5pZmVzdF97ZXZhbHVhdGlvbl9uYW1lfS5qc29uIiwKICAgICAgICAgICAgZGF0YXNldF9pZD0ib2x5bXB1cy1yZWxpYWJpbGl0eS1mdXNpb24iLAogICAgICAgICAgICB2ZXJzaW9uPSIzLjAuMCIsCiAgICAgICAgICAgIGxpY2Vuc2VfbmFtZT0iQ0MtQlktNC4wIiwKICAgICAgICAgICAgdHJhaW5fYmF0Y2g9b3V0cHV0IC8gInRyYWluLnB0IiwKICAgICAgICAgICAgZXZhbHVhdGlvbl9iYXRjaD1vdXRwdXQgLyBmIntldmFsdWF0aW9uX25hbWV9LnB0IiwKICAgICAgICAgICAgdHJhaW5fc291cmNlX2lkcz1zb3VyY2VfaWRzWyJ0cmFpbiJdLAogICAgICAgICAgICBldmFsdWF0aW9uX3NvdXJjZV9pZHM9c291cmNlX2lkc1tldmFsdWF0aW9uX25hbWVdLAogICAgICAgICkKICAgICAgICBhc3NlcnQgbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2IGlzIG5vdCBOb25lCiAgICAgICAgbWFuaWZlc3RfaGFzaGVzW2V2YWx1YXRpb25fbmFtZV0gPSBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYKCiAgICByZWNlaXB0ID0gUmVsaWFiaWxpdHlEYXRhc2V0UmVjZWlwdCgKICAgICAgICBzcGxpdHM9bGlzdChSRUdJU1RFUkVEX1JFTElBQklMSVRZX1NQTElUUyksCiAgICAgICAgYnVuZGxlX3NoYTI1Nj1idW5kbGVfaGFzaGVzLAogICAgICAgIG1hbmlmZXN0X3NoYTI1Nj1tYW5pZmVzdF9oYXNoZXMsCiAgICAgICAgc291cmNlX2lkX3NoYTI1Nj17CiAgICAgICAgICAgIG5hbWU6IF9zaGEyNTYoKCJcbiIuam9pbih2YWx1ZXMpICsgIlxuIikuZW5jb2RlKCkpCiAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZXMgaW4gc291cmNlX2lkcy5pdGVtcygpCiAgICAgICAgfSwKICAgICAgICBzb3VyY2Vfb3ZlcmxhcF9jb3VudD1vdmVybGFwX2NvdW50LAogICAgICAgIHByb3RlY3RlZF9zcGxpdHM9WyJjbGVhbl90ZXN0IiwgImNvcnJ1cHRlZF90ZXN0IiwgIm9vZF90ZXN0Il0sCiAgICAgICAgY29tcG9uZW50X29yZGVyPWxpc3QoQ09NUE9ORU5UX09SREVSKSwKICAgICAgICByZWxpYWJpbGl0eV9zZW1hbnRpY3M9UkVMSUFCSUxJVFlfVkFMVUVTLAogICAgICAgIGNsYWltX2JvdW5kYXJ5PSgKICAgICAgICAgICAgIkNvbnRyb2xsZWQgc3ludGhldGljIHJlbGlhYmlsaXR5LW1lY2hhbmlzbSBldmlkZW5jZSBvbmx5OyBubyBuYXR1cmFsLWRhdGEsICIKICAgICAgICAgICAgInRhcmdldC1zY2FsZSwgZGVwbG95bWVudCwgc2FmZXR5LCBvciBKRVBBIGNsYWltIGlzIGxpY2Vuc2VkLiIKICAgICAgICApLAogICAgKQogICAgX2F0b21pY193cml0ZSgKICAgICAgICBvdXRwdXQgLyAicmVjZWlwdC5qc29uIiwKICAgICAgICAoanNvbi5kdW1wcyhyZWNlaXB0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIikuZW5jb2RlKCksCiAgICApCiAgICByZXR1cm4gcmVjZWlwdAo=', 'olympus/data/retrieval.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIG1hdGggaW1wb3J0IHNxcnQKCmZyb20gb2x5bXB1cy5jb3JlLm1vZGFsaXRpZXMgaW1wb3J0IFJlZmVyZW5jZUVuY29kZXIsIFRleHRBcnRpZmFjdApmcm9tIG9seW1wdXMuZGF0YS5pbmdlc3Rpb24gaW1wb3J0IEluZ2VzdGVkRG9jdW1lbnQKCgpjbGFzcyBSZXRyaWV2YWxQbGF0Zm9ybToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuZW5jb2RlciA9IFJlZmVyZW5jZUVuY29kZXIoKQoKICAgIGRlZiBsZXhpY2FsX3NlYXJjaCgKICAgICAgICBzZWxmLCBxdWVyeTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0KICAgICkgLT4gbGlzdFt0dXBsZVtzdHIsIGZsb2F0XV06CiAgICAgICAgcXVlcnlfdG9rZW5zID0gc2V0KHF1ZXJ5Lmxvd2VyKCkuc3BsaXQoKSkKICAgICAgICBzY29yZWQgPSBbXQogICAgICAgIGZvciBkb2N1bWVudCBpbiBkb2N1bWVudHM6CiAgICAgICAgICAgIG92ZXJsYXAgPSBxdWVyeV90b2tlbnMuaW50ZXJzZWN0aW9uKGRvY3VtZW50LnRleHQubG93ZXIoKS5zcGxpdCgpKQogICAgICAgICAgICBzY29yZWQuYXBwZW5kKChkb2N1bWVudC5pZGVudGlmaWVyLCBsZW4ob3ZlcmxhcCkgLyBtYXgoMSwgbGVuKHF1ZXJ5X3Rva2VucykpKSkKICAgICAgICByZXR1cm4gc29ydGVkKHNjb3JlZCwga2V5PWxhbWJkYSBpdGVtOiBpdGVtWzFdLCByZXZlcnNlPVRydWUpCgogICAgZGVmIHNlbWFudGljX3NlYXJjaCgKICAgICAgICBzZWxmLCBxdWVyeTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0KICAgICkgLT4gbGlzdFt0dXBsZVtzdHIsIGZsb2F0XV06CiAgICAgICAgcXVlcnlfdmVjdG9yID0gc2VsZi5lbmNvZGVyLmVuY29kZShUZXh0QXJ0aWZhY3QoY29udGVudD1xdWVyeSkpLnZlY3RvcgogICAgICAgIHNjb3JlZCA9IFtdCiAgICAgICAgZm9yIGRvY3VtZW50IGluIGRvY3VtZW50czoKICAgICAgICAgICAgZG9jdW1lbnRfdmVjdG9yID0gc2VsZi5lbmNvZGVyLmVuY29kZShUZXh0QXJ0aWZhY3QoY29udGVudD1kb2N1bWVudC50ZXh0KSkudmVjdG9yCiAgICAgICAgICAgIG51bWVyYXRvciA9IHN1bShhICogYiBmb3IgYSwgYiBpbiB6aXAocXVlcnlfdmVjdG9yLCBkb2N1bWVudF92ZWN0b3IsIHN0cmljdD1GYWxzZSkpCiAgICAgICAgICAgIGRlbm9taW5hdG9yID0gKAogICAgICAgICAgICAgICAgc3FydChzdW0oYSAqIGEgZm9yIGEgaW4gcXVlcnlfdmVjdG9yKSAqIHN1bShiICogYiBmb3IgYiBpbiBkb2N1bWVudF92ZWN0b3IpKSBvciAxLjAKICAgICAgICAgICAgKQogICAgICAgICAgICBzY29yZWQuYXBwZW5kKChkb2N1bWVudC5pZGVudGlmaWVyLCBudW1lcmF0b3IgLyBkZW5vbWluYXRvcikpCiAgICAgICAgcmV0dXJuIHNvcnRlZChzY29yZWQsIGtleT1sYW1iZGEgaXRlbTogaXRlbVsxXSwgcmV2ZXJzZT1UcnVlKQoKICAgIGRlZiBjb250cmFkaWN0aW9uX3NlYXJjaChzZWxmLCBjbGFpbTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0pIC0+IGxpc3Rbc3RyXToKICAgICAgICBjbGFpbV9jb3VudGVyID0gQ291bnRlcihjbGFpbS5sb3dlcigpLnNwbGl0KCkpCiAgICAgICAgY29udHJhZGljdG9yczogbGlzdFtzdHJdID0gW10KICAgICAgICBmb3IgZG9jdW1lbnQgaW4gZG9jdW1lbnRzOgogICAgICAgICAgICB0ZXh0ID0gZG9jdW1lbnQudGV4dC5sb3dlcigpCiAgICAgICAgICAgIGlmIGFueShmIm5vdCB7d29yZH0iIGluIHRleHQgZm9yIHdvcmQgaW4gY2xhaW1fY291bnRlcik6CiAgICAgICAgICAgICAgICBjb250cmFkaWN0b3JzLmFwcGVuZChkb2N1bWVudC5pZGVudGlmaWVyKQogICAgICAgIHJldHVybiBjb250cmFkaWN0b3JzCg==', 'olympus/data/robust_fusion_v5.py': 'IiIiVjUgZnJlc2ggZXhhbXBsZXMsIGV4cGFuZGVkIHRyYWluaW5nIGNvdmVyYWdlLCBhbmQgcmVzZXJ2ZWQgT09EIGZhbWlsaWVzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmRhdGEub3JhY2xlX2Z1c2lvbl92NCBpbXBvcnQgZ2VuZXJhdGUgYXMgcHJldmlvdXNfZ2VuZXJhdGUKCgpkZWYgZ2VuZXJhdGUoc2VlZDogaW50LCBjb3VudDogaW50LCByZWdpbWU6IHN0cikgLT4gZGljdFtzdHIsIFRlbnNvcl06CiAgICBpZiByZWdpbWUgbm90IGluIHsiY29yZSIsICJicm9hZCIsICJjbGVhbiIsICJjb3JydXB0ZWQiLCAia25vd25fc2hpZnQiLCAibm92ZWxfc2hpZnQifToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKHJlZ2ltZSkKICAgIHNvdXJjZSA9IHsKICAgICAgICAiY29yZSI6ICJtaXhlZCIsCiAgICAgICAgImJyb2FkIjogIm1peGVkIiwKICAgICAgICAia25vd25fc2hpZnQiOiAib29kIiwKICAgICAgICAibm92ZWxfc2hpZnQiOiAiY29ycnVwdGVkIiwKICAgIH0uZ2V0KHJlZ2ltZSwgcmVnaW1lKQogICAgYmF0Y2ggPSBwcmV2aW91c19nZW5lcmF0ZShzZWVkLCBjb3VudCwgc291cmNlKQogICAgZyA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQgKyA5MTAwMDApCiAgICBpZiByZWdpbWUgPT0gImJyb2FkIjoKICAgICAgICAjIFNhbWUgbGF0ZW50IGV4YW1wbGVzIGFuZCBtaXNzaW5nbmVzcyBhcyBjb3JlLiBPbmx5IGRhbWFnZWQgb2JzZXJ2YXRpb25zCiAgICAgICAgIyBjaGFuZ2U7IGNsZWFuIGFuZCBtaXNzaW5nLW9ubHkgZXhhbXBsZXMgcmVtYWluIGV4YWN0bHkgbWF0Y2hlZC4KICAgICAgICBjbGVhbiA9IHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNvdW50LCAiY2xlYW4iKQogICAgICAgIGNob29zZSA9IHRvcmNoLnJhbmQoY291bnQsIGdlbmVyYXRvcj1nKSA8IDAuNQogICAgICAgIGZvciBpbmRleCwga2V5IGluIGVudW1lcmF0ZSgoInRva2VucyIsICJpbWFnZSIsICJzdGF0ZXMiKSk6CiAgICAgICAgICAgIGRhbWFnZWQgPSAoYmF0Y2hbImNvcnJ1cHRpb25fc3RhdHVzIl1bOiwgaW5kZXhdID09IDIpICYgY2hvb3NlCiAgICAgICAgICAgIHggPSBjbGVhbltrZXldLmNsb25lKCkKICAgICAgICAgICAgaWYgaW5kZXggPT0gMDoKICAgICAgICAgICAgICAgIHhbOiwgOjNdID0geFs6LCA6M10uZmxpcCgtMSkKICAgICAgICAgICAgZWxpZiBpbmRleCA9PSAxOgogICAgICAgICAgICAgICAgeCA9ICgxIC0geCkuY2xhbXAoMCwgMSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHggKz0gMS41CiAgICAgICAgICAgIGJhdGNoW2tleV1bZGFtYWdlZF0gPSB4W2RhbWFnZWRdCiAgICBlbGlmIHJlZ2ltZSA9PSAibm92ZWxfc2hpZnQiOgogICAgICAgIGNsZWFuID0gcHJldmlvdXNfZ2VuZXJhdGUoc2VlZCwgY291bnQsICJjbGVhbiIpCiAgICAgICAgZm9yIGluZGV4LCBrZXkgaW4gZW51bWVyYXRlKCgidG9rZW5zIiwgImltYWdlIiwgInN0YXRlcyIpKToKICAgICAgICAgICAgZGFtYWdlZCA9IGJhdGNoWyJjb3JydXB0aW9uX3N0YXR1cyJdWzosIGluZGV4XSA9PSAyCiAgICAgICAgICAgIHggPSBjbGVhbltrZXldLmNsb25lKCkKICAgICAgICAgICAgaWYgaW5kZXggPT0gMDoKICAgICAgICAgICAgICAgICMgUmVsb2NhdGlvbiBpcyBhYnNlbnQgZnJvbSBjb3JlL2Jyb2FkL2RldmVsb3BtZW50IGNvcnJ1cHRpb24uCiAgICAgICAgICAgICAgICB4ID0geC5yb2xsKDQsIC0xKQogICAgICAgICAgICBlbGlmIGluZGV4ID09IDE6CiAgICAgICAgICAgICAgICAjIFNwYXRpYWxseSBzdHJ1Y3R1cmVkIG9jY2x1c2lvbiwgbm90IGludmVyc2lvbiBvciByYW5kb20gbm9pc2UuCiAgICAgICAgICAgICAgICB4WzosIDosIDo6MiwgOjoyXSA9IDAuOAogICAgICAgICAgICAgICAgeFs6LCA6LCAxOjoyLCAxOjoyXSA9IDAuOAogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgIyBOZWdhdGl2ZSB0aW1lLXZhcnlpbmcgZHJpZnQsIG5vdCB0aGUga25vd24gY29uc3RhbnQgcG9zaXRpdmUgYmlhcy4KICAgICAgICAgICAgICAgIHggLT0gdG9yY2gubGluc3BhY2UoMC40LCAxLjYsIDQpW05vbmUsIDosIE5vbmVdCiAgICAgICAgICAgIGJhdGNoW2tleV1bZGFtYWdlZF0gPSB4W2RhbWFnZWRdCiAgICByZXR1cm4gYmF0Y2gK', 'olympus/demos.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIG9seW1wdXMuY29yZS5kaXZpc2lvbiBpbXBvcnQgTmV1cmFsRGl2aXNpb25SZWFzc2VtYmx5LCBTcGVjaWFsaXN0UmVzdWx0LCBTcGVjaWFsaXN0VGFzawpmcm9tIG9seW1wdXMuY29yZS5pbnRlcnByZXRhdGlvbiBpbXBvcnQgSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsKZnJvbSBvbHltcHVzLmNvcmUucmVwcmVzZW50YXRpb24gaW1wb3J0IER5bmFtaWNSZXByZXNlbnRhdGlvblRoZW9yeSwgUmVwcmVzZW50YXRpb25GYW1pbHkKZnJvbSBvbHltcHVzLmNvcmUucmV0cm9kaWN0aW9uIGltcG9ydCBSZXRyb2RpY3RpdmVJZGVudGl0eU5ldHdvcmssIGNvZGVfYnVnX2dlbmVyYXRvcnMKZnJvbSBvbHltcHVzLmNvcmUudHJhbnNmb3JtIGltcG9ydCB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybQpmcm9tIG9seW1wdXMuZm9yZ2UuY29tcGlsZXIgaW1wb3J0IE5hdHVyYWxMYW5ndWFnZUJlaGF2aW9yQ29tcGlsZXIKZnJvbSBvbHltcHVzLmZvcmdlLnJ1bnRpbWUgaW1wb3J0IEZvcmdlUnVudGltZQpmcm9tIG9seW1wdXMubWVtb3J5LnN0b3JlIGltcG9ydCBNZW1vcnlTdG9yZQpmcm9tIG9seW1wdXMubW9kZWxzLmZhbWlsaWVzIGltcG9ydCBIZXJtZXNOYW5vLCBIZXJtZXNSZXNwb25zZQpmcm9tIG9seW1wdXMudHJhaW5pbmcuc3ludGhldGljIGltcG9ydCB0cmFuc2Zvcm1fc2lnbmFscwoKCmRlZiBydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8oKSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIG9iamVjdF1dOgogICAgbmV0d29yayA9IEludGVycHJldGl2ZVN1cGVycG9zaXRpb25OZXR3b3JrKCkKICAgIGJyYW5jaGVzID0gbmV0d29yay5hbmFseXplKCJUaGUgY2xvdWQgd2FzIGEgZHJhZ29uIGFib3ZlIHRoZSBjaXR5LiIpCiAgICB0b3BfY2F0ZWdvcnkgPSBicmFuY2hlc1swXS5jYXRlZ29yeS52YWx1ZQogICAgcmV0dXJuIHsKICAgICAgICAiYW1iaWd1b3VzX2ludGVycHJldGF0aW9uIjogewogICAgICAgICAgICAicGFzc2VkIjogbGVuKGJyYW5jaGVzKSA+PSAzIGFuZCB0b3BfY2F0ZWdvcnkgaW4geyJsaXRlcmFsIiwgImNvbmNyZXRlX3Jlc2VtYmxhbmNlIn0sCiAgICAgICAgICAgICJkZXRhaWwiOiBmIlRvcCBpbnRlcnByZXRhdGlvbiBjYXRlZ29yeToge3RvcF9jYXRlZ29yeX0iLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX3JldHJvZGljdGlvbl9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3RdXToKICAgIGN1cnJlbnQgPSAiZGVmIGF2ZXJhZ2UodG90YWwsIGNvdW50KTpcbiAgICByZXR1cm4gdG90YWwgLyBjb3VudFxuIgogICAgbmV0d29yayA9IFJldHJvZGljdGl2ZUlkZW50aXR5TmV0d29yaygpCiAgICByZXN1bHRzID0gbmV0d29yay5pbmZlcigKICAgICAgICBjdXJyZW50X3N0YXRlPWN1cnJlbnQsCiAgICAgICAgZ2VuZXJhdG9ycz1jb2RlX2J1Z19nZW5lcmF0b3JzKCksCiAgICAgICAgZm9yd2FyZF9zaW11bGF0b3I9bGFtYmRhIGNhbmRpZGF0ZTogKAogICAgICAgICAgICBjYW5kaWRhdGUucmVwbGFjZSgicmV0dXJuIHRvdGFsIC8gKGNvdW50IC0gMSkiLCAicmV0dXJuIHRvdGFsIC8gY291bnQiKQogICAgICAgICAgICAucmVwbGFjZSgicmV0dXJuIHRvdGFsIC8gY291bnQgaWYgY291bnQgZWxzZSAwIiwgInJldHVybiB0b3RhbCAvIGNvdW50IikKICAgICAgICAgICAgLnJlcGxhY2UoInRvdGFsID0rIHZhbHVlIiwgInRvdGFsICs9IHZhbHVlIikKICAgICAgICApLAogICAgICAgIGludmFyaWFudF9jaGVja2VyPWxhbWJkYSBjYW5kaWRhdGU6ICJyZXR1cm4iIGluIGNhbmRpZGF0ZSBhbmQgImRlZiBhdmVyYWdlIiBpbiBjYW5kaWRhdGUsCiAgICApCiAgICByZXR1cm4gewogICAgICAgICJyZXRyb2RpY3Rpb24iOiB7CiAgICAgICAgICAgICJwYXNzZWQiOiByZXN1bHRzWzBdLmZvcndhcmRfbWF0Y2gsCiAgICAgICAgICAgICJkZXRhaWwiOiByZXN1bHRzWzBdLmNhbmRpZGF0ZV9zdGF0ZS5zcGxpdGxpbmVzKClbLTFdLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX3RyYW5zZm9ybV9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3RdXToKICAgIG1ldHJpY3MgPSB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybSh0cmFuc2Zvcm1fc2lnbmFscygpKQogICAgcmV0dXJuIHsKICAgICAgICAibGVhcm5lZF90cmFuc2Zvcm0iOiB7CiAgICAgICAgICAgICJwYXNzZWQiOiBtZXRyaWNzLnJlY29uc3RydWN0aW9uX2xvc3MgPCAwLjIsCiAgICAgICAgICAgICJkZXRhaWwiOiBmInJlY29uc3RydWN0aW9uX2xvc3M9e21ldHJpY3MucmVjb25zdHJ1Y3Rpb25fbG9zc30iLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX3JlcHJlc2VudGF0aW9uX2RlbW8oKSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIG9iamVjdF1dOgogICAgdGhlb3J5ID0gRHluYW1pY1JlcHJlc2VudGF0aW9uVGhlb3J5KCkKICAgIGRlc2NyaXB0b3IgPSB0aGVvcnkuc2VsZWN0KHsiZWRnZXMiOiBbKCJhIiwgImIiKV0sICJoaWVyYXJjaHlfZGVwdGgiOiAzfSkKICAgIHJldHVybiB7CiAgICAgICAgInJlcHJlc2VudGF0aW9uX3NlbGVjdGlvbiI6IHsKICAgICAgICAgICAgInBhc3NlZCI6IGRlc2NyaXB0b3IuZmFtaWx5ID09IFJlcHJlc2VudGF0aW9uRmFtaWx5LlRSRUUsCiAgICAgICAgICAgICJkZXRhaWwiOiBkZXNjcmlwdG9yLmZhbWlseS52YWx1ZSwKICAgICAgICB9CiAgICB9CgoKZGVmIHJ1bl9kaXZpc2lvbl9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3RdXToKICAgIGRpdmlzaW9uID0gTmV1cmFsRGl2aXNpb25SZWFzc2VtYmx5KCkKICAgIHRhc2tzID0gWwogICAgICAgIFNwZWNpYWxpc3RUYXNrKG5hbWU9InJlc2VhcmNoIiwgcGF5bG9hZD0iY29sbGVjdCBldmlkZW5jZSIpLAogICAgICAgIFNwZWNpYWxpc3RUYXNrKG5hbWU9InBsYW5uaW5nIiwgcGF5bG9hZD0ic2VxdWVuY2UgdGhlIHdvcmsiKSwKICAgIF0KICAgIHNwZWNpYWxpc3RzID0gewogICAgICAgICJyZXNlYXJjaCI6IGxhbWJkYSBwYXlsb2FkOiBTcGVjaWFsaXN0UmVzdWx0KAogICAgICAgICAgICBuYW1lPSJyZXNlYXJjaCIsIG91dHB1dD1mInJlc2VhcmNoZWQge3BheWxvYWR9IiwgY29uZmlkZW5jZT0wLjgyCiAgICAgICAgKSwKICAgICAgICAicGxhbm5pbmciOiBsYW1iZGEgcGF5bG9hZDogU3BlY2lhbGlzdFJlc3VsdCgKICAgICAgICAgICAgbmFtZT0icGxhbm5pbmciLCBvdXRwdXQ9ZiJwbGFubmVkIHtwYXlsb2FkfSIsIGNvbmZpZGVuY2U9MC43OAogICAgICAgICksCiAgICB9CiAgICBfLCBmaW5hbCA9IGRpdmlzaW9uLmV4ZWN1dGUodGFza3MsIHNwZWNpYWxpc3RzKQogICAgcmV0dXJuIHsKICAgICAgICAiZGl2aXNpb25fcmVhc3NlbWJseSI6IHsKICAgICAgICAgICAgInBhc3NlZCI6ICJyZXNlYXJjaCIgaW4gZmluYWwub3V0cHV0IGFuZCAicGxhbm5pbmciIGluIGZpbmFsLm91dHB1dCwKICAgICAgICAgICAgImRldGFpbCI6IGZpbmFsLm91dHB1dCwKICAgICAgICB9CiAgICB9CgoKZGVmIHJ1bl9mb3JnZV9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3RdXToKICAgIGNvbXBpbGVyID0gTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcigpCiAgICBydW50aW1lID0gRm9yZ2VSdW50aW1lKCkKICAgIGNvbXBpbGF0aW9uID0gY29tcGlsZXIuY29tcGlsZSgKICAgICAgICAiTWFpbnRhaW4gc2V2ZXJhbCBwb3NzaWJsZSBpbnRlcnByZXRhdGlvbnMsIGdpdmUgZWFjaCBhIGNvbmZpZGVuY2UgIgogICAgICAgICJzY29yZSwgdXNlIHRvb2xzIHRvIHRlc3QgcHJlZGljdGlvbnMsIGFuZCBtZXJnZSB0aGUgcmVtYWluaW5nIGNvbmNsdXNpb25zLiIKICAgICkKICAgIHJlc3VsdCA9IHJ1bnRpbWUuZXhlY3V0ZShjb21waWxhdGlvbi5zcGVjLCAiQSBkZXBlbmRlbmN5IGdyYXBoIGxvb2tlZCBsaWtlIGEgZm9yZXN0IGNhbm9weS4iKQogICAgcmV0dXJuIHsKICAgICAgICAiZm9yZ2UiOiB7CiAgICAgICAgICAgICJwYXNzZWQiOiAib3V0cHV0IiBpbiByZXN1bHQub3V0cHV0cyBhbmQgbGVuKHJlc3VsdC5ldmVudHMpID49IDQsCiAgICAgICAgICAgICJkZXRhaWwiOiByZXN1bHQub3V0cHV0c1sib3V0cHV0Il1bInN0YXRlbWVudCJdLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX2hlcm1lc19kZW1vKG1lbW9yeV9wYXRoOiBQYXRoKSAtPiBIZXJtZXNSZXNwb25zZToKICAgIHdpdGggTWVtb3J5U3RvcmUobWVtb3J5X3BhdGgpIGFzIG1lbW9yeToKICAgICAgICBoZXJtZXMgPSBIZXJtZXNOYW5vKG1lbW9yeSkKICAgICAgICByZXR1cm4gaGVybWVzLnJlc3BvbmQoIlN1bW1hcml6ZSB0aGUgcHJvamVjdCBzdGF0dXMgYW5kIGtlZXAgaXQgcHJpdmF0ZS4iKQo=', 'olympus/evaluation/__init__.py': 'IiIiRXZhbHVhdGlvbiBoYXJuZXNzLiIiIgo=', 'olympus/evaluation/decision_dossiers.py': 'IiIiU291cmNlLWJvdW5kLCBwZXItZXhhbXBsZSBhdWRpdHMgZm9yIHRoZSBjb21wb3NlZCBPbHltcHVzIHJlZmVyZW5jZSBzeXN0ZW0uCgpUaGUgYWdncmVnYXRlIGV2YWx1YXRvciBkZWxpYmVyYXRlbHkgcmV0YWlucyBvbmx5IGNvbXBhY3QgcmVkdWN0aW9ucy4gIFRoaXMKbW9kdWxlIGV4cG9ydHMgYW4gaW5zcGVjdGFibGUgY29tcGFuaW9uIGFydGlmYWN0IGZvciBhIGRlY2xhcmVkIHNwbGl0IHdpdGhvdXQKc2VyaWFsaXppbmcgcmF3IHRva2VucywgaW1hZ2VzLCBvciBoaWRkZW4gc3RhdGVzLiAgQSBkb3NzaWVyIHJlcG9ydHMgd2hhdCB0aGUKbW9kZWwgZXhwb3NlZCwgd2hhdCB0aGUgaG9zdCBhbGxvd2VkLCBhbmQgaG93IGxlYXZlLW9uZS1mYW1pbHktb3V0IGxvZ2l0cyBkaWZmZXI7Cml0IGRvZXMgbm90IGNsYWltIHRoYXQgYSBmYW1pbHkgZ2F0ZSBvciBjb3VudGVyZmFjdHVhbCBpcyBhIGNhdXNhbCBleHBsYW5hdGlvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IE1hcHBpbmcsIFNlcXVlbmNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmV2YWx1YXRpb24ub2x5bXB1c19zeXN0ZW0gaW1wb3J0IG9seW1wdXNfYmF0Y2hfaW5wdXRzX3RhcmdldHMKZnJvbSBvbHltcHVzLm1vZGVscy5vbHltcHVzX3N5c3RlbSBpbXBvcnQgT2x5bXB1c1JlZmVyZW5jZVN5c3RlbQpmcm9tIG9seW1wdXMubW9kZWxzLnJ1bnRpbWUgaW1wb3J0IF9ldmFsdWF0aW9uX21vZGUKCgpjbGFzcyBPbHltcHVzRG9zc2llckJpbmRpbmcoU3RyaWN0TW9kZWwpOgogICAgIiIiSW1tdXRhYmxlIGlkZW50aXRpZXMgbmVlZGVkIHRvIGludGVycHJldCBhIGRlY2lzaW9uLWRvc3NpZXIgYnVuZGxlLiIiIgoKICAgIGRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2OiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXlswLTlhLWZdezY0fSQiKQogICAgY2hlY2twb2ludF9zaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBzcGxpdDogc3RyID0gRmllbGQocGF0dGVybj1yIl5bYS16XVthLXowLTlfLV0qJCIpCgoKY2xhc3MgT2x5bXB1c0NvdW50ZXJmYWN0dWFsRGVjaXNpb24oU3RyaWN0TW9kZWwpOgogICAgIiIiUmF3IGRlY2lzaW9uIGJlaGF2aW9yIGFmdGVyIHJlbW92aW5nIG9uZSBwcm9qZWN0ZWQgY29tcG9uZW50LiIiIgoKICAgIHJhd19wcmVkaWN0ZWRfY2xhc3M6IGludCA9IEZpZWxkKGdlPTApCiAgICByYXdfcHJlZGljdGVkX3Byb2JhYmlsaXR5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAga2xfZnJvbV9mdWxsOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCkKICAgIGNoYW5nZXNfcmF3X3ByZWRpY3Rpb246IGJvb2wKCgpjbGFzcyBPbHltcHVzRGVjaXNpb25Eb3NzaWVyKFN0cmljdE1vZGVsKToKICAgICIiIkEgY29tcGFjdCBhdWRpdCByZWNvcmQgZm9yIG9uZSBzb3VyY2UtYm91bmQgZXhhbXBsZS4iIiIKCiAgICBzb3VyY2VfaWQ6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGRlY2lzaW9uX2F2YWlsYWJsZTogYm9vbAogICAgcmF3X3ByZWRpY3RlZF9jbGFzczogaW50ID0gRmllbGQoZ2U9MCkKICAgIHJhd19wcmVkaWN0ZWRfcHJvYmFiaWxpdHk6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBleHBvc2VkX3ByZWRpY3RlZF9jbGFzczogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ2U9MCkKICAgIGV4cG9zZWRfcHJlZGljdGVkX3Byb2JhYmlsaXR5OiBmbG9hdCB8IE5vbmUgPSBGaWVsZChkZWZhdWx0PU5vbmUsIGdlPTAuMCwgbGU9MS4wKQogICAgYWxsb3dlZF90cmFuc2l0aW9uczogbGlzdFtib29sXSA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIHNlbGVjdGVkX3RyYW5zaXRpb246IGludCA9IEZpZWxkKGdlPTApCiAgICBzZWxlY3RlZF90cmFuc2l0aW9uX3Byb2JhYmlsaXR5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgZm9yY2VkX3N0b3A6IGJvb2wKICAgIGhvc3RfcmlzazogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGludHJpbnNpY19yaXNrOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgZWZmZWN0aXZlX3Jpc2s6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICB1bmNlcnRhaW50eTogZmxvYXQgPSBGaWVsZChndD0wLjApCiAgICBmYW1pbHlfZGlzYWdyZWVtZW50OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgY29tcG9uZW50X2dhdGVzOiBkaWN0W3N0ciwgZmxvYXRdID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgY291bnRlcmZhY3R1YWxzOiBkaWN0W3N0ciwgT2x5bXB1c0NvdW50ZXJmYWN0dWFsRGVjaXNpb25dID0gRmllbGQobWluX2xlbmd0aD0xKQoKICAgIEBtb2RlbF92YWxpZGF0b3IobW9kZT0iYWZ0ZXIiKQogICAgZGVmIGludGVybmFsbHlfY29uc2lzdGVudChzZWxmKSAtPiBPbHltcHVzRGVjaXNpb25Eb3NzaWVyOgogICAgICAgIGlmIHNldChzZWxmLmNvbXBvbmVudF9nYXRlcykgIT0gc2V0KHNlbGYuY291bnRlcmZhY3R1YWxzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY29tcG9uZW50IGdhdGVzIGFuZCBjb3VudGVyZmFjdHVhbHMgbXVzdCBuYW1lIHRoZSBzYW1lIGNvbXBvbmVudHMiKQogICAgICAgIGlmIGFueSh2YWx1ZSA8IDAuMCBvciB2YWx1ZSA+IDEuMCBmb3IgdmFsdWUgaW4gc2VsZi5jb21wb25lbnRfZ2F0ZXMudmFsdWVzKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb21wb25lbnQgZ2F0ZXMgbXVzdCBmYWxsIGluIFswLCAxXSIpCiAgICAgICAgaWYgbm90IGFicyhzdW0oc2VsZi5jb21wb25lbnRfZ2F0ZXMudmFsdWVzKCkpIC0gMS4wKSA8PSAxZS01OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb21wb25lbnQgZ2F0ZXMgbXVzdCBzdW0gdG8gb25lIikKICAgICAgICBpZiBzZWxmLmZvcmNlZF9zdG9wID09IHNlbGYuZGVjaXNpb25fYXZhaWxhYmxlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZWNpc2lvbiBhdmFpbGFiaWxpdHkgbXVzdCBiZSB0aGUgaW52ZXJzZSBvZiBmb3JjZWQgU1RPUCIpCiAgICAgICAgaWYgc2VsZi5mb3JjZWRfc3RvcCBhbmQgKAogICAgICAgICAgICBzZWxmLmV4cG9zZWRfcHJlZGljdGVkX2NsYXNzIGlzIG5vdCBOb25lCiAgICAgICAgICAgIG9yIHNlbGYuZXhwb3NlZF9wcmVkaWN0ZWRfcHJvYmFiaWxpdHkgaXMgbm90IE5vbmUKICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJmb3JjZWQgU1RPUCBtdXN0IG5vdCBleHBvc2UgYSBkZWNpc2lvbiIpCiAgICAgICAgaWYgbm90IHNlbGYuZm9yY2VkX3N0b3AgYW5kICgKICAgICAgICAgICAgc2VsZi5leHBvc2VkX3ByZWRpY3RlZF9jbGFzcyAhPSBzZWxmLnJhd19wcmVkaWN0ZWRfY2xhc3MKICAgICAgICAgICAgb3Igc2VsZi5leHBvc2VkX3ByZWRpY3RlZF9wcm9iYWJpbGl0eSAhPSBzZWxmLnJhd19wcmVkaWN0ZWRfcHJvYmFiaWxpdHkKICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhdmFpbGFibGUgZGVjaXNpb25zIG11c3QgZXhhY3RseSBleHBvc2UgdGhlIHJhdyBwcmVkaWN0aW9uIikKICAgICAgICBpZiBzZWxmLnNlbGVjdGVkX3RyYW5zaXRpb24gPj0gbGVuKHNlbGYuYWxsb3dlZF90cmFuc2l0aW9ucyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNlbGVjdGVkIHRyYW5zaXRpb24gZmFsbHMgb3V0c2lkZSB0aGUgaG9zdCB0cmFuc2l0aW9uIG1hc2siKQogICAgICAgIGlmIG5vdCBzZWxmLmFsbG93ZWRfdHJhbnNpdGlvbnNbc2VsZi5zZWxlY3RlZF90cmFuc2l0aW9uXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2VsZWN0ZWQgdHJhbnNpdGlvbiBpcyBmb3JiaWRkZW4gYnkgdGhlIGhvc3QgdHJhbnNpdGlvbiBtYXNrIikKICAgICAgICBpZiBzZWxmLmVmZmVjdGl2ZV9yaXNrICsgMWUtNyA8IHNlbGYuaG9zdF9yaXNrOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlZmZlY3RpdmUgcmlzayBtdXN0IG5vdCBmYWxsIGJlbG93IHRoZSBob3N0IHJpc2siKQogICAgICAgIHJldHVybiBzZWxmCgoKY2xhc3MgT2x5bXB1c0RlY2lzaW9uRG9zc2llckJ1bmRsZShTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gRmllbGQoZGVmYXVsdD0xLCBmcm96ZW49VHJ1ZSkKICAgIGJpbmRpbmc6IE9seW1wdXNEb3NzaWVyQmluZGluZwogICAgZG9zc2llcnM6IGxpc3RbT2x5bXB1c0RlY2lzaW9uRG9zc2llcl0gPSBGaWVsZChtaW5fbGVuZ3RoPTEpCiAgICBidW5kbGVfc2hhMjU2OiBzdHIgfCBOb25lID0gRmllbGQoZGVmYXVsdD1Ob25lLCBwYXR0ZXJuPXIiXlswLTlhLWZdezY0fSQiKQoKICAgIEBtb2RlbF92YWxpZGF0b3IobW9kZT0iYWZ0ZXIiKQogICAgZGVmIHVuaXF1ZV9zb3VyY2VzKHNlbGYpIC0+IE9seW1wdXNEZWNpc2lvbkRvc3NpZXJCdW5kbGU6CiAgICAgICAgc291cmNlcyA9IFtkb3NzaWVyLnNvdXJjZV9pZCBmb3IgZG9zc2llciBpbiBzZWxmLmRvc3NpZXJzXQogICAgICAgIGlmIGxlbihzb3VyY2VzKSAhPSBsZW4oc2V0KHNvdXJjZXMpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGVjaXNpb24gZG9zc2llcnMgbXVzdCBoYXZlIHVuaXF1ZSBzb3VyY2UgaWRlbnRpdGllcyIpCiAgICAgICAgaWYgc2VsZi5idW5kbGVfc2hhMjU2IGlzIG5vdCBOb25lOgogICAgICAgICAgICBleHBlY3RlZCA9IGhhc2hsaWIuc2hhMjU2KHNlbGYuY2Fub25pY2FsX2J5dGVzKGluY2x1ZGVfaGFzaD1GYWxzZSkpLmhleGRpZ2VzdCgpCiAgICAgICAgICAgIGlmIHNlbGYuYnVuZGxlX3NoYTI1NiAhPSBleHBlY3RlZDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRlY2lzaW9uLWRvc3NpZXIgYnVuZGxlIGRpZ2VzdCBkb2VzIG5vdCBtYXRjaCBpdHMgY29udGVudHMiKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIGNhbm9uaWNhbF9ieXRlcyhzZWxmLCAqLCBpbmNsdWRlX2hhc2g6IGJvb2wgPSBUcnVlKSAtPiBieXRlczoKICAgICAgICBwYXlsb2FkID0gc2VsZi5tb2RlbF9kdW1wKG1vZGU9Impzb24iKQogICAgICAgIGlmIG5vdCBpbmNsdWRlX2hhc2g6CiAgICAgICAgICAgIHBheWxvYWRbImJ1bmRsZV9zaGEyNTYiXSA9IE5vbmUKICAgICAgICByZXR1cm4gKGpzb24uZHVtcHMocGF5bG9hZCwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkgKyAiXG4iKS5lbmNvZGUoKQoKCmRlZiByZWFkX29seW1wdXNfZGVjaXNpb25fZG9zc2llcl9idW5kbGUoCiAgICBwYXRoOiBQYXRoLAogICAgKiwKICAgIGV4cGVjdGVkX2JpbmRpbmc6IE9seW1wdXNEb3NzaWVyQmluZGluZywKICAgIGV4cGVjdGVkX3NvdXJjZV9pZHM6IFNlcXVlbmNlW3N0cl0sCiAgICBleHBlY3RlZF9maWxlX3NoYTI1Njogc3RyIHwgTm9uZSA9IE5vbmUsCikgLT4gT2x5bXB1c0RlY2lzaW9uRG9zc2llckJ1bmRsZToKICAgICIiIlJlYWQgb25lIGltbXV0YWJsZSBidW5kbGUgYW5kIHZlcmlmeSBpdHMgZXh0ZXJuYWwgYW5kIHNlbWFudGljIGJpbmRpbmdzLiIiIgoKICAgIGlmIHBhdGguaXNfc3ltbGluaygpIG9yIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZWNpc2lvbi1kb3NzaWVyIGFydGlmYWN0IG11c3QgYmUgYSByZWd1bGFyIG5vbi1zeW1saW5rIGZpbGUiKQogICAgcGF5bG9hZCA9IHBhdGgucmVhZF9ieXRlcygpCiAgICBhY3R1YWxfZmlsZV9zaGEyNTYgPSBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3QoKQogICAgaWYgZXhwZWN0ZWRfZmlsZV9zaGEyNTYgaXMgbm90IE5vbmUgYW5kIGFjdHVhbF9maWxlX3NoYTI1NiAhPSBleHBlY3RlZF9maWxlX3NoYTI1NjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZWNpc2lvbi1kb3NzaWVyIGZpbGUgZGlnZXN0IGRvZXMgbm90IG1hdGNoIHRoZSByZXBvcnQiKQogICAgYnVuZGxlID0gT2x5bXB1c0RlY2lzaW9uRG9zc2llckJ1bmRsZS5tb2RlbF92YWxpZGF0ZV9qc29uKHBheWxvYWQpCiAgICBpZiBidW5kbGUuYmluZGluZyAhPSBleHBlY3RlZF9iaW5kaW5nOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRlY2lzaW9uLWRvc3NpZXIgYmluZGluZyBkb2VzIG5vdCBtYXRjaCB0aGUgZXhwZWN0ZWQgcnVuIikKICAgIGlmIFtkb3NzaWVyLnNvdXJjZV9pZCBmb3IgZG9zc2llciBpbiBidW5kbGUuZG9zc2llcnNdICE9IGxpc3QoZXhwZWN0ZWRfc291cmNlX2lkcyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGVjaXNpb24tZG9zc2llciBzb3VyY2VzIGRvIG5vdCBtYXRjaCB0aGUgZXhwZWN0ZWQgc3BsaXQiKQogICAgcmV0dXJuIGJ1bmRsZQoKCmRlZiB2ZXJpZnlfb2x5bXB1c19kZWNpc2lvbl9kb3NzaWVyX3JlcGxheSgKICAgIG1vZGVsOiBPbHltcHVzUmVmZXJlbmNlU3lzdGVtLAogICAgYmF0Y2g6IE1hcHBpbmdbc3RyLCBUZW5zb3JdLAogICAgZXh0ZW5zaW9uczogU2VxdWVuY2Vbc3RyXSwKICAgIHNvdXJjZV9pZHM6IFNlcXVlbmNlW3N0cl0sCiAgICAqLAogICAgYmluZGluZzogT2x5bXB1c0Rvc3NpZXJCaW5kaW5nLAogICAgb2JzZXJ2ZWQ6IE9seW1wdXNEZWNpc2lvbkRvc3NpZXJCdW5kbGUsCikgLT4gTm9uZToKICAgICIiIlJlY29tcHV0ZSBldmVyeSBhdHRyaWJ1dGlvbiBmcm9tIHRoZSBkZWNsYXJlZCBjaGVja3BvaW50IGFuZCBzcGxpdC4KCiAgICBUaGlzIGRlbGliZXJhdGVseSByZXF1aXJlcyBieXRlLWlkZW50aWNhbCBjYW5vbmljYWwgdmFsdWVzLiAgQSB2ZXJpZmllciBtdXN0CiAgICByZXBsYXkgb24gdGhlIHJ1bidzIGRlY2xhcmVkIGRldmljZS9ydW50aW1lIHJhdGhlciB0aGFuIHNpbGVudGx5IGFjY2VwdGluZyBhCiAgICBudW1lcmljYWxseSBzaW1pbGFyIGJ1bmRsZSBwcm9kdWNlZCBieSBkaWZmZXJlbnQgc29mdHdhcmUgb3IgaGFyZHdhcmUuCiAgICAiIiIKCiAgICByZXBsYXllZCA9IE9seW1wdXNEZWNpc2lvbkRvc3NpZXJCdW5kbGUoCiAgICAgICAgYmluZGluZz1iaW5kaW5nLAogICAgICAgIGRvc3NpZXJzPWJ1aWxkX29seW1wdXNfZGVjaXNpb25fZG9zc2llcnMoCiAgICAgICAgICAgIG1vZGVsLAogICAgICAgICAgICBiYXRjaCwKICAgICAgICAgICAgZXh0ZW5zaW9ucywKICAgICAgICAgICAgc291cmNlX2lkcywKICAgICAgICApLAogICAgKQogICAgcmVwbGF5ZWQuYnVuZGxlX3NoYTI1NiA9IGhhc2hsaWIuc2hhMjU2KAogICAgICAgIHJlcGxheWVkLmNhbm9uaWNhbF9ieXRlcyhpbmNsdWRlX2hhc2g9RmFsc2UpCiAgICApLmhleGRpZ2VzdCgpCiAgICBpZiBvYnNlcnZlZC5jYW5vbmljYWxfYnl0ZXMoKSAhPSByZXBsYXllZC5jYW5vbmljYWxfYnl0ZXMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAiZGVjaXNpb24tZG9zc2llciByZXBsYXkgZG9lcyBub3QgbWF0Y2ggdGhlIHNlbGVjdGVkIGNoZWNrcG9pbnQgYW5kIHNwbGl0IgogICAgICAgICkKCgpkZWYgYnVpbGRfb2x5bXB1c19kZWNpc2lvbl9kb3NzaWVycygKICAgIG1vZGVsOiBPbHltcHVzUmVmZXJlbmNlU3lzdGVtLAogICAgYmF0Y2g6IE1hcHBpbmdbc3RyLCBUZW5zb3JdLAogICAgZXh0ZW5zaW9uczogU2VxdWVuY2Vbc3RyXSwKICAgIHNvdXJjZV9pZHM6IFNlcXVlbmNlW3N0cl0sCikgLT4gbGlzdFtPbHltcHVzRGVjaXNpb25Eb3NzaWVyXToKICAgICIiIlByb2R1Y2UgQ1BVLW5hdGl2ZSBkb3NzaWVycyB3aXRob3V0IHJldGFpbmluZyBtb2RlbCBhY3RpdmF0aW9ucy4KCiAgICBgYHNvdXJjZV9pZHNgYCBtdXN0IGJlIHRoZSBvcmRlcmVkIHNvdXJjZSBpZGVudGl0aWVzIGZyb20gdGhlIGV4YWN0IHNwbGl0CiAgICBtYW5pZmVzdC4gIFRoaXMgZnVuY3Rpb24gaW50ZW50aW9uYWxseSBkb2VzIG5vdCBhY2NlcHQgYXJiaXRyYXJ5IGxhYmVscyBvcgogICAgcmF3IHByb3ZlbmFuY2UgdGV4dCwgcHJldmVudGluZyBhbiBpbnNwZWN0aW9uIGFydGlmYWN0IGZyb20gc2lsZW50bHkgYmVpbmcKICAgIGRpc2Nvbm5lY3RlZCBmcm9tIHRoZSBldmFsdWF0ZWQgdGVuc29yIGJ1bmRsZS4KICAgICIiIgoKICAgIGNvdW50ID0gYmF0Y2hbInRva2VucyJdLnNoYXBlWzBdCiAgICBpZiAoCiAgICAgICAgbGVuKHNvdXJjZV9pZHMpICE9IGNvdW50CiAgICAgICAgb3IgbGVuKHNldChzb3VyY2VfaWRzKSkgIT0gY291bnQKICAgICAgICBvciBhbnkobm90IHZhbHVlIGZvciB2YWx1ZSBpbiBzb3VyY2VfaWRzKQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzb3VyY2UgaWRlbnRpdGllcyBtdXN0IGJlIHVuaXF1ZSBhbmQgYWxpZ24gd2l0aCB0aGUgdGVuc29yIGJhdGNoIikKICAgIHBhcmFtZXRlciA9IG5leHQobW9kZWwucGFyYW1ldGVycygpKQogICAgZGV2aWNlID0gcGFyYW1ldGVyLmRldmljZQogICAgc2VsZWN0ZWQgPSB7CiAgICAgICAgbmFtZTogdGVuc29yLnRvKAogICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICBkdHlwZT1wYXJhbWV0ZXIuZHR5cGUgaWYgdGVuc29yLmlzX2Zsb2F0aW5nX3BvaW50KCkgZWxzZSB0ZW5zb3IuZHR5cGUsCiAgICAgICAgKQogICAgICAgIGZvciBuYW1lLCB0ZW5zb3IgaW4gYmF0Y2guaXRlbXMoKQogICAgfQogICAgaW5wdXRzLCBfID0gb2x5bXB1c19iYXRjaF9pbnB1dHNfdGFyZ2V0cyhzZWxlY3RlZCwgZXh0ZW5zaW9ucykKICAgIHdpdGggX2V2YWx1YXRpb25fbW9kZShtb2RlbCksIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgb3V0cHV0ID0gbW9kZWwoKippbnB1dHMpCgogICAgZnVsbF9wcm9iYWJpbGl0aWVzID0gb3V0cHV0LnJhd19kZWNpc2lvbl9sb2dpdHMuc29mdG1heChkaW09LTEpLmRldGFjaCgpLmNwdSgpCiAgICBjb3VudGVyZmFjdHVhbF9wcm9iYWJpbGl0aWVzID0gKAogICAgICAgIG91dHB1dC5jb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHMuc29mdG1heChkaW09LTEpLmRldGFjaCgpLmNwdSgpCiAgICApCiAgICBjb3VudGVyZmFjdHVhbF9sb2dfcHJvYmFiaWxpdGllcyA9ICgKICAgICAgICBvdXRwdXQuY291bnRlcmZhY3R1YWxfZGVjaXNpb25fbG9naXRzLmxvZ19zb2Z0bWF4KGRpbT0tMSkuZGV0YWNoKCkuY3B1KCkKICAgICkKICAgIGZ1bGxfY3B1ID0gZnVsbF9wcm9iYWJpbGl0aWVzCiAgICBjb3VudGVyZmFjdHVhbF9rbCA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwua2xfZGl2KAogICAgICAgIGNvdW50ZXJmYWN0dWFsX2xvZ19wcm9iYWJpbGl0aWVzLAogICAgICAgIGZ1bGxfY3B1LnVuc3F1ZWV6ZSgxKS5leHBhbmRfYXMoY291bnRlcmZhY3R1YWxfbG9nX3Byb2JhYmlsaXRpZXMpLAogICAgICAgIHJlZHVjdGlvbj0ibm9uZSIsCiAgICApLnN1bShkaW09LTEpCiAgICBnYXRlcyA9IG91dHB1dC5mYW1pbHlfZ2F0ZXMuZGV0YWNoKCkuY3B1KCkKICAgIGRlY2lzaW9uX2F2YWlsYWJsZSA9IG91dHB1dC5kZWNpc2lvbl9hdmFpbGFibGUuZGV0YWNoKCkuY3B1KCkKICAgIGZvcmNlZF9zdG9wID0gb3V0cHV0LmFpb24uZm9yY2VkX3N0b3AuZGV0YWNoKCkuY3B1KCkKICAgIHNlbGVjdGVkX3RyYW5zaXRpb24gPSBvdXRwdXQuYWlvbi5zZWxlY3RlZF9zdGFnZS5kZXRhY2goKS5jcHUoKQogICAgdHJhbnNpdGlvbl9wcm9iYWJpbGl0aWVzID0gb3V0cHV0LmFpb24udHJhbnNpdGlvbl9wcm9iYWJpbGl0aWVzLmRldGFjaCgpLmNwdSgpCiAgICBhbGxvd2VkX3RyYW5zaXRpb25zID0gYmF0Y2hbImFsbG93ZWRfdHJhbnNpdGlvbnMiXS5kZXRhY2goKS5jcHUoKQogICAgaG9zdF9yaXNrID0gYmF0Y2hbInJpc2tfc2NvcmUiXS5kZXRhY2goKS5jcHUoKQogICAgaW50cmluc2ljX3Jpc2sgPSBvdXRwdXQuaW50cmluc2ljX3Jpc2suZGV0YWNoKCkuY3B1KCkKICAgIGVmZmVjdGl2ZV9yaXNrID0gb3V0cHV0LmVmZmVjdGl2ZV9yaXNrLmRldGFjaCgpLmNwdSgpCiAgICB1bmNlcnRhaW50eSA9IG91dHB1dC51bmNlcnRhaW50eS5kZXRhY2goKS5jcHUoKQogICAgZGlzYWdyZWVtZW50ID0gb3V0cHV0LmZhbWlseV9kaXNhZ3JlZW1lbnQuZGV0YWNoKCkuY3B1KCkKCiAgICBkb3NzaWVyczogbGlzdFtPbHltcHVzRGVjaXNpb25Eb3NzaWVyXSA9IFtdCiAgICBmb3IgaW5kZXgsIHNvdXJjZV9pZCBpbiBlbnVtZXJhdGUoc291cmNlX2lkcyk6CiAgICAgICAgcmF3X2NsYXNzID0gaW50KGZ1bGxfcHJvYmFiaWxpdGllc1tpbmRleF0uYXJnbWF4KCkuaXRlbSgpKQogICAgICAgIHJhd19wcm9iYWJpbGl0eSA9IGZsb2F0KGZ1bGxfcHJvYmFiaWxpdGllc1tpbmRleCwgcmF3X2NsYXNzXS5pdGVtKCkpCiAgICAgICAgdHJhbnNpdGlvbiA9IGludChzZWxlY3RlZF90cmFuc2l0aW9uW2luZGV4XS5pdGVtKCkpCiAgICAgICAgYXZhaWxhYmxlID0gYm9vbChkZWNpc2lvbl9hdmFpbGFibGVbaW5kZXhdLml0ZW0oKSkKICAgICAgICBjb21wb25lbnRfZ2F0ZXMgPSB7CiAgICAgICAgICAgIG5hbWU6IGZsb2F0KGdhdGVzW2luZGV4LCBjb21wb25lbnRfaW5kZXhdLml0ZW0oKSkKICAgICAgICAgICAgZm9yIGNvbXBvbmVudF9pbmRleCwgbmFtZSBpbiBlbnVtZXJhdGUob3V0cHV0LmNvbXBvbmVudF9uYW1lcykKICAgICAgICB9CiAgICAgICAgY291bnRlcmZhY3R1YWxzID0ge30KICAgICAgICBmb3IgY29tcG9uZW50X2luZGV4LCBuYW1lIGluIGVudW1lcmF0ZShvdXRwdXQuY29tcG9uZW50X25hbWVzKToKICAgICAgICAgICAgcHJvYmFiaWxpdGllcyA9IGNvdW50ZXJmYWN0dWFsX3Byb2JhYmlsaXRpZXNbaW5kZXgsIGNvbXBvbmVudF9pbmRleF0KICAgICAgICAgICAgY291bnRlcmZhY3R1YWxfY2xhc3MgPSBpbnQocHJvYmFiaWxpdGllcy5hcmdtYXgoKS5pdGVtKCkpCiAgICAgICAgICAgIGNvdW50ZXJmYWN0dWFsc1tuYW1lXSA9IE9seW1wdXNDb3VudGVyZmFjdHVhbERlY2lzaW9uKAogICAgICAgICAgICAgICAgcmF3X3ByZWRpY3RlZF9jbGFzcz1jb3VudGVyZmFjdHVhbF9jbGFzcywKICAgICAgICAgICAgICAgIHJhd19wcmVkaWN0ZWRfcHJvYmFiaWxpdHk9ZmxvYXQocHJvYmFiaWxpdGllc1tjb3VudGVyZmFjdHVhbF9jbGFzc10uaXRlbSgpKSwKICAgICAgICAgICAgICAgIGtsX2Zyb21fZnVsbD1mbG9hdChjb3VudGVyZmFjdHVhbF9rbFtpbmRleCwgY29tcG9uZW50X2luZGV4XS5pdGVtKCkpLAogICAgICAgICAgICAgICAgY2hhbmdlc19yYXdfcHJlZGljdGlvbj1jb3VudGVyZmFjdHVhbF9jbGFzcyAhPSByYXdfY2xhc3MsCiAgICAgICAgICAgICkKICAgICAgICBkb3NzaWVycy5hcHBlbmQoCiAgICAgICAgICAgIE9seW1wdXNEZWNpc2lvbkRvc3NpZXIoCiAgICAgICAgICAgICAgICBzb3VyY2VfaWQ9c3RyKHNvdXJjZV9pZCksCiAgICAgICAgICAgICAgICBkZWNpc2lvbl9hdmFpbGFibGU9YXZhaWxhYmxlLAogICAgICAgICAgICAgICAgcmF3X3ByZWRpY3RlZF9jbGFzcz1yYXdfY2xhc3MsCiAgICAgICAgICAgICAgICByYXdfcHJlZGljdGVkX3Byb2JhYmlsaXR5PXJhd19wcm9iYWJpbGl0eSwKICAgICAgICAgICAgICAgIGV4cG9zZWRfcHJlZGljdGVkX2NsYXNzPXJhd19jbGFzcyBpZiBhdmFpbGFibGUgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgZXhwb3NlZF9wcmVkaWN0ZWRfcHJvYmFiaWxpdHk9cmF3X3Byb2JhYmlsaXR5IGlmIGF2YWlsYWJsZSBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICBhbGxvd2VkX3RyYW5zaXRpb25zPVtib29sKHZhbHVlKSBmb3IgdmFsdWUgaW4gYWxsb3dlZF90cmFuc2l0aW9uc1tpbmRleF0udG9saXN0KCldLAogICAgICAgICAgICAgICAgc2VsZWN0ZWRfdHJhbnNpdGlvbj10cmFuc2l0aW9uLAogICAgICAgICAgICAgICAgc2VsZWN0ZWRfdHJhbnNpdGlvbl9wcm9iYWJpbGl0eT1mbG9hdCgKICAgICAgICAgICAgICAgICAgICB0cmFuc2l0aW9uX3Byb2JhYmlsaXRpZXNbaW5kZXgsIHRyYW5zaXRpb25dLml0ZW0oKQogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgIGZvcmNlZF9zdG9wPWJvb2woZm9yY2VkX3N0b3BbaW5kZXhdLml0ZW0oKSksCiAgICAgICAgICAgICAgICBob3N0X3Jpc2s9ZmxvYXQoaG9zdF9yaXNrW2luZGV4XS5pdGVtKCkpLAogICAgICAgICAgICAgICAgaW50cmluc2ljX3Jpc2s9ZmxvYXQoaW50cmluc2ljX3Jpc2tbaW5kZXhdLml0ZW0oKSksCiAgICAgICAgICAgICAgICBlZmZlY3RpdmVfcmlzaz1mbG9hdChlZmZlY3RpdmVfcmlza1tpbmRleF0uaXRlbSgpKSwKICAgICAgICAgICAgICAgIHVuY2VydGFpbnR5PWZsb2F0KHVuY2VydGFpbnR5W2luZGV4XS5pdGVtKCkpLAogICAgICAgICAgICAgICAgZmFtaWx5X2Rpc2FncmVlbWVudD1mbG9hdChkaXNhZ3JlZW1lbnRbaW5kZXhdLml0ZW0oKSksCiAgICAgICAgICAgICAgICBjb21wb25lbnRfZ2F0ZXM9Y29tcG9uZW50X2dhdGVzLAogICAgICAgICAgICAgICAgY291bnRlcmZhY3R1YWxzPWNvdW50ZXJmYWN0dWFscywKICAgICAgICAgICAgKQogICAgICAgICkKICAgIHJldHVybiBkb3NzaWVycwoKCmRlZiB3cml0ZV9vbHltcHVzX2RlY2lzaW9uX2Rvc3NpZXJzKAogICAgcGF0aDogUGF0aCwKICAgICosCiAgICBiaW5kaW5nOiBPbHltcHVzRG9zc2llckJpbmRpbmcsCiAgICBkb3NzaWVyczogU2VxdWVuY2VbT2x5bXB1c0RlY2lzaW9uRG9zc2llcl0sCikgLT4gT2x5bXB1c0RlY2lzaW9uRG9zc2llckJ1bmRsZToKICAgICIiIkF0b21pY2FsbHkgZXhwb3J0IGEgaGFzaC1ib3VuZCBkb3NzaWVyIGJ1bmRsZSB3aXRob3V0IG92ZXJ3cml0aW5nIG9uZS4iIiIKCiAgICBpZiBwYXRoLmV4aXN0cygpIG9yIHBhdGguaXNfc3ltbGluaygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcigicmVmdXNpbmcgdG8gb3ZlcndyaXRlIGFuIGV4aXN0aW5nIGRlY2lzaW9uLWRvc3NpZXIgYnVuZGxlIikKICAgIGJ1bmRsZSA9IE9seW1wdXNEZWNpc2lvbkRvc3NpZXJCdW5kbGUoYmluZGluZz1iaW5kaW5nLCBkb3NzaWVycz1saXN0KGRvc3NpZXJzKSkKICAgIGJ1bmRsZS5idW5kbGVfc2hhMjU2ID0gaGFzaGxpYi5zaGEyNTYoYnVuZGxlLmNhbm9uaWNhbF9ieXRlcyhpbmNsdWRlX2hhc2g9RmFsc2UpKS5oZXhkaWdlc3QoKQogICAgcGF5bG9hZCA9IGJ1bmRsZS5jYW5vbmljYWxfYnl0ZXMoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxlLndyaXRlKHBheWxvYWQpCiAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5ID0gUGF0aChoYW5kbGUubmFtZSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCiAgICByZXR1cm4gYnVuZGxlCg==', 'olympus/evaluation/harness.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBjYXN0Cgpmcm9tIG9seW1wdXMuZGVtb3MgaW1wb3J0ICgKICAgIHJ1bl9hbWJpZ3VvdXNfaW50ZXJwcmV0YXRpb25fZGVtbywKICAgIHJ1bl9mb3JnZV9kZW1vLAogICAgcnVuX3JlcHJlc2VudGF0aW9uX2RlbW8sCiAgICBydW5fcmV0cm9kaWN0aW9uX2RlbW8sCiAgICBydW5fdHJhbnNmb3JtX2RlbW8sCikKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIEV2YWx1YXRpb25SZXN1bHQ6CiAgICBuYW1lOiBzdHIKICAgIHBhc3NlZDogYm9vbAogICAgZGV0YWlsOiBzdHIKCgpjbGFzcyBFdmFsdWF0aW9uSGFybmVzczoKICAgIGRlZiBydW4oc2VsZikgLT4gbGlzdFtFdmFsdWF0aW9uUmVzdWx0XToKICAgICAgICByZXN1bHRzID0gWwogICAgICAgICAgICBydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8oKSwKICAgICAgICAgICAgcnVuX3JldHJvZGljdGlvbl9kZW1vKCksCiAgICAgICAgICAgIHJ1bl90cmFuc2Zvcm1fZGVtbygpLAogICAgICAgICAgICBydW5fcmVwcmVzZW50YXRpb25fZGVtbygpLAogICAgICAgICAgICBydW5fZm9yZ2VfZGVtbygpLAogICAgICAgIF0KICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBFdmFsdWF0aW9uUmVzdWx0KAogICAgICAgICAgICAgICAgbmFtZT1uYW1lLAogICAgICAgICAgICAgICAgcGFzc2VkPWNhc3QoYm9vbCwgcGF5bG9hZFsicGFzc2VkIl0pLAogICAgICAgICAgICAgICAgZGV0YWlsPWNhc3Qoc3RyLCBwYXlsb2FkWyJkZXRhaWwiXSksCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIGRlbW8gaW4gcmVzdWx0cwogICAgICAgICAgICBmb3IgbmFtZSwgcGF5bG9hZCBpbiBkZW1vLml0ZW1zKCkKICAgICAgICBdCg==', 'olympus/evaluation/olympus_system.py': 'IiIiSGVsZC1vdXQgZGlhZ25vc3RpY3MgZm9yIHRoZSBjb21wb3NlZCBPbHltcHVzIHJlZmVyZW5jZSBzeXN0ZW0uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgTWFwcGluZywgU2VxdWVuY2UKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZApmcm9tIHRvcmNoIGltcG9ydCBUZW5zb3IsIG5uCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbApmcm9tIG9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIGltcG9ydCAoCiAgICBPbHltcHVzT2JqZWN0aXZlT3V0cHV0LAogICAgT2x5bXB1c1JlZmVyZW5jZVN5c3RlbSwKICAgIE9seW1wdXNTeXN0ZW1Mb3NzQ29uZmlnLAogICAgT2x5bXB1c1N5c3RlbU91dHB1dCwKICAgIG9seW1wdXNfc3lzdGVtX2xvc3MsCikKZnJvbSBvbHltcHVzLm1vZGVscy5yZWZlcmVuY2VfZmFtaWxpZXMgaW1wb3J0IEFpb25SZWZlcmVuY2VPdXRwdXQKZnJvbSBvbHltcHVzLm1vZGVscy5ydW50aW1lIGltcG9ydCBSdW50aW1lSW5wdXQsIF9ldmFsdWF0aW9uX21vZGUKCl9JTlBVVF9OQU1FUyA9IGZyb3plbnNldCgKICAgIHsKICAgICAgICAidG9rZW5zIiwKICAgICAgICAiaW1hZ2UiLAogICAgICAgICJzdGF0ZXMiLAogICAgICAgICJhY3Rpb25zIiwKICAgICAgICAiYWxsb3dlZF90cmFuc2l0aW9ucyIsCiAgICAgICAgInJpc2tfc2NvcmUiLAogICAgICAgICJhdHRlbnRpb25fbWFzayIsCiAgICAgICAgImRvbWFpbl9pZHMiLAogICAgICAgICJ0ZW1wb3JhbF9tYXNrIiwKICAgICAgICAidGltZXN0YW1wIiwKICAgIH0KKQpfUkVRVUlSRURfVEFSR0VUUyA9IGZyb3plbnNldCh7ImRlY2lzaW9uX3RhcmdldHMiLCAidHJhbnNpdGlvbl90YXJnZXRzIiwgIm5leHRfc3RhdGVfdGFyZ2V0cyJ9KQoKCmNsYXNzIFJpc2tDb3ZlcmFnZVBvaW50KFN0cmljdE1vZGVsKToKICAgIGNvdmVyYWdlOiBmbG9hdCA9IEZpZWxkKGd0PTAuMCwgbGU9MS4wKQogICAgcmV0YWluZWRfZXhhbXBsZXM6IGludCA9IEZpZWxkKGd0PTApCiAgICBhY2N1cmFjeTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIG1heGltdW1fZWZmZWN0aXZlX3Jpc2s6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICAjIEhpc3RvcmljYWwgc2NoZW1hLTEgcmVwb3J0cyBvbWl0dGVkIHRoaXMgZGVub21pbmF0b3I7IGRvIG5vdCBpbnZlbnQgaXQgb24gbG9hZC4KICAgIHBvcHVsYXRpb25fY292ZXJhZ2U6IGZsb2F0IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9MC4wLCBsZT0xLjApCgoKY2xhc3MgT2x5bXB1c1JvdXRpbmdEaWFnbm9zdGljcyhTdHJpY3RNb2RlbCk6CiAgICBtZWFuX2dhdGVzOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBub3JtYWxpemVkX2dhdGVfZW50cm9weTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGVmZmVjdGl2ZV9jb21wb25lbnRzOiBmbG9hdCA9IEZpZWxkKGdlPTEuMCkKICAgIGNvdW50ZXJmYWN0dWFsX2tsOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBjb3VudGVyZmFjdHVhbF9mbGlwX3JhdGU6IGRpY3Rbc3RyLCBmbG9hdF0KCgpjbGFzcyBPbHltcHVzUmlza0RpYWdub3N0aWNzKFN0cmljdE1vZGVsKToKICAgIG1lYW5faW50cmluc2ljX3Jpc2s6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBtZWFuX2VmZmVjdGl2ZV9yaXNrOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgaW50cmluc2ljX2JyaWVyOiBmbG9hdCB8IE5vbmUgPSBGaWVsZChkZWZhdWx0PU5vbmUsIGdlPTAuMCkKICAgIGludHJpbnNpY19lY2U6IGZsb2F0IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ2U9MC4wLCBsZT0xLjApCiAgICByaXNrX2NvdmVyYWdlOiBsaXN0W1Jpc2tDb3ZlcmFnZVBvaW50XQoKCmNsYXNzIE9seW1wdXNTeXN0ZW1FdmFsdWF0aW9uKFN0cmljdE1vZGVsKToKICAgIGV4YW1wbGVzOiBpbnQgPSBGaWVsZChndD0wKQogICAgZGVjaXNpb25fYXZhaWxhYmxlX2V4YW1wbGVzOiBpbnQgPSBGaWVsZChnZT0wKQogICAgZm9yY2VkX3N0b3BfZXhhbXBsZXM6IGludCA9IEZpZWxkKGdlPTApCiAgICB0b3RhbF9sb3NzOiBmbG9hdAogICAgbG9zc19jb21wb25lbnRzOiBkaWN0W3N0ciwgZmxvYXRdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBkZWNpc2lvbl9hY2N1cmFjeTogZmxvYXQgfCBOb25lID0gRmllbGQoZGVmYXVsdD1Ob25lLCBnZT0wLjAsIGxlPTEuMCkKICAgIHRyYW5zaXRpb25fYWNjdXJhY3k6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBuZXh0X3N0YXRlX21zZTogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBtZWFuX3VuY2VydGFpbnR5OiBmbG9hdCA9IEZpZWxkKGd0PTAuMCkKICAgIHJvdXRpbmc6IE9seW1wdXNSb3V0aW5nRGlhZ25vc3RpY3MKICAgIHJpc2s6IE9seW1wdXNSaXNrRGlhZ25vc3RpY3MKCgpkZWYgb2x5bXB1c19iYXRjaF9pbnB1dHNfdGFyZ2V0cygKICAgIGJhdGNoOiBNYXBwaW5nW3N0ciwgVGVuc29yXSwgZXh0ZW5zaW9uczogU2VxdWVuY2Vbc3RyXQopIC0+IHR1cGxlW2RpY3Rbc3RyLCBSdW50aW1lSW5wdXRdLCBkaWN0W3N0ciwgVGVuc29yXV06CiAgICAiIiJTcGxpdCBvbmUgdmVyaWZpZWQgdGVuc29yIGJ1bmRsZSBpbnRvIHRoZSBtb2RlbCBhbmQgb2JqZWN0aXZlIGNvbnRyYWN0cy4iIiIKCiAgICBtaXNzaW5nX3RhcmdldHMgPSBzb3J0ZWQoX1JFUVVJUkVEX1RBUkdFVFMgLSBzZXQoYmF0Y2gpKQogICAgaWYgbWlzc2luZ190YXJnZXRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJPbHltcHVzIGJhdGNoIGlzIG1pc3NpbmcgdGFyZ2V0czoge21pc3NpbmdfdGFyZ2V0c30iKQogICAgaW5wdXRzOiBkaWN0W3N0ciwgUnVudGltZUlucHV0XSA9IHsKICAgICAgICBuYW1lOiB2YWx1ZSBmb3IgbmFtZSwgdmFsdWUgaW4gYmF0Y2guaXRlbXMoKSBpZiBuYW1lIGluIF9JTlBVVF9OQU1FUwogICAgfQogICAgaW5wdXRzWyJleHRlbnNpb25fZmVhdHVyZXMiXSA9IHtuYW1lOiBiYXRjaFtmImV4dGVuc2lvbl9fe25hbWV9Il0gZm9yIG5hbWUgaW4gZXh0ZW5zaW9uc30KICAgIHRhcmdldHMgPSB7bmFtZTogYmF0Y2hbbmFtZV0gZm9yIG5hbWUgaW4gX1JFUVVJUkVEX1RBUkdFVFN9CiAgICBpZiAiaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyIgaW4gYmF0Y2g6CiAgICAgICAgdGFyZ2V0c1siaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyJdID0gYmF0Y2hbImludHJpbnNpY19yaXNrX3RhcmdldHMiXQogICAgcmV0dXJuIGlucHV0cywgdGFyZ2V0cwoKCmRlZiBfZXhwZWN0ZWRfY2FsaWJyYXRpb25fZXJyb3IocHJlZGljdGlvbnM6IFRlbnNvciwgdGFyZ2V0czogVGVuc29yLCAqLCBiaW5zOiBpbnQgPSAxMCkgLT4gZmxvYXQ6CiAgICBib3VuZGFyaWVzID0gdG9yY2gubGluc3BhY2UoMC4wLCAxLjAsIGJpbnMgKyAxLCBkZXZpY2U9cHJlZGljdGlvbnMuZGV2aWNlKQogICAgZXJyb3IgPSBwcmVkaWN0aW9ucy5uZXdfemVyb3MoKCkpCiAgICBmb3IgaW5kZXggaW4gcmFuZ2UoYmlucyk6CiAgICAgICAgbG93ZXIgPSBib3VuZGFyaWVzW2luZGV4XQogICAgICAgIHVwcGVyID0gYm91bmRhcmllc1tpbmRleCArIDFdCiAgICAgICAgc2VsZWN0ZWQgPSAocHJlZGljdGlvbnMgPj0gbG93ZXIpICYgKAogICAgICAgICAgICBwcmVkaWN0aW9ucyA8PSB1cHBlciBpZiBpbmRleCA9PSBiaW5zIC0gMSBlbHNlIHByZWRpY3Rpb25zIDwgdXBwZXIKICAgICAgICApCiAgICAgICAgaWYgYm9vbChzZWxlY3RlZC5hbnkoKSk6CiAgICAgICAgICAgIGdhcCA9IChwcmVkaWN0aW9uc1tzZWxlY3RlZF0ubWVhbigpIC0gdGFyZ2V0c1tzZWxlY3RlZF0ubWVhbigpKS5hYnMoKQogICAgICAgICAgICBlcnJvciA9IGVycm9yICsgc2VsZWN0ZWQuZmxvYXQoKS5tZWFuKCkgKiBnYXAKICAgIHJldHVybiBmbG9hdChlcnJvci5pdGVtKCkpCgoKZGVmIF9yaXNrX2NvdmVyYWdlKAogICAgb3V0cHV0OiBPbHltcHVzU3lzdGVtT3V0cHV0IHwgT2x5bXB1c09iamVjdGl2ZU91dHB1dCwgZGVjaXNpb25fdGFyZ2V0czogVGVuc29yCikgLT4gbGlzdFtSaXNrQ292ZXJhZ2VQb2ludF06CiAgICBhY3RpdmUgPSBvdXRwdXQuZGVjaXNpb25fYXZhaWxhYmxlCiAgICBpZiBub3QgYm9vbChhY3RpdmUuYW55KCkpOgogICAgICAgIHJldHVybiBbXQogICAgY29ycmVjdCA9IG91dHB1dC5yYXdfZGVjaXNpb25fbG9naXRzLmFyZ21heChkaW09LTEpLmVxKGRlY2lzaW9uX3RhcmdldHMpW2FjdGl2ZV0KICAgIHJpc2tzID0gb3V0cHV0LmVmZmVjdGl2ZV9yaXNrW2FjdGl2ZV0KICAgIG9yZGVyID0gcmlza3MuYXJnc29ydChzdGFibGU9VHJ1ZSkKICAgIG9yZGVyZWRfY29ycmVjdCA9IGNvcnJlY3Rbb3JkZXJdLmZsb2F0KCkKICAgIG9yZGVyZWRfcmlzayA9IHJpc2tzW29yZGVyXQogICAgcmVzdWx0OiBsaXN0W1Jpc2tDb3ZlcmFnZVBvaW50XSA9IFtdCiAgICBmb3IgY292ZXJhZ2UgaW4gKDAuMjUsIDAuNSwgMC43NSwgMS4wKToKICAgICAgICByZXRhaW5lZCA9IG1heCgxLCBtYXRoLmNlaWwoY292ZXJhZ2UgKiBvcmRlcmVkX2NvcnJlY3QubnVtZWwoKSkpCiAgICAgICAgcmVzdWx0LmFwcGVuZCgKICAgICAgICAgICAgUmlza0NvdmVyYWdlUG9pbnQoCiAgICAgICAgICAgICAgICBjb3ZlcmFnZT1yZXRhaW5lZCAvIG9yZGVyZWRfY29ycmVjdC5udW1lbCgpLAogICAgICAgICAgICAgICAgcmV0YWluZWRfZXhhbXBsZXM9cmV0YWluZWQsCiAgICAgICAgICAgICAgICBhY2N1cmFjeT1mbG9hdChvcmRlcmVkX2NvcnJlY3RbOnJldGFpbmVkXS5tZWFuKCkuaXRlbSgpKSwKICAgICAgICAgICAgICAgIG1heGltdW1fZWZmZWN0aXZlX3Jpc2s9ZmxvYXQob3JkZXJlZF9yaXNrWzpyZXRhaW5lZF0ubWF4KCkuaXRlbSgpKSwKICAgICAgICAgICAgICAgIHBvcHVsYXRpb25fY292ZXJhZ2U9cmV0YWluZWQgLyBhY3RpdmUubnVtZWwoKSwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgZXZhbHVhdGVfb2x5bXB1c19zeXN0ZW0oCiAgICBtb2RlbDogT2x5bXB1c1JlZmVyZW5jZVN5c3RlbSwKICAgIGJhdGNoOiBNYXBwaW5nW3N0ciwgVGVuc29yXSwKICAgIGV4dGVuc2lvbnM6IFNlcXVlbmNlW3N0cl0sCiAgICAqLAogICAgbG9zc19jb25maWc6IE9seW1wdXNTeXN0ZW1Mb3NzQ29uZmlnIHwgTm9uZSA9IE5vbmUsCiAgICBiYXRjaF9zaXplOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBPbHltcHVzU3lzdGVtRXZhbHVhdGlvbjoKICAgICIiIkJhdGNoZWQgZm9yd2FyZCBwYXNzZXMsIGZvbGxvd2VkIGJ5IGV4YWN0IGdsb2JhbCBkaWFnbm9zdGljIHJlZHVjdGlvbnMuCgogICAgVGhpcyBkb2VzIG5vdCBhdmVyYWdlIG1pbmliYXRjaCBsb3NzZXM6IGFjdGl2ZS1leGFtcGxlIGFuZCB2YWxpZC10aW1lc3RlcAogICAgZGVub21pbmF0b3JzIGRpZmZlciwgYW5kIGdhdGUgYmFsYW5jZSBkZXBlbmRzIG9uIHRoZSBnbG9iYWwgbWVhbiBnYXRlcy4KICAgIE9ubHkgY29tcGFjdCBkZXRhY2hlZCBwcmVkaWN0aW9ucyBhcmUgcmV0YWluZWQgb24gQ1BVLiBDYWxsZXIgbW9kZXMgYXJlIHJlc3RvcmVkLgogICAgYGBjb3ZlcmFnZWBgIGlzIGNvbmRpdGlvbmFsIG9uIGRlY2lzaW9uIGF2YWlsYWJpbGl0eTsgYGBwb3B1bGF0aW9uX2NvdmVyYWdlYGAKICAgIGluY2x1ZGVzIGV4YW1wbGVzIGZvcmNlZCB0byBTVE9QIGluIGl0cyBkZW5vbWluYXRvci4KICAgICIiIgoKICAgIGNvdW50ID0gYmF0Y2hbInRva2VucyJdLnNoYXBlWzBdCiAgICBpZiBiYXRjaF9zaXplIGlzIG5vdCBOb25lIGFuZCBiYXRjaF9zaXplIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJldmFsdWF0aW9uIGJhdGNoX3NpemUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBpZiBjb3VudCA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXZhbHVhdGlvbiByZXF1aXJlcyBhdCBsZWFzdCBvbmUgZXhhbXBsZSIpCiAgICB3aWR0aCA9IGNvdW50IGlmIGJhdGNoX3NpemUgaXMgTm9uZSBlbHNlIGJhdGNoX3NpemUKICAgIHBhcmFtZXRlciA9IG5leHQobW9kZWwucGFyYW1ldGVycygpKQogICAgZGV2aWNlID0gcGFyYW1ldGVyLmRldmljZQogICAgY2h1bmtzOiBsaXN0W09seW1wdXNPYmplY3RpdmVPdXRwdXRdID0gW10KICAgIHdpdGggX2V2YWx1YXRpb25fbW9kZShtb2RlbCksIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGNvdW50LCB3aWR0aCk6CiAgICAgICAgICAgIHNlbGVjdGVkID0gewogICAgICAgICAgICAgICAgbmFtZTogdGVuc29yW3N0YXJ0IDogc3RhcnQgKyB3aWR0aF0udG8oCiAgICAgICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgICAgICAgICBkdHlwZT1wYXJhbWV0ZXIuZHR5cGUgaWYgdGVuc29yLmlzX2Zsb2F0aW5nX3BvaW50KCkgZWxzZSB0ZW5zb3IuZHR5cGUsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBmb3IgbmFtZSwgdGVuc29yIGluIGJhdGNoLml0ZW1zKCkKICAgICAgICAgICAgfQogICAgICAgICAgICBjaHVua19pbnB1dHMsIF8gPSBvbHltcHVzX2JhdGNoX2lucHV0c190YXJnZXRzKHNlbGVjdGVkLCBleHRlbnNpb25zKQogICAgICAgICAgICBwcmVkaWN0aW9uID0gbW9kZWwoKipjaHVua19pbnB1dHMpCiAgICAgICAgICAgIGNodW5rcy5hcHBlbmQoX2NvbXBhY3QocHJlZGljdGlvbikpCiAgICAgICAgICAgIGRlbCBwcmVkaWN0aW9uLCBzZWxlY3RlZCwgY2h1bmtfaW5wdXRzCiAgICBvdXRwdXQgPSBfam9pbihjaHVua3MpCiAgICAjIEdsb2JhbCBkaWFnbm9zdGljcyBzdGF5IGluIHJlZmVyZW5jZSBwcmVjaXNpb24gYW5kIG9uIHRoZSBDUFUuCiAgICBiYXRjaCA9IHsKICAgICAgICBuYW1lOiB0ZW5zb3IuZGV0YWNoKCkudG8oCiAgICAgICAgICAgIGRldmljZT0iY3B1IiwgZHR5cGU9dG9yY2guZmxvYXQzMiBpZiB0ZW5zb3IuaXNfZmxvYXRpbmdfcG9pbnQoKSBlbHNlIHRlbnNvci5kdHlwZQogICAgICAgICkKICAgICAgICBmb3IgbmFtZSwgdGVuc29yIGluIGJhdGNoLml0ZW1zKCkKICAgIH0KICAgIGlucHV0cywgdGFyZ2V0cyA9IG9seW1wdXNfYmF0Y2hfaW5wdXRzX3RhcmdldHMoYmF0Y2gsIGV4dGVuc2lvbnMpCiAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgdGVtcG9yYWxfbWFzayA9IGlucHV0cy5nZXQoInRlbXBvcmFsX21hc2siKQogICAgICAgIGFzc2VydCB0ZW1wb3JhbF9tYXNrIGlzIE5vbmUgb3IgaXNpbnN0YW5jZSh0ZW1wb3JhbF9tYXNrLCBUZW5zb3IpCiAgICAgICAgbG9zcyA9IG9seW1wdXNfc3lzdGVtX2xvc3MoCiAgICAgICAgICAgIG91dHB1dCwKICAgICAgICAgICAgZGVjaXNpb25fdGFyZ2V0cz10YXJnZXRzWyJkZWNpc2lvbl90YXJnZXRzIl0sCiAgICAgICAgICAgIHRyYW5zaXRpb25fdGFyZ2V0cz10YXJnZXRzWyJ0cmFuc2l0aW9uX3RhcmdldHMiXSwKICAgICAgICAgICAgbmV4dF9zdGF0ZV90YXJnZXRzPXRhcmdldHNbIm5leHRfc3RhdGVfdGFyZ2V0cyJdLAogICAgICAgICAgICBpbnRyaW5zaWNfcmlza190YXJnZXRzPXRhcmdldHMuZ2V0KCJpbnRyaW5zaWNfcmlza190YXJnZXRzIiksCiAgICAgICAgICAgIHRlbXBvcmFsX21hc2s9dGVtcG9yYWxfbWFzaywKICAgICAgICAgICAgKioobG9zc19jb25maWcgb3IgT2x5bXB1c1N5c3RlbUxvc3NDb25maWcoKSkubW9kZWxfZHVtcCgpLAogICAgICAgICkKICAgIGZpbml0ZV9vdXRwdXRzID0gKAogICAgICAgIG91dHB1dC5yYXdfZGVjaXNpb25fbG9naXRzLAogICAgICAgIG91dHB1dC5haW9uLnRyYW5zaXRpb25fbG9naXRzLm1hc2tlZF9maWxsKG91dHB1dC5haW9uLnRyYW5zaXRpb25fbG9naXRzLmlzbmVnaW5mKCksIDAuMCksCiAgICAgICAgb3V0cHV0Lm5leHRfc3RhdGUsCiAgICAgICAgb3V0cHV0LnVuY2VydGFpbnR5LAogICAgICAgIG91dHB1dC5mYW1pbHlfZ2F0ZXMsCiAgICAgICAgb3V0cHV0LmNvdW50ZXJmYWN0dWFsX2RlY2lzaW9uX2xvZ2l0cywKICAgICAgICBvdXRwdXQuaW50cmluc2ljX3Jpc2ssCiAgICAgICAgb3V0cHV0LmVmZmVjdGl2ZV9yaXNrLAogICAgICAgIGxvc3MudG90YWwudW5zcXVlZXplKDApLAogICAgKQogICAgaWYgbm90IGFsbChib29sKHRvcmNoLmlzZmluaXRlKHZhbHVlKS5hbGwoKSkgZm9yIHZhbHVlIGluIGZpbml0ZV9vdXRwdXRzKToKICAgICAgICByYWlzZSBGbG9hdGluZ1BvaW50RXJyb3IoIk9seW1wdXMgZXZhbHVhdGlvbiBwcm9kdWNlZCBub24tZmluaXRlIGRpYWdub3N0aWNzIikKCiAgICBkZWNpc2lvbl90YXJnZXRzID0gdGFyZ2V0c1siZGVjaXNpb25fdGFyZ2V0cyJdLnRvKAogICAgICAgIGRldmljZT1vdXRwdXQucmF3X2RlY2lzaW9uX2xvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmxvbmcKICAgICkKICAgIHRyYW5zaXRpb25fdGFyZ2V0cyA9IHRhcmdldHNbInRyYW5zaXRpb25fdGFyZ2V0cyJdLnRvKAogICAgICAgIGRldmljZT1vdXRwdXQuYWlvbi50cmFuc2l0aW9uX2xvZ2l0cy5kZXZpY2UsIGR0eXBlPXRvcmNoLmxvbmcKICAgICkKICAgIGFjdGl2ZSA9IG91dHB1dC5kZWNpc2lvbl9hdmFpbGFibGUKICAgIGRlY2lzaW9uX2FjY3VyYWN5ID0gTm9uZQogICAgaWYgYm9vbChhY3RpdmUuYW55KCkpOgogICAgICAgIGRlY2lzaW9uX2FjY3VyYWN5ID0gZmxvYXQoCiAgICAgICAgICAgIG91dHB1dC5yYXdfZGVjaXNpb25fbG9naXRzLmFyZ21heChkaW09LTEpW2FjdGl2ZV0KICAgICAgICAgICAgLmVxKGRlY2lzaW9uX3RhcmdldHNbYWN0aXZlXSkKICAgICAgICAgICAgLmZsb2F0KCkKICAgICAgICAgICAgLm1lYW4oKQogICAgICAgICAgICAuaXRlbSgpCiAgICAgICAgKQoKICAgIHRlbXBvcmFsX2Vycm9yID0gKG91dHB1dC5uZXh0X3N0YXRlIC0gdGFyZ2V0c1sibmV4dF9zdGF0ZV90YXJnZXRzIl0pLnNxdWFyZSgpLm1lYW4oZGltPS0xKQogICAgdGVtcG9yYWxfbWFzayA9IGlucHV0cy5nZXQoInRlbXBvcmFsX21hc2siKQogICAgaWYgaXNpbnN0YW5jZSh0ZW1wb3JhbF9tYXNrLCBUZW5zb3IpOgogICAgICAgIG5leHRfc3RhdGVfbXNlID0gZmxvYXQodGVtcG9yYWxfZXJyb3JbdGVtcG9yYWxfbWFza10ubWVhbigpLml0ZW0oKSkKICAgIGVsc2U6CiAgICAgICAgbmV4dF9zdGF0ZV9tc2UgPSBmbG9hdCh0ZW1wb3JhbF9lcnJvci5tZWFuKCkuaXRlbSgpKQoKICAgIGdhdGVzID0gb3V0cHV0LmZhbWlseV9nYXRlcy5jbGFtcF9taW4odG9yY2guZmluZm8ob3V0cHV0LmZhbWlseV9nYXRlcy5kdHlwZSkudGlueSkKICAgIGVudHJvcHkgPSAtKGdhdGVzICogZ2F0ZXMubG9nKCkpLnN1bShkaW09LTEpCiAgICBub3JtYWxpemVkX2VudHJvcHkgPSBmbG9hdCgoZW50cm9weSAvIG1hdGgubG9nKGdhdGVzLnNoYXBlWzFdKSkubWVhbigpLml0ZW0oKSkKICAgIGZ1bGxfcHJvYmFiaWxpdGllcyA9IG91dHB1dC5yYXdfZGVjaXNpb25fbG9naXRzLnNvZnRtYXgoZGltPS0xKQogICAgY291bnRlcmZhY3R1YWxfbG9nX3Byb2JhYmlsaXRpZXMgPSBvdXRwdXQuY291bnRlcmZhY3R1YWxfZGVjaXNpb25fbG9naXRzLmxvZ19zb2Z0bWF4KGRpbT0tMSkKICAgIGNvdW50ZXJmYWN0dWFsX2tsX3RlbnNvciA9IG5uLmZ1bmN0aW9uYWwua2xfZGl2KAogICAgICAgIGNvdW50ZXJmYWN0dWFsX2xvZ19wcm9iYWJpbGl0aWVzLAogICAgICAgIGZ1bGxfcHJvYmFiaWxpdGllcy51bnNxdWVlemUoMSkuZXhwYW5kX2FzKGNvdW50ZXJmYWN0dWFsX2xvZ19wcm9iYWJpbGl0aWVzKSwKICAgICAgICByZWR1Y3Rpb249Im5vbmUiLAogICAgKS5zdW0oZGltPS0xKQogICAgZnVsbF9jbGFzc2VzID0gb3V0cHV0LnJhd19kZWNpc2lvbl9sb2dpdHMuYXJnbWF4KGRpbT0tMSkudW5zcXVlZXplKDEpCiAgICBjb3VudGVyZmFjdHVhbF9mbGlwcyA9IG91dHB1dC5jb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHMuYXJnbWF4KGRpbT0tMSkubmUoZnVsbF9jbGFzc2VzKQoKICAgIHJpc2tfdGFyZ2V0cyA9IHRhcmdldHMuZ2V0KCJpbnRyaW5zaWNfcmlza190YXJnZXRzIikKICAgIGJyaWVyID0gTm9uZQogICAgZWNlID0gTm9uZQogICAgaWYgcmlza190YXJnZXRzIGlzIG5vdCBOb25lOgogICAgICAgIHJpc2tfdGFyZ2V0cyA9IHJpc2tfdGFyZ2V0cy50bygKICAgICAgICAgICAgZGV2aWNlPW91dHB1dC5pbnRyaW5zaWNfcmlzay5kZXZpY2UsIGR0eXBlPW91dHB1dC5pbnRyaW5zaWNfcmlzay5kdHlwZQogICAgICAgICkKICAgICAgICBicmllciA9IGZsb2F0KChvdXRwdXQuaW50cmluc2ljX3Jpc2sgLSByaXNrX3RhcmdldHMpLnNxdWFyZSgpLm1lYW4oKS5pdGVtKCkpCiAgICAgICAgZWNlID0gX2V4cGVjdGVkX2NhbGlicmF0aW9uX2Vycm9yKG91dHB1dC5pbnRyaW5zaWNfcmlzaywgcmlza190YXJnZXRzKQoKICAgIHJldHVybiBPbHltcHVzU3lzdGVtRXZhbHVhdGlvbigKICAgICAgICBleGFtcGxlcz1vdXRwdXQucmF3X2RlY2lzaW9uX2xvZ2l0cy5zaGFwZVswXSwKICAgICAgICBkZWNpc2lvbl9hdmFpbGFibGVfZXhhbXBsZXM9aW50KGFjdGl2ZS5zdW0oKS5pdGVtKCkpLAogICAgICAgIGZvcmNlZF9zdG9wX2V4YW1wbGVzPWludChvdXRwdXQuYWlvbi5mb3JjZWRfc3RvcC5zdW0oKS5pdGVtKCkpLAogICAgICAgIHRvdGFsX2xvc3M9ZmxvYXQobG9zcy50b3RhbC5pdGVtKCkpLAogICAgICAgIGxvc3NfY29tcG9uZW50cz17CiAgICAgICAgICAgIG5hbWU6IGZsb2F0KGdldGF0dHIobG9zcywgbmFtZSkuaXRlbSgpKQogICAgICAgICAgICBmb3IgbmFtZSBpbiAoCiAgICAgICAgICAgICAgICAiZGVjaXNpb24iLAogICAgICAgICAgICAgICAgInRyYW5zaXRpb24iLAogICAgICAgICAgICAgICAgInRlbXBvcmFsIiwKICAgICAgICAgICAgICAgICJ1bmNlcnRhaW50eV9jYWxpYnJhdGlvbiIsCiAgICAgICAgICAgICAgICAicmlza19jYWxpYnJhdGlvbiIsCiAgICAgICAgICAgICAgICAiZ2F0ZV9iYWxhbmNlIiwKICAgICAgICAgICAgICAgICJjb3VudGVyZmFjdHVhbF9jb25zaXN0ZW5jeSIsCiAgICAgICAgICAgICkKICAgICAgICB9LAogICAgICAgIGRlY2lzaW9uX2FjY3VyYWN5PWRlY2lzaW9uX2FjY3VyYWN5LAogICAgICAgIHRyYW5zaXRpb25fYWNjdXJhY3k9ZmxvYXQoCiAgICAgICAgICAgIG91dHB1dC5haW9uLnNlbGVjdGVkX3N0YWdlLmVxKHRyYW5zaXRpb25fdGFyZ2V0cykuZmxvYXQoKS5tZWFuKCkuaXRlbSgpCiAgICAgICAgKSwKICAgICAgICBuZXh0X3N0YXRlX21zZT1uZXh0X3N0YXRlX21zZSwKICAgICAgICBtZWFuX3VuY2VydGFpbnR5PWZsb2F0KG91dHB1dC51bmNlcnRhaW50eS5tZWFuKCkuaXRlbSgpKSwKICAgICAgICByb3V0aW5nPU9seW1wdXNSb3V0aW5nRGlhZ25vc3RpY3MoCiAgICAgICAgICAgIG1lYW5fZ2F0ZXM9ewogICAgICAgICAgICAgICAgbmFtZTogZmxvYXQodmFsdWUpCiAgICAgICAgICAgICAgICBmb3IgbmFtZSwgdmFsdWUgaW4gemlwKAogICAgICAgICAgICAgICAgICAgIG91dHB1dC5jb21wb25lbnRfbmFtZXMsCiAgICAgICAgICAgICAgICAgICAgb3V0cHV0LmZhbWlseV9nYXRlcy5tZWFuKGRpbT0wKS50b2xpc3QoKSwKICAgICAgICAgICAgICAgICAgICBzdHJpY3Q9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgfSwKICAgICAgICAgICAgbm9ybWFsaXplZF9nYXRlX2VudHJvcHk9bm9ybWFsaXplZF9lbnRyb3B5LAogICAgICAgICAgICBlZmZlY3RpdmVfY29tcG9uZW50cz1mbG9hdChlbnRyb3B5LmV4cCgpLm1lYW4oKS5pdGVtKCkpLAogICAgICAgICAgICBjb3VudGVyZmFjdHVhbF9rbD17CiAgICAgICAgICAgICAgICBuYW1lOiBmbG9hdCh2YWx1ZSkKICAgICAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZSBpbiB6aXAoCiAgICAgICAgICAgICAgICAgICAgb3V0cHV0LmNvbXBvbmVudF9uYW1lcywKICAgICAgICAgICAgICAgICAgICBjb3VudGVyZmFjdHVhbF9rbF90ZW5zb3IubWVhbihkaW09MCkudG9saXN0KCksCiAgICAgICAgICAgICAgICAgICAgc3RyaWN0PVRydWUsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgIGNvdW50ZXJmYWN0dWFsX2ZsaXBfcmF0ZT17CiAgICAgICAgICAgICAgICBuYW1lOiBmbG9hdCh2YWx1ZSkKICAgICAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZSBpbiB6aXAoCiAgICAgICAgICAgICAgICAgICAgb3V0cHV0LmNvbXBvbmVudF9uYW1lcywKICAgICAgICAgICAgICAgICAgICBjb3VudGVyZmFjdHVhbF9mbGlwcy5mbG9hdCgpLm1lYW4oZGltPTApLnRvbGlzdCgpLAogICAgICAgICAgICAgICAgICAgIHN0cmljdD1UcnVlLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICB9LAogICAgICAgICksCiAgICAgICAgcmlzaz1PbHltcHVzUmlza0RpYWdub3N0aWNzKAogICAgICAgICAgICBtZWFuX2ludHJpbnNpY19yaXNrPWZsb2F0KG91dHB1dC5pbnRyaW5zaWNfcmlzay5tZWFuKCkuaXRlbSgpKSwKICAgICAgICAgICAgbWVhbl9lZmZlY3RpdmVfcmlzaz1mbG9hdChvdXRwdXQuZWZmZWN0aXZlX3Jpc2subWVhbigpLml0ZW0oKSksCiAgICAgICAgICAgIGludHJpbnNpY19icmllcj1icmllciwKICAgICAgICAgICAgaW50cmluc2ljX2VjZT1lY2UsCiAgICAgICAgICAgIHJpc2tfY292ZXJhZ2U9X3Jpc2tfY292ZXJhZ2Uob3V0cHV0LCBkZWNpc2lvbl90YXJnZXRzKSwKICAgICAgICApLAogICAgKQoKCmRlZiBfY29tcGFjdChvdXRwdXQ6IE9seW1wdXNTeXN0ZW1PdXRwdXQpIC0+IE9seW1wdXNPYmplY3RpdmVPdXRwdXQ6CiAgICBkZWYgY3B1KHRlbnNvcjogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgdmFsdWUgPSB0ZW5zb3IuZGV0YWNoKCkuY3B1KCkKICAgICAgICByZXR1cm4gdmFsdWUuZmxvYXQoKSBpZiB2YWx1ZS5pc19mbG9hdGluZ19wb2ludCgpIGVsc2UgdmFsdWUKCiAgICByZXR1cm4gT2x5bXB1c09iamVjdGl2ZU91dHB1dCgKICAgICAgICBhaW9uPUFpb25SZWZlcmVuY2VPdXRwdXQoCiAgICAgICAgICAgIHRyYW5zaXRpb25fbG9naXRzPWNwdShvdXRwdXQuYWlvbi50cmFuc2l0aW9uX2xvZ2l0cyksCiAgICAgICAgICAgIHRyYW5zaXRpb25fcHJvYmFiaWxpdGllcz1jcHUob3V0cHV0LmFpb24udHJhbnNpdGlvbl9wcm9iYWJpbGl0aWVzKSwKICAgICAgICAgICAgc2VsZWN0ZWRfc3RhZ2U9Y3B1KG91dHB1dC5haW9uLnNlbGVjdGVkX3N0YWdlKSwKICAgICAgICAgICAgZm9yY2VkX3N0b3A9Y3B1KG91dHB1dC5haW9uLmZvcmNlZF9zdG9wKSwKICAgICAgICApLAogICAgICAgIG5leHRfc3RhdGU9Y3B1KG91dHB1dC5rcm9ub3MubmV4dF9zdGF0ZSksCiAgICAgICAgY29tcG9uZW50X25hbWVzPW91dHB1dC5jb21wb25lbnRfbmFtZXMsCiAgICAgICAgcmF3X2RlY2lzaW9uX2xvZ2l0cz1jcHUob3V0cHV0LnJhd19kZWNpc2lvbl9sb2dpdHMpLAogICAgICAgIGRlY2lzaW9uX2F2YWlsYWJsZT1jcHUob3V0cHV0LmRlY2lzaW9uX2F2YWlsYWJsZSksCiAgICAgICAgdW5jZXJ0YWludHk9Y3B1KG91dHB1dC51bmNlcnRhaW50eSksCiAgICAgICAgZmFtaWx5X2dhdGVzPWNwdShvdXRwdXQuZmFtaWx5X2dhdGVzKSwKICAgICAgICBjb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHM9Y3B1KG91dHB1dC5jb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHMpLAogICAgICAgIGludHJpbnNpY19yaXNrPWNwdShvdXRwdXQuaW50cmluc2ljX3Jpc2spLAogICAgICAgIGVmZmVjdGl2ZV9yaXNrPWNwdShvdXRwdXQuZWZmZWN0aXZlX3Jpc2spLAogICAgKQoKCmRlZiBfam9pbihjaHVua3M6IGxpc3RbT2x5bXB1c09iamVjdGl2ZU91dHB1dF0pIC0+IE9seW1wdXNPYmplY3RpdmVPdXRwdXQ6CiAgICBkZWYgY2F0KG5hbWU6IHN0cikgLT4gVGVuc29yOgogICAgICAgIHJldHVybiB0b3JjaC5jYXQoW2dldGF0dHIoY2h1bmssIG5hbWUpIGZvciBjaHVuayBpbiBjaHVua3NdLCBkaW09MCkKCiAgICBkZWYgYWlvbl9jYXQobmFtZTogc3RyKSAtPiBUZW5zb3I6CiAgICAgICAgcmV0dXJuIHRvcmNoLmNhdChbZ2V0YXR0cihjaHVuay5haW9uLCBuYW1lKSBmb3IgY2h1bmsgaW4gY2h1bmtzXSwgZGltPTApCgogICAgcmV0dXJuIE9seW1wdXNPYmplY3RpdmVPdXRwdXQoCiAgICAgICAgYWlvbj1BaW9uUmVmZXJlbmNlT3V0cHV0KAogICAgICAgICAgICB0cmFuc2l0aW9uX2xvZ2l0cz1haW9uX2NhdCgidHJhbnNpdGlvbl9sb2dpdHMiKSwKICAgICAgICAgICAgdHJhbnNpdGlvbl9wcm9iYWJpbGl0aWVzPWFpb25fY2F0KCJ0cmFuc2l0aW9uX3Byb2JhYmlsaXRpZXMiKSwKICAgICAgICAgICAgc2VsZWN0ZWRfc3RhZ2U9YWlvbl9jYXQoInNlbGVjdGVkX3N0YWdlIiksCiAgICAgICAgICAgIGZvcmNlZF9zdG9wPWFpb25fY2F0KCJmb3JjZWRfc3RvcCIpLAogICAgICAgICksCiAgICAgICAgbmV4dF9zdGF0ZT1jYXQoIm5leHRfc3RhdGUiKSwKICAgICAgICBjb21wb25lbnRfbmFtZXM9Y2h1bmtzWzBdLmNvbXBvbmVudF9uYW1lcywKICAgICAgICByYXdfZGVjaXNpb25fbG9naXRzPWNhdCgicmF3X2RlY2lzaW9uX2xvZ2l0cyIpLAogICAgICAgIGRlY2lzaW9uX2F2YWlsYWJsZT1jYXQoImRlY2lzaW9uX2F2YWlsYWJsZSIpLAogICAgICAgIHVuY2VydGFpbnR5PWNhdCgidW5jZXJ0YWludHkiKSwKICAgICAgICBmYW1pbHlfZ2F0ZXM9Y2F0KCJmYW1pbHlfZ2F0ZXMiKSwKICAgICAgICBjb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHM9Y2F0KCJjb3VudGVyZmFjdHVhbF9kZWNpc2lvbl9sb2dpdHMiKSwKICAgICAgICBpbnRyaW5zaWNfcmlzaz1jYXQoImludHJpbnNpY19yaXNrIiksCiAgICAgICAgZWZmZWN0aXZlX3Jpc2s9Y2F0KCJlZmZlY3RpdmVfcmlzayIpLAogICAgKQo=', 'olympus/forge/__init__.py': 'IiIiT2x5bXB1cyBGb3JnZSBiZWhhdmlvciBjb21waWxlciBhbmQgcnVudGltZS4iIiIK', 'olympus/forge/compiler.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5mb3JnZS5sYW5ndWFnZSBpbXBvcnQgQmVoYXZpb3JOb2RlLCBCZWhhdmlvclNwZWMsIE5vZGVLaW5kCgoKY2xhc3MgQ29tcGlsZXJJc3N1ZShTdHJpY3RNb2RlbCk6CiAgICBsZXZlbDogc3RyCiAgICBkZXRhaWw6IHN0cgoKCmNsYXNzIE5hdHVyYWxMYW5ndWFnZUNvbXBpbGF0aW9uKFN0cmljdE1vZGVsKToKICAgIHNwZWM6IEJlaGF2aW9yU3BlYwogICAgaXNzdWVzOiBsaXN0W0NvbXBpbGVySXNzdWVdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgoKY2xhc3MgTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcjoKICAgIGRlZiBjb21waWxlKAogICAgICAgIHNlbGYsIGRlc2NyaXB0aW9uOiBzdHIsIG5hbWU6IHN0ciA9ICJjb21waWxlZC1iZWhhdmlvciIKICAgICkgLT4gTmF0dXJhbExhbmd1YWdlQ29tcGlsYXRpb246CiAgICAgICAgbG93ZXJlZCA9IGRlc2NyaXB0aW9uLmxvd2VyKCkKICAgICAgICBub2RlczogbGlzdFtCZWhhdmlvck5vZGVdID0gWwogICAgICAgICAgICBCZWhhdmlvck5vZGUoCiAgICAgICAgICAgICAgICBpZD0iaW50ZXJwcmV0Iiwga2luZD1Ob2RlS2luZC5JTlRFUlBSRVQsIGRlc2NyaXB0aW9uPSJHZW5lcmF0ZSBpbnRlcnByZXRhdGlvbnMiCiAgICAgICAgICAgICksCiAgICAgICAgXQogICAgICAgIGlzc3VlczogbGlzdFtDb21waWxlcklzc3VlXSA9IFtdCiAgICAgICAgcHJldmlvdXNfbm9kZSA9ICJpbnRlcnByZXQiCgogICAgICAgIGlmICJjb25maWRlbmNlIiBpbiBsb3dlcmVkIG9yICJzY29yZSIgaW4gbG93ZXJlZCBvciAidmVyaWZ5IiBpbiBsb3dlcmVkOgogICAgICAgICAgICBub2Rlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICBCZWhhdmlvck5vZGUoCiAgICAgICAgICAgICAgICAgICAgaWQ9InZlcmlmeSIsCiAgICAgICAgICAgICAgICAgICAga2luZD1Ob2RlS2luZC5WRVJJRlksCiAgICAgICAgICAgICAgICAgICAgZGVzY3JpcHRpb249IlZlcmlmeSBicmFuY2ggc3VwcG9ydCIsCiAgICAgICAgICAgICAgICAgICAgZGVwZW5kc19vbj1bcHJldmlvdXNfbm9kZV0sCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcHJldmlvdXNfbm9kZSA9ICJ2ZXJpZnkiCiAgICAgICAgaWYgInRvb2wiIGluIGxvd2VyZWQgb3IgInRlc3QiIGluIGxvd2VyZWQ6CiAgICAgICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAgICAgICAgICAgIEJlaGF2aW9yTm9kZSgKICAgICAgICAgICAgICAgICAgICBpZD0idG9vbCIsCiAgICAgICAgICAgICAgICAgICAga2luZD1Ob2RlS2luZC5UT09MLAogICAgICAgICAgICAgICAgICAgIGRlc2NyaXB0aW9uPSJFeGVjdXRlIGEgZGV0ZXJtaW5pc3RpYyB0b29sIGNoZWNrIiwKICAgICAgICAgICAgICAgICAgICBkZXBlbmRzX29uPVtwcmV2aW91c19ub2RlXSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgICAgICBwcmV2aW91c19ub2RlID0gInRvb2wiCiAgICAgICAgaWYgIm1lbW9yeSIgaW4gbG93ZXJlZDoKICAgICAgICAgICAgbm9kZXMuYXBwZW5kKAogICAgICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAogICAgICAgICAgICAgICAgICAgIGlkPSJtZW1vcnkiLAogICAgICAgICAgICAgICAgICAgIGtpbmQ9Tm9kZUtpbmQuTUVNT1JZX1dSSVRFLAogICAgICAgICAgICAgICAgICAgIGRlc2NyaXB0aW9uPSJQZXJzaXN0IHVzZWZ1bCBmaW5kaW5ncyIsCiAgICAgICAgICAgICAgICAgICAgZGVwZW5kc19vbj1bcHJldmlvdXNfbm9kZV0sCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcHJldmlvdXNfbm9kZSA9ICJtZW1vcnkiCgogICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAogICAgICAgICAgICAgICAgaWQ9Im1lcmdlIiwKICAgICAgICAgICAgICAgIGtpbmQ9Tm9kZUtpbmQuTUVSR0UsCiAgICAgICAgICAgICAgICBkZXNjcmlwdGlvbj0iTWVyZ2Ugc3Vydml2aW5nIGJyYW5jaGVzIiwKICAgICAgICAgICAgICAgIGRlcGVuZHNfb249W3ByZXZpb3VzX25vZGVdLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAogICAgICAgICAgICAgICAgaWQ9Im91dHB1dCIsCiAgICAgICAgICAgICAgICBraW5kPU5vZGVLaW5kLk9VVFBVVCwKICAgICAgICAgICAgICAgIGRlc2NyaXB0aW9uPSJSZXR1cm4gZmluYWwgcmVzdWx0IiwKICAgICAgICAgICAgICAgIGRlcGVuZHNfb249WyJtZXJnZSJdLAogICAgICAgICAgICApCiAgICAgICAgKQoKICAgICAgICBpZiAiY3JlYXRpdmUiIGluIGxvd2VyZWQgYW5kICJsb3ctY29uZmlkZW5jZSIgbm90IGluIGxvd2VyZWQ6CiAgICAgICAgICAgIGlzc3Vlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICBDb21waWxlcklzc3VlKAogICAgICAgICAgICAgICAgICAgIGxldmVsPSJ3YXJuaW5nIiwKICAgICAgICAgICAgICAgICAgICBkZXRhaWw9KAogICAgICAgICAgICAgICAgICAgICAgICAiQ3JlYXRpdmUgZXhwbG9yYXRpb24gcmVxdWVzdGVkIHdpdGhvdXQgZXhwbGljaXQgbG93LWNvbmZpZGVuY2UgYWxsb3dhbmNlLiIKICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCgogICAgICAgIHNwZWMgPSBCZWhhdmlvclNwZWMoCiAgICAgICAgICAgIG5hbWU9bmFtZSwKICAgICAgICAgICAgZGVzY3JpcHRpb249ZGVzY3JpcHRpb24sCiAgICAgICAgICAgIG9iamVjdGl2ZT1kZXNjcmlwdGlvbiwKICAgICAgICAgICAgbm9kZXM9bm9kZXMsCiAgICAgICAgICAgIG91dHB1dHM9WyJvdXRwdXQiXSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIE5hdHVyYWxMYW5ndWFnZUNvbXBpbGF0aW9uKHNwZWM9c3BlYywgaXNzdWVzPWlzc3VlcykK', 'olympus/forge/experiments.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gc3RhdGlzdGljcyBpbXBvcnQgbWVhbgoKZnJvbSBvbHltcHVzLmZvcmdlLmNvbXBpbGVyIGltcG9ydCBOYXR1cmFsTGFuZ3VhZ2VCZWhhdmlvckNvbXBpbGVyCmZyb20gb2x5bXB1cy5mb3JnZS5ydW50aW1lIGltcG9ydCBGb3JnZVJ1bnRpbWUKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIEV4cGVyaW1lbnRTdW1tYXJ5OgogICAgdHJpYWxfY291bnQ6IGludAogICAgbWVhbl9jb25maWRlbmNlOiBmbG9hdAogICAgbWVhbl9zdXJ2aXZvcnM6IGZsb2F0CgoKY2xhc3MgRXhwZXJpbWVudENvbnRyb2xsZXI6CiAgICBkZWYgX19pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmNvbXBpbGVyID0gTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcigpCiAgICAgICAgc2VsZi5ydW50aW1lID0gRm9yZ2VSdW50aW1lKCkKCiAgICBkZWYgcnVuX3RyaWFscyhzZWxmLCBiZWhhdmlvcl90ZXh0OiBzdHIsIHByb21wdDogc3RyLCB0cmlhbHM6IGludCA9IDMpIC0+IEV4cGVyaW1lbnRTdW1tYXJ5OgogICAgICAgIGlmIHRyaWFscyA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyaWFscyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICBjb21waWxhdGlvbiA9IHNlbGYuY29tcGlsZXIuY29tcGlsZShiZWhhdmlvcl90ZXh0KQogICAgICAgIGNvbmZpZGVuY2VzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgc3Vydml2b3JzOiBsaXN0W2ludF0gPSBbXQogICAgICAgIGZvciBfIGluIHJhbmdlKHRyaWFscyk6CiAgICAgICAgICAgIHJlc3VsdCA9IHNlbGYucnVudGltZS5leGVjdXRlKGNvbXBpbGF0aW9uLnNwZWMsIHByb21wdCkKICAgICAgICAgICAgbWVyZ2VkID0gcmVzdWx0Lm91dHB1dHNbIm1lcmdlZCJdCiAgICAgICAgICAgIGNvbmZpZGVuY2VzLmFwcGVuZChmbG9hdChtZXJnZWRbImNvbmZpZGVuY2UiXSkpCiAgICAgICAgICAgIHN1cnZpdm9ycy5hcHBlbmQoaW50KG1lcmdlZFsic3Vydml2b3JzIl0pKQogICAgICAgIHJldHVybiBFeHBlcmltZW50U3VtbWFyeSgKICAgICAgICAgICAgdHJpYWxfY291bnQ9dHJpYWxzLAogICAgICAgICAgICBtZWFuX2NvbmZpZGVuY2U9cm91bmQobWVhbihjb25maWRlbmNlcyksIDQpLAogICAgICAgICAgICBtZWFuX3N1cnZpdm9ycz1yb3VuZChtZWFuKHN1cnZpdm9ycyksIDQpLAogICAgICAgICkK', 'olympus/forge/language.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gZ3JhcGhsaWIgaW1wb3J0IFRvcG9sb2dpY2FsU29ydGVyCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB5YW1sCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIE5vZGVLaW5kKFN0ckVudW0pOgogICAgSU5URVJQUkVUID0gImludGVycHJldCIKICAgIFZFUklGWSA9ICJ2ZXJpZnkiCiAgICBNRVJHRSA9ICJtZXJnZSIKICAgIE1FTU9SWV9XUklURSA9ICJtZW1vcnlfd3JpdGUiCiAgICBUT09MID0gInRvb2wiCiAgICBPVVRQVVQgPSAib3V0cHV0IgoKCmNsYXNzIEJlaGF2aW9yTm9kZShTdHJpY3RNb2RlbCk6CiAgICBpZDogc3RyCiAgICBraW5kOiBOb2RlS2luZAogICAgZGVzY3JpcHRpb246IHN0cgogICAgY29uZmlnOiBkaWN0W3N0ciwgQW55XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgZGVwZW5kc19vbjogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgoKY2xhc3MgQmVoYXZpb3JTcGVjKFN0cmljdE1vZGVsKToKICAgIHZlcnNpb246IHN0ciA9ICIxLjAiCiAgICBuYW1lOiBzdHIKICAgIGRlc2NyaXB0aW9uOiBzdHIKICAgIG9iamVjdGl2ZTogc3RyCiAgICBub2RlczogbGlzdFtCZWhhdmlvck5vZGVdCiAgICBvdXRwdXRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgdG9fbWVybWFpZChzZWxmKSAtPiBzdHI6CiAgICAgICAgbGluZXMgPSBbImdyYXBoIFREIl0KICAgICAgICBmb3Igbm9kZSBpbiBzZWxmLm5vZGVzOgogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZicgIHtub2RlLmlkfVsie25vZGUuaWR9OiB7bm9kZS5raW5kLnZhbHVlfSJdJykKICAgICAgICAgICAgZm9yIGRlcGVuZGVuY3kgaW4gbm9kZS5kZXBlbmRzX29uOgogICAgICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiICB7ZGVwZW5kZW5jeX0gLS0+IHtub2RlLmlkfSIpCiAgICAgICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCiAgICBkZWYgdG9wb2xvZ2ljYWxfb3JkZXIoc2VsZikgLT4gbGlzdFtzdHJdOgogICAgICAgIGdyYXBoID0ge25vZGUuaWQ6IHNldChub2RlLmRlcGVuZHNfb24pIGZvciBub2RlIGluIHNlbGYubm9kZXN9CiAgICAgICAgcmV0dXJuIGxpc3QoVG9wb2xvZ2ljYWxTb3J0ZXIoZ3JhcGgpLnN0YXRpY19vcmRlcigpKQoKCmNsYXNzIEJlaGF2aW9yTGFuZ3VhZ2U6CiAgICBkZWYgcGFyc2Uoc2VsZiwgcGF5bG9hZDogc3RyKSAtPiBCZWhhdmlvclNwZWM6CiAgICAgICAgY29udGVudCA9ICgKICAgICAgICAgICAgeWFtbC5zYWZlX2xvYWQocGF5bG9hZCkKICAgICAgICAgICAgaWYgcGF5bG9hZC5zdHJpcCgpLnN0YXJ0c3dpdGgoKCJ2ZXJzaW9uOiIsICJuYW1lOiIpKQogICAgICAgICAgICBlbHNlIGpzb24ubG9hZHMocGF5bG9hZCkKICAgICAgICApCiAgICAgICAgcmV0dXJuIEJlaGF2aW9yU3BlYy5tb2RlbF92YWxpZGF0ZShjb250ZW50KQoKICAgIGRlZiBmb3JtYXQoc2VsZiwgc3BlYzogQmVoYXZpb3JTcGVjKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHlhbWwuc2FmZV9kdW1wKHNwZWMubW9kZWxfZHVtcChtb2RlPSJqc29uIiksIHNvcnRfa2V5cz1GYWxzZSkKCiAgICBkZWYgc2NoZW1hKHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHJldHVybiBCZWhhdmlvclNwZWMubW9kZWxfanNvbl9zY2hlbWEoKQoKICAgIGRlZiBtaWdyYXRlKHNlbGYsIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBCZWhhdmlvclNwZWM6CiAgICAgICAgcGF5bG9hZC5zZXRkZWZhdWx0KCJ2ZXJzaW9uIiwgIjEuMCIpCiAgICAgICAgcGF5bG9hZC5zZXRkZWZhdWx0KCJvdXRwdXRzIiwgWyJmaW5hbF9vdXRwdXQiXSkKICAgICAgICByZXR1cm4gQmVoYXZpb3JTcGVjLm1vZGVsX3ZhbGlkYXRlKHBheWxvYWQpCgogICAgZGVmIHNlbWFudGljX2NoZWNrKHNlbGYsIHNwZWM6IEJlaGF2aW9yU3BlYykgLT4gbGlzdFtzdHJdOgogICAgICAgIG5vZGVfaWRzID0ge25vZGUuaWQgZm9yIG5vZGUgaW4gc3BlYy5ub2Rlc30KICAgICAgICBlcnJvcnMgPSBbCiAgICAgICAgICAgIGYiTWlzc2luZyBkZXBlbmRlbmN5OiB7ZGVwZW5kZW5jeX0iCiAgICAgICAgICAgIGZvciBub2RlIGluIHNwZWMubm9kZXMKICAgICAgICAgICAgZm9yIGRlcGVuZGVuY3kgaW4gbm9kZS5kZXBlbmRzX29uCiAgICAgICAgICAgIGlmIGRlcGVuZGVuY3kgbm90IGluIG5vZGVfaWRzCiAgICAgICAgXQogICAgICAgIHRyeToKICAgICAgICAgICAgc3BlYy50b3BvbG9naWNhbF9vcmRlcigpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoc3RyKGVycm9yKSkKICAgICAgICByZXR1cm4gZXJyb3JzCg==', 'olympus/forge/runtime.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gb2x5bXB1cy5jb3JlLmludGVycHJldGF0aW9uIGltcG9ydCBJbnRlcnByZXRpdmVTdXBlcnBvc2l0aW9uTmV0d29yawpmcm9tIG9seW1wdXMuY29yZS5vdXRjb21lIGltcG9ydCBPdXRjb21lUXVhbGl0eVByZWRpY3Rvcgpmcm9tIG9seW1wdXMuY29yZS52ZXJpZmljYXRpb24gaW1wb3J0IENhbGlicmF0aW9uVmVyaWZpZXIsIENvbnRyYWRpY3Rpb25WZXJpZmllcgpmcm9tIG9seW1wdXMuZm9yZ2UubGFuZ3VhZ2UgaW1wb3J0IEJlaGF2aW9yU3BlYywgTm9kZUtpbmQKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIFJ1bnRpbWVFdmVudDoKICAgIG5vZGVfaWQ6IHN0cgogICAga2luZDogc3RyCiAgICBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUnVudGltZVJlc3VsdDoKICAgIG91dHB1dHM6IGRpY3Rbc3RyLCBBbnldCiAgICBldmVudHM6IGxpc3RbUnVudGltZUV2ZW50XQoKCmNsYXNzIEZvcmdlUnVudGltZToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuaW50ZXJwcmV0ZXIgPSBJbnRlcnByZXRpdmVTdXBlcnBvc2l0aW9uTmV0d29yaygpCiAgICAgICAgc2VsZi5vdXRjb21lX3ByZWRpY3RvciA9IE91dGNvbWVRdWFsaXR5UHJlZGljdG9yKCkKICAgICAgICBzZWxmLmNvbnRyYWRpY3Rpb25fdmVyaWZpZXIgPSBDb250cmFkaWN0aW9uVmVyaWZpZXIoKQogICAgICAgIHNlbGYuY2FsaWJyYXRpb25fdmVyaWZpZXIgPSBDYWxpYnJhdGlvblZlcmlmaWVyKCkKCiAgICBkZWYgZXhlY3V0ZShzZWxmLCBzcGVjOiBCZWhhdmlvclNwZWMsIHByb21wdDogc3RyKSAtPiBSdW50aW1lUmVzdWx0OgogICAgICAgIG91dHB1dHM6IGRpY3Rbc3RyLCBBbnldID0geyJwcm9tcHQiOiBwcm9tcHR9CiAgICAgICAgZXZlbnRzOiBsaXN0W1J1bnRpbWVFdmVudF0gPSBbXQogICAgICAgIGludGVycHJldGF0aW9ucyA9IHNlbGYuaW50ZXJwcmV0ZXIuYW5hbHl6ZShwcm9tcHQpCgogICAgICAgIGZvciBub2RlX2lkIGluIHNwZWMudG9wb2xvZ2ljYWxfb3JkZXIoKToKICAgICAgICAgICAgbm9kZSA9IG5leHQoaXRlbSBmb3IgaXRlbSBpbiBzcGVjLm5vZGVzIGlmIGl0ZW0uaWQgPT0gbm9kZV9pZCkKICAgICAgICAgICAgaWYgbm9kZS5raW5kID09IE5vZGVLaW5kLklOVEVSUFJFVDoKICAgICAgICAgICAgICAgIG91dHB1dHNbImludGVycHJldGF0aW9ucyJdID0gWwogICAgICAgICAgICAgICAgICAgIGJyYW5jaC5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSBmb3IgYnJhbmNoIGluIGludGVycHJldGF0aW9ucwogICAgICAgICAgICAgICAgXQogICAgICAgICAgICAgICAgZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICBSdW50aW1lRXZlbnQoCiAgICAgICAgICAgICAgICAgICAgICAgIG5vZGVfaWQ9bm9kZS5pZCwKICAgICAgICAgICAgICAgICAgICAgICAga2luZD1ub2RlLmtpbmQudmFsdWUsCiAgICAgICAgICAgICAgICAgICAgICAgIHBheWxvYWQ9eyJjb3VudCI6IGxlbihpbnRlcnByZXRhdGlvbnMpfSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGVsaWYgbm9kZS5raW5kID09IE5vZGVLaW5kLlZFUklGWToKICAgICAgICAgICAgICAgIHRvcF9zdGF0ZW1lbnQgPSBpbnRlcnByZXRhdGlvbnNbMF0uc3RhdGVtZW50CiAgICAgICAgICAgICAgICBjb250cmFkaWN0aW9uID0gc2VsZi5jb250cmFkaWN0aW9uX3ZlcmlmaWVyLnZlcmlmeShbdG9wX3N0YXRlbWVudF0pCiAgICAgICAgICAgICAgICBjYWxpYnJhdGlvbiA9IHNlbGYuY2FsaWJyYXRpb25fdmVyaWZpZXIudmVyaWZ5KAogICAgICAgICAgICAgICAgICAgIGludGVycHJldGF0aW9uc1swXS5jb25maWRlbmNlLCBjb250cmFkaWN0aW9uLnBhc3NlZAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgb3V0cHV0c1sidmVyaWZpY2F0aW9uIl0gPSBbY29udHJhZGljdGlvbi5tb2RlbF9kdW1wKCksIGNhbGlicmF0aW9uLm1vZGVsX2R1bXAoKV0KICAgICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUnVudGltZUV2ZW50KAogICAgICAgICAgICAgICAgICAgICAgICBub2RlX2lkPW5vZGUuaWQsCiAgICAgICAgICAgICAgICAgICAgICAgIGtpbmQ9bm9kZS5raW5kLnZhbHVlLAogICAgICAgICAgICAgICAgICAgICAgICBwYXlsb2FkPXsicGFzc2VkIjogY29udHJhZGljdGlvbi5wYXNzZWQgYW5kIGNhbGlicmF0aW9uLnBhc3NlZH0sCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBOb2RlS2luZC5UT09MOgogICAgICAgICAgICAgICAgcXVhbGl0eSA9IHNlbGYub3V0Y29tZV9wcmVkaWN0b3IucHJlZGljdChwcm9tcHQsIGV2aWRlbmNlX2NvdW50PTEsIHRvb2xfY291bnQ9MSkKICAgICAgICAgICAgICAgIG91dHB1dHNbInRvb2xfcmVzdWx0Il0gPSBxdWFsaXR5Lm1vZGVsX2R1bXAoKQogICAgICAgICAgICAgICAgZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICBSdW50aW1lRXZlbnQoCiAgICAgICAgICAgICAgICAgICAgICAgIG5vZGVfaWQ9bm9kZS5pZCwga2luZD1ub2RlLmtpbmQudmFsdWUsIHBheWxvYWQ9b3V0cHV0c1sidG9vbF9yZXN1bHQiXQogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxpZiBub2RlLmtpbmQgPT0gTm9kZUtpbmQuTUVNT1JZX1dSSVRFOgogICAgICAgICAgICAgICAgb3V0cHV0c1sibWVtb3J5X3dyaXRlIl0gPSB7CiAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJjYXB0dXJlZCIsCiAgICAgICAgICAgICAgICAgICAgImtleXMiOiBbImludGVycHJldGF0aW9ucyIsICJ2ZXJpZmljYXRpb24iXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUnVudGltZUV2ZW50KAogICAgICAgICAgICAgICAgICAgICAgICBub2RlX2lkPW5vZGUuaWQsIGtpbmQ9bm9kZS5raW5kLnZhbHVlLCBwYXlsb2FkPW91dHB1dHNbIm1lbW9yeV93cml0ZSJdCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBOb2RlS2luZC5NRVJHRToKICAgICAgICAgICAgICAgIGJlc3QgPSBpbnRlcnByZXRhdGlvbnNbMF0KICAgICAgICAgICAgICAgIG91dHB1dHNbIm1lcmdlZCJdID0gewogICAgICAgICAgICAgICAgICAgICJzdGF0ZW1lbnQiOiBiZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgICAgICAgICAiY29uZmlkZW5jZSI6IGJlc3QuY29uZmlkZW5jZSwKICAgICAgICAgICAgICAgICAgICAic3Vydml2b3JzIjogbGVuKGludGVycHJldGF0aW9ucyksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICBldmVudHMuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgIFJ1bnRpbWVFdmVudChub2RlX2lkPW5vZGUuaWQsIGtpbmQ9bm9kZS5raW5kLnZhbHVlLCBwYXlsb2FkPW91dHB1dHNbIm1lcmdlZCJdKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBOb2RlS2luZC5PVVRQVVQ6CiAgICAgICAgICAgICAgICBvdXRwdXRzWyJvdXRwdXQiXSA9IG91dHB1dHMuZ2V0KCJtZXJnZWQiLCB7fSkKICAgICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUnVudGltZUV2ZW50KG5vZGVfaWQ9bm9kZS5pZCwga2luZD1ub2RlLmtpbmQudmFsdWUsIHBheWxvYWQ9b3V0cHV0c1sib3V0cHV0Il0pCiAgICAgICAgICAgICAgICApCgogICAgICAgIHJldHVybiBSdW50aW1lUmVzdWx0KG91dHB1dHM9b3V0cHV0cywgZXZlbnRzPWV2ZW50cykK', 'olympus/foundry/__init__.py': 'IiIiRHVyYWJsZSBtb2RlbC1kZXZlbG9wbWVudCBpbmZyYXN0cnVjdHVyZSBmb3IgT2x5bXB1cy4iIiIKCmZyb20gb2x5bXB1cy5mb3VuZHJ5LnNlcnZpY2UgaW1wb3J0IEZvdW5kcnlTZXJ2aWNlCgpfX2FsbF9fID0gWyJGb3VuZHJ5U2VydmljZSJdCg==', 'olympus/foundry/bigram.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcmFuZG9tIGltcG9ydCBSYW5kb20KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgU2VsZgoKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xhc3MgQmlncmFtQ2hlY2twb2ludChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gMQogICAgbW9kZWxfdHlwZTogc3RyID0gImNoYXJhY3Rlci1iaWdyYW0iCiAgICBhbHBoYWJldDogbGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0yKQogICAgY291bnRzOiBsaXN0W2xpc3RbaW50XV0KICAgIHNtb290aGluZzogZmxvYXQgPSBGaWVsZChndD0wLjAsIGxlPTEwLjApCiAgICBzZWVkOiBpbnQgPSBGaWVsZChnZT0wLCBsZT0yKiozMiAtIDEpCiAgICB0cmFpbmluZ19jaGFyYWN0ZXJzOiBpbnQgPSBGaWVsZChndD0xKQogICAgdHJhaW5pbmdfcHJvdmVuYW5jZTogZGljdFtzdHIsIEFueV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKCiAgICBkZWYgY2Fub25pY2FsX2J5dGVzKHNlbGYpIC0+IGJ5dGVzOgogICAgICAgIHJldHVybiBqc29uLmR1bXBzKAogICAgICAgICAgICBzZWxmLm1vZGVsX2R1bXAobW9kZT0ianNvbiIpLAogICAgICAgICAgICBlbnN1cmVfYXNjaWk9RmFsc2UsCiAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlLAogICAgICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IiksCiAgICAgICAgKS5lbmNvZGUoInV0Zi04IikKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBmcm9tX2J5dGVzKGNscywgcGF5bG9hZDogYnl0ZXMpIC0+IFNlbGY6CiAgICAgICAgcmV0dXJuIGNscy5tb2RlbF92YWxpZGF0ZV9qc29uKHBheWxvYWQpCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgTGFuZ3VhZ2VNb2RlbE1ldHJpY3M6CiAgICBuZWdhdGl2ZV9sb2dfbGlrZWxpaG9vZDogZmxvYXQKICAgIHBlcnBsZXhpdHk6IGZsb2F0CiAgICB0b3AxX2FjY3VyYWN5OiBmbG9hdAogICAgZXZhbHVhdGVkX3RyYW5zaXRpb25zOiBpbnQKCiAgICBkZWYgYXNfZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJuZWdhdGl2ZV9sb2dfbGlrZWxpaG9vZCI6IHNlbGYubmVnYXRpdmVfbG9nX2xpa2VsaWhvb2QsCiAgICAgICAgICAgICJwZXJwbGV4aXR5Ijogc2VsZi5wZXJwbGV4aXR5LAogICAgICAgICAgICAidG9wMV9hY2N1cmFjeSI6IHNlbGYudG9wMV9hY2N1cmFjeSwKICAgICAgICAgICAgImV2YWx1YXRlZF90cmFuc2l0aW9ucyI6IGZsb2F0KHNlbGYuZXZhbHVhdGVkX3RyYW5zaXRpb25zKSwKICAgICAgICB9CgoKY2xhc3MgQ2hhcmFjdGVyQmlncmFtTW9kZWw6CiAgICAiIiJBIHNtYWxsIHJlYWwgbGFuZ3VhZ2UgbW9kZWwgdXNlZCB0byB2ZXJpZnkgdGhlIEZvdW5kcnkgbGlmZWN5Y2xlIGNoZWFwbHkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoZWNrcG9pbnQ6IEJpZ3JhbUNoZWNrcG9pbnQpIC0+IE5vbmU6CiAgICAgICAgc2l6ZSA9IGxlbihjaGVja3BvaW50LmFscGhhYmV0KQogICAgICAgIGlmIGxlbihzZXQoY2hlY2twb2ludC5hbHBoYWJldCkpICE9IHNpemUgb3IgYW55KAogICAgICAgICAgICBsZW4oY2hhcmFjdGVyKSAhPSAxIGZvciBjaGFyYWN0ZXIgaW4gY2hlY2twb2ludC5hbHBoYWJldAogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgYWxwaGFiZXQgbXVzdCBjb250YWluIHVuaXF1ZSBzaW5nbGUgY2hhcmFjdGVycyIpCiAgICAgICAgaWYgbGVuKGNoZWNrcG9pbnQuY291bnRzKSAhPSBzaXplIG9yIGFueShsZW4ocm93KSAhPSBzaXplIGZvciByb3cgaW4gY2hlY2twb2ludC5jb3VudHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGNvdW50IG1hdHJpeCBkb2VzIG5vdCBtYXRjaCBhbHBoYWJldCIpCiAgICAgICAgaWYgYW55KHZhbHVlIDwgMCBmb3Igcm93IGluIGNoZWNrcG9pbnQuY291bnRzIGZvciB2YWx1ZSBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGNvdW50cyBjYW5ub3QgYmUgbmVnYXRpdmUiKQogICAgICAgIHNlbGYuY2hlY2twb2ludCA9IGNoZWNrcG9pbnQKICAgICAgICBzZWxmLl9pbmRleCA9IHtjaGFyYWN0ZXI6IGluZGV4IGZvciBpbmRleCwgY2hhcmFjdGVyIGluIGVudW1lcmF0ZShjaGVja3BvaW50LmFscGhhYmV0KX0KCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiB0cmFpbihjbHMsIHRleHQ6IHN0ciwgKiwgc21vb3RoaW5nOiBmbG9hdCA9IDAuMjUsIHNlZWQ6IGludCA9IDcpIC0+IFNlbGY6CiAgICAgICAgaWYgbGVuKHRleHQpIDwgMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW5pbmcgdGV4dCBtdXN0IGNvbnRhaW4gYXQgbGVhc3QgdHdvIGNoYXJhY3RlcnMiKQogICAgICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHNtb290aGluZykgb3Igc21vb3RoaW5nIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNtb290aGluZyBtdXN0IGJlIGEgZmluaXRlIHBvc2l0aXZlIHZhbHVlIikKICAgICAgICBhbHBoYWJldCA9IHNvcnRlZChzZXQodGV4dCkpCiAgICAgICAgaWYgbGVuKGFscGhhYmV0KSA8IDI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluaW5nIHRleHQgbXVzdCBjb250YWluIGF0IGxlYXN0IHR3byBkaXN0aW5jdCBjaGFyYWN0ZXJzIikKICAgICAgICBpbmRleCA9IHtjaGFyYWN0ZXI6IHBvc2l0aW9uIGZvciBwb3NpdGlvbiwgY2hhcmFjdGVyIGluIGVudW1lcmF0ZShhbHBoYWJldCl9CiAgICAgICAgY291bnRzID0gW1swIGZvciBfIGluIGFscGhhYmV0XSBmb3IgXyBpbiBhbHBoYWJldF0KICAgICAgICBmb3IgY3VycmVudCwgZm9sbG93aW5nIGluIHppcCh0ZXh0LCB0ZXh0WzE6XSwgc3RyaWN0PUZhbHNlKToKICAgICAgICAgICAgY291bnRzW2luZGV4W2N1cnJlbnRdXVtpbmRleFtmb2xsb3dpbmddXSArPSAxCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAgQmlncmFtQ2hlY2twb2ludCgKICAgICAgICAgICAgICAgIGFscGhhYmV0PWFscGhhYmV0LAogICAgICAgICAgICAgICAgY291bnRzPWNvdW50cywKICAgICAgICAgICAgICAgIHNtb290aGluZz1zbW9vdGhpbmcsCiAgICAgICAgICAgICAgICBzZWVkPXNlZWQsCiAgICAgICAgICAgICAgICB0cmFpbmluZ19jaGFyYWN0ZXJzPWxlbih0ZXh0KSwKICAgICAgICAgICAgKQogICAgICAgICkKCiAgICBkZWYgX3Byb2JhYmlsaXRpZXMoc2VsZiwgY2hhcmFjdGVyOiBzdHIpIC0+IGxpc3RbZmxvYXRdOgogICAgICAgIHJvd19pbmRleCA9IHNlbGYuX2luZGV4LmdldChjaGFyYWN0ZXIpCiAgICAgICAgc2l6ZSA9IGxlbihzZWxmLmNoZWNrcG9pbnQuYWxwaGFiZXQpCiAgICAgICAgaWYgcm93X2luZGV4IGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBbMS4wIC8gc2l6ZV0gKiBzaXplCiAgICAgICAgcm93ID0gc2VsZi5jaGVja3BvaW50LmNvdW50c1tyb3dfaW5kZXhdCiAgICAgICAgZGVub21pbmF0b3IgPSBzdW0ocm93KSArIHNlbGYuY2hlY2twb2ludC5zbW9vdGhpbmcgKiBzaXplCiAgICAgICAgcmV0dXJuIFsodmFsdWUgKyBzZWxmLmNoZWNrcG9pbnQuc21vb3RoaW5nKSAvIGRlbm9taW5hdG9yIGZvciB2YWx1ZSBpbiByb3ddCgogICAgZGVmIGV2YWx1YXRlKHNlbGYsIHRleHQ6IHN0cikgLT4gTGFuZ3VhZ2VNb2RlbE1ldHJpY3M6CiAgICAgICAgdHJhbnNpdGlvbnMgPSBsaXN0KHppcCh0ZXh0LCB0ZXh0WzE6XSwgc3RyaWN0PUZhbHNlKSkKICAgICAgICBpZiBub3QgdHJhbnNpdGlvbnM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2YWx1YXRpb24gdGV4dCBtdXN0IGNvbnRhaW4gYXQgbGVhc3QgdHdvIGNoYXJhY3RlcnMiKQogICAgICAgIG5lZ2F0aXZlX2xvZ19saWtlbGlob29kID0gMC4wCiAgICAgICAgY29ycmVjdCA9IDAKICAgICAgICBmb3IgY3VycmVudCwgZXhwZWN0ZWQgaW4gdHJhbnNpdGlvbnM6CiAgICAgICAgICAgIHByb2JhYmlsaXRpZXMgPSBzZWxmLl9wcm9iYWJpbGl0aWVzKGN1cnJlbnQpCiAgICAgICAgICAgIGV4cGVjdGVkX2luZGV4ID0gc2VsZi5faW5kZXguZ2V0KGV4cGVjdGVkKQogICAgICAgICAgICBwcm9iYWJpbGl0eSA9ICgKICAgICAgICAgICAgICAgIHByb2JhYmlsaXRpZXNbZXhwZWN0ZWRfaW5kZXhdCiAgICAgICAgICAgICAgICBpZiBleHBlY3RlZF9pbmRleCBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgZWxzZSAxLjAgLyBsZW4oc2VsZi5jaGVja3BvaW50LmFscGhhYmV0KQogICAgICAgICAgICApCiAgICAgICAgICAgIG5lZ2F0aXZlX2xvZ19saWtlbGlob29kIC09IG1hdGgubG9nKG1heChwcm9iYWJpbGl0eSwgMWUtMTIpKQogICAgICAgICAgICBwcmVkaWN0ZWQgPSBtYXgocmFuZ2UobGVuKHByb2JhYmlsaXRpZXMpKSwga2V5PXByb2JhYmlsaXRpZXMuX19nZXRpdGVtX18pCiAgICAgICAgICAgIGNvcnJlY3QgKz0gaW50KGV4cGVjdGVkX2luZGV4ID09IHByZWRpY3RlZCkKICAgICAgICBhdmVyYWdlX25sbCA9IG5lZ2F0aXZlX2xvZ19saWtlbGlob29kIC8gbGVuKHRyYW5zaXRpb25zKQogICAgICAgIHJldHVybiBMYW5ndWFnZU1vZGVsTWV0cmljcygKICAgICAgICAgICAgbmVnYXRpdmVfbG9nX2xpa2VsaWhvb2Q9YXZlcmFnZV9ubGwsCiAgICAgICAgICAgIHBlcnBsZXhpdHk9bWF0aC5leHAoYXZlcmFnZV9ubGwpLAogICAgICAgICAgICB0b3AxX2FjY3VyYWN5PWNvcnJlY3QgLyBsZW4odHJhbnNpdGlvbnMpLAogICAgICAgICAgICBldmFsdWF0ZWRfdHJhbnNpdGlvbnM9bGVuKHRyYW5zaXRpb25zKSwKICAgICAgICApCgogICAgZGVmIGdlbmVyYXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgKiwKICAgICAgICBtYXhfY2hhcmFjdGVyczogaW50ID0gMTYwLAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuNywKICAgICAgICBzZWVkOiBpbnQgfCBOb25lID0gTm9uZSwKICAgICkgLT4gc3RyOgogICAgICAgIGlmIG1heF9jaGFyYWN0ZXJzIDw9IDAgb3IgbWF4X2NoYXJhY3RlcnMgPiA0XzA5NjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X2NoYXJhY3RlcnMgbXVzdCBiZSBiZXR3ZWVuIDEgYW5kIDQwOTYiKQogICAgICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHRlbXBlcmF0dXJlKSBvciB0ZW1wZXJhdHVyZSA8IDAgb3IgdGVtcGVyYXR1cmUgPiAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0ZW1wZXJhdHVyZSBtdXN0IGJlIGJldHdlZW4gMCBhbmQgMiIpCiAgICAgICAgY3VycmVudCA9IHByb21wdFstMV0gaWYgcHJvbXB0IGVsc2UgIlxuIgogICAgICAgIGlmIGN1cnJlbnQgbm90IGluIHNlbGYuX2luZGV4OgogICAgICAgICAgICBjdXJyZW50ID0gIlxuIiBpZiAiXG4iIGluIHNlbGYuX2luZGV4IGVsc2Ugc2VsZi5jaGVja3BvaW50LmFscGhhYmV0WzBdCiAgICAgICAgcmFuZG9tID0gUmFuZG9tKHNlbGYuY2hlY2twb2ludC5zZWVkIGlmIHNlZWQgaXMgTm9uZSBlbHNlIHNlZWQpCiAgICAgICAgZ2VuZXJhdGVkOiBsaXN0W3N0cl0gPSBbXQogICAgICAgIGZvciBfIGluIHJhbmdlKG1heF9jaGFyYWN0ZXJzKToKICAgICAgICAgICAgcHJvYmFiaWxpdGllcyA9IHNlbGYuX3Byb2JhYmlsaXRpZXMoY3VycmVudCkKICAgICAgICAgICAgaWYgdGVtcGVyYXR1cmUgPT0gMDoKICAgICAgICAgICAgICAgIG5leHRfaW5kZXggPSBtYXgocmFuZ2UobGVuKHByb2JhYmlsaXRpZXMpKSwga2V5PXByb2JhYmlsaXRpZXMuX19nZXRpdGVtX18pCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAjIENvbW1vbiBwb3NpdGl2ZSByZXNjYWxpbmcgbGVhdmVzIHRoZSBjYXRlZ29yaWNhbCBsYXcgdW5jaGFuZ2VkLgogICAgICAgICAgICAgICAgIyBEaXZpZGUgYmVmb3JlIHRha2luZyBsb2dzIHRvIHJldGFpbiBkaXN0aW5jdCBuZWFyLW1heGltdW0gZmxvYXRzLgogICAgICAgICAgICAgICAgIyBBdCBsZWFzdCBvbmUgd2VpZ2h0IGlzIDEgZXZlbiB3aGVuIGV2ZXJ5IHAqKigxL1QpIHdvdWxkIHVuZGVyZmxvdy4KICAgICAgICAgICAgICAgIG1heGltdW1fcHJvYmFiaWxpdHkgPSBtYXgocHJvYmFiaWxpdGllcykKICAgICAgICAgICAgICAgIHdlaWdodHMgPSBbCiAgICAgICAgICAgICAgICAgICAgMC4wCiAgICAgICAgICAgICAgICAgICAgaWYgcHJvYmFiaWxpdHkgPT0gMC4wCiAgICAgICAgICAgICAgICAgICAgZWxzZSBtYXRoLmV4cChtYXRoLmxvZyhwcm9iYWJpbGl0eSAvIG1heGltdW1fcHJvYmFiaWxpdHkpIC8gdGVtcGVyYXR1cmUpCiAgICAgICAgICAgICAgICAgICAgZm9yIHByb2JhYmlsaXR5IGluIHByb2JhYmlsaXRpZXMKICAgICAgICAgICAgICAgIF0KICAgICAgICAgICAgICAgIG5leHRfaW5kZXggPSByYW5kb20uY2hvaWNlcyhyYW5nZShsZW4od2VpZ2h0cykpLCB3ZWlnaHRzPXdlaWdodHMsIGs9MSlbMF0KICAgICAgICAgICAgY3VycmVudCA9IHNlbGYuY2hlY2twb2ludC5hbHBoYWJldFtuZXh0X2luZGV4XQogICAgICAgICAgICBnZW5lcmF0ZWQuYXBwZW5kKGN1cnJlbnQpCiAgICAgICAgcmV0dXJuICIiLmpvaW4oZ2VuZXJhdGVkKQoKCmRlZiB1bmlmb3JtX2Jhc2VsaW5lX21ldHJpY3ModGV4dDogc3RyLCBhbHBoYWJldF9zaXplOiBpbnQpIC0+IExhbmd1YWdlTW9kZWxNZXRyaWNzOgogICAgaWYgbGVuKHRleHQpIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJldmFsdWF0aW9uIHRleHQgbXVzdCBjb250YWluIGF0IGxlYXN0IHR3byBjaGFyYWN0ZXJzIikKICAgIGlmIGFscGhhYmV0X3NpemUgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFscGhhYmV0X3NpemUgbXVzdCBiZSBhdCBsZWFzdCB0d28iKQogICAgdHJhbnNpdGlvbnMgPSBsZW4odGV4dCkgLSAxCiAgICBubGwgPSBtYXRoLmxvZyhhbHBoYWJldF9zaXplKQogICAgcmV0dXJuIExhbmd1YWdlTW9kZWxNZXRyaWNzKAogICAgICAgIG5lZ2F0aXZlX2xvZ19saWtlbGlob29kPW5sbCwKICAgICAgICBwZXJwbGV4aXR5PWZsb2F0KGFscGhhYmV0X3NpemUpLAogICAgICAgIHRvcDFfYWNjdXJhY3k9MS4wIC8gYWxwaGFiZXRfc2l6ZSwKICAgICAgICBldmFsdWF0ZWRfdHJhbnNpdGlvbnM9dHJhbnNpdGlvbnMsCiAgICApCg==', 'olympus/foundry/data_pipeline.py': '...', 'olympus/foundry/eval_suite.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbApmcm9tIG9seW1wdXMuZm91bmRyeS5kYXRhX3BpcGVsaW5lIGltcG9ydCBJbnN0cnVjdGlvbkV4YW1wbGUsIGxvYWRfdmVyaWZpZWRfZGF0YXNldApmcm9tIG9seW1wdXMuZm91bmRyeS5yZXNvdXJjZXMgaW1wb3J0IG1lbW9yeV9zbmFwc2hvdApmcm9tIG9seW1wdXMuZm91bmRyeS5zZnQgaW1wb3J0ICgKICAgIExvc3NOb3JtYWxpemF0aW9uLAogICAgVGlueUNhdXNhbExNLAogICAgVGlueU1vZGVsQ29uZmlnLAogICAgX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmlnLAogICAgX2VuY29kZV9leGFtcGxlcywKICAgIF9tb2RlbF9mcm9tX2NoZWNrcG9pbnQsCiAgICBldmFsdWF0ZV9sb3NzLAogICAgZ2VuZXJhdGVfdGV4dCwKKQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNlcmlhbGl6ZWQgPSAoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIikuZW5jb2RlKCkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIGhhbmRsZS53cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmNsYXNzIENhdGVnb3J5RXZhbHVhdGlvbihTdHJpY3RNb2RlbCk6CiAgICBjYXRlZ29yeTogc3RyCiAgICByZWNvcmRzOiBpbnQgPSBGaWVsZChndD0wKQogICAgYmFzZWxpbmVfbG9zczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBjYW5kaWRhdGVfbG9zczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBsb3NzX2NoYW5nZV9mcmFjdGlvbjogZmxvYXQKICAgIHJlZ3Jlc3NlZDogYm9vbAoKCmNsYXNzIFdvcmtmbG93U2NvcmUoU3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyCiAgICBjYXRlZ29yaWVzOiBsaXN0W3N0cl0KICAgIGV4YW1wbGVzOiBpbnQgPSBGaWVsZChndD0wKQogICAgZXhhY3RfbWF0Y2hfcmF0ZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIG5vbmVtcHR5X3JhdGU6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCgoKY2xhc3MgSGVsZE91dEV2YWx1YXRpb24oU3RyaWN0TW9kZWwpOgogICAgc2NoZW1hX3ZlcnNpb246IGludCA9IDEKICAgIHN1aXRlOiBzdHIgPSAib2x5bXB1cy1oZWxkLW91dC1jYXBhYmlsaXRpZXMtdjEiCiAgICBjaGVja3BvaW50X3BhdGg6IHN0cgogICAgY2hlY2twb2ludF9zaGEyNTY6IHN0cgogICAgZGF0YXNldF9tYW5pZmVzdF9zaGEyNTY6IHN0cgogICAgYmFzZWxpbmVfaWRlbnRpdHk6IHN0cgogICAgdGVzdF9yZWNvcmRzOiBpbnQgPSBGaWVsZChndD0wKQogICAgb3ZlcmFsbF9iYXNlbGluZV9sb3NzOiBmbG9hdAogICAgb3ZlcmFsbF9jYW5kaWRhdGVfbG9zczogZmxvYXQKICAgIG92ZXJhbGxfYmFzZWxpbmVfcGVycGxleGl0eTogZmxvYXQKICAgIG92ZXJhbGxfY2FuZGlkYXRlX3BlcnBsZXhpdHk6IGZsb2F0CiAgICBjYXRlZ29yeV9yZXN1bHRzOiBsaXN0W0NhdGVnb3J5RXZhbHVhdGlvbl0KICAgIHJlZ3Jlc3Npb25fY291bnQ6IGludCA9IEZpZWxkKGdlPTApCiAgICBleGFjdF9tYXRjaF9yYXRlOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgZm9ybWF0X2NvbXBsaWFuY2VfcmF0ZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIHdvcmtmbG93X3Njb3JlczogbGlzdFtXb3JrZmxvd1Njb3JlXQogICAgbWVkaWFuX2dlbmVyYXRpb25fbGF0ZW5jeV9tczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBnZW5lcmF0ZWRfYnl0ZXNfcGVyX3NlY29uZDogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBwZWFrX3Jzc19ieXRlczogaW50ID0gRmllbGQoZ2U9MCkKICAgIHBhc3NlZF9zbW9rZV9xdWFsaXR5X2dhdGU6IGJvb2wKICAgIGRlY2lzaW9uOiBzdHIKICAgIGxvc3Nfbm9ybWFsaXphdGlvbjogTG9zc05vcm1hbGl6YXRpb24gPSAibGVnYWN5X2JhdGNoX21lYW5fdjEiCgoKZGVmIF9sb2FkX3Rlc3RfZXhhbXBsZXMobWFuaWZlc3RfcGF0aDogUGF0aCkgLT4gdHVwbGVbc3RyLCBsaXN0W0luc3RydWN0aW9uRXhhbXBsZV1dOgogICAgbWFuaWZlc3QsIGV4YW1wbGVzX2J5X3NwbGl0ID0gbG9hZF92ZXJpZmllZF9kYXRhc2V0KG1hbmlmZXN0X3BhdGgpCiAgICBpZiBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYgaXMgTm9uZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkYXRhc2V0IG1hbmlmZXN0IGhhcyBubyBoYXNoIikKICAgIHJldHVybiBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYsIGV4YW1wbGVzX2J5X3NwbGl0WyJ0ZXN0Il0KCgpkZWYgX2Jhc2VsaW5lX21vZGVsKAogICAgY2hlY2twb2ludDogZGljdFtzdHIsIEFueV0sIGJhc2VfY2hlY2twb2ludDogUGF0aCB8IE5vbmUKKSAtPiB0dXBsZVtUaW55Q2F1c2FsTE0sIHN0cl06CiAgICBpZiBiYXNlX2NoZWNrcG9pbnQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwsIF8gPSBfbW9kZWxfZnJvbV9jaGVja3BvaW50KGJhc2VfY2hlY2twb2ludCwgdG9yY2guZGV2aWNlKCJjcHUiKSkKICAgICAgICByZXR1cm4gbW9kZWwsIGYiY2hlY2twb2ludDp7X3NoYTI1NihiYXNlX2NoZWNrcG9pbnQpfSIKICAgIHRyYWluaW5nID0gX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmlnKGNoZWNrcG9pbnQpCiAgICB0b3JjaC5tYW51YWxfc2VlZCh0cmFpbmluZy5zZWVkKQogICAgcmV0dXJuIFRpbnlDYXVzYWxMTShUaW55TW9kZWxDb25maWcubW9kZWxfdmFsaWRhdGUoY2hlY2twb2ludFsibW9kZWxfY29uZmlnIl0pKSwgKAogICAgICAgIGYiZGV0ZXJtaW5pc3RpYy11bnRyYWluZWQtc2VlZC17dHJhaW5pbmcuc2VlZH0iCiAgICApCgoKZGVmIF9ub3JtYWxpemUodGV4dDogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gIiAiLmpvaW4odGV4dC5jYXNlZm9sZCgpLnN0cmlwKCkuc3BsaXQoKSkKCgpkZWYgX2Zvcm1hdF9jb21wbGlhbnQoZXhwZWN0ZWQ6IHN0ciwgZ2VuZXJhdGVkOiBzdHIpIC0+IGJvb2w6CiAgICBpZiBleHBlY3RlZC5sc3RyaXAoKS5zdGFydHN3aXRoKCJ7Iik6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBqc29uLmxvYWRzKGdlbmVyYXRlZCkKICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHJldHVybiBpc2luc3RhbmNlKHBhcnNlZCwgZGljdCkKICAgIHJldHVybiBib29sKGdlbmVyYXRlZC5zdHJpcCgpKSBhbmQgIlx1ZmZmZCIgbm90IGluIGdlbmVyYXRlZAoKCmRlZiBfd29ya2Zsb3dfc2NvcmVzKAogICAgZXhhbXBsZXM6IGxpc3RbSW5zdHJ1Y3Rpb25FeGFtcGxlXSwgZ2VuZXJhdGVkOiBkaWN0W3N0ciwgc3RyXQopIC0+IGxpc3RbV29ya2Zsb3dTY29yZV06CiAgICBkZWZpbml0aW9ucyA9IHsKICAgICAgICAidG9vbF93b3JrZmxvdyI6IFsidG9vbF91c2UiLCAiYWdlbnRfYmVoYXZpb3IiXSwKICAgICAgICAicmVzdWx0X2NvbXBhcmlzb24iOiBbInJlYXNvbmluZyIsICJzZWxmX2NvcnJlY3Rpb24iLCAicmVzZWFyY2giXSwKICAgICAgICAibXVsdGlfc3RlcF9wbGFubmluZyI6IFsicGxhbm5pbmciLCAic2FmZXR5Il0sCiAgICB9CiAgICByZXN1bHRzOiBsaXN0W1dvcmtmbG93U2NvcmVdID0gW10KICAgIGZvciBuYW1lLCBjYXRlZ29yaWVzIGluIGRlZmluaXRpb25zLml0ZW1zKCk6CiAgICAgICAgc2VsZWN0ZWQgPSBbZXhhbXBsZSBmb3IgZXhhbXBsZSBpbiBleGFtcGxlcyBpZiBleGFtcGxlLmNhdGVnb3J5IGluIGNhdGVnb3JpZXNdCiAgICAgICAgZXhhY3QgPSBzdW0oCiAgICAgICAgICAgIF9ub3JtYWxpemUoZ2VuZXJhdGVkW2l0ZW0uaWRdKSA9PSBfbm9ybWFsaXplKGl0ZW0ucmVzcG9uc2UpIGZvciBpdGVtIGluIHNlbGVjdGVkCiAgICAgICAgKQogICAgICAgIG5vbmVtcHR5ID0gc3VtKGJvb2woZ2VuZXJhdGVkW2l0ZW0uaWRdLnN0cmlwKCkpIGZvciBpdGVtIGluIHNlbGVjdGVkKQogICAgICAgIHJlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICBXb3JrZmxvd1Njb3JlKAogICAgICAgICAgICAgICAgbmFtZT1uYW1lLAogICAgICAgICAgICAgICAgY2F0ZWdvcmllcz1jYXRlZ29yaWVzLAogICAgICAgICAgICAgICAgZXhhbXBsZXM9bGVuKHNlbGVjdGVkKSwKICAgICAgICAgICAgICAgIGV4YWN0X21hdGNoX3JhdGU9ZXhhY3QgLyBsZW4oc2VsZWN0ZWQpLAogICAgICAgICAgICAgICAgbm9uZW1wdHlfcmF0ZT1ub25lbXB0eSAvIGxlbihzZWxlY3RlZCksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBldmFsdWF0ZV9jaGVja3BvaW50KAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgbWFuaWZlc3RfcGF0aDogUGF0aCwKICAgIG91dHB1dF9wYXRoOiBQYXRoLAogICAgKiwKICAgIGJhc2VfY2hlY2twb2ludDogUGF0aCB8IE5vbmUgPSBOb25lLAogICAgbWF4X2dlbmVyYXRpb25fdG9rZW5zOiBpbnQgPSA0OCwKKSAtPiBIZWxkT3V0RXZhbHVhdGlvbjoKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3B1IikKICAgIGNhbmRpZGF0ZSwgY2hlY2twb2ludCA9IF9tb2RlbF9mcm9tX2NoZWNrcG9pbnQoY2hlY2twb2ludF9wYXRoLCBkZXZpY2UpCiAgICBiYXNlbGluZSwgYmFzZWxpbmVfaWRlbnRpdHkgPSBfYmFzZWxpbmVfbW9kZWwoY2hlY2twb2ludCwgYmFzZV9jaGVja3BvaW50KQogICAgbWFuaWZlc3Rfc2hhLCBleGFtcGxlcyA9IF9sb2FkX3Rlc3RfZXhhbXBsZXMobWFuaWZlc3RfcGF0aCkKICAgIGlmIGNoZWNrcG9pbnRbImRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2Il0gIT0gbWFuaWZlc3Rfc2hhOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgYW5kIGV2YWx1YXRpb24gZGF0YXNldCBoYXNoZXMgZG8gbm90IG1hdGNoIikKICAgIHRyYWluaW5nID0gX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmlnKGNoZWNrcG9pbnQpCiAgICBldmFsdWF0aW9uX2NvbmZpZyA9IHRyYWluaW5nLm1vZGVsX2NvcHkodXBkYXRlPXsicGFja19zZXF1ZW5jZXMiOiBGYWxzZX0pCiAgICBhbGxfcm93cyA9IF9lbmNvZGVfZXhhbXBsZXMoZXhhbXBsZXMsIGV2YWx1YXRpb25fY29uZmlnKQogICAgYmFzZWxpbmVfbG9zcyA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgYmFzZWxpbmUsIGFsbF9yb3dzLCBkZXZpY2U9ZGV2aWNlLCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAgKQogICAgY2FuZGlkYXRlX2xvc3MgPSBldmFsdWF0ZV9sb3NzKAogICAgICAgIGNhbmRpZGF0ZSwgYWxsX3Jvd3MsIGRldmljZT1kZXZpY2UsIG5vcm1hbGl6YXRpb249dHJhaW5pbmcubG9zc19ub3JtYWxpemF0aW9uCiAgICApCgogICAgY2F0ZWdvcnlfcmVzdWx0czogbGlzdFtDYXRlZ29yeUV2YWx1YXRpb25dID0gW10KICAgIGZvciBjYXRlZ29yeSBpbiBzb3J0ZWQoe2V4YW1wbGUuY2F0ZWdvcnkgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXN9KToKICAgICAgICBzZWxlY3RlZCA9IFtleGFtcGxlIGZvciBleGFtcGxlIGluIGV4YW1wbGVzIGlmIGV4YW1wbGUuY2F0ZWdvcnkgPT0gY2F0ZWdvcnldCiAgICAgICAgcm93cyA9IF9lbmNvZGVfZXhhbXBsZXMoc2VsZWN0ZWQsIGV2YWx1YXRpb25fY29uZmlnKQogICAgICAgIGJlZm9yZSA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgICAgIGJhc2VsaW5lLCByb3dzLCBkZXZpY2U9ZGV2aWNlLCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAgICAgICkKICAgICAgICBhZnRlciA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgICAgIGNhbmRpZGF0ZSwgcm93cywgZGV2aWNlPWRldmljZSwgbm9ybWFsaXphdGlvbj10cmFpbmluZy5sb3NzX25vcm1hbGl6YXRpb24KICAgICAgICApCiAgICAgICAgY2hhbmdlID0gKGFmdGVyIC0gYmVmb3JlKSAvIG1heChiZWZvcmUsIDFlLTEyKQogICAgICAgIGNhdGVnb3J5X3Jlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICBDYXRlZ29yeUV2YWx1YXRpb24oCiAgICAgICAgICAgICAgICBjYXRlZ29yeT1jYXRlZ29yeSwKICAgICAgICAgICAgICAgIHJlY29yZHM9bGVuKHNlbGVjdGVkKSwKICAgICAgICAgICAgICAgIGJhc2VsaW5lX2xvc3M9YmVmb3JlLAogICAgICAgICAgICAgICAgY2FuZGlkYXRlX2xvc3M9YWZ0ZXIsCiAgICAgICAgICAgICAgICBsb3NzX2NoYW5nZV9mcmFjdGlvbj1jaGFuZ2UsCiAgICAgICAgICAgICAgICByZWdyZXNzZWQ9Y2hhbmdlID4gMC4wNSwKICAgICAgICAgICAgKQogICAgICAgICkKCiAgICBnZW5lcmF0ZWQ6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGxhdGVuY2llczogbGlzdFtmbG9hdF0gPSBbXQogICAgZ2VuZXJhdGVkX2J5dGVzID0gMAogICAgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXM6CiAgICAgICAgc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgYW5zd2VyID0gZ2VuZXJhdGVfdGV4dCgKICAgICAgICAgICAgY2FuZGlkYXRlLAogICAgICAgICAgICBleGFtcGxlLnByb21wdCwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X2dlbmVyYXRpb25fdG9rZW5zLAogICAgICAgICkKICAgICAgICBsYXRlbmNpZXMuYXBwZW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydCkKICAgICAgICBnZW5lcmF0ZWRfYnl0ZXMgKz0gbGVuKGFuc3dlci5lbmNvZGUoKSkKICAgICAgICBnZW5lcmF0ZWRbZXhhbXBsZS5pZF0gPSBhbnN3ZXIKICAgIGV4YWN0ID0gc3VtKF9ub3JtYWxpemUoZ2VuZXJhdGVkW2l0ZW0uaWRdKSA9PSBfbm9ybWFsaXplKGl0ZW0ucmVzcG9uc2UpIGZvciBpdGVtIGluIGV4YW1wbGVzKQogICAgY29tcGxpYW50ID0gc3VtKF9mb3JtYXRfY29tcGxpYW50KGl0ZW0ucmVzcG9uc2UsIGdlbmVyYXRlZFtpdGVtLmlkXSkgZm9yIGl0ZW0gaW4gZXhhbXBsZXMpCiAgICByZWdyZXNzaW9ucyA9IHN1bShyZXN1bHQucmVncmVzc2VkIGZvciByZXN1bHQgaW4gY2F0ZWdvcnlfcmVzdWx0cykKICAgIGV4YWN0X3JhdGUgPSBleGFjdCAvIGxlbihleGFtcGxlcykKICAgIGZvcm1hdF9yYXRlID0gY29tcGxpYW50IC8gbGVuKGV4YW1wbGVzKQogICAgcGFzc2VkID0gY2FuZGlkYXRlX2xvc3MgPCBiYXNlbGluZV9sb3NzIGFuZCByZWdyZXNzaW9ucyA9PSAwIGFuZCBmb3JtYXRfcmF0ZSA+PSAwLjgKICAgIGRlY2lzaW9uID0gKAogICAgICAgICJQYXNzZWQgdGhlIGluZnJhc3RydWN0dXJlIHNtb2tlIGdhdGU7IHRoaXMgZG9lcyBub3Qgc2F0aXNmeSBIZXJtZXMgcHJvbW90aW9uIGdhdGVzLiIKICAgICAgICBpZiBwYXNzZWQKICAgICAgICBlbHNlICJGYWlsZWQgdGhlIGluZnJhc3RydWN0dXJlIHNtb2tlIGdhdGUgYW5kIGlzIGluZWxpZ2libGUgZm9yIG1vZGVsIHByb21vdGlvbi4iCiAgICApCiAgICByZXBvcnQgPSBIZWxkT3V0RXZhbHVhdGlvbigKICAgICAgICBsb3NzX25vcm1hbGl6YXRpb249dHJhaW5pbmcubG9zc19ub3JtYWxpemF0aW9uLAogICAgICAgIGNoZWNrcG9pbnRfcGF0aD1zdHIoY2hlY2twb2ludF9wYXRoLnJlc29sdmUoKSksCiAgICAgICAgY2hlY2twb2ludF9zaGEyNTY9X3NoYTI1NihjaGVja3BvaW50X3BhdGgpLAogICAgICAgIGRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2PW1hbmlmZXN0X3NoYSwKICAgICAgICBiYXNlbGluZV9pZGVudGl0eT1iYXNlbGluZV9pZGVudGl0eSwKICAgICAgICB0ZXN0X3JlY29yZHM9bGVuKGV4YW1wbGVzKSwKICAgICAgICBvdmVyYWxsX2Jhc2VsaW5lX2xvc3M9YmFzZWxpbmVfbG9zcywKICAgICAgICBvdmVyYWxsX2NhbmRpZGF0ZV9sb3NzPWNhbmRpZGF0ZV9sb3NzLAogICAgICAgIG92ZXJhbGxfYmFzZWxpbmVfcGVycGxleGl0eT1tYXRoLmV4cChtaW4oYmFzZWxpbmVfbG9zcywgMjAuMCkpLAogICAgICAgIG92ZXJhbGxfY2FuZGlkYXRlX3BlcnBsZXhpdHk9bWF0aC5leHAobWluKGNhbmRpZGF0ZV9sb3NzLCAyMC4wKSksCiAgICAgICAgY2F0ZWdvcnlfcmVzdWx0cz1jYXRlZ29yeV9yZXN1bHRzLAogICAgICAgIHJlZ3Jlc3Npb25fY291bnQ9cmVncmVzc2lvbnMsCiAgICAgICAgZXhhY3RfbWF0Y2hfcmF0ZT1leGFjdF9yYXRlLAogICAgICAgIGZvcm1hdF9jb21wbGlhbmNlX3JhdGU9Zm9ybWF0X3JhdGUsCiAgICAgICAgd29ya2Zsb3dfc2NvcmVzPV93b3JrZmxvd19zY29yZXMoZXhhbXBsZXMsIGdlbmVyYXRlZCksCiAgICAgICAgbWVkaWFuX2dlbmVyYXRpb25fbGF0ZW5jeV9tcz1zdGF0aXN0aWNzLm1lZGlhbihsYXRlbmNpZXMpICogMV8wMDAsCiAgICAgICAgZ2VuZXJhdGVkX2J5dGVzX3Blcl9zZWNvbmQ9KAogICAgICAgICAgICBnZW5lcmF0ZWRfYnl0ZXMgLyBzdW0obGF0ZW5jaWVzKSBpZiBzdW0obGF0ZW5jaWVzKSBlbHNlIDAuMAogICAgICAgICksCiAgICAgICAgcGVha19yc3NfYnl0ZXM9bWVtb3J5X3NuYXBzaG90KCkucHJvY2Vzc19wZWFrX3Jzc19ieXRlcywKICAgICAgICBwYXNzZWRfc21va2VfcXVhbGl0eV9nYXRlPXBhc3NlZCwKICAgICAgICBkZWNpc2lvbj1kZWNpc2lvbiwKICAgICkKICAgIF9hdG9taWNfanNvbihvdXRwdXRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo=', 'olympus/foundry/load_probe.py': 'IiIiQm91bmRlZCBBU0dJIGxvYWQgcHJvYmUgZm9yIGVuZ2luZWVyaW5nIHF1YWxpZmljYXRpb24uCgpUaGUgcHJvYmUgaXMgaW50ZW50aW9uYWxseSBsYWJlbGxlZCBhcyBhIHNtb2tlIHRlc3QuICBJdHMgb3V0cHV0IGNhbm5vdCBzYXRpc2Z5CnRoZSAyNC1ob3VyIHByb2R1Y3Rpb24gZ2F0ZSBpbiBgYHJlYWRpbmVzcy5weWBgLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhc3luY2lvCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCBvcwppbXBvcnQgc3RhdGlzdGljcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0ZW1wZmlsZSBpbXBvcnQgTmFtZWRUZW1wb3JhcnlGaWxlCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBodHRweApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBMb2FkU21va2VSZXBvcnQoU3RyaWN0TW9kZWwpOgogICAgc2NoZW1hX3ZlcnNpb246IGludCA9IDEKICAgIGVuZHBvaW50OiBzdHIKICAgIHJlcXVlc3RzOiBpbnQgPSBGaWVsZChndD0wKQogICAgY29uY3VycmVuY3k6IGludCA9IEZpZWxkKGd0PTApCiAgICBzdWNjZXNzZnVsX3JlcXVlc3RzOiBpbnQgPSBGaWVsZChnZT0wKQogICAgZmFpbGVkX3JlcXVlc3RzOiBpbnQgPSBGaWVsZChnZT0wKQogICAgZXJyb3JfcmF0ZTogZmxvYXQgPSBGaWVsZChnZT0wLCBsZT0xKQogICAgZWxhcHNlZF9zZWNvbmRzOiBmbG9hdCA9IEZpZWxkKGd0PTApCiAgICB0aHJvdWdocHV0X3JlcXVlc3RzX3Blcl9zZWNvbmQ6IGZsb2F0ID0gRmllbGQoZ2U9MCkKICAgIHA1MF9sYXRlbmN5X21zOiBmbG9hdCA9IEZpZWxkKGdlPTApCiAgICBwOTVfbGF0ZW5jeV9tczogZmxvYXQgPSBGaWVsZChnZT0wKQogICAgcDk5X2xhdGVuY3lfbXM6IGZsb2F0ID0gRmllbGQoZ2U9MCkKICAgIHN0YXR1c19jb3VudHM6IGRpY3Rbc3RyLCBpbnRdCiAgICBxdWFsaWZpY2F0aW9uOiBzdHIgPSAiQk9VTkRFRF9FTkdJTkVFUklOR19TTU9LRV9OT1RfUFJPRFVDVElPTl9FVklERU5DRSIKCgpkZWYgX3BlcmNlbnRpbGUodmFsdWVzOiBsaXN0W2Zsb2F0XSwgcXVhbnRpbGU6IGZsb2F0KSAtPiBmbG9hdDoKICAgIGlmIG5vdCB2YWx1ZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGVyY2VudGlsZSByZXF1aXJlcyBhdCBsZWFzdCBvbmUgb2JzZXJ2YXRpb24iKQogICAgaWYgbm90IDAgPD0gcXVhbnRpbGUgPD0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJxdWFudGlsZSBtdXN0IGJlIGluIFswLCAxXSIpCiAgICBvcmRlcmVkID0gc29ydGVkKHZhbHVlcykKICAgIGluZGV4ID0gbWluKGxlbihvcmRlcmVkKSAtIDEsIG1heCgwLCBtYXRoLmNlaWwobGVuKG9yZGVyZWQpICogcXVhbnRpbGUpIC0gMSkpCiAgICByZXR1cm4gb3JkZXJlZFtpbmRleF0KCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNlcmlhbGl6ZWQgPSAoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIikuZW5jb2RlKCkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIGhhbmRsZS53cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmFzeW5jIGRlZiBwcm9iZV9hc2dpX2FwcCgKICAgIGFwcDogQW55LAogICAgKiwKICAgIGVuZHBvaW50OiBzdHIgPSAiL2hlYWx0aCIsCiAgICByZXF1ZXN0czogaW50ID0gMjAwLAogICAgY29uY3VycmVuY3k6IGludCA9IDE2LAopIC0+IExvYWRTbW9rZVJlcG9ydDoKICAgIGlmIHJlcXVlc3RzIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyZXF1ZXN0cyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIGNvbmN1cnJlbmN5IDwgMSBvciBjb25jdXJyZW5jeSA+IHJlcXVlc3RzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNvbmN1cnJlbmN5IG11c3QgYmUgaW4gWzEsIHJlcXVlc3RzXSIpCiAgICBzZW1hcGhvcmUgPSBhc3luY2lvLlNlbWFwaG9yZShjb25jdXJyZW5jeSkKICAgIGxhdGVuY2llczogbGlzdFtmbG9hdF0gPSBbXQogICAgc3RhdHVzZXM6IGRpY3Rbc3RyLCBpbnRdID0ge30KCiAgICBhc3luYyB3aXRoIGh0dHB4LkFzeW5jQ2xpZW50KAogICAgICAgIHRyYW5zcG9ydD1odHRweC5BU0dJVHJhbnNwb3J0KGFwcD1hcHApLCBiYXNlX3VybD0iaHR0cDovL29seW1wdXMudGVzdCIKICAgICkgYXMgY2xpZW50OgoKICAgICAgICBhc3luYyBkZWYgb25lKCkgLT4gTm9uZToKICAgICAgICAgICAgYXN5bmMgd2l0aCBzZW1hcGhvcmU6CiAgICAgICAgICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gYXdhaXQgY2xpZW50LmdldChlbmRwb2ludCkKICAgICAgICAgICAgICAgICAgICBrZXkgPSBzdHIocmVzcG9uc2Uuc3RhdHVzX2NvZGUpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOgogICAgICAgICAgICAgICAgICAgIGtleSA9IGYiZXJyb3I6e3R5cGUoZXJyb3IpLl9fbmFtZV9ffSIKICAgICAgICAgICAgICAgIGxhdGVuY2llcy5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAqIDFfMDAwKQogICAgICAgICAgICAgICAgc3RhdHVzZXNba2V5XSA9IHN0YXR1c2VzLmdldChrZXksIDApICsgMQoKICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGF3YWl0IGFzeW5jaW8uZ2F0aGVyKCoob25lKCkgZm9yIF8gaW4gcmFuZ2UocmVxdWVzdHMpKSkKICAgICAgICBlbGFwc2VkID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKCiAgICBzdWNjZXNzZnVsID0gc3VtKGNvdW50IGZvciBzdGF0dXMsIGNvdW50IGluIHN0YXR1c2VzLml0ZW1zKCkgaWYgc3RhdHVzLnN0YXJ0c3dpdGgoIjIiKSkKICAgIGZhaWxlZCA9IHJlcXVlc3RzIC0gc3VjY2Vzc2Z1bAogICAgcmV0dXJuIExvYWRTbW9rZVJlcG9ydCgKICAgICAgICBlbmRwb2ludD1lbmRwb2ludCwKICAgICAgICByZXF1ZXN0cz1yZXF1ZXN0cywKICAgICAgICBjb25jdXJyZW5jeT1jb25jdXJyZW5jeSwKICAgICAgICBzdWNjZXNzZnVsX3JlcXVlc3RzPXN1Y2Nlc3NmdWwsCiAgICAgICAgZmFpbGVkX3JlcXVlc3RzPWZhaWxlZCwKICAgICAgICBlcnJvcl9yYXRlPWZhaWxlZCAvIHJlcXVlc3RzLAogICAgICAgIGVsYXBzZWRfc2Vjb25kcz1lbGFwc2VkLAogICAgICAgIHRocm91Z2hwdXRfcmVxdWVzdHNfcGVyX3NlY29uZD1yZXF1ZXN0cyAvIGVsYXBzZWQsCiAgICAgICAgcDUwX2xhdGVuY3lfbXM9c3RhdGlzdGljcy5tZWRpYW4obGF0ZW5jaWVzKSwKICAgICAgICBwOTVfbGF0ZW5jeV9tcz1fcGVyY2VudGlsZShsYXRlbmNpZXMsIDAuOTUpLAogICAgICAgIHA5OV9sYXRlbmN5X21zPV9wZXJjZW50aWxlKGxhdGVuY2llcywgMC45OSksCiAgICAgICAgc3RhdHVzX2NvdW50cz1kaWN0KHNvcnRlZChzdGF0dXNlcy5pdGVtcygpKSksCiAgICApCgoKZGVmIHdyaXRlX2xvYWRfc21va2VfcmVwb3J0KHJlcG9ydDogTG9hZFNtb2tlUmVwb3J0LCBvdXRwdXRfcGF0aDogUGF0aCkgLT4gTm9uZToKICAgIF9hdG9taWNfanNvbihvdXRwdXRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQo=', 'olympus/foundry/ollama.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgaHR0cHgKCmZyb20gb2x5bXB1cy5mb3VuZHJ5LnNjaGVtYXMgaW1wb3J0IEdlbmVyYXRpb25SZXN1bHQKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE9sbGFtYUNsaWVudDoKICAgIGJhc2VfdXJsOiBzdHIgPSAiaHR0cDovLzEyNy4wLjAuMToxMTQzNCIKICAgIHRpbWVvdXRfc2Vjb25kczogZmxvYXQgPSAxMjAuMAogICAgdHJhbnNwb3J0OiBodHRweC5CYXNlVHJhbnNwb3J0IHwgTm9uZSA9IGZpZWxkKGRlZmF1bHQ9Tm9uZSwgcmVwcj1GYWxzZSkKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYmFzZV91cmwgPSBzZWxmLmJhc2VfdXJsLnJzdHJpcCgiLyIpCiAgICAgICAgaWYgbm90IHNlbGYuYmFzZV91cmwuc3RhcnRzd2l0aCgoImh0dHA6Ly8iLCAiaHR0cHM6Ly8iKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9sbGFtYSBiYXNlX3VybCBtdXN0IGJlIGFuIEhUVFAgb3IgSFRUUFMgVVJMIikKICAgICAgICBpZiBzZWxmLnRpbWVvdXRfc2Vjb25kcyA8PSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0aW1lb3V0X3NlY29uZHMgbXVzdCBiZSBwb3NpdGl2ZSIpCgogICAgZGVmIF9yZXF1ZXN0KHNlbGYsIG1ldGhvZDogc3RyLCBwYXRoOiBzdHIsICoqa3dhcmdzOiBBbnkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHdpdGggaHR0cHguQ2xpZW50KAogICAgICAgICAgICBiYXNlX3VybD1zZWxmLmJhc2VfdXJsLAogICAgICAgICAgICB0aW1lb3V0PXNlbGYudGltZW91dF9zZWNvbmRzLAogICAgICAgICAgICB0cmFuc3BvcnQ9c2VsZi50cmFuc3BvcnQsCiAgICAgICAgKSBhcyBjbGllbnQ6CiAgICAgICAgICAgIHJlc3BvbnNlID0gY2xpZW50LnJlcXVlc3QobWV0aG9kLCBwYXRoLCAqKmt3YXJncykKICAgICAgICAgICAgcmVzcG9uc2UucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgICAgIHBheWxvYWQgPSByZXNwb25zZS5qc29uKCkKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwYXlsb2FkLCBkaWN0KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2xsYW1hIHJldHVybmVkIGEgbm9uLW9iamVjdCByZXNwb25zZSIpCiAgICAgICAgcmV0dXJuIHBheWxvYWQKCiAgICBkZWYgbGlzdF9tb2RlbHMoc2VsZikgLT4gbGlzdFtzdHJdOgogICAgICAgIHBheWxvYWQgPSBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2FwaS90YWdzIikKICAgICAgICBtb2RlbHMgPSBwYXlsb2FkLmdldCgibW9kZWxzIikKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShtb2RlbHMsIGxpc3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbGxhbWEgbW9kZWwgbGlzdCBpcyBtYWxmb3JtZWQiKQogICAgICAgIG5hbWVzOiBsaXN0W3N0cl0gPSBbXQogICAgICAgIGZvciBtb2RlbCBpbiBtb2RlbHM6CiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKG1vZGVsLCBkaWN0KSBvciBub3QgaXNpbnN0YW5jZShtb2RlbC5nZXQoIm5hbWUiKSwgc3RyKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9sbGFtYSBtb2RlbCBlbnRyeSBpcyBtYWxmb3JtZWQiKQogICAgICAgICAgICBuYW1lcy5hcHBlbmQobW9kZWxbIm5hbWUiXSkKICAgICAgICByZXR1cm4gbmFtZXMKCiAgICBkZWYgaGVhbHRoKHNlbGYpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIG1vZGVscyA9IHNlbGYubGlzdF9tb2RlbHMoKQogICAgICAgIHJldHVybiB7InN0YXR1cyI6ICJvayIsICJwcm92aWRlciI6ICJvbGxhbWEiLCAibW9kZWxzIjogbW9kZWxzfQoKICAgIGRlZiBnZW5lcmF0ZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAgbW9kZWw6IHN0ciwKICAgICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBzeXN0ZW06IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIG1heF90b2tlbnM6IGludCA9IDI1NiwKICAgICAgICBjb250ZXh0X3Rva2VuczogaW50ID0gMl8wNDgsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAogICAgKSAtPiBHZW5lcmF0aW9uUmVzdWx0OgogICAgICAgIGlmIG5vdCBtb2RlbC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbCBpcyByZXF1aXJlZCIpCiAgICAgICAgaWYgbm90IHByb21wdC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9tcHQgaXMgcmVxdWlyZWQiKQogICAgICAgIGlmIG5vdCAxIDw9IG1heF90b2tlbnMgPD0gNF8wOTY6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF90b2tlbnMgbXVzdCBiZSBiZXR3ZWVuIDEgYW5kIDQwOTYiKQogICAgICAgIGlmIG5vdCAyNTYgPD0gY29udGV4dF90b2tlbnMgPD0gMTMxXzA3MjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY29udGV4dF90b2tlbnMgbXVzdCBiZSBiZXR3ZWVuIDI1NiBhbmQgMTMxMDcyIikKICAgICAgICBpZiBub3QgMCA8PSB0ZW1wZXJhdHVyZSA8PSAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0ZW1wZXJhdHVyZSBtdXN0IGJlIGJldHdlZW4gMCBhbmQgMiIpCiAgICAgICAgbWVzc2FnZXMgPSBbXQogICAgICAgIGlmIHN5c3RlbToKICAgICAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IHN5c3RlbX0pCiAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwcm9tcHR9KQogICAgICAgIHBheWxvYWQgPSBzZWxmLl9yZXF1ZXN0KAogICAgICAgICAgICAiUE9TVCIsCiAgICAgICAgICAgICIvYXBpL2NoYXQiLAogICAgICAgICAgICBqc29uPXsKICAgICAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAgICAgIm1lc3NhZ2VzIjogbWVzc2FnZXMsCiAgICAgICAgICAgICAgICAic3RyZWFtIjogRmFsc2UsCiAgICAgICAgICAgICAgICAidGhpbmsiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJvcHRpb25zIjogewogICAgICAgICAgICAgICAgICAgICJudW1fcHJlZGljdCI6IG1heF90b2tlbnMsCiAgICAgICAgICAgICAgICAgICAgIm51bV9jdHgiOiBjb250ZXh0X3Rva2VucywKICAgICAgICAgICAgICAgICAgICAidGVtcGVyYXR1cmUiOiB0ZW1wZXJhdHVyZSwKICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgIH0sCiAgICAgICAgKQogICAgICAgIG1lc3NhZ2UgPSBwYXlsb2FkLmdldCgibWVzc2FnZSIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobWVzc2FnZSwgZGljdCkgb3Igbm90IGlzaW5zdGFuY2UobWVzc2FnZS5nZXQoImNvbnRlbnQiKSwgc3RyKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2xsYW1hIGNoYXQgcmVzcG9uc2UgaXMgbWFsZm9ybWVkIikKICAgICAgICBjb250ZW50ID0gbWVzc2FnZVsiY29udGVudCJdCiAgICAgICAgcmV0dXJuIEdlbmVyYXRpb25SZXN1bHQoCiAgICAgICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgICAgICBjb250ZW50PWNvbnRlbnQsCiAgICAgICAgICAgIGZpbmlzaF9yZWFzb249InN0b3AiIGlmIHBheWxvYWQuZ2V0KCJkb25lIikgaXMgVHJ1ZSBlbHNlICJsZW5ndGgiLAogICAgICAgICAgICBwcm9tcHRfY2hhcmFjdGVycz1sZW4ocHJvbXB0KSwKICAgICAgICAgICAgZ2VuZXJhdGVkX2NoYXJhY3RlcnM9bGVuKGNvbnRlbnQpLAogICAgICAgICAgICBldmlkZW5jZT17CiAgICAgICAgICAgICAgICAicHJvdmlkZXIiOiAib2xsYW1hIiwKICAgICAgICAgICAgICAgICJkb25lX3JlYXNvbiI6IHBheWxvYWQuZ2V0KCJkb25lX3JlYXNvbiIpLAogICAgICAgICAgICAgICAgInRvdGFsX2R1cmF0aW9uX25zIjogcGF5bG9hZC5nZXQoInRvdGFsX2R1cmF0aW9uIiksCiAgICAgICAgICAgICAgICAiZXZhbF9jb3VudCI6IHBheWxvYWQuZ2V0KCJldmFsX2NvdW50IiksCiAgICAgICAgICAgICAgICAibG9hZF9kdXJhdGlvbl9ucyI6IHBheWxvYWQuZ2V0KCJsb2FkX2R1cmF0aW9uIiksCiAgICAgICAgICAgIH0sCiAgICAgICAgKQo=', 'olympus/foundry/promotion.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmZvdW5kcnkuZGF0YV9waXBlbGluZSBpbXBvcnQgRGF0YXNldE1hbmlmZXN0VjIsIHZlcmlmeV9kYXRhc2V0X21hbmlmZXN0CmZyb20gb2x5bXB1cy5mb3VuZHJ5LmV2YWxfc3VpdGUgaW1wb3J0IEhlbGRPdXRFdmFsdWF0aW9uCmZyb20gb2x5bXB1cy5mb3VuZHJ5LnF1YW50aXphdGlvbiBpbXBvcnQgUXVhbnRpemF0aW9uUmVwb3J0CgoKZGVmIF9zaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKCmRlZiBfYXRvbWljX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogb2JqZWN0KSAtPiBOb25lOgogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc2VyaWFsaXplZCA9IChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKS5lbmNvZGUoKQogICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxlLndyaXRlKHNlcmlhbGl6ZWQpCiAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5ID0gUGF0aChoYW5kbGUubmFtZSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKY2xhc3MgR2F0ZVJlc3VsdChTdHJpY3RNb2RlbCk6CiAgICBnYXRlOiBzdHIKICAgIHBhc3NlZDogYm9vbAogICAgb2JzZXJ2ZWQ6IHN0cgogICAgcmVxdWlyZW1lbnQ6IHN0cgoKCmNsYXNzIFByb21vdGlvblJlcG9ydChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gMQogICAgcmVxdWVzdGVkX21vZGVsX2lkOiBzdHIKICAgIGNoZWNrcG9pbnRfc2hhMjU2OiBzdHIKICAgIGdhdGVzOiBsaXN0W0dhdGVSZXN1bHRdCiAgICBwYXNzZWQ6IGJvb2wKICAgIHN0YXR1czogc3RyCiAgICByZWxlYXNlX21hbmlmZXN0X3BhdGg6IHN0ciB8IE5vbmUKICAgIGJsb2NrZXJzOiBsaXN0W3N0cl0KCgpkZWYgX2dhdGUobmFtZTogc3RyLCBwYXNzZWQ6IGJvb2wsIG9ic2VydmVkOiBvYmplY3QsIHJlcXVpcmVtZW50OiBzdHIpIC0+IEdhdGVSZXN1bHQ6CiAgICByZXR1cm4gR2F0ZVJlc3VsdCgKICAgICAgICBnYXRlPW5hbWUsCiAgICAgICAgcGFzc2VkPXBhc3NlZCwKICAgICAgICBvYnNlcnZlZD1zdHIob2JzZXJ2ZWQpLAogICAgICAgIHJlcXVpcmVtZW50PXJlcXVpcmVtZW50LAogICAgKQoKCmRlZiBfd29ya2Zsb3dfZmxvb3IoZXZhbHVhdGlvbjogSGVsZE91dEV2YWx1YXRpb24pIC0+IGZsb2F0OgogICAgcmV0dXJuIG1pbihzY29yZS5leGFjdF9tYXRjaF9yYXRlIGZvciBzY29yZSBpbiBldmFsdWF0aW9uLndvcmtmbG93X3Njb3JlcykKCgpkZWYgZXZhbHVhdGVfcHJvbW90aW9uKAogICAgKiwKICAgIHJlcXVlc3RlZF9tb2RlbF9pZDogc3RyLAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgZGF0YXNldF9tYW5pZmVzdF9wYXRoOiBQYXRoLAogICAgZXZhbHVhdGlvbl9wYXRoOiBQYXRoLAogICAgcXVhbnRpemF0aW9uX3JlcG9ydF9wYXRoOiBQYXRoLAogICAgbW9kZWxfY2FyZF9wYXRoOiBQYXRoLAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBhcHByb3ZlZF9iYXNlX2xpY2Vuc2U6IHN0ciwKICAgIHNlcnZpbmdfdmVyaWZpY2F0aW9uOiBkaWN0W3N0ciwgQW55XSB8IE5vbmUgPSBOb25lLAopIC0+IFByb21vdGlvblJlcG9ydDoKICAgIGNoZWNrcG9pbnRfc2hhID0gX3NoYTI1NihjaGVja3BvaW50X3BhdGgpCiAgICBtYW5pZmVzdDogRGF0YXNldE1hbmlmZXN0VjIgPSB2ZXJpZnlfZGF0YXNldF9tYW5pZmVzdChkYXRhc2V0X21hbmlmZXN0X3BhdGgpCiAgICBldmFsdWF0aW9uID0gSGVsZE91dEV2YWx1YXRpb24ubW9kZWxfdmFsaWRhdGVfanNvbihldmFsdWF0aW9uX3BhdGgucmVhZF9ieXRlcygpKQogICAgcXVhbnRpemF0aW9uID0gUXVhbnRpemF0aW9uUmVwb3J0Lm1vZGVsX3ZhbGlkYXRlX2pzb24ocXVhbnRpemF0aW9uX3JlcG9ydF9wYXRoLnJlYWRfYnl0ZXMoKSkKICAgIHRyYWluID0gbmV4dChzcGxpdCBmb3Igc3BsaXQgaW4gbWFuaWZlc3Quc3BsaXRzIGlmIHNwbGl0Lm5hbWUgPT0gInRyYWluIikKICAgIHRlc3QgPSBuZXh0KHNwbGl0IGZvciBzcGxpdCBpbiBtYW5pZmVzdC5zcGxpdHMgaWYgc3BsaXQubmFtZSA9PSAidGVzdCIpCiAgICBtaW5pbXVtX2NhdGVnb3J5X3JlY29yZHMgPSBtaW4odGVzdC5jYXRlZ29yaWVzLnZhbHVlcygpKQogICAgbW9kZWxfY2FyZCA9IG1vZGVsX2NhcmRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikgaWYgbW9kZWxfY2FyZF9wYXRoLmlzX2ZpbGUoKSBlbHNlICIiCiAgICBzZXJ2aW5nID0gc2VydmluZ192ZXJpZmljYXRpb24gb3Ige30KICAgIGFsbG93ZWRfbGljZW5zZSA9IGFwcHJvdmVkX2Jhc2VfbGljZW5zZSBpbiB7IkFwYWNoZS0yLjAiLCAiTUlUIiwgIkNDLUJZLTQuMCJ9CiAgICBnYXRlcyA9IFsKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgImNoZWNrcG9pbnRfaWRlbnRpdHkiLAogICAgICAgICAgICBldmFsdWF0aW9uLmNoZWNrcG9pbnRfc2hhMjU2ID09IGNoZWNrcG9pbnRfc2hhCiAgICAgICAgICAgIGFuZCBxdWFudGl6YXRpb24uc291cmNlX2NoZWNrcG9pbnRfc2hhMjU2ID09IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICBjaGVja3BvaW50X3NoYSwKICAgICAgICAgICAgIkV2YWx1YXRpb24gYW5kIHF1YW50aXphdGlvbiBldmlkZW5jZSBtdXN0IHJlZmVyZW5jZSB0aGUgZXhhY3QgY2hlY2twb2ludCBoYXNoLiIsCiAgICAgICAgKSwKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgImRhdGFzZXRfaWRlbnRpdHkiLAogICAgICAgICAgICBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYgPT0gZXZhbHVhdGlvbi5kYXRhc2V0X21hbmlmZXN0X3NoYTI1NiwKICAgICAgICAgICAgbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2LAogICAgICAgICAgICAiRXZhbHVhdGlvbiBtdXN0IHJlZmVyZW5jZSB0aGUgZXhhY3QgaW1tdXRhYmxlIGRhdGFzZXQgbWFuaWZlc3QuIiwKICAgICAgICApLAogICAgICAgIF9nYXRlKAogICAgICAgICAgICAiYmFzZV9saWNlbnNlIiwKICAgICAgICAgICAgYWxsb3dlZF9saWNlbnNlLAogICAgICAgICAgICBhcHByb3ZlZF9iYXNlX2xpY2Vuc2UsCiAgICAgICAgICAgICJCYXNlLW1vZGVsIHJpZ2h0cyBtdXN0IGJlIGV4cGxpY2l0bHkgYXBwcm92ZWQgZm9yIHRyYWluaW5nIGFuZCBkaXN0cmlidXRpb24uIiwKICAgICAgICApLAogICAgICAgIF9nYXRlKAogICAgICAgICAgICAidHJhaW5pbmdfc2NhbGUiLAogICAgICAgICAgICB0cmFpbi5yZWNvcmRzID49IDEwXzAwMCwKICAgICAgICAgICAgdHJhaW4ucmVjb3JkcywKICAgICAgICAgICAgIkF0IGxlYXN0IDEwLDAwMCByZXZpZXdlZCBTRlQgdHJhaW5pbmcgcmVjb3JkcyBhcmUgcmVxdWlyZWQgZm9yIEhlcm1lcyBBbHBoYS4iLAogICAgICAgICksCiAgICAgICAgX2dhdGUoCiAgICAgICAgICAgICJoZWxkX291dF9zY2FsZSIsCiAgICAgICAgICAgIHRlc3QucmVjb3JkcyA+PSAxXzIwMCBhbmQgbWluaW11bV9jYXRlZ29yeV9yZWNvcmRzID49IDEwMCwKICAgICAgICAgICAgZiJ0b3RhbD17dGVzdC5yZWNvcmRzfSwgbWluaW11bV9wZXJfY2F0ZWdvcnk9e21pbmltdW1fY2F0ZWdvcnlfcmVjb3Jkc30iLAogICAgICAgICAgICAiQXQgbGVhc3QgMSwyMDAgaGVsZC1vdXQgcmVjb3JkcyBhbmQgMTAwIHBlciBjYXRlZ29yeSBhcmUgcmVxdWlyZWQuIiwKICAgICAgICApLAogICAgICAgIF9nYXRlKAogICAgICAgICAgICAibG9zc19hbmRfcmVncmVzc2lvbiIsCiAgICAgICAgICAgIGV2YWx1YXRpb24ub3ZlcmFsbF9jYW5kaWRhdGVfbG9zcyA8IGV2YWx1YXRpb24ub3ZlcmFsbF9iYXNlbGluZV9sb3NzCiAgICAgICAgICAgIGFuZCBldmFsdWF0aW9uLnJlZ3Jlc3Npb25fY291bnQgPT0gMCwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgZiJiYXNlbGluZT17ZXZhbHVhdGlvbi5vdmVyYWxsX2Jhc2VsaW5lX2xvc3M6LjZmfSwgIgogICAgICAgICAgICAgICAgZiJjYW5kaWRhdGU9e2V2YWx1YXRpb24ub3ZlcmFsbF9jYW5kaWRhdGVfbG9zczouNmZ9LCAiCiAgICAgICAgICAgICAgICBmInJlZ3Jlc3Npb25zPXtldmFsdWF0aW9uLnJlZ3Jlc3Npb25fY291bnR9IgogICAgICAgICAgICApLAogICAgICAgICAgICAiQ2FuZGlkYXRlIGxvc3MgbXVzdCBpbXByb3ZlIGFuZCBubyBjYXBhYmlsaXR5IGNhdGVnb3J5IG1heSByZWdyZXNzIG92ZXIgNSUuIiwKICAgICAgICApLAogICAgICAgIF9nYXRlKAogICAgICAgICAgICAidGFza19xdWFsaXR5IiwKICAgICAgICAgICAgZXZhbHVhdGlvbi5leGFjdF9tYXRjaF9yYXRlID49IDAuODAKICAgICAgICAgICAgYW5kIGV2YWx1YXRpb24uZm9ybWF0X2NvbXBsaWFuY2VfcmF0ZSA+PSAwLjk1CiAgICAgICAgICAgIGFuZCBfd29ya2Zsb3dfZmxvb3IoZXZhbHVhdGlvbikgPj0gMC43NSwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgZiJleGFjdD17ZXZhbHVhdGlvbi5leGFjdF9tYXRjaF9yYXRlOi4zZn0sICIKICAgICAgICAgICAgICAgIGYiZm9ybWF0PXtldmFsdWF0aW9uLmZvcm1hdF9jb21wbGlhbmNlX3JhdGU6LjNmfSwgIgogICAgICAgICAgICAgICAgZiJ3b3JrZmxvd19mbG9vcj17X3dvcmtmbG93X2Zsb29yKGV2YWx1YXRpb24pOi4zZn0iCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJFeGFjdCBtYXRjaCA+PTgwJSwgZm9ybWF0IGNvbXBsaWFuY2UgPj05NSUsIHdvcmtmbG93IGZsb29yID49NzUlLiIsCiAgICAgICAgKSwKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgInF1YW50aXphdGlvbl9xdWFsaXR5IiwKICAgICAgICAgICAgcXVhbnRpemF0aW9uLnBhc3NlZF9xdWFsaXR5X2dhdGUKICAgICAgICAgICAgYW5kIHF1YW50aXphdGlvbi50b29sX2V4YWN0X21hdGNoX3JhdGUgPj0gMC43NSwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgZiJsb3NzX2NoYW5nZT17cXVhbnRpemF0aW9uLmxvc3NfY2hhbmdlX2ZyYWN0aW9uOi40Zn0sICIKICAgICAgICAgICAgICAgIGYidG9vbF9leGFjdD17cXVhbnRpemF0aW9uLnRvb2xfZXhhY3RfbWF0Y2hfcmF0ZTouM2Z9IgogICAgICAgICAgICApLAogICAgICAgICAgICAiUXVhbnRpemVkIGxvc3MgY2hhbmdlIDw9MiUsIGVuZm9yY2VkIGNvbnRleHQsIGFuZCB0b29sIGV4YWN0IG1hdGNoID49NzUlLiIsCiAgICAgICAgKSwKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgInNlcnZpbmdfcmVwcm9kdWNpYmlsaXR5IiwKICAgICAgICAgICAgc2VydmluZy5nZXQoInBhc3NlZCIpIGlzIFRydWUKICAgICAgICAgICAgYW5kIHNlcnZpbmcuZ2V0KCJjaGVja3BvaW50X3NoYTI1NiIpID09IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICBzZXJ2aW5nIG9yICJtaXNzaW5nIiwKICAgICAgICAgICAgIkZyZXNoLXByb2Nlc3MgQVBJLCBDTEksIGFuZCB3ZWIgc2VydmluZyBtdXN0IHBhc3MgYWdhaW5zdCB0aGUgZXhhY3QgY2hlY2twb2ludC4iLAogICAgICAgICksCiAgICAgICAgX2dhdGUoCiAgICAgICAgICAgICJtb2RlbF9jYXJkIiwKICAgICAgICAgICAgYm9vbChtb2RlbF9jYXJkKQogICAgICAgICAgICBhbmQgY2hlY2twb2ludF9zaGEgaW4gbW9kZWxfY2FyZAogICAgICAgICAgICBhbmQgIkxpbWl0YXRpb25zIiBpbiBtb2RlbF9jYXJkCiAgICAgICAgICAgIGFuZCAiTGljZW5zZSIgaW4gbW9kZWxfY2FyZCwKICAgICAgICAgICAgc3RyKG1vZGVsX2NhcmRfcGF0aCksCiAgICAgICAgICAgICJQdWJsaXNoZWQgY2FyZCBtdXN0IG5hbWUgdGhlIGV4YWN0IGhhc2gsIGxpY2Vuc2UsIGludGVuZGVkIHVzZSwgYW5kIGxpbWl0YXRpb25zLiIsCiAgICAgICAgKSwKICAgIF0KICAgIHBhc3NlZCA9IGFsbChnYXRlLnBhc3NlZCBmb3IgZ2F0ZSBpbiBnYXRlcykKICAgIHJlbGVhc2VfbWFuaWZlc3RfcGF0aDogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGlmIHBhc3NlZDoKICAgICAgICByZWxlYXNlX21hbmlmZXN0ID0gb3V0cHV0X3BhdGgud2l0aF9uYW1lKCJyZWxlYXNlLW1hbmlmZXN0Lmpzb24iKQogICAgICAgIF9hdG9taWNfanNvbigKICAgICAgICAgICAgcmVsZWFzZV9tYW5pZmVzdCwKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAgICAgICAgICAgICAgICJtb2RlbF9pZCI6IHJlcXVlc3RlZF9tb2RlbF9pZCwKICAgICAgICAgICAgICAgICJjaGVja3BvaW50X3NoYTI1NiI6IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICAgICAgImRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2IjogbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2LAogICAgICAgICAgICAgICAgImV2YWx1YXRpb25fc2hhMjU2IjogX3NoYTI1NihldmFsdWF0aW9uX3BhdGgpLAogICAgICAgICAgICAgICAgInF1YW50aXphdGlvbl9yZXBvcnRfc2hhMjU2IjogX3NoYTI1NihxdWFudGl6YXRpb25fcmVwb3J0X3BhdGgpLAogICAgICAgICAgICAgICAgIm1vZGVsX2NhcmRfc2hhMjU2IjogX3NoYTI1Nihtb2RlbF9jYXJkX3BhdGgpLAogICAgICAgICAgICAgICAgImJhc2VfbGljZW5zZSI6IGFwcHJvdmVkX2Jhc2VfbGljZW5zZSwKICAgICAgICAgICAgfSwKICAgICAgICApCiAgICAgICAgcmVsZWFzZV9tYW5pZmVzdF9wYXRoID0gc3RyKHJlbGVhc2VfbWFuaWZlc3QucmVzb2x2ZSgpKQogICAgYmxvY2tlcnMgPSBbZ2F0ZS5nYXRlIGZvciBnYXRlIGluIGdhdGVzIGlmIG5vdCBnYXRlLnBhc3NlZF0KICAgIHJlcG9ydCA9IFByb21vdGlvblJlcG9ydCgKICAgICAgICByZXF1ZXN0ZWRfbW9kZWxfaWQ9cmVxdWVzdGVkX21vZGVsX2lkLAogICAgICAgIGNoZWNrcG9pbnRfc2hhMjU2PWNoZWNrcG9pbnRfc2hhLAogICAgICAgIGdhdGVzPWdhdGVzLAogICAgICAgIHBhc3NlZD1wYXNzZWQsCiAgICAgICAgc3RhdHVzPSJQUk9NT1RFRCIgaWYgcGFzc2VkIGVsc2UgIk5PVF9QUk9NT1RFRCIsCiAgICAgICAgcmVsZWFzZV9tYW5pZmVzdF9wYXRoPXJlbGVhc2VfbWFuaWZlc3RfcGF0aCwKICAgICAgICBibG9ja2Vycz1ibG9ja2VycywKICAgICkKICAgIF9hdG9taWNfanNvbihvdXRwdXRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo=', 'olympus/foundry/quantization.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgTGl0ZXJhbCwgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbApmcm9tIG9seW1wdXMuZm91bmRyeS5kYXRhX3BpcGVsaW5lIGltcG9ydCBsb2FkX3ZlcmlmaWVkX2RhdGFzZXQKZnJvbSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2VzIGltcG9ydCBtZW1vcnlfc25hcHNob3QKZnJvbSBvbHltcHVzLmZvdW5kcnkuc2Z0IGltcG9ydCAoCiAgICBMb3NzTm9ybWFsaXphdGlvbiwKICAgIFRpbnlDYXVzYWxMTSwKICAgIFRpbnlNb2RlbENvbmZpZywKICAgIF9jaGVja3BvaW50X3RyYWluaW5nX2NvbmZpZywKICAgIF9lbmNvZGVfZXhhbXBsZXMsCiAgICBfbW9kZWxfZnJvbV9jaGVja3BvaW50LAogICAgX3BhY2tfbmliYmxlcywKICAgIF91bnBhY2tfbmliYmxlcywKICAgIGV2YWx1YXRlX2xvc3MsCiAgICBnZW5lcmF0ZV90ZXh0LAopCgpRdWFudGl6YXRpb25CaXRzID0gTGl0ZXJhbFs0LCA4XQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2F0b21pY190b3JjaF9zYXZlKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAgICB0cnk6CiAgICAgICAgdG9yY2guc2F2ZShwYXlsb2FkLCB0ZW1wb3JhcnkpCiAgICAgICAgd2l0aCB0ZW1wb3Jhcnkub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQogICAgZmluYWxseToKICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHNlcmlhbGl6ZWQgPSAoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIikuZW5jb2RlKCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIGhhbmRsZS53cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmNsYXNzIFF1YW50aXplZFRlbnNvcihTdHJpY3RNb2RlbCk6CiAgICBuYW1lOiBzdHIKICAgIHNoYXBlOiBsaXN0W2ludF0KICAgIGZsb2F0aW5nOiBib29sCiAgICBheGlzOiBpbnQgfCBOb25lCgoKY2xhc3MgUXVhbnRpemF0aW9uUmVwb3J0KFN0cmljdE1vZGVsKToKICAgIHNjaGVtYV92ZXJzaW9uOiBpbnQgPSAxCiAgICBmb3JtYXQ6IHN0cgogICAgYml0czogUXVhbnRpemF0aW9uQml0cwogICAgc291cmNlX2NoZWNrcG9pbnRfc2hhMjU2OiBzdHIKICAgIGFydGlmYWN0X3BhdGg6IHN0cgogICAgYXJ0aWZhY3Rfc2hhMjU2OiBzdHIKICAgIGZsb2F0X3dlaWdodHNfYnl0ZXM6IGludCA9IEZpZWxkKGd0PTApCiAgICBxdWFudGl6ZWRfYnl0ZXM6IGludCA9IEZpZWxkKGd0PTApCiAgICBzaXplX3JlZHVjdGlvbl9mcmFjdGlvbjogZmxvYXQKICAgIHNvdXJjZV9sb3NzOiBmbG9hdAogICAgcXVhbnRpemVkX2xvc3M6IGZsb2F0CiAgICBsb3NzX2NoYW5nZV9mcmFjdGlvbjogZmxvYXQKICAgIHNvdXJjZV9sYXRlbmN5X21zOiBmbG9hdAogICAgcXVhbnRpemVkX2xhdGVuY3lfbXM6IGZsb2F0CiAgICBzdGFydHVwX2xhdGVuY3lfbXM6IGZsb2F0CiAgICBtYXhfY29udGV4dF90b2tlbnM6IGludAogICAgY29udGV4dF9saW1pdF9lbmZvcmNlZDogYm9vbAogICAgdG9vbF9leGFjdF9tYXRjaF9yYXRlOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgcGVha19yc3NfYnl0ZXM6IGludCA9IEZpZWxkKGdlPTApCiAgICBwYXNzZWRfcXVhbGl0eV9nYXRlOiBib29sCiAgICBsb3NzX25vcm1hbGl6YXRpb246IExvc3NOb3JtYWxpemF0aW9uID0gImxlZ2FjeV9iYXRjaF9tZWFuX3YxIgoKCmRlZiBfcXVhbnRpemVfdGVuc29yKHRlbnNvcjogdG9yY2guVGVuc29yLCBiaXRzOiBRdWFudGl6YXRpb25CaXRzKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHZhbHVlID0gdGVuc29yLmRldGFjaCgpLmNwdSgpCiAgICBpZiBub3QgdmFsdWUuaXNfZmxvYXRpbmdfcG9pbnQoKToKICAgICAgICByZXR1cm4geyJmbG9hdGluZyI6IEZhbHNlLCAic2hhcGUiOiBsaXN0KHZhbHVlLnNoYXBlKSwgInZhbHVlIjogdmFsdWV9CiAgICBmbG9hdGluZyA9IHZhbHVlLmZsb2F0KCkKICAgIGF4aXMgPSAwIGlmIGZsb2F0aW5nLm5kaW0gPj0gMiBlbHNlIE5vbmUKICAgIGxpbWl0ID0gMTI3IGlmIGJpdHMgPT0gOCBlbHNlIDcKICAgIGlmIGF4aXMgPT0gMDoKICAgICAgICBzY2FsZSA9IGZsb2F0aW5nLmFicygpLnJlc2hhcGUoZmxvYXRpbmcuc2hhcGVbMF0sIC0xKS5hbWF4KGRpbT0xKS5jbGFtcF9taW4oMWUtMTIpIC8gbGltaXQKICAgICAgICBzaGFwZWRfc2NhbGUgPSBzY2FsZS5yZXNoYXBlKGZsb2F0aW5nLnNoYXBlWzBdLCAqKFsxXSAqIChmbG9hdGluZy5uZGltIC0gMSkpKQogICAgZWxzZToKICAgICAgICBzY2FsZSA9IGZsb2F0aW5nLmFicygpLmFtYXgoKS5yZXNoYXBlKDEpLmNsYW1wX21pbigxZS0xMikgLyBsaW1pdAogICAgICAgIHNoYXBlZF9zY2FsZSA9IHNjYWxlCiAgICBzaWduZWQgPSB0b3JjaC5yb3VuZChmbG9hdGluZyAvIHNoYXBlZF9zY2FsZSkuY2xhbXAoLWxpbWl0LCBsaW1pdCkudG8odG9yY2guaW50OCkKICAgIGlmIGJpdHMgPT0gODoKICAgICAgICBxdWFudGl6ZWQgPSBzaWduZWQKICAgICAgICBjb3VudCA9IHNpZ25lZC5udW1lbCgpCiAgICBlbHNlOgogICAgICAgIHF1YW50aXplZCA9IF9wYWNrX25pYmJsZXMoc2lnbmVkICsgOCkKICAgICAgICBjb3VudCA9IHNpZ25lZC5udW1lbCgpCiAgICByZXR1cm4gewogICAgICAgICJmbG9hdGluZyI6IFRydWUsCiAgICAgICAgInNoYXBlIjogbGlzdCh2YWx1ZS5zaGFwZSksCiAgICAgICAgImF4aXMiOiBheGlzLAogICAgICAgICJzY2FsZSI6IHNjYWxlLAogICAgICAgICJxdWFudGl6ZWQiOiBxdWFudGl6ZWQsCiAgICAgICAgImNvdW50IjogY291bnQsCiAgICB9CgoKZGVmIF9kZXF1YW50aXplX3RlbnNvcihwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSwgYml0czogUXVhbnRpemF0aW9uQml0cykgLT4gdG9yY2guVGVuc29yOgogICAgaWYgbm90IHBheWxvYWRbImZsb2F0aW5nIl06CiAgICAgICAgcmV0dXJuIGNhc3QodG9yY2guVGVuc29yLCBwYXlsb2FkWyJ2YWx1ZSJdKQogICAgc2hhcGUgPSBjYXN0KGxpc3RbaW50XSwgcGF5bG9hZFsic2hhcGUiXSkKICAgIGlmIGJpdHMgPT0gODoKICAgICAgICBzaWduZWQgPSBjYXN0KHRvcmNoLlRlbnNvciwgcGF5bG9hZFsicXVhbnRpemVkIl0pLnRvKHRvcmNoLmludDgpCiAgICBlbHNlOgogICAgICAgIHBhY2tlZCA9IGNhc3QodG9yY2guVGVuc29yLCBwYXlsb2FkWyJxdWFudGl6ZWQiXSkKICAgICAgICBzaWduZWQgPSBfdW5wYWNrX25pYmJsZXMocGFja2VkLCBpbnQocGF5bG9hZFsiY291bnQiXSkpLnRvKHRvcmNoLmludDgpIC0gOAogICAgc2lnbmVkID0gc2lnbmVkLnJlc2hhcGUoc2hhcGUpLmZsb2F0KCkKICAgIHNjYWxlID0gY2FzdCh0b3JjaC5UZW5zb3IsIHBheWxvYWRbInNjYWxlIl0pLmZsb2F0KCkKICAgIGlmIHBheWxvYWRbImF4aXMiXSA9PSAwOgogICAgICAgIHNjYWxlID0gc2NhbGUucmVzaGFwZShzaGFwZVswXSwgKihbMV0gKiAobGVuKHNoYXBlKSAtIDEpKSkKICAgIHJldHVybiBzaWduZWQgKiBzY2FsZQoKCmRlZiBsb2FkX3F1YW50aXplZF9tb2RlbChwYXRoOiBQYXRoKSAtPiBUaW55Q2F1c2FsTE06CiAgICBwYXlsb2FkID0gY2FzdChkaWN0W3N0ciwgQW55XSwgdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1UcnVlKSkKICAgIGJpdHMgPSBjYXN0KFF1YW50aXphdGlvbkJpdHMsIGludChwYXlsb2FkWyJiaXRzIl0pKQogICAgbW9kZWwgPSBUaW55Q2F1c2FsTE0oVGlueU1vZGVsQ29uZmlnLm1vZGVsX3ZhbGlkYXRlKHBheWxvYWRbIm1vZGVsX2NvbmZpZyJdKSkKICAgIHN0YXRlID0gewogICAgICAgIG5hbWU6IF9kZXF1YW50aXplX3RlbnNvcih0ZW5zb3JfcGF5bG9hZCwgYml0cykKICAgICAgICBmb3IgbmFtZSwgdGVuc29yX3BheWxvYWQgaW4gY2FzdChkaWN0W3N0ciwgZGljdFtzdHIsIEFueV1dLCBwYXlsb2FkWyJ0ZW5zb3JzIl0pLml0ZW1zKCkKICAgIH0KICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsCgoKZGVmIF9sYXRlbmN5KG1vZGVsOiBUaW55Q2F1c2FsTE0sIHByb21wdDogc3RyKSAtPiBmbG9hdDoKICAgIHN0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZ2VuZXJhdGVfdGV4dChtb2RlbCwgcHJvbXB0LCBkZXZpY2U9dG9yY2guZGV2aWNlKCJjcHUiKSwgbWF4X25ld190b2tlbnM9MTYpCiAgICByZXR1cm4gKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydCkgKiAxXzAwMAoKCmRlZiBxdWFudGl6ZV9jaGVja3BvaW50KAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgbWFuaWZlc3RfcGF0aDogUGF0aCwKICAgIG91dHB1dF9yb290OiBQYXRoLAogICAgKiwKICAgIGJpdHM6IFF1YW50aXphdGlvbkJpdHMsCikgLT4gUXVhbnRpemF0aW9uUmVwb3J0OgogICAgc291cmNlLCBjaGVja3BvaW50ID0gX21vZGVsX2Zyb21fY2hlY2twb2ludChjaGVja3BvaW50X3BhdGgsIHRvcmNoLmRldmljZSgiY3B1IikpCiAgICB0cmFpbmluZyA9IF9jaGVja3BvaW50X3RyYWluaW5nX2NvbmZpZyhjaGVja3BvaW50KQogICAgaWYgdHJhaW5pbmcubW9kZSAhPSAiZnVsbCI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicG9zdC10cmFpbmluZyBxdWFudGl6YXRpb24gY3VycmVudGx5IHJlcXVpcmVzIGEgbWVyZ2VkIGZ1bGwgY2hlY2twb2ludCIpCiAgICBtYW5pZmVzdCwgZXhhbXBsZXNfYnlfc3BsaXQgPSBsb2FkX3ZlcmlmaWVkX2RhdGFzZXQobWFuaWZlc3RfcGF0aCkKICAgIGlmIGNoZWNrcG9pbnQuZ2V0KCJkYXRhc2V0X21hbmlmZXN0X3NoYTI1NiIpICE9IG1hbmlmZXN0Lm1hbmlmZXN0X3NoYTI1NjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGFuZCBxdWFudGl6YXRpb24gZGF0YXNldCBoYXNoZXMgZG8gbm90IG1hdGNoIikKICAgIGV4YW1wbGVzID0gZXhhbXBsZXNfYnlfc3BsaXRbInRlc3QiXQogICAgZGVsIGV4YW1wbGVzX2J5X3NwbGl0CiAgICBvdXRwdXRfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmbG9hdF9wYXRoID0gb3V0cHV0X3Jvb3QgLyAiZmxvYXQzMi1tb2RlbC5wdCIKICAgIF9hdG9taWNfdG9yY2hfc2F2ZShmbG9hdF9wYXRoLCBzb3VyY2Uuc3RhdGVfZGljdCgpKQogICAgYXJ0aWZhY3QgPSBvdXRwdXRfcm9vdCAvIGYibW9kZWwtaW50e2JpdHN9LnB0IgogICAgdGVuc29yX3BheWxvYWRzID0gewogICAgICAgIG5hbWU6IF9xdWFudGl6ZV90ZW5zb3IodGVuc29yLCBiaXRzKSBmb3IgbmFtZSwgdGVuc29yIGluIHNvdXJjZS5zdGF0ZV9kaWN0KCkuaXRlbXMoKQogICAgfQogICAgX2F0b21pY190b3JjaF9zYXZlKAogICAgICAgIGFydGlmYWN0LAogICAgICAgIHsKICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAgICAgICAgICAgImZvcm1hdCI6IGYib2x5bXB1cy1zeW1tZXRyaWMtaW50e2JpdHN9LXYxIiwKICAgICAgICAgICAgImJpdHMiOiBiaXRzLAogICAgICAgICAgICAibW9kZWxfY29uZmlnIjogdHJhaW5pbmcubW9kZWwubW9kZWxfZHVtcChtb2RlPSJqc29uIiksCiAgICAgICAgICAgICJzb3VyY2VfY2hlY2twb2ludF9zaGEyNTYiOiBfc2hhMjU2KGNoZWNrcG9pbnRfcGF0aCksCiAgICAgICAgICAgICJ0ZW5zb3JzIjogdGVuc29yX3BheWxvYWRzLAogICAgICAgIH0sCiAgICApCiAgICBzdGFydCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHF1YW50aXplZCA9IGxvYWRfcXVhbnRpemVkX21vZGVsKGFydGlmYWN0KQogICAgc3RhcnR1cF9tcyA9ICh0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnQpICogMV8wMDAKCiAgICBldmFsX2NvbmZpZyA9IHRyYWluaW5nLm1vZGVsX2NvcHkodXBkYXRlPXsicGFja19zZXF1ZW5jZXMiOiBGYWxzZX0pCiAgICByb3dzID0gX2VuY29kZV9leGFtcGxlcyhleGFtcGxlcywgZXZhbF9jb25maWcpCiAgICBzb3VyY2VfbG9zcyA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgc291cmNlLCByb3dzLCBkZXZpY2U9dG9yY2guZGV2aWNlKCJjcHUiKSwgbm9ybWFsaXphdGlvbj10cmFpbmluZy5sb3NzX25vcm1hbGl6YXRpb24KICAgICkKICAgIHF1YW50aXplZF9sb3NzID0gZXZhbHVhdGVfbG9zcygKICAgICAgICBxdWFudGl6ZWQsIHJvd3MsIGRldmljZT10b3JjaC5kZXZpY2UoImNwdSIpLCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAgKQogICAgcHJvbXB0ID0gIkxpc3QgdGhlIHNhZmUgc3RlcHMgYmVmb3JlIHN0YXJ0aW5nIGEgbWVtb3J5LWhlYXZ5IGxvY2FsIG1vZGVsIHdvcmtsb2FkLiIKICAgIHNvdXJjZV9sYXRlbmN5ID0gX2xhdGVuY3koc291cmNlLCBwcm9tcHQpCiAgICBxdWFudGl6ZWRfbGF0ZW5jeSA9IF9sYXRlbmN5KHF1YW50aXplZCwgcHJvbXB0KQogICAgdG9vbF9leGFtcGxlcyA9IFtpdGVtIGZvciBpdGVtIGluIGV4YW1wbGVzIGlmIGl0ZW0uY2F0ZWdvcnkgaW4geyJ0b29sX3VzZSIsICJhZ2VudF9iZWhhdmlvciJ9XQogICAgdG9vbF9leGFjdCA9IHN1bSgKICAgICAgICBnZW5lcmF0ZV90ZXh0KHF1YW50aXplZCwgaXRlbS5wcm9tcHQsIGRldmljZT10b3JjaC5kZXZpY2UoImNwdSIpLCBtYXhfbmV3X3Rva2Vucz00OCkKICAgICAgICAuc3RyaXAoKQogICAgICAgIC5jYXNlZm9sZCgpCiAgICAgICAgPT0gaXRlbS5yZXNwb25zZS5zdHJpcCgpLmNhc2Vmb2xkKCkKICAgICAgICBmb3IgaXRlbSBpbiB0b29sX2V4YW1wbGVzCiAgICApIC8gbGVuKHRvb2xfZXhhbXBsZXMpCiAgICBjb250ZXh0X2xpbWl0X2VuZm9yY2VkID0gRmFsc2UKICAgIHRyeToKICAgICAgICBxdWFudGl6ZWQodG9yY2guemVyb3MoKDEsIHRyYWluaW5nLm1vZGVsLm1heF9zZXF1ZW5jZV90b2tlbnMgKyAxKSwgZHR5cGU9dG9yY2gubG9uZykpCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICBjb250ZXh0X2xpbWl0X2VuZm9yY2VkID0gVHJ1ZQogICAgZmxvYXRfc2l6ZSA9IGZsb2F0X3BhdGguc3RhdCgpLnN0X3NpemUKICAgIHF1YW50aXplZF9zaXplID0gYXJ0aWZhY3Quc3RhdCgpLnN0X3NpemUKICAgIGxvc3NfY2hhbmdlID0gKHF1YW50aXplZF9sb3NzIC0gc291cmNlX2xvc3MpIC8gbWF4KHNvdXJjZV9sb3NzLCAxZS0xMikKICAgIHJlcG9ydCA9IFF1YW50aXphdGlvblJlcG9ydCgKICAgICAgICBsb3NzX25vcm1hbGl6YXRpb249dHJhaW5pbmcubG9zc19ub3JtYWxpemF0aW9uLAogICAgICAgIGZvcm1hdD1mIm9seW1wdXMtc3ltbWV0cmljLWludHtiaXRzfS12MSIsCiAgICAgICAgYml0cz1iaXRzLAogICAgICAgIHNvdXJjZV9jaGVja3BvaW50X3NoYTI1Nj1fc2hhMjU2KGNoZWNrcG9pbnRfcGF0aCksCiAgICAgICAgYXJ0aWZhY3RfcGF0aD1zdHIoYXJ0aWZhY3QucmVzb2x2ZSgpKSwKICAgICAgICBhcnRpZmFjdF9zaGEyNTY9X3NoYTI1NihhcnRpZmFjdCksCiAgICAgICAgZmxvYXRfd2VpZ2h0c19ieXRlcz1mbG9hdF9zaXplLAogICAgICAgIHF1YW50aXplZF9ieXRlcz1xdWFudGl6ZWRfc2l6ZSwKICAgICAgICBzaXplX3JlZHVjdGlvbl9mcmFjdGlvbj0xLjAgLSBxdWFudGl6ZWRfc2l6ZSAvIGZsb2F0X3NpemUsCiAgICAgICAgc291cmNlX2xvc3M9c291cmNlX2xvc3MsCiAgICAgICAgcXVhbnRpemVkX2xvc3M9cXVhbnRpemVkX2xvc3MsCiAgICAgICAgbG9zc19jaGFuZ2VfZnJhY3Rpb249bG9zc19jaGFuZ2UsCiAgICAgICAgc291cmNlX2xhdGVuY3lfbXM9c291cmNlX2xhdGVuY3ksCiAgICAgICAgcXVhbnRpemVkX2xhdGVuY3lfbXM9cXVhbnRpemVkX2xhdGVuY3ksCiAgICAgICAgc3RhcnR1cF9sYXRlbmN5X21zPXN0YXJ0dXBfbXMsCiAgICAgICAgbWF4X2NvbnRleHRfdG9rZW5zPXRyYWluaW5nLm1vZGVsLm1heF9zZXF1ZW5jZV90b2tlbnMsCiAgICAgICAgY29udGV4dF9saW1pdF9lbmZvcmNlZD1jb250ZXh0X2xpbWl0X2VuZm9yY2VkLAogICAgICAgIHRvb2xfZXhhY3RfbWF0Y2hfcmF0ZT10b29sX2V4YWN0LAogICAgICAgIHBlYWtfcnNzX2J5dGVzPW1lbW9yeV9zbmFwc2hvdCgpLnByb2Nlc3NfcGVha19yc3NfYnl0ZXMsCiAgICAgICAgcGFzc2VkX3F1YWxpdHlfZ2F0ZT0oYWJzKGxvc3NfY2hhbmdlKSA8PSAwLjAyIGFuZCBjb250ZXh0X2xpbWl0X2VuZm9yY2VkKSwKICAgICkKICAgIHJlcG9ydF9wYXRoID0gb3V0cHV0X3Jvb3QgLyBmImludHtiaXRzfS1yZXBvcnQuanNvbiIKICAgIF9hdG9taWNfanNvbihyZXBvcnRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo=', 'olympus/foundry/readiness.py': '...', 'olympus/foundry/resources.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGZjbnRsCmltcG9ydCBvcwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHJlCmltcG9ydCByZXNvdXJjZQppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBJTywgU2VsZgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUsIHNsb3RzPVRydWUpCmNsYXNzIE1lbW9yeVNuYXBzaG90OgogICAgdG90YWxfYnl0ZXM6IGludAogICAgYXZhaWxhYmxlX2J5dGVzOiBpbnQKICAgIHN3YXBfdG90YWxfYnl0ZXM6IGludAogICAgc3dhcF91c2VkX2J5dGVzOiBpbnQKICAgIHByb2Nlc3NfcGVha19yc3NfYnl0ZXM6IGludAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHN3YXBfZnJhY3Rpb24oc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIDAuMCBpZiBzZWxmLnN3YXBfdG90YWxfYnl0ZXMgPT0gMCBlbHNlIHNlbGYuc3dhcF91c2VkX2J5dGVzIC8gc2VsZi5zd2FwX3RvdGFsX2J5dGVzCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIGludCB8IGZsb2F0XToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAidG90YWxfYnl0ZXMiOiBzZWxmLnRvdGFsX2J5dGVzLAogICAgICAgICAgICAiYXZhaWxhYmxlX2J5dGVzIjogc2VsZi5hdmFpbGFibGVfYnl0ZXMsCiAgICAgICAgICAgICJzd2FwX3RvdGFsX2J5dGVzIjogc2VsZi5zd2FwX3RvdGFsX2J5dGVzLAogICAgICAgICAgICAic3dhcF91c2VkX2J5dGVzIjogc2VsZi5zd2FwX3VzZWRfYnl0ZXMsCiAgICAgICAgICAgICJzd2FwX2ZyYWN0aW9uIjogc2VsZi5zd2FwX2ZyYWN0aW9uLAogICAgICAgICAgICAicHJvY2Vzc19wZWFrX3Jzc19ieXRlcyI6IHNlbGYucHJvY2Vzc19wZWFrX3Jzc19ieXRlcywKICAgICAgICB9CgoKZGVmIF9ydW4oY29tbWFuZDogbGlzdFtzdHJdKSAtPiBzdHI6CiAgICB0cnk6CiAgICAgICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oY29tbWFuZCwgY2hlY2s9VHJ1ZSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTUpCiAgICBleGNlcHQgKE9TRXJyb3IsIHN1YnByb2Nlc3MuU3VicHJvY2Vzc0Vycm9yKSBhcyBlcnJvcjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJyZXNvdXJjZSBpbnNwZWN0aW9uIGNvbW1hbmQgZmFpbGVkOiB7Y29tbWFuZFswXX0iKSBmcm9tIGVycm9yCiAgICByZXR1cm4gcmVzdWx0LnN0ZG91dAoKCmRlZiBfcGVha19yc3NfYnl0ZXMoKSAtPiBpbnQ6CiAgICB2YWx1ZSA9IHJlc291cmNlLmdldHJ1c2FnZShyZXNvdXJjZS5SVVNBR0VfU0VMRikucnVfbWF4cnNzCiAgICByZXR1cm4gaW50KHZhbHVlIGlmIHBsYXRmb3JtLnN5c3RlbSgpID09ICJEYXJ3aW4iIGVsc2UgdmFsdWUgKiAxMDI0KQoKCmRlZiBfbWFjb3NfbWVtb3J5KCkgLT4gTWVtb3J5U25hcHNob3Q6CiAgICB0b3RhbCA9IGludChfcnVuKFsic3lzY3RsIiwgIi1uIiwgImh3Lm1lbXNpemUiXSkuc3RyaXAoKSkKICAgIHZtX291dHB1dCA9IF9ydW4oWyJ2bV9zdGF0Il0pCiAgICBwYWdlX3NpemVfbWF0Y2ggPSByZS5zZWFyY2gociJwYWdlIHNpemUgb2YgKFxkKykgYnl0ZXMiLCB2bV9vdXRwdXQpCiAgICBpZiBwYWdlX3NpemVfbWF0Y2ggaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNhbm5vdCBwYXJzZSBtYWNPUyBWTSBwYWdlIHNpemUiKQogICAgcGFnZV9zaXplID0gaW50KHBhZ2Vfc2l6ZV9tYXRjaC5ncm91cCgxKSkKICAgIHBhZ2VzOiBkaWN0W3N0ciwgaW50XSA9IHt9CiAgICBmb3IgbGluZSBpbiB2bV9vdXRwdXQuc3BsaXRsaW5lcygpOgogICAgICAgIG1hdGNoID0gcmUubWF0Y2gociIoW146XSspOlxzKyhcZCspXC4iLCBsaW5lKQogICAgICAgIGlmIG1hdGNoOgogICAgICAgICAgICBwYWdlc1ttYXRjaC5ncm91cCgxKV0gPSBpbnQobWF0Y2guZ3JvdXAoMikpCiAgICBhdmFpbGFibGVfcGFnZXMgPSBzdW0oCiAgICAgICAgcGFnZXMuZ2V0KG5hbWUsIDApCiAgICAgICAgZm9yIG5hbWUgaW4gKCJQYWdlcyBmcmVlIiwgIlBhZ2VzIGluYWN0aXZlIiwgIlBhZ2VzIHNwZWN1bGF0aXZlIiwgIlBhZ2VzIHB1cmdlYWJsZSIpCiAgICApCiAgICBzd2FwX291dHB1dCA9IF9ydW4oWyJzeXNjdGwiLCAidm0uc3dhcHVzYWdlIl0pCiAgICBzd2FwX21hdGNoID0gcmUuc2VhcmNoKHIidG90YWwgPSAoWzAtOS5dKylNXHMrdXNlZCA9IChbMC05Ll0rKU0iLCBzd2FwX291dHB1dCkKICAgIGlmIHN3YXBfbWF0Y2ggaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNhbm5vdCBwYXJzZSBtYWNPUyBzd2FwIHVzYWdlIikKICAgIHJldHVybiBNZW1vcnlTbmFwc2hvdCgKICAgICAgICB0b3RhbF9ieXRlcz10b3RhbCwKICAgICAgICBhdmFpbGFibGVfYnl0ZXM9YXZhaWxhYmxlX3BhZ2VzICogcGFnZV9zaXplLAogICAgICAgIHN3YXBfdG90YWxfYnl0ZXM9aW50KGZsb2F0KHN3YXBfbWF0Y2guZ3JvdXAoMSkpICogMTAyNCAqIDEwMjQpLAogICAgICAgIHN3YXBfdXNlZF9ieXRlcz1pbnQoZmxvYXQoc3dhcF9tYXRjaC5ncm91cCgyKSkgKiAxMDI0ICogMTAyNCksCiAgICAgICAgcHJvY2Vzc19wZWFrX3Jzc19ieXRlcz1fcGVha19yc3NfYnl0ZXMoKSwKICAgICkKCgpkZWYgX2xpbnV4X21lbW9yeSgpIC0+IE1lbW9yeVNuYXBzaG90OgogICAgdmFsdWVzOiBkaWN0W3N0ciwgaW50XSA9IHt9CiAgICBmb3IgbGluZSBpbiBQYXRoKCIvcHJvYy9tZW1pbmZvIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICBrZXksIHJhdyA9IGxpbmUuc3BsaXQoIjoiLCAxKQogICAgICAgIHZhbHVlc1trZXldID0gaW50KHJhdy5zdHJpcCgpLnNwbGl0KClbMF0pICogMTAyNAogICAgcmV0dXJuIE1lbW9yeVNuYXBzaG90KAogICAgICAgIHRvdGFsX2J5dGVzPXZhbHVlc1siTWVtVG90YWwiXSwKICAgICAgICBhdmFpbGFibGVfYnl0ZXM9dmFsdWVzWyJNZW1BdmFpbGFibGUiXSwKICAgICAgICBzd2FwX3RvdGFsX2J5dGVzPXZhbHVlc1siU3dhcFRvdGFsIl0sCiAgICAgICAgc3dhcF91c2VkX2J5dGVzPXZhbHVlc1siU3dhcFRvdGFsIl0gLSB2YWx1ZXNbIlN3YXBGcmVlIl0sCiAgICAgICAgcHJvY2Vzc19wZWFrX3Jzc19ieXRlcz1fcGVha19yc3NfYnl0ZXMoKSwKICAgICkKCgpkZWYgbWVtb3J5X3NuYXBzaG90KCkgLT4gTWVtb3J5U25hcHNob3Q6CiAgICBzeXN0ZW0gPSBwbGF0Zm9ybS5zeXN0ZW0oKQogICAgaWYgc3lzdGVtID09ICJEYXJ3aW4iOgogICAgICAgIHJldHVybiBfbWFjb3NfbWVtb3J5KCkKICAgIGlmIHN5c3RlbSA9PSAiTGludXgiOgogICAgICAgIHJldHVybiBfbGludXhfbWVtb3J5KCkKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmInJlc291cmNlIGdvdmVybm9yIGRvZXMgbm90IHN1cHBvcnQge3N5c3RlbX0iKQoKCmNsYXNzIFJlc291cmNlR292ZXJub3I6CiAgICAiIiJFeGNsdXNpdmUgbG9jYWwgd29ya2xvYWQgbG9jayBwbHVzIHByZWZsaWdodCBtZW1vcnkgYW5kIHN3YXAgZW5mb3JjZW1lbnQuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbG9ja19wYXRoOiBQYXRoLAogICAgICAgICosCiAgICAgICAgbWluX2F2YWlsYWJsZV9ieXRlczogaW50ID0gMiAqIDEwMjQqKjMsCiAgICAgICAgbWF4X3N3YXBfZnJhY3Rpb246IGZsb2F0ID0gMC45MCwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBtaW5fYXZhaWxhYmxlX2J5dGVzIDwgMjU2ICogMTAyNCoqMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWluX2F2YWlsYWJsZV9ieXRlcyBtdXN0IGJlIGF0IGxlYXN0IDI1NiBNaUIiKQogICAgICAgIGlmIG5vdCAwLjAgPCBtYXhfc3dhcF9mcmFjdGlvbiA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF9zd2FwX2ZyYWN0aW9uIG11c3QgYmUgaW4gKDAsIDFdIikKICAgICAgICBzZWxmLmxvY2tfcGF0aCA9IGxvY2tfcGF0aC5yZXNvbHZlKCkKICAgICAgICBzZWxmLm1pbl9hdmFpbGFibGVfYnl0ZXMgPSBtaW5fYXZhaWxhYmxlX2J5dGVzCiAgICAgICAgc2VsZi5tYXhfc3dhcF9mcmFjdGlvbiA9IG1heF9zd2FwX2ZyYWN0aW9uCiAgICAgICAgc2VsZi5faGFuZGxlOiBJT1tzdHJdIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLnByZWZsaWdodDogTWVtb3J5U25hcHNob3QgfCBOb25lID0gTm9uZQoKICAgIGRlZiBhY3F1aXJlKHNlbGYpIC0+IE1lbW9yeVNuYXBzaG90OgogICAgICAgIHNlbGYubG9ja19wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgaGFuZGxlID0gc2VsZi5sb2NrX3BhdGgub3BlbigiYSsiLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHRyeToKICAgICAgICAgICAgZmNudGwuZmxvY2soaGFuZGxlLmZpbGVubygpLCBmY250bC5MT0NLX0VYIHwgZmNudGwuTE9DS19OQikKICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICBoYW5kbGUuY2xvc2UoKQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICAgICAiYW5vdGhlciBtb2RlbC1sb2FkaW5nIG9yIHRyYWluaW5nIHdvcmtsb2FkIGhvbGRzIHRoZSBsb2NrIgogICAgICAgICAgICApIGZyb20gZXJyb3IKCiAgICAgICAgdHJ5OgogICAgICAgICAgICBzbmFwc2hvdCA9IG1lbW9yeV9zbmFwc2hvdCgpCiAgICAgICAgICAgIGlmIHNuYXBzaG90LmF2YWlsYWJsZV9ieXRlcyA8IHNlbGYubWluX2F2YWlsYWJsZV9ieXRlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICAgICBmImF2YWlsYWJsZSBtZW1vcnkge3NuYXBzaG90LmF2YWlsYWJsZV9ieXRlc30gaXMgYmVsb3cgdGhlICIKICAgICAgICAgICAgICAgICAgICBmIntzZWxmLm1pbl9hdmFpbGFibGVfYnl0ZXN9LWJ5dGUgc2FmZXR5IGZsb29yIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBzbmFwc2hvdC5zd2FwX2ZyYWN0aW9uID4gc2VsZi5tYXhfc3dhcF9mcmFjdGlvbjoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICAgICBmInN3YXAgdXRpbGl6YXRpb24ge3NuYXBzaG90LnN3YXBfZnJhY3Rpb246LjElfSBleGNlZWRzIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgZiJ7c2VsZi5tYXhfc3dhcF9mcmFjdGlvbjouMSV9IHNhZmV0eSBjZWlsaW5nIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBoYW5kbGUuc2VlaygwKQogICAgICAgICAgICBoYW5kbGUudHJ1bmNhdGUoKQogICAgICAgICAgICBoYW5kbGUud3JpdGUoZiJwaWQ9e29zLmdldHBpZCgpfVxuIikKICAgICAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZjbnRsLmZsb2NrKGhhbmRsZS5maWxlbm8oKSwgZmNudGwuTE9DS19VTikKICAgICAgICAgICAgZmluYWxseToKICAgICAgICAgICAgICAgIGhhbmRsZS5jbG9zZSgpCiAgICAgICAgICAgIHJhaXNlCgogICAgICAgIHNlbGYuX2hhbmRsZSA9IGhhbmRsZQogICAgICAgIHNlbGYucHJlZmxpZ2h0ID0gc25hcHNob3QKICAgICAgICByZXR1cm4gc25hcHNob3QKCiAgICBkZWYgcmVsZWFzZShzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX2hhbmRsZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgZmNudGwuZmxvY2soc2VsZi5faGFuZGxlLmZpbGVubygpLCBmY250bC5MT0NLX1VOKQogICAgICAgICAgICBzZWxmLl9oYW5kbGUuY2xvc2UoKQogICAgICAgICAgICBzZWxmLl9oYW5kbGUgPSBOb25lCgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBTZWxmOgogICAgICAgIHNlbGYuYWNxdWlyZSgpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKl86IG9iamVjdCkgLT4gTm9uZToKICAgICAgICBzZWxmLnJlbGVhc2UoKQo=', 'olympus/foundry/schemas.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIExpdGVyYWwKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkLCBtb2RlbF92YWxpZGF0b3IKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xhc3MgQXJ0aWZhY3RTdGF0dXMoU3RyRW51bSk6CiAgICBERVNJR05FRCA9ICJERVNJR05FRCIKICAgIFJVTk5JTkdfRVhQRVJJTUVOVCA9ICJSVU5OSU5HX0VYUEVSSU1FTlQiCiAgICBDSEVDS1BPSU5URUQgPSAiQ0hFQ0tQT0lOVEVEIgogICAgRVZBTFVBVEVEID0gIkVWQUxVQVRFRCIKICAgIFZFUklGSUVEID0gIlZFUklGSUVEIgogICAgTkVHQVRJVkVfUkVTVUxUID0gIk5FR0FUSVZFX1JFU1VMVCIKICAgIElOQ09OQ0xVU0lWRSA9ICJJTkNPTkNMVVNJVkUiCiAgICBGQUlMRUQgPSAiRkFJTEVEIgoKCmNsYXNzIERhdGFzZXRSZWNvcmQoU3RyaWN0TW9kZWwpOgogICAgZGF0YXNldF9pZDogc3RyID0gRmllbGQocGF0dGVybj1yIl5bYS16MC05XVthLXowLTkuXy1dezEsNjN9JCIpCiAgICB2ZXJzaW9uOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NjQpCiAgICBzaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBtYXRlcmlhbGl6ZWRfcGF0aDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgc291cmNlOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9Ml8wMDApCiAgICBvd25lcjogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTIwMCkKICAgIGxpY2Vuc2U6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwgbWF4X2xlbmd0aD0yMDApCiAgICBwcm92ZW5hbmNlOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NF8wMDApCiAgICBwcml2YWN5X2NsYXNzaWZpY2F0aW9uOiBMaXRlcmFsWyJwdWJsaWMiLCAiaW50ZXJuYWwiLCAicHJpdmF0ZSIsICJyZXN0cmljdGVkIl0KICAgIHN5bnRoZXRpYzogYm9vbAogICAgZ2VuZXJhdG9yOiBzdHIgfCBOb25lID0gRmllbGQoZGVmYXVsdD1Ob25lLCBtYXhfbGVuZ3RoPTUwMCkKICAgIGJ5dGVfY291bnQ6IGludCA9IEZpZWxkKGd0PTApCiAgICBjaGFyYWN0ZXJfY291bnQ6IGludCA9IEZpZWxkKGd0PTApCiAgICBjcmVhdGVkX2F0OiBzdHIKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiByZXF1aXJlX3N5bnRoZXRpY19nZW5lcmF0b3Ioc2VsZikgLT4gRGF0YXNldFJlY29yZDoKICAgICAgICBpZiBzZWxmLnN5bnRoZXRpYyBhbmQgbm90IHNlbGYuZ2VuZXJhdG9yOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzeW50aGV0aWMgZGF0YXNldHMgbXVzdCBpZGVudGlmeSB0aGVpciBnZW5lcmF0b3IiKQogICAgICAgIHJldHVybiBzZWxmCgoKY2xhc3MgRXhwZXJpbWVudFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBleHBlcmltZW50X2lkOiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXltBLVpdW0EtWjAtOV8tXXsyLDYzfSQiKQogICAgaHlwb3RoZXNpczogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTRfMDAwKQogICAgbW9kZWxfZmFtaWx5OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MTAwKQogICAgZGF0YXNldF9pZDogc3RyCiAgICBkYXRhc2V0X3ZlcnNpb246IHN0cgogICAgZGF0YXNldF9zaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBzZWVkOiBpbnQgPSBGaWVsZChnZT0wLCBsZT0yKiozMiAtIDEpCiAgICBjb25maWc6IGRpY3Rbc3RyLCBBbnldCiAgICBjb2RlX2NvbW1pdDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTY0KQogICAgaGFyZHdhcmU6IGRpY3Rbc3RyLCBBbnldCiAgICBzdGF0dXM6IEFydGlmYWN0U3RhdHVzCiAgICBmYWlsdXJlX3JlYXNvbjogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGNyZWF0ZWRfYXQ6IHN0cgogICAgdXBkYXRlZF9hdDogc3RyCgoKY2xhc3MgQ2hlY2twb2ludFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBjaGVja3BvaW50X2lkOiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXmNrcHRfWzAtOWEtZl17MTZ9JCIpCiAgICBleHBlcmltZW50X2lkOiBzdHIKICAgIHNoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0kIikKICAgIHBhdGg6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGZvcm1hdDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTEwMCkKICAgIGJ5dGVfY291bnQ6IGludCA9IEZpZWxkKGd0PTApCiAgICBtZXRyaWNzOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBjcmVhdGVkX2F0OiBzdHIKCgpjbGFzcyBFdmFsdWF0aW9uUmVjb3JkKFN0cmljdE1vZGVsKToKICAgIGV2YWx1YXRpb25faWQ6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeZXZhbF9bMC05YS1mXXsxNn0kIikKICAgIGNoZWNrcG9pbnRfaWQ6IHN0cgogICAgc3VpdGU6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwgbWF4X2xlbmd0aD0yMDApCiAgICBkYXRhc2V0X3NoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0kIikKICAgIGJhc2VsaW5lX21ldHJpY3M6IGRpY3Rbc3RyLCBmbG9hdF0KICAgIGNhbmRpZGF0ZV9tZXRyaWNzOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBwYXNzZWQ6IGJvb2wKICAgIGRlY2lzaW9uOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9Ml8wMDApCiAgICBjcmVhdGVkX2F0OiBzdHIKCgpjbGFzcyBNb2RlbFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBtb2RlbF9pZDogc3RyID0gRmllbGQocGF0dGVybj1yIl5bYS16MC05XVthLXowLTkuXy1dezEsMTI3fSQiKQogICAgZmFtaWx5OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MTAwKQogICAgdmVyc2lvbjogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTY0KQogICAgY2hlY2twb2ludF9pZDogc3RyCiAgICBzdGF0dXM6IEFydGlmYWN0U3RhdHVzCiAgICBydW50aW1lOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MTAwKQogICAgY2FwYWJpbGl0aWVzOiBsaXN0W3N0cl0KICAgIGludGVuZGVkX3VzZTogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTJfMDAwKQogICAgbGltaXRhdGlvbnM6IGxpc3Rbc3RyXQogICAgY3JlYXRlZF9hdDogc3RyCgoKY2xhc3MgRXZpZGVuY2VSZWNvcmQoU3RyaWN0TW9kZWwpOgogICAgc2VxdWVuY2U6IGludCA9IEZpZWxkKGd0PTApCiAgICB0aW1lc3RhbXA6IHN0cgogICAgZW50aXR5X3R5cGU6IHN0cgogICAgZW50aXR5X2lkOiBzdHIKICAgIGFjdGlvbjogc3RyCiAgICBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIEZvdW5kcnlQaXBlbGluZVJlc3VsdChTdHJpY3RNb2RlbCk6CiAgICBkYXRhc2V0OiBEYXRhc2V0UmVjb3JkCiAgICBleHBlcmltZW50OiBFeHBlcmltZW50UmVjb3JkCiAgICBjaGVja3BvaW50OiBDaGVja3BvaW50UmVjb3JkCiAgICBldmFsdWF0aW9uOiBFdmFsdWF0aW9uUmVjb3JkCiAgICBtb2RlbDogTW9kZWxSZWNvcmQKICAgIGV4cG9ydF9wYXRoOiBzdHIKCgpjbGFzcyBHZW5lcmF0aW9uUmVzdWx0KFN0cmljdE1vZGVsKToKICAgIG1vZGVsOiBzdHIKICAgIGNvbnRlbnQ6IHN0cgogICAgZmluaXNoX3JlYXNvbjogTGl0ZXJhbFsic3RvcCIsICJsZW5ndGgiXQogICAgcHJvbXB0X2NoYXJhY3RlcnM6IGludCA9IEZpZWxkKGdlPTApCiAgICBnZW5lcmF0ZWRfY2hhcmFjdGVyczogaW50ID0gRmllbGQoZ2U9MCkKICAgIGV2aWRlbmNlOiBkaWN0W3N0ciwgQW55XQo=', 'olympus/foundry/service.py': '...', 'olympus/foundry/sft.py': '...', 'olympus/foundry/store.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHNxbGl0ZTMKaW1wb3J0IHRocmVhZGluZwpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgSXRlcmF0b3IKZnJvbSBjb250ZXh0bGliIGltcG9ydCBjb250ZXh0bWFuYWdlcgpmcm9tIGRhdGV0aW1lIGltcG9ydCBVVEMsIGRhdGV0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIG9seW1wdXMuY29yZS5zcWxpdGVfam91cm5hbCBpbXBvcnQgYXBwbHlfZHVyYWJsZV9qb3VybmFsX21vZGUKZnJvbSBvbHltcHVzLmZvdW5kcnkuc2NoZW1hcyBpbXBvcnQgKAogICAgQXJ0aWZhY3RTdGF0dXMsCiAgICBDaGVja3BvaW50UmVjb3JkLAogICAgRGF0YXNldFJlY29yZCwKICAgIEV2YWx1YXRpb25SZWNvcmQsCiAgICBFdmlkZW5jZVJlY29yZCwKICAgIEV4cGVyaW1lbnRSZWNvcmQsCiAgICBNb2RlbFJlY29yZCwKKQoKCmNsYXNzIEZvdW5kcnlTdG9yZToKICAgICIiIlNRTGl0ZSByZWdpc3RyeSB3aXRoIGZvcmVpZ24ga2V5cyBhbmQgYXBwZW5kLW9ubHkgZXZpZGVuY2UuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHNlbGYucGF0aCA9IHBhdGgKICAgICAgICBzZWxmLl9jb25uZWN0aW9uID0gc3FsaXRlMy5jb25uZWN0KHBhdGgsIHRpbWVvdXQ9MzAuMCwgY2hlY2tfc2FtZV90aHJlYWQ9RmFsc2UpCiAgICAgICAgc2VsZi5fbG9jayA9IHRocmVhZGluZy5STG9jaygpCiAgICAgICAgc2VsZi5fY29ubmVjdGlvbi5yb3dfZmFjdG9yeSA9IHNxbGl0ZTMuUm93CiAgICAgICAgc2VsZi5fY29ubmVjdGlvbi5leGVjdXRlKCJQUkFHTUEgZm9yZWlnbl9rZXlzID0gT04iKQogICAgICAgIGFwcGx5X2R1cmFibGVfam91cm5hbF9tb2RlKHNlbGYuX2Nvbm5lY3Rpb24pCiAgICAgICAgc2VsZi5fY29ubmVjdGlvbi5leGVjdXRlKCJQUkFHTUEgc3luY2hyb25vdXMgPSBGVUxMIikKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYuX21pZ3JhdGUoKQogICAgICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgICAgICAjIENvbnN0cnVjdGlvbiBtdXN0IG5vdCBsZWFrIGEgbGl2ZSBkYXRhYmFzZSBoYW5kbGUgd2hlbiBzY2hlbWEKICAgICAgICAgICAgIyBjcmVhdGlvbiBmYWlscyAoZm9yIGV4YW1wbGUsIG9uIGEgcmVhZC1vbmx5IGFydGlmYWN0IHJvb3QpLgogICAgICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmNsb3NlKCkKICAgICAgICAgICAgcmFpc2UKCiAgICBkZWYgX21pZ3JhdGUoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGVzY3JpcHQoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBzY2hlbWFfbWlncmF0aW9ucyAoCiAgICAgICAgICAgICAgICB2ZXJzaW9uIElOVEVHRVIgUFJJTUFSWSBLRVksCiAgICAgICAgICAgICAgICBhcHBsaWVkX2F0IFRFWFQgTk9UIE5VTEwKICAgICAgICAgICAgKTsKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgZGF0YXNldHMgKAogICAgICAgICAgICAgICAgZGF0YXNldF9pZCBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgdmVyc2lvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgc2hhMjU2IFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICByZWNvcmRfanNvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgUFJJTUFSWSBLRVkgKGRhdGFzZXRfaWQsIHZlcnNpb24pLAogICAgICAgICAgICAgICAgVU5JUVVFIChzaGEyNTYpCiAgICAgICAgICAgICk7CiAgICAgICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIGV4cGVyaW1lbnRzICgKICAgICAgICAgICAgICAgIGV4cGVyaW1lbnRfaWQgVEVYVCBQUklNQVJZIEtFWSwKICAgICAgICAgICAgICAgIGRhdGFzZXRfaWQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGRhdGFzZXRfdmVyc2lvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgc3RhdHVzIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICByZWNvcmRfanNvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgRk9SRUlHTiBLRVkgKGRhdGFzZXRfaWQsIGRhdGFzZXRfdmVyc2lvbikKICAgICAgICAgICAgICAgICAgICBSRUZFUkVOQ0VTIGRhdGFzZXRzKGRhdGFzZXRfaWQsIHZlcnNpb24pCiAgICAgICAgICAgICk7CiAgICAgICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIGNoZWNrcG9pbnRzICgKICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfaWQgVEVYVCBQUklNQVJZIEtFWSwKICAgICAgICAgICAgICAgIGV4cGVyaW1lbnRfaWQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHNoYTI1NiBURVhUIE5PVCBOVUxMIFVOSVFVRSwKICAgICAgICAgICAgICAgIHJlY29yZF9qc29uIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICBGT1JFSUdOIEtFWSAoZXhwZXJpbWVudF9pZCkgUkVGRVJFTkNFUyBleHBlcmltZW50cyhleHBlcmltZW50X2lkKQogICAgICAgICAgICApOwogICAgICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBldmFsdWF0aW9ucyAoCiAgICAgICAgICAgICAgICBldmFsdWF0aW9uX2lkIFRFWFQgUFJJTUFSWSBLRVksCiAgICAgICAgICAgICAgICBjaGVja3BvaW50X2lkIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICByZWNvcmRfanNvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgRk9SRUlHTiBLRVkgKGNoZWNrcG9pbnRfaWQpIFJFRkVSRU5DRVMgY2hlY2twb2ludHMoY2hlY2twb2ludF9pZCkKICAgICAgICAgICAgKTsKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgbW9kZWxzICgKICAgICAgICAgICAgICAgIG1vZGVsX2lkIFRFWFQgUFJJTUFSWSBLRVksCiAgICAgICAgICAgICAgICBjaGVja3BvaW50X2lkIFRFWFQgTk9UIE5VTEwgVU5JUVVFLAogICAgICAgICAgICAgICAgc3RhdHVzIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICByZWNvcmRfanNvbiBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgRk9SRUlHTiBLRVkgKGNoZWNrcG9pbnRfaWQpIFJFRkVSRU5DRVMgY2hlY2twb2ludHMoY2hlY2twb2ludF9pZCkKICAgICAgICAgICAgKTsKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgZXZpZGVuY2UgKAogICAgICAgICAgICAgICAgc2VxdWVuY2UgSU5URUdFUiBQUklNQVJZIEtFWSBBVVRPSU5DUkVNRU5ULAogICAgICAgICAgICAgICAgdGltZXN0YW1wIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICBlbnRpdHlfdHlwZSBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgZW50aXR5X2lkIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICBhY3Rpb24gVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHBheWxvYWRfanNvbiBURVhUIE5PVCBOVUxMCiAgICAgICAgICAgICk7CiAgICAgICAgICAgICIiIgogICAgICAgICkKICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICJJTlNFUlQgT1IgSUdOT1JFIElOVE8gc2NoZW1hX21pZ3JhdGlvbnModmVyc2lvbiwgYXBwbGllZF9hdCkgVkFMVUVTICg/LCA/KSIsCiAgICAgICAgICAgICgxLCBkYXRldGltZS5ub3coVVRDKS5pc29mb3JtYXQoKSksCiAgICAgICAgKQogICAgICAgIHNlbGYuX21pZ3JhdGVfdmVyaWZpY2F0aW9uX2RhdGFzZXRfc291cmNlKCkKICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmNvbW1pdCgpCgogICAgZGVmIF9taWdyYXRlX3ZlcmlmaWNhdGlvbl9kYXRhc2V0X3NvdXJjZShzZWxmKSAtPiBOb25lOgogICAgICAgIHZlcnNpb24gPSBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICJTRUxFQ1QgMSBGUk9NIHNjaGVtYV9taWdyYXRpb25zIFdIRVJFIHZlcnNpb24gPSAyIgogICAgICAgICkuZmV0Y2hvbmUoKQogICAgICAgIGlmIHZlcnNpb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIGxlZ2FjeV9zb3VyY2UgPSAicmVwb3NpdG9yeTovL2RhdGFzZXRzL3NhbXBsZXMvZm91bmRyeV92ZXJpZmljYXRpb24udHh0IgogICAgICAgIGNhbm9uaWNhbF9zb3VyY2UgPSAicmVwb3NpdG9yeTovL29seW1wdXMvZm91bmRyeS9mb3VuZHJ5X3ZlcmlmaWNhdGlvbi50eHQiCiAgICAgICAgcm93ID0gc2VsZi5fY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAiIiJTRUxFQ1QgcmVjb3JkX2pzb24gRlJPTSBkYXRhc2V0cwogICAgICAgICAgICBXSEVSRSBkYXRhc2V0X2lkID0gPyBBTkQgdmVyc2lvbiA9ID8gQU5EIHNoYTI1NiA9ID8iIiIsCiAgICAgICAgICAgICgKICAgICAgICAgICAgICAgICJmb3VuZHJ5LXZlcmlmaWNhdGlvbi1jb3JwdXMiLAogICAgICAgICAgICAgICAgIjEuMC4wIiwKICAgICAgICAgICAgICAgICI2ZjQ4MjNlMDUwMzQyNmJkYTExYWE0YmZmZTgzNTc5NjNmM2JjOTdhNjJlYTQwYzJiNjk3MDBhZjE2NmE4ZWFiIiwKICAgICAgICAgICAgKSwKICAgICAgICApLmZldGNob25lKCkKICAgICAgICBpZiByb3cgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlY29yZCA9IERhdGFzZXRSZWNvcmQubW9kZWxfdmFsaWRhdGVfanNvbihyb3dbInJlY29yZF9qc29uIl0pCiAgICAgICAgICAgIGlmIHJlY29yZC5zb3VyY2UgPT0gbGVnYWN5X3NvdXJjZToKICAgICAgICAgICAgICAgIHJlY29yZC5zb3VyY2UgPSBjYW5vbmljYWxfc291cmNlCiAgICAgICAgICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAgICAgIiIiVVBEQVRFIGRhdGFzZXRzIFNFVCByZWNvcmRfanNvbiA9ID8KICAgICAgICAgICAgICAgICAgICBXSEVSRSBkYXRhc2V0X2lkID0gPyBBTkQgdmVyc2lvbiA9ID8iIiIsCiAgICAgICAgICAgICAgICAgICAgKHNlbGYuX2pzb24ocmVjb3JkKSwgcmVjb3JkLmRhdGFzZXRfaWQsIHJlY29yZC52ZXJzaW9uKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIHNlbGYuX2Nvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgICAgICAgICAiIiJJTlNFUlQgSU5UTyBldmlkZW5jZSgKICAgICAgICAgICAgICAgICAgICAgICAgdGltZXN0YW1wLCBlbnRpdHlfdHlwZSwgZW50aXR5X2lkLCBhY3Rpb24sIHBheWxvYWRfanNvbgogICAgICAgICAgICAgICAgICAgICkgVkFMVUVTICg/LCA/LCA/LCA/LCA/KSIiIiwKICAgICAgICAgICAgICAgICAgICAoCiAgICAgICAgICAgICAgICAgICAgICAgIGRhdGV0aW1lLm5vdyhVVEMpLmlzb2Zvcm1hdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAiZGF0YXNldCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJmb3VuZHJ5LXZlcmlmaWNhdGlvbi1jb3JwdXNAMS4wLjAiLAogICAgICAgICAgICAgICAgICAgICAgICAibWV0YWRhdGFfbWlncmF0ZWQiLAogICAgICAgICAgICAgICAgICAgICAgICBqc29uLmR1bXBzKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgeyJmaWVsZCI6ICJzb3VyY2UiLCAiZnJvbSI6IGxlZ2FjeV9zb3VyY2UsICJ0byI6IGNhbm9uaWNhbF9zb3VyY2V9LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgc29ydF9rZXlzPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IiksCiAgICAgICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgICkKICAgICAgICBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICJJTlNFUlQgSU5UTyBzY2hlbWFfbWlncmF0aW9ucyh2ZXJzaW9uLCBhcHBsaWVkX2F0KSBWQUxVRVMgKD8sID8pIiwKICAgICAgICAgICAgKDIsIGRhdGV0aW1lLm5vdyhVVEMpLmlzb2Zvcm1hdCgpKSwKICAgICAgICApCgogICAgQGNvbnRleHRtYW5hZ2VyCiAgICBkZWYgdHJhbnNhY3Rpb24oc2VsZikgLT4gSXRlcmF0b3Jbc3FsaXRlMy5Db25uZWN0aW9uXToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHNlbGYuX2Nvbm5lY3Rpb24uZXhlY3V0ZSgiQkVHSU4gSU1NRURJQVRFIikKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgeWllbGQgc2VsZi5fY29ubmVjdGlvbgogICAgICAgICAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHNlbGYuX2Nvbm5lY3Rpb24ucm9sbGJhY2soKQogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHNlbGYuX2Nvbm5lY3Rpb24uY29tbWl0KCkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX2pzb24oCiAgICAgICAgcmVjb3JkOiAoCiAgICAgICAgICAgIERhdGFzZXRSZWNvcmQKICAgICAgICAgICAgfCBFeHBlcmltZW50UmVjb3JkCiAgICAgICAgICAgIHwgQ2hlY2twb2ludFJlY29yZAogICAgICAgICAgICB8IEV2YWx1YXRpb25SZWNvcmQKICAgICAgICAgICAgfCBNb2RlbFJlY29yZAogICAgICAgICksCiAgICApIC0+IHN0cjoKICAgICAgICByZXR1cm4gcmVjb3JkLm1vZGVsX2R1bXBfanNvbigpCgogICAgZGVmIHJlZ2lzdGVyX2RhdGFzZXQoc2VsZiwgcmVjb3JkOiBEYXRhc2V0UmVjb3JkKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi50cmFuc2FjdGlvbigpIGFzIGNvbm5lY3Rpb246CiAgICAgICAgICAgIGV4aXN0aW5nID0gY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAgIiIiU0VMRUNUIHNoYTI1NiwgcmVjb3JkX2pzb24gRlJPTSBkYXRhc2V0cwogICAgICAgICAgICAgICAgV0hFUkUgZGF0YXNldF9pZCA9ID8gQU5EIHZlcnNpb24gPSA/IiIiLAogICAgICAgICAgICAgICAgKHJlY29yZC5kYXRhc2V0X2lkLCByZWNvcmQudmVyc2lvbiksCiAgICAgICAgICAgICkuZmV0Y2hvbmUoKQogICAgICAgICAgICBpZiBleGlzdGluZyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGlmIGV4aXN0aW5nWyJzaGEyNTYiXSAhPSByZWNvcmQuc2hhMjU2OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgICAgIGYiZGF0YXNldCBpZGVudGl0eSBhbHJlYWR5IHBvaW50cyB0byBkaWZmZXJlbnQgY29udGVudDogIgogICAgICAgICAgICAgICAgICAgICAgICBmIntyZWNvcmQuZGF0YXNldF9pZH1Ae3JlY29yZC52ZXJzaW9ufSIKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICByZWdpc3RlcmVkID0gRGF0YXNldFJlY29yZC5tb2RlbF92YWxpZGF0ZV9qc29uKGV4aXN0aW5nWyJyZWNvcmRfanNvbiJdKQogICAgICAgICAgICAgICAgcmVnaXN0ZXJlZF9tZXRhZGF0YSA9IHJlZ2lzdGVyZWQubW9kZWxfZHVtcChleGNsdWRlPXsiY3JlYXRlZF9hdCJ9KQogICAgICAgICAgICAgICAgaW5jb21pbmdfbWV0YWRhdGEgPSByZWNvcmQubW9kZWxfZHVtcChleGNsdWRlPXsiY3JlYXRlZF9hdCJ9KQogICAgICAgICAgICAgICAgaWYgcmVnaXN0ZXJlZF9tZXRhZGF0YSAhPSBpbmNvbWluZ19tZXRhZGF0YToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgICAgICAgICBmImRhdGFzZXQgaWRlbnRpdHkgYWxyZWFkeSBoYXMgZGlmZmVyZW50IG1ldGFkYXRhOiAiCiAgICAgICAgICAgICAgICAgICAgICAgIGYie3JlY29yZC5kYXRhc2V0X2lkfUB7cmVjb3JkLnZlcnNpb259IgogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBjb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiIiJJTlNFUlQgSU5UTyBkYXRhc2V0cyhkYXRhc2V0X2lkLCB2ZXJzaW9uLCBzaGEyNTYsIHJlY29yZF9qc29uKQogICAgICAgICAgICAgICAgVkFMVUVTICg/LCA/LCA/LCA/KSIiIiwKICAgICAgICAgICAgICAgIChyZWNvcmQuZGF0YXNldF9pZCwgcmVjb3JkLnZlcnNpb24sIHJlY29yZC5zaGEyNTYsIHNlbGYuX2pzb24ocmVjb3JkKSksCiAgICAgICAgICAgICkKCiAgICBkZWYgcmVnaXN0ZXJfZXhwZXJpbWVudChzZWxmLCByZWNvcmQ6IEV4cGVyaW1lbnRSZWNvcmQpIC0+IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLnRyYW5zYWN0aW9uKCkgYXMgY29ubmVjdGlvbjoKICAgICAgICAgICAgY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAgIiIiSU5TRVJUIElOVE8gZXhwZXJpbWVudHMoCiAgICAgICAgICAgICAgICAgICAgZXhwZXJpbWVudF9pZCwgZGF0YXNldF9pZCwgZGF0YXNldF92ZXJzaW9uLCBzdGF0dXMsIHJlY29yZF9qc29uCiAgICAgICAgICAgICAgICApIFZBTFVFUyAoPywgPywgPywgPywgPykiIiIsCiAgICAgICAgICAgICAgICAoCiAgICAgICAgICAgICAgICAgICAgcmVjb3JkLmV4cGVyaW1lbnRfaWQsCiAgICAgICAgICAgICAgICAgICAgcmVjb3JkLmRhdGFzZXRfaWQsCiAgICAgICAgICAgICAgICAgICAgcmVjb3JkLmRhdGFzZXRfdmVyc2lvbiwKICAgICAgICAgICAgICAgICAgICByZWNvcmQuc3RhdHVzLnZhbHVlLAogICAgICAgICAgICAgICAgICAgIHNlbGYuX2pzb24ocmVjb3JkKSwKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICkKCiAgICBkZWYgdXBkYXRlX2V4cGVyaW1lbnQoc2VsZiwgcmVjb3JkOiBFeHBlcmltZW50UmVjb3JkKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi50cmFuc2FjdGlvbigpIGFzIGNvbm5lY3Rpb246CiAgICAgICAgICAgIGNoYW5nZWQgPSBjb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiVVBEQVRFIGV4cGVyaW1lbnRzIFNFVCBzdGF0dXMgPSA/LCByZWNvcmRfanNvbiA9ID8gV0hFUkUgZXhwZXJpbWVudF9pZCA9ID8iLAogICAgICAgICAgICAgICAgKHJlY29yZC5zdGF0dXMudmFsdWUsIHNlbGYuX2pzb24ocmVjb3JkKSwgcmVjb3JkLmV4cGVyaW1lbnRfaWQpLAogICAgICAgICAgICApLnJvd2NvdW50CiAgICAgICAgICAgIGlmIGNoYW5nZWQgIT0gMToKICAgICAgICAgICAgICAgIHJhaXNlIEtleUVycm9yKGYidW5rbm93biBleHBlcmltZW50OiB7cmVjb3JkLmV4cGVyaW1lbnRfaWR9IikKCiAgICBkZWYgcmVnaXN0ZXJfY2hlY2twb2ludChzZWxmLCByZWNvcmQ6IENoZWNrcG9pbnRSZWNvcmQpIC0+IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLnRyYW5zYWN0aW9uKCkgYXMgY29ubmVjdGlvbjoKICAgICAgICAgICAgY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAgIiIiSU5TRVJUIElOVE8gY2hlY2twb2ludHMoCiAgICAgICAgICAgICAgICAgICAgY2hlY2twb2ludF9pZCwgZXhwZXJpbWVudF9pZCwgc2hhMjU2LCByZWNvcmRfanNvbgogICAgICAgICAgICAgICAgKSBWQUxVRVMgKD8sID8sID8sID8pIiIiLAogICAgICAgICAgICAgICAgKHJlY29yZC5jaGVja3BvaW50X2lkLCByZWNvcmQuZXhwZXJpbWVudF9pZCwgcmVjb3JkLnNoYTI1Niwgc2VsZi5fanNvbihyZWNvcmQpKSwKICAgICAgICAgICAgKQoKICAgIGRlZiByZWdpc3Rlcl9ldmFsdWF0aW9uKHNlbGYsIHJlY29yZDogRXZhbHVhdGlvblJlY29yZCkgLT4gTm9uZToKICAgICAgICB3aXRoIHNlbGYudHJhbnNhY3Rpb24oKSBhcyBjb25uZWN0aW9uOgogICAgICAgICAgICBjb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiIiJJTlNFUlQgSU5UTyBldmFsdWF0aW9ucyhldmFsdWF0aW9uX2lkLCBjaGVja3BvaW50X2lkLCByZWNvcmRfanNvbikKICAgICAgICAgICAgICAgIFZBTFVFUyAoPywgPywgPykiIiIsCiAgICAgICAgICAgICAgICAocmVjb3JkLmV2YWx1YXRpb25faWQsIHJlY29yZC5jaGVja3BvaW50X2lkLCBzZWxmLl9qc29uKHJlY29yZCkpLAogICAgICAgICAgICApCgogICAgZGVmIHJlZ2lzdGVyX21vZGVsKHNlbGYsIHJlY29yZDogTW9kZWxSZWNvcmQpIC0+IE5vbmU6CiAgICAgICAgaWYgcmVjb3JkLnN0YXR1cyBpcyBub3QgQXJ0aWZhY3RTdGF0dXMuVkVSSUZJRUQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9ubHkgdmVyaWZpZWQgY2hlY2twb2ludHMgY2FuIGJlIHByb21vdGVkIHRvIHRoZSBtb2RlbCByZWdpc3RyeSIpCiAgICAgICAgd2l0aCBzZWxmLnRyYW5zYWN0aW9uKCkgYXMgY29ubmVjdGlvbjoKICAgICAgICAgICAgY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAgIiIiSU5TRVJUIElOVE8gbW9kZWxzKG1vZGVsX2lkLCBjaGVja3BvaW50X2lkLCBzdGF0dXMsIHJlY29yZF9qc29uKQogICAgICAgICAgICAgICAgVkFMVUVTICg/LCA/LCA/LCA/KSIiIiwKICAgICAgICAgICAgICAgIChyZWNvcmQubW9kZWxfaWQsIHJlY29yZC5jaGVja3BvaW50X2lkLCByZWNvcmQuc3RhdHVzLnZhbHVlLCBzZWxmLl9qc29uKHJlY29yZCkpLAogICAgICAgICAgICApCgogICAgZGVmIGFwcGVuZF9ldmlkZW5jZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAgdGltZXN0YW1wOiBzdHIsCiAgICAgICAgZW50aXR5X3R5cGU6IHN0ciwKICAgICAgICBlbnRpdHlfaWQ6IHN0ciwKICAgICAgICBhY3Rpb246IHN0ciwKICAgICAgICBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSwKICAgICkgLT4gaW50OgogICAgICAgIHdpdGggc2VsZi50cmFuc2FjdGlvbigpIGFzIGNvbm5lY3Rpb246CiAgICAgICAgICAgIGN1cnNvciA9IGNvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgICAgICIiIklOU0VSVCBJTlRPIGV2aWRlbmNlKAogICAgICAgICAgICAgICAgICAgIHRpbWVzdGFtcCwgZW50aXR5X3R5cGUsIGVudGl0eV9pZCwgYWN0aW9uLCBwYXlsb2FkX2pzb24KICAgICAgICAgICAgICAgICkgVkFMVUVTICg/LCA/LCA/LCA/LCA/KSIiIiwKICAgICAgICAgICAgICAgICgKICAgICAgICAgICAgICAgICAgICB0aW1lc3RhbXAsCiAgICAgICAgICAgICAgICAgICAgZW50aXR5X3R5cGUsCiAgICAgICAgICAgICAgICAgICAgZW50aXR5X2lkLAogICAgICAgICAgICAgICAgICAgIGFjdGlvbiwKICAgICAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHBheWxvYWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgiLCIsICI6IikpLAogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBjdXJzb3IubGFzdHJvd2lkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlNRTGl0ZSBkaWQgbm90IHJldHVybiBhbiBldmlkZW5jZSBzZXF1ZW5jZSIpCiAgICAgICAgICAgIHJldHVybiBjdXJzb3IubGFzdHJvd2lkCgogICAgZGVmIF9yZWNvcmRzKAogICAgICAgIHNlbGYsCiAgICAgICAgdGFibGU6IHN0ciwKICAgICAgICBtb2RlbDogKAogICAgICAgICAgICB0eXBlW0RhdGFzZXRSZWNvcmRdCiAgICAgICAgICAgIHwgdHlwZVtFeHBlcmltZW50UmVjb3JkXQogICAgICAgICAgICB8IHR5cGVbQ2hlY2twb2ludFJlY29yZF0KICAgICAgICAgICAgfCB0eXBlW0V2YWx1YXRpb25SZWNvcmRdCiAgICAgICAgICAgIHwgdHlwZVtNb2RlbFJlY29yZF0KICAgICAgICApLAogICAgKSAtPiBsaXN0W0FueV06CiAgICAgICAgYWxsb3dlZCA9IHsiZGF0YXNldHMiLCAiZXhwZXJpbWVudHMiLCAiY2hlY2twb2ludHMiLCAiZXZhbHVhdGlvbnMiLCAibW9kZWxzIn0KICAgICAgICBpZiB0YWJsZSBub3QgaW4gYWxsb3dlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuc3VwcG9ydGVkIHJlZ2lzdHJ5IHRhYmxlOiB7dGFibGV9IikKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJvd3MgPSBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoZiJTRUxFQ1QgcmVjb3JkX2pzb24gRlJPTSB7dGFibGV9IikuZmV0Y2hhbGwoKQogICAgICAgIHJldHVybiBbbW9kZWwubW9kZWxfdmFsaWRhdGVfanNvbihyb3dbInJlY29yZF9qc29uIl0pIGZvciByb3cgaW4gcm93c10KCiAgICBkZWYgZGF0YXNldHMoc2VsZikgLT4gbGlzdFtEYXRhc2V0UmVjb3JkXToKICAgICAgICByZXR1cm4gc2VsZi5fcmVjb3JkcygiZGF0YXNldHMiLCBEYXRhc2V0UmVjb3JkKQoKICAgIGRlZiBleHBlcmltZW50cyhzZWxmKSAtPiBsaXN0W0V4cGVyaW1lbnRSZWNvcmRdOgogICAgICAgIHJldHVybiBzZWxmLl9yZWNvcmRzKCJleHBlcmltZW50cyIsIEV4cGVyaW1lbnRSZWNvcmQpCgogICAgZGVmIGNoZWNrcG9pbnRzKHNlbGYpIC0+IGxpc3RbQ2hlY2twb2ludFJlY29yZF06CiAgICAgICAgcmV0dXJuIHNlbGYuX3JlY29yZHMoImNoZWNrcG9pbnRzIiwgQ2hlY2twb2ludFJlY29yZCkKCiAgICBkZWYgZXZhbHVhdGlvbnMoc2VsZikgLT4gbGlzdFtFdmFsdWF0aW9uUmVjb3JkXToKICAgICAgICByZXR1cm4gc2VsZi5fcmVjb3JkcygiZXZhbHVhdGlvbnMiLCBFdmFsdWF0aW9uUmVjb3JkKQoKICAgIGRlZiBtb2RlbHMoc2VsZikgLT4gbGlzdFtNb2RlbFJlY29yZF06CiAgICAgICAgcmV0dXJuIHNlbGYuX3JlY29yZHMoIm1vZGVscyIsIE1vZGVsUmVjb3JkKQoKICAgIGRlZiBtb2RlbChzZWxmLCBtb2RlbF9pZDogc3RyKSAtPiBNb2RlbFJlY29yZCB8IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByb3cgPSBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiU0VMRUNUIHJlY29yZF9qc29uIEZST00gbW9kZWxzIFdIRVJFIG1vZGVsX2lkID0gPyIsIChtb2RlbF9pZCwpCiAgICAgICAgICAgICkuZmV0Y2hvbmUoKQogICAgICAgIHJldHVybiBOb25lIGlmIHJvdyBpcyBOb25lIGVsc2UgTW9kZWxSZWNvcmQubW9kZWxfdmFsaWRhdGVfanNvbihyb3dbInJlY29yZF9qc29uIl0pCgogICAgZGVmIGNoZWNrcG9pbnQoc2VsZiwgY2hlY2twb2ludF9pZDogc3RyKSAtPiBDaGVja3BvaW50UmVjb3JkIHwgTm9uZToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJvdyA9IHNlbGYuX2Nvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgICAgICJTRUxFQ1QgcmVjb3JkX2pzb24gRlJPTSBjaGVja3BvaW50cyBXSEVSRSBjaGVja3BvaW50X2lkID0gPyIsIChjaGVja3BvaW50X2lkLCkKICAgICAgICAgICAgKS5mZXRjaG9uZSgpCiAgICAgICAgcmV0dXJuIE5vbmUgaWYgcm93IGlzIE5vbmUgZWxzZSBDaGVja3BvaW50UmVjb3JkLm1vZGVsX3ZhbGlkYXRlX2pzb24ocm93WyJyZWNvcmRfanNvbiJdKQoKICAgIGRlZiBldmlkZW5jZShzZWxmKSAtPiBsaXN0W0V2aWRlbmNlUmVjb3JkXToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJvd3MgPSBzZWxmLl9jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiIiJTRUxFQ1Qgc2VxdWVuY2UsIHRpbWVzdGFtcCwgZW50aXR5X3R5cGUsIGVudGl0eV9pZCwgYWN0aW9uLCBwYXlsb2FkX2pzb24KICAgICAgICAgICAgICAgIEZST00gZXZpZGVuY2UgT1JERVIgQlkgc2VxdWVuY2UiIiIKICAgICAgICAgICAgKS5mZXRjaGFsbCgpCiAgICAgICAgcmV0dXJuIFsKICAgICAgICAgICAgRXZpZGVuY2VSZWNvcmQoCiAgICAgICAgICAgICAgICBzZXF1ZW5jZT1yb3dbInNlcXVlbmNlIl0sCiAgICAgICAgICAgICAgICB0aW1lc3RhbXA9cm93WyJ0aW1lc3RhbXAiXSwKICAgICAgICAgICAgICAgIGVudGl0eV90eXBlPXJvd1siZW50aXR5X3R5cGUiXSwKICAgICAgICAgICAgICAgIGVudGl0eV9pZD1yb3dbImVudGl0eV9pZCJdLAogICAgICAgICAgICAgICAgYWN0aW9uPXJvd1siYWN0aW9uIl0sCiAgICAgICAgICAgICAgICBwYXlsb2FkPWpzb24ubG9hZHMocm93WyJwYXlsb2FkX2pzb24iXSksCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIHJvdyBpbiByb3dzCiAgICAgICAgXQoKICAgIGRlZiBpbnRlZ3JpdHlfY2hlY2soc2VsZikgLT4gc3RyOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgcmVzdWx0ID0gc2VsZi5fY29ubmVjdGlvbi5leGVjdXRlKCJQUkFHTUEgaW50ZWdyaXR5X2NoZWNrIikuZmV0Y2hvbmUoKQogICAgICAgIHJldHVybiBzdHIocmVzdWx0WzBdKQoKICAgIGRlZiBjbG9zZShzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgc2VsZi5fY29ubmVjdGlvbi5jbG9zZSgpCgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBGb3VuZHJ5U3RvcmU6CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKl86IG9iamVjdCkgLT4gTm9uZToKICAgICAgICBzZWxmLmNsb3NlKCkK', 'olympus/labos/__init__.py': 'IiIiTGFiT1MgcG9ydGZvbGlvIG9yY2hlc3RyYXRpb24uIiIiCgpmcm9tIG9seW1wdXMubGFib3MuZGlzY292ZXJ5IGltcG9ydCBkaXNjb3Zlcl9wcm9qZWN0cwpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0IFByb2plY3RNYW5pZmVzdApmcm9tIG9seW1wdXMubGFib3MucG9ydGZvbGlvIGltcG9ydCBQb3J0Zm9saW9TZXJ2aWNlCmZyb20gb2x5bXB1cy5sYWJvcy5yZXBvcnRzIGltcG9ydCBQb3J0Zm9saW9SZXBvcnRlcgpmcm9tIG9seW1wdXMubGFib3MucnVubmVyIGltcG9ydCBQb3J0Zm9saW9SdW5uZXIKZnJvbSBvbHltcHVzLmxhYm9zLnZhbGlkYXRpb24gaW1wb3J0IHZhbGlkYXRlX3Byb2plY3QKCl9fYWxsX18gPSBbCiAgICAiUG9ydGZvbGlvU2VydmljZSIsCiAgICAiUG9ydGZvbGlvUmVwb3J0ZXIiLAogICAgIlBvcnRmb2xpb1J1bm5lciIsCiAgICAiUHJvamVjdE1hbmlmZXN0IiwKICAgICJkaXNjb3Zlcl9wcm9qZWN0cyIsCiAgICAidmFsaWRhdGVfcHJvamVjdCIsCl0K', 'olympus/labos/discovery.py': '...', 'olympus/labos/manifest.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJFbnVtCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBNYW5pZmVzdFNvdXJjZShTdHJFbnVtKToKICAgIERFQ0xBUkVEID0gImRlY2xhcmVkIgogICAgSU5GRVJSRUQgPSAiaW5mZXJyZWQiCgoKY2xhc3MgUHJvamVjdExhbmd1YWdlKFN0ckVudW0pOgogICAgUFlUSE9OID0gInB5dGhvbiIKICAgIFRZUEVTQ1JJUFQgPSAidHlwZXNjcmlwdCIKICAgIE1JWEVEID0gIm1peGVkIgogICAgVU5LTk9XTiA9ICJ1bmtub3duIgoKCmNsYXNzIFByb2plY3RTdGF0dXMoU3RyRW51bSk6CiAgICBDT01QTEVURSA9ICJjb21wbGV0ZSIKICAgIFNDSUVOVElGSUNBTExZX1JVTk5BQkxFID0gInNjaWVudGlmaWNhbGx5X3J1bm5hYmxlIgogICAgU01PS0VfVEVTVEVEID0gInNtb2tlX3Rlc3RlZCIKICAgIEZVTExfQkVOQ0hNQVJLX1BFTkRJTkdfQ09NUFVURSA9ICJmdWxsX2JlbmNobWFya19wZW5kaW5nX2NvbXB1dGUiCiAgICBCTE9DS0VEX0JZX0VYVEVSTkFMX0RBVEFTRVQgPSAiYmxvY2tlZF9ieV9leHRlcm5hbF9kYXRhc2V0IgogICAgQkxPQ0tFRF9CWV9DUkVERU5USUFMID0gImJsb2NrZWRfYnlfY3JlZGVudGlhbCIKICAgIEJMT0NLRURfQllfUEVSTUlTU0lPTiA9ICJibG9ja2VkX2J5X3Blcm1pc3Npb24iCiAgICBESVNDT1ZFUkVEX09OTFkgPSAiZGlzY292ZXJlZF9vbmx5IgoKCmNsYXNzIFByb2plY3RDbGFzc2lmaWNhdGlvbihTdHJFbnVtKToKICAgIEZMQUdTSElQX0NPTVBBTlkgPSAiZmxhZ3NoaXBfY29tcGFueSIKICAgIEhJR0hfUE9URU5USUFMX01WUCA9ICJoaWdoX3BvdGVudGlhbF9tdnAiCiAgICBVU0VGVUxfRkVBVFVSRSA9ICJ1c2VmdWxfZmVhdHVyZV9mb3JfYW5vdGhlcl9jb21wYW55IgogICAgUkVTRUFSQ0hfUFJPSkVDVCA9ICJyZXNlYXJjaF9wcm9qZWN0IgogICAgUkVRVUlSRVNfVkFMSURBVElPTiA9ICJyZXF1aXJlc192YWxpZGF0aW9uIgogICAgV0VBS19ESUZGRVJFTlRJQVRJT04gPSAid2Vha19kaWZmZXJlbnRpYXRpb24iCiAgICBNRVJHRV9XSVRIX0FOT1RIRVJfSURFQSA9ICJtZXJnZV93aXRoX2Fub3RoZXJfaWRlYSIKICAgIEFSQ0hJVkUgPSAiYXJjaGl2ZSIKCgpjbGFzcyBSZXNvdXJjZVByb2ZpbGUoU3RyRW51bSk6CiAgICBTTU9LRSA9ICJzbW9rZSIKICAgIERFVkVMT1BNRU5UID0gImRldmVsb3BtZW50IgogICAgQkVOQ0hNQVJLID0gImJlbmNobWFyayIKICAgIEZVTEwgPSAiZnVsbCIKCgpjbGFzcyBEYXRhc2V0RGVwZW5kZW5jeShTdHJpY3RNb2RlbCk6CiAgICBuYW1lOiBzdHIKICAgIHJlcXVpcmVkOiBib29sID0gVHJ1ZQogICAgbm90ZXM6IHN0ciA9ICIiCiAgICBibG9ja2VkX2J5X3VzZXJfYWN0aW9uOiBib29sID0gRmFsc2UKCgpjbGFzcyBFbnRyeVBvaW50KFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgY29tbWFuZDogc3RyCiAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUKICAgIGV4cGVjdGVkX291dHB1dHM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZW52OiBkaWN0W3N0ciwgc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgdGltZW91dF9zZWNvbmRzOiBpbnQgPSAxMjAKCgpjbGFzcyBSZXNvdXJjZVJlcXVpcmVtZW50cyhTdHJpY3RNb2RlbCk6CiAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUgPSBSZXNvdXJjZVByb2ZpbGUuU01PS0UKICAgIGNwdV9vbmx5OiBib29sID0gVHJ1ZQogICAgZ3B1X29wdGlvbmFsOiBib29sID0gRmFsc2UKICAgIG1lbW9yeV9nYjogaW50ID0gMgogICAgbm90ZXM6IHN0ciA9ICIiCgoKY2xhc3MgUHJvamVjdE1hbmlmZXN0KFN0cmljdE1vZGVsKToKICAgIHByb2plY3RfaWQ6IHN0cgogICAgdGl0bGU6IHN0cgogICAgZGVzY3JpcHRpb246IHN0cgogICAgaHlwb3RoZXNpczogc3RyCiAgICByb290X3BhdGg6IHN0cgogICAgZG9tYWluOiBzdHIKICAgIGxhbmd1YWdlOiBQcm9qZWN0TGFuZ3VhZ2UgPSBQcm9qZWN0TGFuZ3VhZ2UuVU5LTk9XTgogICAgbWFuaWZlc3Rfc291cmNlOiBNYW5pZmVzdFNvdXJjZSA9IE1hbmlmZXN0U291cmNlLklORkVSUkVECiAgICBlbnRyeV9wb2ludHM6IGxpc3RbRW50cnlQb2ludF0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGRhdGFzZXRzOiBsaXN0W0RhdGFzZXREZXBlbmRlbmN5XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZXhwZWN0ZWRfb3V0cHV0czogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICB2YWxpZGF0aW9uX2NvbW1hbmRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHRhZ3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgY2xhc3NpZmljYXRpb246IFByb2plY3RDbGFzc2lmaWNhdGlvbiA9IFByb2plY3RDbGFzc2lmaWNhdGlvbi5SRVNFQVJDSF9QUk9KRUNUCiAgICB0YXJnZXRfdXNlcjogc3RyID0gIlJlc2VhcmNoIGVuZ2luZWVyIG9yIHRlY2huaWNhbCBmb3VuZGVyIGV2YWx1YXRpbmcgdGhlIHByb2plY3QuIgogICAgY3VycmVudF9hbHRlcm5hdGl2ZTogc3RyID0gKAogICAgICAgICJNYW51YWwgbm90ZWJvb2tzLCBvbmUtb2ZmIHNjcmlwdHMsIG9yIHVuZ292ZXJuZWQgcHJvdG90eXBlIHJlcG9zaXRvcmllcy4iCiAgICApCiAgICBwcm9wb3NlZF9zb2x1dGlvbjogc3RyID0gKAogICAgICAgICJBIHJlcHJvZHVjaWJsZSBsb2NhbCBpbXBsZW1lbnRhdGlvbiB3aXRoIG1hbmlmZXN0cywgdGVzdHMsIHJlcG9ydHMsICIKICAgICAgICAiYW5kIGV4cGxpY2l0IHJlbGVhc2UgZ2F0ZXMuIgogICAgKQogICAgZGlmZmVyZW50aWF0aW9uOiBzdHIgPSAoCiAgICAgICAgIkV2aWRlbmNlLWZpcnN0IGV4ZWN1dGlvbiB3aXRoIG1hY2hpbmUtcmVhZGFibGUgc3RhdHVzIGFuZCBibG9ja2VkLWFjdGlvbiB0cmFja2luZy4iCiAgICApCiAgICBtYXJrZXQ6IHN0ciA9ICgKICAgICAgICAiU3BlY2lhbGl6ZWQgcmVzZWFyY2gsIGRldmVsb3BlciwgYW5kIGZpbnRlY2gvTUwgaW5mcmFzdHJ1Y3R1cmUgbWFya2V0cyAiCiAgICAgICAgImRlcGVuZGluZyBvbiB0aGUgcHJvamVjdCBkb21haW4uIgogICAgKQogICAgYnVzaW5lc3NfbW9kZWw6IHN0ciA9ICgKICAgICAgICAiUmVxdWlyZXMgcHJvamVjdC1zcGVjaWZpYyB2YWxpZGF0aW9uIGJlZm9yZSBwcmljaW5nOyBsaWtlbHkgc3RhcnRzIGFzICIKICAgICAgICAiY29uc3VsdGluZywgU2FhUywgb3IgaW5mcmFzdHJ1Y3R1cmUgbGljZW5zaW5nLiIKICAgICkKICAgIGRpc3RyaWJ1dGlvbl9zdHJhdGVneTogc3RyID0gKAogICAgICAgICJUZWNobmljYWwgY29udGVudCwgZm91bmRlci1sZWQgZGVzaWduIHBhcnRuZXJzaGlwcywgb3BlbiBkZW1vcywgYW5kIHRhcmdldGVkIHBpbG90cy4iCiAgICApCiAgICB0ZWNobmljYWxfcmVxdWlyZW1lbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGRhdGFfcmVxdWlyZW1lbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHJlZ3VsYXRvcnlfY29uc2lkZXJhdGlvbnM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgc2VjdXJpdHlfcmlza3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgY2FwaXRhbF9pbnRlbnNpdHk6IHN0ciA9ICJsb3ctdG8tbWVkaXVtIGJlZm9yZSBmdWxsIGJlbmNobWFyayBzY2FsZS11cCIKICAgIHRpbWVfdG9fbXZwOiBzdHIgPSAiMS00IHdlZWtzIGZvciBhIG5hcnJvdyBzbW9rZS10ZXN0ZWQgTVZQIHdoZW4gZGVwZW5kZW5jaWVzIGFyZSBsb2NhbCIKICAgIGV2aWRlbmNlX2F2YWlsYWJsZTogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBmYXRhbF9hc3N1bXB0aW9uczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICB2YWxpZGF0aW9uX2V4cGVyaW1lbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGdvX25vX2dvX2NyaXRlcmlhOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGJlbmNobWFya19wbGFuOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGFibGF0aW9uX3BsYW46IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgcmVsZWFzZV9nYXRlczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBzY2llbnRpZmljX2NhdmVhdDogc3RyID0gKAogICAgICAgICJTbW9rZSB0ZXN0cyBhbmQgc3ludGhldGljIGV4cGVyaW1lbnRzIGFyZSBlbmdpbmVlcmluZyBldmlkZW5jZSwgbm90IHByb29mIG9mICIKICAgICAgICAic2NpZW50aWZpYyBzdXBlcmlvcml0eSBvciByZWd1bGF0ZWQgcHJvZHVjdGlvbiByZWFkaW5lc3MuIgogICAgKQogICAgcmVzb3VyY2VfcmVxdWlyZW1lbnRzOiBSZXNvdXJjZVJlcXVpcmVtZW50cyA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1SZXNvdXJjZVJlcXVpcmVtZW50cykKICAgIG1ldGFkYXRhOiBkaWN0W3N0ciwgQW55XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHBhdGgoc2VsZikgLT4gUGF0aDoKICAgICAgICByZXR1cm4gUGF0aChzZWxmLnJvb3RfcGF0aCkKCiAgICBkZWYgc21va2VfZW50cnlfcG9pbnRzKHNlbGYpIC0+IGxpc3RbRW50cnlQb2ludF06CiAgICAgICAgcmV0dXJuIFtpdGVtIGZvciBpdGVtIGluIHNlbGYuZW50cnlfcG9pbnRzIGlmIGl0ZW0ucHJvZmlsZSA9PSBSZXNvdXJjZVByb2ZpbGUuU01PS0VdCgoKY2xhc3MgVmFsaWRhdGlvbklzc3VlKFN0cmljdE1vZGVsKToKICAgIHNldmVyaXR5OiBzdHIKICAgIG1lc3NhZ2U6IHN0cgoKCmNsYXNzIFZhbGlkYXRpb25SZXN1bHQoU3RyaWN0TW9kZWwpOgogICAgcHJvamVjdF9pZDogc3RyCiAgICB2YWxpZDogYm9vbAogICAgaXNzdWVzOiBsaXN0W1ZhbGlkYXRpb25Jc3N1ZV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCgpjbGFzcyBQcm9qZWN0UmVjb3JkKFN0cmljdE1vZGVsKToKICAgIG1hbmlmZXN0OiBQcm9qZWN0TWFuaWZlc3QKICAgIHZhbGlkYXRpb246IFZhbGlkYXRpb25SZXN1bHQKICAgIHN0YXR1czogUHJvamVjdFN0YXR1cyA9IFByb2plY3RTdGF0dXMuRElTQ09WRVJFRF9PTkxZCiAgICBzdW1tYXJ5OiBzdHIgPSAiIgogICAgYmxvY2tlZF9yZWFzb246IHN0ciA9ICIiCg==', 'olympus/labos/portfolio.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIG9seW1wdXMubGFib3MuZGlzY292ZXJ5IGltcG9ydCBkaXNjb3Zlcl9wcm9qZWN0cwpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0ICgKICAgIFByb2plY3RNYW5pZmVzdCwKICAgIFByb2plY3RSZWNvcmQsCiAgICBQcm9qZWN0U3RhdHVzLAogICAgUmVzb3VyY2VQcm9maWxlLAopCmZyb20gb2x5bXB1cy5sYWJvcy5yZXBvcnRzIGltcG9ydCBQb3J0Zm9saW9SZXBvcnRlcgpmcm9tIG9seW1wdXMubGFib3MucnVubmVyIGltcG9ydCBQb3J0Zm9saW9SdW5uZXIKZnJvbSBvbHltcHVzLmxhYm9zLnZhbGlkYXRpb24gaW1wb3J0IHZhbGlkYXRlX3Byb2plY3QKCgpjbGFzcyBQb3J0Zm9saW9TZXJ2aWNlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHdvcmtzcGFjZV9yb290OiBQYXRoLCBhcnRpZmFjdF9yb290OiBQYXRoKSAtPiBOb25lOgogICAgICAgIHNlbGYud29ya3NwYWNlX3Jvb3QgPSB3b3Jrc3BhY2Vfcm9vdAogICAgICAgIHNlbGYuYXJ0aWZhY3Rfcm9vdCA9IGFydGlmYWN0X3Jvb3QKICAgICAgICBzZWxmLnJ1bm5lciA9IFBvcnRmb2xpb1J1bm5lcihhcnRpZmFjdF9yb290KQogICAgICAgIHNlbGYucmVwb3J0ZXIgPSBQb3J0Zm9saW9SZXBvcnRlcihhcnRpZmFjdF9yb290LCB3b3Jrc3BhY2Vfcm9vdCkKCiAgICBkZWYgY2xvc2Uoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLnJlcG9ydGVyLmNsb3NlKCkKICAgICAgICBzZWxmLnJ1bm5lci5jbG9zZSgpCgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBQb3J0Zm9saW9TZXJ2aWNlOgogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIF9fZXhpdF9fKHNlbGYsICphcmdzOiBvYmplY3QpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5jbG9zZSgpCgogICAgZGVmIGRpc2NvdmVyKHNlbGYpIC0+IGxpc3RbUHJvamVjdFJlY29yZF06CiAgICAgICAgbWFuaWZlc3RzID0gZGlzY292ZXJfcHJvamVjdHMoc2VsZi53b3Jrc3BhY2Vfcm9vdCkKICAgICAgICByZWNvcmRzID0gW3NlbGYuX3JlY29yZF9mcm9tX21hbmlmZXN0KG1hbmlmZXN0KSBmb3IgbWFuaWZlc3QgaW4gbWFuaWZlc3RzXQogICAgICAgIHJldHVybiBzZWxmLl9hdHRhY2hfcnVuX2hpc3RvcnkocmVjb3JkcykKCiAgICBkZWYgdmFsaWRhdGUoc2VsZikgLT4gbGlzdFtQcm9qZWN0UmVjb3JkXToKICAgICAgICByZXR1cm4gc2VsZi5kaXNjb3ZlcigpCgogICAgZGVmIHJ1bl9zbW9rZSgKICAgICAgICBzZWxmLAogICAgICAgIHByb2plY3RfaWRzOiBzZXRbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgIHByb2ZpbGU6IFJlc291cmNlUHJvZmlsZSA9IFJlc291cmNlUHJvZmlsZS5TTU9LRSwKICAgICkgLT4gbGlzdFtQcm9qZWN0UmVjb3JkXToKICAgICAgICByZWNvcmRzID0gc2VsZi5kaXNjb3ZlcigpCiAgICAgICAgZm9yIHJlY29yZCBpbiByZWNvcmRzOgogICAgICAgICAgICBpZiBwcm9qZWN0X2lkcyBpcyBub3QgTm9uZSBhbmQgcmVjb3JkLm1hbmlmZXN0LnByb2plY3RfaWQgbm90IGluIHByb2plY3RfaWRzOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcmVzdWx0cyA9IHNlbGYucnVubmVyLnJ1bl9tYW5pZmVzdChyZWNvcmQubWFuaWZlc3QsIHByb2ZpbGU9cHJvZmlsZSkKICAgICAgICAgICAgaWYgcmVzdWx0czoKICAgICAgICAgICAgICAgIHJlY29yZC5zdGF0dXMgPSByZXN1bHRzWy0xXS5zdGF0dXMKICAgICAgICAgICAgICAgIGlmICgKICAgICAgICAgICAgICAgICAgICByZWNvcmQuc3RhdHVzID09IFByb2plY3RTdGF0dXMuU01PS0VfVEVTVEVECiAgICAgICAgICAgICAgICAgICAgYW5kIGFueShkYXRhc2V0LmJsb2NrZWRfYnlfdXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gcmVjb3JkLm1hbmlmZXN0LmRhdGFzZXRzKQogICAgICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgICAgICByZWNvcmQuc3RhdHVzID0gUHJvamVjdFN0YXR1cy5GVUxMX0JFTkNITUFSS19QRU5ESU5HX0NPTVBVVEUKICAgICAgICAgICAgICAgIHJlY29yZC5zdW1tYXJ5ID0gKAogICAgICAgICAgICAgICAgICAgIGYiRXhlY3V0ZWQge2xlbihyZXN1bHRzKX0ge3Byb2ZpbGUudmFsdWV9IGNvbW1hbmQocyk7ICIKICAgICAgICAgICAgICAgICAgICBmImxhc3QgY2xhc3NpZmljYXRpb249e3Jlc3VsdHNbLTFdLmNsYXNzaWZpY2F0aW9ufS4iCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBpZiByZXN1bHRzWy0xXS5jbGFzc2lmaWNhdGlvbiAhPSAib2siOgogICAgICAgICAgICAgICAgICAgIHJlY29yZC5ibG9ja2VkX3JlYXNvbiA9IF9kZXNjcmliZV9ydW5fZmFpbHVyZSgKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1stMV0uY2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdHNbLTFdLnN0ZG91dCwKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1stMV0uc3RkZXJyLAogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGVsaWYgYW55KAogICAgICAgICAgICAgICAgICAgIGRhdGFzZXQuYmxvY2tlZF9ieV91c2VyX2FjdGlvbiBmb3IgZGF0YXNldCBpbiByZWNvcmQubWFuaWZlc3QuZGF0YXNldHMKICAgICAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICAgICAgcmVjb3JkLmJsb2NrZWRfcmVhc29uID0gKAogICAgICAgICAgICAgICAgICAgICAgICAiT25lIG9yIG1vcmUgZGF0YXNldHMgcmVxdWlyZSBtYW51YWwgZG93bmxvYWQgb3IgYWNjZXB0YW5jZS4iCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICByZWNvcmQuYmxvY2tlZF9yZWFzb24gPSAiIgogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmVjb3JkLnN1bW1hcnkgPSAiTm8gbWF0Y2hpbmcgZW50cnkgcG9pbnRzIGZvciB0aGUgcmVxdWVzdGVkIHByb2ZpbGUuIgogICAgICAgIHJldHVybiByZWNvcmRzCgogICAgZGVmIHdyaXRlX3JlcG9ydHMoCiAgICAgICAgc2VsZiwKICAgICAgICByZWNvcmRzOiBsaXN0W1Byb2plY3RSZWNvcmRdLAogICAgICAgIG91dHB1dF9yb290OiBQYXRoLAogICAgKSAtPiBOb25lOgogICAgICAgIG91dHB1dF9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxmLnJlcG9ydGVyLndyaXRlX3N0YXR1c19qc29uKHJlY29yZHMsIG91dHB1dF9yb290IC8gInBvcnRmb2xpb19zdGF0dXMuanNvbiIpCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9jb21wbGV0aW9uX3JlcG9ydCgKICAgICAgICAgICAgcmVjb3JkcywKICAgICAgICAgICAgb3V0cHV0X3Jvb3QgLyAiUE9SVEZPTElPX0NPTVBMRVRJT05fUkVQT1JULm1kIiwKICAgICAgICApCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9ydW5zX2pzb24ob3V0cHV0X3Jvb3QgLyAicG9ydGZvbGlvX3J1bnMuanNvbiIpCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9zY2llbnRpZmljX3ZhbGlkYXRpb25fcmVwb3J0KAogICAgICAgICAgICByZWNvcmRzLAogICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJTQ0lFTlRJRklDX1ZBTElEQVRJT05fUkVQT1JULm1kIiwKICAgICAgICApCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9yZXByb2R1Y2liaWxpdHlfcmVwb3J0KAogICAgICAgICAgICByZWNvcmRzLAogICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJSRVBST0RVQ0lCSUxJVFkubWQiLAogICAgICAgICkKICAgICAgICBzZWxmLnJlcG9ydGVyLndyaXRlX3JlbWFpbmluZ19hY3Rpb25zKAogICAgICAgICAgICByZWNvcmRzLAogICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJSRU1BSU5JTkdfRVhURVJOQUxfQUNUSU9OUy5tZCIsCiAgICAgICAgKQoKICAgIGRlZiBfcmVjb3JkX2Zyb21fbWFuaWZlc3Qoc2VsZiwgbWFuaWZlc3Q6IFByb2plY3RNYW5pZmVzdCkgLT4gUHJvamVjdFJlY29yZDoKICAgICAgICB2YWxpZGF0aW9uID0gdmFsaWRhdGVfcHJvamVjdChtYW5pZmVzdCkKICAgICAgICBzdGF0dXMgPSBQcm9qZWN0U3RhdHVzLkRJU0NPVkVSRURfT05MWQogICAgICAgIGJsb2NrZWRfcmVhc29uID0gIiIKICAgICAgICBpZiBtYW5pZmVzdC5tYW5pZmVzdF9zb3VyY2UudmFsdWUgPT0gImRlY2xhcmVkIiBhbmQgdmFsaWRhdGlvbi52YWxpZDoKICAgICAgICAgICAgc3RhdHVzID0gUHJvamVjdFN0YXR1cy5TQ0lFTlRJRklDQUxMWV9SVU5OQUJMRQogICAgICAgIGlmIGFueShkYXRhc2V0LmJsb2NrZWRfYnlfdXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gbWFuaWZlc3QuZGF0YXNldHMpOgogICAgICAgICAgICBibG9ja2VkX3JlYXNvbiA9ICJPbmUgb3IgbW9yZSBkYXRhc2V0cyByZXF1aXJlIG1hbnVhbCBkb3dubG9hZCBvciBhY2NlcHRhbmNlLiIKICAgICAgICAgICAgaWYgc3RhdHVzID09IFByb2plY3RTdGF0dXMuRElTQ09WRVJFRF9PTkxZOgogICAgICAgICAgICAgICAgc3RhdHVzID0gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX0VYVEVSTkFMX0RBVEFTRVQKICAgICAgICBpZiAoCiAgICAgICAgICAgIGFueShpc3N1ZS5zZXZlcml0eSA9PSAid2FybmluZyIgZm9yIGlzc3VlIGluIHZhbGlkYXRpb24uaXNzdWVzKQogICAgICAgICAgICBhbmQgc3RhdHVzID09IFByb2plY3RTdGF0dXMuQ09NUExFVEUKICAgICAgICApOgogICAgICAgICAgICBzdGF0dXMgPSBQcm9qZWN0U3RhdHVzLlNDSUVOVElGSUNBTExZX1JVTk5BQkxFCiAgICAgICAgc3VtbWFyeSA9IG1hbmlmZXN0LmRlc2NyaXB0aW9uCiAgICAgICAgcmV0dXJuIFByb2plY3RSZWNvcmQoCiAgICAgICAgICAgIG1hbmlmZXN0PW1hbmlmZXN0LAogICAgICAgICAgICB2YWxpZGF0aW9uPXZhbGlkYXRpb24sCiAgICAgICAgICAgIHN0YXR1cz1zdGF0dXMsCiAgICAgICAgICAgIHN1bW1hcnk9c3VtbWFyeSwKICAgICAgICAgICAgYmxvY2tlZF9yZWFzb249YmxvY2tlZF9yZWFzb24sCiAgICAgICAgKQoKICAgIGRlZiBfYXR0YWNoX3J1bl9oaXN0b3J5KHNlbGYsIHJlY29yZHM6IGxpc3RbUHJvamVjdFJlY29yZF0pIC0+IGxpc3RbUHJvamVjdFJlY29yZF06CiAgICAgICAgbGF0ZXN0X2J5X3Byb2plY3Q6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgb2JqZWN0XV0gPSB7fQogICAgICAgIGZvciBoaXN0b3J5X3JvdyBpbiBzZWxmLnJ1bm5lci5ydW5fZGIuZGV0YWlsZWRfcm93cygpOgogICAgICAgICAgICBsYXRlc3RfYnlfcHJvamVjdFtzdHIoaGlzdG9yeV9yb3dbInByb2plY3RfaWQiXSldID0gaGlzdG9yeV9yb3cKICAgICAgICBmb3IgcmVjb3JkIGluIHJlY29yZHM6CiAgICAgICAgICAgIGxhdGVzdF9yb3cgPSBsYXRlc3RfYnlfcHJvamVjdC5nZXQocmVjb3JkLm1hbmlmZXN0LnByb2plY3RfaWQpCiAgICAgICAgICAgIGlmIGxhdGVzdF9yb3cgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHN0YXR1c19uYW1lID0gc3RyKGxhdGVzdF9yb3dbInN0YXR1cyJdKQogICAgICAgICAgICByZWNvcmQuc3RhdHVzID0gUHJvamVjdFN0YXR1cyhzdGF0dXNfbmFtZSkKICAgICAgICAgICAgaWYgKAogICAgICAgICAgICAgICAgcmVjb3JkLnN0YXR1cyA9PSBQcm9qZWN0U3RhdHVzLlNNT0tFX1RFU1RFRAogICAgICAgICAgICAgICAgYW5kIGFueShkYXRhc2V0LmJsb2NrZWRfYnlfdXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gcmVjb3JkLm1hbmlmZXN0LmRhdGFzZXRzKQogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgcmVjb3JkLnN0YXR1cyA9IFByb2plY3RTdGF0dXMuRlVMTF9CRU5DSE1BUktfUEVORElOR19DT01QVVRFCiAgICAgICAgICAgIHJlY29yZC5zdW1tYXJ5ID0gKAogICAgICAgICAgICAgICAgZiJMYXN0IHJ1biBge2xhdGVzdF9yb3dbJ2NvbW1hbmQnXX1gIGNvbXBsZXRlZCB3aXRoIHN0YXR1cyAiCiAgICAgICAgICAgICAgICBmImB7cmVjb3JkLnN0YXR1cy52YWx1ZX1gLiIKICAgICAgICAgICAgKQogICAgICAgICAgICBjbGFzc2lmaWNhdGlvbiA9IHN0cihsYXRlc3Rfcm93WyJjbGFzc2lmaWNhdGlvbiJdKQogICAgICAgICAgICBpZiBjbGFzc2lmaWNhdGlvbiAhPSAib2siOgogICAgICAgICAgICAgICAgcmVjb3JkLmJsb2NrZWRfcmVhc29uID0gX2Rlc2NyaWJlX3J1bl9mYWlsdXJlKAogICAgICAgICAgICAgICAgICAgIGNsYXNzaWZpY2F0aW9uLAogICAgICAgICAgICAgICAgICAgIHN0cihsYXRlc3Rfcm93LmdldCgic3Rkb3V0IiwgIiIpKSwKICAgICAgICAgICAgICAgICAgICBzdHIobGF0ZXN0X3Jvdy5nZXQoInN0ZGVyciIsICIiKSksCiAgICAgICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHJlY29yZHMKCgpkZWYgX2Rlc2NyaWJlX3J1bl9mYWlsdXJlKGNsYXNzaWZpY2F0aW9uOiBzdHIsIHN0ZG91dDogc3RyLCBzdGRlcnI6IHN0cikgLT4gc3RyOgogICAgaWYgY2xhc3NpZmljYXRpb24gPT0gIm9rIjoKICAgICAgICByZXR1cm4gIiIKCiAgICB0ZXh0ID0gIlxuIi5qb2luKHBhcnQgZm9yIHBhcnQgaW4gKHN0ZG91dCwgc3RkZXJyKSBpZiBwYXJ0KQogICAgbGluZXMgPSBbbGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHRleHQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KCiAgICBpZiBjbGFzc2lmaWNhdGlvbiA9PSAiZXh0ZXJuYWxfZGF0YXNldCI6CiAgICAgICAgcmV0dXJuICJPbmUgb3IgbW9yZSByZXF1aXJlZCBkYXRhc2V0cyBhcmUgbWlzc2luZyBvciByZXF1aXJlIG1hbnVhbCBhY2Nlc3MuIgogICAgaWYgY2xhc3NpZmljYXRpb24gPT0gImNyZWRlbnRpYWwiOgogICAgICAgIHJldHVybiAiQSByZXF1aXJlZCBjcmVkZW50aWFsIG9yIEFQSSBrZXkgaXMgbWlzc2luZy4iCiAgICBpZiBjbGFzc2lmaWNhdGlvbiA9PSAiZGVwZW5kZW5jeSI6CiAgICAgICAgcmV0dXJuICJBIHJlcXVpcmVkIG1vZHVsZSBvciBjb21tYW5kIGlzIG1pc3NpbmcgZnJvbSB0aGUgcnVudGltZSBlbnZpcm9ubWVudC4iCgogICAgcGVybWlzc2lvbl9saW5lID0gbmV4dCgKICAgICAgICAoCiAgICAgICAgICAgIGxpbmUKICAgICAgICAgICAgZm9yIGxpbmUgaW4gbGluZXMKICAgICAgICAgICAgaWYgIlBlcm1pc3Npb25FcnJvcjoiIGluIGxpbmUKICAgICAgICAgICAgb3IgIk9wZXJhdGlvbiBub3QgcGVybWl0dGVkIiBpbiBsaW5lCiAgICAgICAgICAgIG9yICJQZXJtaXNzaW9uIGRlbmllZCIgaW4gbGluZQogICAgICAgICksCiAgICAgICAgIiIsCiAgICApCiAgICBpZiBjbGFzc2lmaWNhdGlvbiA9PSAicGVybWlzc2lvbiIgYW5kIHBlcm1pc3Npb25fbGluZToKICAgICAgICByZXR1cm4gcGVybWlzc2lvbl9saW5lCgogICAgZmFpbGVkX2xpbmUgPSBuZXh0KAogICAgICAgICgKICAgICAgICAgICAgbGluZS5yZW1vdmVwcmVmaXgoIkZBSUxFRCAiKS5zdHJpcCgpCiAgICAgICAgICAgIGZvciBsaW5lIGluIHJldmVyc2VkKGxpbmVzKQogICAgICAgICAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIkZBSUxFRCAiKQogICAgICAgICksCiAgICAgICAgIiIsCiAgICApCiAgICBlcnJvcl9saW5lID0gbmV4dCgKICAgICAgICAoCiAgICAgICAgICAgIGxpbmUucmVtb3ZlcHJlZml4KCJFIikuc3RyaXAoKQogICAgICAgICAgICBmb3IgbGluZSBpbiByZXZlcnNlZChsaW5lcykKICAgICAgICAgICAgaWYgIkFzc2VydGlvbkVycm9yOiIgaW4gbGluZQogICAgICAgICAgICBvciAiUGVybWlzc2lvbkVycm9yOiIgaW4gbGluZQogICAgICAgICAgICBvciAiTW9kdWxlTm90Rm91bmRFcnJvcjoiIGluIGxpbmUKICAgICAgICAgICAgb3IgIkltcG9ydEVycm9yOiIgaW4gbGluZQogICAgICAgICAgICBvciAiRmlsZU5vdEZvdW5kRXJyb3I6IiBpbiBsaW5lCiAgICAgICAgKSwKICAgICAgICAiIiwKICAgICkKICAgIGFzc2VydF9saW5lID0gbmV4dCgKICAgICAgICAoCiAgICAgICAgICAgIGxpbmUubHN0cmlwKCI+ICIpLnN0cmlwKCkKICAgICAgICAgICAgZm9yIGxpbmUgaW4gcmV2ZXJzZWQobGluZXMpCiAgICAgICAgICAgIGlmIGxpbmUubHN0cmlwKCI+ICIpLnN0cmlwKCkuc3RhcnRzd2l0aCgiYXNzZXJ0ICIpCiAgICAgICAgKSwKICAgICAgICAiIiwKICAgICkKCiAgICBpZiBmYWlsZWRfbGluZSBhbmQgYXNzZXJ0X2xpbmU6CiAgICAgICAgcmV0dXJuIGYie2ZhaWxlZF9saW5lfToge2Fzc2VydF9saW5lfSIKICAgIGlmIGZhaWxlZF9saW5lIGFuZCBlcnJvcl9saW5lOgogICAgICAgIHJldHVybiBmIntmYWlsZWRfbGluZX06IHtlcnJvcl9saW5lfSIKICAgIGlmIGZhaWxlZF9saW5lOgogICAgICAgIHJldHVybiBmYWlsZWRfbGluZQogICAgaWYgcGVybWlzc2lvbl9saW5lOgogICAgICAgIHJldHVybiBwZXJtaXNzaW9uX2xpbmUKICAgIGlmIGVycm9yX2xpbmU6CiAgICAgICAgcmV0dXJuIGVycm9yX2xpbmUKICAgIHJldHVybiBjbGFzc2lmaWNhdGlvbgo=', 'olympus/labos/reports.py': '...', 'olympus/labos/runner.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzaGxleAppbXBvcnQgc3FsaXRlMwppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBkYXRldGltZSBpbXBvcnQgVVRDLCBkYXRldGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gb2x5bXB1cy5sYWJvcy5tYW5pZmVzdCBpbXBvcnQgUHJvamVjdE1hbmlmZXN0LCBQcm9qZWN0U3RhdHVzLCBSZXNvdXJjZVByb2ZpbGUKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIFJ1blJlc3VsdDoKICAgIHByb2plY3RfaWQ6IHN0cgogICAgY29tbWFuZDogc3RyCiAgICByZXR1cm5fY29kZTogaW50CiAgICBzdGRvdXQ6IHN0cgogICAgc3RkZXJyOiBzdHIKICAgIHN0YXR1czogUHJvamVjdFN0YXR1cwogICAgc3RhcnRlZF9hdDogc3RyCiAgICBmaW5pc2hlZF9hdDogc3RyCiAgICBwcm9maWxlOiBzdHIKICAgIGN3ZDogc3RyCiAgICBjbGFzc2lmaWNhdGlvbjogc3RyCgoKY2xhc3MgUnVuRGF0YWJhc2U6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBzZWxmLnBhdGggPSBwYXRoCiAgICAgICAgc2VsZi5wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5jb25uZWN0aW9uID0gc3FsaXRlMy5jb25uZWN0KHBhdGgpCiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBydW5zICgKICAgICAgICAgICAgICAgIGlkIElOVEVHRVIgUFJJTUFSWSBLRVkgQVVUT0lOQ1JFTUVOVCwKICAgICAgICAgICAgICAgIHByb2plY3RfaWQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGNvbW1hbmQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHByb2ZpbGUgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGN3ZCBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgcmV0dXJuX2NvZGUgSU5URUdFUiBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHN0YXR1cyBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgY2xhc3NpZmljYXRpb24gVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHN0YXJ0ZWRfYXQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGZpbmlzaGVkX2F0IFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICBzdGRvdXQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHN0ZGVyciBURVhUIE5PVCBOVUxMCiAgICAgICAgICAgICkKICAgICAgICAgICAgIiIiCiAgICAgICAgKQogICAgICAgIHNlbGYuY29ubmVjdGlvbi5jb21taXQoKQoKICAgIGRlZiBjbG9zZShzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuY29ubmVjdGlvbi5jbG9zZSgpCgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBSdW5EYXRhYmFzZToKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiBfX2V4aXRfXyhzZWxmLCAqYXJnczogb2JqZWN0KSAtPiBOb25lOgogICAgICAgIHNlbGYuY2xvc2UoKQoKICAgIGRlZiBfX2RlbF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzZWxmLmNsb3NlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgcGFzcwoKICAgIGRlZiBpbnNlcnQoc2VsZiwgcmVzdWx0OiBSdW5SZXN1bHQpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBJTlNFUlQgSU5UTyBydW5zICgKICAgICAgICAgICAgICAgIHByb2plY3RfaWQsIGNvbW1hbmQsIHByb2ZpbGUsIGN3ZCwgcmV0dXJuX2NvZGUsIHN0YXR1cywKICAgICAgICAgICAgICAgIGNsYXNzaWZpY2F0aW9uLCBzdGFydGVkX2F0LCBmaW5pc2hlZF9hdCwgc3Rkb3V0LCBzdGRlcnIKICAgICAgICAgICAgKSBWQUxVRVMgKD8sID8sID8sID8sID8sID8sID8sID8sID8sID8sID8pCiAgICAgICAgICAgICIiIiwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgcmVzdWx0LnByb2plY3RfaWQsCiAgICAgICAgICAgICAgICByZXN1bHQuY29tbWFuZCwKICAgICAgICAgICAgICAgIHJlc3VsdC5wcm9maWxlLAogICAgICAgICAgICAgICAgcmVzdWx0LmN3ZCwKICAgICAgICAgICAgICAgIHJlc3VsdC5yZXR1cm5fY29kZSwKICAgICAgICAgICAgICAgIHJlc3VsdC5zdGF0dXMudmFsdWUsCiAgICAgICAgICAgICAgICByZXN1bHQuY2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICByZXN1bHQuc3RhcnRlZF9hdCwKICAgICAgICAgICAgICAgIHJlc3VsdC5maW5pc2hlZF9hdCwKICAgICAgICAgICAgICAgIHJlc3VsdC5zdGRvdXQsCiAgICAgICAgICAgICAgICByZXN1bHQuc3RkZXJyLAogICAgICAgICAgICApLAogICAgICAgICkKICAgICAgICBzZWxmLmNvbm5lY3Rpb24uY29tbWl0KCkKCiAgICBkZWYgZmFpbGVkX3J1bnMoc2VsZikgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgogICAgICAgIHJvd3MgPSBzZWxmLmNvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVDVCBwcm9qZWN0X2lkLCBjb21tYW5kIEZST00gcnVucyBXSEVSRSByZXR1cm5fY29kZSAhPSAwIE9SREVSIEJZIGlkIERFU0MiCiAgICAgICAgKS5mZXRjaGFsbCgpCiAgICAgICAgcmV0dXJuIFsoc3RyKHJvd1swXSksIHN0cihyb3dbMV0pKSBmb3Igcm93IGluIHJvd3NdCgogICAgZGVmIHJvd3Moc2VsZikgLT4gbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV06CiAgICAgICAgY3Vyc29yID0gc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBTRUxFQ1QgcHJvamVjdF9pZCwgY29tbWFuZCwgcHJvZmlsZSwgY3dkLCByZXR1cm5fY29kZSwgc3RhdHVzLAogICAgICAgICAgICAgICAgICAgY2xhc3NpZmljYXRpb24sIHN0YXJ0ZWRfYXQsIGZpbmlzaGVkX2F0CiAgICAgICAgICAgIEZST00gcnVucyBPUkRFUiBCWSBpZAogICAgICAgICAgICAiIiIKICAgICAgICApCiAgICAgICAgY29sdW1ucyA9IFtpdGVtWzBdIGZvciBpdGVtIGluIGN1cnNvci5kZXNjcmlwdGlvbl0KICAgICAgICByZXR1cm4gW2RpY3QoemlwKGNvbHVtbnMsIHJvdywgc3RyaWN0PVRydWUpKSBmb3Igcm93IGluIGN1cnNvci5mZXRjaGFsbCgpXQoKICAgIGRlZiBkZXRhaWxlZF9yb3dzKHNlbGYpIC0+IGxpc3RbZGljdFtzdHIsIG9iamVjdF1dOgogICAgICAgIGN1cnNvciA9IHNlbGYuY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAiIiIKICAgICAgICAgICAgU0VMRUNUIHByb2plY3RfaWQsIGNvbW1hbmQsIHByb2ZpbGUsIGN3ZCwgcmV0dXJuX2NvZGUsIHN0YXR1cywKICAgICAgICAgICAgICAgICAgIGNsYXNzaWZpY2F0aW9uLCBzdGFydGVkX2F0LCBmaW5pc2hlZF9hdCwgc3Rkb3V0LCBzdGRlcnIKICAgICAgICAgICAgRlJPTSBydW5zIE9SREVSIEJZIGlkCiAgICAgICAgICAgICIiIgogICAgICAgICkKICAgICAgICBjb2x1bW5zID0gW2l0ZW1bMF0gZm9yIGl0ZW0gaW4gY3Vyc29yLmRlc2NyaXB0aW9uXQogICAgICAgIHJldHVybiBbZGljdCh6aXAoY29sdW1ucywgcm93LCBzdHJpY3Q9VHJ1ZSkpIGZvciByb3cgaW4gY3Vyc29yLmZldGNoYWxsKCldCgoKY2xhc3MgRXZlbnRMb2dnZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBzZWxmLnBhdGggPSBwYXRoCiAgICAgICAgc2VsZi5wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGxvZyhzZWxmLCBwYXlsb2FkOiBkaWN0W3N0ciwgb2JqZWN0XSkgLT4gTm9uZToKICAgICAgICB3aXRoIHNlbGYucGF0aC5vcGVuKCJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhwYXlsb2FkLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKQoKCmNsYXNzIFBvcnRmb2xpb1J1bm5lcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhcnRpZmFjdF9yb290OiBQYXRoKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXJ0aWZhY3Rfcm9vdCA9IGFydGlmYWN0X3Jvb3QKICAgICAgICBzZWxmLnJ1bl9kYiA9IFJ1bkRhdGFiYXNlKGFydGlmYWN0X3Jvb3QgLyAibGFib3NfcnVucy5zcWxpdGUzIikKICAgICAgICBzZWxmLmV2ZW50X2xvZ2dlciA9IEV2ZW50TG9nZ2VyKGFydGlmYWN0X3Jvb3QgLyAibGFib3NfZXZlbnRzLmpzb25sIikKCiAgICBkZWYgY2xvc2Uoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLnJ1bl9kYi5jbG9zZSgpCgogICAgZGVmIHJ1bl9tYW5pZmVzdCgKICAgICAgICBzZWxmLAogICAgICAgIG1hbmlmZXN0OiBQcm9qZWN0TWFuaWZlc3QsCiAgICAgICAgcHJvZmlsZTogUmVzb3VyY2VQcm9maWxlID0gUmVzb3VyY2VQcm9maWxlLlNNT0tFLAogICAgICAgIHRpbWVvdXRfc2Vjb25kczogaW50ID0gMTIwLAogICAgKSAtPiBsaXN0W1J1blJlc3VsdF06CiAgICAgICAgY29tbWFuZHMgPSBbZW50cnkgZm9yIGVudHJ5IGluIG1hbmlmZXN0LmVudHJ5X3BvaW50cyBpZiBlbnRyeS5wcm9maWxlID09IHByb2ZpbGVdCiAgICAgICAgaWYgbm90IGNvbW1hbmRzIGFuZCBwcm9maWxlID09IFJlc291cmNlUHJvZmlsZS5TTU9LRToKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgcmVzdWx0czogbGlzdFtSdW5SZXN1bHRdID0gW10KICAgICAgICBmb3IgZW50cnkgaW4gY29tbWFuZHM6CiAgICAgICAgICAgIHJlc3VsdCA9IHNlbGYucnVuX2NvbW1hbmQoCiAgICAgICAgICAgICAgICBwcm9qZWN0X2lkPW1hbmlmZXN0LnByb2plY3RfaWQsCiAgICAgICAgICAgICAgICBjb21tYW5kPWVudHJ5LmNvbW1hbmQsCiAgICAgICAgICAgICAgICBjd2Q9bWFuaWZlc3QucGF0aCwKICAgICAgICAgICAgICAgIHByb2ZpbGU9cHJvZmlsZSwKICAgICAgICAgICAgICAgIHRpbWVvdXRfc2Vjb25kcz1lbnRyeS50aW1lb3V0X3NlY29uZHMgb3IgdGltZW91dF9zZWNvbmRzLAogICAgICAgICAgICAgICAgZXh0cmFfZW52PWVudHJ5LmVudiwKICAgICAgICAgICAgKQogICAgICAgICAgICByZXN1bHRzLmFwcGVuZChyZXN1bHQpCiAgICAgICAgcmV0dXJuIHJlc3VsdHMKCiAgICBkZWYgcnVuX2NvbW1hbmQoCiAgICAgICAgc2VsZiwKICAgICAgICBwcm9qZWN0X2lkOiBzdHIsCiAgICAgICAgY29tbWFuZDogc3RyLAogICAgICAgIGN3ZDogUGF0aCwKICAgICAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUsCiAgICAgICAgdGltZW91dF9zZWNvbmRzOiBmbG9hdCA9IDEyMCwKICAgICAgICBleHRyYV9lbnY6IGRpY3Rbc3RyLCBzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IFJ1blJlc3VsdDoKICAgICAgICBzdGFydGVkX2F0ID0gZGF0ZXRpbWUubm93KFVUQykuaXNvZm9ybWF0KCkKICAgICAgICBlbnYgPSBvcy5lbnZpcm9uLmNvcHkoKQogICAgICAgIGVudlsiUFlUSE9ORE9OVFdSSVRFQllURUNPREUiXSA9ICIxIgogICAgICAgIGVudlsiUFlURVNUX0FERE9QVFMiXSA9ICItcCBubzpjYWNoZXByb3ZpZGVyIgogICAgICAgIGlmIGV4dHJhX2VudjoKICAgICAgICAgICAgZW52LnVwZGF0ZShleHRyYV9lbnYpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjb21wbGV0ZWQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIHNobGV4LnNwbGl0KGNvbW1hbmQpLAogICAgICAgICAgICAgICAgY3dkPWN3ZCwKICAgICAgICAgICAgICAgIGVudj1lbnYsCiAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICAgICAgdGltZW91dD10aW1lb3V0X3NlY29uZHMsCiAgICAgICAgICAgICAgICBjaGVjaz1GYWxzZSwKICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm5fY29kZSA9IGNvbXBsZXRlZC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHN0ZG91dCA9IGNvbXBsZXRlZC5zdGRvdXQKICAgICAgICAgICAgc3RkZXJyID0gY29tcGxldGVkLnN0ZGVycgogICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkIGFzIGV4YzoKICAgICAgICAgICAgcmV0dXJuX2NvZGUgPSAxMjQKICAgICAgICAgICAgc3Rkb3V0ID0gX2RlY29kZV90aW1lb3V0X3N0cmVhbShleGMuc3Rkb3V0KQogICAgICAgICAgICBzdGRlcnIgPSBfZGVjb2RlX3RpbWVvdXRfc3RyZWFtKGV4Yy5zdGRlcnIpCiAgICAgICAgICAgIHN0ZGVyciA9IChzdGRlcnIgKyAiXG4iIGlmIHN0ZGVyciBlbHNlICIiKSArICgKICAgICAgICAgICAgICAgIGYiQ29tbWFuZCB0aW1lZCBvdXQgYWZ0ZXIge3RpbWVvdXRfc2Vjb25kc30gc2Vjb25kcy4iCiAgICAgICAgICAgICkKICAgICAgICBmaW5pc2hlZF9hdCA9IGRhdGV0aW1lLm5vdyhVVEMpLmlzb2Zvcm1hdCgpCiAgICAgICAgY2xhc3NpZmljYXRpb24gPSBjbGFzc2lmeV9ydW4ocmV0dXJuX2NvZGUsIHN0ZGVycikKICAgICAgICBzdGF0dXMgPSBzdGF0dXNfZnJvbV9ydW4ocmV0dXJuX2NvZGUsIGNsYXNzaWZpY2F0aW9uLCBwcm9maWxlKQogICAgICAgIHJlc3VsdCA9IFJ1blJlc3VsdCgKICAgICAgICAgICAgcHJvamVjdF9pZD1wcm9qZWN0X2lkLAogICAgICAgICAgICBjb21tYW5kPWNvbW1hbmQsCiAgICAgICAgICAgIHJldHVybl9jb2RlPXJldHVybl9jb2RlLAogICAgICAgICAgICBzdGRvdXQ9c3Rkb3V0Wy0yMF8wMDA6XSwKICAgICAgICAgICAgc3RkZXJyPXN0ZGVyclstMjBfMDAwOl0sCiAgICAgICAgICAgIHN0YXR1cz1zdGF0dXMsCiAgICAgICAgICAgIHN0YXJ0ZWRfYXQ9c3RhcnRlZF9hdCwKICAgICAgICAgICAgZmluaXNoZWRfYXQ9ZmluaXNoZWRfYXQsCiAgICAgICAgICAgIHByb2ZpbGU9cHJvZmlsZS52YWx1ZSwKICAgICAgICAgICAgY3dkPXN0cihjd2QpLAogICAgICAgICAgICBjbGFzc2lmaWNhdGlvbj1jbGFzc2lmaWNhdGlvbiwKICAgICAgICApCiAgICAgICAgc2VsZi5ydW5fZGIuaW5zZXJ0KHJlc3VsdCkKICAgICAgICBzZWxmLmV2ZW50X2xvZ2dlci5sb2coCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJwcm9qZWN0X2lkIjogcHJvamVjdF9pZCwKICAgICAgICAgICAgICAgICJjb21tYW5kIjogY29tbWFuZCwKICAgICAgICAgICAgICAgICJwcm9maWxlIjogcHJvZmlsZS52YWx1ZSwKICAgICAgICAgICAgICAgICJjd2QiOiBzdHIoY3dkKSwKICAgICAgICAgICAgICAgICJyZXR1cm5fY29kZSI6IHJldHVybl9jb2RlLAogICAgICAgICAgICAgICAgImNsYXNzaWZpY2F0aW9uIjogY2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICAic3RhdHVzIjogc3RhdHVzLnZhbHVlLAogICAgICAgICAgICAgICAgInN0YXJ0ZWRfYXQiOiBzdGFydGVkX2F0LAogICAgICAgICAgICAgICAgImZpbmlzaGVkX2F0IjogZmluaXNoZWRfYXQsCiAgICAgICAgICAgICAgICAiZW52X292ZXJyaWRlcyI6IGV4dHJhX2VudiBvciB7fSwKICAgICAgICAgICAgfQogICAgICAgICkKICAgICAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGNsYXNzaWZ5X3J1bihyZXR1cm5fY29kZTogaW50LCBzdGRlcnI6IHN0cikgLT4gc3RyOgogICAgbG93ZXJlZCA9IHN0ZGVyci5sb3dlcigpCiAgICBpZiByZXR1cm5fY29kZSA9PSAwOgogICAgICAgIHJldHVybiAib2siCiAgICBpZiByZXR1cm5fY29kZSA9PSAxMjQgb3IgInRpbWVkIG91dCIgaW4gbG93ZXJlZCBvciAidGltZW91dCIgaW4gbG93ZXJlZDoKICAgICAgICByZXR1cm4gInRpbWVvdXQiCiAgICBpZiAicGVybWlzc2lvbiBkZW5pZWQiIGluIGxvd2VyZWQgb3IgIm9wZXJhdGlvbiBub3QgcGVybWl0dGVkIiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAicGVybWlzc2lvbiIKICAgIGlmICJubyBtb2R1bGUgbmFtZWQiIGluIGxvd2VyZWQgb3IgImNvbW1hbmQgbm90IGZvdW5kIiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAiZGVwZW5kZW5jeSIKICAgIGlmICJub3QgZm91bmQiIGluIGxvd2VyZWQgYW5kICJkYXRhc2V0IiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAiZXh0ZXJuYWxfZGF0YXNldCIKICAgIGlmICJjcmVkZW50aWFsIiBpbiBsb3dlcmVkIG9yICJhcGkga2V5IiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAiY3JlZGVudGlhbCIKICAgIHJldHVybiAiZmFpbGVkX2NvbW1hbmQiCgoKZGVmIHN0YXR1c19mcm9tX3J1bigKICAgIHJldHVybl9jb2RlOiBpbnQsCiAgICBjbGFzc2lmaWNhdGlvbjogc3RyLAogICAgcHJvZmlsZTogUmVzb3VyY2VQcm9maWxlLAopIC0+IFByb2plY3RTdGF0dXM6CiAgICBpZiByZXR1cm5fY29kZSA9PSAwIGFuZCBwcm9maWxlID09IFJlc291cmNlUHJvZmlsZS5TTU9LRToKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5TTU9LRV9URVNURUQKICAgIGlmIHJldHVybl9jb2RlID09IDA6CiAgICAgICAgcmV0dXJuIFByb2plY3RTdGF0dXMuU0NJRU5USUZJQ0FMTFlfUlVOTkFCTEUKICAgIGlmIGNsYXNzaWZpY2F0aW9uID09ICJleHRlcm5hbF9kYXRhc2V0IjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX0VYVEVSTkFMX0RBVEFTRVQKICAgIGlmIGNsYXNzaWZpY2F0aW9uID09ICJjcmVkZW50aWFsIjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX0NSRURFTlRJQUwKICAgIGlmIGNsYXNzaWZpY2F0aW9uID09ICJwZXJtaXNzaW9uIjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX1BFUk1JU1NJT04KICAgIGlmIGNsYXNzaWZpY2F0aW9uID09ICJ0aW1lb3V0IjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5GVUxMX0JFTkNITUFSS19QRU5ESU5HX0NPTVBVVEUKICAgIHJldHVybiBQcm9qZWN0U3RhdHVzLkRJU0NPVkVSRURfT05MWQoKCmRlZiBfZGVjb2RlX3RpbWVvdXRfc3RyZWFtKHZhbHVlOiBzdHIgfCBieXRlcyB8IE5vbmUpIC0+IHN0cjoKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICIiCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBieXRlcyk6CiAgICAgICAgcmV0dXJuIHZhbHVlLmRlY29kZShlcnJvcnM9InJlcGxhY2UiKQogICAgcmV0dXJuIHZhbHVlCg==', 'olympus/labos/scheduler.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0IFByb2plY3RNYW5pZmVzdCwgUmVzb3VyY2VQcm9maWxlCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBTY2hlZHVsZWRUYXNrOgogICAgcHJvamVjdF9pZDogc3RyCiAgICBjb21tYW5kOiBzdHIKICAgIHByaW9yaXR5OiBpbnQKICAgIHByb2ZpbGU6IFJlc291cmNlUHJvZmlsZQoKCmNsYXNzIFJlc291cmNlQXdhcmVTY2hlZHVsZXI6CiAgICBkZWYgc2NoZWR1bGUoCiAgICAgICAgc2VsZiwKICAgICAgICBtYW5pZmVzdHM6IGxpc3RbUHJvamVjdE1hbmlmZXN0XSwKICAgICAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUsCiAgICApIC0+IGxpc3RbU2NoZWR1bGVkVGFza106CiAgICAgICAgdGFza3M6IGxpc3RbU2NoZWR1bGVkVGFza10gPSBbXQogICAgICAgIGZvciBtYW5pZmVzdCBpbiBtYW5pZmVzdHM6CiAgICAgICAgICAgIGZvciBlbnRyeSBpbiBtYW5pZmVzdC5lbnRyeV9wb2ludHM6CiAgICAgICAgICAgICAgICBpZiBlbnRyeS5wcm9maWxlICE9IHByb2ZpbGU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHByaW9yaXR5ID0gMAogICAgICAgICAgICAgICAgaWYgbWFuaWZlc3QubWFuaWZlc3Rfc291cmNlLnZhbHVlID09ICJkZWNsYXJlZCI6CiAgICAgICAgICAgICAgICAgICAgcHJpb3JpdHkgKz0gMgogICAgICAgICAgICAgICAgaWYgbWFuaWZlc3QucHJvamVjdF9pZCA9PSAib2x5bXB1cyI6CiAgICAgICAgICAgICAgICAgICAgcHJpb3JpdHkgKz0gMwogICAgICAgICAgICAgICAgaWYgbWFuaWZlc3QucmVzb3VyY2VfcmVxdWlyZW1lbnRzLmdwdV9vcHRpb25hbDoKICAgICAgICAgICAgICAgICAgICBwcmlvcml0eSAtPSAxCiAgICAgICAgICAgICAgICB0YXNrcy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgU2NoZWR1bGVkVGFzaygKICAgICAgICAgICAgICAgICAgICAgICAgcHJvamVjdF9pZD1tYW5pZmVzdC5wcm9qZWN0X2lkLAogICAgICAgICAgICAgICAgICAgICAgICBjb21tYW5kPWVudHJ5LmNvbW1hbmQsCiAgICAgICAgICAgICAgICAgICAgICAgIHByaW9yaXR5PXByaW9yaXR5LAogICAgICAgICAgICAgICAgICAgICAgICBwcm9maWxlPXByb2ZpbGUsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBzb3J0ZWQodGFza3MsIGtleT1sYW1iZGEgaXRlbTogaXRlbS5wcmlvcml0eSwgcmV2ZXJzZT1UcnVlKQo=', 'olympus/labos/validation.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0IFByb2plY3RNYW5pZmVzdCwgVmFsaWRhdGlvbklzc3VlLCBWYWxpZGF0aW9uUmVzdWx0CgoKZGVmIHZhbGlkYXRlX3Byb2plY3QobWFuaWZlc3Q6IFByb2plY3RNYW5pZmVzdCkgLT4gVmFsaWRhdGlvblJlc3VsdDoKICAgIGlzc3VlczogbGlzdFtWYWxpZGF0aW9uSXNzdWVdID0gW10KICAgIHByb2plY3Rfcm9vdCA9IFBhdGgobWFuaWZlc3Qucm9vdF9wYXRoKQogICAgaWYgbm90IHByb2plY3Rfcm9vdC5leGlzdHMoKToKICAgICAgICBpc3N1ZXMuYXBwZW5kKFZhbGlkYXRpb25Jc3N1ZShzZXZlcml0eT0iZXJyb3IiLCBtZXNzYWdlPSJQcm9qZWN0IHJvb3QgZG9lcyBub3QgZXhpc3QuIikpCiAgICBpZiBub3QgX2hhc19yZWFkbWUocHJvamVjdF9yb290KToKICAgICAgICBpc3N1ZXMuYXBwZW5kKFZhbGlkYXRpb25Jc3N1ZShzZXZlcml0eT0id2FybmluZyIsIG1lc3NhZ2U9IlJFQURNRSBpcyBtaXNzaW5nLiIpKQogICAgaWYgbm90IG1hbmlmZXN0LmVudHJ5X3BvaW50czoKICAgICAgICBpc3N1ZXMuYXBwZW5kKAogICAgICAgICAgICBWYWxpZGF0aW9uSXNzdWUoCiAgICAgICAgICAgICAgICBzZXZlcml0eT0id2FybmluZyIsCiAgICAgICAgICAgICAgICBtZXNzYWdlPSJObyBydW5uYWJsZSBlbnRyeSBwb2ludHMgZGV0ZWN0ZWQuIiwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIGlmIG5vdCBtYW5pZmVzdC52YWxpZGF0aW9uX2NvbW1hbmRzOgogICAgICAgIGlzc3Vlcy5hcHBlbmQoCiAgICAgICAgICAgIFZhbGlkYXRpb25Jc3N1ZSgKICAgICAgICAgICAgICAgIHNldmVyaXR5PSJ3YXJuaW5nIiwKICAgICAgICAgICAgICAgIG1lc3NhZ2U9Ik5vIHZhbGlkYXRpb24gY29tbWFuZHMgZGV0ZWN0ZWQuIiwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIGZvciBkYXRhc2V0IGluIG1hbmlmZXN0LmRhdGFzZXRzOgogICAgICAgIGlmIGRhdGFzZXQuYmxvY2tlZF9ieV91c2VyX2FjdGlvbjoKICAgICAgICAgICAgaXNzdWVzLmFwcGVuZCgKICAgICAgICAgICAgICAgIFZhbGlkYXRpb25Jc3N1ZSgKICAgICAgICAgICAgICAgICAgICBzZXZlcml0eT0iaW5mbyIsCiAgICAgICAgICAgICAgICAgICAgbWVzc2FnZT1mIkRhdGFzZXQgJ3tkYXRhc2V0Lm5hbWV9JyBtYXkgcmVxdWlyZSBtYW51YWwgZG93bmxvYWQgb3IgYWNjZXB0YW5jZS4iLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICByZXR1cm4gVmFsaWRhdGlvblJlc3VsdCgKICAgICAgICBwcm9qZWN0X2lkPW1hbmlmZXN0LnByb2plY3RfaWQsCiAgICAgICAgdmFsaWQ9bm90IGFueShpc3N1ZS5zZXZlcml0eSA9PSAiZXJyb3IiIGZvciBpc3N1ZSBpbiBpc3N1ZXMpLAogICAgICAgIGlzc3Vlcz1pc3N1ZXMsCiAgICApCgoKZGVmIF9oYXNfcmVhZG1lKHByb2plY3Rfcm9vdDogUGF0aCkgLT4gYm9vbDoKICAgIHJldHVybiBhbnkoKHByb2plY3Rfcm9vdCAvIG5hbWUpLmV4aXN0cygpIGZvciBuYW1lIGluICgiUkVBRE1FLm1kIiwgIlJFQURNRS5NRCIsICJyZWFkbWUubWQiKSkK', 'olympus/memory/__init__.py': 'IiIiTWVtb3J5IHN5c3RlbS4iIiIK', 'olympus/memory/store.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHNxbGl0ZTMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGhyZWFkaW5nIGltcG9ydCBSTG9jawoKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5jb3JlLnNxbGl0ZV9qb3VybmFsIGltcG9ydCBhcHBseV9kdXJhYmxlX2pvdXJuYWxfbW9kZQoKCmNsYXNzIE1lbW9yeVJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBraW5kOiBzdHIKICAgIGtleTogc3RyCiAgICB2YWx1ZTogc3RyCiAgICBzYWxpZW5jZTogZmxvYXQgPSAwLjUKICAgIHRhZ3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoKCmNsYXNzIE1lbW9yeVN0b3JlOgogICAgIiIiU2VyaWFsaXplIGluZGl2aWR1YWwgcHVibGljIG9wZXJhdGlvbnMgb24gdGhlIHNoYXJlZCBTUUxpdGUgY29ubmVjdGlvbi4KCiAgICBEaXJlY3QgYWNjZXNzIHRvIGBgY29ubmVjdGlvbmBgIG11c3QgYmUgZXh0ZXJuYWxseSBzeW5jaHJvbml6ZWQuIFRoaXMgbG9jawogICAgZG9lcyBub3QgbWFrZSBhIHNlcXVlbmNlIG9mIG1ldGhvZCBjYWxscyBvbmUgdHJhbnNhY3Rpb24gb3IgY29vcmRpbmF0ZQogICAgc2VwYXJhdGUgc3RvcmUgaW5zdGFuY2VzL3Byb2Nlc3NlczsgU1FMaXRlIGhhbmRsZXMgdGhlaXIgZGF0YWJhc2UgbG9ja2luZy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIHNlbGYuX2xvY2sgPSBSTG9jaygpCiAgICAgICAgc2VsZi5wYXRoID0gcGF0aAogICAgICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxmLmNvbm5lY3Rpb24gPSBzcWxpdGUzLmNvbm5lY3QocGF0aCwgdGltZW91dD0zMC4wLCBjaGVja19zYW1lX3RocmVhZD1GYWxzZSkKICAgICAgICBhcHBseV9kdXJhYmxlX2pvdXJuYWxfbW9kZShzZWxmLmNvbm5lY3Rpb24pCiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoIlBSQUdNQSBzeW5jaHJvbm91cyA9IEZVTEwiKQogICAgICAgIHNlbGYuY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAiIiIKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgbWVtb3J5ICgKICAgICAgICAgICAgICAgIGtpbmQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGtleSBURVhUIFBSSU1BUlkgS0VZLAogICAgICAgICAgICAgICAgdmFsdWUgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHNhbGllbmNlIFJFQUwgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICB0YWdzIFRFWFQgTk9UIE5VTEwKICAgICAgICAgICAgKQogICAgICAgICAgICAiIiIKICAgICAgICApCiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmNvbW1pdCgpCgogICAgZGVmIGNsb3NlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLmNvbm5lY3Rpb24uY2xvc2UoKQoKICAgIGRlZiBfX2VudGVyX18oc2VsZikgLT4gTWVtb3J5U3RvcmU6CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmFyZ3M6IG9iamVjdCkgLT4gTm9uZToKICAgICAgICBzZWxmLmNsb3NlKCkKCiAgICBkZWYgcHV0KHNlbGYsIHJlY29yZDogTWVtb3J5UmVjb3JkKSAtPiBOb25lOgogICAgICAgICMgSG9sZCB0aGUgbG9jayB0aHJvdWdoIGNvbW1pdC9yb2xsYmFjaywgbm90IGp1c3Qgc3RhdGVtZW50IGV4ZWN1dGlvbi4KICAgICAgICB3aXRoIHNlbGYuX2xvY2ssIHNlbGYuY29ubmVjdGlvbjoKICAgICAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiUkVQTEFDRSBJTlRPIG1lbW9yeSAoa2luZCwga2V5LCB2YWx1ZSwgc2FsaWVuY2UsIHRhZ3MpIFZBTFVFUyAoPywgPywgPywgPywgPykiLAogICAgICAgICAgICAgICAgKHJlY29yZC5raW5kLCByZWNvcmQua2V5LCByZWNvcmQudmFsdWUsIHJlY29yZC5zYWxpZW5jZSwgIiwiLmpvaW4ocmVjb3JkLnRhZ3MpKSwKICAgICAgICAgICAgKQoKICAgIGRlZiBnZXQoc2VsZiwga2V5OiBzdHIpIC0+IE1lbW9yeVJlY29yZCB8IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByb3cgPSBzZWxmLmNvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgICAgICJTRUxFQ1Qga2luZCwga2V5LCB2YWx1ZSwgc2FsaWVuY2UsIHRhZ3MgRlJPTSBtZW1vcnkgV0hFUkUga2V5ID0gPyIsCiAgICAgICAgICAgICAgICAoa2V5LCksCiAgICAgICAgICAgICkuZmV0Y2hvbmUoKQogICAgICAgIGlmIHJvdyBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHJldHVybiBNZW1vcnlSZWNvcmQoCiAgICAgICAgICAgIGtpbmQ9cm93WzBdLAogICAgICAgICAgICBrZXk9cm93WzFdLAogICAgICAgICAgICB2YWx1ZT1yb3dbMl0sCiAgICAgICAgICAgIHNhbGllbmNlPXJvd1szXSwKICAgICAgICAgICAgdGFncz1baXRlbSBmb3IgaXRlbSBpbiByb3dbNF0uc3BsaXQoIiwiKSBpZiBpdGVtXSwKICAgICAgICApCgogICAgZGVmIHF1ZXJ5KHNlbGYsIGtpbmQ6IHN0cikgLT4gbGlzdFtNZW1vcnlSZWNvcmRdOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgcm93cyA9IHNlbGYuY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgICAgICJTRUxFQ1Qga2luZCwga2V5LCB2YWx1ZSwgc2FsaWVuY2UsIHRhZ3MgRlJPTSBtZW1vcnkgIgogICAgICAgICAgICAgICAgICAgICJXSEVSRSBraW5kID0gPyBPUkRFUiBCWSBzYWxpZW5jZSBERVNDIgogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgIChraW5kLCksCiAgICAgICAgICAgICkuZmV0Y2hhbGwoKQogICAgICAgIHJldHVybiBbCiAgICAgICAgICAgIE1lbW9yeVJlY29yZCgKICAgICAgICAgICAgICAgIGtpbmQ9cm93WzBdLAogICAgICAgICAgICAgICAga2V5PXJvd1sxXSwKICAgICAgICAgICAgICAgIHZhbHVlPXJvd1syXSwKICAgICAgICAgICAgICAgIHNhbGllbmNlPXJvd1szXSwKICAgICAgICAgICAgICAgIHRhZ3M9W2l0ZW0gZm9yIGl0ZW0gaW4gcm93WzRdLnNwbGl0KCIsIikgaWYgaXRlbV0sCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIHJvdyBpbiByb3dzCiAgICAgICAgXQo=', 'olympus/models/__init__.py': 'IiIiTGF6aWx5IGV4cG9ydGVkIG1vZGVsIGltcGxlbWVudGF0aW9uczsgbWV0YWRhdGEgaW1wb3J0cyBuZXZlciBsb2FkIFRvcmNoLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBpbXBvcnRsaWIgaW1wb3J0IGltcG9ydF9tb2R1bGUKZnJvbSB0eXBpbmcgaW1wb3J0IFRZUEVfQ0hFQ0tJTkcsIEFueQoKaWYgVFlQRV9DSEVDS0lORzoKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMuY2FwYWNpdHkgaW1wb3J0IENhcGFjaXR5RXN0aW1hdGUsIGVzdGltYXRlX2NhcGFjaXR5CiAgICBmcm9tIG9seW1wdXMubW9kZWxzLmVkaXRjbGV2ciBpbXBvcnQgKAogICAgICAgIEVESVRfRkFDVE9SUywKICAgICAgICBGVVNJT05fTU9ERVMsCiAgICAgICAgRWRpdENMRVZST3V0cHV0LAogICAgICAgIE9seW1wdXNFZGl0Q0xFVlJNb2RlbCwKICAgICAgICBlZGl0Y2xldnJfbG9zcywKICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAgaW1wb3J0ICgKICAgICAgICBBaW9uRmxhZ3NoaXBNb2RlbCwKICAgICAgICBBdGxhc0ZsYWdzaGlwTW9kZWwsCiAgICAgICAgSGVybWVzRmxhZ3NoaXBNb2RlbCwKICAgICAgICBLcm9ub3NGbGFnc2hpcE1vZGVsLAogICAgICAgIE9seW1wdXNGbGFnc2hpcFN5c3RlbSwKICAgICAgICBQZXJzZXVzRmxhZ3NoaXBNb2RlbCwKICAgICAgICBQcm9tZXRoZXVzRmxhZ3NoaXBNb2RlbCwKICAgICAgICBkZWZhdWx0X2ZsYWdzaGlwX2NvbmZpZywKICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgKICAgICAgICBGQU1JTFlfVkVDVE9SX05BTUVTLAogICAgICAgIE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICAgICAgT2x5bXB1c1N5c3RlbUNvbmZpZywKICAgICAgICBPbHltcHVzU3lzdGVtTG9zcywKICAgICAgICBPbHltcHVzU3lzdGVtT3V0cHV0LAogICAgICAgIGJ1aWxkX29seW1wdXNfc3lzdGVtLAogICAgICAgIG9seW1wdXNfc3lzdGVtX2xvc3MsCiAgICApCiAgICBmcm9tIG9seW1wdXMubW9kZWxzLnBhcmFtZXRlcl9ncm93dGggaW1wb3J0ICgKICAgICAgICBHcm93dGhSZWNlaXB0LAogICAgICAgIEpFUEFQYXJhbWV0ZXJHcm93dGgsCiAgICAgICAgd2lkZW5fZmVlZGZvcndhcmQsCiAgICApCiAgICBmcm9tIG9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyBpbXBvcnQgKAogICAgICAgIEFpb25SZWZlcmVuY2VDb25maWcsCiAgICAgICAgQWlvblJlZmVyZW5jZU1vZGVsLAogICAgICAgIEF0bGFzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIEF0bGFzUmVmZXJlbmNlTW9kZWwsCiAgICAgICAgSGVybWVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIEhlcm1lc1JlZmVyZW5jZU1vZGVsLAogICAgICAgIEtyb25vc1JlZmVyZW5jZUNvbmZpZywKICAgICAgICBLcm9ub3NSZWZlcmVuY2VNb2RlbCwKICAgICAgICBQZXJzZXVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIFBlcnNldXNSZWZlcmVuY2VNb2RlbCwKICAgICAgICBQcm9tZXRoZXVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIFByb21ldGhldXNSZWZlcmVuY2VNb2RlbCwKICAgICAgICBSZWZlcmVuY2VUcmFuc2Zvcm1lckNvbmZpZywKICAgICAgICBidWlsZF9yZWZlcmVuY2VfbW9kZWwsCiAgICAgICAgZGVmYXVsdF9yZWZlcmVuY2VfY29uZmlncywKICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMucmVsaWFiaWxpdHlfZnVzaW9uIGltcG9ydCAoCiAgICAgICAgUmVsaWFiaWxpdHlBd2FyZU9seW1wdXNTeXN0ZW0sCiAgICAgICAgUmVsaWFiaWxpdHlGdXNpb25Db25maWcsCiAgICAgICAgUmVsaWFiaWxpdHlGdXNpb25Mb3NzLAogICAgICAgIFJlbGlhYmlsaXR5RnVzaW9uT3V0cHV0LAogICAgICAgIFJlbGlhYmlsaXR5TG9zc0NvbmZpZywKICAgICAgICByZWxpYWJpbGl0eV9mdXNpb25fbG9zcywKICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMucnVudGltZSBpbXBvcnQgKAogICAgICAgIE9seW1wdXNDaGVja3BvaW50UmVjb3JkLAogICAgICAgIE9seW1wdXNJbmZlcmVuY2VSZXN1bHQsCiAgICAgICAgT2x5bXB1c1J1bnRpbWVDb25maWcsCiAgICAgICAgT2x5bXB1c1N5c3RlbVJ1bnRpbWUsCiAgICAgICAgT2x5bXB1c1RyYWluU3RlcFJlc3VsdCwKICAgICAgICBsb2FkX29seW1wdXNfY2hlY2twb2ludCwKICAgICAgICBzYXZlX29seW1wdXNfY2hlY2twb2ludCwKICAgICAgICB0cmFpbl9vbHltcHVzX3N0ZXAsCiAgICApCiAgICBmcm9tIG9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IGltcG9ydCAoCiAgICAgICAgU0VWRU5fTU9ERUxfSURTLAogICAgICAgIFNldmVuTW9kZWxFbmdpbmVlcmluZ1N0YXR1cywKICAgICAgICBTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVzLAogICAgICAgIGF1ZGl0X3NldmVuX21vZGVsX2NvbnRyYWN0cywKICAgICAgICBidWlsZF9zZXZlbl9tb2RlbF9yZWZlcmVuY2UsCiAgICAgICAgZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzLAogICAgICAgIGluc3BlY3Rfc2V2ZW5fbW9kZWxfcmVmZXJlbmNlcywKICAgICkKCl9FWFBPUlRfTU9EVUxFUyA9IHsKICAgICJFRElUX0ZBQ1RPUlMiOiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJGVVNJT05fTU9ERVMiOiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJFZGl0Q0xFVlJPdXRwdXQiOiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJPbHltcHVzRWRpdENMRVZSTW9kZWwiOiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJlZGl0Y2xldnJfbG9zcyI6ICJvbHltcHVzLm1vZGVscy5lZGl0Y2xldnIiLAogICAgIkZBTUlMWV9WRUNUT1JfTkFNRVMiOiAib2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNSZWZlcmVuY2VTeXN0ZW0iOiAib2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNTeXN0ZW1Db25maWciOiAib2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNTeXN0ZW1Mb3NzIjogIm9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJPbHltcHVzU3lzdGVtT3V0cHV0IjogIm9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJvbHltcHVzX3N5c3RlbV9sb3NzIjogIm9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJBaW9uUmVmZXJlbmNlQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiQWlvblJlZmVyZW5jZU1vZGVsIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiQXRsYXNSZWZlcmVuY2VDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJBdGxhc1JlZmVyZW5jZU1vZGVsIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiSGVybWVzUmVmZXJlbmNlQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiSGVybWVzUmVmZXJlbmNlTW9kZWwiOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJLcm9ub3NSZWZlcmVuY2VDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJLcm9ub3NSZWZlcmVuY2VNb2RlbCI6ICJvbHltcHVzLm1vZGVscy5yZWZlcmVuY2VfZmFtaWxpZXMiLAogICAgIlBlcnNldXNSZWZlcmVuY2VDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJQZXJzZXVzUmVmZXJlbmNlTW9kZWwiOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJQcm9tZXRoZXVzUmVmZXJlbmNlQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiUHJvbWV0aGV1c1JlZmVyZW5jZU1vZGVsIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJidWlsZF9yZWZlcmVuY2VfbW9kZWwiOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJkZWZhdWx0X3JlZmVyZW5jZV9jb25maWdzIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiUmVsaWFiaWxpdHlBd2FyZU9seW1wdXNTeXN0ZW0iOiAib2x5bXB1cy5tb2RlbHMucmVsaWFiaWxpdHlfZnVzaW9uIiwKICAgICJSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZyI6ICJvbHltcHVzLm1vZGVscy5yZWxpYWJpbGl0eV9mdXNpb24iLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uTG9zcyI6ICJvbHltcHVzLm1vZGVscy5yZWxpYWJpbGl0eV9mdXNpb24iLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uT3V0cHV0IjogIm9seW1wdXMubW9kZWxzLnJlbGlhYmlsaXR5X2Z1c2lvbiIsCiAgICAiUmVsaWFiaWxpdHlMb3NzQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlbGlhYmlsaXR5X2Z1c2lvbiIsCiAgICAicmVsaWFiaWxpdHlfZnVzaW9uX2xvc3MiOiAib2x5bXB1cy5tb2RlbHMucmVsaWFiaWxpdHlfZnVzaW9uIiwKICAgICJPbHltcHVzQ2hlY2twb2ludFJlY29yZCI6ICJvbHltcHVzLm1vZGVscy5ydW50aW1lIiwKICAgICJPbHltcHVzSW5mZXJlbmNlUmVzdWx0IjogIm9seW1wdXMubW9kZWxzLnJ1bnRpbWUiLAogICAgIk9seW1wdXNSdW50aW1lQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJ1bnRpbWUiLAogICAgIk9seW1wdXNTeXN0ZW1SdW50aW1lIjogIm9seW1wdXMubW9kZWxzLnJ1bnRpbWUiLAogICAgIk9seW1wdXNUcmFpblN0ZXBSZXN1bHQiOiAib2x5bXB1cy5tb2RlbHMucnVudGltZSIsCiAgICAibG9hZF9vbHltcHVzX2NoZWNrcG9pbnQiOiAib2x5bXB1cy5tb2RlbHMucnVudGltZSIsCiAgICAic2F2ZV9vbHltcHVzX2NoZWNrcG9pbnQiOiAib2x5bXB1cy5tb2RlbHMucnVudGltZSIsCiAgICAidHJhaW5fb2x5bXB1c19zdGVwIjogIm9seW1wdXMubW9kZWxzLnJ1bnRpbWUiLAogICAgIlNFVkVOX01PREVMX0lEUyI6ICJvbHltcHVzLm1vZGVscy5zZXZlbl9tb2RlbF9jb250cmFjdCIsCiAgICAiU2V2ZW5Nb2RlbEVuZ2luZWVyaW5nU3RhdHVzIjogIm9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVzIjogIm9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJhdWRpdF9zZXZlbl9tb2RlbF9jb250cmFjdHMiOiAib2x5bXB1cy5tb2RlbHMuc2V2ZW5fbW9kZWxfY29udHJhY3QiLAogICAgImJ1aWxkX3NldmVuX21vZGVsX3JlZmVyZW5jZSI6ICJvbHltcHVzLm1vZGVscy5zZXZlbl9tb2RlbF9jb250cmFjdCIsCiAgICAiZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzIjogIm9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJpbnNwZWN0X3NldmVuX21vZGVsX3JlZmVyZW5jZXMiOiAib2x5bXB1cy5tb2RlbHMuc2V2ZW5fbW9kZWxfY29udHJhY3QiLAogICAgIkNhcGFjaXR5RXN0aW1hdGUiOiAib2x5bXB1cy5tb2RlbHMuY2FwYWNpdHkiLAogICAgImVzdGltYXRlX2NhcGFjaXR5IjogIm9seW1wdXMubW9kZWxzLmNhcGFjaXR5IiwKICAgICJBaW9uRmxhZ3NoaXBNb2RlbCI6ICJvbHltcHVzLm1vZGVscy5mbGFnc2hpcCIsCiAgICAiQXRsYXNGbGFnc2hpcE1vZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJIZXJtZXNGbGFnc2hpcE1vZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJLcm9ub3NGbGFnc2hpcE1vZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJPbHltcHVzRmxhZ3NoaXBTeXN0ZW0iOiAib2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAiLAogICAgIlBlcnNldXNGbGFnc2hpcE1vZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJQcm9tZXRoZXVzRmxhZ3NoaXBNb2RlbCI6ICJvbHltcHVzLm1vZGVscy5mbGFnc2hpcCIsCiAgICAiZGVmYXVsdF9mbGFnc2hpcF9jb25maWciOiAib2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAiLAogICAgIkpFUEFQYXJhbWV0ZXJHcm93dGgiOiAib2x5bXB1cy5tb2RlbHMucGFyYW1ldGVyX2dyb3d0aCIsCiAgICAiR3Jvd3RoUmVjZWlwdCI6ICJvbHltcHVzLm1vZGVscy5wYXJhbWV0ZXJfZ3Jvd3RoIiwKICAgICJ3aWRlbl9mZWVkZm9yd2FyZCI6ICJvbHltcHVzLm1vZGVscy5wYXJhbWV0ZXJfZ3Jvd3RoIiwKICAgICJidWlsZF9vbHltcHVzX3N5c3RlbSI6ICJvbHltcHVzLm1vZGVscy5vbHltcHVzX3N5c3RlbSIsCn0KCl9fYWxsX18gPSBbCiAgICAiRURJVF9GQUNUT1JTIiwKICAgICJGVVNJT05fTU9ERVMiLAogICAgIkVkaXRDTEVWUk91dHB1dCIsCiAgICAiRkFNSUxZX1ZFQ1RPUl9OQU1FUyIsCiAgICAiT2x5bXB1c1JlZmVyZW5jZVN5c3RlbSIsCiAgICAiT2x5bXB1c1N5c3RlbUNvbmZpZyIsCiAgICAiT2x5bXB1c1N5c3RlbUxvc3MiLAogICAgIk9seW1wdXNTeXN0ZW1PdXRwdXQiLAogICAgIk9seW1wdXNDaGVja3BvaW50UmVjb3JkIiwKICAgICJPbHltcHVzSW5mZXJlbmNlUmVzdWx0IiwKICAgICJPbHltcHVzRWRpdENMRVZSTW9kZWwiLAogICAgIk9seW1wdXNSdW50aW1lQ29uZmlnIiwKICAgICJPbHltcHVzU3lzdGVtUnVudGltZSIsCiAgICAiT2x5bXB1c1RyYWluU3RlcFJlc3VsdCIsCiAgICAiUmVsaWFiaWxpdHlBd2FyZU9seW1wdXNTeXN0ZW0iLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uQ29uZmlnIiwKICAgICJSZWxpYWJpbGl0eUZ1c2lvbkxvc3MiLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uT3V0cHV0IiwKICAgICJSZWxpYWJpbGl0eUxvc3NDb25maWciLAogICAgIlNFVkVOX01PREVMX0lEUyIsCiAgICAiU2V2ZW5Nb2RlbEVuZ2luZWVyaW5nU3RhdHVzIiwKICAgICJTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVzIiwKICAgICJBaW9uUmVmZXJlbmNlQ29uZmlnIiwKICAgICJBaW9uUmVmZXJlbmNlTW9kZWwiLAogICAgIkF0bGFzUmVmZXJlbmNlQ29uZmlnIiwKICAgICJBdGxhc1JlZmVyZW5jZU1vZGVsIiwKICAgICJIZXJtZXNSZWZlcmVuY2VDb25maWciLAogICAgIkhlcm1lc1JlZmVyZW5jZU1vZGVsIiwKICAgICJLcm9ub3NSZWZlcmVuY2VDb25maWciLAogICAgIktyb25vc1JlZmVyZW5jZU1vZGVsIiwKICAgICJQZXJzZXVzUmVmZXJlbmNlQ29uZmlnIiwKICAgICJQZXJzZXVzUmVmZXJlbmNlTW9kZWwiLAogICAgIlByb21ldGhldXNSZWZlcmVuY2VDb25maWciLAogICAgIlByb21ldGhldXNSZWZlcmVuY2VNb2RlbCIsCiAgICAiUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25maWciLAogICAgImJ1aWxkX3JlZmVyZW5jZV9tb2RlbCIsCiAgICAiYnVpbGRfc2V2ZW5fbW9kZWxfcmVmZXJlbmNlIiwKICAgICJhdWRpdF9zZXZlbl9tb2RlbF9jb250cmFjdHMiLAogICAgImRlZmF1bHRfcmVmZXJlbmNlX2NvbmZpZ3MiLAogICAgImRlZmF1bHRfc2V2ZW5fbW9kZWxfY29uZmlncyIsCiAgICAiaW5zcGVjdF9zZXZlbl9tb2RlbF9yZWZlcmVuY2VzIiwKICAgICJlZGl0Y2xldnJfbG9zcyIsCiAgICAibG9hZF9vbHltcHVzX2NoZWNrcG9pbnQiLAogICAgIm9seW1wdXNfc3lzdGVtX2xvc3MiLAogICAgInJlbGlhYmlsaXR5X2Z1c2lvbl9sb3NzIiwKICAgICJzYXZlX29seW1wdXNfY2hlY2twb2ludCIsCiAgICAidHJhaW5fb2x5bXB1c19zdGVwIiwKICAgICJDYXBhY2l0eUVzdGltYXRlIiwKICAgICJlc3RpbWF0ZV9jYXBhY2l0eSIsCiAgICAiQWlvbkZsYWdzaGlwTW9kZWwiLAogICAgIkF0bGFzRmxhZ3NoaXBNb2RlbCIsCiAgICAiSGVybWVzRmxhZ3NoaXBNb2RlbCIsCiAgICAiS3Jvbm9zRmxhZ3NoaXBNb2RlbCIsCiAgICAiT2x5bXB1c0ZsYWdzaGlwU3lzdGVtIiwKICAgICJQZXJzZXVzRmxhZ3NoaXBNb2RlbCIsCiAgICAiUHJvbWV0aGV1c0ZsYWdzaGlwTW9kZWwiLAogICAgImRlZmF1bHRfZmxhZ3NoaXBfY29uZmlnIiwKICAgICJKRVBBUGFyYW1ldGVyR3Jvd3RoIiwKICAgICJHcm93dGhSZWNlaXB0IiwKICAgICJ3aWRlbl9mZWVkZm9yd2FyZCIsCiAgICAiYnVpbGRfb2x5bXB1c19zeXN0ZW0iLApdCgoKZGVmIF9fZ2V0YXR0cl9fKG5hbWU6IHN0cikgLT4gQW55OgogICAgbW9kdWxlID0gX0VYUE9SVF9NT0RVTEVTLmdldChuYW1lKQogICAgaWYgbW9kdWxlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgQXR0cmlidXRlRXJyb3IoZiJtb2R1bGUge19fbmFtZV9fIXJ9IGhhcyBubyBhdHRyaWJ1dGUge25hbWUhcn0iKQogICAgdmFsdWUgPSBnZXRhdHRyKGltcG9ydF9tb2R1bGUobW9kdWxlKSwgbmFtZSkKICAgIGdsb2JhbHMoKVtuYW1lXSA9IHZhbHVlCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgX19kaXJfXygpIC0+IGxpc3Rbc3RyXToKICAgIHJldHVybiBzb3J0ZWQoc2V0KGdsb2JhbHMoKSkgfCBzZXQoX19hbGxfXykpCg==', 'olympus/models/capacity.py': 'IiIiQWxsb2NhdGlvbi1mcmVlIHBhcmFtZXRlciBhY2NvdW50aW5nOyBzcGFyc2UgYWN0aXZhdGlvbiBuZXZlciByZW1vdmVzIGJhY2tpbmcgd2VpZ2h0cy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ2FwYWNpdHlFc3RpbWF0ZToKICAgIHRvdGFsX3BhcmFtZXRlcnM6IGludAogICAgYWN0aXZlX3BhcmFtZXRlcnM6IGludAogICAgcmVzaWRlbnRfcGFyYW1ldGVyczogaW50CiAgICB3ZWlnaHRfYml0czogaW50CiAgICBiYWNraW5nX3dlaWdodF9ieXRlczogaW50CiAgICByZXNpZGVudF93ZWlnaHRfYnl0ZXM6IGludAogICAgYWN0aXZlX3dlaWdodF9ieXRlczogaW50CiAgICB0cmFpbmFibGVfc3RhdGVfYnl0ZXM6IGludAogICAgYWN0aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlczogaW50CiAgICByZXF1aXJlZF9kZXZpY2VfYnl0ZXM6IGludAogICAgZGV2aWNlX2J1ZGdldF9ieXRlczogaW50CiAgICBmaXRzX2RlY2xhcmVkX2RldmljZV9idWRnZXQ6IGJvb2wKICAgIGJhY2tpbmdfc3RvcmFnZV9hdmFpbGFibGU6IGJvb2wKICAgIHJ1bm5hYmxlOiBib29sCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIGludCB8IGJvb2xdOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpkZWYgZXN0aW1hdGVfY2FwYWNpdHkoCiAgICAqLAogICAgdG90YWxfcGFyYW1ldGVyczogaW50LAogICAgYWN0aXZlX3BhcmFtZXRlcnM6IGludCwKICAgIHJlc2lkZW50X3BhcmFtZXRlcnM6IGludCwKICAgIGRldmljZV9idWRnZXRfYnl0ZXM6IGludCwKICAgIGJhY2tpbmdfc3RvcmFnZV9ieXRlczogaW50LAogICAgd2VpZ2h0X2JpdHM6IGludCA9IDE2LAogICAgdHJhaW5hYmxlX3BhcmFtZXRlcnM6IGludCA9IDAsCiAgICB0cmFpbmFibGVfYnl0ZXNfcGVyX3BhcmFtZXRlcjogaW50ID0gMTYsCiAgICBhY3RpdmF0aW9uX2FuZF9rdl9yZXNlcnZlX2J5dGVzOiBpbnQgPSAwLAopIC0+IENhcGFjaXR5RXN0aW1hdGU6CiAgICAiIiJDb25zZXJ2YXRpdmUgZGVjbGFyZWQgd2VpZ2h0L3N0YXRlIGJ1ZGdldCwgd2l0aG91dCBhbGxvY2F0aW5nIGFueSB0ZW5zb3IuCgogICAgVHJhaW5pbmcgc3RhdGUgYW5kIHJlc2VydmUgbXVzdCBiZSBzcGVjaWZpZWQgYnkgdGhlIGNhbGxpbmcgd29ya2xvYWQuIFF1YW50aXplcgogICAgbWV0YWRhdGEsIHRyYW5zZmVycywgY29sbGVjdGl2ZSBidWZmZXJzIGFuZCBhbGxvY2F0b3Igb3ZlcmhlYWQgbXVzdCBiZSBpbmNsdWRlZAogICAgaW4gdGhhdCByZXNlcnZlOyB0aGlzIGNhbGN1bGF0aW9uIGlzIGEgbmVjZXNzYXJ5IGFkbWlzc2lvbiB0ZXN0LCBub3QgYSBiZW5jaG1hcmsuCiAgICAiIiIKICAgIHZhbHVlcyA9ICgKICAgICAgICB0b3RhbF9wYXJhbWV0ZXJzLAogICAgICAgIGFjdGl2ZV9wYXJhbWV0ZXJzLAogICAgICAgIHJlc2lkZW50X3BhcmFtZXRlcnMsCiAgICAgICAgZGV2aWNlX2J1ZGdldF9ieXRlcywKICAgICAgICBiYWNraW5nX3N0b3JhZ2VfYnl0ZXMsCiAgICAgICAgdHJhaW5hYmxlX3BhcmFtZXRlcnMsCiAgICAgICAgdHJhaW5hYmxlX2J5dGVzX3Blcl9wYXJhbWV0ZXIsCiAgICAgICAgYWN0aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlcywKICAgICkKICAgIGlmIGFueSh0eXBlKHZhbHVlKSBpcyBub3QgaW50IG9yIHZhbHVlIDwgMCBmb3IgdmFsdWUgaW4gdmFsdWVzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjYXBhY2l0eSBjb3VudHMgYW5kIGJ1ZGdldHMgbXVzdCBiZSBub25uZWdhdGl2ZSBpbnRlZ2VycyIpCiAgICBpZiBub3QgMCA8IGFjdGl2ZV9wYXJhbWV0ZXJzIDw9IHJlc2lkZW50X3BhcmFtZXRlcnMgPD0gdG90YWxfcGFyYW1ldGVyczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyZXF1aXJlIDAgPCBhY3RpdmUgPD0gcmVzaWRlbnQgPD0gdG90YWwgcGFyYW1ldGVycyIpCiAgICBpZiB3ZWlnaHRfYml0cyBub3QgaW4gezQsIDgsIDE2LCAzMn0gb3IgdHlwZSh3ZWlnaHRfYml0cykgaXMgbm90IGludDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ3ZWlnaHQgYml0cyBtdXN0IGJlIDQsIDgsIDE2LCBvciAzMiIpCiAgICBpZiB0cmFpbmFibGVfcGFyYW1ldGVycyA+IHJlc2lkZW50X3BhcmFtZXRlcnMgb3IgdHJhaW5hYmxlX2J5dGVzX3Blcl9wYXJhbWV0ZXIgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluYWJsZSBwYXJhbWV0ZXJzIG11c3QgZml0IHJlc2lkZW5jeSB3aXRoIHBvc2l0aXZlIHN0YXRlIGFjY291bnRpbmciKQogICAgYmFja2luZyA9ICh0b3RhbF9wYXJhbWV0ZXJzICogd2VpZ2h0X2JpdHMgKyA3KSAvLyA4CiAgICByZXNpZGVudCA9IChyZXNpZGVudF9wYXJhbWV0ZXJzICogd2VpZ2h0X2JpdHMgKyA3KSAvLyA4CiAgICBhY3RpdmUgPSAoYWN0aXZlX3BhcmFtZXRlcnMgKiB3ZWlnaHRfYml0cyArIDcpIC8vIDgKICAgIHRyYWluaW5nID0gdHJhaW5hYmxlX3BhcmFtZXRlcnMgKiB0cmFpbmFibGVfYnl0ZXNfcGVyX3BhcmFtZXRlcgogICAgcmVxdWlyZWQgPSByZXNpZGVudCArIHRyYWluaW5nICsgYWN0aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlcwogICAgZml0cyA9IHJlcXVpcmVkIDw9IGRldmljZV9idWRnZXRfYnl0ZXMKICAgIHN0b3JlZCA9IGJhY2tpbmcgPD0gYmFja2luZ19zdG9yYWdlX2J5dGVzCiAgICByZXR1cm4gQ2FwYWNpdHlFc3RpbWF0ZSgKICAgICAgICB0b3RhbF9wYXJhbWV0ZXJzLAogICAgICAgIGFjdGl2ZV9wYXJhbWV0ZXJzLAogICAgICAgIHJlc2lkZW50X3BhcmFtZXRlcnMsCiAgICAgICAgd2VpZ2h0X2JpdHMsCiAgICAgICAgYmFja2luZywKICAgICAgICByZXNpZGVudCwKICAgICAgICBhY3RpdmUsCiAgICAgICAgdHJhaW5pbmcsCiAgICAgICAgYWN0aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlcywKICAgICAgICByZXF1aXJlZCwKICAgICAgICBkZXZpY2VfYnVkZ2V0X2J5dGVzLAogICAgICAgIGZpdHMsCiAgICAgICAgc3RvcmVkLAogICAgICAgIGZpdHMgYW5kIHN0b3JlZCwKICAgICkK', 'olympus/models/conservative_routing_v6.py': 'IiIiRGlzam9pbnQtc3RhY2tlZCByb3V0aW5nIHdpdGggZXhwbGljaXQgcmVmZXJlbmNlIGludGVydmVudGlvbiAodjYpLgoKVGhpcyBpcyBhIGJvdW5kZWQgcmVmZXJlbmNlIG1vZGVsLCBub3QgYSBzYWZldHkgb3IgaW1wcm92ZW1lbnQgZ3VhcmFudGVlLgpHcm91cCBpZGVudGlmaWVycyBhcmUgdXNlZCBvbmx5IGJ5IHRoZSB0cmFpbmluZyBvYmplY3RpdmUsIG5ldmVyIGluZmVyZW5jZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29weQoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCgpkZWYgdmFsaWRhdGVfcHJvYmFiaWxpdGllcyhwOiBUZW5zb3IpIC0+IE5vbmU6CiAgICBpZiBwLm5kaW0gPCAyIG9yIHAuc2hhcGVbMF0gPT0gMCBvciBwLnNoYXBlWy0xXSA8IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicHJvYmFiaWxpdGllcyByZXF1aXJlIG5vbmVtcHR5IGV4YW1wbGVzIGFuZCA+PTIgY2xhc3NlcyIpCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkgb3IgYm9vbCgocCA8IDApLmFueSgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9iYWJpbGl0aWVzIG11c3QgYmUgZmluaXRlIGFuZCBub25uZWdhdGl2ZSIpCiAgICBpZiBub3QgdG9yY2guYWxsY2xvc2UocC5zdW0oLTEpLCB0b3JjaC5vbmVzX2xpa2UocFsuLi4sIDBdKSwgYXRvbD0xZS01LCBydG9sPTApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByb2JhYmlsaXRpZXMgbXVzdCBzdW0gdG8gb25lIikKCgpkZWYgYmxlbmQocmVmZXJlbmNlOiBUZW5zb3IsIGFsdGVybmF0aXZlOiBUZW5zb3IsIGFscGhhOiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgIHZhbGlkYXRlX3Byb2JhYmlsaXRpZXMocmVmZXJlbmNlKQogICAgdmFsaWRhdGVfcHJvYmFiaWxpdGllcyhhbHRlcm5hdGl2ZSkKICAgIGlmIHJlZmVyZW5jZS5zaGFwZSAhPSBhbHRlcm5hdGl2ZS5zaGFwZSBvciBhbHBoYS5zaGFwZSAhPSByZWZlcmVuY2Uuc2hhcGVbOi0xXSArICgxLCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVmZXJlbmNlL2FsdGVybmF0aXZlL2FscGhhIHNoYXBlcyBkaXNhZ3JlZSIpCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShhbHBoYSkuYWxsKCkpIG9yIGJvb2woKChhbHBoYSA8IDApIHwgKGFscGhhID4gMSkpLmFueSgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhbHBoYSBtdXN0IGJlIGZpbml0ZSBpbiBbMCwxXSIpCiAgICByZXR1cm4gKDEgLSBhbHBoYSkgKiByZWZlcmVuY2UgKyBhbHBoYSAqIGFsdGVybmF0aXZlCgoKZGVmIGZlYXR1cmVzKHo6IFRlbnNvciwgcHJvYmFiaWxpdGllczogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBpZiB6Lm5kaW0gIT0gMyBvciB6LnNoYXBlWzE6XSAhPSAoNSwgMTYpIG9yIHByb2JhYmlsaXRpZXMuc2hhcGUgIT0gKGxlbih6KSwgNSwgNCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXhwZWN0ZWQgW04sNSwxNl0gc3RhdGVzIGFuZCBbTiw1LDRdIHByb2JhYmlsaXRpZXMiKQogICAgdmFsaWRhdGVfcHJvYmFiaWxpdGllcyhwcm9iYWJpbGl0aWVzKQogICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoeikuYWxsKCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vbmZpbml0ZSBleHBlcnQgc3RhdGVzIikKICAgIGVudHJvcHkgPSAtKHByb2JhYmlsaXRpZXMgKiBwcm9iYWJpbGl0aWVzLmNsYW1wX21pbigxZS04KS5sb2coKSkuc3VtKC0xKQogICAgcmV0dXJuIHRvcmNoLmNhdCgoei5mbGF0dGVuKDEpLCBwcm9iYWJpbGl0aWVzLmZsYXR0ZW4oMSksIGVudHJvcHkpLCAtMSkuZGV0YWNoKCkKCgpkZWYgcm91dGluZ19uZXR3b3JrKCkgLT4gbm4uU2VxdWVudGlhbDoKICAgIHJldHVybiBubi5TZXF1ZW50aWFsKG5uLkxheWVyTm9ybSgxMDUpLCBubi5MaW5lYXIoMTA1LCAzMiksIG5uLkdFTFUoKSwgbm4uTGluZWFyKDMyLCA1KSkKCgpjbGFzcyBSZWZlcmVuY2VSb3V0ZXMobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuc2VtYW50aWMgPSByb3V0aW5nX25ldHdvcmsoKQogICAgICAgIHNlbGYuZml4ZWQgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoNSkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogVGVuc29yLCBleHBlcnRzOiBUZW5zb3IpIC0+IGRpY3Rbc3RyLCBUZW5zb3JdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICoqe2YiZXhwZXJ0X3tpfSI6IGV4cGVydHNbOiwgaV0gZm9yIGkgaW4gcmFuZ2UoNSl9LAogICAgICAgICAgICAidW5pZm9ybSI6IGV4cGVydHMubWVhbigxKSwKICAgICAgICAgICAgImZpeGVkIjogKHNlbGYuZml4ZWQuc29mdG1heCgtMSlbTm9uZSwgOiwgTm9uZV0gKiBleHBlcnRzKS5zdW0oMSksCiAgICAgICAgICAgICJzZW1hbnRpYyI6IChzZWxmLnNlbWFudGljKHgpLnNvZnRtYXgoLTEpWy4uLiwgTm9uZV0gKiBleHBlcnRzKS5zdW0oMSksCiAgICAgICAgfQoKCmNsYXNzIENvcnJlY3Rpb24obm4uTW9kdWxlKToKICAgICIiIlNoYXJlZCBhbHRlcm5hdGl2ZSBpbml0aWFsaXphdGlvbjsgb25seSB0aGUgZGVjbGFyZWQgY29ycmVjdGlvbiBkaWZmZXJzLgoKICAgIERpcmVjdCB2YXJpYW50cyBoYXZlIG5vIGludGVydmVudGlvbiBwYXJhbWV0ZXJzLiBDb25zdGFudCB2YXJpYW50cyBoYXZlIG9uZTsKICAgIGFkYXB0aXZlIHZhcmlhbnRzIGhhdmUgMTA2LiBDb3VudHMgYXJlIHJlcG9ydGVkIHJhdGhlciB0aGFuIHBhZGRlZCB3aXRoIGRlYWQKICAgIHBhcmFtZXRlcnMuIEEgY29uc3RhbnQgYmxlbmQgaXMgYSB0cmFpbmVkIGlucHV0LWluZGVwZW5kZW50IGFscGhhIGNvbnRyb2wuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5pdGlhbDogbm4uU2VxdWVudGlhbCwga2luZDogc3RyLCBhbHBoYV9pbml0aWFsOiBmbG9hdCA9IDAuMSkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBraW5kIG5vdCBpbiB7ImRpcmVjdCIsICJhbmNob3JlZCIsICJjb25zdGFudCJ9IG9yIG5vdCAwIDwgYWxwaGFfaW5pdGlhbCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgY29ycmVjdGlvbiBraW5kIG9yIGFscGhhIGluaXRpYWxpemF0aW9uIikKICAgICAgICBzZWxmLmtpbmQgPSBraW5kCiAgICAgICAgc2VsZi5hbHRlcm5hdGl2ZSA9IGNvcHkuZGVlcGNvcHkoaW5pdGlhbCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICBpZiBraW5kID09ICJhbmNob3JlZCI6CiAgICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50aW9uOiBubi5Nb2R1bGUgPSBubi5MaW5lYXIoMTA1LCAxKQogICAgICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmludGVydmVudGlvbi53ZWlnaHQpCiAgICAgICAgICAgIG5uLmluaXQuY29uc3RhbnRfKAogICAgICAgICAgICAgICAgc2VsZi5pbnRlcnZlbnRpb24uYmlhcywgZmxvYXQodG9yY2gubG9naXQodG9yY2gudGVuc29yKGFscGhhX2luaXRpYWwpKSkKICAgICAgICAgICAgKQogICAgICAgIGVsaWYga2luZCA9PSAiY29uc3RhbnQiOgogICAgICAgICAgICBzZWxmLmludGVydmVudGlvbiA9IG5uLkxpbmVhcigxLCAxLCBiaWFzPUZhbHNlKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XygKICAgICAgICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50aW9uLndlaWdodCwgZmxvYXQodG9yY2gubG9naXQodG9yY2gudGVuc29yKGFscGhhX2luaXRpYWwpKSkKICAgICAgICAgICAgKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50aW9uID0gbm4uSWRlbnRpdHkoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IFRlbnNvciwgZXhwZXJ0czogVGVuc29yLCByZWZlcmVuY2U6IFRlbnNvcikgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3JdOgogICAgICAgIGFsdGVybmF0aXZlID0gKHNlbGYuYWx0ZXJuYXRpdmUoeCkuc29mdG1heCgtMSlbLi4uLCBOb25lXSAqIGV4cGVydHMpLnN1bSgxKQogICAgICAgIGlmIHNlbGYua2luZCA9PSAiZGlyZWN0IjoKICAgICAgICAgICAgYWxwaGEgPSB0b3JjaC5vbmVzX2xpa2UocmVmZXJlbmNlWzosIDoxXSkKICAgICAgICBlbGlmIHNlbGYua2luZCA9PSAiY29uc3RhbnQiOgogICAgICAgICAgICBhbHBoYSA9IHNlbGYuaW50ZXJ2ZW50aW9uKHRvcmNoLm9uZXNfbGlrZShyZWZlcmVuY2VbOiwgOjFdKSkuc2lnbW9pZCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYWxwaGEgPSBzZWxmLmludGVydmVudGlvbih4KS5zaWdtb2lkKCkKICAgICAgICByZXR1cm4gYmxlbmQocmVmZXJlbmNlLCBhbHRlcm5hdGl2ZSwgYWxwaGEpLCBhbHBoYQoKCmRlZiBjb3JyZWN0aW9uX2xvc3MoCiAgICBwcmVkaWN0aW9uczogVGVuc29yLAogICAgbGFiZWxzOiBUZW5zb3IsCiAgICBncm91cHM6IFRlbnNvciwKICAgIGFscGhhOiBUZW5zb3IsCiAgICAqLAogICAgcm9idXN0OiBib29sLAogICAgcGVuYWx0eTogZmxvYXQsCiAgICBncm91cF9jb3VudDogaW50ID0gMywKKSAtPiBUZW5zb3I6CiAgICB2YWxpZGF0ZV9wcm9iYWJpbGl0aWVzKHByZWRpY3Rpb25zKQogICAgaWYgbGFiZWxzLnNoYXBlICE9IChsZW4ocHJlZGljdGlvbnMpLCkgb3IgZ3JvdXBzLnNoYXBlICE9IGxhYmVscy5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYWJlbHMvZ3JvdXBzIG11c3QgaGF2ZSBvbmUgZW50cnkgcGVyIHByZWRpY3Rpb24iKQogICAgaWYgbGFiZWxzLmR0eXBlICE9IHRvcmNoLmxvbmcgb3IgZ3JvdXBzLmR0eXBlICE9IHRvcmNoLmxvbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW50ZWdlciBsYWJlbHMvZ3JvdXBzIHJlcXVpcmVkIikKICAgIGlmIGJvb2woKChsYWJlbHMgPCAwKSB8IChsYWJlbHMgPj0gcHJlZGljdGlvbnMuc2hhcGVbLTFdKSkuYW55KCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNsYXNzIG91dCBvZiByYW5nZSIpCiAgICBpZiBncm91cF9jb3VudCA8IDEgb3IgYm9vbCgoKGdyb3VwcyA8IDApIHwgKGdyb3VwcyA+PSBncm91cF9jb3VudCkpLmFueSgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm91cCBvdXQgb2YgcmFuZ2UiKQogICAgaWYgcGVuYWx0eSA8IDAgb3Igbm90IF9faW1wb3J0X18oIm1hdGgiKS5pc2Zpbml0ZShwZW5hbHR5KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGFuY2hvciBwZW5hbHR5IikKICAgIGlmIGFscGhhLnNoYXBlICE9IChsZW4ocHJlZGljdGlvbnMpLCAxKSBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShhbHBoYSkuYWxsKCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgYWxwaGEiKQogICAgaWYgYm9vbCgoKGFscGhhIDwgMCkgfCAoYWxwaGEgPiAxKSkuYW55KCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFscGhhIG91dCBvZiByYW5nZSIpCiAgICBsb3NzZXMgPSAtcHJlZGljdGlvbnMuZ2F0aGVyKDEsIGxhYmVsc1s6LCBOb25lXSkuY2xhbXBfbWluKDFlLTgpLmxvZygpLnNxdWVlemUoMSkKICAgIHBlcl9ncm91cCA9IFtdCiAgICBmb3IgZ3JvdXAgaW4gcmFuZ2UoZ3JvdXBfY291bnQpOgogICAgICAgIG1hc2sgPSBncm91cHMgPT0gZ3JvdXAKICAgICAgICBpZiBub3QgYm9vbChtYXNrLmFueSgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXZlcnkgZGVjbGFyZWQgZ3JvdXAgbXVzdCBiZSBwcmVzZW50IikKICAgICAgICBwZXJfZ3JvdXAuYXBwZW5kKGxvc3Nlc1ttYXNrXS5tZWFuKCkpCiAgICB2YWx1ZXMgPSB0b3JjaC5zdGFjayhwZXJfZ3JvdXApCiAgICBvYmplY3RpdmUgPSB2YWx1ZXMubWF4KCkgaWYgcm9idXN0IGVsc2UgdmFsdWVzLm1lYW4oKQogICAgcmV0dXJuIG9iamVjdGl2ZSArIHBlbmFsdHkgKiBhbHBoYS5tZWFuKCkKCgpkZWYgaW50ZXJ2ZW50aW9uX2RpYWdub3N0aWNzKHA6IFRlbnNvciwgcmVmZXJlbmNlOiBUZW5zb3IsIHk6IFRlbnNvcikgLT4gZGljdFtzdHIsIGludCB8IGZsb2F0XToKICAgIGdvb2QsIHJlZl9nb29kID0gcC5hcmdtYXgoLTEpLmVxKHkpLCByZWZlcmVuY2UuYXJnbWF4KC0xKS5lcSh5KQogICAgcmVzY3VlcyA9IGludCgoZ29vZCAmIH5yZWZfZ29vZCkuc3VtKCkpCiAgICBoYXJtcyA9IGludCgofmdvb2QgJiByZWZfZ29vZCkuc3VtKCkpCiAgICByZXR1cm4gewogICAgICAgICJleGFtcGxlcyI6IGxlbih5KSwKICAgICAgICAicmVzY3VlcyI6IHJlc2N1ZXMsCiAgICAgICAgImhhcm1zIjogaGFybXMsCiAgICAgICAgImFjY3VyYWN5X2RlbHRhIjogKHJlc2N1ZXMgLSBoYXJtcykgLyBsZW4oeSksCiAgICAgICAgInByZWRpY3Rpb25fY2hhbmdlcyI6IGludChwLmFyZ21heCgtMSkubmUocmVmZXJlbmNlLmFyZ21heCgtMSkpLnN1bSgpKSwKICAgIH0K', 'olympus/models/editclevr.py': 'IiIiUGFpcmVkLWltYWdlIE9seW1wdXMgbW9kZWwgZm9yIHRoZSBFZGl0Q0xFVlIgaW50ZXJ2ZW50aW9uIGJlbmNobWFyay4KClRoaXMgbW9kZWwgaXMgaW50ZW50aW9uYWxseSBuYXJyb3c6IGl0IHByZWRpY3RzIHdoaWNoIG9iamVjdCBmYWN0b3IgY2hhbmdlZCBpbgphIGJlZm9yZS9hZnRlciBpbWFnZSBwYWlyLiBJdCBkb2VzIG5vdCBmYWJyaWNhdGUgbGFuZ3VhZ2UsIHRlbXBvcmFsLCByaXNrLCBvcgpnb3Zlcm5hbmNlIGlucHV0cyB0aGF0IEVkaXRDTEVWUiBkb2VzIG5vdCBjb250YWluLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCkVESVRfRkFDVE9SUyA9ICgibm9uZSIsICJjb2xvciIsICJtYXRlcmlhbCIsICJzaXplIiwgInNoYXBlIikKRlVTSU9OX01PREVTID0gKCJsZWFybmVkIiwgInVuaWZvcm0iLCAicGFpcl9vbmx5IiwgImRlbHRhX29ubHkiKQoKCmNsYXNzIF9JbWFnZUVuY29kZXIobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB3aWR0aDogaW50KSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubmV0d29yayA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZCgzLCB3aWR0aCwgNSwgc3RyaWRlPTIsIHBhZGRpbmc9MiksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybSg0LCB3aWR0aCksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uQ29udjJkKHdpZHRoLCB3aWR0aCAqIDIsIDMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5Hcm91cE5vcm0oOCwgd2lkdGggKiAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Db252MmQod2lkdGggKiAyLCB3aWR0aCAqIDQsIDMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5Hcm91cE5vcm0oOCwgd2lkdGggKiA0KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Db252MmQod2lkdGggKiA0LCB3aWR0aCAqIDQsIDMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5Hcm91cE5vcm0oOCwgd2lkdGggKiA0KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgxKSwKICAgICAgICAgICAgbm4uRmxhdHRlbigpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZTogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgcmV0dXJuIGNhc3QoVGVuc29yLCBzZWxmLm5ldHdvcmsoaW1hZ2UpKQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgRWRpdENMRVZST3V0cHV0OgogICAgcHJvYmFiaWxpdGllczogZGljdFtzdHIsIFRlbnNvcl0KICAgIGdhdGVzOiBUZW5zb3IKICAgIGV4cGVydF9sb2dpdHM6IFRlbnNvcgogICAgcGFpcl9lbWJlZGRpbmc6IFRlbnNvcgoKCmNsYXNzIE9seW1wdXNFZGl0Q0xFVlJNb2RlbChubi5Nb2R1bGUpOgogICAgIiIiU2hhcmVkIGVuY29kZXJzIHBsdXMgZXhwbGljaXQgZXhwZXJ0cyBhbmQgYW4gb2JzZXJ2YWJsZSByb3V0ZXIuCgogICAgVGhlIGRpcmVjdCBwYWlyIGV4cGVydCBpcyBhIHN0cm9uZyBjb250cm9sLiBUaGUgbGVhcm5lZCByb3V0ZXIgbXVzdCBpbXByb3ZlCiAgICBvbiBpdCB1bmRlciBoZWxkLW91dCBoYXJkIGFuZCBDb0dlblQgc2hpZnRzOyBiZWF0aW5nIGEgZGVsaWJlcmF0ZWx5IHdlYWsKICAgIHVuaWZvcm0gYXZlcmFnZSBpcyBpbnN1ZmZpY2llbnQgZm9yIHRoZSByZWdpc3RlcmVkIHByaW1hcnkuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgKiwgd2lkdGg6IGludCA9IDMyLCBlbWJlZGRpbmdfZGltOiBpbnQgPSAxMjgpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgd2lkdGggPCA4IG9yIHdpZHRoICUgNDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGggbXVzdCBiZSBhdCBsZWFzdCA4IGFuZCBkaXZpc2libGUgYnkgNCIpCiAgICAgICAgaWYgZW1iZWRkaW5nX2RpbSA8IDMyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbWJlZGRpbmdfZGltIG11c3QgYmUgYXQgbGVhc3QgMzIiKQogICAgICAgIGVuY29kZWQgPSB3aWR0aCAqIDQKICAgICAgICBzZWxmLmltYWdlX2VuY29kZXIgPSBfSW1hZ2VFbmNvZGVyKHdpZHRoKQogICAgICAgIHNlbGYuZGVsdGFfZW5jb2RlciA9IF9JbWFnZUVuY29kZXIod2lkdGgpCiAgICAgICAgc2VsZi5wcm9qZWN0ID0gbm4uU2VxdWVudGlhbChubi5MYXllck5vcm0oZW5jb2RlZCksIG5uLkxpbmVhcihlbmNvZGVkLCBlbWJlZGRpbmdfZGltKSkKICAgICAgICBzZWxmLmV4cGVydF9oZWFkcyA9IG5uLk1vZHVsZUxpc3QoCiAgICAgICAgICAgIG5uLkxpbmVhcihlbWJlZGRpbmdfZGltLCBsZW4oRURJVF9GQUNUT1JTKSkgZm9yIF8gaW4gcmFuZ2UoNCkKICAgICAgICApCiAgICAgICAgcGFpcl93aWR0aCA9IGVtYmVkZGluZ19kaW0gKiA0CiAgICAgICAgc2VsZi5wYWlyX25ldHdvcmsgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MYXllck5vcm0ocGFpcl93aWR0aCksCiAgICAgICAgICAgIG5uLkxpbmVhcihwYWlyX3dpZHRoLCBlbWJlZGRpbmdfZGltICogMiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uRHJvcG91dCgwLjEpLAogICAgICAgICAgICBubi5MaW5lYXIoZW1iZWRkaW5nX2RpbSAqIDIsIGVtYmVkZGluZ19kaW0pLAogICAgICAgICkKICAgICAgICByb3V0ZXJfd2lkdGggPSBlbWJlZGRpbmdfZGltICogNCArIGxlbihFRElUX0ZBQ1RPUlMpICogNCArIDQKICAgICAgICBzZWxmLnJvdXRlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShyb3V0ZXJfd2lkdGgpLAogICAgICAgICAgICBubi5MaW5lYXIocm91dGVyX3dpZHRoLCBlbWJlZGRpbmdfZGltKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoZW1iZWRkaW5nX2RpbSwgNCksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGJlZm9yZTogVGVuc29yLCBhZnRlcjogVGVuc29yKSAtPiBFZGl0Q0xFVlJPdXRwdXQ6CiAgICAgICAgaWYgYmVmb3JlLnNoYXBlICE9IGFmdGVyLnNoYXBlIG9yIGJlZm9yZS5uZGltICE9IDQgb3IgYmVmb3JlLnNoYXBlWzFdICE9IDM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJlZm9yZSBhbmQgYWZ0ZXIgbXVzdCBzaGFyZSBzaGFwZSBbYmF0Y2gsIDMsIGhlaWdodCwgd2lkdGhdIikKICAgICAgICBpZiBiZWZvcmUuc2hhcGVbLTJdIDwgMzIgb3IgYmVmb3JlLnNoYXBlWy0xXSA8IDMyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbWFnZXMgbXVzdCBiZSBhdCBsZWFzdCAzMiBieSAzMiIpCiAgICAgICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoYmVmb3JlKS5hbGwoKSBhbmQgdG9yY2guaXNmaW5pdGUoYWZ0ZXIpLmFsbCgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW1hZ2VzIGNvbnRhaW4gbm9uLWZpbml0ZSB2YWx1ZXMiKQoKICAgICAgICBiZWZvcmVfeiA9IHNlbGYucHJvamVjdChzZWxmLmltYWdlX2VuY29kZXIoYmVmb3JlKSkKICAgICAgICBhZnRlcl96ID0gc2VsZi5wcm9qZWN0KHNlbGYuaW1hZ2VfZW5jb2RlcihhZnRlcikpCiAgICAgICAgIyBJbnB1dHMgYXJlIG5vcm1hbGl6ZWQgdG8gWy0xLCAxXS4gS2VlcCBib3RoIGRlcml2ZWQgdmlld3MgaW4gYQogICAgICAgICMgYm91bmRlZCByYW5nZSBzbyB0aGVpciBlbmNvZGVyIHJlY2VpdmVzIHRoZSBzYW1lIHNjYWxlIGFzIFJHQi4KICAgICAgICBzaWduZWRfeiA9IHNlbGYucHJvamVjdChzZWxmLmRlbHRhX2VuY29kZXIoKGFmdGVyIC0gYmVmb3JlKSAvIDIuMCkpCiAgICAgICAgYWJzb2x1dGVfeiA9IHNlbGYucHJvamVjdChzZWxmLmRlbHRhX2VuY29kZXIoKGFmdGVyIC0gYmVmb3JlKS5hYnMoKSAvIDIuMCkpCiAgICAgICAgcGFpcl96ID0gc2VsZi5wYWlyX25ldHdvcmsodG9yY2guY2F0KChiZWZvcmVfeiwgYWZ0ZXJfeiwgc2lnbmVkX3osIGFic29sdXRlX3opLCBkaW09LTEpKQogICAgICAgIGVtYmVkZGluZ3MgPSB0b3JjaC5zdGFjaygoYmVmb3JlX3osIGFmdGVyX3osIGFic29sdXRlX3osIHBhaXJfeiksIGRpbT0xKQogICAgICAgIGxvZ2l0cyA9IHRvcmNoLnN0YWNrKAogICAgICAgICAgICBbaGVhZChlbWJlZGRpbmdzWzosIGluZGV4XSkgZm9yIGluZGV4LCBoZWFkIGluIGVudW1lcmF0ZShzZWxmLmV4cGVydF9oZWFkcyldLCBkaW09MQogICAgICAgICkKICAgICAgICBleHBlcnRfcHJvYmFiaWxpdGllcyA9IGxvZ2l0cy5zb2Z0bWF4KGRpbT0tMSkKICAgICAgICBlbnRyb3B5ID0gLSgKICAgICAgICAgICAgZXhwZXJ0X3Byb2JhYmlsaXRpZXMgKiBleHBlcnRfcHJvYmFiaWxpdGllcy5jbGFtcF9taW4oMWUtOCkubG9nKCkKICAgICAgICApLnN1bShkaW09LTEpCiAgICAgICAgZmVhdHVyZXMgPSB0b3JjaC5jYXQoCiAgICAgICAgICAgIChlbWJlZGRpbmdzLmZsYXR0ZW4oMSksIGV4cGVydF9wcm9iYWJpbGl0aWVzLmZsYXR0ZW4oMSksIGVudHJvcHkpLCBkaW09LTEKICAgICAgICApCiAgICAgICAgZ2F0ZXMgPSBzZWxmLnJvdXRlcihmZWF0dXJlcykuc29mdG1heChkaW09LTEpCiAgICAgICAgcHJvYmFiaWxpdGllcyA9IHsKICAgICAgICAgICAgImxlYXJuZWQiOiAoZ2F0ZXNbLi4uLCBOb25lXSAqIGV4cGVydF9wcm9iYWJpbGl0aWVzKS5zdW0oZGltPTEpLAogICAgICAgICAgICAidW5pZm9ybSI6IGV4cGVydF9wcm9iYWJpbGl0aWVzLm1lYW4oZGltPTEpLAogICAgICAgICAgICAicGFpcl9vbmx5IjogZXhwZXJ0X3Byb2JhYmlsaXRpZXNbOiwgM10sCiAgICAgICAgICAgICJkZWx0YV9vbmx5IjogZXhwZXJ0X3Byb2JhYmlsaXRpZXNbOiwgMl0sCiAgICAgICAgfQogICAgICAgIHJldHVybiBFZGl0Q0xFVlJPdXRwdXQoCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXM9cHJvYmFiaWxpdGllcywKICAgICAgICAgICAgZ2F0ZXM9Z2F0ZXMsCiAgICAgICAgICAgIGV4cGVydF9sb2dpdHM9bG9naXRzLAogICAgICAgICAgICBwYWlyX2VtYmVkZGluZz1wYWlyX3osCiAgICAgICAgKQoKCmRlZiBlZGl0Y2xldnJfbG9zcyhvdXRwdXQ6IEVkaXRDTEVWUk91dHB1dCwgbGFiZWxzOiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgIGlmIGxhYmVscy5uZGltICE9IDEgb3IgbGFiZWxzLnNoYXBlWzBdICE9IG91dHB1dC5leHBlcnRfbG9naXRzLnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVscyBtdXN0IGhhdmUgc2hhcGUgW2JhdGNoXSIpCiAgICBpZiBsYWJlbHMuZHR5cGUgIT0gdG9yY2gubG9uZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYWJlbHMgbXVzdCB1c2UgdG9yY2gubG9uZyIpCiAgICBpZiBsYWJlbHMubnVtZWwoKSA9PSAwIG9yIGxhYmVscy5taW4oKSA8IDAgb3IgbGFiZWxzLm1heCgpID49IGxlbihFRElUX0ZBQ1RPUlMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVscyBmYWxsIG91dHNpZGUgRWRpdENMRVZSIGNsYXNzZXMiKQogICAgIyBFdmVyeSBleHBlcnQgaXMgaW5kZXBlbmRlbnRseSB1c2VmdWw7IHRoZSByb3V0ZXIgY2Fubm90IG1hbnVmYWN0dXJlIGdhaW5zCiAgICAjIGJ5IHN0YXJ2aW5nIGEgY29udHJvbCBoZWFkIG9mIHN1cGVydmlzaW9uLgogICAgZXhwZXJ0ID0gbm4uZnVuY3Rpb25hbC5jcm9zc19lbnRyb3B5KAogICAgICAgIG91dHB1dC5leHBlcnRfbG9naXRzLmZsYXR0ZW4oMCwgMSksIGxhYmVscy5yZXBlYXRfaW50ZXJsZWF2ZSg0KQogICAgKQogICAgbGVhcm5lZCA9IG5uLmZ1bmN0aW9uYWwubmxsX2xvc3Mob3V0cHV0LnByb2JhYmlsaXRpZXNbImxlYXJuZWQiXS5jbGFtcF9taW4oMWUtOCkubG9nKCksIGxhYmVscykKICAgIGJhbGFuY2UgPSAob3V0cHV0LmdhdGVzLm1lYW4oZGltPTApIC0gMC4yNSkuc3F1YXJlKCkubWVhbigpCiAgICByZXR1cm4gZXhwZXJ0ICsgbGVhcm5lZCArIDAuMDEgKiBiYWxhbmNlCg==', 'olympus/models/encoder_quantization.py': 'IiIiRGlmZmVyZW50aWFibGUgZW5jb2Rlci10by1yZXNpZHVhbC12ZWN0b3ItcXVhbnRpemF0aW9uIHBhdGh3YXkuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5cGluZyBpbXBvcnQgTmFtZWRUdXBsZSwgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xhc3MgRW5jb2RlclF1YW50aXphdGlvbkNvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICBpbnB1dF9kaW06IGludCA9IEZpZWxkKGdlPTEpCiAgICBsYXRlbnRfZGltOiBpbnQgPSBGaWVsZChnZT0yKQogICAgaGlkZGVuX2RpbTogaW50ID0gRmllbGQoZGVmYXVsdD0yNTYsIGdlPTIpCiAgICBjb2RlYm9va19zaXplOiBpbnQgPSBGaWVsZChkZWZhdWx0PTI1NiwgZ2U9MiwgbGU9NjVfNTM2KQogICAgcmVzaWR1YWxfY29kZWJvb2tzOiBpbnQgPSBGaWVsZChkZWZhdWx0PTIsIGdlPTEsIGxlPTE2KQogICAgY29tbWl0bWVudF93ZWlnaHQ6IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0wLjI1LCBnZT0wLjAsIGxlPTEwLjApCgoKY2xhc3MgUXVhbnRpemF0aW9uT3V0cHV0KE5hbWVkVHVwbGUpOgogICAgcXVhbnRpemVkOiBUZW5zb3IKICAgIGluZGljZXM6IFRlbnNvcgogICAgbG9zczogVGVuc29yCiAgICBjb21taXRtZW50X2xvc3M6IFRlbnNvcgogICAgY29kZWJvb2tfbG9zczogVGVuc29yCiAgICBxdWFudGl6YXRpb25fZXJyb3I6IFRlbnNvcgogICAgdXRpbGl6YXRpb246IFRlbnNvcgoKCmNsYXNzIFJlc2lkdWFsVmVjdG9yUXVhbnRpemVyKG5uLk1vZHVsZSk6CiAgICAiIiJSZXNpZHVhbCBWUSB3aXRoIG1hc2tlZCB0b2tlbnMgYW5kIGEgc3RyYWlnaHQtdGhyb3VnaCBlbmNvZGVyIGdyYWRpZW50LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWc6IEVuY29kZXJRdWFudGl6YXRpb25Db25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLmNvZGVib29rcyA9IG5uLlBhcmFtZXRlcigKICAgICAgICAgICAgdG9yY2guZW1wdHkoY29uZmlnLnJlc2lkdWFsX2NvZGVib29rcywgY29uZmlnLmNvZGVib29rX3NpemUsIGNvbmZpZy5sYXRlbnRfZGltKQogICAgICAgICkKICAgICAgICBubi5pbml0LnVuaWZvcm1fKAogICAgICAgICAgICBzZWxmLmNvZGVib29rcywKICAgICAgICAgICAgLTEuMCAvIGNvbmZpZy5jb2RlYm9va19zaXplLAogICAgICAgICAgICAxLjAgLyBjb25maWcuY29kZWJvb2tfc2l6ZSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgbGF0ZW50OiBUZW5zb3IsIG1hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lKSAtPiBRdWFudGl6YXRpb25PdXRwdXQ6CiAgICAgICAgaWYgbGF0ZW50Lm5kaW0gPCAyIG9yIGxhdGVudC5zaGFwZVstMV0gIT0gc2VsZi5jb25maWcubGF0ZW50X2RpbToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImV4cGVjdGVkIFsuLi4sIHtzZWxmLmNvbmZpZy5sYXRlbnRfZGltfV0gbGF0ZW50IHRlbnNvciIpCiAgICAgICAgdG9rZW5fc2hhcGUgPSBsYXRlbnQuc2hhcGVbOi0xXQogICAgICAgIGZsYXQgPSBsYXRlbnQucmVzaGFwZSgtMSwgc2VsZi5jb25maWcubGF0ZW50X2RpbSkKICAgICAgICBpZiBtYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIGFjdGl2ZSA9IHRvcmNoLm9uZXMoZmxhdC5zaGFwZVswXSwgZHR5cGU9dG9yY2guYm9vbCwgZGV2aWNlPWZsYXQuZGV2aWNlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGlmIHR1cGxlKG1hc2suc2hhcGUpICE9IHR1cGxlKHRva2VuX3NoYXBlKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1hc2sgc2hhcGUgbXVzdCBlcXVhbCB0aGUgbGF0ZW50IHRva2VuIGRpbWVuc2lvbnMiKQogICAgICAgICAgICBhY3RpdmUgPSBtYXNrLnRvKGRldmljZT1mbGF0LmRldmljZSwgZHR5cGU9dG9yY2guYm9vbCkucmVzaGFwZSgtMSkKICAgICAgICBpZiBub3QgYm9vbChhY3RpdmUuYW55KCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhdCBsZWFzdCBvbmUgbGF0ZW50IHRva2VuIG11c3QgYmUgYWN0aXZlIikKCiAgICAgICAgcmVzaWR1YWwgPSBmbGF0CiAgICAgICAgcXVhbnRpemVkX3N1bSA9IHRvcmNoLnplcm9zX2xpa2UoZmxhdCkKICAgICAgICBpbmRpY2VzOiBsaXN0W1RlbnNvcl0gPSBbXQogICAgICAgIGNvbW1pdG1lbnRfdGVybXM6IGxpc3RbVGVuc29yXSA9IFtdCiAgICAgICAgY29kZWJvb2tfdGVybXM6IGxpc3RbVGVuc29yXSA9IFtdCiAgICAgICAgdXRpbGl6YXRpb25fdGVybXM6IGxpc3RbVGVuc29yXSA9IFtdCgogICAgICAgIGZvciBjb2RlYm9vayBpbiBzZWxmLmNvZGVib29rczoKICAgICAgICAgICAgYWN0aXZlX3Jlc2lkdWFsID0gcmVzaWR1YWxbYWN0aXZlXQogICAgICAgICAgICBkaXN0YW5jZXMgPSAoCiAgICAgICAgICAgICAgICBhY3RpdmVfcmVzaWR1YWwuc3F1YXJlKCkuc3VtKGRpbT0xLCBrZWVwZGltPVRydWUpCiAgICAgICAgICAgICAgICAtIDIuMCAqIGFjdGl2ZV9yZXNpZHVhbCBAIGNvZGVib29rLnRyYW5zcG9zZSgwLCAxKQogICAgICAgICAgICAgICAgKyBjb2RlYm9vay5zcXVhcmUoKS5zdW0oZGltPTEpLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICApCiAgICAgICAgICAgIGFjdGl2ZV9pbmRpY2VzID0gZGlzdGFuY2VzLmFyZ21pbihkaW09MSkKICAgICAgICAgICAgc2VsZWN0ZWRfYWN0aXZlID0gY29kZWJvb2tbYWN0aXZlX2luZGljZXNdCiAgICAgICAgICAgIHNlbGVjdGVkID0gdG9yY2guemVyb3NfbGlrZShmbGF0KQogICAgICAgICAgICBzZWxlY3RlZFthY3RpdmVdID0gc2VsZWN0ZWRfYWN0aXZlCiAgICAgICAgICAgIGZ1bGxfaW5kaWNlcyA9IHRvcmNoLmZ1bGwoKGZsYXQuc2hhcGVbMF0sKSwgLTEsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1mbGF0LmRldmljZSkKICAgICAgICAgICAgZnVsbF9pbmRpY2VzW2FjdGl2ZV0gPSBhY3RpdmVfaW5kaWNlcwogICAgICAgICAgICBpbmRpY2VzLmFwcGVuZChmdWxsX2luZGljZXMpCiAgICAgICAgICAgIGNvbW1pdG1lbnRfdGVybXMuYXBwZW5kKAogICAgICAgICAgICAgICAgbm4uZnVuY3Rpb25hbC5tc2VfbG9zcyhhY3RpdmVfcmVzaWR1YWwsIHNlbGVjdGVkX2FjdGl2ZS5kZXRhY2goKSkKICAgICAgICAgICAgKQogICAgICAgICAgICBjb2RlYm9va190ZXJtcy5hcHBlbmQobm4uZnVuY3Rpb25hbC5tc2VfbG9zcyhzZWxlY3RlZF9hY3RpdmUsIGFjdGl2ZV9yZXNpZHVhbC5kZXRhY2goKSkpCiAgICAgICAgICAgIHV0aWxpemF0aW9uX3Rlcm1zLmFwcGVuZCgKICAgICAgICAgICAgICAgIGFjdGl2ZV9pbmRpY2VzLnVuaXF1ZSgpLm51bWVsKCkKICAgICAgICAgICAgICAgIC8gdG9yY2gudGVuc29yKAogICAgICAgICAgICAgICAgICAgIHNlbGYuY29uZmlnLmNvZGVib29rX3NpemUsCiAgICAgICAgICAgICAgICAgICAgZHR5cGU9ZmxhdC5kdHlwZSwKICAgICAgICAgICAgICAgICAgICBkZXZpY2U9ZmxhdC5kZXZpY2UsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcXVhbnRpemVkX3N1bSA9IHF1YW50aXplZF9zdW0gKyBzZWxlY3RlZAogICAgICAgICAgICByZXNpZHVhbCA9IHJlc2lkdWFsIC0gc2VsZWN0ZWQuZGV0YWNoKCkKCiAgICAgICAgY29tbWl0bWVudCA9IHRvcmNoLnN0YWNrKGNvbW1pdG1lbnRfdGVybXMpLm1lYW4oKQogICAgICAgIGNvZGVib29rX2xvc3MgPSB0b3JjaC5zdGFjayhjb2RlYm9va190ZXJtcykubWVhbigpCiAgICAgICAgbG9zcyA9IGNvZGVib29rX2xvc3MgKyBzZWxmLmNvbmZpZy5jb21taXRtZW50X3dlaWdodCAqIGNvbW1pdG1lbnQKICAgICAgICBzdHJhaWdodF90aHJvdWdoID0gZmxhdCArIChxdWFudGl6ZWRfc3VtIC0gZmxhdCkuZGV0YWNoKCkKICAgICAgICBzdHJhaWdodF90aHJvdWdoID0gdG9yY2gud2hlcmUoYWN0aXZlLnVuc3F1ZWV6ZSgxKSwgc3RyYWlnaHRfdGhyb3VnaCwgZmxhdCkKICAgICAgICBlcnJvciA9IG5uLmZ1bmN0aW9uYWwubXNlX2xvc3MocXVhbnRpemVkX3N1bVthY3RpdmVdLCBmbGF0W2FjdGl2ZV0pCiAgICAgICAgcmV0dXJuIFF1YW50aXphdGlvbk91dHB1dCgKICAgICAgICAgICAgcXVhbnRpemVkPXN0cmFpZ2h0X3Rocm91Z2gucmVzaGFwZSgqdG9rZW5fc2hhcGUsIHNlbGYuY29uZmlnLmxhdGVudF9kaW0pLAogICAgICAgICAgICBpbmRpY2VzPXRvcmNoLnN0YWNrKGluZGljZXMsIGRpbT0tMSkucmVzaGFwZSgKICAgICAgICAgICAgICAgICp0b2tlbl9zaGFwZSwgc2VsZi5jb25maWcucmVzaWR1YWxfY29kZWJvb2tzCiAgICAgICAgICAgICksCiAgICAgICAgICAgIGxvc3M9bG9zcywKICAgICAgICAgICAgY29tbWl0bWVudF9sb3NzPWNvbW1pdG1lbnQsCiAgICAgICAgICAgIGNvZGVib29rX2xvc3M9Y29kZWJvb2tfbG9zcywKICAgICAgICAgICAgcXVhbnRpemF0aW9uX2Vycm9yPWVycm9yLAogICAgICAgICAgICB1dGlsaXphdGlvbj10b3JjaC5zdGFjayh1dGlsaXphdGlvbl90ZXJtcykubWVhbigpLAogICAgICAgICkKCgpjbGFzcyBRdWFudGl6ZWRFbmNvZGVyKG5uLk1vZHVsZSk6CiAgICAiIiJUcmFpbmFibGUgZW5jb2RlciB3aG9zZSBjb250aW51b3VzIHN0YXRlcyBwYXNzIHRocm91Z2ggcmVzaWR1YWwgVlEuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogRW5jb2RlclF1YW50aXphdGlvbkNvbmZpZykgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihjb25maWcuaW5wdXRfZGltLCBjb25maWcuaGlkZGVuX2RpbSksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKGNvbmZpZy5oaWRkZW5fZGltLCBjb25maWcubGF0ZW50X2RpbSksCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShjb25maWcubGF0ZW50X2RpbSksCiAgICAgICAgKQogICAgICAgIHNlbGYucXVhbnRpemVyID0gUmVzaWR1YWxWZWN0b3JRdWFudGl6ZXIoY29uZmlnKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlucHV0czogVGVuc29yLCBtYXNrOiBUZW5zb3IgfCBOb25lID0gTm9uZSkgLT4gUXVhbnRpemF0aW9uT3V0cHV0OgogICAgICAgIGlmIGlucHV0cy5uZGltIDwgMiBvciBpbnB1dHMuc2hhcGVbLTFdICE9IHNlbGYuY29uZmlnLmlucHV0X2RpbToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImV4cGVjdGVkIFsuLi4sIHtzZWxmLmNvbmZpZy5pbnB1dF9kaW19XSBlbmNvZGVyIGlucHV0IikKICAgICAgICByZXR1cm4gY2FzdChRdWFudGl6YXRpb25PdXRwdXQsIHNlbGYucXVhbnRpemVyKHNlbGYuZW5jb2RlcihpbnB1dHMpLCBtYXNrPW1hc2spKQoKCmNsYXNzIFF1YW50aXplZEF1dG9lbmNvZGVyT3V0cHV0KE5hbWVkVHVwbGUpOgogICAgcmVjb25zdHJ1Y3Rpb246IFRlbnNvcgogICAgcXVhbnRpemF0aW9uOiBRdWFudGl6YXRpb25PdXRwdXQKCgpjbGFzcyBRdWFudGl6ZWRBdXRvZW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiUmF0ZS1kaXN0b3J0aW9uIHRyYWluaW5nIHdyYXBwZXIgYXJvdW5kIHRoZSBlbmNvZGVyLXF1YW50aXphdGlvbiBwYXRod2F5LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWc6IEVuY29kZXJRdWFudGl6YXRpb25Db25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLmVuY29kZXIgPSBRdWFudGl6ZWRFbmNvZGVyKGNvbmZpZykKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoY29uZmlnLmxhdGVudF9kaW0sIGNvbmZpZy5oaWRkZW5fZGltKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoY29uZmlnLmhpZGRlbl9kaW0sIGNvbmZpZy5pbnB1dF9kaW0pLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbnB1dHM6IFRlbnNvciwgbWFzazogVGVuc29yIHwgTm9uZSA9IE5vbmUpIC0+IFF1YW50aXplZEF1dG9lbmNvZGVyT3V0cHV0OgogICAgICAgIHF1YW50aXphdGlvbiA9IHNlbGYuZW5jb2RlcihpbnB1dHMsIG1hc2s9bWFzaykKICAgICAgICByZWNvbnN0cnVjdGlvbiA9IGNhc3QoVGVuc29yLCBzZWxmLmRlY29kZXIocXVhbnRpemF0aW9uLnF1YW50aXplZCkpCiAgICAgICAgcmV0dXJuIFF1YW50aXplZEF1dG9lbmNvZGVyT3V0cHV0KAogICAgICAgICAgICByZWNvbnN0cnVjdGlvbj1yZWNvbnN0cnVjdGlvbiwKICAgICAgICAgICAgcXVhbnRpemF0aW9uPXF1YW50aXphdGlvbiwKICAgICAgICApCg==', 'olympus/models/families.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSB0eXBpbmcgaW1wb3J0IFR5cGVkRGljdAoKZnJvbSBvbHltcHVzLmNvcmUuaW50ZXJwcmV0YXRpb24gaW1wb3J0IEludGVycHJldGl2ZVN1cGVycG9zaXRpb25OZXR3b3JrCmZyb20gb2x5bXB1cy5tZW1vcnkuc3RvcmUgaW1wb3J0IE1lbW9yeVJlY29yZCwgTWVtb3J5U3RvcmUKCgpjbGFzcyBIZXJtZXNSZXNwb25zZShUeXBlZERpY3QpOgogICAgcmVzcG9uc2U6IHN0cgogICAgY29uZmlkZW5jZTogZmxvYXQKICAgIGNhdGVnb3J5OiBzdHIKCgpjbGFzcyBIZXJtZXNOYW5vOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1lbW9yeTogTWVtb3J5U3RvcmUpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5tZW1vcnkgPSBtZW1vcnkKICAgICAgICBzZWxmLmludGVycHJldGVyID0gSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsoKQoKICAgIGRlZiByZXNwb25kKHNlbGYsIHByb21wdDogc3RyKSAtPiBIZXJtZXNSZXNwb25zZToKICAgICAgICBicmFuY2hlcyA9IHNlbGYuaW50ZXJwcmV0ZXIuYW5hbHl6ZShwcm9tcHQpCiAgICAgICAgYmVzdCA9IGJyYW5jaGVzWzBdCiAgICAgICAgc2VsZi5tZW1vcnkucHV0KAogICAgICAgICAgICBNZW1vcnlSZWNvcmQoCiAgICAgICAgICAgICAgICBraW5kPSJlcGlzb2RpYyIsCiAgICAgICAgICAgICAgICBrZXk9ZiJwcm9tcHQ6e2FicyhoYXNoKHByb21wdCkpfSIsCiAgICAgICAgICAgICAgICB2YWx1ZT1iZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgICAgIHNhbGllbmNlPWJlc3QuY29uZmlkZW5jZSwKICAgICAgICAgICAgICAgIHRhZ3M9WyJoZXJtZXMiLCBiZXN0LmNhdGVnb3J5LnZhbHVlXSwKICAgICAgICAgICAgKQogICAgICAgICkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAicmVzcG9uc2UiOiBiZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgImNvbmZpZGVuY2UiOiBiZXN0LmNvbmZpZGVuY2UsCiAgICAgICAgICAgICJjYXRlZ29yeSI6IGJlc3QuY2F0ZWdvcnkudmFsdWUsCiAgICAgICAgfQo=', 'olympus/models/flagship.py': '...', 'olympus/models/olympus_system.py': '...', 'olympus/models/oracle_fusion_v4.py': 'IiIiVjQ6IGZyb3plbiBPbHltcHVzIGV4cGVydCBiYW5rcyB3aXRoIGNvbW1vbiwgY3Jvc3MtZml0dGVkIHJvdXRpbmcgY29udHJvbHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKZnJvbSBvbHltcHVzLm1vZGVscy5vbHltcHVzX3N5c3RlbSBpbXBvcnQgT2x5bXB1c1JlZmVyZW5jZVN5c3RlbSwgT2x5bXB1c1N5c3RlbUNvbmZpZwpmcm9tIG9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyBpbXBvcnQgKAogICAgQWlvblJlZmVyZW5jZUNvbmZpZywKICAgIEF0bGFzUmVmZXJlbmNlQ29uZmlnLAogICAgSGVybWVzUmVmZXJlbmNlQ29uZmlnLAogICAgS3Jvbm9zUmVmZXJlbmNlQ29uZmlnLAogICAgUGVyc2V1c1JlZmVyZW5jZUNvbmZpZywKICAgIFByb21ldGhldXNSZWZlcmVuY2VDb25maWcsCiAgICBSZWZlcmVuY2VUcmFuc2Zvcm1lckNvbmZpZywKKQoKTU9ERVMgPSAoCiAgICAidW5pZm9ybSIsCiAgICAiZml4ZWQiLAogICAgInNlbWFudGljIiwKICAgICJ1dGlsaXR5IiwKICAgICJjb3JydXB0aW9uIiwKICAgICJvcmFjbGVfY29ycnVwdGlvbiIsCiAgICAib3JhY2xlX3Rhc2siLAogICAgImJlc3Rfc2luZ2xlIiwKICAgICJzZW1hbnRpY190cmFpbl9tZWFuIiwKKQpJTlBVVFMgPSAoInRva2VucyIsICJpbWFnZSIsICJzdGF0ZXMiLCAiYWN0aW9ucyIsICJhbGxvd2VkX3RyYW5zaXRpb25zIiwgInJpc2tfc2NvcmUiKQoKCmRlZiBzeXN0ZW1fY29uZmlnKCkgLT4gT2x5bXB1c1N5c3RlbUNvbmZpZzoKICAgIGJhY2tib25lID0gUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25maWcoCiAgICAgICAgdm9jYWJfc2l6ZT0zMiwKICAgICAgICBtb2RlbF9kaW09MTYsCiAgICAgICAgbGF5ZXJzPTEsCiAgICAgICAgaGVhZHM9NCwKICAgICAgICBmZWVkZm9yd2FyZF9kaW09MzIsCiAgICAgICAgbWF4X3NlcXVlbmNlX2xlbmd0aD04LAogICAgKQogICAgcmV0dXJuIE9seW1wdXNTeXN0ZW1Db25maWcoCiAgICAgICAgaGVybWVzPUhlcm1lc1JlZmVyZW5jZUNvbmZpZyhiYWNrYm9uZT1iYWNrYm9uZSwgdG9vbF9jb3VudD00KSwKICAgICAgICBwcm9tZXRoZXVzPVByb21ldGhldXNSZWZlcmVuY2VDb25maWcoCiAgICAgICAgICAgIGJhY2tib25lPWJhY2tib25lLCBkb21haW5zPTMsIGFkYXB0ZXJfZGltPTgsIGV2aWRlbmNlX2NsYXNzZXM9NAogICAgICAgICksCiAgICAgICAgcGVyc2V1cz1QZXJzZXVzUmVmZXJlbmNlQ29uZmlnKAogICAgICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwgZXhwZXJ0cz00LCBhY3RpdmVfZXhwZXJ0cz0yLCBleHBlcnRfZGltPTMyCiAgICAgICAgKSwKICAgICAgICBhdGxhcz1BdGxhc1JlZmVyZW5jZUNvbmZpZygKICAgICAgICAgICAgYmFja2JvbmU9YmFja2JvbmUsIGltYWdlX2NoYW5uZWxzPTMsIHBhdGNoX3NpemU9MiwgcmV0cmlldmFsX2RpbT0xMgogICAgICAgICksCiAgICAgICAga3Jvbm9zPUtyb25vc1JlZmVyZW5jZUNvbmZpZygKICAgICAgICAgICAgc3RhdGVfZGltPTUsIGFjdGlvbl9kaW09MywgaGlkZGVuX2RpbT0xMiwgZGlzY3JldGVfYWN0aW9ucz00LCBwbGFubmluZ19ob3Jpem9uPTQKICAgICAgICApLAogICAgICAgIGFpb249QWlvblJlZmVyZW5jZUNvbmZpZyhzdGF0ZV9kaW09OCwgaGlkZGVuX2RpbT0xMiwgcmVzZWFyY2hfc3RhZ2VzPTQsIHN0b3Bfc3RhZ2U9MyksCiAgICAgICAgZnVzaW9uX2RpbT0xNiwKICAgICAgICBkZWNpc2lvbl9jbGFzc2VzPTQsCiAgICAgICAgZnVzaW9uX3N0cmF0ZWd5PSJ1bmlmb3JtIiwKICAgICkKCgpjbGFzcyBFeHBlcnRCYW5rKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnN5c3RlbSA9IE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0oc3lzdGVtX2NvbmZpZygpKQogICAgICAgIHNlbGYuaGVhZHMgPSBubi5Nb2R1bGVMaXN0KG5uLkxpbmVhcigxNiwgNCkgZm9yIF8gaW4gcmFuZ2UoNSkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgYmF0Y2g6IGRpY3Rbc3RyLCBUZW5zb3JdKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvciwgVGVuc29yXToKICAgICAgICBvdXRwdXQgPSBzZWxmLnN5c3RlbSgqKntrZXk6IGJhdGNoW2tleV0gZm9yIGtleSBpbiBJTlBVVFN9LCBkb21haW5faWRzPWJhdGNoWyJkb21haW5faWRzIl0pCiAgICAgICAgeiA9IG91dHB1dC5jb21wb25lbnRfc3RhdGVzCiAgICAgICAgbG9naXRzID0gdG9yY2guc3RhY2soW2hlYWQoels6LCBpXSkgZm9yIGksIGhlYWQgaW4gZW51bWVyYXRlKHNlbGYuaGVhZHMpXSwgMSkKICAgICAgICByZXR1cm4geiwgbG9naXRzLCBvdXRwdXQua3Jvbm9zLm5leHRfc3RhdGUKCgpkZWYgZXhwZXJ0X2xvc3MobG9naXRzOiBUZW5zb3IsIHRhcmdldDogVGVuc29yLCB3ZWlnaHQ6IGZsb2F0KSAtPiBUZW5zb3I6CiAgICAjIEFsbCBleHBlcnRzIGhhdmUgZXhwbGljaXQgdGFzayBzdXBlcnZpc2lvbjsgbm8gaGVhZCBpcyByZXdhcmRlZCBmb3IgYSBsYWJlbAogICAgIyBpdCBvbmx5IHNhdyB0aHJvdWdoIHRhcmdldC1kZXJpdmVkIG1ldGFkYXRhLiBDRiBpcyB0aGUgT05MWSBwYWlyZWQgY2hhbmdlLgogICAgY2UgPSBubi5mdW5jdGlvbmFsLmNyb3NzX2VudHJvcHkobG9naXRzLmZsYXR0ZW4oMCwgMSksIHRhcmdldC5yZXBlYXRfaW50ZXJsZWF2ZSg1KSkKICAgIHAgPSBsb2dpdHMuc29mdG1heCgtMSkKICAgIGZ1bGwgPSBwLm1lYW4oMSkuZGV0YWNoKCkKICAgIGxlYXZlID0gKHAuc3VtKDEsIGtlZXBkaW09VHJ1ZSkgLSBwKSAvIDQKICAgIGNvbnNpc3RlbmN5ID0gKAogICAgICAgIG5uLmZ1bmN0aW9uYWwua2xfZGl2KAogICAgICAgICAgICBsZWF2ZS5jbGFtcF9taW4oMWUtOCkubG9nKCksIGZ1bGxbOiwgTm9uZV0uZXhwYW5kX2FzKGxlYXZlKSwgcmVkdWN0aW9uPSJub25lIgogICAgICAgICkKICAgICAgICAuc3VtKC0xKQogICAgICAgIC5tZWFuKCkKICAgICkKICAgIHJldHVybiBjZSArIHdlaWdodCAqIGNvbnNpc3RlbmN5CgoKZGVmIHJvdXRlcl9mZWF0dXJlcyh6OiBUZW5zb3IsIGxvZ2l0czogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBwID0gbG9naXRzLnNvZnRtYXgoLTEpCiAgICBlbnRyb3B5ID0gLShwICogcC5jbGFtcF9taW4oMWUtOCkubG9nKCkpLnN1bSgtMSkKICAgIHJldHVybiB0b3JjaC5jYXQoW3ouZmxhdHRlbigxKSwgcC5mbGF0dGVuKDEpLCBlbnRyb3B5XSwgMSkuZGV0YWNoKCkKCgpjbGFzcyBSb3V0ZXJzKG5uLk1vZHVsZSk6CiAgICAiIiJFdmVyeSBjb250cm9sIHNoYXJlcyB0aGlzIGV4YWN0IHRyYWluZWQgZ3JhcGggYW5kIGZyb3plbiBleHBlcnQgcHJlZGljdGlvbnMuCgogICAgSGVhZHMgZG8gbm90IHNoYXJlIHRyYWluYWJsZSBoaWRkZW4gbGF5ZXJzLCBhdm9pZGluZyBhdXhpbGlhcnktbG9zcyB0cmFuc2ZlcgogICAgYmV0d2VlbiBzZW1hbnRpYywgdXRpbGl0eSwgYW5kIGNvcnJ1cHRpb24gY29udHJvbHMuIEFsbCBhcmUgYWx3YXlzIGV4ZWN1dGVkLgogICAgIiIiCgogICAgdHJhaW5pbmdfbWVhbl9nYXRlczogVGVuc29yCgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5maXhlZCA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyg1KSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigidHJhaW5pbmdfbWVhbl9nYXRlcyIsIHRvcmNoLm9uZXMoNSkgLyA1KQogICAgICAgIHNlbGYuaGVhZHMgPSBubi5Nb2R1bGVEaWN0KAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICBuYW1lOiBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgICAgIG5uLkxheWVyTm9ybSgxMDUpLCBubi5MaW5lYXIoMTA1LCAzMiksIG5uLkdFTFUoKSwgbm4uTGluZWFyKDMyLCA1KQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCJzZW1hbnRpYyIsICJ1dGlsaXR5IiwgImNvcnJ1cHRpb24iKQogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGZlYXR1cmVzOiBUZW5zb3IpIC0+IGRpY3Rbc3RyLCBUZW5zb3JdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICoqe25hbWU6IGhlYWQoZmVhdHVyZXMpIGZvciBuYW1lLCBoZWFkIGluIHNlbGYuaGVhZHMuaXRlbXMoKX0sCiAgICAgICAgICAgICJmaXhlZCI6IHNlbGYuZml4ZWQuZXhwYW5kKGZlYXR1cmVzLnNoYXBlWzBdLCAtMSksCiAgICAgICAgICAgICJzZW1hbnRpY190cmFpbl9tZWFuIjogc2VsZi50cmFpbmluZ19tZWFuX2dhdGVzLmV4cGFuZChmZWF0dXJlcy5zaGFwZVswXSwgLTEpLAogICAgICAgIH0KCgpkZWYgbm9ybWFsaXplX3F1YWxpdHkocXVhbGl0eTogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBtYXNzID0gcXVhbGl0eS5zdW0oLTEsIGtlZXBkaW09VHJ1ZSkKICAgIHJldHVybiB0b3JjaC53aGVyZShtYXNzID4gMCwgcXVhbGl0eSAvIG1hc3MuY2xhbXBfbWluKDFlLTgpLCB0b3JjaC5vbmVzX2xpa2UocXVhbGl0eSkgLyA1KQoKCmRlZiBwcm9iYWJpbGl0aWVzKAogICAgbG9naXRzOiBUZW5zb3IsCiAgICBzY29yZXM6IGRpY3Rbc3RyLCBUZW5zb3JdLAogICAgbW9kZTogc3RyLAogICAgKiwKICAgIHF1YWxpdHk6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgbGFiZWxzOiBUZW5zb3IgfCBOb25lID0gTm9uZSwKICAgIGJlc3Rfc2luZ2xlOiBpbnQgPSAwLAopIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIHAgPSBsb2dpdHMuc29mdG1heCgtMSkKICAgIGlmIG1vZGUgPT0gInVuaWZvcm0iOgogICAgICAgIGdhdGVzID0gdG9yY2gub25lc19saWtlKHBbLi4uLCAwXSkgLyA1CiAgICBlbGlmIG1vZGUgaW4geyJmaXhlZCIsICJzZW1hbnRpYyJ9OgogICAgICAgIGdhdGVzID0gc2NvcmVzW21vZGVdLnNvZnRtYXgoLTEpCiAgICBlbGlmIG1vZGUgPT0gInNlbWFudGljX3RyYWluX21lYW4iOgogICAgICAgIGdhdGVzID0gc2NvcmVzW21vZGVdCiAgICBlbGlmIG1vZGUgaW4geyJ1dGlsaXR5IiwgImNvcnJ1cHRpb24ifToKICAgICAgICBnYXRlcyA9IG5vcm1hbGl6ZV9xdWFsaXR5KHNjb3Jlc1ttb2RlXS5zaWdtb2lkKCkpCiAgICBlbGlmIG1vZGUgPT0gIm9yYWNsZV9jb3JydXB0aW9uIjoKICAgICAgICBpZiBxdWFsaXR5IGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByaXZpbGVnZWQgY29ycnVwdGlvbiBvcmFjbGUgcmVxdWlyZXMgcXVhbGl0eSBtZXRhZGF0YSIpCiAgICAgICAgZ2F0ZXMgPSBub3JtYWxpemVfcXVhbGl0eShxdWFsaXR5KQogICAgZWxpZiBtb2RlID09ICJvcmFjbGVfdGFzayI6CiAgICAgICAgaWYgbGFiZWxzIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByaXZpbGVnZWQgdGFzayBvcmFjbGUgcmVxdWlyZXMgdHJ1ZSBsYWJlbHMiKQogICAgICAgIHV0aWxpdHkgPSBwLmdhdGhlcigyLCBsYWJlbHNbOiwgTm9uZSwgTm9uZV0uZXhwYW5kKC0xLCA1LCAxKSkuc3F1ZWV6ZSgtMSkKICAgICAgICBnYXRlcyA9IG5uLmZ1bmN0aW9uYWwub25lX2hvdCh1dGlsaXR5LmFyZ21heCgtMSksIDUpLnRvKHApCiAgICBlbGlmIG1vZGUgPT0gImJlc3Rfc2luZ2xlIjoKICAgICAgICBnYXRlcyA9IHRvcmNoLnplcm9zX2xpa2UocFsuLi4sIDBdKQogICAgICAgIGdhdGVzWzosIGJlc3Rfc2luZ2xlXSA9IDEKICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihtb2RlKQogICAgcmV0dXJuIChnYXRlc1suLi4sIE5vbmVdICogcCkuc3VtKDEpLCBnYXRlcwo=', 'olympus/models/parameter_growth.py': 'IiIiRnVuY3Rpb24tcHJlc2VydmluZyBGRk4gZ3Jvd3RoIGFuZCBtYXNrZWQgbGF0ZW50IGxlYXJuaW5nIGZvciBhZGRlZCBjYXBhY2l0eS4KCldpZGVuaW5nIGZvbGxvd3MgTmV0Mk5ldCdzIHJlcGxpY2F0aW9uL2NvdW50IGNvbXBlbnNhdGlvbi4gSkVQQSBzdXBwbGllcyBhIHRyYWluaW5nCm9iamVjdGl2ZSBmb3IgYSBuZXcgcmVzaWR1YWwgYWRhcHRlcjsgbmVpdGhlciBvcGVyYXRpb24gY3JlYXRlcyBwcmV0cmFpbmVkIGtub3dsZWRnZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29weQppbXBvcnQgbWF0aApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IGNhc3QKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBUZW5zb3IsIG5uCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgR3Jvd3RoUmVjZWlwdDoKICAgIG9yaWdpbmFsX3BhcmFtZXRlcnM6IGludAogICAgZXhwYW5kZWRfcGFyYW1ldGVyczogaW50CiAgICBvcmlnaW5hbF93aWR0aDogaW50CiAgICBleHBhbmRlZF93aWR0aDogaW50CiAgICBvcHRpbWl6ZXJfcmVidWlsZF9yZXF1aXJlZDogYm9vbCA9IFRydWUKICAgIHRyYWluZWRfdGFyZ2V0X3NjYWxlOiBib29sID0gRmFsc2UKCgpkZWYgd2lkZW5fZmVlZGZvcndhcmQoCiAgICBibG9jazogbm4uU2VxdWVudGlhbCwKICAgIHdpZHRoOiBpbnQsCiAgICAqLAogICAgbWF4X3BhcmFtZXRlcl9ieXRlczogaW50LAopIC0+IHR1cGxlW25uLlNlcXVlbnRpYWwsIEdyb3d0aFJlY2VpcHRdOgogICAgIiIiQ29weSBhIExpbmVhci9hY3RpdmF0aW9uWy96ZXJvLWRyb3BvdXRdL0xpbmVhciBibG9jayBpbnRvIGEgd2lkZXIgZXF1aXZhbGVudC4KCiAgICBUaGUgb2xkIG1vZHVsZSBhbmQgb3B0aW1pemVyIGFyZSBuZXZlciBtdXRhdGVkLiBUaGUgY2FsbGVyIG11c3QgYnVpbGQgYSBuZXcKICAgIG9wdGltaXplcjsgcHJlc2VydmluZyBvdXRwdXRzIGF0IGluaXRpYWxpemF0aW9uIGRvZXMgbm90IHByZXNlcnZlIHRyYWluaW5nIGR5bmFtaWNzLgogICAgIiIiCiAgICBsYXllcnMgPSBsaXN0KGJsb2NrLmNoaWxkcmVuKCkpCiAgICBpZiAoCiAgICAgICAgbGVuKGxheWVycykgbm90IGluIHszLCA0fQogICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKGxheWVyc1swXSwgbm4uTGluZWFyKQogICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKGxheWVyc1stMV0sIG5uLkxpbmVhcikKICAgICAgICBvciBub3QgaXNpbnN0YW5jZShsYXllcnNbMV0sIChubi5HRUxVLCBubi5TaUxVLCBubi5SZUxVKSkKICAgICAgICBvciAobGVuKGxheWVycykgPT0gNCBhbmQgKG5vdCBpc2luc3RhbmNlKGxheWVyc1syXSwgbm4uRHJvcG91dCkgb3IgbGF5ZXJzWzJdLnAgIT0gMCkpCiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImdyb3d0aCBzdXBwb3J0cyBMaW5lYXIvYWN0aXZhdGlvblsvemVyby1kcm9wb3V0XS9MaW5lYXIgb25seSIpCiAgICBmaXJzdCwgbGFzdCA9IGxheWVyc1swXSwgbGF5ZXJzWy0xXQogICAgYXNzZXJ0IGlzaW5zdGFuY2UoZmlyc3QsIG5uLkxpbmVhcikgYW5kIGlzaW5zdGFuY2UobGFzdCwgbm4uTGluZWFyKQogICAgb2xkID0gZmlyc3Qub3V0X2ZlYXR1cmVzCiAgICBpZiB0eXBlKHdpZHRoKSBpcyBub3QgaW50IG9yIHdpZHRoIDw9IG9sZCBvciBsYXN0LmluX2ZlYXR1cmVzICE9IG9sZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm93dGggd2lkdGggbXVzdCBleGNlZWQgdGhlIGFsaWduZWQgb3JpZ2luYWwgaGlkZGVuIHdpZHRoIikKICAgIHBhcmFtZXRlcnMgPSBsaXN0KGJsb2NrLnBhcmFtZXRlcnMoKSkKICAgIGlmIGFueSgKICAgICAgICBwLmlzX21ldGEgb3Igbm90IHAuaXNfZmxvYXRpbmdfcG9pbnQoKSBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkKICAgICAgICBmb3IgcCBpbiBwYXJhbWV0ZXJzCiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImdyb3d0aCByZXF1aXJlcyBtYXRlcmlhbGl6ZWQgZmluaXRlIGZsb2F0aW5nLXBvaW50IHBhcmFtZXRlcnMiKQogICAgaWYgYW55KHAuZHR5cGUgIT0gZmlyc3Qud2VpZ2h0LmR0eXBlIG9yIHAuZGV2aWNlICE9IGZpcnN0LndlaWdodC5kZXZpY2UgZm9yIHAgaW4gcGFyYW1ldGVycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ3Jvd3RoIHBhcmFtZXRlcnMgbXVzdCBzaGFyZSBkdHlwZSBhbmQgZGV2aWNlIikKICAgIGNvdW50ID0gd2lkdGggKiAoZmlyc3QuaW5fZmVhdHVyZXMgKyBsYXN0Lm91dF9mZWF0dXJlcyArIGludChmaXJzdC5iaWFzIGlzIG5vdCBOb25lKSkKICAgIGNvdW50ICs9IGxhc3Qub3V0X2ZlYXR1cmVzIGlmIGxhc3QuYmlhcyBpcyBub3QgTm9uZSBlbHNlIDAKICAgIG9sZF9jb3VudCA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gcGFyYW1ldGVycykKICAgIGlmICgKICAgICAgICB0eXBlKG1heF9wYXJhbWV0ZXJfYnl0ZXMpIGlzIG5vdCBpbnQKICAgICAgICBvciAoY291bnQgKyBvbGRfY291bnQpICogZmlyc3Qud2VpZ2h0LmVsZW1lbnRfc2l6ZSgpID4gbWF4X3BhcmFtZXRlcl9ieXRlcwogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm93dGggZXhjZWVkcyB0aGUgY29tYmluZWQgb2xkL25ldyBwYXJhbWV0ZXIgYnl0ZSBidWRnZXQiKQogICAgZGV2aWNlLCBkdHlwZSA9IGZpcnN0LndlaWdodC5kZXZpY2UsIGZpcnN0LndlaWdodC5kdHlwZQogICAgIyBBbGxvY2F0ZSB1bmluaXRpYWxpemVkIHRlbnNvcnMgd2l0aG91dCBjb25zdW1pbmcgYW55IGFwcGxpY2F0aW9uIFJORyBzdGF0ZS4KICAgIGxlZnQgPSBubi5MaW5lYXIoZmlyc3QuaW5fZmVhdHVyZXMsIHdpZHRoLCBiaWFzPWZpcnN0LmJpYXMgaXMgbm90IE5vbmUsIGRldmljZT0ibWV0YSIpCiAgICByaWdodCA9IG5uLkxpbmVhcih3aWR0aCwgbGFzdC5vdXRfZmVhdHVyZXMsIGJpYXM9bGFzdC5iaWFzIGlzIG5vdCBOb25lLCBkZXZpY2U9Im1ldGEiKQogICAgbWFwcGluZyA9IHRvcmNoLmFyYW5nZSh3aWR0aCwgZGV2aWNlPWRldmljZSkgJSBvbGQKICAgIGNvdW50cyA9IHRvcmNoLmJpbmNvdW50KG1hcHBpbmcsIG1pbmxlbmd0aD1vbGQpLnRvKGR0eXBlKQogICAgbGVmdC53ZWlnaHQgPSBubi5QYXJhbWV0ZXIoZmlyc3Qud2VpZ2h0LmRldGFjaCgpW21hcHBpbmddLmNsb25lKCksIGZpcnN0LndlaWdodC5yZXF1aXJlc19ncmFkKQogICAgcmlnaHQud2VpZ2h0ID0gbm4uUGFyYW1ldGVyKAogICAgICAgIChsYXN0LndlaWdodC5kZXRhY2goKVs6LCBtYXBwaW5nXSAvIGNvdW50c1ttYXBwaW5nXSkuY2xvbmUoKSwgbGFzdC53ZWlnaHQucmVxdWlyZXNfZ3JhZAogICAgKQogICAgaWYgZmlyc3QuYmlhcyBpcyBub3QgTm9uZToKICAgICAgICBsZWZ0LmJpYXMgPSBubi5QYXJhbWV0ZXIoZmlyc3QuYmlhcy5kZXRhY2goKVttYXBwaW5nXS5jbG9uZSgpLCBmaXJzdC5iaWFzLnJlcXVpcmVzX2dyYWQpCiAgICBpZiBsYXN0LmJpYXMgaXMgbm90IE5vbmU6CiAgICAgICAgcmlnaHQuYmlhcyA9IG5uLlBhcmFtZXRlcihsYXN0LmJpYXMuZGV0YWNoKCkuY2xvbmUoKSwgbGFzdC5iaWFzLnJlcXVpcmVzX2dyYWQpCiAgICByZXN1bHQgPSBubi5TZXF1ZW50aWFsKGxlZnQsICooY29weS5kZWVwY29weShsYXllcikgZm9yIGxheWVyIGluIGxheWVyc1sxOi0xXSksIHJpZ2h0KQogICAgcmVzdWx0LnRyYWluKGJsb2NrLnRyYWluaW5nKQogICAgcmV0dXJuIHJlc3VsdCwgR3Jvd3RoUmVjZWlwdChvbGRfY291bnQsIGNvdW50LCBvbGQsIHdpZHRoKQoKCmNsYXNzIEpFUEFQYXJhbWV0ZXJHcm93dGgobm4uTW9kdWxlKToKICAgICIiIklkZW50aXR5LWluaXRpYWxpemVkIHJlc2lkdWFsIGNhcGFjaXR5IHRyYWluZWQgd2l0aCBtYXNrZWQgbGF0ZW50IHByZWRpY3Rpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHdpZHRoOiBpbnQsIGFkZGVkX3dpZHRoOiBpbnQsICosIG1vbWVudHVtOiBmbG9hdCA9IDAuOTkpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgdHlwZSh3aWR0aCkgaXMgbm90IGludCBvciB0eXBlKGFkZGVkX3dpZHRoKSBpcyBub3QgaW50IG9yIG1pbih3aWR0aCwgYWRkZWRfd2lkdGgpIDwgMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGhzIG11c3QgYmUgaW50ZWdlcnMgPj0gMiIpCiAgICAgICAgaWYgbWF4KHdpZHRoLCBhZGRlZF93aWR0aCkgPiA4MTkyIG9yIG5vdCBtYXRoLmlzZmluaXRlKG1vbWVudHVtKSBvciBub3QgMCA8PSBtb21lbnR1bSA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImdyb3d0aCBkaW1lbnNpb25zIG9yIEVNQSBtb21lbnR1bSBleGNlZWQgdGhlIGJvdW5kZWQgY29udHJhY3QiKQogICAgICAgIHNlbGYud2lkdGgsIHNlbGYubW9tZW50dW0gPSB3aWR0aCwgbW9tZW50dW0KICAgICAgICBzZWxmLmNvbnRleHRfZW5jb2RlciA9IG5uLkxpbmVhcih3aWR0aCwgYWRkZWRfd2lkdGgpCiAgICAgICAgc2VsZi50YXJnZXRfZW5jb2RlciA9IGNvcHkuZGVlcGNvcHkoc2VsZi5jb250ZXh0X2VuY29kZXIpLnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIHNlbGYucHJlZGljdG9yID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIoYWRkZWRfd2lkdGgsIGFkZGVkX3dpZHRoKSwgbm4uR0VMVSgpKQogICAgICAgIHNlbGYucmVzaWR1YWxfaGVhZCA9IG5uLkxpbmVhcihhZGRlZF93aWR0aCwgd2lkdGgpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5yZXNpZHVhbF9oZWFkLndlaWdodCkKICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLnJlc2lkdWFsX2hlYWQuYmlhcykKCiAgICBkZWYgX3ZhbGlkYXRlKHNlbGYsIGhpZGRlbjogVGVuc29yKSAtPiBOb25lOgogICAgICAgIHJlZmVyZW5jZSA9IHNlbGYuY29udGV4dF9lbmNvZGVyLndlaWdodAogICAgICAgIGlmICgKICAgICAgICAgICAgaGlkZGVuLm5kaW0gIT0gMwogICAgICAgICAgICBvciBoaWRkZW4uc2hhcGVbLTFdICE9IHNlbGYud2lkdGgKICAgICAgICAgICAgb3IgbWluKGhpZGRlbi5zaGFwZVs6Ml0pIDwgMQogICAgICAgICAgICBvciBoaWRkZW4uZHR5cGUgIT0gcmVmZXJlbmNlLmR0eXBlCiAgICAgICAgICAgIG9yIGhpZGRlbi5kZXZpY2UgIT0gcmVmZXJlbmNlLmRldmljZQogICAgICAgICAgICBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShoaWRkZW4pLmFsbCgpKQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAiaGlkZGVuIHN0YXRlcyBtdXN0IGJlIGZpbml0ZSBbYmF0Y2gsIHRva2Vucywgd2lkdGhdIG1hdGNoaW5nIHBhcmFtZXRlcnMiCiAgICAgICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBoaWRkZW46IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIHNlbGYuX3ZhbGlkYXRlKGhpZGRlbikKICAgICAgICByZXR1cm4gY2FzdCgKICAgICAgICAgICAgVGVuc29yLCBoaWRkZW4gKyBzZWxmLnJlc2lkdWFsX2hlYWQoc2VsZi5wcmVkaWN0b3Ioc2VsZi5jb250ZXh0X2VuY29kZXIoaGlkZGVuKSkpCiAgICAgICAgKQoKICAgIGRlZiBtYXNrZWRfb2JqZWN0aXZlKHNlbGYsIGhpZGRlbjogVGVuc29yLCB0YXJnZXRfbWFzazogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgIiIiRXhjbHVkZSB0YXJnZXQgcG9zaXRpb25zIGZyb20gdGhpcyBhZGFwdGVyJ3MgbGF0ZW50IHByZWRpY3Rpb24gY29udGV4dC4KCiAgICAgICAgVGhpcyBjYW5ub3QgdW5kbyBpbmZvcm1hdGlvbiBhbHJlYWR5IG1peGVkIGJ5IGFuIHVwc3RyZWFtIGVuY29kZXIuIEZvcgogICAgICAgIHJhdy1pbnB1dCBtYXNraW5nIHN0dWRpZXMgdGhlIGNhbGxlciBtdXN0IGVuY29kZSBjb250ZXh0IGZyb20gc2VwYXJhdGVseQogICAgICAgIG1hc2tlZCBpbnB1dHM7IHRoaXMgYm91bmRlZCBvYmplY3RpdmUgYWxvbmUgZG9lcyBub3QgY2VydGlmeSB0aGF0IGJvdW5kYXJ5LgogICAgICAgICIiIgogICAgICAgIHNlbGYuX3ZhbGlkYXRlKGhpZGRlbikKICAgICAgICBpZiAoCiAgICAgICAgICAgIHRhcmdldF9tYXNrLnNoYXBlICE9IGhpZGRlbi5zaGFwZVs6Ml0KICAgICAgICAgICAgb3IgdGFyZ2V0X21hc2suZHR5cGUgIT0gdG9yY2guYm9vbAogICAgICAgICAgICBvciB0YXJnZXRfbWFzay5kZXZpY2UgIT0gaGlkZGVuLmRldmljZQogICAgICAgICAgICBvciBub3QgYm9vbCh0YXJnZXRfbWFzay5hbnkoLTEpLmFsbCgpKQogICAgICAgICAgICBvciBub3QgYm9vbCgofnRhcmdldF9tYXNrKS5hbnkoLTEpLmFsbCgpKQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVhY2ggc2VxdWVuY2UgbmVlZHMgQm9vbGVhbiB2aXNpYmxlIGFuZCB0YXJnZXQgcG9zaXRpb25zIikKICAgICAgICB2aXNpYmxlID0gfnRhcmdldF9tYXNrCiAgICAgICAgY29udGV4dCA9IHNlbGYuY29udGV4dF9lbmNvZGVyKGhpZGRlbi5tYXNrZWRfZmlsbCh0YXJnZXRfbWFza1suLi4sIE5vbmVdLCAwKSkKICAgICAgICBwb29sZWQgPSAoY29udGV4dCAqIHZpc2libGVbLi4uLCBOb25lXSkuc3VtKDEpIC8gdmlzaWJsZS5zdW0oMSwga2VlcGRpbT1UcnVlKQogICAgICAgIHByZWRpY3RlZCA9IHNlbGYucHJlZGljdG9yKHBvb2xlZClbOiwgTm9uZV0uZXhwYW5kKC0xLCBoaWRkZW4uc2hhcGVbMV0sIC0xKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICB0YXJnZXQgPSBzZWxmLnRhcmdldF9lbmNvZGVyKGhpZGRlbi5kZXRhY2goKSkKICAgICAgICByZXR1cm4gY2FzdChUZW5zb3IsIChwcmVkaWN0ZWRbdGFyZ2V0X21hc2tdIC0gdGFyZ2V0W3RhcmdldF9tYXNrXSkuc3F1YXJlKCkubWVhbigpKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiB1cGRhdGVfdGFyZ2V0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgb25saW5lID0gdHVwbGUoc2VsZi5jb250ZXh0X2VuY29kZXIucGFyYW1ldGVycygpKQogICAgICAgIHRhcmdldCA9IHR1cGxlKHNlbGYudGFyZ2V0X2VuY29kZXIucGFyYW1ldGVycygpKQogICAgICAgIGlmIGFueShub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkgZm9yIHAgaW4gKCpvbmxpbmUsICp0YXJnZXQpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRU1BIHVwZGF0ZSByZXF1aXJlcyBmaW5pdGUgb25saW5lIGFuZCB0YXJnZXQgcGFyYW1ldGVycyIpCiAgICAgICAgZm9yIHRlYWNoZXIsIHN0dWRlbnQgaW4gemlwKHRhcmdldCwgb25saW5lLCBzdHJpY3Q9VHJ1ZSk6CiAgICAgICAgICAgIHRlYWNoZXIubGVycF8oc3R1ZGVudCwgMSAtIHNlbGYubW9tZW50dW0pCg==', 'olympus/models/reference_families.py': '...', 'olympus/models/reliability_fusion.py': 'IiIiUmVsaWFiaWxpdHktc3VwZXJ2aXNlZCBzdWNjZXNzb3IgdG8gdGhlIGZyb3plbiBPbHltcHVzIHYyIGZ1c2lvbiBtZWNoYW5pc20uCgpUaGUgbW9kdWxlIGlzIGFkZGl0aXZlOiBpdCBsZWF2ZXMgdGhlIHYyIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpbnRhY3QgYW5kCnJldXNlcyBpdHMgZmFtaWx5IGVuY29kZXJzLiAgRXZlcnkgcm91dGluZyBjb250cm9sIGluc3RhbnRpYXRlcyBhbmQgZXhlY3V0ZXMKdGhlIHNhbWUgcmVsaWFiaWxpdHkgaGVhZHMgc28gcGFyYW1ldGVyIGNvdW50IGFuZCBmb3J3YXJkIGNvbXB1dGUgcmVtYWluCm1hdGNoZWQuICBSZWxpYWJpbGl0eSBlc3RpbWF0ZXMgYXJlIHByZWRpY3Rpb25zIGZyb20gbW9kZWwgc3RhdGU7IHByb3RlY3RlZAp0YXJnZXRzIGFyZSBuZXZlciBpbnB1dHMgdG8gdGhlIGZvcndhcmQgcGFzcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIHJlcGxhY2UKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgKICAgIE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICBPbHltcHVzU3lzdGVtQ29uZmlnLAogICAgT2x5bXB1c1N5c3RlbUxvc3MsCiAgICBPbHltcHVzU3lzdGVtT3V0cHV0LAogICAgb2x5bXB1c19zeXN0ZW1fbG9zcywKKQoKUmVsaWFiaWxpdHlSb3V0aW5nTW9kZSA9IExpdGVyYWxbCiAgICAicmVsaWFiaWxpdHlfc2hyaW5rYWdlIiwKICAgICJyZWxpYWJpbGl0eV9vbmx5IiwKICAgICJzZW1hbnRpYyIsCiAgICAidW5pZm9ybSIsCl0KCgpjbGFzcyBSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICAiIiJGcm96ZW4sIGJvdW5kZWQgY29uZmlndXJhdGlvbiBmb3IgdGhlIHYzIHN1Y2Nlc3NvciBtZWNoYW5pc20uIiIiCgogICAgc3lzdGVtOiBPbHltcHVzU3lzdGVtQ29uZmlnCiAgICByb3V0aW5nX21vZGU6IFJlbGlhYmlsaXR5Um91dGluZ01vZGUgPSAicmVsaWFiaWxpdHlfc2hyaW5rYWdlIgogICAgcmVsaWFiaWxpdHlfbG9naXRfc3RyZW5ndGg6IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xLjAsIGdlPTAuMCwgbGU9MTYuMCkKICAgIHNocmlua2FnZV90b19hZGFwdGl2ZTogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuNzUsIGdlPTAuMCwgbGU9MS4wKQogICAgcHJvYmFiaWxpdHlfZmxvb3I6IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xZS02LCBndD0wLjAsIGxlPTFlLTIpCgogICAgQG1vZGVsX3ZhbGlkYXRvcihtb2RlPSJhZnRlciIpCiAgICBkZWYgdmFsaWRfbWVjaGFuaXNtKHNlbGYpIC0+IFJlbGlhYmlsaXR5RnVzaW9uQ29uZmlnOgogICAgICAgIGlmIHNlbGYucm91dGluZ19tb2RlIGluIHsicmVsaWFiaWxpdHlfc2hyaW5rYWdlIiwgInJlbGlhYmlsaXR5X29ubHkifToKICAgICAgICAgICAgaWYgc2VsZi5yZWxpYWJpbGl0eV9sb2dpdF9zdHJlbmd0aCA8PSAwLjA6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyZWxpYWJpbGl0eS1hd2FyZSByb3V0aW5nIHJlcXVpcmVzIHBvc2l0aXZlIGxvZ2l0IHN0cmVuZ3RoIikKICAgICAgICAgICAgaWYgc2VsZi5zaHJpbmthZ2VfdG9fYWRhcHRpdmUgPD0gMC4wOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFiaWxpdHktYXdhcmUgcm91dGluZyByZXF1aXJlcyBwb3NpdGl2ZSBhZGFwdGl2ZSBtYXNzIikKICAgICAgICByZXR1cm4gc2VsZgoKCmNsYXNzIFJlbGlhYmlsaXR5TG9zc0NvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICAiIiJKb2ludCB2MyBvYmplY3RpdmUgd2l0aCBleHBsaWNpdCByZWxpYWJpbGl0eSBjYWxpYnJhdGlvbi4iIiIKCiAgICB0cmFuc2l0aW9uX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuNSwgZ2U9MC4wKQogICAgdGVtcG9yYWxfd2VpZ2h0OiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MC4yNSwgZ2U9MC4wKQogICAgdW5jZXJ0YWludHlfd2VpZ2h0OiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MC4xLCBnZT0wLjApCiAgICByaXNrX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMSwgZ2U9MC4wKQogICAgZ2F0ZV9iYWxhbmNlX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMDEsIGdlPTAuMCkKICAgIGNvdW50ZXJmYWN0dWFsX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMDUsIGdlPTAuMCkKICAgIHJlbGlhYmlsaXR5X3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMjUsIGdlPTAuMCkKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiBmaW5pdGVfd2VpZ2h0cyhzZWxmKSAtPiBSZWxpYWJpbGl0eUxvc3NDb25maWc6CiAgICAgICAgaWYgbm90IGFsbChtYXRoLmlzZmluaXRlKHZhbHVlKSBmb3IgdmFsdWUgaW4gc2VsZi5tb2RlbF9kdW1wKCkudmFsdWVzKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsb3NzIHdlaWdodHMgbXVzdCBiZSBmaW5pdGUiKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIGJhc2Vfd2VpZ2h0cyhzZWxmKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgIHZhbHVlcyA9IHNlbGYubW9kZWxfZHVtcCgpCiAgICAgICAgdmFsdWVzLnBvcCgicmVsaWFiaWxpdHlfd2VpZ2h0IikKICAgICAgICByZXR1cm4gdmFsdWVzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dDoKICAgIHN5c3RlbTogT2x5bXB1c1N5c3RlbU91dHB1dAogICAgc2VtYW50aWNfZ2F0ZXM6IFRlbnNvcgogICAgY29tcG9uZW50X3JlbGlhYmlsaXR5X2xvZ2l0czogVGVuc29yCiAgICBjb21wb25lbnRfcmVsaWFiaWxpdHk6IFRlbnNvcgoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUmVsaWFiaWxpdHlGdXNpb25Mb3NzOgogICAgdG90YWw6IFRlbnNvcgogICAgc3lzdGVtOiBPbHltcHVzU3lzdGVtTG9zcwogICAgcmVsaWFiaWxpdHlfY2FsaWJyYXRpb246IFRlbnNvcgoKCmNsYXNzIFJlbGlhYmlsaXR5QXdhcmVPbHltcHVzU3lzdGVtKG5uLk1vZHVsZSk6CiAgICAiIiJSb3V0ZSBmYW1pbHkgc3RhdGVzIHVzaW5nIGNhbGlicmF0ZWQgcmVsaWFiaWxpdHkgd2l0aCB1bmlmb3JtIHNocmlua2FnZS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZykgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAgICAgIHNlbGYuc3lzdGVtID0gT2x5bXB1c1JlZmVyZW5jZVN5c3RlbShjb25maWcuc3lzdGVtKQogICAgICAgIHNlbGYucmVsaWFiaWxpdHlfaGVhZHMgPSBubi5Nb2R1bGVEaWN0KAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICBuYW1lOiBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgICAgIG5uLkxheWVyTm9ybShjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0pLAogICAgICAgICAgICAgICAgICAgIG5uLkxpbmVhcihjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0sIGNvbmZpZy5zeXN0ZW0uZnVzaW9uX2RpbSksCiAgICAgICAgICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICAgICAgICAgIG5uLkxpbmVhcihjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0sIDEpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gc2VsZi5zeXN0ZW0uY29tcG9uZW50X25hbWVzCiAgICAgICAgICAgIH0KICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgY29tcG9uZW50X25hbWVzKHNlbGYpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgICAgICByZXR1cm4gc2VsZi5zeXN0ZW0uY29tcG9uZW50X25hbWVzCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICB0b2tlbnM6IFRlbnNvciwKICAgICAgICBpbWFnZTogVGVuc29yLAogICAgICAgIHN0YXRlczogVGVuc29yLAogICAgICAgIGFjdGlvbnM6IFRlbnNvciwKICAgICAgICBhbGxvd2VkX3RyYW5zaXRpb25zOiBUZW5zb3IsCiAgICAgICAgcmlza19zY29yZTogVGVuc29yLAogICAgICAgICosCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgIGRvbWFpbl9pZHM6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgIHRlbXBvcmFsX21hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgIHRpbWVzdGFtcDogVGVuc29yIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgZXh0ZW5zaW9uX2ZlYXR1cmVzOiBkaWN0W3N0ciwgVGVuc29yXSB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dDoKICAgICAgICBiYXNlID0gc2VsZi5zeXN0ZW0oCiAgICAgICAgICAgIHRva2VucywKICAgICAgICAgICAgaW1hZ2UsCiAgICAgICAgICAgIHN0YXRlcywKICAgICAgICAgICAgYWN0aW9ucywKICAgICAgICAgICAgYWxsb3dlZF90cmFuc2l0aW9ucywKICAgICAgICAgICAgcmlza19zY29yZSwKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2ssCiAgICAgICAgICAgIGRvbWFpbl9pZHM9ZG9tYWluX2lkcywKICAgICAgICAgICAgdGVtcG9yYWxfbWFzaz10ZW1wb3JhbF9tYXNrLAogICAgICAgICAgICB0aW1lc3RhbXA9dGltZXN0YW1wLAogICAgICAgICAgICBleHRlbnNpb25fZmVhdHVyZXM9ZXh0ZW5zaW9uX2ZlYXR1cmVzLAogICAgICAgICkKICAgICAgICB2ZWN0b3JzID0gYmFzZS5jb21wb25lbnRfc3RhdGVzCiAgICAgICAgcmVsaWFiaWxpdHlfbG9naXRzID0gdG9yY2guY2F0KAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgICBzZWxmLnJlbGlhYmlsaXR5X2hlYWRzW25hbWVdKHZlY3RvcnNbOiwgaW5kZXhdKQogICAgICAgICAgICAgICAgZm9yIGluZGV4LCBuYW1lIGluIGVudW1lcmF0ZShzZWxmLmNvbXBvbmVudF9uYW1lcykKICAgICAgICAgICAgXSwKICAgICAgICAgICAgZGltPS0xLAogICAgICAgICkKICAgICAgICByZWxpYWJpbGl0eSA9IHJlbGlhYmlsaXR5X2xvZ2l0cy5zaWdtb2lkKCkKICAgICAgICBnYXRlcyA9IHNlbGYuX3JvdXRlKGJhc2UuZmFtaWx5X2dhdGVzLCByZWxpYWJpbGl0eSwgcmVsaWFiaWxpdHlfbG9naXRzKQogICAgICAgIHN5c3RlbV9vdXRwdXQgPSBzZWxmLl9yZWNvbXB1dGVfc3lzdGVtX291dHB1dCgKICAgICAgICAgICAgYmFzZSwKICAgICAgICAgICAgZ2F0ZXMsCiAgICAgICAgICAgIGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAgICAgICAgICAgIHJpc2tfc2NvcmUsCiAgICAgICAgKQogICAgICAgIHJldHVybiBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dCgKICAgICAgICAgICAgc3lzdGVtPXN5c3RlbV9vdXRwdXQsCiAgICAgICAgICAgIHNlbWFudGljX2dhdGVzPWJhc2UuZmFtaWx5X2dhdGVzLAogICAgICAgICAgICBjb21wb25lbnRfcmVsaWFiaWxpdHlfbG9naXRzPXJlbGlhYmlsaXR5X2xvZ2l0cywKICAgICAgICAgICAgY29tcG9uZW50X3JlbGlhYmlsaXR5PXJlbGlhYmlsaXR5LAogICAgICAgICkKCiAgICBkZWYgX3JvdXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgc2VtYW50aWNfZ2F0ZXM6IFRlbnNvciwKICAgICAgICByZWxpYWJpbGl0eTogVGVuc29yLAogICAgICAgIHJlbGlhYmlsaXR5X2xvZ2l0czogVGVuc29yLAogICAgKSAtPiBUZW5zb3I6CiAgICAgICAgdW5pZm9ybSA9IHRvcmNoLmZ1bGxfbGlrZShzZW1hbnRpY19nYXRlcywgMS4wIC8gc2VtYW50aWNfZ2F0ZXMuc2hhcGVbLTFdKQogICAgICAgIG1vZGUgPSBzZWxmLmNvbmZpZy5yb3V0aW5nX21vZGUKICAgICAgICBpZiBtb2RlID09ICJ1bmlmb3JtIjoKICAgICAgICAgICAgIyBQcmVzZXJ2ZSB0aGUgZnVsbCBtYXRjaGVkIGdyYXBoIHdoaWxlIGRpc2FibGluZyBib3RoIGxlYXJuZWQgc2lnbmFscy4KICAgICAgICAgICAgcmV0dXJuIHVuaWZvcm0gKyAwLjAgKiAoc2VtYW50aWNfZ2F0ZXMgKyByZWxpYWJpbGl0eV9sb2dpdHMpCiAgICAgICAgaWYgbW9kZSA9PSAic2VtYW50aWMiOgogICAgICAgICAgICByZXR1cm4gc2VtYW50aWNfZ2F0ZXMgKyAwLjAgKiByZWxpYWJpbGl0eV9sb2dpdHMKCiAgICAgICAgZmxvb3IgPSBzZWxmLmNvbmZpZy5wcm9iYWJpbGl0eV9mbG9vcgogICAgICAgIHJlbGlhYmlsaXR5X3Rlcm0gPSAoCiAgICAgICAgICAgIHNlbGYuY29uZmlnLnJlbGlhYmlsaXR5X2xvZ2l0X3N0cmVuZ3RoICogcmVsaWFiaWxpdHkuY2xhbXBfbWluKGZsb29yKS5sb2coKQogICAgICAgICkKICAgICAgICBpZiBtb2RlID09ICJyZWxpYWJpbGl0eV9vbmx5IjoKICAgICAgICAgICAgYWRhcHRpdmUgPSByZWxpYWJpbGl0eV90ZXJtLnNvZnRtYXgoZGltPS0xKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbWFudGljX3Rlcm0gPSBzZW1hbnRpY19nYXRlcy5jbGFtcF9taW4oZmxvb3IpLmxvZygpCiAgICAgICAgICAgIGFkYXB0aXZlID0gKHNlbWFudGljX3Rlcm0gKyByZWxpYWJpbGl0eV90ZXJtKS5zb2Z0bWF4KGRpbT0tMSkKICAgICAgICBtYXNzID0gc2VsZi5jb25maWcuc2hyaW5rYWdlX3RvX2FkYXB0aXZlCiAgICAgICAgcmV0dXJuICgxLjAgLSBtYXNzKSAqIHVuaWZvcm0gKyBtYXNzICogYWRhcHRpdmUKCiAgICBkZWYgX3JlY29tcHV0ZV9zeXN0ZW1fb3V0cHV0KAogICAgICAgIHNlbGYsCiAgICAgICAgYmFzZTogT2x5bXB1c1N5c3RlbU91dHB1dCwKICAgICAgICBnYXRlczogVGVuc29yLAogICAgICAgIGFsbG93ZWRfdHJhbnNpdGlvbnM6IFRlbnNvciwKICAgICAgICByaXNrX3Njb3JlOiBUZW5zb3IsCiAgICApIC0+IE9seW1wdXNTeXN0ZW1PdXRwdXQ6CiAgICAgICAgdmVjdG9ycyA9IGJhc2UuY29tcG9uZW50X3N0YXRlcwogICAgICAgIHdlaWdodGVkID0gdmVjdG9ycyAqIGdhdGVzLnVuc3F1ZWV6ZSgtMSkKICAgICAgICBwcmVfbm9ybWFsaXplZCA9IHdlaWdodGVkLnN1bShkaW09MSkKICAgICAgICBmdXNlZCA9IHNlbGYuc3lzdGVtLmZ1c2lvbl9ub3JtKHByZV9ub3JtYWxpemVkKQoKICAgICAgICBub3JtYWxpemVkX3ZlY3RvcnMgPSBubi5mdW5jdGlvbmFsLm5vcm1hbGl6ZSh2ZWN0b3JzLCBkaW09LTEpCiAgICAgICAgbm9ybWFsaXplZF9jZW50cm9pZCA9IG5uLmZ1bmN0aW9uYWwubm9ybWFsaXplKAogICAgICAgICAgICAobm9ybWFsaXplZF92ZWN0b3JzICogZ2F0ZXMudW5zcXVlZXplKC0xKSkuc3VtKGRpbT0xKSwgZGltPS0xCiAgICAgICAgKQogICAgICAgIGNvc2luZV90b19jZW50cm9pZCA9IChub3JtYWxpemVkX3ZlY3RvcnMgKiBub3JtYWxpemVkX2NlbnRyb2lkLnVuc3F1ZWV6ZSgxKSkuc3VtKGRpbT0tMSkKICAgICAgICBkaXNhZ3JlZW1lbnQgPSAoKDEuMCAtIGNvc2luZV90b19jZW50cm9pZCkuY2xhbXAoMC4wLCAyLjApICogZ2F0ZXMpLnN1bShkaW09LTEpIC8gMi4wCiAgICAgICAgcmlza19kaXNhZ3JlZW1lbnQgPSAoCiAgICAgICAgICAgIGRpc2FncmVlbWVudCBpZiBzZWxmLmNvbmZpZy5zeXN0ZW0uZGlzYWdyZWVtZW50X3Jpc2sgZWxzZSBkaXNhZ3JlZW1lbnQgKiAwLjAKICAgICAgICApCiAgICAgICAgcmlza19mZWF0dXJlcyA9IHRvcmNoLmNhdCgoZnVzZWQsIHJpc2tfZGlzYWdyZWVtZW50LnVuc3F1ZWV6ZSgtMSkpLCBkaW09LTEpCiAgICAgICAgaW50cmluc2ljX3Jpc2sgPSBzZWxmLnN5c3RlbS5pbnRyaW5zaWNfcmlza19oZWFkKHJpc2tfZmVhdHVyZXMpLnNxdWVlemUoLTEpLnNpZ21vaWQoKQoKICAgICAgICBiYXRjaCA9IGZ1c2VkLnNoYXBlWzBdCiAgICAgICAgaG9zdF9yaXNrID0gcmlza19zY29yZS50byhkZXZpY2U9ZnVzZWQuZGV2aWNlLCBkdHlwZT1mdXNlZC5kdHlwZSkKICAgICAgICBpZiBob3N0X3Jpc2suc2hhcGUgIT0gKGJhdGNoLCkgb3Igbm90IGJvb2wodG9yY2guaXNmaW5pdGUoaG9zdF9yaXNrKS5hbGwoKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJpc2sgc2NvcmUgbXVzdCBjb250YWluIG9uZSBmaW5pdGUgdmFsdWUgcGVyIGJhdGNoIGl0ZW0iKQogICAgICAgIGlmIGJvb2woKChob3N0X3Jpc2sgPCAwLjApIHwgKGhvc3RfcmlzayA+IDEuMCkpLmFueSgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmlzayBzY29yZXMgbXVzdCBmYWxsIGluIFswLCAxXSIpCiAgICAgICAgZWZmZWN0aXZlX3Jpc2sgPSAxLjAgLSAoMS4wIC0gaG9zdF9yaXNrKSAqICgxLjAgLSBpbnRyaW5zaWNfcmlzaykKICAgICAgICBhaW9uID0gc2VsZi5zeXN0ZW0uYWlvbihzZWxmLnN5c3RlbS5haW9uX3N0YXRlKGZ1c2VkKSwgYWxsb3dlZF90cmFuc2l0aW9ucywgZWZmZWN0aXZlX3Jpc2spCiAgICAgICAgcmF3X2RlY2lzaW9uID0gc2VsZi5zeXN0ZW0uZGVjaXNpb25faGVhZChmdXNlZCkKCiAgICAgICAgbGVhdmVfb25lX291dDogbGlzdFtUZW5zb3JdID0gW10KICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2UodmVjdG9ycy5zaGFwZVsxXSk6CiAgICAgICAgICAgIHJlbWFpbmluZ19tYXNzID0gKDEuMCAtIGdhdGVzWzosIGluZGV4XSkuY2xhbXBfbWluKDFlLTYpLnVuc3F1ZWV6ZSgtMSkKICAgICAgICAgICAgY291bnRlcmZhY3R1YWwgPSBzZWxmLnN5c3RlbS5mdXNpb25fbm9ybSgKICAgICAgICAgICAgICAgIChwcmVfbm9ybWFsaXplZCAtIHdlaWdodGVkWzosIGluZGV4XSkgLyByZW1haW5pbmdfbWFzcwogICAgICAgICAgICApCiAgICAgICAgICAgIGxlYXZlX29uZV9vdXQuYXBwZW5kKHNlbGYuc3lzdGVtLmRlY2lzaW9uX2hlYWQoY291bnRlcmZhY3R1YWwpKQogICAgICAgIGNvdW50ZXJmYWN0dWFsX2xvZ2l0cyA9IHRvcmNoLnN0YWNrKGxlYXZlX29uZV9vdXQsIGRpbT0xKQogICAgICAgIGRlY2lzaW9uX2F2YWlsYWJsZSA9IH5haW9uLmZvcmNlZF9zdG9wCiAgICAgICAgZGVjaXNpb24gPSB0b3JjaC53aGVyZSgKICAgICAgICAgICAgZGVjaXNpb25fYXZhaWxhYmxlLnVuc3F1ZWV6ZSgtMSksIHJhd19kZWNpc2lvbiwgdG9yY2guemVyb3NfbGlrZShyYXdfZGVjaXNpb24pCiAgICAgICAgKQogICAgICAgIHVuY2VydGFpbnR5ID0gbm4uZnVuY3Rpb25hbC5zb2Z0cGx1cyhzZWxmLnN5c3RlbS51bmNlcnRhaW50eV9oZWFkKGZ1c2VkKS5zcXVlZXplKC0xKSkgKyAxZS02CiAgICAgICAgcmV0dXJuIHJlcGxhY2UoCiAgICAgICAgICAgIGJhc2UsCiAgICAgICAgICAgIGFpb249YWlvbiwKICAgICAgICAgICAgZmFtaWx5X2dhdGVzPWdhdGVzLAogICAgICAgICAgICBmdXNlZF9zdGF0ZT1mdXNlZCwKICAgICAgICAgICAgY291bnRlcmZhY3R1YWxfZGVjaXNpb25fbG9naXRzPWNvdW50ZXJmYWN0dWFsX2xvZ2l0cywKICAgICAgICAgICAgcmF3X2RlY2lzaW9uX2xvZ2l0cz1yYXdfZGVjaXNpb24sCiAgICAgICAgICAgIGRlY2lzaW9uX2xvZ2l0cz1kZWNpc2lvbiwKICAgICAgICAgICAgZGVjaXNpb25fYXZhaWxhYmxlPWRlY2lzaW9uX2F2YWlsYWJsZSwKICAgICAgICAgICAgdW5jZXJ0YWludHk9dW5jZXJ0YWludHksCiAgICAgICAgICAgIGZhbWlseV9kaXNhZ3JlZW1lbnQ9ZGlzYWdyZWVtZW50LAogICAgICAgICAgICBpbnRyaW5zaWNfcmlzaz1pbnRyaW5zaWNfcmlzaywKICAgICAgICAgICAgZWZmZWN0aXZlX3Jpc2s9ZWZmZWN0aXZlX3Jpc2ssCiAgICAgICAgKQoKCmRlZiByZWxpYWJpbGl0eV9mdXNpb25fbG9zcygKICAgIG91dHB1dDogUmVsaWFiaWxpdHlGdXNpb25PdXRwdXQsCiAgICAqLAogICAgZGVjaXNpb25fdGFyZ2V0czogVGVuc29yLAogICAgdHJhbnNpdGlvbl90YXJnZXRzOiBUZW5zb3IsCiAgICBuZXh0X3N0YXRlX3RhcmdldHM6IFRlbnNvciwKICAgIGNvbXBvbmVudF9yZWxpYWJpbGl0eV90YXJnZXRzOiBUZW5zb3IsCiAgICBpbnRyaW5zaWNfcmlza190YXJnZXRzOiBUZW5zb3IgfCBOb25lID0gTm9uZSwKICAgIHRlbXBvcmFsX21hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgY29uZmlnOiBSZWxpYWJpbGl0eUxvc3NDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBSZWxpYWJpbGl0eUZ1c2lvbkxvc3M6CiAgICAiIiJPcHRpbWl6ZSB0YXNrIGJlaGF2aW9yIGFuZCBpbmRlcGVuZGVudGx5IG9ic2VydmFibGUgcmVsaWFiaWxpdHkgdGFyZ2V0cy4iIiIKCiAgICB3ZWlnaHRzID0gY29uZmlnIG9yIFJlbGlhYmlsaXR5TG9zc0NvbmZpZygpCiAgICB0YXJnZXRzID0gY29tcG9uZW50X3JlbGlhYmlsaXR5X3RhcmdldHMudG8oCiAgICAgICAgZGV2aWNlPW91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuZGV2aWNlLAogICAgICAgIGR0eXBlPW91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuZHR5cGUsCiAgICApCiAgICBpZiB0YXJnZXRzLnNoYXBlICE9IG91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFiaWxpdHkgdGFyZ2V0cyBtdXN0IG1hdGNoIFtiYXRjaCwgY29tcG9uZW50XSIpCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZSh0YXJnZXRzKS5hbGwoKSkgb3IgYm9vbCgoKHRhcmdldHMgPCAwLjApIHwgKHRhcmdldHMgPiAxLjApKS5hbnkoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFiaWxpdHkgdGFyZ2V0cyBtdXN0IGJlIGZpbml0ZSB2YWx1ZXMgaW4gWzAsIDFdIikKICAgIHN5c3RlbV9sb3NzID0gb2x5bXB1c19zeXN0ZW1fbG9zcygKICAgICAgICBvdXRwdXQuc3lzdGVtLAogICAgICAgIGRlY2lzaW9uX3RhcmdldHM9ZGVjaXNpb25fdGFyZ2V0cywKICAgICAgICB0cmFuc2l0aW9uX3RhcmdldHM9dHJhbnNpdGlvbl90YXJnZXRzLAogICAgICAgIG5leHRfc3RhdGVfdGFyZ2V0cz1uZXh0X3N0YXRlX3RhcmdldHMsCiAgICAgICAgaW50cmluc2ljX3Jpc2tfdGFyZ2V0cz1pbnRyaW5zaWNfcmlza190YXJnZXRzLAogICAgICAgIHRlbXBvcmFsX21hc2s9dGVtcG9yYWxfbWFzaywKICAgICAgICAqKndlaWdodHMuYmFzZV93ZWlnaHRzKCksCiAgICApCiAgICByZWxpYWJpbGl0eV9jYWxpYnJhdGlvbiA9IG5uLmZ1bmN0aW9uYWwuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoCiAgICAgICAgb3V0cHV0LmNvbXBvbmVudF9yZWxpYWJpbGl0eV9sb2dpdHMsCiAgICAgICAgdGFyZ2V0cywKICAgICkKICAgIHRvdGFsID0gc3lzdGVtX2xvc3MudG90YWwgKyB3ZWlnaHRzLnJlbGlhYmlsaXR5X3dlaWdodCAqIHJlbGlhYmlsaXR5X2NhbGlicmF0aW9uCiAgICByZXR1cm4gUmVsaWFiaWxpdHlGdXNpb25Mb3NzKAogICAgICAgIHRvdGFsPXRvdGFsLAogICAgICAgIHN5c3RlbT1zeXN0ZW1fbG9zcywKICAgICAgICByZWxpYWJpbGl0eV9jYWxpYnJhdGlvbj1yZWxpYWJpbGl0eV9jYWxpYnJhdGlvbiwKICAgICkK', 'olympus/models/robust_fusion_v5.py': 'IiIiTWF0Y2hlZCByb3V0aW5nLWZlYXR1cmUgYW5kIHRlbXBvcmFsLXNraXAgYWJsYXRpb25zIG9uIGZyb3plbiB2NCBiYW5rcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjb3B5CgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKZnJvbSBvbHltcHVzLm1vZGVscy5vcmFjbGVfZnVzaW9uX3Y0IGltcG9ydCBub3JtYWxpemVfcXVhbGl0eSwgcm91dGVyX2ZlYXR1cmVzCgpST1VURVMgPSAoCiAgICAidW5pZm9ybSIsCiAgICAiZml4ZWQiLAogICAgInNlbWFudGljIiwKICAgICJwcmVkaWN0aW9uX29ubHkiLAogICAgInV0aWxpdHkiLAogICAgImNvcnJ1cHRpb24iLAogICAgImJsZW5kIiwKICAgICJvcmFjbGVfdGFzayIsCikKVEVNUE9SQUwgPSAoInY0IiwgImRpcmVjdCIsICJyZXNpZHVhbCIsICJwZXJzaXN0ZW5jZSIpCgoKY2xhc3MgSGVhZHMobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZml4ZWQgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoNSkpCiAgICAgICAgbmV0ID0gbm4uU2VxdWVudGlhbChubi5MYXllck5vcm0oMTA1KSwgbm4uTGluZWFyKDEwNSwgMzIpLCBubi5HRUxVKCksIG5uLkxpbmVhcigzMiwgNSkpCiAgICAgICAgc2VsZi5yb3V0aW5nID0gbm4uTW9kdWxlRGljdCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgbmFtZTogY29weS5kZWVwY29weShuZXQpCiAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiAoInNlbWFudGljIiwgInByZWRpY3Rpb25fb25seSIsICJ1dGlsaXR5IiwgImNvcnJ1cHRpb24iKQogICAgICAgICAgICB9CiAgICAgICAgKQogICAgICAgICMgSWRlbnRpY2FsIGluaXRpYWxpemF0aW9uL3BhcmFtZXRlcnMvdXBkYXRlcyBmb3IgdGhlIHRlbXBvcmFsIHBhaXIuCiAgICAgICAgdGVtcG9yYWwgPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcig4OCwgNjQpLCBubi5HRUxVKCksIG5uLkxpbmVhcig2NCwgNSkpCiAgICAgICAgYXNzZXJ0IGlzaW5zdGFuY2UodGVtcG9yYWxbLTFdLCBubi5MaW5lYXIpCiAgICAgICAgbm4uaW5pdC56ZXJvc18odGVtcG9yYWxbLTFdLndlaWdodCkKICAgICAgICBubi5pbml0Lnplcm9zXyh0ZW1wb3JhbFstMV0uYmlhcykKICAgICAgICBzZWxmLnRlbXBvcmFsID0gbm4uTW9kdWxlRGljdCgKICAgICAgICAgICAge25hbWU6IGNvcHkuZGVlcGNvcHkodGVtcG9yYWwpIGZvciBuYW1lIGluICgiZGlyZWN0IiwgInJlc2lkdWFsIil9CiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKAogICAgICAgIHNlbGYsIHo6IFRlbnNvciwgbG9naXRzOiBUZW5zb3IsIHN0YXRlczogVGVuc29yLCBhY3Rpb25zOiBUZW5zb3IKICAgICkgLT4gdHVwbGVbZGljdFtzdHIsIFRlbnNvcl0sIGRpY3Rbc3RyLCBUZW5zb3JdXToKICAgICAgICBmZWF0dXJlcyA9IHJvdXRlcl9mZWF0dXJlcyh6LCBsb2dpdHMpCiAgICAgICAgIyBTYW1lIDEwNS1kaW1lbnNpb25hbCBuZXR3b3JrOyB6ZXJvaW5nIE9OTFkgdGhlIGxhdGVudCBmZWF0dXJlcyBpc29sYXRlcwogICAgICAgICMgcmVwcmVzZW50YXRpb24gZGVwZW5kZW5jZSB3aXRob3V0IGNoYW5naW5nIHBhcmFtZXRlciBjb3VudHMuCiAgICAgICAgcHJlZGljdGlvbl9mZWF0dXJlcyA9IHRvcmNoLmNhdCgodG9yY2guemVyb3NfbGlrZShmZWF0dXJlc1s6LCA6ODBdKSwgZmVhdHVyZXNbOiwgODA6XSksIC0xKQogICAgICAgIHNjb3JlcyA9IHsKICAgICAgICAgICAgbmFtZTogbmV0KHByZWRpY3Rpb25fZmVhdHVyZXMgaWYgbmFtZSA9PSAicHJlZGljdGlvbl9vbmx5IiBlbHNlIGZlYXR1cmVzKQogICAgICAgICAgICBmb3IgbmFtZSwgbmV0IGluIHNlbGYucm91dGluZy5pdGVtcygpCiAgICAgICAgfQogICAgICAgIHNjb3Jlc1siZml4ZWQiXSA9IHNlbGYuZml4ZWQuZXhwYW5kKGxlbih6KSwgLTEpCiAgICAgICAgY29udGV4dCA9IHouZmxhdHRlbigxKS5kZXRhY2goKVs6LCBOb25lXS5leHBhbmQoLTEsIHN0YXRlcy5zaGFwZVsxXSwgLTEpCiAgICAgICAgcmF3ID0gdG9yY2guY2F0KChzdGF0ZXMsIGFjdGlvbnMsIGNvbnRleHQpLCAtMSkKICAgICAgICB0ZW1wb3JhbCA9IHsKICAgICAgICAgICAgbmFtZTogbmV0KHJhdykgKyAoc3RhdGVzIGlmIG5hbWUgPT0gInJlc2lkdWFsIiBlbHNlIDApCiAgICAgICAgICAgIGZvciBuYW1lLCBuZXQgaW4gc2VsZi50ZW1wb3JhbC5pdGVtcygpCiAgICAgICAgfQogICAgICAgIHJldHVybiBzY29yZXMsIHRlbXBvcmFsCgoKZGVmIHJvdXRlZCgKICAgIGxvZ2l0czogVGVuc29yLCBzY29yZXM6IGRpY3Rbc3RyLCBUZW5zb3JdLCBsYWJlbHM6IFRlbnNvciB8IE5vbmUgPSBOb25lCikgLT4gZGljdFtzdHIsIFRlbnNvcl06CiAgICBwID0gbG9naXRzLnNvZnRtYXgoLTEpCiAgICBnYXRlcyA9IHtuYW1lOiBzY29yZXNbbmFtZV0uc29mdG1heCgtMSkgZm9yIG5hbWUgaW4gKCJmaXhlZCIsICJzZW1hbnRpYyIsICJwcmVkaWN0aW9uX29ubHkiKX0KICAgIGdhdGVzLnVwZGF0ZSgKICAgICAgICB7bmFtZTogbm9ybWFsaXplX3F1YWxpdHkoc2NvcmVzW25hbWVdLnNpZ21vaWQoKSkgZm9yIG5hbWUgaW4gKCJ1dGlsaXR5IiwgImNvcnJ1cHRpb24iKX0KICAgICkKICAgIGdhdGVzWyJ1bmlmb3JtIl0gPSB0b3JjaC5vbmVzX2xpa2UocFsuLi4sIDBdKSAvIDUKICAgIGdhdGVzWyJibGVuZCJdID0gKGdhdGVzWyJwcmVkaWN0aW9uX29ubHkiXSArIGdhdGVzWyJ1dGlsaXR5Il0pIC8gMgogICAgdmFsdWVzID0ge25hbWU6IChnYXRlWy4uLiwgTm9uZV0gKiBwKS5zdW0oMSkgZm9yIG5hbWUsIGdhdGUgaW4gZ2F0ZXMuaXRlbXMoKX0KICAgIGlmIGxhYmVscyBpcyBub3QgTm9uZToKICAgICAgICBiZXN0ID0gcC5nYXRoZXIoMiwgbGFiZWxzWzosIE5vbmUsIE5vbmVdLmV4cGFuZCgtMSwgNSwgMSkpLnNxdWVlemUoLTEpLmFyZ21heCgtMSkKICAgICAgICB2YWx1ZXNbIm9yYWNsZV90YXNrIl0gPSBwW3RvcmNoLmFyYW5nZShsZW4ocCkpLCBiZXN0XQogICAgcmV0dXJuIHZhbHVlcwo=', 'olympus/models/runtime.py': 'IiIiUmVzb3VyY2UtZ292ZXJuZWQgcnVudGltZSBhbmQgaW1tdXRhYmxlIGNoZWNrcG9pbnRzIGZvciB0aGUgT2x5bXB1cyBzeXN0ZW0uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBJdGVyYXRvciwgTWFwcGluZwpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyLCBudWxsY29udGV4dApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGVtcGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBMaXRlcmFsLCBjYXN0CgppbXBvcnQgdG9yY2gKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2VzIGltcG9ydCBSZXNvdXJjZUdvdmVybm9yCmZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgKICAgIE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICBPbHltcHVzU3lzdGVtQ29uZmlnLAogICAgT2x5bXB1c1N5c3RlbUxvc3MsCiAgICBPbHltcHVzU3lzdGVtTG9zc0NvbmZpZywKICAgIE9seW1wdXNTeXN0ZW1PdXRwdXQsCiAgICBidWlsZF9vbHltcHVzX3N5c3RlbSwKICAgIG9seW1wdXNfc3lzdGVtX2xvc3MsCikKClJ1bnRpbWVEZXZpY2UgPSBMaXRlcmFsWyJhdXRvIiwgImNwdSIsICJjdWRhIiwgIm1wcyJdClJ1bnRpbWVJbnB1dCA9IFRlbnNvciB8IGRpY3Rbc3RyLCBUZW5zb3JdCgoKY2xhc3MgT2x5bXB1c1J1bnRpbWVDb25maWcoU3RyaWN0TW9kZWwpOgogICAgIiIiRXhlY3V0aW9uIGNvbnRyb2xzIHRoYXQgYXJlIHNlcGFyYXRlIGZyb20gbGVhcm5lZCBtb2RlbCBjb25maWd1cmF0aW9uLiIiIgoKICAgIGRldmljZTogUnVudGltZURldmljZSA9ICJhdXRvIgogICAgbWl4ZWRfcHJlY2lzaW9uOiBib29sID0gRmFsc2UKICAgIGNvbXBpbGVfbW9kZWw6IGJvb2wgPSBGYWxzZQogICAgZGV0ZXJtaW5pc3RpYzogYm9vbCA9IFRydWUKICAgIHJlc291cmNlX3ByZWZsaWdodDogYm9vbCA9IFRydWUKICAgIHdvcmtsb2FkX2xvY2s6IFBhdGggPSBQYXRoKCJhcnRpZmFjdHMvbG9ja3Mvb2x5bXB1cy1zeXN0ZW0ubG9jayIpCiAgICBtaW5fYXZhaWxhYmxlX2J5dGVzOiBpbnQgPSBGaWVsZChkZWZhdWx0PTIgKiAxMDI0KiozLCBnZT0yNTYgKiAxMDI0KioyKQogICAgbWF4X3N3YXBfZnJhY3Rpb246IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0wLjkwLCBndD0wLjAsIGxlPTEuMCkKCgpjbGFzcyBPbHltcHVzQ2hlY2twb2ludFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogTGl0ZXJhbFsxXSA9IDEKICAgIHBhdGg6IHN0cgogICAgc2hhMjU2OiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXlswLTlhLWZdezY0fSQiKQogICAgcGFyYW1ldGVyczogaW50ID0gRmllbGQoZ3Q9MCkKICAgIHN5c3RlbV9jb25maWc6IE9seW1wdXNTeXN0ZW1Db25maWcKICAgIG1ldGFkYXRhOiBkaWN0W3N0ciwgc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgT2x5bXB1c0luZmVyZW5jZVJlc3VsdDoKICAgIG91dHB1dDogT2x5bXB1c1N5c3RlbU91dHB1dAogICAgZWxhcHNlZF9zZWNvbmRzOiBmbG9hdAogICAgZGV2aWNlOiBzdHIKICAgIG1peGVkX3ByZWNpc2lvbl9lZmZlY3RpdmU6IGJvb2wKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE9seW1wdXNUcmFpblN0ZXBSZXN1bHQ6CiAgICBvdXRwdXQ6IE9seW1wdXNTeXN0ZW1PdXRwdXQKICAgIGxvc3M6IE9seW1wdXNTeXN0ZW1Mb3NzCiAgICBncmFkaWVudF9ub3JtOiBmbG9hdAoKCmRlZiBfc2hhMjU2KHBheWxvYWQ6IGJ5dGVzKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkKCgpkZWYgcmVzb2x2ZV9vbHltcHVzX2RldmljZShyZXF1ZXN0ZWQ6IFJ1bnRpbWVEZXZpY2UpIC0+IHRvcmNoLmRldmljZToKICAgIGlmIHJlcXVlc3RlZCA9PSAiYXV0byI6CiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgiY3VkYSIpCiAgICAgICAgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKCJtcHMiKQogICAgICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImNwdSIpCiAgICBpZiByZXF1ZXN0ZWQgPT0gImN1ZGEiIGFuZCBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNVREEgd2FzIHJlcXVlc3RlZCBidXQgaXMgdW5hdmFpbGFibGUiKQogICAgaWYgcmVxdWVzdGVkID09ICJtcHMiIGFuZCBub3QgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTVBTIHdhcyByZXF1ZXN0ZWQgYnV0IGlzIHVuYXZhaWxhYmxlIikKICAgIHJldHVybiB0b3JjaC5kZXZpY2UocmVxdWVzdGVkKQoKCmRlZiBfbW92ZV9pbnB1dHMoCiAgICBpbnB1dHM6IE1hcHBpbmdbc3RyLCBSdW50aW1lSW5wdXRdLCBkZXZpY2U6IHRvcmNoLmRldmljZQopIC0+IGRpY3Rbc3RyLCBSdW50aW1lSW5wdXRdOgogICAgbW92ZWQ6IGRpY3Rbc3RyLCBSdW50aW1lSW5wdXRdID0ge30KICAgIGZvciBuYW1lLCB2YWx1ZSBpbiBpbnB1dHMuaXRlbXMoKToKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBUZW5zb3IpOgogICAgICAgICAgICBtb3ZlZFtuYW1lXSA9IHZhbHVlLnRvKGRldmljZT1kZXZpY2UpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbW92ZWRbbmFtZV0gPSB7a2V5OiB0ZW5zb3IudG8oZGV2aWNlPWRldmljZSkgZm9yIGtleSwgdGVuc29yIGluIHZhbHVlLml0ZW1zKCl9CiAgICByZXR1cm4gbW92ZWQKCgpjbGFzcyBPbHltcHVzU3lzdGVtUnVudGltZToKICAgICIiIkJvdW5kZWQgaW5mZXJlbmNlIHJ1bnRpbWUgd2l0aCBleHBsaWNpdCByZXNvdXJjZSBhbmQgcHJlY2lzaW9uIGNvbnRyb2xzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsOiBPbHltcHVzUmVmZXJlbmNlU3lzdGVtLAogICAgICAgIGNvbmZpZzogT2x5bXB1c1J1bnRpbWVDb25maWcgfCBOb25lID0gTm9uZSwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZyBvciBPbHltcHVzUnVudGltZUNvbmZpZygpCiAgICAgICAgc2VsZi5kZXZpY2UgPSByZXNvbHZlX29seW1wdXNfZGV2aWNlKHNlbGYuY29uZmlnLmRldmljZSkKICAgICAgICB3aXRoIHNlbGYuX3Jlc291cmNlX2NvbnRleHQoKToKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgICAgICBzZWxmLl9jYWxsYWJsZTogbm4uTW9kdWxlID0gc2VsZi5tb2RlbAogICAgICAgICAgICBpZiBzZWxmLmNvbmZpZy5jb21waWxlX21vZGVsOgogICAgICAgICAgICAgICAgc2VsZi5fY2FsbGFibGUgPSBjYXN0KG5uLk1vZHVsZSwgdG9yY2guY29tcGlsZShzZWxmLm1vZGVsLCBkeW5hbWljPUZhbHNlKSkKCiAgICBkZWYgX3Jlc291cmNlX2NvbnRleHQoc2VsZikgLT4gUmVzb3VyY2VHb3Zlcm5vciB8IG51bGxjb250ZXh0W05vbmVdOgogICAgICAgIGlmIG5vdCBzZWxmLmNvbmZpZy5yZXNvdXJjZV9wcmVmbGlnaHQ6CiAgICAgICAgICAgIHJldHVybiBudWxsY29udGV4dCgpCiAgICAgICAgcmV0dXJuIFJlc291cmNlR292ZXJub3IoCiAgICAgICAgICAgIHNlbGYuY29uZmlnLndvcmtsb2FkX2xvY2ssCiAgICAgICAgICAgIG1pbl9hdmFpbGFibGVfYnl0ZXM9c2VsZi5jb25maWcubWluX2F2YWlsYWJsZV9ieXRlcywKICAgICAgICAgICAgbWF4X3N3YXBfZnJhY3Rpb249c2VsZi5jb25maWcubWF4X3N3YXBfZnJhY3Rpb24sCiAgICAgICAgKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG1peGVkX3ByZWNpc2lvbl9lZmZlY3RpdmUoc2VsZikgLT4gYm9vbDoKICAgICAgICByZXR1cm4gc2VsZi5jb25maWcubWl4ZWRfcHJlY2lzaW9uIGFuZCBzZWxmLmRldmljZS50eXBlIGluIHsiY3VkYSIsICJtcHMifQoKICAgIGRlZiBpbmZlcihzZWxmLCBpbnB1dHM6IE1hcHBpbmdbc3RyLCBSdW50aW1lSW5wdXRdKSAtPiBPbHltcHVzSW5mZXJlbmNlUmVzdWx0OgogICAgICAgIHByZWNpc2lvbiA9ICgKICAgICAgICAgICAgdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9c2VsZi5kZXZpY2UudHlwZSwgZHR5cGU9dG9yY2guZmxvYXQxNikKICAgICAgICAgICAgaWYgc2VsZi5taXhlZF9wcmVjaXNpb25fZWZmZWN0aXZlCiAgICAgICAgICAgIGVsc2UgbnVsbGNvbnRleHQoKQogICAgICAgICkKICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHdpdGggKAogICAgICAgICAgICBzZWxmLl9yZXNvdXJjZV9jb250ZXh0KCksCiAgICAgICAgICAgIF9kZXRlcm1pbmlzdGljX2FsZ29yaXRobXMoc2VsZi5jb25maWcuZGV0ZXJtaW5pc3RpYyksCiAgICAgICAgICAgIF9ldmFsdWF0aW9uX21vZGUoc2VsZi5tb2RlbCksCiAgICAgICAgICAgIHByZWNpc2lvbiwKICAgICAgICApOgogICAgICAgICAgICBtb3ZlZCA9IF9tb3ZlX2lucHV0cyhpbnB1dHMsIHNlbGYuZGV2aWNlKQogICAgICAgICAgICBvdXRwdXQgPSBjYXN0KE9seW1wdXNTeXN0ZW1PdXRwdXQsIHNlbGYuX2NhbGxhYmxlKCoqbW92ZWQpKQogICAgICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAogICAgICAgIGlmIG5vdCBib29sKHRvcmNoLmlzZmluaXRlKG91dHB1dC5yYXdfZGVjaXNpb25fbG9naXRzKS5hbGwoKSk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcigiT2x5bXB1cyBpbmZlcmVuY2UgcHJvZHVjZWQgbm9uLWZpbml0ZSBkZWNpc2lvbiBsb2dpdHMiKQogICAgICAgIHJldHVybiBPbHltcHVzSW5mZXJlbmNlUmVzdWx0KAogICAgICAgICAgICBvdXRwdXQ9b3V0cHV0LAogICAgICAgICAgICBlbGFwc2VkX3NlY29uZHM9ZWxhcHNlZCwKICAgICAgICAgICAgZGV2aWNlPXN0cihzZWxmLmRldmljZSksCiAgICAgICAgICAgIG1peGVkX3ByZWNpc2lvbl9lZmZlY3RpdmU9c2VsZi5taXhlZF9wcmVjaXNpb25fZWZmZWN0aXZlLAogICAgICAgICkKCgpAY29udGV4dG1hbmFnZXIKZGVmIF9ldmFsdWF0aW9uX21vZGUobW9kZWw6IG5uLk1vZHVsZSkgLT4gSXRlcmF0b3JbTm9uZV06CiAgICAiIiJSZXN0b3JlIGV2ZXJ5IG1vZHVsZSdzIG1vZGUsIGluY2x1ZGluZyBkZWxpYmVyYXRlbHkgZnJvemVuIHN1Ym1vZHVsZXMuIiIiCgogICAgcHJldmlvdXNfbW9kZXMgPSB0dXBsZSgobW9kdWxlLCBtb2R1bGUudHJhaW5pbmcpIGZvciBtb2R1bGUgaW4gbW9kZWwubW9kdWxlcygpKQogICAgbW9kZWwuZXZhbCgpCiAgICB0cnk6CiAgICAgICAgeWllbGQKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIG1vZHVsZSwgdHJhaW5pbmcgaW4gcHJldmlvdXNfbW9kZXM6CiAgICAgICAgICAgIG1vZHVsZS50cmFpbmluZyA9IHRyYWluaW5nCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiBfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKGVuYWJsZWQ6IGJvb2wpIC0+IEl0ZXJhdG9yW05vbmVdOgogICAgcHJldmlvdXNfZW5hYmxlZCA9IHRvcmNoLmFyZV9kZXRlcm1pbmlzdGljX2FsZ29yaXRobXNfZW5hYmxlZCgpCiAgICBwcmV2aW91c193YXJuX29ubHkgPSB0b3JjaC5pc19kZXRlcm1pbmlzdGljX2FsZ29yaXRobXNfd2Fybl9vbmx5X2VuYWJsZWQoKQogICAgaWYgZW5hYmxlZDoKICAgICAgICB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUsIHdhcm5fb25seT1UcnVlKQogICAgdHJ5OgogICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAgICAgeWllbGQKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgZW5hYmxlZDoKICAgICAgICAgICAgdG9yY2gudXNlX2RldGVybWluaXN0aWNfYWxnb3JpdGhtcygKICAgICAgICAgICAgICAgIHByZXZpb3VzX2VuYWJsZWQsCiAgICAgICAgICAgICAgICB3YXJuX29ubHk9cHJldmlvdXNfd2Fybl9vbmx5LAogICAgICAgICAgICApCgoKZGVmIHRyYWluX29seW1wdXNfc3RlcCgKICAgIG1vZGVsOiBPbHltcHVzUmVmZXJlbmNlU3lzdGVtLAogICAgb3B0aW1pemVyOiB0b3JjaC5vcHRpbS5PcHRpbWl6ZXIsCiAgICBpbnB1dHM6IE1hcHBpbmdbc3RyLCBSdW50aW1lSW5wdXRdLAogICAgdGFyZ2V0czogTWFwcGluZ1tzdHIsIFRlbnNvcl0sCiAgICAqLAogICAgZ3JhZGllbnRfY2xpcF9ub3JtOiBmbG9hdCA9IDEuMCwKICAgIGxvc3NfY29uZmlnOiBPbHltcHVzU3lzdGVtTG9zc0NvbmZpZyB8IE5vbmUgPSBOb25lLAopIC0+IE9seW1wdXNUcmFpblN0ZXBSZXN1bHQ6CiAgICAiIiJFeGVjdXRlIG9uZSBmaW5pdGUsIGNsaXBwZWQgam9pbnQtc3lzdGVtIG9wdGltaXphdGlvbiBzdGVwLiIiIgoKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKGdyYWRpZW50X2NsaXBfbm9ybSkgb3IgZ3JhZGllbnRfY2xpcF9ub3JtIDw9IDAuMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncmFkaWVudCBjbGlwIG5vcm0gbXVzdCBiZSBmaW5pdGUgYW5kIHBvc2l0aXZlIikKICAgIG1vZGVsLnRyYWluKCkKICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIG91dHB1dCA9IGNhc3QoT2x5bXB1c1N5c3RlbU91dHB1dCwgbW9kZWwoKipkaWN0KGlucHV0cykpKQogICAgcmVxdWlyZWRfdGFyZ2V0cyA9IHsKICAgICAgICAiZGVjaXNpb25fdGFyZ2V0cyIsCiAgICAgICAgInRyYW5zaXRpb25fdGFyZ2V0cyIsCiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyIsCiAgICB9CiAgICBvcHRpb25hbF90YXJnZXRzID0geyJpbnRyaW5zaWNfcmlza190YXJnZXRzIn0KICAgIG1pc3NpbmdfdGFyZ2V0cyA9IHNvcnRlZChyZXF1aXJlZF90YXJnZXRzIC0gc2V0KHRhcmdldHMpKQogICAgdW5leHBlY3RlZF90YXJnZXRzID0gc29ydGVkKHNldCh0YXJnZXRzKSAtIHJlcXVpcmVkX3RhcmdldHMgLSBvcHRpb25hbF90YXJnZXRzKQogICAgaWYgbWlzc2luZ190YXJnZXRzIG9yIHVuZXhwZWN0ZWRfdGFyZ2V0czoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAidHJhaW5pbmcgdGFyZ2V0cyBkbyBub3QgbWF0Y2ggdGhlIE9seW1wdXMgbG9zcyBjb250cmFjdDsgIgogICAgICAgICAgICBmIm1pc3Npbmc9e21pc3NpbmdfdGFyZ2V0c30sIHVuZXhwZWN0ZWQ9e3VuZXhwZWN0ZWRfdGFyZ2V0c30iCiAgICAgICAgKQogICAgdGVtcG9yYWxfbWFzayA9IGlucHV0cy5nZXQoInRlbXBvcmFsX21hc2siKQogICAgaWYgdGVtcG9yYWxfbWFzayBpcyBub3QgTm9uZSBhbmQgbm90IGlzaW5zdGFuY2UodGVtcG9yYWxfbWFzaywgVGVuc29yKToKICAgICAgICByYWlzZSBUeXBlRXJyb3IoInRlbXBvcmFsIG1hc2sgbXVzdCBiZSBhIHRlbnNvciIpCiAgICB3ZWlnaHRzID0gKGxvc3NfY29uZmlnIG9yIE9seW1wdXNTeXN0ZW1Mb3NzQ29uZmlnKCkpLm1vZGVsX2R1bXAoKQogICAgbG9zcyA9IG9seW1wdXNfc3lzdGVtX2xvc3MoCiAgICAgICAgb3V0cHV0LAogICAgICAgIGRlY2lzaW9uX3RhcmdldHM9dGFyZ2V0c1siZGVjaXNpb25fdGFyZ2V0cyJdLAogICAgICAgIHRyYW5zaXRpb25fdGFyZ2V0cz10YXJnZXRzWyJ0cmFuc2l0aW9uX3RhcmdldHMiXSwKICAgICAgICBuZXh0X3N0YXRlX3RhcmdldHM9dGFyZ2V0c1sibmV4dF9zdGF0ZV90YXJnZXRzIl0sCiAgICAgICAgaW50cmluc2ljX3Jpc2tfdGFyZ2V0cz10YXJnZXRzLmdldCgiaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyIpLAogICAgICAgIHRlbXBvcmFsX21hc2s9dGVtcG9yYWxfbWFzaywKICAgICAgICAqKndlaWdodHMsCiAgICApCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShsb3NzLnRvdGFsKSk6CiAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCJPbHltcHVzIHRyYWluaW5nIGxvc3MgaXMgbm9uLWZpbml0ZSIpCiAgICBsb3NzLnRvdGFsLmJhY2t3YXJkKCkgICMgdHlwZTogaWdub3JlW25vLXVudHlwZWQtY2FsbF0KICAgIGdyYWRpZW50ID0gbm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgZ3JhZGllbnRfY2xpcF9ub3JtKQogICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoZ3JhZGllbnQpKToKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCJPbHltcHVzIGdyYWRpZW50IG5vcm0gaXMgbm9uLWZpbml0ZSIpCiAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICByZXR1cm4gT2x5bXB1c1RyYWluU3RlcFJlc3VsdCgKICAgICAgICBvdXRwdXQ9b3V0cHV0LAogICAgICAgIGxvc3M9bG9zcywKICAgICAgICBncmFkaWVudF9ub3JtPWZsb2F0KGdyYWRpZW50LmRldGFjaCgpLml0ZW0oKSksCiAgICApCgoKZGVmIHNhdmVfb2x5bXB1c19jaGVja3BvaW50KAogICAgbW9kZWw6IE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICBwYXRoOiBQYXRoLAogICAgKiwKICAgIG1ldGFkYXRhOiBNYXBwaW5nW3N0ciwgc3RyXSB8IE5vbmUgPSBOb25lLAopIC0+IE9seW1wdXNDaGVja3BvaW50UmVjb3JkOgogICAgIiIiV3JpdGUgYSBuZXcgaW1tdXRhYmxlIGNoZWNrcG9pbnQgYW5kIHJldHVybiBpdHMgY29udGVudCBpZGVudGl0eS4iIiIKCiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcigicmVmdXNpbmcgdG8gb3ZlcndyaXRlIGFuIGV4aXN0aW5nIE9seW1wdXMgY2hlY2twb2ludCIpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYXlsb2FkID0gewogICAgICAgICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgICAgICAgIm1vZGVsX2NvbmZpZyI6IG1vZGVsLmNvbmZpZy5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwKICAgICAgICAibW9kZWxfc3RhdGUiOiBtb2RlbC5zdGF0ZV9kaWN0KCksCiAgICAgICAgIm1ldGFkYXRhIjogZGljdChtZXRhZGF0YSBvciB7fSksCiAgICB9CiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdHJ5OgogICAgICAgIHRvcmNoLnNhdmUocGF5bG9hZCwgdGVtcG9yYXJ5KQogICAgICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInJiIikgYXMgaGFuZGxlOgogICAgICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgIyBIYXJkLWxpbmsgcHVibGljYXRpb24gaXMgYXRvbWljIGFuZCByZWZ1c2VzIGFuIGV4aXN0aW5nIGRlc3RpbmF0aW9uLgogICAgICAgICMgQW4gZXhpc3RzKCkvcmVwbGFjZSgpIHBhaXIgY2FuIG92ZXJ3cml0ZSBhbm90aGVyIHdyaXRlcidzIGNoZWNrcG9pbnQuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBvcy5saW5rKHRlbXBvcmFyeSwgcGF0aCkKICAgICAgICBleGNlcHQgRmlsZUV4aXN0c0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoInJlZnVzaW5nIHRvIG92ZXJ3cml0ZSBhbiBleGlzdGluZyBPbHltcHVzIGNoZWNrcG9pbnQiKSBmcm9tIGVycm9yCiAgICBmaW5hbGx5OgogICAgICAgIHRlbXBvcmFyeS51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgY29udGVudCA9IHBhdGgucmVhZF9ieXRlcygpCiAgICByZXR1cm4gT2x5bXB1c0NoZWNrcG9pbnRSZWNvcmQoCiAgICAgICAgcGF0aD1zdHIocGF0aC5yZXNvbHZlKCkpLAogICAgICAgIHNoYTI1Nj1fc2hhMjU2KGNvbnRlbnQpLAogICAgICAgIHBhcmFtZXRlcnM9c3VtKHBhcmFtZXRlci5udW1lbCgpIGZvciBwYXJhbWV0ZXIgaW4gbW9kZWwucGFyYW1ldGVycygpKSwKICAgICAgICBzeXN0ZW1fY29uZmlnPW1vZGVsLmNvbmZpZywKICAgICAgICBtZXRhZGF0YT1kaWN0KG1ldGFkYXRhIG9yIHt9KSwKICAgICkKCgpkZWYgbG9hZF9vbHltcHVzX2NoZWNrcG9pbnQoCiAgICBwYXRoOiBQYXRoLAogICAgKiwKICAgIGV4cGVjdGVkX3NoYTI1Njogc3RyLAopIC0+IHR1cGxlW09seW1wdXNSZWZlcmVuY2VTeXN0ZW0sIE9seW1wdXNDaGVja3BvaW50UmVjb3JkXToKICAgICIiIkxvYWQgYSBkaWdlc3QtcGlubmVkIGNoZWNrcG9pbnQgd2l0aG91dCBhbGxvd2luZyBhcmJpdHJhcnkgcGlja2xlIG9iamVjdHMuIiIiCgogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBiZSBhIHJlZ3VsYXIgbm9uLXN5bWxpbmsgZmlsZSIpCiAgICBjb250ZW50ID0gcGF0aC5yZWFkX2J5dGVzKCkKICAgIG9ic2VydmVkID0gX3NoYTI1Nihjb250ZW50KQogICAgaWYgb2JzZXJ2ZWQgIT0gZXhwZWN0ZWRfc2hhMjU2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgZGlnZXN0IGRvZXMgbm90IG1hdGNoIHRoZSBleHBlY3RlZCBTSEEtMjU2IikKICAgIHBheWxvYWQ6IEFueSA9IHRvcmNoLmxvYWQoaW8uQnl0ZXNJTyhjb250ZW50KSwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpIG9yIHBheWxvYWQuZ2V0KCJzY2hlbWFfdmVyc2lvbiIpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBoYXMgYW4gdW5zdXBwb3J0ZWQgc2NoZW1hIikKICAgIHJhd19jb25maWcgPSBwYXlsb2FkLmdldCgibW9kZWxfY29uZmlnIikKICAgIHJhd19zdGF0ZSA9IHBheWxvYWQuZ2V0KCJtb2RlbF9zdGF0ZSIpCiAgICByYXdfbWV0YWRhdGEgPSBwYXlsb2FkLmdldCgibWV0YWRhdGEiLCB7fSkKICAgIGlmIG5vdCBpc2luc3RhbmNlKHJhd19jb25maWcsIGRpY3QpIG9yIG5vdCBpc2luc3RhbmNlKHJhd19zdGF0ZSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBpcyBtaXNzaW5nIGl0cyBtb2RlbCBjb25maWd1cmF0aW9uIG9yIHN0YXRlIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHJhd19tZXRhZGF0YSwgZGljdCkgb3Igbm90IGFsbCgKICAgICAgICBpc2luc3RhbmNlKGtleSwgc3RyKSBhbmQgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKSBmb3Iga2V5LCB2YWx1ZSBpbiByYXdfbWV0YWRhdGEuaXRlbXMoKQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IG1ldGFkYXRhIG11c3QgbWFwIHN0cmluZ3MgdG8gc3RyaW5ncyIpCiAgICBtb2RlbF9jb25maWcgPSBPbHltcHVzU3lzdGVtQ29uZmlnLm1vZGVsX3ZhbGlkYXRlKHJhd19jb25maWcpCiAgICBtb2RlbCA9IGJ1aWxkX29seW1wdXNfc3lzdGVtKG1vZGVsX2NvbmZpZykKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChjYXN0KGRpY3Rbc3RyLCBUZW5zb3JdLCByYXdfc3RhdGUpLCBzdHJpY3Q9VHJ1ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgcmVjb3JkID0gT2x5bXB1c0NoZWNrcG9pbnRSZWNvcmQoCiAgICAgICAgcGF0aD1zdHIocGF0aC5yZXNvbHZlKCkpLAogICAgICAgIHNoYTI1Nj1vYnNlcnZlZCwKICAgICAgICBwYXJhbWV0ZXJzPXN1bShwYXJhbWV0ZXIubnVtZWwoKSBmb3IgcGFyYW1ldGVyIGluIG1vZGVsLnBhcmFtZXRlcnMoKSksCiAgICAgICAgc3lzdGVtX2NvbmZpZz1tb2RlbF9jb25maWcsCiAgICAgICAgbWV0YWRhdGE9Y2FzdChkaWN0W3N0ciwgc3RyXSwgcmF3X21ldGFkYXRhKSwKICAgICkKICAgIHJldHVybiBtb2RlbCwgcmVjb3JkCg==', 'olympus/models/seven_model_contract.py': 'IiIiVW5pZmllZCBlbmdpbmVlcmluZyBjb250cmFjdCBmb3IgdGhlIHNpeCBmYW1pbHkgcmVmZXJlbmNlcyBhbmQgSkVQQSB1cHNjYWxlci4KClRoZSBwcm9iZXMgaW4gdGhpcyBtb2R1bGUgdXNlIHRpbnkgYXJ0aWZpY2lhbCB0ZW5zb3JzLiAgVGhleSBlc3RhYmxpc2ggZXhlY3V0YWJsZQphbmQgZGlmZmVyZW50aWFibGUgc29mdHdhcmUgcGF0aHMgb25seTsgdGhleSBkbyBub3QgdHJhaW4sIGV2YWx1YXRlLCBvciBwcm9tb3RlIGEKbW9kZWwtZmFtaWx5IGNoZWNrcG9pbnQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IENhbGxhYmxlCmZyb20gdHlwaW5nIGltcG9ydCBMaXRlcmFsLCBUeXBlQWxpYXMsIGNhc3QKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZApmcm9tIHRvcmNoIGltcG9ydCBUZW5zb3IsIG5uCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbApmcm9tIG9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyBpbXBvcnQgKAogICAgQWlvblJlZmVyZW5jZUNvbmZpZywKICAgIEFpb25SZWZlcmVuY2VNb2RlbCwKICAgIEF0bGFzUmVmZXJlbmNlQ29uZmlnLAogICAgQXRsYXNSZWZlcmVuY2VNb2RlbCwKICAgIEhlcm1lc1JlZmVyZW5jZUNvbmZpZywKICAgIEhlcm1lc1JlZmVyZW5jZU1vZGVsLAogICAgS3Jvbm9zUmVmZXJlbmNlQ29uZmlnLAogICAgS3Jvbm9zUmVmZXJlbmNlTW9kZWwsCiAgICBQZXJzZXVzUmVmZXJlbmNlQ29uZmlnLAogICAgUGVyc2V1c1JlZmVyZW5jZU1vZGVsLAogICAgUHJvbWV0aGV1c1JlZmVyZW5jZUNvbmZpZywKICAgIFByb21ldGhldXNSZWZlcmVuY2VNb2RlbCwKICAgIFJlZmVyZW5jZUZhbWlseUNvbmZpZywKICAgIFJlZmVyZW5jZVRyYW5zZm9ybWVyQ29uZmlnLAogICAgYnVpbGRfcmVmZXJlbmNlX21vZGVsLAopCmZyb20gb2x5bXB1cy51cHNjYWxlci5qZXBhIGltcG9ydCBKRVBBVXBzY2FsZXIsIEpFUEFVcHNjYWxlckNvbmZpZywgamVwYV91cHNjYWxlcl9sb3NzCgpTZXZlbk1vZGVsSWQgPSBMaXRlcmFsWwogICAgImhlcm1lcyIsCiAgICAicHJvbWV0aGV1cyIsCiAgICAicGVyc2V1cyIsCiAgICAiYXRsYXMiLAogICAgImtyb25vcyIsCiAgICAiYWlvbiIsCiAgICAiamVwYV91cHNjYWxlciIsCl0KIyBLZWVwIHRoaXMgcGFyc2VhYmxlIGJ5IHRoZSByZXBvc2l0b3J5J3MgUHl0aG9uLTMuMTEgc291cmNlLWNvbXBhdGliaWxpdHkgZ2F0ZS4KU2V2ZW5Nb2RlbENvbmZpZzogVHlwZUFsaWFzID0gUmVmZXJlbmNlRmFtaWx5Q29uZmlnIHwgSkVQQVVwc2NhbGVyQ29uZmlnICAjIG5vcWE6IFVQMDQwCgpTRVZFTl9NT0RFTF9JRFM6IHR1cGxlW1NldmVuTW9kZWxJZCwgLi4uXSA9ICgKICAgICJoZXJtZXMiLAogICAgInByb21ldGhldXMiLAogICAgInBlcnNldXMiLAogICAgImF0bGFzIiwKICAgICJrcm9ub3MiLAogICAgImFpb24iLAogICAgImplcGFfdXBzY2FsZXIiLAopCgoKY2xhc3MgU2V2ZW5Nb2RlbEVuZ2luZWVyaW5nU3RhdHVzKFN0cmljdE1vZGVsKToKICAgICIiIk1hY2hpbmUtcmVhZGFibGUgcmVzdWx0IGZyb20gb25lIGJvdW5kZWQgYXJ0aWZpY2lhbCBjb250cmFjdCBwcm9iZS4iIiIKCiAgICBtb2RlbF9pZDogU2V2ZW5Nb2RlbElkCiAgICBkaXNwbGF5X25hbWU6IHN0cgogICAgbW9kZWxfa2luZDogTGl0ZXJhbFsiZmFtaWx5X3JlZmVyZW5jZSIsICJ1cHNjYWxlcl9yZWZlcmVuY2UiXQogICAgaW1wbGVtZW50YXRpb246IHN0cgogICAgcmVmZXJlbmNlX3BhcmFtZXRlcnM6IGludCA9IEZpZWxkKGd0PTApCiAgICB0cmFpbmFibGVfcGFyYW1ldGVyczogaW50ID0gRmllbGQoZ3Q9MCkKICAgIHRyYWluYWJsZV9wYXJhbWV0ZXJfdGVuc29yczogaW50ID0gRmllbGQoZ3Q9MCkKICAgIGdyYWRpZW50X3BhcmFtZXRlcl90ZW5zb3JzOiBpbnQgPSBGaWVsZChndD0wKQogICAgZ3JhZGllbnRfY292ZXJhZ2U6IExpdGVyYWxbIlBBU1MiXSA9ICJQQVNTIgogICAgZm9yd2FyZF9jb250cmFjdDogTGl0ZXJhbFsiUEFTUyJdID0gIlBBU1MiCiAgICBmaW5pdGVfY29udHJhY3Q6IExpdGVyYWxbIlBBU1MiXSA9ICJQQVNTIgogICAgYmFja3dhcmRfY29udHJhY3Q6IExpdGVyYWxbIlBBU1MiXSA9ICJQQVNTIgogICAgc3RhdHVzOiBMaXRlcmFsWyJFTkdJTkVFUklOR19SRUZFUkVOQ0VfQ09NUExFVEVfTk9UX1BST01PVEVEIl0gPSAoCiAgICAgICAgIkVOR0lORUVSSU5HX1JFRkVSRU5DRV9DT01QTEVURV9OT1RfUFJPTU9URUQiCiAgICApCiAgICB0YXJnZXRfY2FwYWJpbGl0eV9jbGFpbWVkOiBMaXRlcmFsW0ZhbHNlXSA9IEZhbHNlCiAgICBwcm9tb3Rpb25fYXV0aG9yaXplZDogTGl0ZXJhbFtGYWxzZV0gPSBGYWxzZQogICAgc2NpZW50aWZpY19ldmFsdWF0aW9uOiBMaXRlcmFsW0ZhbHNlXSA9IEZhbHNlCgoKY2xhc3MgU2V2ZW5Nb2RlbEludmVudG9yeVN0YXR1cyhTdHJpY3RNb2RlbCk6CiAgICAiIiJGYXN0LCBub24tZXhlY3V0aW5nIGludmVudG9yeSBmb3IgdGhlIGJvdW5kZWQgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9ucy4iIiIKCiAgICBtb2RlbF9pZDogU2V2ZW5Nb2RlbElkCiAgICBkaXNwbGF5X25hbWU6IHN0cgogICAgbW9kZWxfa2luZDogTGl0ZXJhbFsiZmFtaWx5X3JlZmVyZW5jZSIsICJ1cHNjYWxlcl9yZWZlcmVuY2UiXQogICAgaW1wbGVtZW50YXRpb246IHN0cgogICAgcmVmZXJlbmNlX3BhcmFtZXRlcnM6IGludCA9IEZpZWxkKGd0PTApCiAgICB0cmFpbmFibGVfcGFyYW1ldGVyczogaW50ID0gRmllbGQoZ3Q9MCkKICAgIHN0YXR1czogTGl0ZXJhbFsiRU5HSU5FRVJJTkdfUkVGRVJFTkNFX0xPQ0FMX1ZFUklGSUVEX05PVF9QUk9NT1RFRCJdID0gKAogICAgICAgICJFTkdJTkVFUklOR19SRUZFUkVOQ0VfTE9DQUxfVkVSSUZJRURfTk9UX1BST01PVEVEIgogICAgKQogICAgY29udHJhY3RfZXhlY3V0aW9uX3JlcXVpcmVkOiBMaXRlcmFsW1RydWVdID0gVHJ1ZQogICAgdGFyZ2V0X2NhcGFiaWxpdHlfY2xhaW1lZDogTGl0ZXJhbFtGYWxzZV0gPSBGYWxzZQogICAgcHJvbW90aW9uX2F1dGhvcml6ZWQ6IExpdGVyYWxbRmFsc2VdID0gRmFsc2UKICAgIHNjaWVudGlmaWNfZXZhbHVhdGlvbjogTGl0ZXJhbFtGYWxzZV0gPSBGYWxzZQoKCmRlZiBkZWZhdWx0X3NldmVuX21vZGVsX2NvbmZpZ3MoKSAtPiB0dXBsZVtTZXZlbk1vZGVsQ29uZmlnLCAuLi5dOgogICAgIiIiUmV0dXJuIGJvdW5kZWQgY29uZmlndXJhdGlvbnMgZm9yIHRoZSBleGFjdCBzZXZlbi1tb2RlbCBlbmdpbmVlcmluZyBsaW5ldXAuIiIiCgogICAgYmFja2JvbmUgPSBSZWZlcmVuY2VUcmFuc2Zvcm1lckNvbmZpZygKICAgICAgICB2b2NhYl9zaXplPTMyLAogICAgICAgIG1vZGVsX2RpbT0xNiwKICAgICAgICBsYXllcnM9MSwKICAgICAgICBoZWFkcz00LAogICAgICAgIGZlZWRmb3J3YXJkX2RpbT0yNCwKICAgICAgICBtYXhfc2VxdWVuY2VfbGVuZ3RoPTgsCiAgICApCiAgICByZXR1cm4gKAogICAgICAgIEhlcm1lc1JlZmVyZW5jZUNvbmZpZyhiYWNrYm9uZT1iYWNrYm9uZSwgdG9vbF9jb3VudD0zKSwKICAgICAgICBQcm9tZXRoZXVzUmVmZXJlbmNlQ29uZmlnKAogICAgICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwKICAgICAgICAgICAgZG9tYWlucz0zLAogICAgICAgICAgICBhZGFwdGVyX2RpbT04LAogICAgICAgICAgICBldmlkZW5jZV9jbGFzc2VzPTIsCiAgICAgICAgKSwKICAgICAgICBQZXJzZXVzUmVmZXJlbmNlQ29uZmlnKAogICAgICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwKICAgICAgICAgICAgZXhwZXJ0cz0zLAogICAgICAgICAgICBhY3RpdmVfZXhwZXJ0cz0yLAogICAgICAgICAgICBleHBlcnRfZGltPTI0LAogICAgICAgICksCiAgICAgICAgQXRsYXNSZWZlcmVuY2VDb25maWcoCiAgICAgICAgICAgIGJhY2tib25lPWJhY2tib25lLAogICAgICAgICAgICBpbWFnZV9jaGFubmVscz0zLAogICAgICAgICAgICBwYXRjaF9zaXplPTIsCiAgICAgICAgICAgIHJldHJpZXZhbF9kaW09NiwKICAgICAgICApLAogICAgICAgIEtyb25vc1JlZmVyZW5jZUNvbmZpZygKICAgICAgICAgICAgc3RhdGVfZGltPTUsCiAgICAgICAgICAgIGFjdGlvbl9kaW09MywKICAgICAgICAgICAgaGlkZGVuX2RpbT04LAogICAgICAgICAgICBkaXNjcmV0ZV9hY3Rpb25zPTQsCiAgICAgICAgICAgIHBsYW5uaW5nX2hvcml6b249MywKICAgICAgICApLAogICAgICAgIEFpb25SZWZlcmVuY2VDb25maWcoCiAgICAgICAgICAgIHN0YXRlX2RpbT01LAogICAgICAgICAgICBoaWRkZW5fZGltPTgsCiAgICAgICAgICAgIHJlc2VhcmNoX3N0YWdlcz00LAogICAgICAgICAgICBzdG9wX3N0YWdlPTMsCiAgICAgICAgICAgIGtpbGxfdGhyZXNob2xkPTAuNzUsCiAgICAgICAgKSwKICAgICAgICBKRVBBVXBzY2FsZXJDb25maWcoCiAgICAgICAgICAgIHNjYWxlPTIsCiAgICAgICAgICAgIGlucHV0X2NoYW5uZWxzPTMsCiAgICAgICAgICAgIGZlYXR1cmVfY2hhbm5lbHM9OCwKICAgICAgICAgICAgcmVzaWR1YWxfYmxvY2tzPTEsCiAgICAgICAgICAgIHRhcmdldF9tb21lbnR1bT0wLjksCiAgICAgICAgICAgIHByZWRpY3Rvcl9kaWxhdGlvbnM9KDEsIDIpLAogICAgICAgICksCiAgICApCgoKZGVmIHNldmVuX21vZGVsX2lkKGNvbmZpZzogU2V2ZW5Nb2RlbENvbmZpZykgLT4gU2V2ZW5Nb2RlbElkOgogICAgIiIiUmV0dXJuIHRoZSBzdGFibGUgZW5naW5lZXJpbmcgaWRlbnRpZmllciBmb3Igb25lIHR5cGVkIG1vZGVsIGNvbmZpZy4iIiIKCiAgICBpZiBpc2luc3RhbmNlKGNvbmZpZywgSkVQQVVwc2NhbGVyQ29uZmlnKToKICAgICAgICByZXR1cm4gImplcGFfdXBzY2FsZXIiCiAgICByZXR1cm4gY29uZmlnLmZhbWlseV9pZAoKCmRlZiBidWlsZF9zZXZlbl9tb2RlbF9yZWZlcmVuY2UoY29uZmlnOiBTZXZlbk1vZGVsQ29uZmlnKSAtPiBubi5Nb2R1bGU6CiAgICAiIiJCdWlsZCBvbmUgYm91bmRlZCBpbXBsZW1lbnRhdGlvbiB3aXRob3V0IGltcGx5aW5nIGEgdHJhaW5lZCBmYW1pbHkgbW9kZWwuIiIiCgogICAgaWYgaXNpbnN0YW5jZShjb25maWcsIEpFUEFVcHNjYWxlckNvbmZpZyk6CiAgICAgICAgcmV0dXJuIEpFUEFVcHNjYWxlcihjb25maWcpCiAgICByZXR1cm4gYnVpbGRfcmVmZXJlbmNlX21vZGVsKGNvbmZpZykKCgpkZWYgaW5zcGVjdF9zZXZlbl9tb2RlbF9yZWZlcmVuY2VzKCkgLT4gdHVwbGVbU2V2ZW5Nb2RlbEludmVudG9yeVN0YXR1cywgLi4uXToKICAgICIiIkRlc2NyaWJlIHRoZSBjYW5vbmljYWwgcmVmZXJlbmNlcyB3aXRob3V0IHJ1bm5pbmcgdGhlaXIgY29udHJhY3QgcHJvYmVzLiIiIgoKICAgIGRpc3BsYXlfbmFtZXM6IGRpY3RbU2V2ZW5Nb2RlbElkLCBzdHJdID0gewogICAgICAgICJoZXJtZXMiOiAiSGVybWVzIiwKICAgICAgICAicHJvbWV0aGV1cyI6ICJQcm9tZXRoZXVzIiwKICAgICAgICAicGVyc2V1cyI6ICJQZXJzZXVzIiwKICAgICAgICAiYXRsYXMiOiAiQXRsYXMiLAogICAgICAgICJrcm9ub3MiOiAiS3Jvbm9zIiwKICAgICAgICAiYWlvbiI6ICJBaW9uIiwKICAgICAgICAiamVwYV91cHNjYWxlciI6ICJKRVBBIFVwc2NhbGVyIiwKICAgIH0KICAgIHN0YXR1c2VzOiBsaXN0W1NldmVuTW9kZWxJbnZlbnRvcnlTdGF0dXNdID0gW10KICAgIHdpdGggdG9yY2gucmFuZG9tLmZvcmtfcm5nKGRldmljZXM9W10pOgogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKDIwMjYxMDE4KQogICAgICAgIGZvciBjb25maWcgaW4gZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzKCk6CiAgICAgICAgICAgIG1vZGVsX2lkID0gc2V2ZW5fbW9kZWxfaWQoY29uZmlnKQogICAgICAgICAgICBtb2RlbCA9IGJ1aWxkX3NldmVuX21vZGVsX3JlZmVyZW5jZShjb25maWcpCiAgICAgICAgICAgIHN0YXR1c2VzLmFwcGVuZCgKICAgICAgICAgICAgICAgIFNldmVuTW9kZWxJbnZlbnRvcnlTdGF0dXMoCiAgICAgICAgICAgICAgICAgICAgbW9kZWxfaWQ9bW9kZWxfaWQsCiAgICAgICAgICAgICAgICAgICAgZGlzcGxheV9uYW1lPWRpc3BsYXlfbmFtZXNbbW9kZWxfaWRdLAogICAgICAgICAgICAgICAgICAgIG1vZGVsX2tpbmQ9KAogICAgICAgICAgICAgICAgICAgICAgICAidXBzY2FsZXJfcmVmZXJlbmNlIgogICAgICAgICAgICAgICAgICAgICAgICBpZiBtb2RlbF9pZCA9PSAiamVwYV91cHNjYWxlciIKICAgICAgICAgICAgICAgICAgICAgICAgZWxzZSAiZmFtaWx5X3JlZmVyZW5jZSIKICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgICAgIGltcGxlbWVudGF0aW9uPXR5cGUobW9kZWwpLl9fbmFtZV9fLAogICAgICAgICAgICAgICAgICAgIHJlZmVyZW5jZV9wYXJhbWV0ZXJzPXN1bShwYXJhbWV0ZXIubnVtZWwoKSBmb3IgcGFyYW1ldGVyIGluIG1vZGVsLnBhcmFtZXRlcnMoKSksCiAgICAgICAgICAgICAgICAgICAgdHJhaW5hYmxlX3BhcmFtZXRlcnM9c3VtKAogICAgICAgICAgICAgICAgICAgICAgICBwYXJhbWV0ZXIubnVtZWwoKQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgcGFyYW1ldGVyIGluIG1vZGVsLnBhcmFtZXRlcnMoKQogICAgICAgICAgICAgICAgICAgICAgICBpZiBwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZAogICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgIGlmIHR1cGxlKHN0YXR1cy5tb2RlbF9pZCBmb3Igc3RhdHVzIGluIHN0YXR1c2VzKSAhPSBTRVZFTl9NT0RFTF9JRFM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJzZXZlbi1tb2RlbCBpbnZlbnRvcnkgZGlkIG5vdCBwcmVzZXJ2ZSB0aGUgY2Fub25pY2FsIG1vZGVsIG9yZGVyIikKICAgIHJldHVybiB0dXBsZShzdGF0dXNlcykKCgpkZWYgYXVkaXRfc2V2ZW5fbW9kZWxfY29udHJhY3RzKCkgLT4gdHVwbGVbU2V2ZW5Nb2RlbEVuZ2luZWVyaW5nU3RhdHVzLCAuLi5dOgogICAgIiIiRXhlY3V0ZSBkZXRlcm1pbmlzdGljIGFydGlmaWNpYWwgZm9yd2FyZC9iYWNrd2FyZCBwcm9iZXMgZm9yIGFsbCBzZXZlbiBtb2RlbHMuIiIiCgogICAgcHJvYmVfYnlfaWQ6IGRpY3RbU2V2ZW5Nb2RlbElkLCBDYWxsYWJsZVtbbm4uTW9kdWxlXSwgVGVuc29yXV0gPSB7CiAgICAgICAgImhlcm1lcyI6IF9wcm9iZV9oZXJtZXMsCiAgICAgICAgInByb21ldGhldXMiOiBfcHJvYmVfcHJvbWV0aGV1cywKICAgICAgICAicGVyc2V1cyI6IF9wcm9iZV9wZXJzZXVzLAogICAgICAgICJhdGxhcyI6IF9wcm9iZV9hdGxhcywKICAgICAgICAia3Jvbm9zIjogX3Byb2JlX2tyb25vcywKICAgICAgICAiYWlvbiI6IF9wcm9iZV9haW9uLAogICAgICAgICJqZXBhX3Vwc2NhbGVyIjogX3Byb2JlX2plcGEsCiAgICB9CiAgICBkaXNwbGF5X25hbWVzOiBkaWN0W1NldmVuTW9kZWxJZCwgc3RyXSA9IHsKICAgICAgICAiaGVybWVzIjogIkhlcm1lcyIsCiAgICAgICAgInByb21ldGhldXMiOiAiUHJvbWV0aGV1cyIsCiAgICAgICAgInBlcnNldXMiOiAiUGVyc2V1cyIsCiAgICAgICAgImF0bGFzIjogIkF0bGFzIiwKICAgICAgICAia3Jvbm9zIjogIktyb25vcyIsCiAgICAgICAgImFpb24iOiAiQWlvbiIsCiAgICAgICAgImplcGFfdXBzY2FsZXIiOiAiSkVQQSBVcHNjYWxlciIsCiAgICB9CiAgICBzdGF0dXNlczogbGlzdFtTZXZlbk1vZGVsRW5naW5lZXJpbmdTdGF0dXNdID0gW10KICAgIHdpdGggdG9yY2gucmFuZG9tLmZvcmtfcm5nKGRldmljZXM9W10pOgogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKDIwMjYxMDE4KQogICAgICAgIGZvciBjb25maWcgaW4gZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzKCk6CiAgICAgICAgICAgIG1vZGVsX2lkID0gc2V2ZW5fbW9kZWxfaWQoY29uZmlnKQogICAgICAgICAgICBtb2RlbCA9IGJ1aWxkX3NldmVuX21vZGVsX3JlZmVyZW5jZShjb25maWcpLnRyYWluKCkKICAgICAgICAgICAgbW9kZWwuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIG9iamVjdGl2ZSA9IHByb2JlX2J5X2lkW21vZGVsX2lkXShtb2RlbCkKICAgICAgICAgICAgaWYgb2JqZWN0aXZlLm5kaW0gIT0gMCBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShvYmplY3RpdmUpKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmInttb2RlbF9pZH0gYXJ0aWZpY2lhbCBwcm9iZSByZXR1cm5lZCBhIG5vbi1maW5pdGUgc2NhbGFyIikKICAgICAgICAgICAgb2JqZWN0aXZlLmJhY2t3YXJkKCkgICMgdHlwZTogaWdub3JlW25vLXVudHlwZWQtY2FsbF0KICAgICAgICAgICAgdHJhaW5hYmxlX3BhcmFtZXRlcnMgPSBbCiAgICAgICAgICAgICAgICBwYXJhbWV0ZXIgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQKICAgICAgICAgICAgXQogICAgICAgICAgICBncmFkaWVudHMgPSBbcGFyYW1ldGVyLmdyYWQgZm9yIHBhcmFtZXRlciBpbiB0cmFpbmFibGVfcGFyYW1ldGVyc10KICAgICAgICAgICAgaWYgYW55KGdyYWRpZW50IGlzIE5vbmUgZm9yIGdyYWRpZW50IGluIGdyYWRpZW50cyk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgZiJ7bW9kZWxfaWR9IGFydGlmaWNpYWwgcHJvYmUgZGlkIG5vdCByZWFjaCBldmVyeSB0cmFpbmFibGUgcGFyYW1ldGVyIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBmaW5pdGVfZ3JhZGllbnRzID0gW2Nhc3QoVGVuc29yLCBncmFkaWVudCkgZm9yIGdyYWRpZW50IGluIGdyYWRpZW50c10KICAgICAgICAgICAgaWYgbm90IGFsbChib29sKHRvcmNoLmlzZmluaXRlKGdyYWRpZW50KS5hbGwoKSkgZm9yIGdyYWRpZW50IGluIGZpbml0ZV9ncmFkaWVudHMpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie21vZGVsX2lkfSBhcnRpZmljaWFsIHByb2JlIHByb2R1Y2VkIG5vbi1maW5pdGUgZ3JhZGllbnRzIikKICAgICAgICAgICAgdHJhaW5hYmxlID0gc3VtKAogICAgICAgICAgICAgICAgcGFyYW1ldGVyLm51bWVsKCkgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQKICAgICAgICAgICAgKQogICAgICAgICAgICBzdGF0dXNlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICBTZXZlbk1vZGVsRW5naW5lZXJpbmdTdGF0dXMoCiAgICAgICAgICAgICAgICAgICAgbW9kZWxfaWQ9bW9kZWxfaWQsCiAgICAgICAgICAgICAgICAgICAgZGlzcGxheV9uYW1lPWRpc3BsYXlfbmFtZXNbbW9kZWxfaWRdLAogICAgICAgICAgICAgICAgICAgIG1vZGVsX2tpbmQ9KAogICAgICAgICAgICAgICAgICAgICAgICAidXBzY2FsZXJfcmVmZXJlbmNlIgogICAgICAgICAgICAgICAgICAgICAgICBpZiBtb2RlbF9pZCA9PSAiamVwYV91cHNjYWxlciIKICAgICAgICAgICAgICAgICAgICAgICAgZWxzZSAiZmFtaWx5X3JlZmVyZW5jZSIKICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgICAgIGltcGxlbWVudGF0aW9uPXR5cGUobW9kZWwpLl9fbmFtZV9fLAogICAgICAgICAgICAgICAgICAgIHJlZmVyZW5jZV9wYXJhbWV0ZXJzPXN1bSgKICAgICAgICAgICAgICAgICAgICAgICAgcGFyYW1ldGVyLm51bWVsKCkgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkKICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgICAgIHRyYWluYWJsZV9wYXJhbWV0ZXJzPXRyYWluYWJsZSwKICAgICAgICAgICAgICAgICAgICB0cmFpbmFibGVfcGFyYW1ldGVyX3RlbnNvcnM9bGVuKHRyYWluYWJsZV9wYXJhbWV0ZXJzKSwKICAgICAgICAgICAgICAgICAgICBncmFkaWVudF9wYXJhbWV0ZXJfdGVuc29ycz1sZW4oZmluaXRlX2dyYWRpZW50cyksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgIGlmIHR1cGxlKHN0YXR1cy5tb2RlbF9pZCBmb3Igc3RhdHVzIGluIHN0YXR1c2VzKSAhPSBTRVZFTl9NT0RFTF9JRFM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJzZXZlbi1tb2RlbCBhdWRpdCBkaWQgbm90IHByZXNlcnZlIHRoZSBjYW5vbmljYWwgbW9kZWwgb3JkZXIiKQogICAgcmV0dXJuIHR1cGxlKHN0YXR1c2VzKQoKCmRlZiBfdG9rZW5zKCkgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3JdOgogICAgcmV0dXJuICgKICAgICAgICB0b3JjaC50ZW5zb3IoW1sxLCAyLCAzLCAwXSwgWzQsIDUsIDAsIDBdXSwgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgdG9yY2gudGVuc29yKAogICAgICAgICAgICBbW1RydWUsIFRydWUsIFRydWUsIEZhbHNlXSwgW1RydWUsIFRydWUsIEZhbHNlLCBGYWxzZV1dLCBkdHlwZT10b3JjaC5ib29sCiAgICAgICAgKSwKICAgICkKCgpkZWYgX2Zpbml0ZV90ZW5zb3JzKG1vZGVsX2lkOiBTZXZlbk1vZGVsSWQsICp0ZW5zb3JzOiBUZW5zb3IpIC0+IE5vbmU6CiAgICBpZiBub3QgdGVuc29ycyBvciBub3QgYWxsKGJvb2wodG9yY2guaXNmaW5pdGUodGVuc29yKS5hbGwoKSkgZm9yIHRlbnNvciBpbiB0ZW5zb3JzKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7bW9kZWxfaWR9IGFydGlmaWNpYWwgcHJvYmUgcHJvZHVjZWQgbm9uLWZpbml0ZSBvdXRwdXQiKQoKCmRlZiBfcHJvYmVfaGVybWVzKG1vZHVsZTogbm4uTW9kdWxlKSAtPiBUZW5zb3I6CiAgICBtb2RlbCA9IGNhc3QoSGVybWVzUmVmZXJlbmNlTW9kZWwsIG1vZHVsZSkKICAgIHRva2VucywgbWFzayA9IF90b2tlbnMoKQogICAgb3V0cHV0ID0gbW9kZWwodG9rZW5zLCBtYXNrKQogICAgX2Zpbml0ZV90ZW5zb3JzKAogICAgICAgICJoZXJtZXMiLAogICAgICAgIG91dHB1dC50b2tlbl9sb2dpdHMsCiAgICAgICAgb3V0cHV0LnRvb2xfbG9naXRzLAogICAgICAgIG91dHB1dC5hYnN0ZW50aW9uX2xvZ2l0LAogICAgICAgIG91dHB1dC5oaWRkZW5fc3RhdGVzLAogICAgKQogICAgcmV0dXJuIGNhc3QoCiAgICAgICAgVGVuc29yLAogICAgICAgIG91dHB1dC50b2tlbl9sb2dpdHMuc3F1YXJlKCkubWVhbigpCiAgICAgICAgKyBvdXRwdXQudG9vbF9sb2dpdHMuc3F1YXJlKCkubWVhbigpCiAgICAgICAgKyBvdXRwdXQuYWJzdGVudGlvbl9sb2dpdC5zcXVhcmUoKS5tZWFuKCksCiAgICApCgoKZGVmIF9wcm9iZV9wcm9tZXRoZXVzKG1vZHVsZTogbm4uTW9kdWxlKSAtPiBUZW5zb3I6CiAgICBtb2RlbCA9IGNhc3QoUHJvbWV0aGV1c1JlZmVyZW5jZU1vZGVsLCBtb2R1bGUpCiAgICB0b2tlbnMsIG1hc2sgPSBfdG9rZW5zKCkKICAgIG91dHB1dCA9IG1vZGVsKHRva2VucywgbWFzaykKICAgIF9maW5pdGVfdGVuc29ycygKICAgICAgICAicHJvbWV0aGV1cyIsCiAgICAgICAgb3V0cHV0LnRva2VuX2xvZ2l0cywKICAgICAgICBvdXRwdXQuZXZpZGVuY2VfbG9naXRzLAogICAgICAgIG91dHB1dC5kb21haW5fd2VpZ2h0cywKICAgICAgICBvdXRwdXQuaGlkZGVuX3N0YXRlcywKICAgICkKICAgIHJldHVybiBjYXN0KAogICAgICAgIFRlbnNvciwgb3V0cHV0LnRva2VuX2xvZ2l0cy5zcXVhcmUoKS5tZWFuKCkgKyBvdXRwdXQuZXZpZGVuY2VfbG9naXRzLnNxdWFyZSgpLm1lYW4oKQogICAgKQoKCmRlZiBfcHJvYmVfcGVyc2V1cyhtb2R1bGU6IG5uLk1vZHVsZSkgLT4gVGVuc29yOgogICAgbW9kZWwgPSBjYXN0KFBlcnNldXNSZWZlcmVuY2VNb2RlbCwgbW9kdWxlKQogICAgdG9rZW5zLCBtYXNrID0gX3Rva2VucygpCiAgICBvdXRwdXQgPSBtb2RlbCh0b2tlbnMsIG1hc2spCiAgICBfZmluaXRlX3RlbnNvcnMoCiAgICAgICAgInBlcnNldXMiLAogICAgICAgIG91dHB1dC50b2tlbl9sb2dpdHMsCiAgICAgICAgb3V0cHV0LnJvdXRpbmdfcHJvYmFiaWxpdGllcywKICAgICAgICBvdXRwdXQubG9hZF9iYWxhbmNlX2xvc3MsCiAgICAgICAgb3V0cHV0LmhpZGRlbl9zdGF0ZXMsCiAgICApCiAgICByZXR1cm4gY2FzdChUZW5zb3IsIG91dHB1dC50b2tlbl9sb2dpdHMuc3F1YXJlKCkubWVhbigpICsgMC4wMSAqIG91dHB1dC5sb2FkX2JhbGFuY2VfbG9zcykKCgpkZWYgX3Byb2JlX2F0bGFzKG1vZHVsZTogbm4uTW9kdWxlKSAtPiBUZW5zb3I6CiAgICBtb2RlbCA9IGNhc3QoQXRsYXNSZWZlcmVuY2VNb2RlbCwgbW9kdWxlKQogICAgdG9rZW5zLCBtYXNrID0gX3Rva2VucygpCiAgICBpbWFnZSA9IHRvcmNoLmxpbnNwYWNlKDAuMCwgMS4wLCAyICogMyAqIDggKiA4KS5yZXNoYXBlKDIsIDMsIDgsIDgpCiAgICBvdXRwdXQgPSBtb2RlbCh0b2tlbnMsIGltYWdlLCBtYXNrLCB0aW1lc3RhbXA9dG9yY2gudGVuc29yKFswLjAsIDEuMF0pKQogICAgX2Zpbml0ZV90ZW5zb3JzKAogICAgICAgICJhdGxhcyIsCiAgICAgICAgb3V0cHV0LnRva2VuX2xvZ2l0cywKICAgICAgICBvdXRwdXQucmV0cmlldmFsX2VtYmVkZGluZywKICAgICAgICBvdXRwdXQuZ3JvdW5kaW5nX3dlaWdodHMsCiAgICAgICAgb3V0cHV0LmhpZGRlbl9zdGF0ZXMsCiAgICApCiAgICByZXR1cm4gY2FzdCgKICAgICAgICBUZW5zb3IsCiAgICAgICAgb3V0cHV0LnRva2VuX2xvZ2l0cy5zcXVhcmUoKS5tZWFuKCkgKyBvdXRwdXQucmV0cmlldmFsX2VtYmVkZGluZy5zcXVhcmUoKS5tZWFuKCksCiAgICApCgoKZGVmIF9wcm9iZV9rcm9ub3MobW9kdWxlOiBubi5Nb2R1bGUpIC0+IFRlbnNvcjoKICAgIG1vZGVsID0gY2FzdChLcm9ub3NSZWZlcmVuY2VNb2RlbCwgbW9kdWxlKQogICAgc3RhdGVzID0gdG9yY2gubGluc3BhY2UoMC4wLCAxLjAsIDIgKiA0ICogNSkucmVzaGFwZSgyLCA0LCA1KQogICAgYWN0aW9ucyA9IHRvcmNoLmxpbnNwYWNlKC0xLjAsIDEuMCwgMiAqIDQgKiAzKS5yZXNoYXBlKDIsIDQsIDMpCiAgICB2YWxpZCA9IHRvcmNoLnRlbnNvcigKICAgICAgICBbW1RydWUsIFRydWUsIFRydWUsIFRydWVdLCBbVHJ1ZSwgVHJ1ZSwgRmFsc2UsIEZhbHNlXV0sIGR0eXBlPXRvcmNoLmJvb2wKICAgICkKICAgIG91dHB1dCA9IG1vZGVsKHN0YXRlcywgYWN0aW9ucywgdmFsaWQpCiAgICBfZmluaXRlX3RlbnNvcnMoCiAgICAgICAgImtyb25vcyIsCiAgICAgICAgb3V0cHV0Lm5leHRfc3RhdGUsCiAgICAgICAgb3V0cHV0LnJld2FyZCwKICAgICAgICBvdXRwdXQudmFsdWUsCiAgICAgICAgb3V0cHV0LnVuY2VydGFpbnR5LAogICAgICAgIG91dHB1dC5wbGFuX2xvZ2l0cywKICAgICAgICBvdXRwdXQuaGlkZGVuX3N0YXRlcywKICAgICkKICAgIHJldHVybiBjYXN0KAogICAgICAgIFRlbnNvciwKICAgICAgICBvdXRwdXQubmV4dF9zdGF0ZS5zcXVhcmUoKS5tZWFuKCkKICAgICAgICArIG91dHB1dC5yZXdhcmQuc3F1YXJlKCkubWVhbigpCiAgICAgICAgKyBvdXRwdXQudmFsdWUuc3F1YXJlKCkubWVhbigpCiAgICAgICAgKyBvdXRwdXQudW5jZXJ0YWludHkuc3F1YXJlKCkubWVhbigpCiAgICAgICAgKyBvdXRwdXQucGxhbl9sb2dpdHMuc3F1YXJlKCkubWVhbigpLAogICAgKQoKCmRlZiBfcHJvYmVfYWlvbihtb2R1bGU6IG5uLk1vZHVsZSkgLT4gVGVuc29yOgogICAgbW9kZWwgPSBjYXN0KEFpb25SZWZlcmVuY2VNb2RlbCwgbW9kdWxlKQogICAgc3RhdGVzID0gdG9yY2gubGluc3BhY2UoLTEuMCwgMS4wLCAyICogNSkucmVzaGFwZSgyLCA1KQogICAgYWxsb3dlZCA9IHRvcmNoLnRlbnNvcigKICAgICAgICBbW1RydWUsIFRydWUsIEZhbHNlLCBUcnVlXSwgW0ZhbHNlLCBUcnVlLCBUcnVlLCBUcnVlXV0sIGR0eXBlPXRvcmNoLmJvb2wKICAgICkKICAgIG91dHB1dCA9IG1vZGVsKHN0YXRlcywgYWxsb3dlZCwgdG9yY2gudGVuc29yKFswLjIsIDAuN10pKQogICAgX2Zpbml0ZV90ZW5zb3JzKCJhaW9uIiwgb3V0cHV0LnRyYW5zaXRpb25fcHJvYmFiaWxpdGllcykKICAgIHN0YWdlX3dlaWdodCA9IHRvcmNoLmFyYW5nZSg0LCBkdHlwZT1vdXRwdXQudHJhbnNpdGlvbl9wcm9iYWJpbGl0aWVzLmR0eXBlKQogICAgcmV0dXJuIGNhc3QoVGVuc29yLCAob3V0cHV0LnRyYW5zaXRpb25fcHJvYmFiaWxpdGllcyAqIHN0YWdlX3dlaWdodCkuc3VtKGRpbT0tMSkubWVhbigpKQoKCmRlZiBfcHJvYmVfamVwYShtb2R1bGU6IG5uLk1vZHVsZSkgLT4gVGVuc29yOgogICAgbW9kZWwgPSBjYXN0KEpFUEFVcHNjYWxlciwgbW9kdWxlKQogICAgbG93ID0gdG9yY2gubGluc3BhY2UoMC4wLCAxLjAsIDIgKiAzICogNCAqIDUpLnJlc2hhcGUoMiwgMywgNCwgNSkKICAgIGhpZ2ggPSB0b3JjaC5ubi5mdW5jdGlvbmFsLmludGVycG9sYXRlKGxvdywgc2NhbGVfZmFjdG9yPTIsIG1vZGU9ImJpbGluZWFyIikKICAgIG1hc2sgPSB0b3JjaC56ZXJvcygoMiwgNCwgNSksIGR0eXBlPXRvcmNoLmJvb2wpCiAgICBtYXNrWzosIDoyLCA6Ml0gPSBUcnVlCiAgICBvdXRwdXQgPSBtb2RlbChsb3csIGhpZ2gsIHRpbWVzdGFtcD10b3JjaC50ZW5zb3IoWzAuMCwgMS4wXSksIGNvbnRleHRfbWFzaz1tYXNrKQogICAgaWYgb3V0cHV0LnRhcmdldF9sYXRlbnQgaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImplcGFfdXBzY2FsZXIgYXJ0aWZpY2lhbCBwcm9iZSBkaWQgbm90IHByb2R1Y2UgYSB0YXJnZXQgbGF0ZW50IikKICAgIF9maW5pdGVfdGVuc29ycygKICAgICAgICAiamVwYV91cHNjYWxlciIsCiAgICAgICAgb3V0cHV0LnVwc2NhbGVkLAogICAgICAgIG91dHB1dC5yZXNpZHVhbCwKICAgICAgICBvdXRwdXQucHJlZGljdGVkX2xhdGVudCwKICAgICAgICBvdXRwdXQudGFyZ2V0X2xhdGVudCwKICAgICAgICBvdXRwdXQuZ2VvbWV0cnksCiAgICApCiAgICByZXR1cm4gamVwYV91cHNjYWxlcl9sb3NzKG91dHB1dCwgaGlnaCkudG90YWwK', 'olympus/models/spatiotemporal_lazy.py': '...', 'olympus/models/specifications.py': 'IiIiQ2Fub25pY2FsLCBldmlkZW5jZS1ib3VuZCBzcGVjaWZpY2F0aW9ucyBmb3IgdGhlIEJVMUxEIG1vZGVsIHByb2dyYW1tZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5nIGltcG9ydCBMaXRlcmFsCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKTW9kZWxGYW1pbHlJZCA9IExpdGVyYWxbImhlcm1lcyIsICJwcm9tZXRoZXVzIiwgInBlcnNldXMiLCAiYXRsYXMiLCAia3Jvbm9zIiwgImFpb24iXQoKCmNsYXNzIE1vZGVsVGFyZ2V0U3BlY2lmaWNhdGlvbihTdHJpY3RNb2RlbCk6CiAgICAiIiJBbiBpbnRlbmRlZCBtb2RlbCB0YXJnZXQsIG5ldmVyIGV2aWRlbmNlIHRoYXQgdGhlIHRhcmdldCBhbHJlYWR5IGV4aXN0cy4iIiIKCiAgICBmYW1pbHlfaWQ6IE1vZGVsRmFtaWx5SWQKICAgIGRpc3BsYXlfbmFtZTogc3RyCiAgICB0YXJnZXRfcGFyYW1ldGVyczogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9MCkKICAgIHRhcmdldF9wYXJhbWV0ZXJzX21pbjogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9MCkKICAgIHRhcmdldF9wYXJhbWV0ZXJzX21heDogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9MCkKICAgIGludGVuZGVkX3JvbGU6IHN0cgogICAgY2FwYWJpbGl0eV9nYXRlczogbGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgc3RhdHVzOiBMaXRlcmFsWyJUQVJHRVRfU1BFQ0lGSUNBVElPTiJdID0gIlRBUkdFVF9TUEVDSUZJQ0FUSU9OIgogICAgY2FwYWJpbGl0eV9jbGFpbWVkOiBMaXRlcmFsW0ZhbHNlXSA9IEZhbHNlCgogICAgQG1vZGVsX3ZhbGlkYXRvcihtb2RlPSJhZnRlciIpCiAgICBkZWYgdmFsaWRfdGFyZ2V0X3JhbmdlKHNlbGYpIC0+IE1vZGVsVGFyZ2V0U3BlY2lmaWNhdGlvbjoKICAgICAgICBsb3csIGhpZ2ggPSBzZWxmLnRhcmdldF9wYXJhbWV0ZXJzX21pbiwgc2VsZi50YXJnZXRfcGFyYW1ldGVyc19tYXgKICAgICAgICBpZiAobG93IGlzIE5vbmUpICE9IChoaWdoIGlzIE5vbmUpIG9yIChsb3cgaXMgbm90IE5vbmUgYW5kIGhpZ2ggaXMgbm90IE5vbmUgYW5kIGxvdyA+IGhpZ2gpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0YXJnZXQgcmFuZ2VzIHJlcXVpcmUgb3JkZXJlZCBtaW5pbXVtIGFuZCBtYXhpbXVtIikKICAgICAgICBpZiBsb3cgaXMgbm90IE5vbmUgYW5kIGhpZ2ggaXMgbm90IE5vbmUgYW5kIHNlbGYudGFyZ2V0X3BhcmFtZXRlcnMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlmIG5vdCBsb3cgPD0gc2VsZi50YXJnZXRfcGFyYW1ldGVycyA8PSBoaWdoOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicG9pbnQgdGFyZ2V0IG11c3QgbGllIGluc2lkZSB0aGUgdGFyZ2V0IHJhbmdlIikKICAgICAgICByZXR1cm4gc2VsZgoKCkJVMUxEX01PREVMX1NQRUNTOiB0dXBsZVtNb2RlbFRhcmdldFNwZWNpZmljYXRpb24sIC4uLl0gPSAoCiAgICBNb2RlbFRhcmdldFNwZWNpZmljYXRpb24oCiAgICAgICAgZmFtaWx5X2lkPSJoZXJtZXMiLAogICAgICAgIGRpc3BsYXlfbmFtZT0iSGVybWVzIiwKICAgICAgICB0YXJnZXRfcGFyYW1ldGVycz0xMl8wMDBfMDAwXzAwMCwKICAgICAgICBpbnRlbmRlZF9yb2xlPSJQcml2YXRlLCBsb2NhbC1maXJzdCBhc3Npc3RhbnQgd2l0aCBncm91bmRlZCB0b29sIHVzZSBhbmQgcmVjb3ZlcnkuIiwKICAgICAgICBjYXBhYmlsaXR5X2dhdGVzPVsKICAgICAgICAgICAgImhlbGQtb3V0IGV4YWN0IHRhc2sgY29tcGxldGlvbiIsCiAgICAgICAgICAgICJ0eXBlZCB0b29sIGFyZ3VtZW50cyBhbmQgZGVuaWVkLWNhbGwgaGFuZGxpbmciLAogICAgICAgICAgICAiZ3JvdW5kZWQgY2l0YXRpb24gYW5kIGNhbGlicmF0ZWQgYWJzdGVudGlvbiIsCiAgICAgICAgICAgICJib3VuZGVkIGxvY2FsIG1lbW9yeSwgbGF0ZW5jeSwgYW5kIHByaXZhY3kiLAogICAgICAgIF0sCiAgICApLAogICAgTW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAgICAgIGZhbWlseV9pZD0icHJvbWV0aGV1cyIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJQcm9tZXRoZXVzIiwKICAgICAgICB0YXJnZXRfcGFyYW1ldGVyc19taW49MTJfMDAwXzAwMF8wMDAsCiAgICAgICAgdGFyZ2V0X3BhcmFtZXRlcnNfbWF4PTIwXzAwMF8wMDBfMDAwLAogICAgICAgIGludGVuZGVkX3JvbGU9KAogICAgICAgICAgICAiRG9tYWluIHNwZWNpYWxpc3QgYWRhcHRlZCBmb3IgcmVzZWFyY2gsIHNvZnR3YXJlIGVuZ2luZWVyaW5nLCBhbmQgZGVzaWduIHdpdGhvdXQgIgogICAgICAgICAgICAiZXJhc2luZyBnZW5lcmFsIHNhZmV0eSBvciBjYWxpYnJhdGlvbi4iCiAgICAgICAgKSwKICAgICAgICBjYXBhYmlsaXR5X2dhdGVzPVsKICAgICAgICAgICAgImRvbWFpbi1kaXNqb2ludCBzcGVjaWFsaXN0IGV2YWx1YXRpb25zIiwKICAgICAgICAgICAgImJhc2UtdmVyc3VzLWFkYXB0ZXIgcmVncmVzc2lvbiBjb250cm9scyIsCiAgICAgICAgICAgICJzb3VyY2UtZ3JvdW5kZWQgcmVzZWFyY2ggYW5kIGV4ZWN1dGFibGUgdmVyaWZpY2F0aW9uIiwKICAgICAgICAgICAgImFkYXB0ZXIgaXNvbGF0aW9uLCByb2xsYmFjaywgYW5kIHByb3ZlbmFuY2UiLAogICAgICAgIF0sCiAgICApLAogICAgTW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAgICAgIGZhbWlseV9pZD0icGVyc2V1cyIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJQZXJzZXVzIiwKICAgICAgICB0YXJnZXRfcGFyYW1ldGVycz0xXzAyNF8wMDBfMDAwXzAwMCwKICAgICAgICBpbnRlbmRlZF9yb2xlPSgKICAgICAgICAgICAgIlNwYXJzZSBnZW5lcmFsaXN0IGRlc2lnbmVkIGZvciBnbG9iYWxseSBhY2Nlc3NpYmxlIGluZmVyZW5jZSB3aXRoIHJlbGlhYmxlIHJvdXRpbmcuIgogICAgICAgICksCiAgICAgICAgY2FwYWJpbGl0eV9nYXRlcz1bCiAgICAgICAgICAgICJhY3RpdmUtcGFyYW1ldGVyIGFuZCBtZW1vcnkgYnVkZ2V0IGNvbXBsaWFuY2UiLAogICAgICAgICAgICAibXVsdGlsaW5ndWFsIGFuZCBsb3ctcmVzb3VyY2UgZXZhbHVhdGlvbnMiLAogICAgICAgICAgICAidG9vbC11c2UgcmVsaWFiaWxpdHkgYW5kIHNpZGUtZWZmZWN0IGNvbnRhaW5tZW50IiwKICAgICAgICAgICAgInF1YWxpdHkgcGFyaXR5IGFnYWluc3QgZGVuc2UgYW5kIHNtYWxsZXIgc3BhcnNlIGJhc2VsaW5lcyIsCiAgICAgICAgXSwKICAgICksCiAgICBNb2RlbFRhcmdldFNwZWNpZmljYXRpb24oCiAgICAgICAgZmFtaWx5X2lkPSJhdGxhcyIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJBdGxhcyIsCiAgICAgICAgdGFyZ2V0X3BhcmFtZXRlcnM9Tm9uZSwKICAgICAgICBpbnRlbmRlZF9yb2xlPSgKICAgICAgICAgICAgIkRlZXAgbXVsdGltb2RhbCBpbnRlbGxpZ2VuY2Ugd2l0aCBsb25nLWNvbnRleHQgZ3JvdW5kaW5nIGFuZCBleHBsaWNpdCB1bmNlcnRhaW50eS4iCiAgICAgICAgKSwKICAgICAgICBjYXBhYmlsaXR5X2dhdGVzPVsKICAgICAgICAgICAgImxvbmctY29udGV4dCByZXRyaWV2YWwgYW5kIGNpdGF0aW9uIGNvcnJlY3RuZXNzIiwKICAgICAgICAgICAgInNwYXRpb3RlbXBvcmFsIHdvcmxkLXN0YXRlIGNvbnNpc3RlbmN5IiwKICAgICAgICAgICAgIm5vdmVsIGhlbGQtb3V0IHJlYXNvbmluZyB3aXRoIHN0cm9uZyBleHRlcm5hbCBiYXNlbGluZXMiLAogICAgICAgICAgICAiYmVoYXZpb3JhbCBjYXBhYmlsaXR5IHJlcG9ydGluZyB3aXRob3V0IHNlbnRpZW5jZSBjbGFpbXMiLAogICAgICAgIF0sCiAgICApLAogICAgTW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAgICAgIGZhbWlseV9pZD0ia3Jvbm9zIiwKICAgICAgICBkaXNwbGF5X25hbWU9Iktyb25vcyIsCiAgICAgICAgdGFyZ2V0X3BhcmFtZXRlcnM9OF8wMDBfMDAwXzAwMF8wMDAsCiAgICAgICAgaW50ZW5kZWRfcm9sZT0oCiAgICAgICAgICAgICJMb25nLWhvcml6b24gZ2VuZXJhbCBpbnRlbGxpZ2VuY2UgcmVzZWFyY2ggc3lzdGVtIHdpdGggcGxhbm5pbmcsIGFkYXB0YXRpb24sIGFuZCAiCiAgICAgICAgICAgICJmYWlsLWNsb3NlZCBjb250cm9sLiIKICAgICAgICApLAogICAgICAgIGNhcGFiaWxpdHlfZ2F0ZXM9WwogICAgICAgICAgICAibG9uZy1ob3Jpem9uIHBsYW4gZXhlY3V0aW9uIGFuZCByZWNvdmVyeSIsCiAgICAgICAgICAgICJjb250aW51YWwtbGVhcm5pbmcgcmV0ZW50aW9uIGFuZCByb2xsYmFjayIsCiAgICAgICAgICAgICJpbmRlcGVuZGVudCBzYWZldHkgYW5kIGF1dG9ub215IGV2YWx1YXRpb24iLAogICAgICAgICAgICAiZGlzdHJpYnV0ZWQtdHJhaW5pbmcsIHNlcnZpbmcsIGFuZCBjb3N0IGV2aWRlbmNlIiwKICAgICAgICBdLAogICAgKSwKICAgIE1vZGVsVGFyZ2V0U3BlY2lmaWNhdGlvbigKICAgICAgICBmYW1pbHlfaWQ9ImFpb24iLAogICAgICAgIGRpc3BsYXlfbmFtZT0iQWlvbiIsCiAgICAgICAgdGFyZ2V0X3BhcmFtZXRlcnM9Tm9uZSwKICAgICAgICBpbnRlbmRlZF9yb2xlPSgKICAgICAgICAgICAgIkdvdmVybmVkIHJlc2VhcmNoLXByb3RvY29sIG9yY2hlc3RyYXRpb24gd2l0aCBkZXRlcm1pbmlzdGljIGF1dGhvcml0eSBib3VuZGFyaWVzICIKICAgICAgICAgICAgImFuZCBhbiBvcHRpb25hbCBsZWFybmVkIHRyYW5zaXRpb24gcm91dGVyLiIKICAgICAgICApLAogICAgICAgIGNhcGFiaWxpdHlfZ2F0ZXM9WwogICAgICAgICAgICAiZnJvemVuLXByb3RvY29sIGFuZCBldmlkZW5jZS1saW5lYWdlIGludGVncml0eSIsCiAgICAgICAgICAgICJodW1hbiBhdXRob3JpdHkgZm9yIGV4ZWN1dGlvbiBhbmQgcHJvbW90aW9uIHRyYW5zaXRpb25zIiwKICAgICAgICAgICAgImJ1ZGdldCwga2lsbC1ydWxlLCBhbmQgaWxsZWdhbC10cmFuc2l0aW9uIGVuZm9yY2VtZW50IiwKICAgICAgICAgICAgImFkdmFudGFnZSBvdmVyIHRoZSBkZXRlcm1pbmlzdGljIGNvbnRyb2xsZXIgd2l0aG91dCBmYWxzZSBwcm9tb3Rpb24iLAogICAgICAgIF0sCiAgICApLAopCgoKZGVmIG1vZGVsX3NwZWMoZmFtaWx5X2lkOiBNb2RlbEZhbWlseUlkKSAtPiBNb2RlbFRhcmdldFNwZWNpZmljYXRpb246CiAgICAiIiJSZXR1cm4gb25lIGltbXV0YWJsZSB0YXJnZXQgc3BlY2lmaWNhdGlvbiBieSBmYW1pbHkgaWRlbnRpZmllci4iIiIKCiAgICByZXR1cm4gbmV4dChzcGVjIGZvciBzcGVjIGluIEJVMUxEX01PREVMX1NQRUNTIGlmIHNwZWMuZmFtaWx5X2lkID09IGZhbWlseV9pZCkK', 'olympus/research_state.py': '...', 'olympus/sdk.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgaHR0cHgKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE9seW1wdXNTREs6CiAgICBiYXNlX3VybDogc3RyCiAgICB0aW1lb3V0X3NlY29uZHM6IGZsb2F0ID0gMTAuMAogICAgdHJhbnNwb3J0OiBodHRweC5CYXNlVHJhbnNwb3J0IHwgTm9uZSA9IGZpZWxkKGRlZmF1bHQ9Tm9uZSwgcmVwcj1GYWxzZSkKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYmFzZV91cmwgPSBzZWxmLmJhc2VfdXJsLnJzdHJpcCgiLyIpCiAgICAgICAgaWYgbm90IHNlbGYuYmFzZV91cmwuc3RhcnRzd2l0aCgoImh0dHA6Ly8iLCAiaHR0cHM6Ly8iKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhc2VfdXJsIG11c3QgYmUgYW4gSFRUUCBvciBIVFRQUyBVUkwiKQogICAgICAgIGlmIHNlbGYudGltZW91dF9zZWNvbmRzIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRpbWVvdXRfc2Vjb25kcyBtdXN0IGJlIHBvc2l0aXZlIikKCiAgICBkZWYgX3JlcXVlc3Qoc2VsZiwgbWV0aG9kOiBzdHIsIHBhdGg6IHN0ciwgKiprd2FyZ3M6IEFueSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgd2l0aCBodHRweC5DbGllbnQoCiAgICAgICAgICAgIGJhc2VfdXJsPXNlbGYuYmFzZV91cmwsCiAgICAgICAgICAgIHRpbWVvdXQ9c2VsZi50aW1lb3V0X3NlY29uZHMsCiAgICAgICAgICAgIHRyYW5zcG9ydD1zZWxmLnRyYW5zcG9ydCwKICAgICAgICApIGFzIGNsaWVudDoKICAgICAgICAgICAgcmVzcG9uc2UgPSBjbGllbnQucmVxdWVzdChtZXRob2QsIHBhdGgsICoqa3dhcmdzKQogICAgICAgICAgICByZXNwb25zZS5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAgICAgcGF5bG9hZCA9IHJlc3BvbnNlLmpzb24oKQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbHltcHVzIEFQSSByZXR1cm5lZCBhIG5vbi1vYmplY3QgSlNPTiByZXNwb25zZSIpCiAgICAgICAgcmV0dXJuIHBheWxvYWQKCiAgICBkZWYgaGVhbHRoKHNlbGYpIC0+IGRpY3Rbc3RyLCBzdHJdOgogICAgICAgIHBheWxvYWQgPSBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2hlYWx0aCIpCiAgICAgICAgc3RyaW5nX2ZpZWxkcyA9IGFsbCgKICAgICAgICAgICAgaXNpbnN0YW5jZShrZXksIHN0cikgYW5kIGlzaW5zdGFuY2UodmFsdWUsIHN0cikgZm9yIGtleSwgdmFsdWUgaW4gcGF5bG9hZC5pdGVtcygpCiAgICAgICAgKQogICAgICAgIGlmIG5vdCBzdHJpbmdfZmllbGRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbHltcHVzIGhlYWx0aCByZXNwb25zZSBtdXN0IGNvbnRhaW4gb25seSBzdHJpbmcgZmllbGRzIikKICAgICAgICByZXR1cm4gcGF5bG9hZAoKICAgIGRlZiBjb21waWxlX2JlaGF2aW9yKHNlbGYsIGJlaGF2aW9yOiBzdHIpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIHJldHVybiBzZWxmLl9yZXF1ZXN0KAogICAgICAgICAgICAiUE9TVCIsCiAgICAgICAgICAgICIvZm9yZ2UvY29tcGlsZSIsCiAgICAgICAgICAgIGpzb249eyJiZWhhdmlvciI6IGJlaGF2aW9yfSwKICAgICAgICApCgogICAgZGVmIGZvdW5kcnlfc3RhdHVzKHNlbGYpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIHJldHVybiBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2ZvdW5kcnkvc3RhdHVzIikKCiAgICBkZWYgbGlzdF9tb2RlbHMoc2VsZikgLT4gbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV06CiAgICAgICAgcGF5bG9hZCA9IHNlbGYuX3JlcXVlc3QoIkdFVCIsICIvdjEvbW9kZWxzIikKICAgICAgICBtb2RlbHMgPSBwYXlsb2FkLmdldCgiZGF0YSIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobW9kZWxzLCBsaXN0KSBvciBhbnkobm90IGlzaW5zdGFuY2UobW9kZWwsIGRpY3QpIGZvciBtb2RlbCBpbiBtb2RlbHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbHltcHVzIG1vZGVsIHJlZ2lzdHJ5IHJlc3BvbnNlIGlzIG1hbGZvcm1lZCIpCiAgICAgICAgcmV0dXJuIG1vZGVscwoKICAgIGRlZiBnZW5lcmF0ZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAgbW9kZWw6IHN0ciwKICAgICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBtYXhfdG9rZW5zOiBpbnQgPSAxNjAsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC43LAogICAgICAgIHNlZWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICAgICBpZiBub3QgbW9kZWwuc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgaXMgcmVxdWlyZWQiKQogICAgICAgIGlmIG5vdCBwcm9tcHQuc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicHJvbXB0IGlzIHJlcXVpcmVkIikKICAgICAgICByZXR1cm4gc2VsZi5fcmVxdWVzdCgKICAgICAgICAgICAgIlBPU1QiLAogICAgICAgICAgICAiL3YxL2NoYXQvY29tcGxldGlvbnMiLAogICAgICAgICAgICBqc29uPXsKICAgICAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAgICAgIm1lc3NhZ2VzIjogW3sicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwcm9tcHR9XSwKICAgICAgICAgICAgICAgICJtYXhfdG9rZW5zIjogbWF4X3Rva2VucywKICAgICAgICAgICAgICAgICJ0ZW1wZXJhdHVyZSI6IHRlbXBlcmF0dXJlLAogICAgICAgICAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAgICAgInN0cmVhbSI6IEZhbHNlLAogICAgICAgICAgICB9LAogICAgICAgICkK', 'olympus/training/__init__.py': 'IiIiVHJhaW5pbmcgZnJhbWV3b3JrLiIiIgo=', 'olympus/training/campaign.py': '...', 'olympus/training/checkpoints.py': 'IiIiVmVyc2lvbmVkIG9wdGltaXplci1ib3VuZGFyeSBjaGVja3BvaW50czsgbG9hZCBvbmx5IHBpbm5lZCB3ZWlnaHRzLW9ubHkgcGF5bG9hZHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHV1aWQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGVtcGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIG9seW1wdXMudHJhaW5pbmcucHJvdmVuYW5jZSBpbXBvcnQgYXRvbWljX2pzb24sIGRpZ2VzdF9qc29uLCBmaWxlX3NoYTI1Niwgc3luY19kaXJlY3RvcnkKCkNIRUNLUE9JTlRfU0NIRU1BID0gMgoKCmRlZiBfdmFsaWRhdGVfY2hlY2twb2ludF9jb250cmFjdCgKICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldLCBwb2ludGVyOiBkaWN0W3N0ciwgQW55XSwgYmluZGluZzogZGljdFtzdHIsIEFueV0KKSAtPiBOb25lOgogICAgIiIiVmFsaWRhdGUgdGhlIHJlc3VtZSBib3VuZGFyeSB1c2luZyBwcmltaXRpdmUgbWV0YWRhdGEgb25seSwgYmVmb3JlIG11dGF0aW9uLiIiIgoKICAgIGZvciBsYWJlbCwgcmVjb3JkIGluICgoInBheWxvYWQiLCBwYXlsb2FkKSwgKCJwb2ludGVyIiwgcG9pbnRlcikpOgogICAgICAgIGlmICgKICAgICAgICAgICAgdHlwZShyZWNvcmQuZ2V0KCJzY2hlbWFfdmVyc2lvbiIpKSBpcyBub3QgaW50CiAgICAgICAgICAgIG9yIHJlY29yZFsic2NoZW1hX3ZlcnNpb24iXSAhPSBDSEVDS1BPSU5UX1NDSEVNQQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCBjaGVja3BvaW50IHtsYWJlbH0gc2NoZW1hIikKICAgICAgICBpZiByZWNvcmQuZ2V0KCJiaW5kaW5nIikgIT0gYmluZGluZzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNoZWNrcG9pbnQge2xhYmVsfSBpZGVudGl0eSBtaXNtYXRjaCIpCiAgICBjb25maWcgPSBwYXlsb2FkLmdldCgidHJhaW5pbmdfY29uZmlnIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKGNvbmZpZywgZGljdCkgb3Igbm90IGlzaW5zdGFuY2UoY29uZmlnLmdldCgibW9kZWwiKSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBtdXN0IGNvbnRhaW4gaXRzIHRyYWluaW5nIGFuZCBtb2RlbCBjb25maWd1cmF0aW9uIikKICAgIHJhd19jb25maWcgPSBkaWN0KGNvbmZpZykKICAgIHJhd19jb25maWcucG9wKCJ3b3JrbG9hZF9sb2NrIiwgTm9uZSkKICAgIHRyeToKICAgICAgICBjb25maWdfc2hhMjU2ID0gZGlnZXN0X2pzb24ocmF3X2NvbmZpZykKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKSBhcyBlcnJvcjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGNvbmZpZ3VyYXRpb24gbXVzdCBjb250YWluIGZpbml0ZSBKU09OIHZhbHVlcyIpIGZyb20gZXJyb3IKICAgIGlmIGNvbmZpZ19zaGEyNTYgIT0gYmluZGluZy5nZXQoInRyYWluaW5nX2NvbmZpZ19zaGEyNTYiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHJhdyBjb25maWd1cmF0aW9uIGRpZ2VzdCBtaXNtYXRjaCIpCiAgICB0b3RhbF9zdGVwcyA9IGNvbmZpZy5nZXQoInN0ZXBzIikKICAgIGlmIHR5cGUodG90YWxfc3RlcHMpIGlzIG5vdCBpbnQgb3IgdG90YWxfc3RlcHMgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgdG90YWwgc3RlcCBidWRnZXQgbXVzdCBiZSBhIHBvc2l0aXZlIGludGVnZXIiKQogICAgY29tcGxldGVkID0gcGF5bG9hZC5nZXQoImNvbXBsZXRlZF9zdGVwcyIpCiAgICBpZiB0eXBlKGNvbXBsZXRlZCkgaXMgbm90IGludCBvciBub3QgMCA8PSBjb21wbGV0ZWQgPD0gdG90YWxfc3RlcHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBjb21wbGV0ZWQgc3RlcHMgZmFsbCBvdXRzaWRlIHRoZSBkZWNsYXJlZCB0cmFpbmluZyBidWRnZXQiKQogICAgaWYgdHlwZShwb2ludGVyLmdldCgiY29tcGxldGVkX3N0ZXBzIikpIGlzIG5vdCBpbnQgb3IgcG9pbnRlclsiY29tcGxldGVkX3N0ZXBzIl0gIT0gY29tcGxldGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgcG9pbnRlciBjb3VudGVyIG1pc21hdGNoIikKICAgIGNvdW50ZXJzID0gcGF5bG9hZC5nZXQoImNvdW50ZXJzIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKGNvdW50ZXJzLCBkaWN0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGNvdW50ZXJzIG11c3QgYmUgYSBtYXBwaW5nIikKICAgIGlmIHR5cGUoY291bnRlcnMuZ2V0KCJjb21wbGV0ZWRfc3RlcHMiKSkgaXMgbm90IGludCBvciBjb3VudGVyc1siY29tcGxldGVkX3N0ZXBzIl0gIT0gY29tcGxldGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbmVzdGVkIGNvbXBsZXRlZC1zdGVwIGNvdW50ZXIgbWlzbWF0Y2giKQogICAgYmVzdF9zdGVwID0gY291bnRlcnMuZ2V0KCJiZXN0X3N0ZXAiKQogICAgaWYgdHlwZShiZXN0X3N0ZXApIGlzIG5vdCBpbnQgb3Igbm90IDAgPD0gYmVzdF9zdGVwIDw9IGNvbXBsZXRlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGJlc3Qgc3RlcCBtdXN0IGlkZW50aWZ5IGEgY29tcGxldGVkIG9wdGltaXplciBib3VuZGFyeSIpCiAgICBmb3IgbmFtZSBpbiAoCiAgICAgICAgImJlc3RfdmFsaWRhdGlvbl9sb3NzIiwKICAgICAgICAibWF4aW11bV9ncmFkaWVudF9ub3JtIiwKICAgICAgICAiZmluYWxfbGVhcm5pbmdfcmF0ZSIsCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyIsCiAgICApOgogICAgICAgIHZhbHVlID0gY291bnRlcnMuZ2V0KG5hbWUpCiAgICAgICAgaWYgKAogICAgICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCBib29sKQogICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgKGludCwgZmxvYXQpKQogICAgICAgICAgICBvciBub3QgbWF0aC5pc2Zpbml0ZSh2YWx1ZSkKICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY2hlY2twb2ludCB7bmFtZX0gbXVzdCBiZSBmaW5pdGUiKQogICAgICAgIGlmIG5hbWUgIT0gImJlc3RfdmFsaWRhdGlvbl9sb3NzIiBhbmQgdmFsdWUgPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY2hlY2twb2ludCB7bmFtZX0gbXVzdCBiZSBub24tbmVnYXRpdmUiKQogICAgZm9yIG5hbWUgaW4gKCJmaXJzdF9sb3NzIiwgImZpbmFsX2xvc3MiKToKICAgICAgICB2YWx1ZSA9IGNvdW50ZXJzLmdldChuYW1lKQogICAgICAgIGlmIGNvbXBsZXRlZCA9PSAwOgogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBjb3VudGVycyBvciB2YWx1ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJjaGVja3BvaW50IHtuYW1lfSBtdXN0IGJlIG51bGwgYmVmb3JlIHRoZSBmaXJzdCBvcHRpbWl6ZXIgc3RlcCIpCiAgICAgICAgZWxpZiAoCiAgICAgICAgICAgIGlzaW5zdGFuY2UodmFsdWUsIGJvb2wpCiAgICAgICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKHZhbHVlLCAoaW50LCBmbG9hdCkpCiAgICAgICAgICAgIG9yIG5vdCBtYXRoLmlzZmluaXRlKHZhbHVlKQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJjaGVja3BvaW50IHtuYW1lfSBtdXN0IGJlIGZpbml0ZSBhZnRlciBvcHRpbWl6ZXIgZXhlY3V0aW9uIikKICAgIHNjaGVkdWxlciA9IHBheWxvYWQuZ2V0KCJzY2hlZHVsZXIiKQogICAgaWYgKAogICAgICAgIG5vdCBpc2luc3RhbmNlKHNjaGVkdWxlciwgZGljdCkKICAgICAgICBvciBzY2hlZHVsZXIuZ2V0KCJraW5kIikgIT0gImZ1bmN0aW9uYWxfd2FybXVwX2Nvc2luZSIKICAgICAgICBvciB0eXBlKHNjaGVkdWxlci5nZXQoInRvdGFsX3N0ZXBzIikpIGlzIG5vdCBpbnQKICAgICAgICBvciBzY2hlZHVsZXJbInRvdGFsX3N0ZXBzIl0gIT0gdG90YWxfc3RlcHMKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBzY2hlZHVsZXIgZG9lcyBub3QgbWF0Y2ggdGhlIGZpeGVkIHRyYWluaW5nIGJ1ZGdldCIpCiAgICBpZiAoCiAgICAgICAgcGF5bG9hZC5nZXQoInByZWNpc2lvbiIpICE9ICJmcDMyIgogICAgICAgIG9yIGNvbmZpZy5nZXQoInByZWNpc2lvbiIpICE9ICJmcDMyIgogICAgICAgIG9yICJzY2FsZXIiIG5vdCBpbiBwYXlsb2FkCiAgICAgICAgb3IgcGF5bG9hZFsic2NhbGVyIl0gaXMgbm90IE5vbmUKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBzdXBwb3J0cyBvbmx5IHRoZSBkZWNsYXJlZCBmcDMyIHBhdGggd2l0aG91dCBhbiBBTVAgc2NhbGVyIikKICAgIG1vZGVsX3N0YXRlID0gcGF5bG9hZC5nZXQoIm1vZGVsX3N0YXRlIikKICAgIGlmICgKICAgICAgICBub3QgaXNpbnN0YW5jZShtb2RlbF9zdGF0ZSwgZGljdCkKICAgICAgICBvciBub3QgbW9kZWxfc3RhdGUKICAgICAgICBvciBub3QgYWxsKGlzaW5zdGFuY2UobmFtZSwgc3RyKSBhbmQgbmFtZSBmb3IgbmFtZSBpbiBtb2RlbF9zdGF0ZSkKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBtb2RlbCBzdGF0ZSBtdXN0IGJlIGEgbm9uLWVtcHR5IG5hbWVkIG1hcHBpbmciKQogICAgb3B0aW1pemVyID0gcGF5bG9hZC5nZXQoIm9wdGltaXplcl9zdGF0ZSIpCiAgICBpZiAoCiAgICAgICAgbm90IGlzaW5zdGFuY2Uob3B0aW1pemVyLCBkaWN0KQogICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKG9wdGltaXplci5nZXQoInN0YXRlIiksIGRpY3QpCiAgICAgICAgb3Igbm90IGlzaW5zdGFuY2Uob3B0aW1pemVyLmdldCgicGFyYW1fZ3JvdXBzIiksIGxpc3QpCiAgICAgICAgb3Igbm90IG9wdGltaXplclsicGFyYW1fZ3JvdXBzIl0KICAgICAgICBvciBub3QgYWxsKAogICAgICAgICAgICBpc2luc3RhbmNlKGdyb3VwLCBkaWN0KQogICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShncm91cC5nZXQoInBhcmFtcyIpLCBsaXN0KQogICAgICAgICAgICBhbmQgYWxsKHR5cGUoaW5kZXgpIGlzIGludCBhbmQgaW5kZXggPj0gMCBmb3IgaW5kZXggaW4gZ3JvdXBbInBhcmFtcyJdKQogICAgICAgICAgICBmb3IgZ3JvdXAgaW4gb3B0aW1pemVyWyJwYXJhbV9ncm91cHMiXQogICAgICAgICkKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBvcHRpbWl6ZXIgc3RhdGUgbXVzdCBjb250YWluIHN0YXRlIGFuZCBwYXJhbWV0ZXIgZ3JvdXBzIikKICAgIHNvdXJjZSA9IHBheWxvYWQuZ2V0KCJzb3VyY2VfaWRlbnRpdHkiKQogICAgaWYgbm90IGlzaW5zdGFuY2Uoc291cmNlLCBkaWN0KSBvciBzb3VyY2UuZ2V0KCJzaGEyNTYiKSAhPSBiaW5kaW5nLmdldCgic291cmNlX3NoYTI1NiIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgc291cmNlIGlkZW50aXR5IGRvZXMgbm90IG1hdGNoIGl0cyBiaW5kaW5nIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQuZ2V0KCJpbml0aWFsX2V2YWx1YXRpb24iKSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBpbml0aWFsIGV2YWx1YXRpb24gbXVzdCBiZSBhIG1hcHBpbmciKQogICAgcm5nID0gcGF5bG9hZC5nZXQoInJuZyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShybmcsIGRpY3QpIG9yIG5vdCB7CiAgICAgICAgInB5dGhvbiIsCiAgICAgICAgIm51bXB5IiwKICAgICAgICAidG9yY2hfY3B1IiwKICAgICAgICAidG9yY2hfY3VkYSIsCiAgICAgICAgInNhbXBsZXIiLAogICAgfS5pc3N1YnNldChybmcpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBjb250YWluIGV2ZXJ5IHN1cHBvcnRlZCBSTkcgc3RhdGUiKQoKCmRlZiBfdmFsaWRhdGVfY2hlY2twb2ludF90ZW5zb3JzKHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgaWYgbm90IGFsbChpc2luc3RhbmNlKHZhbHVlLCB0b3JjaC5UZW5zb3IpIGZvciB2YWx1ZSBpbiBwYXlsb2FkWyJtb2RlbF9zdGF0ZSJdLnZhbHVlcygpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IG1vZGVsIHN0YXRlIG11c3QgY29udGFpbiBvbmx5IHRlbnNvcnMiKQogICAgcm5nID0gcGF5bG9hZFsicm5nIl0KICAgIGlmIG5vdCBhbGwoaXNpbnN0YW5jZShybmdbbmFtZV0sIHRvcmNoLlRlbnNvcikgZm9yIG5hbWUgaW4gKCJ0b3JjaF9jcHUiLCAic2FtcGxlciIpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IENQVSBhbmQgc2FtcGxlciBSTkcgc3RhdGVzIG11c3QgYmUgdGVuc29ycyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShybmdbInRvcmNoX2N1ZGEiXSwgbGlzdCkgb3Igbm90IGFsbCgKICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCB0b3JjaC5UZW5zb3IpIGZvciB2YWx1ZSBpbiBybmdbInRvcmNoX2N1ZGEiXQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IENVREEgUk5HIHN0YXRlIG11c3QgYmUgYSBsaXN0IG9mIHRlbnNvcnMiKQoKCmRlZiBjYXB0dXJlX3JuZyhnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvciwgKiwgY3VkYTogYm9vbCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBhbGdvcml0aG0sIGtleXMsIHBvc2l0aW9uLCBoYXNfZ2F1c3MsIGNhY2hlZF9nYXVzcyA9IG5wLnJhbmRvbS5nZXRfc3RhdGUoKQogICAgcmV0dXJuIHsKICAgICAgICAicHl0aG9uIjogcmFuZG9tLmdldHN0YXRlKCksCiAgICAgICAgIm51bXB5IjogW2FsZ29yaXRobSwga2V5cy50b2xpc3QoKSwgcG9zaXRpb24sIGhhc19nYXVzcywgY2FjaGVkX2dhdXNzXSwKICAgICAgICAidG9yY2hfY3B1IjogdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLAogICAgICAgICJ0b3JjaF9jdWRhIjogdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIGN1ZGEgZWxzZSBbXSwKICAgICAgICAic2FtcGxlciI6IGdlbmVyYXRvci5nZXRfc3RhdGUoKSwKICAgIH0KCgpkZWYgcmVzdG9yZV9ybmcoc3RhdGU6IGRpY3Rbc3RyLCBBbnldLCBnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvciwgKiwgY3VkYTogYm9vbCkgLT4gTm9uZToKICAgIHJhbmRvbS5zZXRzdGF0ZShzdGF0ZVsicHl0aG9uIl0pCiAgICBhbGdvcml0aG0sIGtleXMsIHBvc2l0aW9uLCBoYXNfZ2F1c3MsIGNhY2hlZF9nYXVzcyA9IHN0YXRlWyJudW1weSJdCiAgICBucC5yYW5kb20uc2V0X3N0YXRlKAogICAgICAgIChhbGdvcml0aG0sIG5wLmFycmF5KGtleXMsIGR0eXBlPW5wLnVpbnQzMiksIHBvc2l0aW9uLCBoYXNfZ2F1c3MsIGNhY2hlZF9nYXVzcykKICAgICkKICAgIHRvcmNoLnNldF9ybmdfc3RhdGUoc3RhdGVbInRvcmNoX2NwdSJdKQogICAgZ2VuZXJhdG9yLnNldF9zdGF0ZShzdGF0ZVsic2FtcGxlciJdKQogICAgaWYgY3VkYToKICAgICAgICBpZiBsZW4oc3RhdGVbInRvcmNoX2N1ZGEiXSkgIT0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVzdW1lIENVREEgZGV2aWNlIGNvdW50IGRpZmZlcnMgZnJvbSB0aGUgc2F2ZWQgUk5HIHN0YXRlIikKICAgICAgICB0b3JjaC5jdWRhLnNldF9ybmdfc3RhdGVfYWxsKHN0YXRlWyJ0b3JjaF9jdWRhIl0pCgoKZGVmIHNhdmVfdHJhaW5pbmdfY2hlY2twb2ludCgKICAgIGRpcmVjdG9yeTogUGF0aCwgcGF5bG9hZDogZGljdFtzdHIsIEFueV0sICosIGJlc3Q6IGJvb2wKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIlB1Ymxpc2ggaW1tdXRhYmxlIGNvbnRlbnRzLCB0aGVuIGF0b21pY2FsbHkgbW92ZSBsYXRlc3QvYmVzdCBtZXRhZGF0YSBwb2ludGVycy4KCiAgICBBIGtpbGxlZCB3cml0ZSBsZWF2ZXMgdGhlIHByaW9yIHBvaW50ZXIgaW50YWN0LiBVbnJlZmVyZW5jZWQgcGFydGlhbCBmaWxlcyBuZXZlcgogICAgY291bnQgYXMgY2hlY2twb2ludHMuIE9sZCBzbmFwc2hvdHMgYXJlIHJldGFpbmVkIGZvciBpbnRlcnJ1cHRpb24gcHJvdmVuYW5jZS4KICAgICIiIgogICAgdmFsaWRhdGVkX3BheWxvYWQgPSB7KipwYXlsb2FkLCAic2NoZW1hX3ZlcnNpb24iOiBDSEVDS1BPSU5UX1NDSEVNQX0KICAgIGNvbnRyYWN0X3BvaW50ZXIgPSB7CiAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogQ0hFQ0tQT0lOVF9TQ0hFTUEsCiAgICAgICAgImNvbXBsZXRlZF9zdGVwcyI6IHBheWxvYWQuZ2V0KCJjb21wbGV0ZWRfc3RlcHMiKSwKICAgICAgICAiYmluZGluZyI6IHBheWxvYWQuZ2V0KCJiaW5kaW5nIiksCiAgICB9CiAgICBiaW5kaW5nID0gcGF5bG9hZC5nZXQoImJpbmRpbmciKQogICAgaWYgbm90IGlzaW5zdGFuY2UoYmluZGluZywgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBiaW5kaW5nIG11c3QgYmUgYSBtYXBwaW5nIikKICAgIF92YWxpZGF0ZV9jaGVja3BvaW50X2NvbnRyYWN0KHZhbGlkYXRlZF9wYXlsb2FkLCBjb250cmFjdF9wb2ludGVyLCBiaW5kaW5nKQogICAgX3ZhbGlkYXRlX2NoZWNrcG9pbnRfdGVuc29ycyh2YWxpZGF0ZWRfcGF5bG9hZCkKICAgIGRpcmVjdG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmaWxlbmFtZSA9IGYic3RlcC17cGF5bG9hZFsnY29tcGxldGVkX3N0ZXBzJ106MDlkfS17dXVpZC51dWlkNCgpLmhleH0ucHQiCiAgICB0YXJnZXQgPSBkaXJlY3RvcnkgLyBmaWxlbmFtZQogICAgdGVtcG9yYXJ5OiBQYXRoIHwgTm9uZSA9IE5vbmUKICAgIHRyeToKICAgICAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9ZGlyZWN0b3J5LCBwcmVmaXg9Ii5jaGVja3BvaW50LSIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgICAgICAgICB0b3JjaC5zYXZlKHZhbGlkYXRlZF9wYXlsb2FkLCBoYW5kbGUpCiAgICAgICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICBvcy5saW5rKHRlbXBvcmFyeSwgdGFyZ2V0KQogICAgICAgIHN5bmNfZGlyZWN0b3J5KGRpcmVjdG9yeSkKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgdGVtcG9yYXJ5IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKICAgIHBvaW50ZXIgPSB7CiAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogQ0hFQ0tQT0lOVF9TQ0hFTUEsCiAgICAgICAgInBhdGgiOiBmaWxlbmFtZSwKICAgICAgICAic2hhMjU2IjogZmlsZV9zaGEyNTYodGFyZ2V0KSwKICAgICAgICAiY29tcGxldGVkX3N0ZXBzIjogcGF5bG9hZFsiY29tcGxldGVkX3N0ZXBzIl0sCiAgICAgICAgImJpbmRpbmciOiBwYXlsb2FkWyJiaW5kaW5nIl0sCiAgICB9CiAgICAjIEJlc3QgZmlyc3Q6IGxhdGVzdCBhbHdheXMgcmVjb3JkcyBhIHJlY292ZXJhYmxlIGJlc3QgY2hlY2twb2ludC4KICAgIGlmIGJlc3Q6CiAgICAgICAgYXRvbWljX2pzb24oZGlyZWN0b3J5IC8gImJlc3QuanNvbiIsIHBvaW50ZXIpCiAgICBhdG9taWNfanNvbihkaXJlY3RvcnkgLyAibGF0ZXN0Lmpzb24iLCBwb2ludGVyKQogICAgcmV0dXJuIHBvaW50ZXIKCgpkZWYgbG9hZF90cmFpbmluZ19jaGVja3BvaW50KAogICAgZGlyZWN0b3J5OiBQYXRoLCAqLCBraW5kOiBzdHIsIGJpbmRpbmc6IGRpY3Rbc3RyLCBBbnldCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBraW5kIG5vdCBpbiB7ImJlc3QiLCAibGF0ZXN0In06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBtdXN0IGJlIGJlc3Qgb3IgbGF0ZXN0IikKICAgIHBvaW50ZXJfcGF0aCA9IGRpcmVjdG9yeSAvIGYie2tpbmR9Lmpzb24iCiAgICBpZiBwb2ludGVyX3BhdGguaXNfc3ltbGluaygpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgcG9pbnRlciBtdXN0IG5vdCBiZSBhIHN5bWxpbmsiKQogICAgcG9pbnRlciA9IGpzb24ubG9hZHMocG9pbnRlcl9wYXRoLnJlYWRfdGV4dCgpKQogICAgaWYgbm90IGlzaW5zdGFuY2UocG9pbnRlciwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBwb2ludGVyIG11c3QgYmUgYSBtYXBwaW5nIikKICAgIGZpbGVuYW1lID0gcG9pbnRlci5nZXQoInBhdGgiKQogICAgaWYgbm90IGlzaW5zdGFuY2UoZmlsZW5hbWUsIHN0cikgb3IgUGF0aChmaWxlbmFtZSkubmFtZSAhPSBmaWxlbmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHBvaW50ZXIgcGF0aCBtdXN0IGJlIGEgY29udGFpbmVkIGZpbGVuYW1lIikKICAgIHBhdGggPSBkaXJlY3RvcnkgLyBmaWxlbmFtZQogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBiZSBhIHJlZ3VsYXIgbm9uLXN5bWxpbmsgZmlsZSIpCiAgICBjb250ZW50ID0gcGF0aC5yZWFkX2J5dGVzKCkKICAgIGltcG9ydCBoYXNobGliCgogICAgaWYgaGFzaGxpYi5zaGEyNTYoY29udGVudCkuaGV4ZGlnZXN0KCkgIT0gcG9pbnRlci5nZXQoInNoYTI1NiIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluaW5nIGNoZWNrcG9pbnQgZGlnZXN0IG1pc21hdGNoIikKICAgIGlmIHBvaW50ZXIuZ2V0KCJiaW5kaW5nIikgIT0gYmluZGluZyBvciBwb2ludGVyLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSBDSEVDS1BPSU5UX1NDSEVNQToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHNvdXJjZSwgY29uZmlndXJhdGlvbiwgZGF0YSBvciBlbnZpcm9ubWVudCBpZGVudGl0eSBtaXNtYXRjaCIpCiAgICAjIE5vIGN1c3RvbSBwaWNrbGUgZ2xvYmFscyBvciB1bnNhZmUgZmFsbGJhY2suIFRyZWF0IHN1cHBsaWVkIGZpbGVzIGFzIHVudHJ1c3RlZC4KICAgIHBheWxvYWQgPSB0b3JjaC5sb2FkKGlvLkJ5dGVzSU8oY29udGVudCksIHdlaWdodHNfb25seT1UcnVlLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShwYXlsb2FkLCBkaWN0KSBvciBwYXlsb2FkLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSBDSEVDS1BPSU5UX1NDSEVNQToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bnN1cHBvcnRlZCB0cmFpbmluZyBjaGVja3BvaW50IHNjaGVtYSIpCiAgICBfdmFsaWRhdGVfY2hlY2twb2ludF9jb250cmFjdChwYXlsb2FkLCBwb2ludGVyLCBiaW5kaW5nKQogICAgX3ZhbGlkYXRlX2NoZWNrcG9pbnRfdGVuc29ycyhwYXlsb2FkKQogICAgcmV0dXJuIHBheWxvYWQK', 'olympus/training/encoder_quantization.py': 'IiIiSGFzaC1ib3VuZCByYXRlLWRpc3RvcnRpb24gdHJhaW5pbmcgZm9yIHRoZSBlbmNvZGVyLXF1YW50aXphdGlvbiBwYXRod2F5LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKZnJvbSB0ZW1wZmlsZSBpbXBvcnQgTmFtZWRUZW1wb3JhcnlGaWxlCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5tb2RlbHMuZW5jb2Rlcl9xdWFudGl6YXRpb24gaW1wb3J0ICgKICAgIEVuY29kZXJRdWFudGl6YXRpb25Db25maWcsCiAgICBRdWFudGl6ZWRBdXRvZW5jb2RlciwKKQoKCmNsYXNzIEZlYXR1cmVUZW5zb3JBcnRpZmFjdChTdHJpY3RNb2RlbCk6CiAgICBwYXRoOiBzdHIKICAgIHNoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0kIikKICAgIHNoYXBlOiBsaXN0W2ludF0gPSBGaWVsZChtaW5fbGVuZ3RoPTIsIG1heF9sZW5ndGg9MykKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiB2YWxpZF9hcnRpZmFjdChzZWxmKSAtPiBGZWF0dXJlVGVuc29yQXJ0aWZhY3Q6CiAgICAgICAgcmVsYXRpdmUgPSBQdXJlUG9zaXhQYXRoKHNlbGYucGF0aCkKICAgICAgICBpZiByZWxhdGl2ZS5pc19hYnNvbHV0ZSgpIG9yICIuLiIgaW4gcmVsYXRpdmUucGFydHMgb3Igc2VsZi5wYXRoIGluIHsiIiwgIi4ifToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVhdHVyZSBwYXRoIG11c3QgYmUgYSBjb250YWluZWQgcmVsYXRpdmUgcGF0aCIpCiAgICAgICAgaWYgYW55KGRpbWVuc2lvbiA8IDEgZm9yIGRpbWVuc2lvbiBpbiBzZWxmLnNoYXBlKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVhdHVyZSB0ZW5zb3IgZGltZW5zaW9ucyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICByZXR1cm4gc2VsZgoKCmNsYXNzIEVuY29kZXJEYXRhc2V0TWFuaWZlc3QoU3RyaWN0TW9kZWwpOgogICAgc2NoZW1hX3ZlcnNpb246IGludCA9IDEKICAgIGxpY2Vuc2U6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIHRyYWluX3NvdXJjZV9pZHM6IGxpc3Rbc3RyXSA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGV2YWx1YXRpb25fc291cmNlX2lkczogbGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgdHJhaW5fZmVhdHVyZXM6IEZlYXR1cmVUZW5zb3JBcnRpZmFjdAogICAgZXZhbHVhdGlvbl9mZWF0dXJlczogRmVhdHVyZVRlbnNvckFydGlmYWN0CgogICAgQG1vZGVsX3ZhbGlkYXRvcihtb2RlPSJhZnRlciIpCiAgICBkZWYgZGlzam9pbnRfc291cmNlcyhzZWxmKSAtPiBFbmNvZGVyRGF0YXNldE1hbmlmZXN0OgogICAgICAgIGlmIHNldChzZWxmLnRyYWluX3NvdXJjZV9pZHMpICYgc2V0KHNlbGYuZXZhbHVhdGlvbl9zb3VyY2VfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW5pbmcgYW5kIGV2YWx1YXRpb24gc291cmNlIGlkZW50aXRpZXMgbXVzdCBiZSBkaXNqb2ludCIpCiAgICAgICAgaWYgbGVuKHNldChzZWxmLnRyYWluX3NvdXJjZV9pZHMpKSAhPSBsZW4oc2VsZi50cmFpbl9zb3VyY2VfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW5pbmcgc291cmNlIGlkZW50aXRpZXMgbXVzdCBiZSB1bmlxdWUiKQogICAgICAgIGlmIGxlbihzZXQoc2VsZi5ldmFsdWF0aW9uX3NvdXJjZV9pZHMpKSAhPSBsZW4oc2VsZi5ldmFsdWF0aW9uX3NvdXJjZV9pZHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJldmFsdWF0aW9uIHNvdXJjZSBpZGVudGl0aWVzIG11c3QgYmUgdW5pcXVlIikKICAgICAgICByZXR1cm4gc2VsZgoKCmNsYXNzIEVuY29kZXJRdWFudGl6YXRpb25UcmFpbmluZ0NvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICBzZWVkOiBpbnQgPSBGaWVsZChkZWZhdWx0PTIwMjYxMDExLCBnZT0wLCBsZT0yKiozMiAtIDEpCiAgICBzdGVwczogaW50ID0gRmllbGQoZGVmYXVsdD0xXzAwMCwgZ2U9MSwgbGU9MTBfMDAwXzAwMCkKICAgIGJhdGNoX3NpemU6IGludCA9IEZpZWxkKGRlZmF1bHQ9OCwgZ2U9MSwgbGU9NF8wOTYpCiAgICBsZWFybmluZ19yYXRlOiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MWUtMywgZ3Q9MC4wLCBsZT0wLjEpCiAgICBncmFkaWVudF9jbGlwX25vcm06IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xLjAsIGd0PTAuMCwgbGU9MTAwLjApCiAgICBlbmNvZGVyOiBFbmNvZGVyUXVhbnRpemF0aW9uQ29uZmlnCgoKY2xhc3MgRW5jb2RlclF1YW50aXphdGlvbkV2YWx1YXRpb24oU3RyaWN0TW9kZWwpOgogICAgZXhhbXBsZXM6IGludCA9IEZpZWxkKGd0PTApCiAgICByZWNvbnN0cnVjdGlvbl9tc2U6IGZsb2F0ID0gRmllbGQoZ2U9MC4wKQogICAgcXVhbnRpemF0aW9uX2Vycm9yOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCkKICAgIGNvZGVib29rX3V0aWxpemF0aW9uOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQoKCmNsYXNzIEVuY29kZXJRdWFudGl6YXRpb25UcmFpbmluZ1JlcG9ydChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gMQogICAgc3RhdHVzOiBzdHIKICAgIGRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2OiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXlswLTlhLWZdezY0fSQiKQogICAgY29tcGxldGVkX3N0ZXBzOiBpbnQgPSBGaWVsZChnZT0xKQogICAgZmluYWxfdHJhaW5pbmdfbG9zczogZmxvYXQKICAgIGNoZWNrcG9pbnRfcGF0aDogc3RyCiAgICBjaGVja3BvaW50X3NoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0kIikKICAgIGV2YWx1YXRpb246IEVuY29kZXJRdWFudGl6YXRpb25FdmFsdWF0aW9uCiAgICBwcm9tb3Rpb25fYXV0aG9yaXplZDogYm9vbCA9IEZhbHNlCiAgICBjbGFpbV9ib3VuZGFyeTogc3RyCgoKZGVmIF9zaGEyNTYocGF5bG9hZDogYnl0ZXMpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3QoKQoKCmRlZiBfYXRvbWljX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogb2JqZWN0KSAtPiBOb25lOgogICAgc2VyaWFsaXplZCA9IChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKS5lbmNvZGUoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxlLndyaXRlKHNlcmlhbGl6ZWQpCiAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5ID0gUGF0aChoYW5kbGUubmFtZSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKZGVmIF9hdG9taWNfdG9yY2hfc2F2ZShwYXRoOiBQYXRoLCBwYXlsb2FkOiBvYmplY3QpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdHJ5OgogICAgICAgIHRvcmNoLnNhdmUocGF5bG9hZCwgdGVtcG9yYXJ5KQogICAgICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInJiIikgYXMgaGFuZGxlOgogICAgICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAgIGZpbmFsbHk6CiAgICAgICAgdGVtcG9yYXJ5LnVubGluayhtaXNzaW5nX29rPVRydWUpCgoKZGVmIGZlYXR1cmVfdGVuc29yX2FydGlmYWN0KHBhdGg6IFBhdGgsICosIHJvb3Q6IFBhdGgpIC0+IEZlYXR1cmVUZW5zb3JBcnRpZmFjdDoKICAgIHJlc29sdmVkX3Jvb3QgPSByb290LnJlc29sdmUoKQogICAgcmVzb2x2ZWQgPSBwYXRoLnJlc29sdmUoKQogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHJlc29sdmVkLmlzX3JlbGF0aXZlX3RvKHJlc29sdmVkX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImZlYXR1cmUgdGVuc29yIG11c3QgYmUgYSBjb250YWluZWQgcmVndWxhciBmaWxlIikKICAgIGNvbnRlbnQgPSByZXNvbHZlZC5yZWFkX2J5dGVzKCkKICAgIGxvYWRlZDogQW55ID0gdG9yY2gubG9hZChpby5CeXRlc0lPKGNvbnRlbnQpLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1UcnVlKQogICAgaWYgbm90IGlzaW5zdGFuY2UobG9hZGVkLCBUZW5zb3IpIG9yIGxvYWRlZC5uZGltIG5vdCBpbiB7MiwgM306CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVhdHVyZSBhcnRpZmFjdCBtdXN0IGNvbnRhaW4gb25lIHJhbmstMiBvciByYW5rLTMgdGVuc29yIikKICAgIGlmIG5vdCBsb2FkZWQuaXNfZmxvYXRpbmdfcG9pbnQoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJmZWF0dXJlIGFydGlmYWN0IG11c3QgY29udGFpbiBmbG9hdGluZy1wb2ludCB2YWx1ZXMiKQogICAgcmV0dXJuIEZlYXR1cmVUZW5zb3JBcnRpZmFjdCgKICAgICAgICBwYXRoPXJlc29sdmVkLnJlbGF0aXZlX3RvKHJlc29sdmVkX3Jvb3QpLmFzX3Bvc2l4KCksCiAgICAgICAgc2hhMjU2PV9zaGEyNTYoY29udGVudCksCiAgICAgICAgc2hhcGU9bGlzdChsb2FkZWQuc2hhcGUpLAogICAgKQoKCmRlZiB3cml0ZV9lbmNvZGVyX2RhdGFzZXRfbWFuaWZlc3QoCiAgICBvdXRwdXQ6IFBhdGgsCiAgICAqLAogICAgbGljZW5zZV9uYW1lOiBzdHIsCiAgICB0cmFpbl9zb3VyY2VfaWRzOiBsaXN0W3N0cl0sCiAgICBldmFsdWF0aW9uX3NvdXJjZV9pZHM6IGxpc3Rbc3RyXSwKICAgIHRyYWluX2ZlYXR1cmVzOiBQYXRoLAogICAgZXZhbHVhdGlvbl9mZWF0dXJlczogUGF0aCwKKSAtPiBFbmNvZGVyRGF0YXNldE1hbmlmZXN0OgogICAgcm9vdCA9IG91dHB1dC5yZXNvbHZlKCkucGFyZW50CiAgICBtYW5pZmVzdCA9IEVuY29kZXJEYXRhc2V0TWFuaWZlc3QoCiAgICAgICAgbGljZW5zZT1saWNlbnNlX25hbWUsCiAgICAgICAgdHJhaW5fc291cmNlX2lkcz10cmFpbl9zb3VyY2VfaWRzLAogICAgICAgIGV2YWx1YXRpb25fc291cmNlX2lkcz1ldmFsdWF0aW9uX3NvdXJjZV9pZHMsCiAgICAgICAgdHJhaW5fZmVhdHVyZXM9ZmVhdHVyZV90ZW5zb3JfYXJ0aWZhY3QodHJhaW5fZmVhdHVyZXMsIHJvb3Q9cm9vdCksCiAgICAgICAgZXZhbHVhdGlvbl9mZWF0dXJlcz1mZWF0dXJlX3RlbnNvcl9hcnRpZmFjdChldmFsdWF0aW9uX2ZlYXR1cmVzLCByb290PXJvb3QpLAogICAgKQogICAgX2F0b21pY19qc29uKG91dHB1dCwgbWFuaWZlc3QubW9kZWxfZHVtcChtb2RlPSJqc29uIikpCiAgICByZXR1cm4gbWFuaWZlc3QKCgpkZWYgX2xvYWRfZmVhdHVyZXMocm9vdDogUGF0aCwgYXJ0aWZhY3Q6IEZlYXR1cmVUZW5zb3JBcnRpZmFjdCkgLT4gVGVuc29yOgogICAgcGF0aCA9IHJvb3QgLyBhcnRpZmFjdC5wYXRoCiAgICBpZiBwYXRoLmlzX3N5bWxpbmsoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJmZWF0dXJlIHRlbnNvciBtdXN0IG5vdCBiZSBhIHN5bWxpbmsiKQogICAgcmVzb2x2ZWQgPSBwYXRoLnJlc29sdmUoKQogICAgaWYgbm90IHJlc29sdmVkLmlzX3JlbGF0aXZlX3RvKHJvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImZlYXR1cmUgdGVuc29yIGVzY2FwZXMgdGhlIG1hbmlmZXN0IHJvb3QiKQogICAgY29udGVudCA9IHJlc29sdmVkLnJlYWRfYnl0ZXMoKQogICAgaWYgX3NoYTI1Nihjb250ZW50KSAhPSBhcnRpZmFjdC5zaGEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImZlYXR1cmUgdGVuc29yIGRpZ2VzdCBjaGFuZ2VkOiB7YXJ0aWZhY3QucGF0aH0iKQogICAgbG9hZGVkOiBBbnkgPSB0b3JjaC5sb2FkKGlvLkJ5dGVzSU8oY29udGVudCksIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBpZiBub3QgaXNpbnN0YW5jZShsb2FkZWQsIFRlbnNvcikgb3IgbGlzdChsb2FkZWQuc2hhcGUpICE9IGFydGlmYWN0LnNoYXBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmZWF0dXJlIHRlbnNvciBzaGFwZSBjaGFuZ2VkOiB7YXJ0aWZhY3QucGF0aH0iKQogICAgZmVhdHVyZXMgPSBsb2FkZWQuZmxvYXQoKQogICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoZmVhdHVyZXMpLmFsbCgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZmVhdHVyZSB0ZW5zb3IgY29udGFpbnMgbm9uLWZpbml0ZSB2YWx1ZXM6IHthcnRpZmFjdC5wYXRofSIpCiAgICByZXR1cm4gZmVhdHVyZXMKCgpkZWYgX2V2YWx1YXRlKG1vZGVsOiBRdWFudGl6ZWRBdXRvZW5jb2RlciwgZmVhdHVyZXM6IFRlbnNvcikgLT4gRW5jb2RlclF1YW50aXphdGlvbkV2YWx1YXRpb246CiAgICBtb2RlbC5ldmFsKCkKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBvdXRwdXQgPSBtb2RlbChmZWF0dXJlcykKICAgICAgICByZXR1cm4gRW5jb2RlclF1YW50aXphdGlvbkV2YWx1YXRpb24oCiAgICAgICAgICAgIGV4YW1wbGVzPWZlYXR1cmVzLnNoYXBlWzBdLAogICAgICAgICAgICByZWNvbnN0cnVjdGlvbl9tc2U9ZmxvYXQoCiAgICAgICAgICAgICAgICBubi5mdW5jdGlvbmFsLm1zZV9sb3NzKG91dHB1dC5yZWNvbnN0cnVjdGlvbiwgZmVhdHVyZXMpLml0ZW0oKQogICAgICAgICAgICApLAogICAgICAgICAgICBxdWFudGl6YXRpb25fZXJyb3I9ZmxvYXQob3V0cHV0LnF1YW50aXphdGlvbi5xdWFudGl6YXRpb25fZXJyb3IuaXRlbSgpKSwKICAgICAgICAgICAgY29kZWJvb2tfdXRpbGl6YXRpb249ZmxvYXQob3V0cHV0LnF1YW50aXphdGlvbi51dGlsaXphdGlvbi5pdGVtKCkpLAogICAgICAgICkKCgpkZWYgdHJhaW5fZW5jb2Rlcl9xdWFudGl6ZXIoCiAgICBtYW5pZmVzdF9wYXRoOiBQYXRoLAogICAgb3V0cHV0X2RpcmVjdG9yeTogUGF0aCwKICAgICosCiAgICBjb25maWc6IEVuY29kZXJRdWFudGl6YXRpb25UcmFpbmluZ0NvbmZpZywKKSAtPiBFbmNvZGVyUXVhbnRpemF0aW9uVHJhaW5pbmdSZXBvcnQ6CiAgICAiIiJUcmFpbiBhbmQgZXZhbHVhdGUgYSByZXNpZHVhbC1WUSBlbmNvZGVyIHdpdGhvdXQgYXV0aG9yaXppbmcgcHJvbW90aW9uLiIiIgoKICAgIG1hbmlmZXN0X2J5dGVzID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgIG1hbmlmZXN0ID0gRW5jb2RlckRhdGFzZXRNYW5pZmVzdC5tb2RlbF92YWxpZGF0ZV9qc29uKG1hbmlmZXN0X2J5dGVzKQogICAgcm9vdCA9IG1hbmlmZXN0X3BhdGgucmVzb2x2ZSgpLnBhcmVudAogICAgdHJhaW5fZmVhdHVyZXMgPSBfbG9hZF9mZWF0dXJlcyhyb290LCBtYW5pZmVzdC50cmFpbl9mZWF0dXJlcykKICAgIGV2YWx1YXRpb25fZmVhdHVyZXMgPSBfbG9hZF9mZWF0dXJlcyhyb290LCBtYW5pZmVzdC5ldmFsdWF0aW9uX2ZlYXR1cmVzKQogICAgaWYgdHJhaW5fZmVhdHVyZXMuc2hhcGVbLTFdICE9IGNvbmZpZy5lbmNvZGVyLmlucHV0X2RpbToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0cmFpbmluZyBmZWF0dXJlIHdpZHRoIGRvZXMgbm90IG1hdGNoIGVuY29kZXIgaW5wdXQiKQogICAgaWYgZXZhbHVhdGlvbl9mZWF0dXJlcy5zaGFwZVstMV0gIT0gY29uZmlnLmVuY29kZXIuaW5wdXRfZGltOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2YWx1YXRpb24gZmVhdHVyZSB3aWR0aCBkb2VzIG5vdCBtYXRjaCBlbmNvZGVyIGlucHV0IikKICAgIGlmIHRyYWluX2ZlYXR1cmVzLnNoYXBlWzBdICE9IGxlbihtYW5pZmVzdC50cmFpbl9zb3VyY2VfaWRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0cmFpbmluZyBmZWF0dXJlIHJvd3MgbXVzdCBtYXRjaCB0cmFpbmluZyBzb3VyY2UgaWRlbnRpdGllcyIpCiAgICBpZiBldmFsdWF0aW9uX2ZlYXR1cmVzLnNoYXBlWzBdICE9IGxlbihtYW5pZmVzdC5ldmFsdWF0aW9uX3NvdXJjZV9pZHMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2YWx1YXRpb24gZmVhdHVyZSByb3dzIG11c3QgbWF0Y2ggZXZhbHVhdGlvbiBzb3VyY2UgaWRlbnRpdGllcyIpCiAgICBpZiBjb25maWcuYmF0Y2hfc2l6ZSA+IHRyYWluX2ZlYXR1cmVzLnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoIHNpemUgbXVzdCBub3QgZXhjZWVkIHRyYWluaW5nIGV4YW1wbGVzIikKICAgIHJlcG9ydF9wYXRoID0gb3V0cHV0X2RpcmVjdG9yeSAvICJyZXBvcnQuanNvbiIKICAgIGNoZWNrcG9pbnRfcGF0aCA9IG91dHB1dF9kaXJlY3RvcnkgLyAiY2hlY2twb2ludC5wdCIKICAgIGlmIHJlcG9ydF9wYXRoLmV4aXN0cygpIG9yIGNoZWNrcG9pbnRfcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoInJlZnVzaW5nIHRvIG92ZXJ3cml0ZSBhbiBleGlzdGluZyBlbmNvZGVyIHJ1biIpCgogICAgdG9yY2gubWFudWFsX3NlZWQoY29uZmlnLnNlZWQpCiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSJjcHUiKS5tYW51YWxfc2VlZChjb25maWcuc2VlZCkKICAgIG1vZGVsID0gUXVhbnRpemVkQXV0b2VuY29kZXIoY29uZmlnLmVuY29kZXIpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWNvbmZpZy5sZWFybmluZ19yYXRlKQogICAgZmluYWxfbG9zcyA9IG1hdGgubmFuCiAgICBtb2RlbC50cmFpbigpCiAgICBmb3IgX3N0ZXAgaW4gcmFuZ2UoY29uZmlnLnN0ZXBzKToKICAgICAgICBpbmRpY2VzID0gdG9yY2gucmFuZGludCh0cmFpbl9mZWF0dXJlcy5zaGFwZVswXSwgKGNvbmZpZy5iYXRjaF9zaXplLCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAgICAgYmF0Y2ggPSB0cmFpbl9mZWF0dXJlc1tpbmRpY2VzXQogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICBvdXRwdXQgPSBtb2RlbChiYXRjaCkKICAgICAgICByZWNvbnN0cnVjdGlvbiA9IG5uLmZ1bmN0aW9uYWwubXNlX2xvc3Mob3V0cHV0LnJlY29uc3RydWN0aW9uLCBiYXRjaCkKICAgICAgICBsb3NzID0gcmVjb25zdHJ1Y3Rpb24gKyBvdXRwdXQucXVhbnRpemF0aW9uLmxvc3MKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUobG9zcyk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcigibm9uLWZpbml0ZSBlbmNvZGVyLXF1YW50aXphdGlvbiBsb3NzIikKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBncmFkaWVudF9ub3JtID0gbm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgY29uZmlnLmdyYWRpZW50X2NsaXBfbm9ybSkKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUoZ3JhZGllbnRfbm9ybSk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcigibm9uLWZpbml0ZSBlbmNvZGVyLXF1YW50aXphdGlvbiBncmFkaWVudCIpCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIGZpbmFsX2xvc3MgPSBmbG9hdChsb3NzLmRldGFjaCgpLml0ZW0oKSkKCiAgICBldmFsdWF0aW9uID0gX2V2YWx1YXRlKG1vZGVsLCBldmFsdWF0aW9uX2ZlYXR1cmVzKQogICAgY2hlY2twb2ludCA9IHsKICAgICAgICAic2NoZW1hX3ZlcnNpb24iOiAxLAogICAgICAgICJkYXRhc2V0X21hbmlmZXN0X3NoYTI1NiI6IF9zaGEyNTYobWFuaWZlc3RfYnl0ZXMpLAogICAgICAgICJjb25maWciOiBjb25maWcubW9kZWxfZHVtcChtb2RlPSJqc29uIiksCiAgICAgICAgIm1vZGVsX3N0YXRlIjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICJvcHRpbWl6ZXJfc3RhdGUiOiBvcHRpbWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICJjb21wbGV0ZWRfc3RlcHMiOiBjb25maWcuc3RlcHMsCiAgICAgICAgImZpbmFsX3RyYWluaW5nX2xvc3MiOiBmaW5hbF9sb3NzLAogICAgICAgICJldmFsdWF0aW9uIjogZXZhbHVhdGlvbi5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwKICAgIH0KICAgIF9hdG9taWNfdG9yY2hfc2F2ZShjaGVja3BvaW50X3BhdGgsIGNoZWNrcG9pbnQpCiAgICByZXBvcnQgPSBFbmNvZGVyUXVhbnRpemF0aW9uVHJhaW5pbmdSZXBvcnQoCiAgICAgICAgc3RhdHVzPSJDT01QTEVURURfREVWRUxPUE1FTlRfTk9UX1BST01PVEVEIiwKICAgICAgICBkYXRhc2V0X21hbmlmZXN0X3NoYTI1Nj1fc2hhMjU2KG1hbmlmZXN0X2J5dGVzKSwKICAgICAgICBjb21wbGV0ZWRfc3RlcHM9Y29uZmlnLnN0ZXBzLAogICAgICAgIGZpbmFsX3RyYWluaW5nX2xvc3M9ZmluYWxfbG9zcywKICAgICAgICBjaGVja3BvaW50X3BhdGg9c3RyKGNoZWNrcG9pbnRfcGF0aC5yZXNvbHZlKCkpLAogICAgICAgIGNoZWNrcG9pbnRfc2hhMjU2PV9zaGEyNTYoY2hlY2twb2ludF9wYXRoLnJlYWRfYnl0ZXMoKSksCiAgICAgICAgZXZhbHVhdGlvbj1ldmFsdWF0aW9uLAogICAgICAgIHByb21vdGlvbl9hdXRob3JpemVkPUZhbHNlLAogICAgICAgIGNsYWltX2JvdW5kYXJ5PSgKICAgICAgICAgICAgIlJhdGUtZGlzdG9ydGlvbiBleGVjdXRpb24gaXMgbm90IGV2aWRlbmNlIG9mIGRvd25zdHJlYW0gdGFzayBxdWFsaXR5LCBhIHRyYWluZWQgIgogICAgICAgICAgICAiQlUxTEQgZmFtaWx5LCBvciB0YXJnZXQtc2NhbGUgcXVhbnRpemVkIGluZmVyZW5jZS4iCiAgICAgICAgKSwKICAgICkKICAgIF9hdG9taWNfanNvbihyZXBvcnRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo=', 'olympus/training/olympus_system.py': '...', 'olympus/training/provenance.py': 'IiIiU21hbGwsIFRvcmNoLWZyZWUgaWRlbnRpdHkgYW5kIGR1cmFibGUtd3JpdGUgaGVscGVycyBmb3IgbG9jYWwgY2FtcGFpZ25zLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKCmRlZiBkaWdlc3RfanNvbih2YWx1ZTogb2JqZWN0KSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoCiAgICAgICAganNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSwgYWxsb3dfbmFuPUZhbHNlKS5lbmNvZGUoKQogICAgKS5oZXhkaWdlc3QoKQoKCmRlZiBmaWxlX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBoYW5kbGUucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGF0b21pY19qc29uKHBhdGg6IFBhdGgsIHZhbHVlOiBvYmplY3QpIC0+IE5vbmU6CiAgICAiIiJSZXBsYWNlIG1ldGFkYXRhIG9ubHkgYWZ0ZXIgaXRzIGNvbXBsZXRlIGNvbnRlbnRzIGhhdmUgcmVhY2hlZCB0aGUgZmlsZXN5c3RlbS4iIiIKICAgIHBheWxvYWQgPSAoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlLCBhbGxvd19uYW49RmFsc2UpICsgIlxuIikuZW5jb2RlKCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRlbXBvcmFyeTogUGF0aCB8IE5vbmUgPSBOb25lCiAgICB0cnk6CiAgICAgICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAgICAgICAgICAgIGhhbmRsZS53cml0ZShwYXlsb2FkKQogICAgICAgICAgICBoYW5kbGUuZmx1c2goKQogICAgICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAgICAgICBzeW5jX2RpcmVjdG9yeShwYXRoLnBhcmVudCkKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgdGVtcG9yYXJ5IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKCgpkZWYgc3luY19kaXJlY3RvcnkocGF0aDogUGF0aCkgLT4gTm9uZToKICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fUkRPTkxZKQogICAgdHJ5OgogICAgICAgIG9zLmZzeW5jKGRlc2NyaXB0b3IpCiAgICBmaW5hbGx5OgogICAgICAgIG9zLmNsb3NlKGRlc2NyaXB0b3IpCgoKZGVmIHNvdXJjZV9pZGVudGl0eSgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiSGFzaCBhY3R1YWwgc291cmNlIGJ5dGVzLCBpbmNsdWRpbmcgdW5jb21taXR0ZWQvdW50cmFja2VkIGltcGxlbWVudGF0aW9uIGZpbGVzLgoKICAgIEdpdCBtZXRhZGF0YSBpcyBpbmZvcm1hdGlvbmFsOyB0aGUgc291cmNlIGRpZ2VzdCBpcyBzdGFibGUgYWNyb3NzIG91dHB1dCB3cml0ZXMsCiAgICByZXBvc2l0b3J5IHJlbG9jYXRpb24gYW5kIGNvbW1pdHMgdGhhdCBkbyBub3QgY2hhbmdlIGltcGxlbWVudGF0aW9uIGJ5dGVzLgogICAgTm8gZW52aXJvbm1lbnQgdmFyaWFibGVzLCByZW1vdGUgVVJMcyBvciBhcmJpdHJhcnkgd29ya3NwYWNlIGZpbGVzIGFyZSByZWNvcmRlZC4KICAgICIiIgogICAgcm9vdCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCiAgICBmaWxlcyA9IHNvcnRlZChbKnJvb3QuZ2xvYigib2x5bXB1cy8qKi8qLnB5IiksICpyb290Lmdsb2IoInNjcmlwdHMvKi5weSIpXSkKICAgIGZpbGVzLmV4dGVuZChwYXRoIGZvciBwYXRoIGluIFtyb290IC8gInB5cHJvamVjdC50b21sIl0gaWYgcGF0aC5pc19maWxlKCkpCiAgICBoYXNoZXMgPSB7c3RyKHBhdGgucmVsYXRpdmVfdG8ocm9vdCkpOiBmaWxlX3NoYTI1NihwYXRoKSBmb3IgcGF0aCBpbiBmaWxlc30KCiAgICBkZWYgZ2l0KCphcmdzOiBzdHIpIC0+IHN0ciB8IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoCiAgICAgICAgICAgICAgICBbImdpdCIsICphcmdzXSwgY3dkPXJvb3QsIHN0ZGVycj1zdWJwcm9jZXNzLkRFVk5VTEwsIHRleHQ9VHJ1ZSwgdGltZW91dD01CiAgICAgICAgICAgICkuc3RyaXAoKQogICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5TdWJwcm9jZXNzRXJyb3IpOiAgIyBmbXQ6IHNraXAKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBzdGF0dXMgPSBnaXQoInN0YXR1cyIsICItLXBvcmNlbGFpbiIsICItLXVudHJhY2tlZC1maWxlcz1ub3JtYWwiKQogICAgcmV0dXJuIHsKICAgICAgICAic2hhMjU2IjogZGlnZXN0X2pzb24oaGFzaGVzKSwKICAgICAgICAiZmlsZXMiOiBoYXNoZXMsCiAgICAgICAgImhlYWQiOiBnaXQoInJldi1wYXJzZSIsICJIRUFEIiksCiAgICAgICAgImRpcnR5IjogTm9uZSBpZiBzdGF0dXMgaXMgTm9uZSBlbHNlIGJvb2woc3RhdHVzKSwKICAgIH0K', 'olympus/training/synthetic.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCgoKZGVmIGFtYmlndW91c19pbnRlcnByZXRhdGlvbl9kYXRhc2V0KCkgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgogICAgcmV0dXJuIFsKICAgICAgICAoIlRoZSBjbG91ZCB3YXMgYSBkcmFnb24gYWJvdmUgdGhlIGNpdHkuIiwgImFuYWxvZ3kiKSwKICAgICAgICAoIlRoZSBsb2cgbGluZSBtZWFucyB0aGUgc2VydmljZSByZXN0YXJ0ZWQuIiwgImxpdGVyYWwiKSwKICAgICAgICAoIlRoZSBtYXJrZXQgYnJlYXRoZWQgYmVmb3JlIHRoZSBjcmFzaC4iLCAic3ltYm9saWMiKSwKICAgIF0KCgpkZWYgdHJhbnNmb3JtX3NpZ25hbHMoKSAtPiB0b3JjaC5UZW5zb3I6CiAgICB0aW1lID0gdG9yY2gubGluc3BhY2UoMC4wLCA2LjI4LCBzdGVwcz02NCkKICAgIHN0YWNrZWQgPSB0b3JjaC5zdGFjaygKICAgICAgICBbCiAgICAgICAgICAgIHRvcmNoLnNpbih0aW1lKSwKICAgICAgICAgICAgdG9yY2guY29zKHRpbWUpLAogICAgICAgICAgICB0b3JjaC5zaW4odGltZSAqIDIuMCkgKiAwLjUsCiAgICAgICAgXSwKICAgICAgICBkaW09MSwKICAgICkKICAgIHJldHVybiBzdGFja2VkCg==', 'olympus/training/trainer.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgpmcm9tIG9seW1wdXMuY29yZS5wcmVkaWN0aW9uIGltcG9ydCBKRVBBVHJhaW5pbmdSZXN1bHQsIHRyYWluX2plcGEKZnJvbSBvbHltcHVzLmNvcmUudHJhbnNmb3JtIGltcG9ydCBUcmFuc2Zvcm1NZXRyaWNzLCB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybQpmcm9tIG9seW1wdXMudHJhaW5pbmcuc3ludGhldGljIGltcG9ydCB0cmFuc2Zvcm1fc2lnbmFscwoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgVHJhaW5pbmdSZXBvcnQ6CiAgICBzZWVkOiBpbnQKICAgIGplcGE6IEpFUEFUcmFpbmluZ1Jlc3VsdAogICAgdHJhbnNmb3JtOiBUcmFuc2Zvcm1NZXRyaWNzCgoKY2xhc3MgT2x5bXB1c1RyYWluZXI6CiAgICBkZWYgcnVuKHNlbGYsIHNlZWQ6IGludCA9IDcpIC0+IFRyYWluaW5nUmVwb3J0OgogICAgICAgIGplcGEgPSB0cmFpbl9qZXBhKHNlZWQ9c2VlZCkKICAgICAgICB0cmFuc2Zvcm0gPSB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybSh0cmFuc2Zvcm1fc2lnbmFscygpLCBzZWVkPXNlZWQpCiAgICAgICAgcmV0dXJuIFRyYWluaW5nUmVwb3J0KHNlZWQ9c2VlZCwgamVwYT1qZXBhLCB0cmFuc2Zvcm09dHJhbnNmb3JtKQo=', 'olympus/upscaler/__init__.py': 'IiIiSkVQQS1iYXNlZCBzdXBlci1yZXNvbHV0aW9uIHJlc2VhcmNoIGNvbXBvbmVudHMuIiIiCgpmcm9tIG9seW1wdXMudXBzY2FsZXIuamVwYSBpbXBvcnQgKAogICAgSkVQQVVwc2NhbGVyLAogICAgSkVQQVVwc2NhbGVyQ29uZmlnLAogICAgSkVQQVVwc2NhbGVyTG9zcywKICAgIEpFUEFVcHNjYWxlck91dHB1dCwKICAgIGplcGFfdXBzY2FsZXJfbG9zcywKICAgIHVwc2NhbGVfc2VsZl9lbnNlbWJsZSwKICAgIHVwc2NhbGVfdGlsZWQsCikKZnJvbSBvbHltcHVzLnVwc2NhbGVyLnRyYWluaW5nIGltcG9ydCBsb2FkX2plcGFfdXBzY2FsZXJfY2hlY2twb2ludAoKX19hbGxfXyA9IFsKICAgICJKRVBBVXBzY2FsZXIiLAogICAgIkpFUEFVcHNjYWxlckNvbmZpZyIsCiAgICAiSkVQQVVwc2NhbGVyTG9zcyIsCiAgICAiSkVQQVVwc2NhbGVyT3V0cHV0IiwKICAgICJqZXBhX3Vwc2NhbGVyX2xvc3MiLAogICAgImxvYWRfamVwYV91cHNjYWxlcl9jaGVja3BvaW50IiwKICAgICJ1cHNjYWxlX3NlbGZfZW5zZW1ibGUiLAogICAgInVwc2NhbGVfdGlsZWQiLApdCg==', 'olympus/upscaler/jepa.py': '...', 'olympus/upscaler/training.py': '...', 'research/olympus_routing_v6/protocol.json': 'ewogICJpZCI6ICJvbHltcHVzLXJvdXRpbmctdjYuMC4yIiwKICAic2NvcGUiOiAiQm91bmRlZCBzeW50aGV0aWMgY29uc2VydmF0aXZlIHN0YWNraW5nIHN0dWR5OyBubyB0YXJnZXQtc2NhbGUgb3IgZnVsbC1mYW1pbHkgY2FwYWJpbGl0eSBjbGFpbSIsCiAgImh5cG90aGVzaXMiOiAiRXhhbXBsZS1kZXBlbmRlbnQgYW5jaG9yZWQgY29ycmVjdGlvbiB3aXRoIHdvcnN0LWdyb3VwIHRyYWluaW5nIGltcHJvdmVzIGZyZXNoIGNvbXBvc2l0ZS1zaGlmdCBhY2N1cmFjeSBieSBhdCBsZWFzdCAyIHBwIG92ZXIgdGhlIGRldmVsb3BtZW50LXNlbGVjdGVkIHN0cm9uZ2VzdCBjb250cm9sIHdpdGhvdXQgZGVjbGFyZWQgcmVnaW1lIHJlZ3Jlc3Npb25zLiIsCiAgImRhdGFzZXRfc2VlZHMiOiBbCiAgICAyMDI2NzEwMCwKICAgIDIwMjY3MjAwLAogICAgMjAyNjczMDAsCiAgICAyMDI2NzQwMCwKICAgIDIwMjY3NTAwCiAgXSwKICAidHJhaW5pbmdfc2VlZHMiOiBbCiAgICAyMDI2NzYwMSwKICAgIDIwMjY3NjAyCiAgXSwKICAiYmFua19leGFtcGxlcyI6IDEwMjQsCiAgIm1ldGFfZXhhbXBsZXNfcGVyX3JlZ2ltZSI6IDI1NiwKICAidGVzdF9leGFtcGxlc19wZXJfcmVnaW1lIjogMTAyNCwKICAiYmFua19zdGVwcyI6IDI1MCwKICAicmVmZXJlbmNlX3N0ZXBzIjogMzAwLAogICJjb3JyZWN0aW9uX3N0ZXBzIjogMzAwLAogICJiYXRjaF9wZXJfZ3JvdXAiOiAzMiwKICAibHIiOiAwLjAwMSwKICAid2VpZ2h0X2RlY2F5IjogMC4wMSwKICAiZ3JhZGllbnRfY2xpcCI6IDEuMCwKICAiYW5jaG9yX3BlbmFsdHkiOiAwLjAxLAogICJhbHBoYV9pbml0aWFsIjogMC4xLAogICJncm91cF9vYmplY3RpdmUiOiAibWF4aW11bSBvZiB0aHJlZSBlcXVhbGx5IHNpemVkIGdyb3VwIG1lYW4gTkxMcyBwZXIgdXBkYXRlOyBkZXRlcm1pbmlzdGljIHRvcmNoLm1heCBzdWJncmFkaWVudDsgbm90IGFkYXB0aXZlLXdlaWdodCBncm91cCBEUk8iLAogICJkZXZlbG9wbWVudF9yZWdpbWVzIjogWwogICAgImNsZWFuIiwKICAgICJjb3JydXB0ZWQiLAogICAgImtub3duX3NoaWZ0IgogIF0sCiAgInByb3RlY3RlZF9yZWdpbWVzIjogWwogICAgImNsZWFuIiwKICAgICJjb3JydXB0ZWQiLAogICAgImtub3duX3NoaWZ0IiwKICAgICJjb21wb3NpdGVfc2hpZnQiCiAgXSwKICAic3BsaXRfb2Zmc2V0cyI6IHsKICAgICJiYW5rIjogMSwKICAgICJyZWZlcmVuY2VfZml0IjogMTAsCiAgICAicmVmZXJlbmNlX3NlbGVjdCI6IDIwLAogICAgImNvcnJlY3Rpb25fZml0IjogMzAsCiAgICAic2VsZWN0aW9uIjogNDAsCiAgICAicHJvdGVjdGVkIjogNjAKICB9LAogICJ2YXJpYW50cyI6IFsKICAgICJkaXJlY3RfZXJtIiwKICAgICJkaXJlY3Rfcm9idXN0IiwKICAgICJhbmNob3JlZF9lcm0iLAogICAgImFuY2hvcmVkX3JvYnVzdCIsCiAgICAiY29uc3RhbnRfZXJtIiwKICAgICJjb25zdGFudF9yb2J1c3QiLAogICAgImFuY2hvcmVkX3JvYnVzdF9ub19wZW5hbHR5IgogIF0sCiAgInByaW1hcnlfY2FuZGlkYXRlIjogImFuY2hvcmVkX3JvYnVzdCIsCiAgImNvbnRyb2xfY2FuZGlkYXRlcyI6IFsKICAgICJ1bmlmb3JtIiwKICAgICJmaXhlZCIsCiAgICAic2VtYW50aWMiLAogICAgImRpcmVjdF9lcm0iLAogICAgImRpcmVjdF9yb2J1c3QiLAogICAgImNvbnN0YW50X3JvYnVzdCIsCiAgICAiY29uc3RhbnRfZXJtIiwKICAgICJleHBlcnRfMCIsCiAgICAiZXhwZXJ0XzEiLAogICAgImV4cGVydF8yIiwKICAgICJleHBlcnRfMyIsCiAgICAiZXhwZXJ0XzQiCiAgXSwKICAic2VsZWN0aW9uIjogIm1heGltaXplIG1pbmltdW0gcmVnaW1lIGFjY3VyYWN5LCB0aGVuIG1lYW4gYWNjdXJhY3ksIHRoZW4gZGVjbGFyZWQgbGlzdCBvcmRlciIsCiAgInByaW1hcnlfcmVnaW1lIjogImNvbXBvc2l0ZV9zaGlmdCIsCiAgIm1pbmltdW1fZWZmZWN0IjogMC4wMiwKICAibm9ucmVncmVzc2lvbl9tZWFuIjogLTAuMDEsCiAgIm5vbnJlZ3Jlc3Npb25faW50ZXJ2YWxfbG93ZXIiOiAtMC4wMiwKICAiYm9vdHN0cmFwX3NhbXBsZXMiOiAxMDAwMCwKICAiYm9vdHN0cmFwX3NlZWQiOiAyMDI2Nzk5OSwKICAiaW50ZXJ2YWwiOiAicGFpcmVkIGRhdGFzZXQtY2x1c3RlciBwZXJjZW50aWxlIDk1JTsgZml2ZSBpbmRlcGVuZGVudCBkYXRhc2V0cywgdHdvIGluaXRpYWxpemF0aW9ucyBhdmVyYWdlZDsgZGVzY3JpcHRpdmUgcGlsb3Qgb25seSIsCiAgIm1heF9kZXZlbG9wbWVudF9yZXZpc2lvbnMiOiAwLAogICJtYXhfY2FtcGFpZ25fc2Vjb25kcyI6IDM2MDAsCiAgIm1heF9xdWFsaWZpY2F0aW9uX3NlY29uZHMiOiAxMjAsCiAgIm1pbmltdW1fYXZhaWxhYmxlX2J5dGVzIjogMjE0NzQ4MzY0OCwKICAibWF4aW11bV9zd2FwX2ZyYWN0aW9uIjogMC45LAogICJsb2NhbF9mcmVlX2J5dGVzX2Zsb29yIjogNTM2ODcwOTEyLAogICJwYWlkX2pvYnMiOiAwLAogICJleHRlcm5hbF9kb3dubG9hZHMiOiAwLAogICJkZXZpY2UiOiAiQ1BVIiwKICAidGhyZWFkcyI6IDEsCiAgInN0b3BwaW5nIjogIk9uZSBmcm96ZW4gY2FtcGFpZ247IHJlc291cmNlL3RpbWUvbm9uZmluaXRlL3Byb3ZlbmFuY2UgZmFpbHVyZXMgc3RvcCBhbmQgcHJlc2VydmUgcGFydGlhbCBhcnRpZmFjdHMuIE5vIHJlcGVhdGVkIGNvbmZpcm1hdG9yeSBzZWFyY2ggZm9yIGEgcG9zaXRpdmUgcmVzdWx0LiIsCiAgImNsYWltX2ZhbHNpZmllciI6ICJQcmltYXJ5IHByYWN0aWNhbCBlZmZlY3Qgb3IgaW50ZXJ2YWwgY3JpdGVyaW9uIGZhaWxzLCBvciBhbnkgY2xlYW4vY29ycnVwdGVkL2tub3duLXNoaWZ0IG5vbnJlZ3Jlc3Npb24gZ2F0ZSBmYWlscy4gSW52YWxpZCBwcm92ZW5hbmNlL251bWVyaWNzIHlpZWxkcyBJTlZBTElELCBhYnNlbnQgZXhlY3V0aW9uIHlpZWxkcyBOT1RfRVZBTFVBVEVELiIsCiAgIm5vbnJlZ3Jlc3Npb25fY29tcGFyYXRvcnMiOiBbCiAgICAicmVmZXJlbmNlIiwKICAgICJzZWxlY3RlZF9jb250cm9sIgogIF0sCiAgImFtZW5kbWVudCI6ICJQcmUtZXhlY3V0aW9uIHNvdXJjZSByZXZpZXcgYWRkZWQgZXhwZXJ0LW9ubHkgY29udHJvbHMsIGV4cGxpY2l0IGJhbmsgdHJhaW5pbmcgaHlwZXJwYXJhbWV0ZXJzIGFuZCBjb3JyZWN0ZWQgaGVsZC1vdXQtc2hpZnQgdGVybWlub2xvZ3kuIE5vIHNjaWVudGlmaWMgb3V0Y29tZXMgb2JzZXJ2ZWQ7IGVhcmxpZXIgcHJvdG9jb2wgdmVyc2lvbnMgcmV0YWluZWQuIiwKICAiYmFua19iYXRjaF9zaXplIjogNjQsCiAgImJhbmtfdGVtcG9yYWxfd2VpZ2h0IjogMC4yNSwKICAicmVmZXJlbmNlX2NhbmRpZGF0ZXMiOiBbCiAgICAidW5pZm9ybSIsCiAgICAiZml4ZWQiLAogICAgInNlbWFudGljIiwKICAgICJleHBlcnRfMCIsCiAgICAiZXhwZXJ0XzEiLAogICAgImV4cGVydF8yIiwKICAgICJleHBlcnRfMyIsCiAgICAiZXhwZXJ0XzQiCiAgXSwKICAic2hpZnRfYm91bmRhcnkiOiAiY29tcG9zaXRlX3NoaWZ0IGlzIGEgcmVzZXJ2ZWQgY29uZmlndXJhdGlvbiBvZiBwcmV2aW91c2x5IG9ic2VydmVkIHJlbG9jYXRpb24sIG9jY2x1c2lvbiBhbmQgYWZmaW5lLXN0YXRlIHBlcnR1cmJhdGlvbiB0eXBlczsgb25lIGRhbWFnZWQgbW9kYWxpdHkgcGVyIGV4YW1wbGUsIG5vdCBhIG5vdmVsIG1lY2hhbmlzbSBvciBuYXR1cmFsLWRhdGEgT09ELiIKfQo=', 'scripts/__init__.py': 'IiIiUmVwb3NpdG9yeSBtYWludGVuYW5jZSBhbmQgcmVzZWFyY2ggZW50cnkgcG9pbnRzLiIiIgo=', 'scripts/qualify_flagship_v2.py': 'IiIiUnVuIGJvdW5kZWQgYXJ0aWZpY2lhbCBmbGFnc2hpcCBxdWFsaWZpY2F0aW9uIGFmdGVyIGhvc3QgYWRtaXNzaW9uLgoKVXNhZ2U6IHB5dGhvbiAtbSBzY3JpcHRzLnF1YWxpZnlfZmxhZ3NoaXBfdjIgLS1vdXRwdXQgTkVXX0RJUkVDVE9SWSAtLWRldmljZSBjcHUKVGhlIHNhbWUgY29tbWFuZCB3b3JrcyBvZmZsaW5lIG9uIEthZ2dsZTsgdXNlIGN1ZGEgb25seSBmb3IgYW4gYWxsb2NhdGVkIEdQVS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmUKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2VzIGltcG9ydCBSZXNvdXJjZUdvdmVybm9yCmZyb20gb2x5bXB1cy50cmFpbmluZy5wcm92ZW5hbmNlIGltcG9ydCBhdG9taWNfanNvbiwgZmlsZV9zaGEyNTYsIHNvdXJjZV9pZGVudGl0eQoKCmRlZiBfbWFjb3Nfdm1fY291bnRlcnMoKSAtPiB0dXBsZVtpbnQsIGludF06CiAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICBbInZtX3N0YXQiXSwgY2hlY2s9VHJ1ZSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTUKICAgICkuc3Rkb3V0CiAgICBwYWdlX3NpemUgPSByZS5zZWFyY2gociJwYWdlIHNpemUgb2YgKFxkKykgYnl0ZXMiLCByZXN1bHQpCiAgICBmcmVlID0gcmUuc2VhcmNoKHIiXlBhZ2VzIGZyZWU6XHMrKFxkKylcLiIsIHJlc3VsdCwgcmUuTVVMVElMSU5FKQogICAgcGFnZW91dHMgPSByZS5zZWFyY2gociJeUGFnZW91dHM6XHMrKFxkKylcLiIsIHJlc3VsdCwgcmUuTVVMVElMSU5FKQogICAgaWYgcGFnZV9zaXplIGlzIE5vbmUgb3IgZnJlZSBpcyBOb25lIG9yIHBhZ2VvdXRzIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJjYW5ub3QgaW5zcGVjdCBtYWNPUyBmcmVlIHBhZ2VzIGFuZCBwYWdlb3V0cyIpCiAgICByZXR1cm4gaW50KHBhZ2Vfc2l6ZVsxXSkgKiBpbnQoZnJlZVsxXSksIGludChwYWdlb3V0c1sxXSkKCgpkZWYgbG9jYWxfdGVuc29yX2FkbWlzc2lvbigpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgIiIiUmV0YWluIHRoZSBsb2NhbCA1MTIgTWlCL3N0YWJsZS1wYWdlb3V0IGZsb29yIGJlZm9yZSBsb2FkaW5nIFRvcmNoLgoKICAgIFRoaXMgaXMgYWRkaXRpb25hbCB0byAobm90IGEgcmVwbGFjZW1lbnQgZm9yKSBSZXNvdXJjZUdvdmVybm9yJ3MgMiBHaUIKICAgIGF2YWlsYWJsZS1tZW1vcnkgYW5kIDkwIHBlcmNlbnQgc3dhcCBjZWlsaW5ncy4gTGludXggdXNlcyB0aGF0IGdvdmVybm9yJ3MKICAgIE1lbUF2YWlsYWJsZSBhZG1pc3Npb247IHJhdyBtYWNPUyBmcmVlIHBhZ2VzIGFyZSBub3QgaW50ZXJjaGFuZ2VhYmxlIHdpdGggaXQuCiAgICAiIiIKICAgIGlmIHBsYXRmb3JtLnN5c3RlbSgpICE9ICJEYXJ3aW4iOgogICAgICAgIHJldHVybiB7InBsYXRmb3JtIjogcGxhdGZvcm0uc3lzdGVtKCksICJleHRyYV9tYWNvc19ndWFyZCI6ICJOT1RfQVBQTElDQUJMRSJ9CiAgICBmaXJzdCA9IF9tYWNvc192bV9jb3VudGVycygpCiAgICB0aW1lLnNsZWVwKDEpCiAgICBzZWNvbmQgPSBfbWFjb3Nfdm1fY291bnRlcnMoKQogICAgaWYgbWluKGZpcnN0WzBdLCBzZWNvbmRbMF0pIDwgNTEyICogMTAyNCoqMiBvciBzZWNvbmRbMV0gPiBmaXJzdFsxXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYibG9jYWwgdGVuc29yIGFkbWlzc2lvbiByZWZ1c2VkOiBmcmVlIGJ5dGVzIHtmaXJzdFswXX0sIHtzZWNvbmRbMF19OyAiCiAgICAgICAgICAgIGYicGFnZW91dHMge2ZpcnN0WzFdfSwge3NlY29uZFsxXX07IHJlcXVpcmUgPj01MTIgTWlCIGFuZCBubyByaXNpbmcgcGFnZW91dHMiCiAgICAgICAgKQogICAgcmV0dXJuIHsKICAgICAgICAicGxhdGZvcm0iOiAiRGFyd2luIiwKICAgICAgICAiZnJlZV9ieXRlcyI6IFtmaXJzdFswXSwgc2Vjb25kWzBdXSwKICAgICAgICAicGFnZW91dHMiOiBsaXN0KChmaXJzdFsxXSwgc2Vjb25kWzFdKSksCiAgICB9CgoKZGVmIHF1YWxpZnkob3V0cHV0OiBQYXRoLCAqLCBkZXZpY2U6IHN0ciA9ICJjcHUiLCB3YWxsX3NlY29uZHM6IGludCA9IDEyMCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBkZXZpY2Ugbm90IGluIHsiY3B1IiwgImN1ZGEifSBvciBub3QgMTAgPD0gd2FsbF9zZWNvbmRzIDw9IDEyMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZXZpY2UgbXVzdCBiZSBjcHUvY3VkYSBhbmQgd2FsbCBidWRnZXQgbXVzdCBiZSAxMC4uMTIwIHNlY29uZHMiKQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgcm9vdCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCiAgICBwcm90b2NvbCA9IHJvb3QgLyAicmVzZWFyY2gvZmxhZ3NoaXBfZW5naW5lZXJpbmdfdjIvUFJPVE9DT0wubWQiCiAgICBzb3VyY2UgPSBzb3VyY2VfaWRlbnRpdHkoKQogICAgcmVjZWlwdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAgICAgICAic3R1ZHlfaWQiOiAiZmxhZ3NoaXAtZW5naW5lZXJpbmctdjIiLAogICAgICAgICJzdGF0dXMiOiAiU1RBUlRFRCIsCiAgICAgICAgImRldmljZV9yZXF1ZXN0ZWQiOiBkZXZpY2UsCiAgICAgICAgInNvdXJjZSI6IHNvdXJjZSwKICAgICAgICAicHJvdG9jb2xfc2hhMjU2IjogZmlsZV9zaGEyNTYocHJvdG9jb2wpLAogICAgICAgICJyZWdyZXNzaW9uX3NvdXJjZV9zaGEyNTYiOiBmaWxlX3NoYTI1Nihyb290IC8gInRlc3RzL3Rlc3RfZmxhZ3NoaXBfdjIucHkiKSwKICAgICAgICAibWF4X3dhbGxfc2Vjb25kcyI6IHdhbGxfc2Vjb25kcywKICAgICAgICAic2NpZW50aWZpY19ldmFsdWF0aW9uIjogRmFsc2UsCiAgICAgICAgInByb21vdGlvbl9hdXRob3JpemVkIjogRmFsc2UsCiAgICAgICAgInRhcmdldF9zY2FsZV90cmFpbmluZyI6IEZhbHNlLAogICAgfQogICAgYXRvbWljX2pzb24ob3V0cHV0IC8gInJlY2VpcHQuanNvbiIsIHJlY2VpcHQpCiAgICB0cnk6CiAgICAgICAgcmVjZWlwdFsibG9jYWxfdGVuc29yX2FkbWlzc2lvbiJdID0gbG9jYWxfdGVuc29yX2FkbWlzc2lvbigpCiAgICAgICAgd2l0aCBSZXNvdXJjZUdvdmVybm9yKHJvb3QgLyAiYXJ0aWZhY3RzL2xvY2tzL2ZsYWdzaGlwLXF1YWxpZmljYXRpb24ubG9jayIpIGFzIGdvdmVybm9yOgogICAgICAgICAgICByZWNlaXB0WyJyZXNvdXJjZV9wcmVmbGlnaHQiXSA9ICgKICAgICAgICAgICAgICAgIGdvdmVybm9yLnByZWZsaWdodC5hc19kaWN0KCkgaWYgZ292ZXJub3IucHJlZmxpZ2h0IGVsc2UgTm9uZQogICAgICAgICAgICApCiAgICAgICAgICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgiQ1VCTEFTX1dPUktTUEFDRV9DT05GSUciLCAiOjQwOTY6OCIpCiAgICAgICAgICAgIG9zLmVudmlyb25bIk9NUF9OVU1fVEhSRUFEUyJdID0gIjEiCiAgICAgICAgICAgIG9zLmVudmlyb25bIk1LTF9OVU1fVEhSRUFEUyJdID0gIjEiCiAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgaWYgZGV2aWNlID09ICJjdWRhIiBhbmQgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNVREEgcmVxdWVzdGVkIGJ1dCB1bmF2YWlsYWJsZTsgQ1BVIGZhbGxiYWNrIGlzIGZvcmJpZGRlbiIpCiAgICAgICAgICAgIHJlY2VpcHRbInJ1bnRpbWUiXSA9IHsKICAgICAgICAgICAgICAgICJweXRob24iOiBzeXMudmVyc2lvbiwKICAgICAgICAgICAgICAgICJ0b3JjaCI6IHN0cih0b3JjaC5fX3ZlcnNpb25fXyksCiAgICAgICAgICAgICAgICAiY3VkYSI6IHRvcmNoLnZlcnNpb24uY3VkYSwKICAgICAgICAgICAgICAgICJkZXZpY2UiOiBkZXZpY2UsCiAgICAgICAgICAgICAgICAiZGV2aWNlX25hbWUiOiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSBpZiBkZXZpY2UgPT0gImN1ZGEiIGVsc2UgIkNQVSIsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgdGVzdHMgPSBbc3RyKHJvb3QgLyAidGVzdHMvdGVzdF9mbGFnc2hpcF92Mi5weSIpXQogICAgICAgICAgICBpZiBkZXZpY2UgPT0gImNwdSI6CiAgICAgICAgICAgICAgICB0ZXN0cyArPSBbIi1rIiwgIm5vdCBjdWRhIl0KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgICAgICAgICBzeXMuZXhlY3V0YWJsZSwKICAgICAgICAgICAgICAgICAgICAgICAgIi1tIiwKICAgICAgICAgICAgICAgICAgICAgICAgInB5dGVzdCIsCiAgICAgICAgICAgICAgICAgICAgICAgICItcSIsCiAgICAgICAgICAgICAgICAgICAgICAgICp0ZXN0cywKICAgICAgICAgICAgICAgICAgICAgICAgIi0tanVuaXR4bWwiLAogICAgICAgICAgICAgICAgICAgICAgICBzdHIob3V0cHV0LnJlc29sdmUoKSAvICJyZWdyZXNzaW9ucy54bWwiKSwKICAgICAgICAgICAgICAgICAgICBdLAogICAgICAgICAgICAgICAgICAgIGN3ZD1yb290LAogICAgICAgICAgICAgICAgICAgIGVudj1kaWN0KG9zLmVudmlyb24sIFBZVEVTVF9ESVNBQkxFX1BMVUdJTl9BVVRPTE9BRD0iMSIpLAogICAgICAgICAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICAgICAgICAgIHRpbWVvdXQ9bWF4KDEsIHdhbGxfc2Vjb25kcyAtICh0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCkpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCBhcyBlcnJvcjoKICAgICAgICAgICAgICAgIHN0ZG91dCA9IGVycm9yLnN0ZG91dCBvciBiIiIKICAgICAgICAgICAgICAgIHN0ZGVyciA9IGVycm9yLnN0ZGVyciBvciBiIiIKICAgICAgICAgICAgICAgIChvdXRwdXQgLyAicmVncmVzc2lvbnMubG9nIikud3JpdGVfYnl0ZXMoCiAgICAgICAgICAgICAgICAgICAgKHN0ZG91dC5lbmNvZGUoKSBpZiBpc2luc3RhbmNlKHN0ZG91dCwgc3RyKSBlbHNlIHN0ZG91dCkKICAgICAgICAgICAgICAgICAgICArIChzdGRlcnIuZW5jb2RlKCkgaWYgaXNpbnN0YW5jZShzdGRlcnIsIHN0cikgZWxzZSBzdGRlcnIpCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICAob3V0cHV0IC8gInJlZ3Jlc3Npb25zLmxvZyIpLndyaXRlX3RleHQocmVzdWx0LnN0ZG91dCArIHJlc3VsdC5zdGRlcnIpCiAgICAgICAgICAgIGlmIHJlc3VsdC5yZXR1cm5jb2RlOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiZmxhZ3NoaXAgcmVncmVzc2lvbnMgZmFpbGVkIHdpdGggZXhpdCB7cmVzdWx0LnJldHVybmNvZGV9IikKICAgICAgICAgICAgIyBBcnRpZmljaWFsIG9wdGltaXplciBxdWFsaWZpY2F0aW9uLCBubyBwcm90ZWN0ZWQgZGF0YXNldCBvciBiZW5jaG1hcmsuCiAgICAgICAgICAgIGZyb20gc2NyaXB0cy5mbGFnc2hpcF9zbW9rZSBpbXBvcnQgdHJhaW5fc21va2UKCiAgICAgICAgICAgIHJlY2VpcHRbIm9wdGltaXplcl9zbW9rZSJdID0gdHJhaW5fc21va2UoCiAgICAgICAgICAgICAgICBvdXRwdXQgLyAic21va2UucHQiLAogICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgICAgIGRlYWRsaW5lPXN0YXJ0ZWQgKyB3YWxsX3NlY29uZHMsCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQgPiB3YWxsX3NlY29uZHM6CiAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoInF1YWxpZmljYXRpb24gZXhjZWVkZWQgdGhlIGZyb3plbiB3YWxsIGJ1ZGdldCIpCiAgICAgICAgICAgIHJlY2VpcHRbInN0YXR1cyJdID0gIkVOR0lORUVSSU5HX1FVQUxJRklFRCIKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6CiAgICAgICAgcmVjZWlwdFsic3RhdHVzIl0gPSAiRkFJTEVEX09SX1JFU09VUkNFX0JMT0NLRUQiCiAgICAgICAgcmVjZWlwdFsiZXJyb3IiXSA9IGYie3R5cGUoZXJyb3IpLl9fbmFtZV9ffToge2Vycm9yfSIKICAgICAgICByYWlzZQogICAgZmluYWxseToKICAgICAgICByZWNlaXB0WyJlbGFwc2VkX3NlY29uZHMiXSA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICAgICAgcmVjZWlwdFsiYXJ0aWZhY3RzIl0gPSB7CiAgICAgICAgICAgIHN0cihwYXRoLnJlbGF0aXZlX3RvKG91dHB1dCkpOiBmaWxlX3NoYTI1NihwYXRoKQogICAgICAgICAgICBmb3IgcGF0aCBpbiBzb3J0ZWQob3V0cHV0LnJnbG9iKCIqIikpCiAgICAgICAgICAgIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLm5hbWUgIT0gInJlY2VpcHQuanNvbiIKICAgICAgICB9CiAgICAgICAgYXRvbWljX2pzb24ob3V0cHV0IC8gInJlY2VpcHQuanNvbiIsIHJlY2VpcHQpCiAgICByZXR1cm4gcmVjZWlwdAoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgY2hvaWNlcz0oImNwdSIsICJjdWRhIiksIGRlZmF1bHQ9ImNwdSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyhxdWFsaWZ5KGFyZ3Mub3V0cHV0LCBkZXZpY2U9YXJncy5kZXZpY2UpLCBpbmRlbnQ9MikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'scripts/run_conservative_routing_v6.py': '...', 'scripts/run_oracle_fusion_v4.py': '...', 'tests/__init__.py': 'IiIiT2x5bXB1cyByZWdyZXNzaW9uIHRlc3RzLiIiIgo=', 'tests/test_conservative_routing_v6.py': 'IiIiQXJ0aWZpY2lhbCBlcXVhdGlvbiwgZ3JhZGllbnQsIGRpc2pvaW50bmVzcyBhbmQgZnVsbCBsaWZlY3ljbGUgcmVncmVzc2lvbnMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29weQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBweXRlc3QKaW1wb3J0IHRvcmNoCgpmcm9tIG9seW1wdXMubW9kZWxzLmNvbnNlcnZhdGl2ZV9yb3V0aW5nX3Y2IGltcG9ydCAoCiAgICBDb3JyZWN0aW9uLAogICAgUmVmZXJlbmNlUm91dGVzLAogICAgYmxlbmQsCiAgICBjb3JyZWN0aW9uX2xvc3MsCiAgICBmZWF0dXJlcywKICAgIGludGVydmVudGlvbl9kaWFnbm9zdGljcywKKQpmcm9tIHNjcmlwdHMgaW1wb3J0IHJ1bl9jb25zZXJ2YXRpdmVfcm91dGluZ192NiBhcyBydW5uZXIKZnJvbSBzY3JpcHRzIGltcG9ydCBydW5fb3JhY2xlX2Z1c2lvbl92NCBhcyB2NAoKCmRlZiBmaXh0dXJlKCkgLT4gdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZCg5OTEpCiAgICBwID0gdG9yY2gucmFuZG4oMTIsIDUsIDQsIGdlbmVyYXRvcj1nZW5lcmF0b3IpLnNvZnRtYXgoLTEpCiAgICB4ID0gZmVhdHVyZXModG9yY2gucmFuZG4oMTIsIDUsIDE2LCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSwgcCkKICAgIHJldHVybiB4LCBwLCBwLm1lYW4oMSkKCgpkZWYgdGVzdF9leGFjdF9lbmRwb2ludHNfYW5kX2NvbnZleF9ib3VuZCgpIC0+IE5vbmU6CiAgICBfLCBwLCByZWYgPSBmaXh0dXJlKCkKICAgIGFsdCA9IHBbOiwgMF0KICAgIGFzc2VydCB0b3JjaC5lcXVhbChibGVuZChyZWYsIGFsdCwgdG9yY2guemVyb3MoMTIsIDEpKSwgcmVmKQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGJsZW5kKHJlZiwgYWx0LCB0b3JjaC5vbmVzKDEyLCAxKSksIGFsdCkKICAgIGFscGhhID0gdG9yY2guZnVsbCgoMTIsIDEpLCAwLjIpCiAgICByZXN1bHQgPSBibGVuZChyZWYsIGFsdCwgYWxwaGEpCiAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UocmVzdWx0LnN1bSgtMSksIHRvcmNoLm9uZXMoMTIpKQogICAgIyBUb3RhbCB2YXJpYXRpb24gZGlzdGFuY2UgZnJvbSByZWZlcmVuY2UgY2Fubm90IGV4Y2VlZCBhbHBoYS4KICAgIGFzc2VydCBib29sKCgocmVzdWx0IC0gcmVmKS5hYnMoKS5zdW0oLTEpIC8gMiA8PSAwLjIwMDAwMSkuYWxsKCkpCgoKQHB5dGVzdC5tYXJrLnBhcmFtZXRyaXplKCJraW5kIiwgWyJkaXJlY3QiLCAiY29uc3RhbnQiLCAiYW5jaG9yZWQiXSkKZGVmIHRlc3RfZ3JhZGllbnRfYW5kX3JlbG9hZChraW5kOiBzdHIpIC0+IE5vbmU6CiAgICB4LCBwLCByZWYgPSBmaXh0dXJlKCkKICAgIG1vZGVsID0gQ29ycmVjdGlvbihSZWZlcmVuY2VSb3V0ZXMoKS5ldmFsKCkucmVxdWlyZXNfZ3JhZF8oRmFsc2UpLnNlbWFudGljLCBraW5kKQogICAgcHJlZCwgYWxwaGEgPSBtb2RlbCh4LCBwLCByZWYpCiAgICBsb3NzID0gY29ycmVjdGlvbl9sb3NzKAogICAgICAgIHByZWQsIHRvcmNoLmFyYW5nZSgxMikgJSA0LCB0b3JjaC5hcmFuZ2UoMTIpICUgMywgYWxwaGEsIHJvYnVzdD1UcnVlLCBwZW5hbHR5PTAuMDEKICAgICkKICAgIHRvcmNoLmF1dG9ncmFkLmJhY2t3YXJkKGxvc3MpCiAgICBhc3NlcnQgYWxsKAogICAgICAgIHYuZ3JhZCBpcyBub3QgTm9uZSBhbmQgYm9vbCh0b3JjaC5pc2Zpbml0ZSh2LmdyYWQpLmFsbCgpKSBmb3IgdiBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkKICAgICkKICAgIG90aGVyID0gY29weS5kZWVwY29weShtb2RlbCkKICAgIG90aGVyLmxvYWRfc3RhdGVfZGljdChtb2RlbC5zdGF0ZV9kaWN0KCkpCiAgICBhc3NlcnQgdG9yY2guZXF1YWwob3RoZXIoeCwgcCwgcmVmKVswXSwgcHJlZCkKCgpkZWYgdGVzdF9jb25zdGFudF9pc19pbnB1dF9pbmRlcGVuZGVudF9hbmRfYWRhcHRpdmVfc3RhcnRzX2F0X2RlY2xhcmVkX2FscGhhKCkgLT4gTm9uZToKICAgIHgsIHAsIHJlZiA9IGZpeHR1cmUoKQogICAgZm9yIGtpbmQgaW4gKCJjb25zdGFudCIsICJhbmNob3JlZCIpOgogICAgICAgIG1vZGVsID0gQ29ycmVjdGlvbihSZWZlcmVuY2VSb3V0ZXMoKS5ldmFsKCkucmVxdWlyZXNfZ3JhZF8oRmFsc2UpLnNlbWFudGljLCBraW5kKQogICAgICAgIF8sIGFscGhhID0gbW9kZWwoeCwgcCwgcmVmKQogICAgICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShhbHBoYSwgdG9yY2guZnVsbF9saWtlKGFscGhhLCAwLjEpKQoKCmRlZiB0ZXN0X2dyb3VwX29iamVjdGl2ZXNfaGFuZF9jYWxjdWxhdGlvbigpIC0+IE5vbmU6CiAgICBwcm9iYWJpbGl0aWVzID0gdG9yY2gudGVuc29yKFtbMC44LCAwLjJdLCBbMC41LCAwLjVdLCBbMC4yNSwgMC43NV1dKQogICAgbGFiZWxzLCBncm91cHMgPSB0b3JjaC56ZXJvcygzLCBkdHlwZT10b3JjaC5sb25nKSwgdG9yY2guYXJhbmdlKDMpCiAgICBhbHBoYSA9IHRvcmNoLnplcm9zKDMsIDEpCiAgICBleHBlY3RlZCA9IC1wcm9iYWJpbGl0aWVzWzosIDBdLmxvZygpCiAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UoCiAgICAgICAgY29ycmVjdGlvbl9sb3NzKHByb2JhYmlsaXRpZXMsIGxhYmVscywgZ3JvdXBzLCBhbHBoYSwgcm9idXN0PUZhbHNlLCBwZW5hbHR5PTApLAogICAgICAgIGV4cGVjdGVkLm1lYW4oKSwKICAgICkKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZSgKICAgICAgICBjb3JyZWN0aW9uX2xvc3MocHJvYmFiaWxpdGllcywgbGFiZWxzLCBncm91cHMsIGFscGhhLCByb2J1c3Q9VHJ1ZSwgcGVuYWx0eT0wKSwKICAgICAgICBleHBlY3RlZC5tYXgoKSwKICAgICkKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhWYWx1ZUVycm9yLCBtYXRjaD0iZXZlcnkgZGVjbGFyZWQgZ3JvdXAiKToKICAgICAgICBjb3JyZWN0aW9uX2xvc3MoCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXMsIGxhYmVscywgdG9yY2guemVyb3NfbGlrZShncm91cHMpLCBhbHBoYSwgcm9idXN0PVRydWUsIHBlbmFsdHk9MAogICAgICAgICkKCgpAcHl0ZXN0Lm1hcmsucGFyYW1ldHJpemUoInZhbHVlIiwgWy0wLjEsIDEuMSwgZmxvYXQoIm5hbiIpXSkKZGVmIHRlc3RfaW52YWxpZF9hbHBoYV9yZWplY3RlZCh2YWx1ZTogZmxvYXQpIC0+IE5vbmU6CiAgICBfLCBwLCByZWYgPSBmaXh0dXJlKCkKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICBibGVuZChyZWYsIHBbOiwgMF0sIHRvcmNoLmZ1bGwoKDEyLCAxKSwgdmFsdWUpKQoKCmRlZiB0ZXN0X2hhcm1fcmVzY3VlX2lkZW50aXR5KCkgLT4gTm9uZToKICAgIHJlZiA9IHRvcmNoLnRlbnNvcihbWzAuOSwgMC4xXSwgWzAuMSwgMC45XSwgWzAuMSwgMC45XV0pCiAgICBwcmVkID0gdG9yY2gudGVuc29yKFtbMC4xLCAwLjldLCBbMC45LCAwLjFdLCBbMC45LCAwLjFdXSkKICAgIHJlc3VsdCA9IGludGVydmVudGlvbl9kaWFnbm9zdGljcyhwcmVkLCByZWYsIHRvcmNoLnplcm9zKDMsIGR0eXBlPXRvcmNoLmxvbmcpKQogICAgYXNzZXJ0IHJlc3VsdCA9PSB7CiAgICAgICAgImV4YW1wbGVzIjogMywKICAgICAgICAicmVzY3VlcyI6IDIsCiAgICAgICAgImhhcm1zIjogMSwKICAgICAgICAiYWNjdXJhY3lfZGVsdGEiOiAxIC8gMywKICAgICAgICAicHJlZGljdGlvbl9jaGFuZ2VzIjogMywKICAgIH0KCgpkZWYgdGVzdF9kaXNqb2ludG5lc3NfcmVqZWN0c19kdXBsaWNhdGVzX2FuZF9vdmVybGFwKCkgLT4gTm9uZToKICAgIHJ1bm5lci5lbnN1cmVfZGlzam9pbnQoeyJhIjogWzEsIDJdLCAiYiI6IFszXX0pCiAgICBmb3IgcGFydHMgaW4gKHsiYSI6IFsxLCAxXX0sIHsiYSI6IFsxLCAyXSwgImIiOiBbMiwgM119KToKICAgICAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9Im92ZXJsYXAiKToKICAgICAgICAgICAgcnVubmVyLmVuc3VyZV9kaXNqb2ludChwYXJ0cykKCgpkZWYgdGVzdF9zZWxlY3Rpb25fdGllX29yZGVyX2FuZF93b3JzdF9yZWdpbWUoKSAtPiBOb25lOgogICAgcmVwb3J0cyA9IHtyOiB7ImEiOiB7ImFjY3VyYWN5IjogMC44fSwgImIiOiB7ImFjY3VyYWN5IjogMC44fX0gZm9yIHIgaW4gcnVubmVyLlJFR0lNRVN9CiAgICBhc3NlcnQgcnVubmVyLnNlbGVjdChyZXBvcnRzLCBbImIiLCAiYSJdKSA9PSAiYiIKICAgIHJlcG9ydHNbcnVubmVyLlJFR0lNRVNbMF1dWyJiIl1bImFjY3VyYWN5Il0gPSAwLjcKICAgIGFzc2VydCBydW5uZXIuc2VsZWN0KHJlcG9ydHMsIFsiYiIsICJhIl0pID09ICJhIgoKCmRlZiB0ZXN0X3Jlc2VydmVkX3NoaWZ0X3ByZXNlcnZlc190YXJnZXRzX2FuZF9zb3VyY2VfaWRlbnRpdHkoKSAtPiBOb25lOgogICAgZnJvbSBvbHltcHVzLmRhdGEuY29uc2VydmF0aXZlX3JvdXRpbmdfdjYgaW1wb3J0IGdlbmVyYXRlCgogICAgY2xlYW4gPSBnZW5lcmF0ZSgyMDI2ODAwMSwgMTYsICJjbGVhbiIpCiAgICBzaGlmdGVkID0gZ2VuZXJhdGUoMjAyNjgwMDEsIDE2LCAiY29tcG9zaXRlX3NoaWZ0IikKICAgIGZvciBrZXkgaW4gKCJzb3VyY2VfaWRzIiwgImRlY2lzaW9uX3RhcmdldHMiLCAibmV4dF9zdGF0ZV90YXJnZXRzIik6CiAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGNsZWFuW2tleV0sIHNoaWZ0ZWRba2V5XSkKCgpkZWYgdGVzdF9hcnRpZmljaWFsX2xpZmVjeWNsZV9hbmRfdGFtcGVyX3JlZnVzYWwoCiAgICB0bXBfcGF0aDogUGF0aCwgbW9ua2V5cGF0Y2g6IHB5dGVzdC5Nb25rZXlQYXRjaAopIC0+IE5vbmU6CiAgICAiIiJPbmUgdGlueSBmaXh0dXJlOyBub3QgYW4gYWRtaXR0ZWQgc2NpZW50aWZpYyBjYW1wYWlnbiBvciByZXN1bHQuIiIiCiAgICBwcm90b2NvbCA9IGNvcHkuZGVlcGNvcHkocnVubmVyLlBST1RPQ09MKQogICAgcHJvdG9jb2wudXBkYXRlKAogICAgICAgIGRhdGFzZXRfc2VlZHM9WzIwMjY4OTAwXSwKICAgICAgICB0cmFpbmluZ19zZWVkcz1bMjAyNjg5NTBdLAogICAgICAgIGJhbmtfZXhhbXBsZXM9MTYsCiAgICAgICAgYmFua19zdGVwcz0yLAogICAgICAgIGJhbmtfYmF0Y2hfc2l6ZT00LAogICAgICAgIG1ldGFfZXhhbXBsZXNfcGVyX3JlZ2ltZT04LAogICAgICAgIHRlc3RfZXhhbXBsZXNfcGVyX3JlZ2ltZT04LAogICAgICAgIHJlZmVyZW5jZV9zdGVwcz0yLAogICAgICAgIGNvcnJlY3Rpb25fc3RlcHM9MiwKICAgICAgICBiYXRjaF9wZXJfZ3JvdXA9NCwKICAgICAgICBib290c3RyYXBfc2FtcGxlcz0yMCwKICAgICkKICAgIG1vbmtleXBhdGNoLnNldGF0dHIocnVubmVyLCAiUFJPVE9DT0wiLCBwcm90b2NvbCkKICAgIGJhbmtfcHJvdG9jb2wgPSBkaWN0KHY0LlBST1RPQ09MLCBleHBlcnRfc3RlcHM9MiwgYmF0Y2hfc2l6ZT00KQogICAgbW9ua2V5cGF0Y2guc2V0YXR0cih2NCwgIlBST1RPQ09MIiwgYmFua19wcm90b2NvbCkKICAgICMgVGhlIGhvc3Qgd2FzIGFkbWl0dGVkIGJ5IHRoZSBvdXRlciBxdWFsaWZpY2F0aW9uIGNvbW1hbmQ7IG5vIHJlc291cmNlCiAgICAjIHRocmVzaG9sZHMgYXJlIGRpc2FibGVkLCBhbmQgdGhlIHNhbWUgZ3VhcmQgaXMgY2FsbGVkIGluc2lkZSB0aGUgZml4dHVyZS4KICAgIHY0LndyaXRlKAogICAgICAgIHRtcF9wYXRoIC8gIkNPTlRSQUNULmpzb24iLAogICAgICAgIHsic291cmNlcyI6IHJ1bm5lci5zb3VyY2VzKCksICJwcm90b2NvbCI6IHByb3RvY29sLCAic2NvcGUiOiAiYXJ0aWZpY2lhbF9maXh0dXJlIn0sCiAgICApCiAgICBpbXBvcnQgdGltZQoKICAgIGRlYWRsaW5lID0gdGltZS5tb25vdG9uaWMoKSArIDkwCiAgICBydW5uZXIuZGV2ZWxvcCh0bXBfcGF0aCwgZGVhZGxpbmUpCiAgICBydW5uZXIuZnJlZXplKHRtcF9wYXRoKQogICAgcnVubmVyLmNvbmZpcm0odG1wX3BhdGgsIGRlYWRsaW5lKQogICAgdjQud3JpdGUodG1wX3BhdGggLyAiU1VNTUFSWS5qc29uIiwgcnVubmVyLnN1bW1hcml6ZSh0bXBfcGF0aCkpCiAgICBhc3NlcnQgcnVubmVyLnZlcmlmeSh0bXBfcGF0aClbInJhd19yZXBvcnRzX3JlY29tcHV0ZWQiXSA9PSA0CiAgICByYXdfcGF0aCA9IG5leHQoKHRtcF9wYXRoIC8gInByb3RlY3RlZCIpLmdsb2IoIiovY2xlYW4ucHQiKSkKICAgIHJlcG9ydF9wYXRoID0gcmF3X3BhdGgud2l0aF9zdWZmaXgoIi5qc29uIikKICAgIG9yaWdpbmFsX3Jhdywgb3JpZ2luYWxfcmVwb3J0ID0gcmF3X3BhdGgucmVhZF9ieXRlcygpLCByZXBvcnRfcGF0aC5yZWFkX2J5dGVzKCkKICAgIHJhdyA9IHRvcmNoLmxvYWQocmF3X3BhdGgsIHdlaWdodHNfb25seT1UcnVlKQogICAgcmF3WyJzb3VyY2VfaWRzIl0gPSByYXdbInNvdXJjZV9pZHMiXSArIDEwMDAwMDAwMDAwMAogICAgdG9yY2guc2F2ZShyYXcsIHJhd19wYXRoKQogICAgcmVwb3J0ID0gdjQucmVhZChyZXBvcnRfcGF0aCkKICAgIHJlcG9ydFsicHJlZGljdGlvbl9zaGEyNTYiXSA9IHY0LnNoYShyYXdfcGF0aCkKICAgIGltcG9ydCBqc29uCgogICAgcmVwb3J0X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCkpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoUnVudGltZUVycm9yLCBtYXRjaD0icmVnZW5lcmF0ZWQgc291cmNlIElEcyIpOgogICAgICAgIHJ1bm5lci52ZXJpZnkodG1wX3BhdGgpCiAgICByYXdfcGF0aC53cml0ZV9ieXRlcyhvcmlnaW5hbF9yYXcpCiAgICByZXBvcnRfcGF0aC53cml0ZV9ieXRlcyhvcmlnaW5hbF9yZXBvcnQpCiAgICAjIEEgc2VsZi1jb25zaXN0ZW50IGNoYW5nZWQgcHJlZGljdGlvbi9oYXNoL21ldHJpYyBpcyBzdGlsbCBub3QgY2hlY2twb2ludCBldmlkZW5jZS4KICAgIHJhdyA9IHRvcmNoLmxvYWQocmF3X3BhdGgsIHdlaWdodHNfb25seT1UcnVlKQogICAgcmF3WyJwcm9iYWJpbGl0aWVzIl1bImFuY2hvcmVkX3JvYnVzdCJdID0gcmF3WyJwcm9iYWJpbGl0aWVzIl1bImFuY2hvcmVkX3JvYnVzdCJdLnJvbGwoMSwgLTEpCiAgICB0b3JjaC5zYXZlKHJhdywgcmF3X3BhdGgpCiAgICByZXBvcnQgPSB2NC5yZWFkKHJlcG9ydF9wYXRoKQogICAgcmVwb3J0WyJwcmVkaWN0aW9uX3NoYTI1NiJdID0gdjQuc2hhKHJhd19wYXRoKQogICAgcmVwb3J0WyJtZXRyaWNzIl0gPSBydW5uZXIuZGVzY3JpYmUocmF3WyJwcm9iYWJpbGl0aWVzIl0sIHJhd1siYWxwaGEiXSwgcmF3WyJsYWJlbHMiXSkKICAgIHJlcG9ydF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQpKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9ImNoZWNrcG9pbnQgcHJlZGljdGlvbiByZXBsYXkiKToKICAgICAgICBydW5uZXIudmVyaWZ5KHRtcF9wYXRoKQogICAgcmF3X3BhdGgud3JpdGVfYnl0ZXMob3JpZ2luYWxfcmF3KQogICAgcmVwb3J0X3BhdGgud3JpdGVfYnl0ZXMob3JpZ2luYWxfcmVwb3J0KQogICAgZnJlZXplID0gdG1wX3BhdGggLyAiU0NJRU5USUZJQ19GUkVFWkUuanNvbiIKICAgIG9yaWdpbmFsX2ZyZWV6ZSA9IGZyZWV6ZS5yZWFkX2J5dGVzKCkKICAgIGFsdGVyZWQgPSB2NC5yZWFkKGZyZWV6ZSkKICAgIGFsdGVyZWRbImZpbGVzIl0ucG9wKG5leHQoaXRlcihhbHRlcmVkWyJmaWxlcyJdKSkpCiAgICBmcmVlemUud3JpdGVfdGV4dChqc29uLmR1bXBzKGFsdGVyZWQpKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9Im1hbmlmZXN0IG1lbWJlcnNoaXAiKToKICAgICAgICBydW5uZXIudmFsaWRhdGVfZnJlZXplKHRtcF9wYXRoKQogICAgZnJlZXplLndyaXRlX2J5dGVzKG9yaWdpbmFsX2ZyZWV6ZSkKICAgIGNvbnRyYWN0ID0gdG1wX3BhdGggLyAiQ09OVFJBQ1QuanNvbiIKICAgIG9yaWdpbmFsX2NvbnRyYWN0ID0gY29udHJhY3QucmVhZF9ieXRlcygpCiAgICBjaGFuZ2VkX2NvbnRyYWN0ID0gdjQucmVhZChjb250cmFjdCkKICAgIGNoYW5nZWRfY29udHJhY3RbInB5dGhvbiJdID0gInRhbXBlcmVkIGVudmlyb25tZW50IgogICAgY29udHJhY3Qud3JpdGVfdGV4dChqc29uLmR1bXBzKGNoYW5nZWRfY29udHJhY3QpKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9ImNoYW5nZWQgZnJvemVuIGZpbGUiKToKICAgICAgICBydW5uZXIudmFsaWRhdGVfZnJlZXplKHRtcF9wYXRoKQogICAgY29udHJhY3Qud3JpdGVfYnl0ZXMob3JpZ2luYWxfY29udHJhY3QpCiAgICBjaGVja3BvaW50ID0gbmV4dCgodG1wX3BhdGggLyAiZGV2ZWxvcG1lbnQiKS5nbG9iKCIqL2NoZWNrcG9pbnQucHQiKSkKICAgIHdpdGggY2hlY2twb2ludC5vcGVuKCJhYiIpIGFzIHN0cmVhbToKICAgICAgICBzdHJlYW0ud3JpdGUoYiJ0YW1wZXIiKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9ImNoZWNrcG9pbnQgaGFzaHxjaGFuZ2VkIGZyb3plbiBmaWxlIik6CiAgICAgICAgcnVubmVyLnZlcmlmeSh0bXBfcGF0aCkK'}
HASHES = {'LICENSE': 'f253151b912014fbdbd7283a4226dc34dd022f639a18eae552e847ad27d4ee7b', 'olympus/__init__.py': '3bc54c19fb529b525983b06e5233d11ef274a8046cc8ef71d67627809d1d958b', 'olympus/api.py': '4a1fc967440d75b7e14b4c093b0d404801d03845e5c318954c822e042ec05516', 'olympus/cli.py': '77ea945f430885185f671dd5b70f63d22d1e5d2d0b3f99463f0271630f585838', 'olympus/core/__init__.py': '990c5157585a49d86fe4e605b667f2985785f7f92b5cc1d3eff69151e6b8a9b6', 'olympus/core/budget.py': 'd4ea7843b31888d60c76410c3c6e22e279b38faeb942c43e6206e0541e54d01c', 'olympus/core/division.py': 'a189dc9f498c9abec25138c8b420d4d5a4d91fe518e7eb5d2420a45b7857dcd4', 'olympus/core/interpretation.py': '6110433e0965c62c33514f6977e8fcb3c3d6ade86ac943264b68b63751ff2ebe', 'olympus/core/mapping.py': 'efde41cf5caa58f64ea80e463413a93824cdacd0917ff10b34eb2a2f93b18e39', 'olympus/core/modalities.py': 'e31662d3f950bf3d87d269d8e61b02bad737d9ec39c52a0b46e9e921d0f2ce40', 'olympus/core/observability.py': '5184a3adf97a0e842f35929af47ed40f00aea99ae781d6a2be8452bbfef4508d', 'olympus/core/outcome.py': '6b06a54a4bc00b822fa3de8f77495938cb91dfbfb0166e8df4ec6031cbc3f296', 'olympus/core/perspective.py': '9067a6540a9b86a1c128932fb266ce10cb50a7ca5a8cf5b5be40c3539062c1a7', 'olympus/core/prediction.py': 'beef7ad50e25a54006abc3d95d46da259bf30673c956d2b0b50ed0999f611b16', 'olympus/core/representation.py': 'e369de60759ddfbfb2aa58624621e88a404c5db4c53b2b2127b6fa319ff4e465', 'olympus/core/retrodiction.py': '81f4baf7dda3a5e89bcefb2df665019b4b01646d3ea0848e2df4f12d94b57c2c', 'olympus/core/schemas.py': 'e89c498176fb86b52f1a49b8c9961aa8f5cb431ef6a953abfff09cbef1e597f0', 'olympus/core/security.py': '1e33d05c8316d39dec9bd4bb27879cb4ae8bbdad958a600ee8f7de1c98ad959b', 'olympus/core/sqlite_journal.py': 'd577ca0e3e5e0c1b3807386cc93efb797ae81c9a0f63c43cd7936db68c3845b5', 'olympus/core/transform.py': '2f608764a1cfe924c3d1b1b565bc053c17f4bd4faf3de10792ac76a9afdc034d', 'olympus/core/verification.py': '7f76edc54a733107798f994de38587d12ceccb61e9e512f1188a6077469f5623', 'olympus/core/workspace.py': '623f077705e27254712decda3160598ff895681abcf4af82f67c9e161d6b3f68', 'olympus/data/__init__.py': 'aed6813343b71dcbb8cedcedf695d690fae94525576445d660da03b7739b3cf4', 'olympus/data/conservative_routing_v6.py': '65697cb3cbb2ca640087f0bbd3d6c4e57d676e9352c377b60d8f1ce9fbb502a3', 'olympus/data/controlled_fusion.py': 'd23804807f19d465a50cb12fa223dc2950c73479973dbe477eff1b3e21bd10ad', 'olympus/data/editclevr.py': '9c1d1f50d01385e573eb1891b3633419f41facaed7e2918b9a7863e9075e55f0', 'olympus/data/ingestion.py': '6d6599177e01eaf84baf41d2c24e6026765cd189ff32027032865e7a9da072bd', 'olympus/data/manifests.py': '91875d590c50d0b3dca7126c6bca235d75cf7fe5ba9489e11498df7e9f2a7999', 'olympus/data/olympus_dataset.py': 'e612a824f6572490423d6894f57d2b7b9e4bf767dd83c2659fbde87c82ff96b3', 'olympus/data/oracle_fusion_v4.py': '93a2221fd675bf660eec86cdabd58085cb1917d08a484d16cc7c8f61951f191f', 'olympus/data/registry.py': '23834ae066cb9b38318baf819b7d23c176cb3d6bc00f003c8df0193ae011bb32', 'olympus/data/reliability_fusion.py': 'acf2d9d0592705e762548c068464d9f6c2d735274d0cd8c3b053f5353261c944', 'olympus/data/retrieval.py': '4316227d671a3dc8c0d7b49b4b76c9472658c2e183da030c428e107d4b0a8861', 'olympus/data/robust_fusion_v5.py': '6d0d92739543d83378a9da8b658e4f6645fe56947a69290240fede95d280d3e0', 'olympus/demos.py': 'ffed31e5a5eede5adaa0274f97588ee6135f71acdbb9c17293712b9860957a85', 'olympus/evaluation/__init__.py': 'f449c9ce782ca03d42fc8d52c372adad800e09326e700d611d382277c48f8eba', 'olympus/evaluation/decision_dossiers.py': 'a5ee4f8dee52d610e9cdc5d76867f3dc2c290a28f7082b75bb4d3933f120a932', 'olympus/evaluation/harness.py': '99d1966c0d7d783a3773c92b61c501e722cc8f25810610cb427f2823754527ec', 'olympus/evaluation/olympus_system.py': 'bac95fd017308fb8c1c60c68929f6d8bb9dd4c49b4c7a9d0a7674c69fcfc9103', 'olympus/forge/__init__.py': '6f11962b71aec233a5417d3dbfe118d5dc9440feeb9af9de79a6147d2fba6b9e', 'olympus/forge/compiler.py': 'c74bafa8606f28bb3ec24f5d3f0d723f252f02ec52359b2daaa6dd678263c2d9', 'olympus/forge/experiments.py': 'c5a8d11609420f52aeb7b8145be4ccb2ccb2ad3c133f03345ecf1c7572cf03e5', 'olympus/forge/language.py': '5d5a355469e8e6d20a768e0eed74fd8b09b2e7bed97a01ea2d9998de88f41af1', 'olympus/forge/runtime.py': '0beeff2d03954bc046ccc9844342539664822a47ce92b52e76754f55b35b0c17', 'olympus/foundry/__init__.py': '8478c8acc64a9acfd87e26b3ef436923caa7ca83afd4e6bf81434ab78b8a9431', 'olympus/foundry/bigram.py': '371ffe3ccd1f7f1e535ad18fa067924f8af0780448f54ec4e1d7848a0ba96f53', 'olympus/foundry/data_pipeline.py': 'ecc3a9b40279bc5b8796cfe0d8625a841ed40711aab7fe8b0b67e434acaacd24', 'olympus/foundry/eval_suite.py': '4ed8843b36fe89a56334843e42b83b520f428ace8cb350bf3353c3115e89f151', 'olympus/foundry/load_probe.py': '2444f85d880e7b894b4b8d0c6ec3c1782d0223b72dc984f08de712ab22cdc8ed', 'olympus/foundry/ollama.py': '4371570711b19015c2720691ac2c37535520e5b6b5c29972f5b3f363087827a3', 'olympus/foundry/promotion.py': '88885642ba659e34e09f475b5e9c93cfbb60c01b6e8d4b80408e553f180bb620', 'olympus/foundry/quantization.py': '5f1a0a41a5d5920210b03c30f1027fb42579a33080188b2a08ba3721452ac0f1', 'olympus/foundry/readiness.py': 'ddf2b29651cedd87e18205d53bfb388a384fdd5e4ff4f1a8323273ee0434943b', 'olympus/foundry/resources.py': '2a09fa333d409dd0f60e10212214a930508e84a1ff71fb3a3f674d4dc77b73ad', 'olympus/foundry/schemas.py': 'cb6bbc4e7f8362245f75a28f210b50bc7b4b54e36f8b8490b5a7c37b5953f19f', 'olympus/foundry/service.py': '869f0e277bed42fb2214e9944012b05652d184f167df9ca8c27007b40207b46e', 'olympus/foundry/sft.py': '5429e6e120106f50be085b6d2d820843a47dbc167b5e80c4c2670f9bbe60be87', 'olympus/foundry/store.py': '0450caa8686116c0f0b72b66907234e0ef4f60ebb2ebc6a43c1f592f8f648e0d', 'olympus/labos/__init__.py': '1785439a11bf594dc7a042eb9005ab0a2586e34ac8cf06d4e36231cee9a18ff7', 'olympus/labos/discovery.py': 'b700da24bb355946e245a0b8097cf75b8fd06d04be7b99ed8ddf32bdaddc8e58', 'olympus/labos/manifest.py': '9da9f6b93b905117af9cddca2a3186c7a77cd5fbf2dee39c53cb9e0819376be7', 'olympus/labos/portfolio.py': '768719f78da837fe015ad33d0d551dde3d0df0025452e3752f2aa20874b9b53a', 'olympus/labos/reports.py': 'ff174d504ead84ac8ecc9d634b9aaf733a783def25908bf6c8586f841fa94d46', 'olympus/labos/runner.py': '21fc396eb8bcb952fa24dc1fdcd7fab6207bd6ef2e4f2ef8ee4bcf9b670b42b6', 'olympus/labos/scheduler.py': '9a9fb0b1834c09d1d6de21f96e984c6cf8aca188093187265e71546fea8064cf', 'olympus/labos/validation.py': '3758f74c38e9148e585e675f7e064909243e384924827420133d4e574bd174a1', 'olympus/memory/__init__.py': '9a8ad140f0403fb7f1d2dbd2440db3201e5eb0a5807b67c8f9e74642959886f4', 'olympus/memory/store.py': '27823c8aac59e2718e420515504f88b12411ff99833859ea9c4452b087bfe2e1', 'olympus/models/__init__.py': '11daeb99f263347b641b68e84973ac84ce01af360294af85691912037855730a', 'olympus/models/capacity.py': 'ab6939d77cb61ae6eca707b16fef6e560314c675deca691290b2d3908a9b8c0e', 'olympus/models/conservative_routing_v6.py': '3b9067f8a518147396d8d3181e107f748990c2968bfa8968ee2977a1a5a26270', 'olympus/models/editclevr.py': 'ba122fee340e5d29f8525406e2fe5809fb74230df1981b0ce7a6a8e0bfefab4b', 'olympus/models/encoder_quantization.py': '047d212d11bed763cc79405adda670efb5fa406c5cb85350551fe526f92073b3', 'olympus/models/families.py': '1144b12e02ba5840157c0546bd541db4147edfa30da4f41a71f553a41a5616a2', 'olympus/models/flagship.py': 'e3f10f99e33133ceb981f2ae6e01bbb762c574f661afb55780a25cb80a34a640', 'olympus/models/olympus_system.py': 'ca9b40f292f62a599714ae756465ee1e46242ec9fa84eb07754ff8350857469f', 'olympus/models/oracle_fusion_v4.py': '1e3bcb8af4814ae303bbd846313f361c72fae8fa8310a11598ce003d2d0fa60a', 'olympus/models/parameter_growth.py': 'faf7eb277513dfbaab53ebeab63f13415dd7c9d2d95438863467c47d5c11a11b', 'olympus/models/reference_families.py': 'b0def64a3c2258cd8b19f2e5bf45f148061f6bcc0e232f14306841209ebc69e4', 'olympus/models/reliability_fusion.py': 'a1c2291fc47f552376791a4880cd9bd12bc039bbe0ed8fd6bfe51fabccd79f78', 'olympus/models/robust_fusion_v5.py': '124f0e7aa7b7f53ca837304b22b192875fa2a93dd2e8a19e8e61affebc235c85', 'olympus/models/runtime.py': '80dae9d7860901f19666fcd2caed40fbf287c50f3178800cea104560532dbe65', 'olympus/models/seven_model_contract.py': 'f8dd9a244fe1797d6b5389c39ab9783e063a80275b3ae48d4d4e4f5da2738154', 'olympus/models/spatiotemporal_lazy.py': '9c9a86d1cbc6ba1da8b9b354d30a5e94b9b46f206077952ea858c7160c703c0b', 'olympus/models/specifications.py': '9156dad138b72ddc69dbf13c3b056de5973606d92c9bbc77d27d944a0b2dd2d7', 'olympus/research_state.py': 'dc19a257db15da6dccec9a99ce5dcac1507c660c1263136d8e206df84255e07a', 'olympus/sdk.py': '1a3811422972240b013291b279418e55ead8a875240f04eaddedec0027807266', 'olympus/training/__init__.py': 'a6c17a8ec8867f67ec9f30764f9579f390f45189913638714a611a7d22dd05e4', 'olympus/training/campaign.py': '9f36427f65ac1480a4edacd7f7952c49cb5a94c9854c8e7dbbfa8159d3e6c830', 'olympus/training/checkpoints.py': 'd35a8ae76fea0a018efafb0728c7a7725044db3a135ebead9d5303385153ea7e', 'olympus/training/encoder_quantization.py': '24e07232438cc59503081c381bbf81023dd51b331b6c50fadd7e73d7bb6f9803', 'olympus/training/olympus_system.py': 'e32e529862878662617b43ba4533ce81a2a35077a5469fa029ddd97f4e19f10c', 'olympus/training/provenance.py': 'b048f0c004038f7a31fcfecc03e1f5d35dc1841b96b534e7172e2e36e0f9e3a3', 'olympus/training/synthetic.py': '1a7290890ea1a8c7750e333cbb0cd03db3ede25d5cd90729c0792548486e5888', 'olympus/training/trainer.py': '2aad19a32437cfb5c1d2672b331623450785c0fa2f90722bf8de92f0ff27ab05', 'olympus/upscaler/__init__.py': '6f1636998ce43e356911a1b5cfebef7e3393216782cb260b7ac549d9c08fd8fe', 'olympus/upscaler/jepa.py': '50676264f84cb2e7d40eb3642645ffe482f4f25f444b079f2eb53cdfa02b5a5c', 'olympus/upscaler/training.py': '2dc6d917cf4df68caa2601aadeda523946c2d00dddc1c81730e9c018a356227e', 'research/olympus_routing_v6/protocol.json': '3ac832bade0e5d1321ee5c6d305d81fe43d3806f119c209a817d2759a8d5cd7a', 'scripts/__init__.py': 'a3f27293bb35acb28573d82c89720dd2acf0b1f512637319d1614ade52742414', 'scripts/qualify_flagship_v2.py': '36f0ea4a0ec339c2e6bd331396bdbed0a51235cb7ab3168e2983ddb291efc56f', 'scripts/run_conservative_routing_v6.py': 'a9d080804819a0f253aab5395869270f438c57d5c3f4476db8592b6b1bc82723', 'scripts/run_oracle_fusion_v4.py': '560e742f7494a8e3df01d5872fa74d17ec576684ac30c44485ae7ca788416c58', 'tests/__init__.py': '34c86913f2bcd1b314d95de2d2a76e04f609e6fcedaf61c3a3f477d341e49a7b', 'tests/test_conservative_routing_v6.py': '51c189b56e362ba7321a3c182a1e358a9c0f52e04d55abc8074fa32d85bfb927'}
EXPECTED = 'e2773e00d757af02e92ef0f2bdc5d15b8de629fd94cf56d8636b47ef040035e9'
assert hashlib.sha256(json.dumps(HASHES, sort_keys=True).encode()).hexdigest() == EXPECTED
BASE = Path('/kaggle/working') / ('olympus-v6-' + EXPECTED[:12])
SOURCE, RESULTS = BASE / 'source', BASE / 'results'
SOURCE.mkdir(parents=True, exist_ok=True)
for name, encoded in PAYLOAD.items():
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError('unsafe source member')
    value = base64.b64decode(encoded)
    assert hashlib.sha256(value).hexdigest() == HASHES[name]
    destination = SOURCE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == value, 'Existing source differs'
    else:
        destination.write_bytes(value)
RESULTS.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(SOURCE), PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
           OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
def command(action, destination, *extra):
    subprocess.run([sys.executable, '-m', 'scripts.run_conservative_routing_v6',
                    action, '--output', str(destination), *map(str, extra)],
                   cwd=SOURCE, env=ENV, check=True)
def preserve():
    archive = BASE / ('retained-' + str(time.time_ns()) + '.tar.gz')
    with tarfile.open(archive, 'w:gz') as stream:
        stream.add(RESULTS, arcname='results')
    print('RETAINED', archive, hashlib.sha256(archive.read_bytes()).hexdigest())
print('SOURCE VERIFIED', EXPECTED)


In [ ]:
# Keep False: the first campaign has already been admitted on GitHub CPU.
# Enable only for a separately authorized independent reproduction, never a
# favorable replacement result. Missing dependencies fail without downloads.
RUN_INDEPENDENT_REPRODUCTION = False
qualification = RESULTS / 'qualification'
try:
    command('qualify', qualification)
    if RUN_INDEPENDENT_REPRODUCTION:
        command('run', RESULTS / 'independent-reproduction', '--qualification', qualification)
finally:
    preserve()
